# ARC-AGI-3 — combined solution v60 (v59 + three speed changes)

**Submission notebook: first impressions OFF** (`ENABLE_FIRST_IMPRESSIONS = False`) **and the difference image kept** (`V60_NO_DIFF_IMAGE = False`). v59c's clean full test (the advisers could not see the game being played) cleared 10-17% fewer levels per token than v45-v47 at 40-80K tokens, and both leaderboard runs with first impressions scored ~24 (v55 24.0, v59b 24.21) against ~28.5 for our other runs: changes to what the player reads cost points on the hidden games. The player is therefore v48's (temperature 0.7; the second opinion with the 25-game catalogue) plus the two mechanical speed changes: the effort ladder and lighter trims. `v60-test` keeps all v60 changes and the first impressions on, so its run measures the speed changes against v59c.

**v60 = v59 (contest code) with three changes that give the model more thinking tokens in the same 9 hours.** On the hidden games later levels weigh more, so +10% tokens is worth about +2.4 points. Each change has its own switch in the config cell; set it to False to return to v59's behaviour.

1. `V60_EFFORT_LADDER`: after a reply is cut off at the 12,288-token cap, the following requests think at 'medium' effort until an action changes the board (Franzen's built-in `ARC3_REASONING_EFFORT_LADDER`, off in his public notebook). Our 2-hour runs had 33-41 cut-off replies each, about 7% of all tokens.
2. `V60_LIGHT_TRIMS`: when the history fills, a trim keeps about 35K tokens instead of half the budget (about 52K). Reading prompts slows generation by ~15% (fit over ~1,000 server windows: 1,125 tokens/s with no prompt reading vs ~970 measured), and re-reading the kept history after trims is ~6.5% of generation; this about halves it. The average history depth drops ~12%; the level notes and shared rules are re-injected after every trim as before.
3. `V60_NO_DIFF_IMAGE`: no per-turn difference image (~400 tokens a turn). The current board image, the game-over diff image, animations, the level inventory and `frame_diff()` stay.

Kept from v59c: the clean test (in a test run the advisers' catalogue hides the game being played) and test runs wait for the model server before their clock starts. The contest rerun is unaffected by both.

**Run checks**: `[vllm ...s] READY` before the games start; in EXPERIMENT_RESULT `v60` shows the switches; `reply_effort` contains some 'medium' replies; `dedup_stats.trims` lower than v59B's rate; decode ~1,000+ tokens/s at 18 running; `fi_errors` 0; `gc_reader_unknown` 0.

---

## v59c

**v59c is a test notebook; the code that plays the contest is v59's.** v56 gave the fresh readers (first impressions + second opinion) a design catalogue with one line per public game. In a Save & Run on the public games that line describes the very game being played, which flatters the first impressions; the contest's hidden games have no line. In a test run (not `TRUE_SUBMISSION`) the readers now see the catalogue without that one line (`GC_LINE_OF_GAME`, counters `gc_reader_hidden` / `gc_reader_unknown`); in the contest it is unchanged.

The run is the full 121-minute evaluation (`COMMIT_PLAY_MINUTES = None`): all 25 games get seats, so it compares directly with the 121-minute runs v45/v46/v47/v49. Every test run waits for the model server before its clock starts (at most 75 minutes).

**Run checks**: `[vllm ...s] READY` before the games start; `gc_reader_hidden` close to `fi_asks + so_asks`; `gc_reader_unknown` 0; `fi_errors` 0; decode ~1,000 tokens/s at 18 running. Expect about 2.5-3.5 hours.

---

## v59b

**v59b = v59 with one change that only affects the short test run (Save & Run).** On Oct 24, v59's Save & Run got a slow machine: the model server was ready at 2,567 s, after the test's 30-minute window (released at 12 min, closed at ~42.5 min) had ended, so no game played and every score was 0. Now a test run releases the games only once the server is up (at most 75 min after the start), so its 30 minutes are all play. The contest rerun (`TRUE_SUBMISSION`) keeps the 12-minute release: the code that plays the competition is v59's.

**Commit run checks**: `[vllm ...s] READY` before the games start; `first impressions installed: True`; `fi_errors` 0; `fi_sync_delivered` close to `fi_asks`; `fi_new_lists` > 0; decode ~1,000 tokens/s at 18 running; levels > 0.

---

## v59

**v59 = v58 with one setting reverted.** v54-v58 ran the player at temperature 1.0 (the model card's thinking setting). v55, the first such run on the leaderboard, scored 24, and the serve logs show why it starts behind: at the same load temperature 1.0 decodes ~7% slower (931-942 vs 965-1,022 tokens/s; MTP mean acceptance 2.43 vs 2.69-2.70 in v47/v48/v49), i.e. ~7% fewer tokens in the 9-hour run, with no measured gain per token. So the player and the second opinion are back at v48's 0.7. The first impressions keep 1.0 (`FI_TEMPERATURE`; short requests, and their answers were judged good at 1.0 in v55): the side-request helper `_so_post` takes an optional `temperature` (None = the harness value, as before).

Against v48 the player is unchanged except for the first impressions it now reads on each level's first turn (v55-v58: two fresh model instances, the level's first board, the controls, the earlier rules, and - v58 - the colours that are new on this level).

**Commit run checks** (30 min): `first impressions installed: True`; `fi_errors` 0; `fi_sync_delivered` close to `fi_asks`; `fi_new_lists` > 0; decode ~1,000 tokens/s at 18 running; the usual v48 checks.

---

## v58

**v58 = v57 with two changes to the first impressions (section R of the add-ons cell).**

1. **They arrive before the player has its own reading.** v55's Save & Run: the two fresh answers took 8-39 s (median 23 s) but joined the player's prompt 1-3 turns into the level, after it had formed its own reading; it used them in about 1 of 22 deliveries. Now the level's first turn waits for them (at most `FI_SYNC_WAIT_S` = 75 s; if they are not back by then, they join a later turn as before). The wait costs the game wall time, not GPU time: the two side requests run on the server meanwhile.
2. **The fresh readers are told what is new.** The harness records the colours of every board the player sees, per level; at a level start, the objects of the first board whose colour appeared on no board of the earlier levels are listed (colour, cells, rows/cols; at most 5 colours) in the first impressions' facts and in the second opinion's facts. The player's own prompt is unchanged (v49's Goal Lens showed that noisy hints in the player's prompt are costly). On the 111 level starts of v47's run, 45 got a list (41%), e.g. tn36 L2 the new green panel, ka59 L3 the yellow pair, tu93 L2 the new red block, m0r0 L2 the lethal red tiles; worst 15 ms per turn.

**Commit run checks** (30 min): `first impressions installed: True`; `fi_errors` 0; `fi_sync_delivered` close to `fi_asks`; `fi_wait_ms` / `fi_asks` about 20-40 s; `fi_new_lists` > 0; the usual v48 checks.

---

## v57

**v57 = v56 with one fix found in v55's Save & Run.** The games are released 12 minutes after the start while the model server may still be loading (v55: ready after 26 minutes on a slow machine). The first impressions of the first 18 games were asked at once and all 36 requests were refused (`fi_errors` 36). Now a game asks only once the vLLM server reports ready (`fi_waiting_server` counts the turns that waited; a later turn of the same level asks), and a refused connection is retried 3 times, 10 s apart (`fi_retries`). Everything else is v56.

v55's Save & Run otherwise: healthy; per generated token it cleared levels at v48's rate (27 levels / 0.82M tokens vs 49 / 1.54M); temperature 1.0 costs ~10% decoding speed (MTP acceptance 2.43 vs 2.69); the delivered first impressions named the right goal on held-out games (ka59 L2: 'move each green piece into its matching outline, click a piece to control it' - exactly the rule; m0r0 L2: mirror pair + lethal checker tiles; vc33 L3-L4: align each band with its marker via the buttons).

**Commit run checks** (30 min): `first impressions installed: True`; `fi_errors` 0 (or only after a server restart); `fi_delivered` > 0; the usual v48 checks.

---

## v56

**v56 = v55 with the design catalogue extended for the fresh readers.** The first impressions (v55) and the second opinion read a catalogue of mechanics from other games of this benchmark in their system prompts. It covered 15 public games; the other 10 (ft09 g50t ka59 m0r0 r11l re86 s5i5 su15 tn36 vc33) had been held out as a local test set. In the contest every public game is just more evidence of how these designers build goals, so v56 adds one line for each of the 10 (from the rules our players stated after clearing them) and four lessons from comparing runs that cleared a level with runs that stalled on it (one no-op is weak evidence; a mirroring panel shows a recipe or hint; a missing action comes from the level's new element; a level that stays unsolved with understood controls means the goal reading is wrong). The player's own prompt is unchanged (`ENABLE_GAME_CATALOGUE` stays False): only the fresh readers see the catalogue.

**Commit run checks** (30 min): as v55 (`first impressions installed: True`, `fi_delivered` > 0, `fi_errors` 0, `so_errors` 0); read a few `first_impressions.jsonl` answers.

---

## v55

**v55 = v54 (v48 with the model's official thinking temperature 1.0) + rival first impressions** (`ENABLE_FIRST_IMPRESSIONS`, section R of the add-ons cell).

**Why.** 26 of the 41 levels our runs never cleared were cleared by the same model in another run, often within 9-21 moves - a stuck level is mostly a run that chose the wrong reading early. Read side by side, the two runs of tn36 level 2 show it: the run that stalled for 359K tokens discovered nearly every mechanic but settled early on a complicated interpretation (rotation pivots of a "ghost" piece); the run that cleared it tested the simplest reading first (the side panel shows the recipe - copy it). Our second opinion (section S) brings a fresh view only after 40K tokens on a level, when the lock-in has already happened.

**What it does.** At the first turn of every level, 2 fresh model instances (no history; sampled independently at temperature 1.0, so they disagree usefully) each look at the level's first board, the controls, the rules of the earlier levels and other games' rules, and give up to 3 hypotheses about the goal, each with its cheapest 1-3-action test - looking especially for what is new on the level and for anything that could be a target, preview, hint, recipe or control panel, simplest explanation first. They run in background threads (the player never waits); once both are back their answers join the player's next turn as ideas to test, never as instructions (header `FIRST IMPRESSIONS ... ideas to test cheaply, not facts`). Short thinking (the second opinion's bounded settings), ~3-6K generated tokens per level, charged to the game.

**Commit run checks** (30 min): `first impressions installed: True (2 per level start)`; ADDON_SUMMARY `fi_asks` ~ levels started, `fi_answers` ~ 2 x `fi_asks`, `fi_delivered` > 0, `fi_errors` 0; `first_impressions.jsonl` in the output (read a few answers); the usual v48 checks (server, REAP, 18 seqs, smoke tests, no errors).

---

## v54

**v54 = v48 with the sampling temperature raised from 0.7 to 1.0.** Qwen's model card for Qwen3.8-Flash-Next recommends, for thinking mode, `temperature=1.0, top_p=0.95, top_k=20, min_p=0.0, presence_penalty=0.0, repetition_penalty=1.0`, and the model's own `generation_config.json` sets temperature 1.0. Our player thinks in thinking mode (enable_thinking true, reasoning effort xhigh) but sampled at 0.7, the value inherited from da-fr's Milestone-2 notebook; top_p and top_k already match the card. Why it may matter on the hidden games: 26 of the 41 levels our runs never cleared were cleared by the same model in another run (often within 9-21 moves) - stuck levels are mostly runs that locked onto one idea, and a model run below its tuned temperature explores fewer ideas. Cost: none in seats or memory (18 games x 104K, every game starts); speculative-decoding acceptance may drop slightly. The second opinion (same harness setting) also samples at 1.0.

**Commit run checks** (30 min): the same as v48 (server, REAP, 18 seqs, smoke tests, no errors); in serve.log the mean acceptance length (v48 ~2.7; a small drop is expected); no repeating thinking blocks.

---

## v48 (First Light only for hopeless level 1s)

**v48 = v47 with First Light's trigger moved from 60K to 180K generated tokens on level 1** (`FL_MIN_LEVEL1_TOKENS`), and a 30-minute commit run (`COMMIT_PLAY_MINUTES = 30`; the contest always plays 532 minutes).

**Why.** In v47's Save & Run First Light cleared level 1 of m0r0 (11,742 search moves, a 15-move winning route), and the game lost most of its score: the model took the search's one history entry for a game move, tried to explain the win from it, misread the controls and stalled on level 3 - 2 levels / 11.7 points, against 5 levels / 69-71 points in v43, v45 and v46, where the model cleared that level 1 itself. Over 236 game-runs of our Save & Runs the model cleared level 1 in every game it had time for; 10% of those clears came after 60K tokens (where v46/v47's First Light steps in), none after 180K. At 180K First Light can only touch games that would otherwise stay on level 1 (score 0).

**What v47's Save & Run showed** (everything else in v48 is v47 unchanged): healthy (errors 0, restarts 0, preemption 0, KV peak 83% as estimated); 18 games at once give only ~+2% generation over 16 (963 vs 941 tok/s; the pruned model is near its limit at 16), harmless; the practice-game library was in 86 of 87 stalled-level blocks, every game id recorded, never a game's own rules, and the model used the rules as new goal hypotheses (12 of 87 blocks); levels cleared within the same generated tokens per game were the best of all seven 121-minute runs at every budget (40K 39, 80K 63, 120K 80, 150K 96, 200K 110; previous best 35 / 64 / 77 / 92 / 104).

**Commit run checks** (30 min): `--max-num-seqs 18` and REAP ok in the server lines; `cross-game practice library: 70 rules seeded, game names recorded`; first_light_smoke.json ok; survival_smoke.json ok; no errors.

---

## v47 (18 games at once + practice-game rule library)

**v47 = v46 + two changes, both switchable** (`VLLM_MAX_NUM_SEQS = 16` + `VLLM_HISTORY_BUDGET_K_BY_REAP[448] = 116` + `ENABLE_XG_LIBRARY = False` = v46 exactly).

**Why these two.** Forum data on hundreds of single submissions of da-fr's Milestone-2 notebook give 25.8 +- 3.9 points per run; our 18 leaderboard runs average 28.4 (about +2.6 over it). With run-to-run noise that large, a version change of 1-2 points cannot be seen in one score, so v47 only takes changes that cannot make play worse:
1. **18 games generate at once (was 16).** Measured on vLLM (v40's fit): 18 running games generate ~11% more tokens than 16 at the same load; above 18 the gain flattens. To keep the KV pool safe at the startup peak, each game's history is trimmed at 104K tokens (was 116K) back to ~52K, still inside the 50-116K range where our clear rate per token was flat. Estimated peak ~84% of the pool (v46 measured 83% at 16 x 116K). If the pruned model fails before the run starts, the full model runs v39's tested shape, 16 games x 88K (`VLLM_FALLBACK_SEQS_BEFORE_RELEASE`); if it fails during the run, 14 games (as before).
2. **Practice-game rule library** (`ENABLE_XG_LIBRARY`). The cross-game rules (v44: shown only to a game stalled on a level, at the 40K / 80K / 120K-token checks) used to come only from hidden games that had already cleared something, so the first stalls of a contest run got no rules at all. The pool now starts with 70 rules our own players wrote when they cleared levels of the 25 public games (25 level-1 rules with the goal first, 45 "what this level added"), cleaned and clipped exactly like in-run rules. They rank like any other rule but count as the oldest (an in-run rule wins a tie) and are labelled "a practice game". A game never sees its own rules: in a Save & Run each public game is matched by its game id and sees only the other 24 (a game whose id is unknown gets none), so the Save & Run measures the library honestly.

**Honest expectation.** About +0.5 to +1.5 points of true average, not a jump; each true point roughly doubles the gold odds in our private-board simulation.

**Save & Run checks** (121-min all25): `--max-num-seqs 18` in the server command; no preemption or restart; KV max below ~92%; generation at 18 running above v46's at 16; `cross-game practice library: 70 rules seeded, game names recorded`; ADDON_SUMMARY `xg_lib_pool` 70, `xg_named_games` 25, `xg_lib_blocks` > 0, `xg_errors` 0; no block in `cross_game_rules.jsonl` shows a game its own rules; plus everything v46 checks (first_light_smoke, survival_smoke, FIRST_LIGHT_SUMMARY).

---

## v46 (First Light + keep-alive ladder + cross-game rules)

**v46 = v45 + FIRST LIGHT: a CPU search for games stuck on level 1, run while they wait for the GPU** (`ENABLE_FIRST_LIGHT`; False = v45 exactly).

**Why.** A game stuck on level 1 scores 0 for the whole game, and level 1 carries the smallest weight, so moves spent on it cost almost nothing while clearing it opens levels 2, 3, ... to the model. In the contest ~110 games share 16 GPU seats: a game whose history was just trimmed hands its seat over and waits - time the CPU can use. A plain graph search with no language model - every key, a click on every object, boards remembered by hash, RESET + replay to reach unexplored boards - cleared level 1 of **16 of the 25** public games within 60-90 s (5 to 11,700 moves), including **3 of the 8** level-1s our model found hardest (cd82, ls20, m0r0): the two are good at different things. (A published search-only agent finished 3rd in the ARC-AGI-3 preview contest: arXiv 2512.24156.)

**What it does.** Only for a game still on level 1 after `FL_MIN_LEVEL1_TOKENS` (60K) generated tokens, and only while it waits at a seat handover: a background thread searches level 1 until the seat comes back (start delay 4 s, so a short wait costs nothing). Then, in the game's own thread:
- level 1 cleared: the harness's game state, history and runtime state move to the level-2 board, the per-level counters restart (the scheduler prices the game on level 2), and the model's next request carries a note with the moves that cleared it;
- not cleared: the board is put back exactly where the model left it (RESET + the model's own moves since its last RESET, checked cell by cell), with a one-line note.

Never on level 2 or later (there the model still clears 74% of its stalled levels at ~1.1x the human move count, and search moves would wipe those scores). Limits: 12,000 moves per game, 60,000 per run (about twice what a contest run plays itself), 2 games at once, none in the last 15 minutes. Any failure leaves the game to the model and is counted.

**Tested.** On the real game engine, offline and over HTTP against the real competition-mode game server: vc33 level 1 cleared by the search, ls20 stopped mid-search and restored exactly. Through stand-ins of the harness classes it hooks: cleared / restored / not eligible / short wait / failing search. In the Save & Run: `first_light_smoke.json` repeats the vc33 / ls20 checks on the real taaf GameAPI.

**Honest expectation.** About +0.5 to +1 point if many hidden games stall on level 1; it cannot show on public games, which our model nearly always clears. Risk: new code in the contest path, and extra game-server load from search moves (capped).

**Save & Run** (the 9-hour rehearsal notebook built from this version is the better test: with 100 games waiting for seats, First Light gets to run): check `first light installed`, `first_light_smoke.json` ok, `FIRST_LIGHT_SUMMARY` (jobs, cleared, restored, restore_mismatch 0, errors 0), `first_light.jsonl`, plus everything v45 checks.

---

## v45 (scorecard keep-alive ladder + cross-game rules)

**v45 = v44 + a keep-alive ladder for the game server's scorecard.** Play is v44's (v43 + cross-game rules).

**Why.** The identical v36 notebook scored 32.76 on the leaderboard and, resubmitted, 24.03 after running the full 9 hours (v7: 31.95, then 23.56). Our 17 leaderboard scores split into ~31 and ~25.5 with almost nothing between. The game server ends a run's scoring when it sees no game command for 15 minutes ("the run will end with the score so far" - Jeroen Cottaar, Tufa Labs, in the Kaggle thread on zero scores); our patient harness then keeps playing for hours while nothing counts, so a 9-hour run can still score low. v32's keep-alive covered this only while some game was still untouched at the start of level 1. Hours into a 110-game contest run there may be none left, and a model-server outage - the watchdog's restart re-reads the weights from network storage, ~10-12 minutes, plus the time to the first new move - would then close the scorecard.

**What v45 changes (submission only; a Save & Run plays offline).** When the game server has seen no command for 4 minutes, one RESET goes to the cheapest target:
1. a game at the very start of level 1 (unchanged board, one action counted on level 1) - as before;
2. a game at the start of a later level, right after clearing one (unchanged board, one action counted there);
3. only after 8 minutes of silence: a game still on level 1 with no points, part-way through it - its level 1 restarts, exactly what its own RESET does;
4. only after 8 minutes: otherwise the lowest-level game part-way through a level - that level restarts; cleared levels stay cleared.

Won games are never touched. A restarted game is a free target for the rest of the outage, so an outage costs at most one restarted level. Normal play sends commands every few seconds, so none of this fires unless the model server is down.

**Tested on the real game-server code** (arc-agi 0.9.8 = the competition wheel, contest versions of ft09/ls20/vc33, competition mode over HTTP, through this notebook's own survival cell): each tier picks the right game; tiers 1-2 leave the board unchanged and keep the level's counter at 0; nothing is sent between 4 and 8 minutes when only part-way games exist; tier 3 restarts ls20's level 1 at its start board; tier 4 restarts ft09's level 2 and level 1 stays cleared on the server's scorecard; the scorecard's idle timer restarts every time. The Save & Run's survival smoke test repeats these steps on Kaggle (`survival_smoke.json`: `ok` and `ladder_ok` true).

**Honest expectation.** This only matters if model-server outages happen in contest runs; we cannot see contest logs, so that is a likely explanation of the low runs, not a proven one. It costs nothing when there is no outage.

**Save & Run: 121 minutes on all 25 games** (this also validates v44's cross-game rules). Check: `survival_smoke.json` ok and ladder_ok; `cross-game rules installed: True`; `xg_errors` 0; the usual server checks.

---

## v44 (cross-game rules for the player)

**v44 = v43 + cross-game rules for the player (`ENABLE_CROSS_GAME_RULES`).** Everything else is v43: REAP-448 at load, 16 games at once, 116K history, exact speculative decoding.

**Why.** All games of a run are played by one process, and every level a game clears leaves that game's own `Rule for level N:` paragraph (goal, controls, what won): 104 of them in v43's Save & Run. Since v25 these rules are pooled, but only the second opinion read them, and it got the *newest* rule of each other game, mostly a late level's details ("Same maze; goal = reach green. Guard dots show ..."). In v43 1 of 74 second-opinion answers used one, and the player itself never saw any. In the contest that pool is knowledge about the hidden games, gathered from the hidden games while they are played: it cannot be fitted to the public games.

**What v44 adds.** When the level-progress facts fire on a stalled level (40K, 80K and 120K generated tokens on a level without clearing it), the same turn also gets up to 5 rules from other games of this run, as information with no instruction:
- stalled on **level 1** (the game has no rule of its own yet): other games' level-1 rules, each with its goal sentence first. Each one describes a whole game from scratch.
- stalled on a **later level**: what later levels of other games *added*, taken only from rules that name a new element ("New mechanic: colored key cells are pressure plates - while either block stands on one, all gates of that color open", "The new dark-red tile is a chaser ...", "a second, smaller mold was added ..."). A later level usually keeps its game's rules and adds one twist.

Each block shows one rule per other game: games with the most similar controls first (MOUSE only / arrows only / arrows + SPACE ...), rules this game has already seen last (so the 80K and 120K checks show others), never the game's own rules. The second opinion gets the same selection instead of the newest rules. A block is ~1,500 characters (~370 tokens), at most 3 per level. Captures that are not rules ("(write paragraph) Level 2: top row has ...", "as written above. Let me map level 5.") are dropped.

**Replayed on real runs through the notebook's own code** (every rule stated in time order with its game's controls, a block built at every real stall):
- v43: 104 stated rules -> 96 in the library (21 level-1, 8 junk captures dropped); at the 87 real stalls 84 blocks (4.0 rules each, mean 1,475 characters). Same on v35, v36, v39, v40, v41, v42: every check passed. Later-level additions found across those runs: pressure plates / buttons that open same-colored gates, chasers that start moving once you enter their line of sight, teleport pads, recolor and paint stations, toggles that latch a gate, collectible controls that unlock a button.
- With the switch off, every prompt and every second-opinion request is byte-identical to v43's (tested on a stub harness with the real wrapper chain); a failure inside the section leaves the prompt as in v43 and is counted (`xg_errors`).

**Honest expectation.** The public games share mechanics only loosely, and the 25 local games rarely stall on level 1 (11 of 87 stalls in v43, all cleared in the end). The gain, if any, comes from hidden games stalled on an idea that another hidden game has already revealed. A local run can only show that the mechanism works and costs nothing per token; the leaderboard decides. If it does not help, v43 stays the base.

**Save & Run: 121 minutes on all 25 games.** Check: `cross-game rules installed: True`; in `ADDON_SUMMARY` `xg_pool` ~90-100, `xg_blocks` ~80, `xg_rejected` small, `xg_errors` 0; `cross_game_rules.jsonl` in the output; then the usual: REAP ok, KV ~1.99M tokens, acceptance ~2.7, no looping thinking blocks, levels per generated token in the usual range.

---

## v43 (REAP-pruned experts + exact speculative decoding)

**v43 = v42 with exact speculative decoding again (relaxed MTP acceptance off).** Everything else is v42: the target model loads 448 of 512 routed experts per MoE layer (REAP), which leaves a 1.99M-token KV cache, so 16 games keep v36's full 116K history.

**What v42's Save & Run showed (121 minutes, 25 games).**
- REAP worked as designed: 448 experts kept in all 48 layers (258,048 expert tensors loaded, 36,864 skipped, routers checked), KV cache 1,985,472 tokens (v41: 1,412,504), peak use 81%, no restart, no fallback.
- Relaxed acceptance raised the mean acceptance length from 2.67 to 3.15 tokens per decode step, but it made the model loop: 34 thinking blocks of up to 50,000 characters that repeat one sentence hundreds of times ("The dots are at the divider's position..." 420 times), in ar25 (26), m0r0 (5) and sp80 (3). The same check finds none in v35, v36, v40 or v41. Those three games used 543K, 547K and 381K tokens and cleared about 10 levels fewer than in v41 (ar25 6 vs 8, m0r0 2 vs 5, sp80 1 vs 6); cut-off replies rose to 69 (v41: 44). Keeping a drafted token with probability min(1, p/0.6) makes a repetition that the model already finds likely almost certain, so a loop no longer breaks.
- Per generated token, v42 cleared levels at the rate of earlier runs: on the 9 standard games without the looping m0r0 it had 26 levels within 120K tokens per game (v36 26, mean of 22 runs 26.5) and 30 within 150K (v35 30, v36 33, mean 31.2). So the pruning itself shows no cost.

**Save & Run: 121 minutes on all 25 games.** Check: the `REAP:` line (status ok), no `relaxed MTP acceptance` line, mean acceptance length back to ~2.7, KV cache ~1.99M tokens and peak below ~92%, no thinking block that repeats itself, cut-off replies back near v41's rate, then levels per generated token in the usual range.

---

## v42 (REAP-pruned experts + relaxed MTP acceptance; not submitted: relaxed acceptance caused repetition loops)

**v42 = v36's play (full 116K history, no trim memory) with 16 games at once, plus two server-side changes: more KV cache from pruned experts, and faster speculative decoding.** v41 (20 games, history trimmed at 68K, trim memory) scored **26.64** on the leaderboard, far below v35/v36/v39 (30.75/32.76/31.67): on the harder hidden games, short history seems to cost more than extra games gain. v42 keeps v36's history and takes its speed from the server instead.

| | v36 | v39 | v41 | v42 |
|---|---|---|---|---|
| games generating at once | 14 | 16 | 20 | **16** |
| routed experts per MoE layer | 512 | 512 | 512 | **448** (REAP) |
| KV cache | 1.42M tokens | 1.42M tokens | 1.42M tokens | **~1.97M tokens** |
| history trimmed at, back to | 116K, ~58K | 88K, ~44K | 68K, ~35K + excerpts | **116K, ~58K** |
| MTP acceptance | exact | exact | exact | **relaxed, t = 0.6** |
| leaderboard | **32.76** | 31.67 | 26.64 | |

**1. REAP-at-load (`VLLM_REAP_KEEP = 448`).** Qwen3.8-Flash-Next routes every token to 10 of 512 experts per MoE layer. REAP (Router-weighted Expert Activation Pruning, Cerebras) ranks experts by how much they contribute when chosen; public REAP builds of this model drop the weakest. v42 drops the same 64 experts per layer while the target model loads: their weights are never loaded, the router loses their rows (so top-10 routing never picks them) and the MTP draft keeps all 512. The kept lists are those of the public `lee-chang-93/Qwen3.8-Flash-Next-NVFP4-REAP-k448` build: REAP saliency over 16.49M tokens of this model's agentic traffic including images; 92.65% of the saliency mass is kept (7.35% removed per layer on average, 9.1% at most). Checked here: all 48 routers of our checkpoint are bit-identical to the source of that build, and its routers are exact rows of ours (the hook re-checks every router's sha256 while loading and refuses anything else). The freed ~7.9 GiB of expert weights go to the KV cache, so 16 games can keep v36's full 116K history (expected peak ~80-87% of the pool; v36's 14 x 116K peaked at 97.8% of 1.42M), and each decode step reads fewer expert weights. Top-20 teams run pruned MoE builds; one public log (REAP-448 on SGLang) measured no score cost from the pruning itself.

**2. Relaxed MTP acceptance (`VLLM_SPEC_RELAX_T = 0.6`).** The MTP head drafts 3 tokens per step and the model verifies them. Exact verification keeps a drafted token with probability p (the model's own probability for it); v42 keeps it with probability min(1, p/0.6). In v41 the mean acceptance length was 2.67 tokens per step (per-position 0.75/0.73/0.71); relaxed, about 3.0 is expected, roughly +12% tokens per decode step. The cost is slightly greedier sampling where the draft agrees with a fairly confident model (p >= 0.6 is always kept). Patched in one line of vLLM's rejection sampler, only if the stock file's sha256 matches (else exact, logged).

**Safety.** If the pruned server dies before it is ready, the launcher starts the full model automatically (before the benchmark release: 16 games with v39's 88K history; after it: 14 games, v36's shape for the 116K history the harness already uses) and keeps the failed log as `serve-reap-failed.log`. `VLLM_REAP_KEEP = None`, `VLLM_SPEC_RELAX_T = None` and `VLLM_MAX_NUM_SEQS = 14` give v36's play exactly.

**Honest expectation.** So far the leaderboard has moved little with speed (SGLang at 10 games ~31.0, vLLM at 14-16 games ~31.7 with 40-60% more generation), so the speed part may add about a point at most; the full history is the protection. Pruning and relaxed acceptance change the model's outputs slightly, which the Save & Run must show to be harmless.

**Save & Run: 121 minutes on all 25 games.** Check, in this order: the `REAP:` line (status ok, keep 448, 48 routers, 12 tensors per expert) and `relaxed MTP acceptance ... 'patched': True` in the log; KV cache ~1.9-2.0M tokens; mean acceptance length ~3.0 (v41 2.67); generation tok/s at 16 running against v39's ~980; KV peak below ~92%; no restart or fallback; then levels per generated token on the 10 standard games inside the range of earlier runs, and no rise in cut-off or looping replies.

---

## v41 (20 games at once + trim memory; leaderboard 26.64)

**v41 = v40 + trim memory.** v40 (20 games at once, history trimmed at 68K) generated about 12% more tokens per second than 16 games, but once its history trims started it cleared about 20% fewer levels per generated token. v41 keeps v40's speed and gives back what a trim takes away.

| | v36 | v39 | v40 | v41 |
|---|---|---|---|---|
| games generating at once | 14 | 16 | 20 | 20 |
| history trimmed at, back to | 116K, ~58K | 88K, ~44K | 68K, ~35K | 68K, ~35K **+ excerpts** |
| leaderboard | **32.76** | 31.67 | not submitted | |

**What v40 showed.** On the 10 games played in about 20 earlier runs, v40 matched v35/v36 (same model and server) until its first trims, at 15-41K generated tokens per game. After them it fell behind every earlier run: at 150K tokens per game it had cleared 22 levels, against 26-33 for each of the 18 earlier runs (mean 29). In the turns right after a trim it cleared levels at about half its usual rate; v35/v36 show no such dip.

**What a trim loses.** Not the facts: every frame and action of the game stays in `history` in Python. A trim removes the model's interpretation - its findings, the ideas it ruled out, its plan.

**Trim memory (cell 6f).** When a trim drops assistant turns of the current level, the harness keeps the first and last 400 characters of each one's reasoning (usually the newest finding and the next test) and the actions that turn executed. They go, oldest first, into one message at the head of the retained history (at most 10,000 characters; the newest win). Nothing is summarised or rewritten, no extra request is made and no tokens are generated. The message changes only at a trim, when the cached prefix is lost anyway, so the prefix cache works as before between trims. A new level starts a new list (cleared levels are covered by the solved-level memory). Checked on all 278 trims of the v40 run with the real harness trimmer and v40's budget: 72% get the message (the others dropped no turn of the current level), median 9.4K characters = 7.7% of the trimmed request, every request within budget, and nothing changes between trims.

**Save & Run: 121 minutes on all 25 games**, as for v40. What decides it is the number of levels per generated token after trims on the 10 standard games (v40: the lowest of all runs from 80K tokens per game on; v35/v36: in the middle). If v41 gets back to v35/v36's level, it keeps v40's speed, about 23% more generation than v36's 14 games. Also check: `trim_memory_stats` in the result line (trims and excerpts > 0, errors 0), no server restart, KV peak below ~92%.

---

## v40 (20 games at once; not submitted: its history trims cost more than its speed gained)

**v40 = v39 with 20 games generating at once instead of 16, and history trimmed at 68K instead of 88K.** Leaderboard at the time: v35 30.75, v36 **32.76** (best), v39 31.67.

| | v36 / v37 | v39 | v40 |
|---|---|---|---|
| games generating at once (`VLLM_MAX_NUM_SEQS`) | 14 | 16 | **20** |
| history trimmed at (`VLLM_HISTORY_BUDGET_K`), back to | 116K, ~58K | 88K, ~44K | **68K, ~35K** |
| Save & Run | 14 games, 68 min | 25 games, 30 min | 25 games, **121 min** |

**Why more games at once.** The model has 512 experts with 10 active per token, and its full-attention layers read only 2,048 selected tokens. So a decoding step costs mostly the expert weights it loads. With 16 games × 4 MTP tokens, a step already touches about 72% of the experts; with 20 games, about 79%. Each extra game adds tokens to the step at little extra cost. Measured so far: 882 generated tokens/s at 14 running, 967-982 at 16 (+42 tok/s per game). Expected at 20: about 1,050-1,120 tok/s (+10-15%).

**Why the history must shrink.** GPU memory is fully used: weights 71.9 GiB, KV pool 19.7 GiB = 1.415M tokens. The pool is already fp8, and the n-gram table and token embeddings already sit in host RAM. So games × history per game is fixed. The KV replay model (built from v35/v36 request logs; it gave 86-89% for v39, which measured 91.7%) gives these peaks (median / 90th percentile over 40 game mixes):

| | first 25 min of play | after 25 min |
|---|---|---|
| 16 games, trim at 88K (v39) | 86% / 89% | 80% / 83% |
| 20 games, trim at 70K | 85% / 87% | 83% / 87% |
| **20 games, trim at 68K (v40)** | **81% / 82%** | **79% / 82%** |

After a trim, a game keeps about 35K tokens: the system prompt plus about 8 of its latest turns (v39: about 11, v36: about 15). Between 50K and 116K the level-clear rate did not depend on history size; below that it is untested. This is v40's main risk.

**Save & Run: the full 121 minutes** (`COMMIT_PLAY_MINUTES = None`) on all 25 public games, so games trim several times. v39's 30-minute run trimmed only 16 times, all near the end, so it never tested play after a trim. What to check: no server restart, KV peak below ~92%, generation at 20 running (expected ~1,050-1,120 tok/s), no preemptions, and levels per generated token after trims compared with earlier runs.

---

## v39 (16 games at once with a startup-safe history budget; leaderboard 31.67)

**v39 = v38 with a history budget that survives the startup peak.** Best leaderboard score at the time: v36, **32.76** (Oct 16).

| | v36 / v37 | v38 | v39 |
|---|---|---|---|
| games generating at once (`VLLM_MAX_NUM_SEQS`) | 14 | 16 | 16 |
| history trimmed at (`VLLM_HISTORY_BUDGET_K`), back to | 116K, ~58K | 100K, ~50K | **88K, ~44K** |
| Save & Run | 14 games, 68 min (v37: 10 min) | 25 games, 121 min | 25 games, **30 min** |

**What went wrong in v38.** About 15 minutes into play the harness logged `analyzer endpoint unreachable (ConnectionError)`: the model server had stopped. The run was cancelled, so there is no server log, but the timing points to the KV pool running out. v38's budget was sized for the long-run average. At the start, however, every game grows its history together until its first trim. v36's own KV curve, scaled from 14 to 16 games, passes 100% in exactly play minutes 10-18.

**How v39 is sized.** A replay model of KV use, built from the real per-game request logs of v35 and v36, reproduces their measured curves (model peaks 91-98% against 95-98% measured). Its peak KV use per setting (median / 90th percentile over 40 random game mixes):

| | first 25 min of play | after 25 min |
|---|---|---|
| 14 games, trim at 116K (v36) | 99% / 102% | 93% / 97% |
| 16 games, trim at 100K (v38) | 98% / 100% | 95% / 98% |
| **16 games, trim at 88K (v39)** | **86% / 89%** | **80% / 83%** |

v39 trims at 88K, back to about 44K. Between 50K and 116K of history, the level-clear rate showed no dependence on history size (v38 notes below); 44-50K is slightly below what our runs cover. The extra speed is v38's: each running game adds about 41.6 generated tokens/s, so 16 games give about +7.5% compute, worth about +1.7 points in the contest simulator. The table also shows that 14 games at 116K (v36) run close to the limit at the start.

**Save & Run: 30 minutes of play** (`COMMIT_PLAY_MINUTES = 30`) on all 25 public games. That is long enough to pass the startup peak (play minutes 10-25) and short enough to be cheap. The contest run ignores it and plays its 532 minutes. What to check: no server restart, KV peak below ~92%, the KV pool size at 16 games, generation speed at 16 running (v36: 880 tok/s at 14).

---

## v38 (16 games at once; its server stopped at the startup KV peak, fixed in v39)

**v38 = v37 with 16 games generating at once instead of 14.** v36 scored **32.76** on the public leaderboard (Oct 16), our best so far (v35: 30.75). v37 is v36 with v35's exact server command. v38 changes only the number of games at once and the history budget that makes room for them; play, prompts, add-ons and the scheduler (`token_scale` 700K) are unchanged.

| | v37 | v38 |
|---|---|---|
| games generating at once (`VLLM_MAX_NUM_SEQS`) | 14 | 16 |
| history per game (`VLLM_HISTORY_BUDGET_K`) | trimmed at 116K tokens, back to ~58K | trimmed at 100K, back to ~50K |
| Save & Run | 14 games, 10 min (commit run) | all 25 public games on 16 slots, 121 min |

**Why more games at once.** Each decoding step reads the model's weights once, and every running game adds its tokens to that step. Over 753 ten-second windows of v35's and v36's Save & Runs, each extra running game added **41.6 ± 1.8** generated tokens/s, steadily from 8 to 14 games (fit: generated tok/s = 407 + 41.6 × running − 0.047 × prompt tok/s; KV fill had no effect). At 16 games that is about +9%, about +7.5% after the extra prompt work.

**Why a smaller history budget.** The KV pool (1.42M tokens) is what held us at 14: with 14 games it is 94% full in the busiest 10% of moments. Trimming at 100K instead of 116K lets 16 games use the same KV that 14 use now. Does the shorter history cost levels? Across 19 Save & Runs, after each game's first trim (415 level clears, 23.3M generated tokens), the level-clear rate does not depend on how much history the model holds:

| context when the level was cleared | 50-65K | 65-80K | 80-95K | 95-110K | 110K+ |
|---|---|---|---|---|---|
| clears per 1M generated tokens | 15.9 | 20.3 | 16.3 | 17.9 | 18.1 |

From just after a trim to just before the next one, the rate is flat too (14-23 per 1M, no trend). The model's level notes and shared rules are re-injected after every trim, which is probably why the oldest history matters little.

**Contest simulator** (realistic persistence, six hidden-difficulty worlds, 120 contests each, same random draws for both): 16 slots at +7.5% speed beat v36's 14 slots by **+1.7 points** on average (+0.9 to +2.3 by world, never negative). If the shorter history made the model need 5% more tokens per level, the gain would be +0.5; if the speed gain were only +4%, +0.9. At 16 slots, `token_scale` 700K, 850K and 1000K are within 0.1 of each other, so it stays at 700K.

**Save & Run.** The 14-game set cannot fill 16 slots, so this Save & Run plays all 25 public games for 121 minutes (532 × 25 / 110: the contest's compute per game). That keeps 16 games running and lets the scheduler choose, as in the contest. Its local score is not comparable with the 14-game runs. What to check: generation speed at 16 running (v35/v36: 880 tok/s at 14), KV peak and preemptions (v36: 98%, none), errors.

---

## v37 (cache experiment rolled back; v35's server command)

**v37 = v36 with its cache experiment rolled back.** The model server is v35's exactly (v35: **30.75** on the public leaderboard, Oct 15). Play, prompts, add-ons and the scheduler (`token_scale` 700K) are v36's.

| | v35 | v36 | v37 |
|---|---|---|---|
| `--prefix-cache-retention-interval` | 0 | left out (runtime default) | 0 |
| scheduler `token_scale` | 400K | 700K | 700K |
| reply-cache probe (Save & Run only) | no | yes | no |
| Save & Run play | 68 min | 68 min | 10 min (`COMMIT_PLAY_MINUTES`) |

**Why.** v36 tried to let each turn reuse the previous reply from the prefix cache. Its Save & Run (Oct 15) showed that it does not: none of 1,348 follow-up turns resumed inside the previous reply (v35: none either), and prompt work stayed at 3.6K tokens per turn. Whatever the retention setting, this runtime only resumes at checkpoints made while reading a prompt. The setting gained nothing and kept extra checkpoint blocks in the cache: v36 computed 9.71M prompt tokens against v35's 9.34M, and long contexts were re-read 6.0% of the time against 4.9% (both within run-to-run noise). Generation speed and KV use at equal load were the same. v36 then scored 32.76 on the leaderboard against v35's 30.75, a gap within submission-to-submission luck; the cache setting is not the reason.

The probe reported a mismatch, but it never reached the reply. It compared a turn resumed at the end of the previous prompt (what every v35 turn does) with the same turn computed from scratch. The two agreed on the first 4 of 16 greedy tokens, with log-probabilities within 0.13, and then diverged after a near-tie. That is the ordinary difference between two computation orders: lordhansolo's notes say this runtime is not bitwise identical across batch shapes. It is not a broken cache.

**Commit run.** Play (v36) and server (v35) are both measured, so `COMMIT_PLAY_MINUTES = 10` keeps this Save & Run short. The competition run ignores it and keeps its 532 minutes. Set it to `None` for a full 68-minute local evaluation.

---

## v36 (reply-cache experiment, rolled back in v37; scheduler for 14 slots kept)

**v36 = v35 + three small changes.** v35 scored **30.75** on the public leaderboard (Oct 15), our best since Oct 11. Play, prompts and add-ons are unchanged.

| | v35 | v36 |
|---|---|---|
| vLLM prefix cache: model-state checkpoints kept | only at the end of each prompt (`--prefix-cache-retention-interval 0`) | at every 1,664-token block, replies included (runtime default) |
| scheduler `token_scale` | 400K | 700K |
| watchdog host-memory log | path missing (its test-run monitor stopped at start) | written to `vllm-host-memory.jsonl` |

**1. Reuse the previous reply from the cache.** In v35's Save & Run, 99% of turns re-read the model's whole previous reply: 4.1K prompt tokens per turn, against 1.8K on SGLang in v33. The server kept the model's internal (Mamba) state only at the end of each prompt, and the reply comes after it. Over the run the server computed 3.0 prompt tokens per generated token, and with 14 games, every 1,000 prompt tokens/s cost about 56 generated tokens/s. Keeping every aligned checkpoint lets the next turn resume at the last block boundary inside the previous reply. From v35's logs that is 11-22% less prompt work and about 2-3.5% faster generation. lordhansolo's production runs used 0, so this path is new to us: in a Save & Run, a probe (`vllm_probe_reply_cache`) runs one turn twice, once resumed inside the previous reply and once computed from scratch, and checks that both predict the same next tokens; a wrong checkpoint would silently change what the model sees. `VLLM_PREFIX_RETENTION_INTERVAL = 0` restores v35.

**2. Scheduler for 14 slots.** The contest simulator, run with v35's 14 slots (realistic persistence, six hidden-difficulty worlds, 150 fresh contests each), finds `token_scale` 700K better than 400K by +0.55 on average: +1.0 to +1.3 when hidden games are 1.6-2.5x harder, -0.27 in a world where 15% of levels are impossible. It only acts in the contest; a Save & Run cannot show it.

**3. Watchdog log.** lordhansolo's watchdog samples host RAM every 15 s in test runs. v35's config lacked the file path, so that thread stopped at start (harmless; it never runs in the contest). v36 gives the path, so the Save & Run shows RAM headroom.

---

## v35 (vLLM serving stack)

**v35 = v34 with a faster model server.** The harness, its add-ons and the prompts are unchanged. Only the program that runs the model changes, so that more tokens are generated per minute.

**Why.** In the contest the score grows almost one-for-one with the tokens the model generates. In our contest simulator, with hidden games twice as hard as public ones, +10% tokens gives about +3 points and +25% about +8. A public comparison of the three best public notebooks, all running Qwen3.8-Flash-Next on one RTX PRO 6000, measured:

| | v34: da-fr's SGLang Pennyroyal | v35: lordhansolo's vLLM main e975732 |
|---|---|---|
| model weights | Intel W4A16 AutoRound + albucino drafter | primitive-ai mixed NVFP4/FP8 with its own MTP head |
| token cache (KV pool) | 1.01M tokens | about 1.4M tokens |
| games thinking at once | 10 | 14 |
| decode speed, median (Save & Run logs) | 722 tok/s | 976 tok/s |
| while a long prompt is read in | every game stops generating (6.4 of 48 min in v33) | prompt chunks are mixed into decode steps |

**Kept from v34.** The same chat template (byte-identical file), the same tokenizer for our text, 640-pixel board images (lordhansolo used 256), temperature 0.7, our context budget, the scheduler and every add-on. The model's past reasoning stays in the prompt: once the server is up, three one-token probe requests check which message field (`reasoning` or `reasoning_content`) this server renders, and the harness stores the reasoning under that one. One field only: the harness's own token count includes every field, so two copies would make it trim history too early.

**Credit.** The vLLM runtime, its overlay patch, the server watchdog and the 32K draft vocabulary are lordhansolo's: Kaggle notebook lordhansolo/arc-agi-3-milestone-2, dataset lordhansolo/vllm-main-e975732-arc3, model lordhansolo/qwen3-8-flash-next-mixed-nvfp4-fp8 (weights from primitive-ai). The draft vocabulary is embedded here and checked against its sha256.

**Inputs to attach**, in addition to v34's: the dataset `lordhansolo/vllm-main-e975732-arc3` and the model `lordhansolo/qwen3-8-flash-next-mixed-nvfp4-fp8` (framework PyTorch, variation hf-mixed-mtp-nvfp4, version 1). Keep v34's inputs attached: they are the fallback.

**Safety net.** If the vLLM server cannot start (an input is missing, the runtime patch fails, it runs out of memory while loading), or if its watchdog gives up after 5 restarts during the run, v34's SGLang server starts instead (`VLLM_FALLBACK_TO_SGLANG`). `SERVER_ENGINE = 'sglang'` runs v34 exactly.

**Risk.** The NVFP4/FP8 weights are a different quantization of the same model. Their publisher reports equal accuracy, but nobody has measured them on ARC-AGI-3. The first Save & Run plays 14 games for 68 minutes (`LOCAL_GAME_SET = 'mixed14'`: the same GPU time per game as the contest, with 14 games thinking at once as in the contest; the 10 games of earlier runs are scored separately for comparison) and checks the server that was used, its token-cache size, the tokens generated per game against v33, and that each prompt still carries the model's past reasoning (`check_run.py`).

---

## v34 (contest scheduler retuned)

**v34 = v33 + two retuned scheduler numbers.** What the model reads and does is identical to v33. Only the choice of which waiting contest game gets one of the 10 model slots changes.

**Why the scheduler.** In the contest, about 110 games share 10 model slots for about 9 hours, so the slot priority (affectify's D′) decides where the compute goes. Local Save & Runs never test this, because there 10 games have 10 slots. We built a contest simulator from our 19 recorded Save & Run runs. It uses the per-level token costs of 10 public games, made harder in several ways to stand in for the hidden set, and the notebook's exact D′ formula (checked equal on 20,000 random states). In it, the way compute is shared moves the score a lot: with hidden games 2x harder than our public-game runs, sharing equally scores about 22, D′ about 28, v34 about 30, and an allocator that knew every level's cost in advance about 46.

**The change** (`DPRIME_OVERRIDES` in the config cell):
- `level_pow` 0.5 → 0.75: deeper levels count more, as in the scorer (level i weighs i).
- `token_scale` 225K → 400K: a level that has already taken many tokens is given up more slowly. In our runs, continuing a long-stuck level cleared it about twice as often per token as starting it fresh.

**Simulated effect** (110 games, 10 slots, 520 minutes, 30 seeds; game difficulty kept only as persistent across levels as in our runs):

| hidden set, compared with our public-game runs | D′ (v33) | v34 | change |
|---|---|---|---|
| same difficulty | 50.4 | 50.4 | +0.0 |
| 1.6x harder | 35.2 | 36.8 | +1.6 |
| 2x harder | 27.8 | 30.0 | +2.2 |
| 2.5x harder | 21.7 | 23.6 | +1.9 |
| 15% of levels impossible | 26.7 | 26.4 | -0.3 |
| 35% of games hopeless | 33.3 | 34.0 | +0.7 |

The expected gain is about +1 point: small, but positive in almost every case. The cost is that about 10-15 of the 110 games may never get a slot, because their compute goes to games that are progressing. The simulator counts those games as 0 and still comes out ahead. If every game is first given a turn, about half the gain is lost. Larger changes gain more where hidden levels are merely hard and lose more where they are impossible, so we kept the change moderate.

**Checks.** A Save & Run cannot show the effect, because with 10 games for 10 slots the scheduler never has to choose. It shows the installed values in the `#OURS_FORM ok version=d_prime ... level_pow=0.75 ... token_scale=400000` line and in `experiment_result.json` (`dprime_params`). Setting `DPRIME_OVERRIDES = {}` restores v33 exactly.

---

## v33 (game-count guard)

**v33 = v32 + one competition guard.** Play is identical to v32.

**Game-count guard.** The contest run lets at most `bm.solver.concurrency` = 120 games run at once (da-fr's note: "game count should be 110"). A game keeps its place until it wins or its 532-minute budget ends, so with more than 120 hidden games the extra ones would wait until the run is over and score 0. v33 raises the limit to the number of games the game server lists. With 110 games nothing changes. The 10 model slots (priority gate) still decide which games are thinking at any moment.

**Not adopted: the "fresh review" context restart** from the public notebook *try-merging-it-you-might-get-lucky-and-win-gold* (da-fr's notebook plus one change: once per level, after 96K generated tokens without clearing it, the conversation history and the retained Python functions are deleted and the model starts over from the current board). Checked on our 19 recorded Save & Run runs (192 game runs, 63 levels that reached 96K tokens):

| restart after | levels reaching it (per 10 games) | cleared by the continuing context | clears per 100K tokens: continue / clean start | levels gained by restarting (per 10 games) |
|---|---|---|---|---|
| 48K | 11.2 | 69% | 1.93 / 0.81 | -1.7 |
| 64K | 6.8 | 62% | 1.62 / 0.78 | -0.7 |
| 80K | 4.5 | 52% | 1.41 / 0.73 | -0.3 |
| **96K (the fork)** | 3.3 | 48% (median 17K more tokens) | 1.51 / 0.70 | **-0.2** |
| 128K | 1.4 | 22% | 0.99 / 0.73 | +0.04 |
| 160K | 0.5 | 0% | 0.00 / 0.11 | +0.01 |

Half of the "stalled" levels at 96K were close to solved: the continuing context cleared them a few turns later. A clean start on the same level (measured in our other runs) needs a median 44K tokens (middle half 28-66K) and clears at half the rate per token. Deleting the context is therefore a loss wherever it fires often, and only neutral where it almost never fires. Our second opinion (v24-v28) already gives a stalled level a fresh model's view without deleting the player's work.

---

## v32 (competition survival)

**v32 = v31 + competition survival.** Play is unchanged from v31; what changes is how the notebook survives the parts of a real submission that a Save & Run never exercises.

**Why.** The unchanged v7 scored 31.95 on Oct 8 and 23.56 on Oct 14; every score since Oct 11 is 26.9 or lower. Our code did not change, so the cause is in the submission environment, and the submission path is the one part a Save & Run cannot see: there every game action is an HTTP call to Kaggle's game server. Reading arc_agi's own client and server code shows two ways that path can cost a large part of a run:

* **A slow answer is dropped.** The client gives each game-server call 10 s; a slower answer comes back to the model as an error although the server may already have applied the action. v32 waits 60 s.
* **An idle scorecard is closed.** The server closes a scorecard after 15 minutes without a game command, and in competition mode one scorecard holds every game, so no game can act for the rest of the run. Nothing is sent while the model server loads (886 s on Oct 13, 260-510 s before) or while the watchdog relaunches it after a crash. After 4 idle minutes v32 sends one RESET to a game standing at the very start of level 1: the server answers with the unchanged board and the scorecard's idle timer restarts. A game that has acted on its level is never touched.

Games also wait up to 40 min for a slow model-server start (was 15) and survive about 100 min of failed model requests (was about 25), so a slow load or a relaunch no longer makes them give up.

All of it is tested on the real arc_agi server code: locally (`test_v32_survival.py`, including the control where the stock client drops a 12 s answer) and again at the end of every Save & Run (`survival_smoke.json`).

---

## v31 (speed fix + stuck-level help)

**v31 = v30 with exact reply replay and history de-duplication switched on together**, as v20 ran them. Nothing else changes.

**Why (v31).** Our two measured improvements are:

* **The speed fix** (exact reply replay with history de-duplication, v20). Locally the server re-reads far less, and runs with replay generated about 6.5% more tokens (1.81M against 1.70M). On the leaderboard, v20 (replay with de-dup) scored 30.37, while v21, v28 and v29 (replay without de-dup) scored 26.79, 26.01 and 26.87, so v31 keeps the two together exactly as v20 did.
* **The stuck-level help** (level-progress facts plus a background second opinion). Pooled over four local runs, it cleared 77% of stuck levels against 70% without it.

Neither is proven on the hidden games, so the remaining submissions run a 2x2 test: v7 (neither), v30 (help only), v20 (speed fix only) and v31 (both), interleaved so that every version sees the same conditions. Averages over several scores decide; no single score does.

---

## v30 (v7's base + stuck-level help)

**v30 = v29 with exact reply replay switched off.** That is v7's competition base exactly (no reply replay, no history de-duplication, 60 Mamba slots), plus the level-progress facts and the background second opinion.

**Why.** The leaderboard scores by setting:

| Version | reply replay | history de-dup | score |
|---|---|---|---|
| v7 | off | off | 31.95 |
| v20 | on | on | 30.37 |
| v21, v28, v29 | **on** | **off** | 26.79, 26.01, 26.87 |

The disabled de-duplication cell installs nothing, so "replay on, de-dup off" is the only competition-relevant setting shared by all three low scores. No mechanism showed in local runs, and with replay on the local runs looked healthy. But three scores this low are very unlikely by chance, so v30 removes the suspect and keeps the stuck-level help. Locally, that help cleared 84% of stuck levels (41 of 49) against 70% without it. Without replay the server re-reads the previous reply more often (as in v7), costing a few percent of speed.

---

## v29 (v28 on da-fr's server memory split)

**v29 = v28 with the Mamba state pool back at 60 slots** (da-fr's value, used by v2-v20). Nothing else changes.

**Why.** Every leaderboard score with 60 slots was 30.4-32.0 (v2 31.22, v5 30.44, v7 31.95, v20 30.37). Both scores with 50 slots were low (v21 26.79, v28 26.01). By chance alone that pair would be expected well under 5% of the time. Locally, 50 slots never constrained anything: exactly 40 were in use with 10 games running, and no request ever waited. But the competition keeps all 10 seats busy for 9 hours while 110 games cycle through them, which no local run reproduces. Going back costs about 4% of token-cache space, which never mattered locally. Keeping a change that coincides with the two lowest scores of the family is the bigger risk.

---

## v28 (background second opinion)

**v28 = v27 with the second opinion fetched in the background.** Everything the second opinion sees and says is v27's.

**Why.** The v26 Save & Run proved the bounded second opinion: 21 of 21 answered, about 1,300 tokens and 22 s each, and the answers were specific and testable. But the game waited idle for each answer, and the server evicted its cached prompt meanwhile. The request right after a second opinion re-read its whole context in 15 of 24 cases (1.4M tokens, about 4% of the run). With the same level-progress facts and no second opinion (v23), that happened in 2 of 30 cases. The same pattern appeared in v24 and v25.

* **Background request.** When the level-progress facts fire, the game's own request goes out at once and a background thread asks for the second opinion. The answer is appended to the game's next turn, which normally starts after the second opinion is back, because the player's own reply takes longer than the second opinion's ~1,300 tokens. If the level is cleared before the answer arrives, the answer is dropped. An answer that takes more than 10 minutes is dropped.
* **Isolation.** The background request is built from read-only fields of the agent (model, server URL, headers). It never changes the agent, so the player's own requests, which can now run at the same time, keep their normal reasoning effort. Its tokens are charged to the game when the answer is delivered or dropped.
* New counters: `so_delivered`, `so_discarded` (level already cleared), `so_dropped` (too slow).

---

## v27 (second opinion in grid coordinates)

**v27 = v26 + one line in the second opinion's request.** In the v25 run, 3 of the 17 second opinions finished within the old 8,192-token cap. They were specific and useful: the ka59 level-2 answer named exactly the move that cleared the level. But one of them gave positions such as "(107, 273)", pixel coordinates of the 10x-enlarged board image, while the player clicks grid cells 0-63. The request now states the grid size and the image scale and asks for grid (row, col) positions only. Everything else is v26's.

---

## v26 (second opinion that answers)

**v26 = v25 with the second opinion's answer bounded.** Everything else is v25's.

**Why.** The v24 Save & Run showed all 17 second opinions empty. Each one spent the whole 8,192-token cap thinking (finish = length, 8,192 reasoning tokens) and took 133-174 s, so about 8% of the run's tokens and over 40 slot-minutes bought nothing.

* **Short thinking first.** The side request now asks the model for brief thinking (the chat template's `reasoning_effort` "low", the same setting the player never uses) with a 3,072-token cap.
* **Direct answer as a fallback.** If that still ends without an answer, one more request asks again without thinking (700 tokens). So a second opinion costs at most about 3.8K tokens instead of 8.2K, and it nearly always answers.
* The low-effort setting exists only on the agent instance for the duration of the side request; the player's own requests keep their default effort (checked in the tests). New counter `so_fallbacks`; the Save & Run log records both attempts.

---

## v25 (second opinion with collective memory)

**v25 = v24 + collective memory for the second opinion.** The 110 games of a run no longer learn alone.

**Why.** The hidden games are made by one design team on one engine, so the mechanics one game had to discover (a key that opens the door of its color, a bar that is a move budget, a switch that must be covered) are good guesses for another. Until now every game started from zero and kept what it learned to itself.

* **Run-wide rule pool.** When a game clears a level, it states the rule ("Rule for level N: ...", captured by the solved-level memory). Each such rule goes into a pool shared by all games of the run. When another game is stuck and asks for a second opinion, that fresh instance sees the newest rule of up to 10 other games, marked as inspiration, not facts. It adapts to the hidden set itself during the run; nothing is tuned on public games.
* **Design catalogue for the second opinion.** The knowledge-only description of how ARC-AGI-3 games tend to work (budget bars, goal shown on screen, select / move / run controls, 15 example mechanics) goes into the second opinion's system prompt.
* Only the second opinion reads either. The playing model's own prompt is unchanged from v24 (v22 had put the catalogue in the player's prompt; that is not repeated).
* Switches: `ENABLE_SO_SHARED_RULES`, `ENABLE_SO_CATALOGUE` (both off = v24). New counters `so_pool_rules` (rules pooled), `so_shared_shown` (second opinions that showed other games' rules).

---

## v24 (second opinion on stuck levels)

**v24 = v23 + an independent second opinion when a level stalls.** Server, history handling and every other add-on are v23's.

**Why.** Most lost points are levels the model never clears, and the same level costs 20-40K tokens in one run and 120-180K in another (vc33 level 4: 31K in v21, 123K in v22; tn36 level 2: 134K, 169K, 181K). The model locks onto one idea, and its own long reasoning history keeps it there. A fresh model instance that has not read that history is not locked in.

* **Second opinion.** When the level-progress facts fire (a level past 40K, then 80K generated tokens without clearing), the harness sends one side request. That fresh instance gets the level-start board, the current board, the action meanings and only measured facts: what each action did on this level, what was never tried, the budget bar, and the rules of earlier levels. It answers with up to 3 hypotheses about the goal or the hidden mechanic, each with its cheapest test. The playing model sees the answer right after the facts, marked "possibilities, not facts". Its own history and prefix cache are untouched: the side request bypasses the exact-replay wrapper, and its tokens are charged to the game. At most 2 per level, about 5% of the run's tokens.
* `ENABLE_SECOND_OPINION` switches it off (= v23). Counters `so_asks`, `so_answers`, `so_empty`, `so_errors`, `so_tokens`. A Save & Run writes every question and answer to `second_opinions.jsonl` (never in a competition rerun).

---

## v23 (level-progress facts)

**v23 = v21 + facts about what has and has not been tried on a level that is taking long.** Server, history handling and every other add-on are exactly v21's.

**Why.** Most lost points are levels the model never clears, and the same level costs 20-40K tokens in one run and 120-170K in another (vc33 level 4: 31K in v21, 123K in v22; tn36 level 2: 134K and 169K). Reading those runs: the model locks onto one idea, and the move that solves the level is often one it has never made (vc33: an object that turned orange was clicked only ~30 moves after it appeared). v10 tested a stuck-level check locally: levels past 40K tokens were cleared 80% of the time against 64% for v7. But v10 also told the model to spend its next 1-3 actions on probes, and on the leaderboard every version that told the model how to behave scored lower. v23 keeps only the facts.

* **Level-progress facts.** Only on a level that has taken 40K generated tokens without clearing (then again at 80K and 120K, at most 3 times), the turn's message ends with what the harness measured: tokens and actions spent on the level, the actions used and never tried there, the colors clicked, and the colors present but never clicked (newly appeared ones first). No advice, no request. Levels that go well never see it.
* **Keep functions on timeout.** When a python call hits the 30 s limit, the harness used to drop all of the model's retained helper functions. They are now kept, and the model is told so (1-5 times per 10-game run).
* Switches: `ENABLE_STUCK_FACTS`, `ENABLE_KEEP_FUNCTIONS_ON_TIMEOUT` (`ENABLE_GAME_CATALOGUE`, v22's, is off). Counters `sb_checks`, `sb_levels`, `sb_cleared_after`, `sb_tokens_after_check`, `sb_functions_kept`, `sb_errors`.

---

## v21 (v7 play + server speed fixes)

**v21 = v7, our best leaderboard version (31.95), plus two server-side speed fixes.** Nothing in what the model is asked or shown changes beyond v7, except that its own earlier replies are sent back exactly as it wrote them.

**Why back to v7's play.** v18-v20 changed how old history is trimmed. Each change did what it was built to do, but none showed a better score, and the local runs drifted down (52.0, 51.8, 48.0, 42.5; late moves per 1K tokens 1.21, 1.11, 1.08, 0.95). That is within noise run by run, but it is not worth the risk, so `ENABLE_HISTORY_DEDUP = False` (cell 6c stays, switched off).

* **Exact reply replay (cell 6d, from v20).** The harness stores replies with blank lines deleted, so the server re-read the model's whole previous reply in 62% of turns (v19). v20 measured the fix: reuse 38% -> 95%, reply re-reads 1.74M -> 0.11M prefill tokens per run.
* **Larger token cache.** In v20 the gain was eaten by an eviction burst. With 9 games near 100K tokens of context, the 1.01M-token cache was full, and 8 games lost their whole cached prompt within 80 seconds (0.92M tokens re-read). The Mamba state pool had 60 slots, but at most 40 were ever used (v19 and v20). It now has 50 (`SERVER_MAMBA_CACHE`), the least that still allows 10 running games (the server keeps 5 slots per game). The freed ~0.55 GB goes to the token cache (about +48K tokens, +5%). In the competition, the 10 active games hold 58K-118K tokens each, so this should make full-cache bursts much rarer.
* The Save & Run diagnostics stay (cache reuse, re-read tokens, KV usage); they never run in a competition rerun.

---

## v20 (exact reply replay)


**v20 = v19 + one fix found by v19's server diagnostics.** The add-on cell is still v7's, byte for byte.

**v19 result (Save & Run, mixed10):** 47.96 / 43 levels (v18 51.79 / 43, v7 52.03 / 45; local noise is about 7 points), all error counters 0. History de-duplication v2 worked as tested: right after each drain the model kept 125K characters of its own work (v18 121K, v7 91K).

**What the diagnostics found.** Of 745 follow-up requests, only 38% reused the previous reply from the server cache. A cache checkpoint existed at the end of the reply every time. The other 462 re-read the whole reply (2,267 tokens on average, about a quarter of all prefill work) because the next prompt no longer matched the generated tokens. In 456 of the 462, the cause was the harness's `_normalize_message_content`. It stores every reply with its blank lines deleted (`\n\n` becomes `\n`, still a single token, so the token count stays the same and nothing else notices).

* **Exact reply replay (cell 6d).** The harness keeps its normalised history, so nothing it decides on changes. Only the copy of the messages sent to the server carries each earlier reply exactly as the server returned it. A message is replaced only when the harness's own normalisation of a returned reply gives exactly its stored text, and it keeps the same text for as long as it stays in history. Checked end to end with the real harness, the real Qwen3.8 chat template and tokenizer, and real reasoning, code and tool results from three v19 games. With the stock harness, the next prompt repeated prompt + reply in only 2-8 of 29-39 turns per game. With v20 it repeated them in every turn, before and after a drain. `ENABLE_EXACT_REPLAY = False` gives v19 exactly.
* The cache diagnostics (Save & Run only) now also report the re-read token total and the diverging tokens, so the next run shows the effect directly.

---

## v19 (history de-duplication v2 + cache diagnostics)


**v19 = v18 + two small additions.** The add-on cell is still v7's, byte for byte.

**v18 result (Save & Run, mixed10):** 51.79 / 43 levels (v7 52.03 / 45), all error counters 0. The de-duplication worked as designed in the real run: right after each drain the model kept 16.5 of its own turns instead of 12.4-12.7 (121K vs 91-99K characters of its own reasoning, code and results), and late-game moves per 1K tokens stayed normal (1.11).

* **History de-duplication v2 (cell 6c).** At drain time, older turn openers now also lose harness blocks that a later message carries again: the solved-level memory, the action-effect summary, the move-budget meter and the tool tip. These are cumulative or a per-turn reading, so only the newest copy matters, and it always stays. On 167 real drains (v7 and v18 runs) through the real trimmer: +3% to +5.5% more of the model's own turns kept. `DEDUP_SUPERSEDED_BLOCKS = False` (in cell 6c) gives v18 exactly.
* **Server cache diagnostics (Save & Run only; never in a competition rerun).** In about 60% of turns the server re-reads the model's previous reply instead of reusing it from cache (about 5% of all GPU time). A measurement-only hook in the SGLang scheduler records, for every request, whether a cache checkpoint was kept at the end of the reply and whether the next prompt repeats prompt+reply token for token. The result line gains `cache_diag`; the raw records are in `cache_diag.jsonl`. `ENABLE_CACHE_DIAG = False` turns it off.
* **`COMMIT_PLAY_MINUTES`** (default `None` = full local evaluation). Set it to e.g. `5` only to make a cheap submittable version of an already-evaluated notebook; it never affects the competition rerun.

---

## v18 (history de-duplication)


**v18 = v7 (our best leaderboard score, 31.95) + history de-duplication.** The add-on cell is v7's, byte for byte; one new cell (6c) changes how old history is trimmed.

**Why.** v17 showed how much the model depends on its own past reasoning: when its history was replaced by summaries, it needed about twice as many tokens per move in 11 of 12 games. The history is also padded: every turn opener repeats the same ten instruction lines, a fresh copy of the retained-function list and one or two board images, and all of it stays in history. In the v7 run that was about 17% of every full prompt.

* When the context is full and the harness is about to drop old messages anyway (the cached prefix is lost at that moment regardless), older turn openers first lose those duplicate lines and their images, so the drain then removes fewer real turns. On 84 real drains from the v7 run, replayed through the real harness trimmer, the model kept 32% more of its own recent work: 11.5 -> 15.1 turns, 76K -> 100K characters of its reasoning, code and tool results. The prompt size after a drain is unchanged.
* The newest turn opener is never touched, so the model sees exactly what it saw in v7 on the current turn. The images of the last 3 openers stay, and the model's own messages and tool results are never modified. Below the context limit nothing changes, so prefix caching behaves as before. `ENABLE_HISTORY_DEDUP = False` restores v7 exactly.
* New in the result line: `dedup_stats` (drains de-duplicated, characters and images removed) and `act_per_1k_late` (moves per 1K generated tokens after each game's first 24 replies: v7..v16 1.07-1.63, v17 0.57).

---

## v7 (move-budget meter)


**v7 = v6 + a move-budget meter, built from an analysis of our own v5 transcripts** (10 games, about 950 model requests). Many games show a HUD bar in the screen border, a full row such as row 0 or row 63, that loses cells with every move and ends the attempt with GAME OVER when it is empty. In v5, **m0r0 died** because it never realised this, and **s5i5 / vc33** spent a large share of their thinking measuring the bar by hand. The harness now measures it exactly and adds one line per turn (about 50 tokens; no game action), for example:

*"MOVE-BUDGET METER (measured by the harness): the 'B' cells in row 0 went 64 -> 10 over the 127 moves of this attempt (-0.43 per move); at this rate it runs out in about 23 more moves. Earlier in this game, 1 GAME OVER(s) happened right when such a bar ran out - treat it as a hard move limit."*

It was validated by replaying the real v5 games move by move. It found vc33's pink row-0 bar (-1.27 per move), s5i5's gray row-63 bar, and m0r0's black row-0 bar (-0.43 per move), and it linked m0r0's GAME OVER at move 152 to that bar running out. The bar is only reported when one border color changed monotonically by at least 3 cells over at least 3 moves of the current attempt. It is reset on level change, RESET and GAME OVER. `ENABLE_BUDGET_METER = False` switches it off.

---


**v6 = v5 + help that does not depend on the model choosing to use it.** In v5's Save & Run, the model never called the world-model tools (`wm_snippets = 0`): an optional tool, described once at the end of a long system prompt, gets ignored. Every game also used its whole token budget, so games run out of compute, not ideas. The lever is therefore **more levels per token**.
* **Automatic action-effect hints.** The harness analyses every move already made on the current level (no game action is spent) and adds a short summary to the turn's message, for example *"- UP x3: the 'b' object moved (-1,+0) 2/3; no visible change 1/3"* or *"- MOUSE on 'Y' cells x1: cells changed 'Y'->'R' 1/1"*. It detects rigid object moves and their direction, plain color changes, game overs and moves with no lasting effect, and it ignores the HUD border, using the same 4-cell border as the harness's no-op guard. The block is re-added only when what it says changes or after the history drain evicted it, so it does not repeat every turn. It costs about 100 tokens. Speed: 300 moves are analysed in 0.07 s; later turns reuse a per-move cache.
* **Tool reminder.** When a level reaches 8 and then 25 recorded moves, the turn's message suggests `check_rules` / `plan_route` once. It never appears again on a level where the model has already used them.

Both go at the end of the turn's message, so they never invalidate the prefix cache, and the solved-level memory's block and its "Rule for level N" request still come last. `ENABLE_ACTION_HINTS` / `ENABLE_TOOL_NUDGE` switch them off. New counters: `ah_blocks`, `ah_chars`, `nudges`, `ah_errors`.

---


**v5 = v4 + affectify's D′ slot priority + competition-matched local testing.**
* **D′ scheduling** (from affectify's public notebook *"ARC-AGI-3 Milestone 2 Solution with a new slot priority (D′)"*, built on their earlier *"Duck plus ours"*; their code is embedded verbatim, and all credit for it is theirs). It replaces only the formula that picks which waiting game gets one of the 10 LLM slots, favoring games that are deep, action-efficient and still have levels left. Unstarted games no longer jump the queue. Their own simulation put it at about +1.5 points. It touches nothing our add-ons touch, and it was tested locally on top of them against the real priority gate. `ENABLE_DPRIME_SCHEDULER = False` restores da-fr's priority.
* **Competition-matched Save & Run.** da-fr's 4-pass study showed that compute per game dominates the score: the same solver got 46.5 on the 25 public games with competition compute, against about 21 at half of it. So a local run now gives each game the same GPU time as the real run: N games for 532·N/110 minutes. The default `LOCAL_GAME_SET = 'mixed10'` is 10 games spanning easy to hard (mean about 49 in that study, close to the 25-game 46.5), and takes about 48 minutes. The other choices are `'demo10'` (da-fr's 10 easiest) and `'all25'` (about 121 minutes).

**Reading results honestly:** the same code scores ±2 points between leaderboard submissions (da-fr: 27.62, 27.89, 31.47), and a 10-game local run swings by about ±7. Local runs are health checks; only clear leaderboard gains count.

---


**v4 = v2's play (31.22 on the public leaderboard) plus our own strategy: world-model tools.** The model already writes Python and is told to search, but it has to guess the game's rules with no way to check them, and it rewrites a search algorithm on almost every turn. v4 gives it two sandbox functions that spend no game actions:
* `check_rules(step)`: the model writes `step(board, action)`, its simulator of the rules. This replays every recorded move of the current level through it and reports the accuracy, with the exact wrong cells.
* `plan_route(step, goal)`: a tested breadth-first search over that simulator. It returns the shortest action route, ready for `action()`. A 116-move route across a full 64×64 board takes about 2.4 s.

A fixed system-prompt section explains the workflow (rules, then check, then plan, then play the whole route). Because the text is the same for the whole run, it does not hurt the prefix cache. `ENABLE_WORLD_MODEL_TOOLS = False` gives exactly v3 with `PROFILE = 'v2'`. `ADDON_SUMMARY` reports how often the model used the tools (`wm_*` counters).

Also new: if the harness patch ever fails to apply, the notebook says why. It either reports that the patch cell was edited (a stray space is enough; checked against a fingerprint of the original) or that the wrong source bundle is attached. The watchdog now also waits until the old server has released its port and GPU memory before relaunching.

---


**v3 is v2 plus a `PROFILE` switch** that turns on strategies the Duck harness already contains but da-fr's configuration leaves off. With `PROFILE = 'v2'` it plays exactly like v2, plus a more tolerant rule capture. Every profile uses only harness code that already exists; v3 only flips its switches.

| Profile | Switches turned on | Aimed at |
|---|---|---|
| `v2` | none | baseline |
| `act` | `OPENER_COMMIT_HINT`, `PREFER_TOOL_CALLS`, `STUB_DEAD_REASONING` | over-thinking: in v2's report, r11l spent 2,473 tokens per action and vc33 29.7K tokens per level |
| `death` | `DEATH_LEDGER`, `DEATH_REPEAT_GUARD`, `REPEAT_STATE_GUARD` | replaying a route that already caused a game over, and loops inside the model's own code |
| `inventory` | `LEVEL_INVENTORY` | spotting what is new in each level, which is usually a new mechanic |
| `act_death`, `act_death_inventory` | combinations | stacking the winners |

The repeat guards stay off on level 1 (`ARC3_GUARDS_FROM_LEVEL=2`, as in da-fr's config), and each has an override: if the model insists by sending the same action again, it executes. `SERVER_SPEC_STEPS = 4` is a separate speed experiment. It needs one run to check that the server boots and stays within memory before it is used for scoring.

**Experiment protocol.** Save & Run now plays **all 25 public games for 60 minutes**. The 10-game set is da-fr's easiest, and tuning on it risks overfitting. Each run takes about 75 minutes and ends with one `EXPERIMENT_RESULT {...}` line, also saved as `/kaggle/working/experiment_result.json`. Run each profile at least twice, because single runs vary by several points, and only keep what wins consistently.


**v2 changes.** v1's Save & Run scored a mean of **39.57** on the 10-game set, against 36.56 for da-fr, but generated 503 tok/s instead of 588.
1. **Solved-level memory no longer touches the system prompt.** v1 pinned the rules there, and each of its 39 updates made the server re-prefill that game's whole context. v2 appends the same block to the normal user prompt, which the harness keeps in history exactly as sent. The block is re-appended only after a new rule, or after the oldest-first history drain has evicted it. So it adds no extra prefix-cache misses. To compare against v1 behavior, set `MEMORY_PLACEMENT = "system"`.
2. **End-of-run speed/cache report** (Save & Run only). It prints prefix-cache reuse, decode throughput and speculative accept length from the SGLang log and `/metrics`, plus tokens per action and per level for every game.
3. **Quieter patch step.** The `git apply` output is captured and summarized in one line. The full output is shown only on failure.

This notebook merges three public ARC-AGI-3 notebooks into one submission.

| Source | What it contributes here |
|---|---|
| **da-fr, *ARC-AGI-3 Milestone 2 solution*** (built on Tufa Labs' Duck harness) | **The backbone, unchanged.** It has the full harness patch (UNDO/ACTION7, no-op and death guards, priority scheduler, animation frames, frame diffs, persistent functions, segmentation), Qwen3.8-Flash-Next W4A16 AutoRound plus MTP speculative decoding on Pennyroyal SGLang v2.5.3, and the tuned context and sampling settings. It was the strongest of the three: its own 10-game Save & Run scored a **mean of 36.56**. On those same 10 games, Duck v18's public run averaged 14.1: da-fr was ahead on 8 games and tied on 2. |
| **Tufa Labs / Duck v18.1gc** (keithtyser / jeroencottaar) | **Solved-level memory** (their M85, adapted). After each cleared level, the model states a `Rule for level N:`, and that rule, the winning action tail and the clearing reasoning are pinned to the system prompt for the rest of the game. The backbone runs with `ARC3_MEMORY_SECTIONS=off`, so it otherwise has no carrier across levels once history drains. Also **compact runtime state** (their M86 finding: the per-action state file is written with `indent=2`) and a **model-server watchdog**. |
| **TAAF launcher** (lordhansolo, *built on Tufa Labs duck harness*) | **Launcher robustness:** inputs are found at either Kaggle mount layout (`/kaggle/input/<slug>` or `/kaggle/input/datasets/<owner>/<slug>`), and the run uses a wall-clock deadline instead of fixed budgets. |

**New safety features** (all can be switched off in the config cell):
* **Deadline guard (submission only):** the per-game budget is capped so that play ends before the Kaggle runtime limit, even if the server or gateway was slow to start. The original used a fixed 532 min per game *after* up to 12 min of startup.
* **Server watchdog:** if SGLang exits mid-run, the watchdog kills its process group, waits for VRAM to free, and relaunches it with identical arguments (at most 2 restarts). Games wait for it instead of giving up after 10 failed requests.

Everything the add-ons do is wrapped so that if one fails, the harness falls back to its original behavior. You can check this in `addon_summary()` at the end of the run.

**Required inputs** (same as the da-fr notebook): the competition data, `dfranzen/pennyroyal-v253`, `dfranzen/taaf-kaggle-source-bundle-copy`, and the models `dfranzen/intel-qwen3.8-flash-next-w4a16-autoround` and `dfranzen/albucino-qwen3-8-flash-next-drafter`. After copying, select the **RTX PRO 6000** accelerator.

Credits: da-fr (Daniel Franzen) for the Milestone-2 solution. Tufa Labs (Harold Bessis, Jeroen Cottaar, Isaiah Pressman, Andries Smit, Michal Tesnar, Stefano Viel) for the Duck harness and TAAF. keithtyser for the Duck v18 notebook. lordhansolo for the TAAF Kaggle launcher. John Pezzulli, Mamy Ratsimbazafy and Gabriel Olympie for Pennyroyal SGLang. Intel and Albucino for the quantized checkpoints. Third-party code and weights remain under their own licenses.


## 0. Harness patch

Patch to add our modifications to the original tufa harness.

In [ ]:
%%writefile /kaggle/harness-changes.patch
diff --git a/ARC3-Inference/inference/agent/action_names.py b/ARC3-Inference/inference/agent/action_names.py
index 4b8dfa3..1ef0773 100644
--- a/ARC3-Inference/inference/agent/action_names.py
+++ b/ARC3-Inference/inference/agent/action_names.py
@@ -1,6 +1,7 @@
 """Shared action-name mapping between model-facing labels and engine actions."""
 from __future__ import annotations
 
+import os
 from typing import Iterable
 
 
@@ -17,8 +18,32 @@ ENGINE_TO_MODEL_ACTION = {
 MODEL_TO_ENGINE_ACTION = {value: key for key, value in ENGINE_TO_MODEL_ACTION.items()}
 
 
+def reset_exposed() -> bool:
+    """Opt-in RESET advertising and snippet guards; off preserves legacy use."""
+    return os.environ.get("EXPOSE_RESET", "off").strip().lower() in {
+        "on", "1", "true", "yes",
+    }
+
+
+def undo_exposure_mode() -> str:
+    """Read at call time; unset/invalid values preserve the original wiring.
+
+    tufa: advertise ACTION7 verbatim but reject execution (legacy behavior).
+    off: hide ACTION7/UNDO and reject execution.
+    on: advertise UNDO and accept either name for engine ACTION7.
+    """
+    mode = os.environ.get("EXPOSE_UNDO", "tufa").strip().lower()
+    return mode if mode in {"tufa", "off", "on"} else "tufa"
+
+
 def to_model_action(name: str | None) -> str:
     raw = str(name or "").strip().upper()
+    if raw in {"ACTION7", "UNDO"}:
+        mode = undo_exposure_mode()
+        if mode == "off":
+            return ""
+        if mode == "on":
+            return "UNDO"
     return ENGINE_TO_MODEL_ACTION.get(raw, raw)
 
 
@@ -26,6 +51,8 @@ def to_engine_action(name: str | None) -> str | None:
     raw = str(name or "").strip().upper()
     if not raw:
         return None
+    if raw in {"ACTION7", "UNDO"}:
+        return "ACTION7" if undo_exposure_mode() == "on" else None
     if raw in ENGINE_TO_MODEL_ACTION:
         return raw
     return MODEL_TO_ENGINE_ACTION.get(raw)
diff --git a/ARC3-Inference/inference/agent/noop_repeat_guard.py b/ARC3-Inference/inference/agent/noop_repeat_guard.py
new file mode 100644
index 0000000..314e47a
--- /dev/null
+++ b/ARC3-Inference/inference/agent/noop_repeat_guard.py
@@ -0,0 +1,178 @@
+"""Known-no-op repeat guard.
+
+Records ``(level, interior_state_hash, action_signature)`` triples that were
+observed to change nothing outside the board border, and refuses to execute
+them again in a bit-identical state.
+
+Two deliberate choices:
+
+* The state key uses ``_interior_state_hash`` - the same border-cropped hash
+  the death ledger uses - so "the same state" means the same thing everywhere
+  in the harness, and a ticking HUD timer does not make every state unique.
+
+* Blocking is overridable. A blocked action arms a single slot; if the model
+  immediately re-issues that same action, it executes. This avoids the
+  self-sealing failure of a hard block: once an action is on the list, a hard
+  block guarantees no contradicting evidence can ever arrive, so a mechanic
+  gated on hidden state (a counter, a cooldown, an off-screen timer) would be
+  foreclosed forever. Each insistence costs exactly one action and the entry
+  survives, so the model can always insist again.
+
+Known limitation: the harness does not receive per-action frame counts, so an
+action whose effect appears only in intermediate animation frames (a rejected
+click, a bounce) looks identical to one that did nothing, and will be
+recorded. The override path is the only recourse there, which is why this
+guard defaults to off.
+"""
+from __future__ import annotations
+
+import re
+from collections import OrderedDict
+from typing import Any, Callable
+
+
+_DISPLAY_COORDS_RE = re.compile(
+    r"^([A-Z0-9_]+)\(\s*(?:ROW\s*=\s*)?(-?\d+)\s*,\s*(?:COL\s*=\s*)?(-?\d+)\s*\)$"
+)
+
+
+def action_signature(action: Any) -> str:
+    """Stable signature for one action. Mouse coordinates are part of the
+    identity: a click is only "the same action" at the same cell.
+
+    Accepts both shapes the harness uses - the dict form the sandbox passes
+    (`{"action": "MOUSE", "row": 4, "col": 7}`) and the display form the death
+    ledger stores (`"MOUSE(row=4, col=7)"`) - and normalizes them to the same
+    string, so a verdict recorded through one path is found through the other.
+    Callers must pass MODEL-facing names (UP/DOWN/...), not engine names
+    (ACTION1/ACTION2/...).
+    """
+    if isinstance(action, dict):
+        name = str(action.get("action") or action.get("name") or "").strip().upper()
+        row = action.get("row")
+        col = action.get("col")
+        if row is not None or col is not None:
+            return f"{name}({row},{col})"
+        return name
+    text = " ".join(str(action or "").split()).upper()
+    match = _DISPLAY_COORDS_RE.match(text)
+    if match:
+        return f"{match.group(1)}({match.group(2)},{match.group(3)})"
+    return text
+
+
+class NoopRepeatGuard:
+    """Per-level store of state+action combinations known to do nothing."""
+
+    def __init__(self, *, max_states_per_level: int = 512, max_actions_per_state: int = 16) -> None:
+        self._max_states = max(1, int(max_states_per_level))
+        self._max_actions = max(1, int(max_actions_per_state))
+        self._states: "OrderedDict[str, OrderedDict[str, None]]" = OrderedDict()
+        self._level: Any = None
+        # One slot per guard kind. A shared slot let whichever guard ran first
+        # consume the other's override: the death check runs before the no-op
+        # check, cleared the arm on every non-match, and re-armed it, so the
+        # no-op guard's "issue it again and it will execute" never fired. A
+        # mouse-only game observed 291 replies and one executed action.
+        self._armed_noop: tuple[str, str] | None = None
+        self._armed_death: tuple[str, str] | None = None
+
+    # -- level scoping -------------------------------------------------------
+    def note_level(self, level: Any) -> None:
+        """Drop everything when the level changes: a new layout means old
+        state hashes describe boards that no longer exist. Deaths and resets
+        deliberately do NOT clear the store - the board returns to the level
+        start, so recorded no-ops still hold and carry across attempts."""
+        if level != self._level:
+            self._level = level
+            self._states.clear()
+            self._armed_noop = None
+            self._armed_death = None
+
+    # -- recording -----------------------------------------------------------
+    def observe(self, state_hash: str, action_sig: str, *, gameplay_changed: Any) -> None:
+        if not state_hash or not action_sig:
+            return
+        if gameplay_changed is False:
+            entry = self._states.get(state_hash)
+            if entry is None:
+                entry = OrderedDict()
+                self._states[state_hash] = entry
+                while len(self._states) > self._max_states:
+                    self._states.popitem(last=False)
+            entry[action_sig] = None
+            while len(entry) > self._max_actions:
+                entry.popitem(last=False)
+        elif gameplay_changed is True:
+            # Evidence contradicts a recorded no-op: drop it. Reachable via the
+            # override path, where a blocked action is executed after all.
+            entry = self._states.get(state_hash)
+            if entry is not None and action_sig in entry:
+                del entry[action_sig]
+                if not entry:
+                    del self._states[state_hash]
+
+    # -- blocking ------------------------------------------------------------
+    def should_block(self, state_hash: str, action_sig: str) -> bool:
+        """True when this combination is a known no-op AND the model has not
+        just been told so. Arms the override slot when it blocks."""
+        if not state_hash or not action_sig:
+            return False
+        key = (state_hash, action_sig)
+        if self._armed_noop == key:
+            # the model saw the refusal and insisted: let it through, and
+            # re-arm nothing so a third attempt is blocked again
+            self._armed_noop = None
+            return False
+        entry = self._states.get(state_hash)
+        if not entry or action_sig not in entry:
+            self._armed_noop = None
+            return False
+        self._armed_noop = key
+        return True
+
+    def note_executed(self, state_hash: str, action_sig: str) -> None:
+        """Any executed action that is not the armed one clears the slot, so
+        the override only applies to an immediate repeat."""
+        key = (state_hash, action_sig)
+        if self._armed_noop and self._armed_noop != key:
+            self._armed_noop = None
+        if self._armed_death and self._armed_death != key:
+            self._armed_death = None
+
+    # -- death verdicts ------------------------------------------------------
+    def should_block_death(
+        self,
+        state_hash: str,
+        action_sig: str,
+        is_known_fatal: "Callable[[str, str], bool]",
+    ) -> bool:
+        """Same block-then-insist contract as `should_block`, but the verdict
+        comes from the death ledger rather than this store.
+
+        The ledger already keeps, per recorded attempt, an `actions` list and
+        an index-aligned `hashes` list of interior state hashes taken BEFORE
+        each action - so "this action, from this exact board, killed us" is a
+        lookup, not a sequence match. Matching per position rather than by
+        sequence prefix is what makes an interleaved or differently-routed
+        approach to the same fatal move still get caught, while a batch whose
+        earlier actions changed the board correctly does NOT match.
+
+        Deaths are more likely than no-ops to hinge on state the board does
+        not show (a hazard's facing, a timer), so blocking must stay
+        overridable: under permadeath a hard block could foreclose the only
+        path through a level with no way to ever discover the error.
+        """
+        if not state_hash or not action_sig:
+            return False
+        key = (state_hash, action_sig)
+        if self._armed_death == key:
+            self._armed_death = None
+            return False
+        if not is_known_fatal(state_hash, action_sig):
+            # deliberately does NOT clear the slot: an action this guard has no
+            # verdict on is none of its business, and clearing here would drop
+            # an override armed for a different action
+            return False
+        self._armed_death = key
+        return True
diff --git a/ARC3-Inference/inference/agent/priority_pace_reference.json b/ARC3-Inference/inference/agent/priority_pace_reference.json
new file mode 100644
index 0000000..9c624f8
--- /dev/null
+++ b/ARC3-Inference/inference/agent/priority_pace_reference.json
@@ -0,0 +1,16 @@
+{
+  "fallback_tokens": 45924.98353694247,
+  "by_level": {
+    "1": 27011.05963663083,
+    "2": 51072.88442280241,
+    "3": 48932.24773612239,
+    "4": 56352.32702962253,
+    "5": 55796.95578535604,
+    "6": 67264.12383313512,
+    "7": 40699.37351375245,
+    "8": 22696.312661663567
+  },
+  "method": "Kaplan-Meier restricted mean token cost, capped at 120000 tokens",
+  "training": "1417 reached-level episodes from historical transcripts; pooled by level number, no game identity at inference",
+  "limitations": "Historical config differs from current config. Reference is a bounded cost scale, not an unbiased uncapped completion time. Last survival value carried to cap beyond observed support."
+}
diff --git a/ARC3-Inference/inference/agent/priority_scheduler.py b/ARC3-Inference/inference/agent/priority_scheduler.py
new file mode 100644
index 0000000..14bb787
--- /dev/null
+++ b/ARC3-Inference/inference/agent/priority_scheduler.py
@@ -0,0 +1,134 @@
+"""Pure scheduler inputs and scoring; never reads hidden game metadata."""
+from __future__ import annotations
+
+import math
+from dataclasses import dataclass
+
+
+# Expected future level-weight sum, conditional on completing the current level.
+# 80k tokens are shared by ALL future levels; efficiency is NOT baked in.
+# Source: arc3-levelup-chances, pooled_late_provisional, N6..10_H80000 / 0.8.
+# KM duration distributions: separate levels 1-4, pooled observed levels 5-8
+# for levels 5 onward. Levels 9-10 are extrapolated. Costs rounded up to 250
+# tokens. 336 historical passes / 1,417 reached levels; not a contest fit.
+# Tuples are indexed by current_level - 1. The final level has no future tail.
+TAIL_LOOKUP_80K: dict[int, tuple[float, ...]] = {
+    6: (3.9082328014, 5.0756939323, 5.3532912917, 5.6080111343,
+        4.0898867400, 0.0),
+    7: (3.9262240771, 5.1669653335, 5.6943514417, 6.6500041602,
+        6.6562876217, 4.7715345300, 0.0),
+    8: (3.9291260268, 5.1855629875, 5.7851444897, 7.0053702980,
+        7.8471367942, 7.7045641090, 5.4531823200, 0.0),
+    9: (3.9295002558, 5.1885168225, 5.8035897108, 7.0958580947,
+        8.2469236991, 9.0442694281, 8.7528405964, 6.1348301100, 0.0),
+    10: (3.9295397677, 5.1888936715, 5.8065473797, 7.1136691211,
+         8.3474656955, 9.4884771003, 10.2414020621, 9.8011170838,
+         6.8164779000, 0.0),
+}
+
+
+# Heuristic continuation value by levels remaining AFTER the current level:
+# 3 or more -> 8; 2 -> 7; 1 -> 5; 0 -> 0. These are final B values,
+# so neither score normalization nor the empirical efficiency discount applies.
+TAIL_LOOKUP_REMAINING: dict[int, tuple[float, ...]] = {
+    n: (8.0,) * (n - 3) + (7.0, 5.0, 0.0) for n in range(6, 11)
+}
+
+
+def parse_total_levels(value: object) -> int | None:
+    """Validate public level-count metadata, then clip to the table domain."""
+    if value is None or isinstance(value, bool):
+        return None
+    try:
+        number = float(value)
+    except (TypeError, ValueError, OverflowError):
+        return None
+    if not math.isfinite(number) or number <= 0 or not number.is_integer():
+        return None
+    return min(10, max(6, int(number)))
+
+
+@dataclass
+class ProgressPace:
+    completed: int = 0
+    log_cost_ratio: float = 0.0
+
+    def record_completion(self, tokens_used: float, reference_tokens: float) -> None:
+        observation = math.log(max(1.0, tokens_used) / max(1.0, reference_tokens))
+        self.log_cost_ratio = (
+            observation if self.completed == 0
+            else 0.6 * self.log_cost_ratio + 0.4 * observation
+        )
+        self.completed += 1
+
+    def cost_multiplier(self) -> float:
+        confidence = self.completed / (self.completed + 3.0)
+        return math.exp(min(math.log(2.0), max(math.log(0.5),
+                                             confidence * self.log_cost_ratio)))
+
+
+@dataclass(frozen=True)
+class PrioritySnapshot:
+    level: int
+    actions: int
+    tokens: float
+    cost_multiplier: float = 1.0
+    # Appended to preserve four-argument callers and older pickled snapshots.
+    total_levels: int | None = None
+
+
+def priority_value(
+    state: PrioritySnapshot, *, endgame: bool = False,
+    tail_fraction: float | None = None, human_actions: float = 25.0,
+    action_scale: float = 115.0, token_scale: float = 62000.0,
+    action_weight: float = 0.25, tail_base: float = 6.0, tail_cap: float = 6.0,
+    normalize_score: bool = False, tail_lookup: bool | str = False,
+    tail_efficiency: float = 0.8,
+) -> int:
+    """No phase offset: all candidates must be evaluated at the same time.
+
+    tail_fraction=None retains the endgame rule. Supplying a fraction
+    keeps the likelihood proxy active and fades the tail instead of switching.
+    The likelihood proxy is heuristic, not a calibrated completion probability.
+
+    tail_lookup=True uses the empirical 80k table; "remaining" uses the
+    fixed 8/7/5/0 table without an efficiency discount. False keeps the capped
+    tail. normalize_score scales only A, never B or the endgame's 0.25 floor.
+    """
+    level = max(1, state.level)
+    total_levels = parse_total_levels(getattr(state, "total_levels", None)) or 10
+    actions, tokens = max(0, state.actions), max(0.0, state.tokens)
+    pace = min(2.0, max(0.5, state.cost_multiplier))
+    efficiency = (human_actions / (human_actions + actions)) ** 2
+    current_value = level * efficiency
+    if normalize_score:
+        # Normalize immediate reward A to a ten-level reference (sum 1..10).
+        # B is a separately chosen continuation value, not normalized reward.
+        current_value *= 55.0 / (total_levels * (total_levels + 1) / 2)
+    if endgame and tail_fraction is None:
+        # Preserve the original arithmetic when normalization is disabled.
+        value = (
+            level * 0.25 + current_value if normalize_score
+            else level * (0.25 + efficiency)
+        ) / pace * 100
+    else:
+        if tail_lookup == "remaining":
+            index = min(total_levels, level) - 1
+            tail = TAIL_LOOKUP_REMAINING[total_levels][index]
+        elif tail_lookup:
+            future_efficiency = (
+                min(1.0, max(0.0, tail_efficiency))
+                if math.isfinite(tail_efficiency) else 0.8
+            )
+            index = min(total_levels, level) - 1
+            tail = TAIL_LOOKUP_80K[total_levels][index] * future_efficiency
+        else:
+            tail = min(tail_cap, tail_base + level - 1)
+        if tail_fraction is not None:
+            tail *= min(1.0, max(0.0, tail_fraction))
+        chance = (
+            action_weight * 0.5 ** ((actions / action_scale) ** 2)
+            + (1.0 - action_weight) * 0.5 ** ((tokens / (token_scale * pace)) ** 2)
+        )
+        value = (current_value + tail) * chance / pace * 100
+    return max(1, int(value))
diff --git a/ARC3-Inference/inference/agent/prompts.py b/ARC3-Inference/inference/agent/prompts.py
index 2e59574..52351db 100644
--- a/ARC3-Inference/inference/agent/prompts.py
+++ b/ARC3-Inference/inference/agent/prompts.py
@@ -2,6 +2,62 @@
 
 from inference.utils.grid_utils import ARC_COLOR_LEGEND
 
+LEVEL_TRANSFER_SYSTEM_GUIDANCE = (
+    "- Levels usually build on mechanics learned in earlier levels, especially the most "
+    "recent one. Carry forward supported knowledge as a starting hypothesis, while "
+    "re-checking anything contradicted by new evidence. New levels often introduce "
+    "additional mechanics, sometimes through unfamiliar board elements. These additions "
+    "are often important for solving the level. The goal may remain the same but require "
+    "new mechanics to reach it, or the goal itself may change.\n"
+)
+
+LEVEL_START_USER_PROMPT = (
+    "You have completed the previous level. `current_frame` now contains the starting "
+    "board of the next level; any accompanying current-grid image shows this new board. "
+    "Build a new plan for this layout rather than continuing the previous level's action "
+    "sequence.\n\n"
+    "Start from the mechanics you established on the previous level; do not rediscover "
+    "them without reason. Inspect the new board for unfamiliar elements, changed "
+    "arrangements, or interactions your previous understanding does not explain. New "
+    "elements often introduce mechanics needed to solve this level, so prioritize small, "
+    "informative tests when their behavior is unclear.\n\n"
+    "Reassess the goal: does the previous objective still apply, now requiring the new "
+    "mechanics, or does the evidence suggest a different objective? Combine retained "
+    "knowledge with new findings to plan for this board."
+)
+
+
+def apply_level_transfer_system_guidance(prompt: str) -> str:
+    """Opt-in system wording; the caller gates ARC3_LEVEL_TRANSFER_GUIDANCE.
+
+    Preserve the original constants for the default arm. The first user prompt
+    after completion gives the new-board instructions; the system retains flag
+    definitions and stop instructions for failure and run completion.
+    """
+    return (
+        prompt.replace(
+            "- Levels often build on earlier mechanics, but layouts and interactions can still change between levels.\n",
+            LEVEL_TRANSFER_SYSTEM_GUIDANCE,
+        )
+        .replace(
+            "- Strategies may transfer loosely across levels, but layouts and mechanics can change. Re-check the new board before repeating a plan.\n",
+            "",
+        )
+        .replace(
+            "- Re-ground on the newest frame after any score increase or abrupt scene change; the returned board may already be the next level.\n",
+            "",
+        )
+        .replace(
+            "- `WIN` means the whole game is solved. Mid-run level completion is more likely to appear as a score increase while play continues.\n",
+            "- `WIN` means the whole game is solved.\n",
+        )
+        .replace(
+            "- If an action result reports `game_over`, `run_complete`, `level_completed`, or `done`, stop acting immediately and re-ground on the next turn.\n",
+            "- If an action result reports `game_over`, `run_complete`, or `done`, stop acting immediately and re-ground on the next turn.\n",
+        )
+    )
+
+
 TOOL_CALL_FORMAT_GUIDANCE = (
     "When calling `python`, emit exactly the tool-call format shown elsewhere in this prompt for this model. "
     "Use only that format; do not add markdown fences, prose wrappers, or alternate tool-call syntax. "
@@ -34,13 +90,50 @@ VISUAL_GAME_ADDENDUM = (
     "- For `MOUSE`, pass `row` and `col` integer arguments. `row` is vertical position, `col` is horizontal position.\n"
 )
 
+ACTION_INFO_ADDENDUM = (
+    "\n\nAction meanings (use only actions currently available):\n"
+    "- `UP`, `DOWN`, `LEFT`, and `RIGHT` are directional controls; what they affect "
+    "depends on the game.\n"
+    "- When available, `SPACE` performs a game-specific action, such as interacting, "
+    "selecting, rotating, attaching/detaching, or executing. Test its effect rather "
+    "than assuming what it does.\n"
+    "- When available, `MOUSE` clicks a board location. Pass integer `row` and `col` "
+    "fields from 0 to 63. Coordinates are zero-based from the top-left: `row` "
+    "increases downward and `col` increases rightward.\n"
+)
+
+UNDO_INFO_ADDENDUM = (
+    "- When available, `UNDO` reverses a previous action, usually the last turn. "
+    "Check what it restores. It cannot recover a failed attempt after game over.\n"
+)
+
+RESET_INFO_ADDENDUM = (
+    "- When available, `RESET` usually restores the current level to its starting "
+    "state, including the remaining-action/time bar, while keeping completed levels. "
+    "Use it to recover from an unrecoverable position or start a different approach. "
+    "RESET itself counts as one action, and actions already spent still count toward "
+    "your score. RESET must be the first game action in a Python snippet; inspection "
+    "or computation may precede it.\n"
+)
+
+GAMEPLAY_CHANGED_ADDENDUM = (
+    # One region, named once and referred back to twice. "outside that margin"
+    # and "inside that border" both read as possibly including the border
+    # cells themselves; "ignores those edge cells" states the rule before any
+    # geometry, and "further in" cannot be read as the border.
+    "- `board_changed` is true when ANY cell differs, including cells within {border} of the "
+    "edge where a timer or remaining-steps bar usually sits; `gameplay_changed` ignores those "
+    "edge cells and is true only when a cell further in differs. An action that only shrinks "
+    "a timer bar in those edge cells sets `board_changed` but not `gameplay_changed`.\n"
+)
+
 STRUCTURED_RUNTIME_STATE_ADDENDUM = (
     "\n\nRuntime variables inside every `python` tool call:\n"
     "- `current_frame` is a lightweight frame view for the latest environment state.\n"
     "- `current_frame` exposes only `.ascii`, `.step`, `.level`, `.shape`, and `.segmentation`.\n"
     "- `current_frame.ascii` is a single newline-delimited string containing the latest board rendered with the letter-coded ARC color symbols.\n"
     "- `current_frame.segmentation` parses the board into objects. It returns `{'nodes': [...], 'adjacency_list': [...]}`.\n"
-    "- Each node in `segmentation['nodes']` is one 4-connected same-color object with: `id` (index, ordered top-most-left-most), `color` (ARC color character), `hash` (a signature of the object's color and shape that ignores its position -- equal hashes mean the same object regardless of where it is, so use it to track an object across frames or to spot multiple identical objects in one frame), `pixels` (cell count), `boundary` (clockwise outer-perimeter corner points as `[row, col]`), and `children` (ids of objects fully enclosed by this one).\n"
+    "- Each node in `segmentation['nodes']` is one 4-connected same-color object with: `id` (index, ordered top-most-left-most), `color` (ARC color character), `hash` (a signature of the object's color and shape that ignores position AND rotation -- equal hashes indicate matching color and shape up to rotation; they do not establish object identity), `rotation` (degrees clockwise from the object's canonical orientation: 0/90/180/270, reduced to 0/90 for 2-fold symmetric shapes, `None` for fully symmetric ones), `rotational_symmetry` (1, 2, or 4), `pose_hash` (the rotation-SENSITIVE variant, if you need to match exact orientation), `shape_hash` (color-independent pose shape signature), `pixels` (cell count), `boundary` (clockwise outer-perimeter corner points as `[row, col]` cell coordinates), and `children` (ids of objects fully enclosed by this one).\n"
     "- `segmentation['adjacency_list']` is a list of `[i, j]` node-id pairs whose objects share an edge.\n"
     
     "- `current_frame.step` is the current environment step count.\n"
@@ -50,20 +143,48 @@ STRUCTURED_RUNTIME_STATE_ADDENDUM = (
 
     "- `history` is a chronological list of action/frame snapshots.\n"
     "- `history` is a Python list of objects, not a dict.\n"
-    "- Each history entry exposes only `.action` and `.frame`; entries are not subscriptable like `entry['action']`.\n"
+    "- Each history entry exposes `.action`, `.frame`, and `.result`; entries are not subscriptable like `entry['action']`.\n"
     "- Each `history[i].frame` is the frame after `history[i].action`; each frame exposes only `.ascii`, `.step`, `.level`, `.shape`, and `.segmentation`.\n"
     "- Important history semantics: when `history` is non-empty, `history[-1].frame` is the same latest/post-action board as `current_frame`. It is not the previous board. To inspect the state before the latest action, use `previous_frame` or `history[-2].frame` when available.\n"
     "- `previous_frame` is the frame before the most recent real environment action, or `None` if no previous frame is available.\n"
     "- `last_action` is the most recent real environment action name/display, or `None` before any real action.\n"
     "- `last_action_frame` is the post-action frame for `last_action`; it matches `current_frame` after a real action.\n"
-    "- `transitions` is a chronological list of actual action transitions, excluding the initial seeded frame. Each transition exposes `.action`, `.before_frame`, `.after_frame`, `.frame` (alias of `.after_frame`), and `.result`.\n"
-    "- `last_transition` is `transitions[-1]` or `None`. Its `.result` mirrors `last_action_result`; older transitions may have an empty `.result`. For before/after diffs, compare `last_transition.before_frame` to `last_transition.after_frame`; do not compare `current_frame` to `history[-1].frame`.\n"
-    "- `last_action_result` is the persisted result dict from the most recent `action(...)` call. It remains available across later Python inspection calls that do not call `action(...)`, and is `{}` before any action result exists. Read transition metadata from fields/keys such as `last_action_result['board_changed']`, `last_action_result['done']`, `last_action_result['level_completed']`, `last_action_result['game_over']`, `last_action_result['run_complete']`, `last_action_result['reward']`, and `last_action_result['valid_actions']`.\n"
+    "- `transitions` is a chronological list of actual action transitions, excluding the initial seeded frame. Each transition exposes `.action`, `.before_frame`, `.after_frame`, `.frame` (alias of `.after_frame`), and `.result`. Each `.result` describes that individual action and is retained with its history entry.\n"
+    "- `last_transition` is `transitions[-1]` or `None`. A refused action creates no transition and does not overwrite existing transition results. For before/after diffs, compare `last_transition.before_frame` to `last_transition.after_frame`; do not compare `current_frame` to `history[-1].frame`.\n"
+    "- `last_action_call_result` contains the result of the latest model-issued `action(...)` call, including batch totals, executed/skipped actions, and stop reasons. `action(...)` returns this result. It remains available across later Python inspection calls and analyzer turns, and is `{}` before any action call result exists.\n"
+    "- On the next analyzer turn after automatic RESET, `current_frame` shows the restarted level. The fatal action and RESET are separate transitions. Each transition result has `automatic=True` for a harness-initiated action and `automatic=False` for a model-issued action. Automatic RESET does not overwrite `last_action_call_result`: it still describes the call that caused death.\n"
     "- `valid_actions` is the current list of valid action names.\n"
     "- Call `action(actions)` to execute one or more real environment actions from Python.\n"
     "- Pass `action(actions)` a list like `['LEFT']` or `[{'action': 'MOUSE', 'row': 4, 'col': 7}]`.\n"
     "- One action usually returns one frame, but a single action can result in a short multi-frame animation.\n"
-    "- After `action(actions)` returns, `current_frame`, `previous_frame`, `history`, `transitions`, `valid_actions`, and `last_action_result` are refreshed.\n"
+    "- After `action(actions)` returns, `current_frame`, `previous_frame`, `history`, `transitions`, `valid_actions`, and `last_action_call_result` are refreshed.\n"
+)
+
+ANIMATION_ADDENDUM_TIMELINE = (
+    "- `last_animation_timeline` is a diff timeline of those frames: it "
+    "shows which cells changed at each step. It is a dict with `action`, `frames` and "
+    "`steps`, one entry per step that changed anything. Each entry has `step`, "
+    "`changed`, `bbox`, and "
+    "either `changes` (the cells, as `old>new @ (row,col) ...`) or `transitions` (a "
+    "count per colour change, when there were too many cells to list).\n"
+)
+
+ANIMATION_ADDENDUM = (
+    "- One action can return a short animation. `current_frame` is its final frame.\n"
+    "- When the final executed action of the latest model-issued call animated, `last_action_call_result['animation']` "
+    "reports `frames`, `transient_pixels` and `transient_bbox`. Transient cells "
+    "changed and then changed BACK during the animation, so they appear in no frame "
+    "you can otherwise reach - not in `current_frame`, not in `previous_frame`, not "
+    "in `history`.\n"
+    "- `last_animation_frames` is the frames of the last executed action that "
+    "animated, as views with "
+    "`.ascii`, `.shape` and `.segmentation`, the same as `current_frame`. Index 0 is "
+    "the first frame of the animation and the last entry is the board it settled on. "
+    "After a death these are the frames of the FATAL action, not of the automatic "
+    "reset that followed it. Crop "
+    "their `.ascii` yourself; printing a whole 64x64 frame will not fit the tool "
+    "budget. Empty when the action did not animate, and reading them costs no "
+    "in-game action.\n"
 )
 
 MULTIMODAL_CONTEXT_ADDENDUM = (
@@ -73,7 +194,7 @@ MULTIMODAL_CONTEXT_ADDENDUM = (
     "- You can use images and other tools to understand the game state and guide your strategy, each may be useful depending on the current uncertainty.\n"
 )
 
-PYTHON_ADDENDUM = (
+PYTHON_ADDENDUM_HEAD = (
     "\n\nPython tool guidance:\n"
     "- Use `current_frame.segmentation` as your primary view of the board -- objects, colors, containment, adjacency, and cross-frame object hashes.\n"
     "- Use `current_frame.ascii` only to read a small, specific region of the board when `segmentation` is not enough; never use it to scan or summarize the whole board.\n"
@@ -82,9 +203,13 @@ PYTHON_ADDENDUM = (
     "- The only tool is `python`; call it with one ephemeral `code` string.\n"
     "- Always inspect `current_frame`, `history`, and `valid_actions` from Python instead of reasoning from the raw board by eye.\n"
     "- For the most recent change, compare `previous_frame` to `current_frame`, or `last_transition.before_frame` to `last_transition.after_frame`. `history[-1].frame` is the current frame, so comparing it to `current_frame` only compares the board to itself.\n"
-    "- Maintain a compact working world model: what entities or regions exist, what actions seem to do, what the goal likely is, what remains uncertain, and what plan best fits the evidence so far.\n"
+)
+
+# split so the world-model lines can be swapped in at their original
+# position - immediately before the IMPORTANT search guidance - rather than
+# appended after the whole addendum
+PYTHON_ADDENDUM_TAIL = (
     "- IMPORTANT: Especially when the game is about making an agent navigate to a target, it is usually safer to write an explicit search algorithm such as BFS. More generally, when the objective is understood but the best action order is unclear, pathfinding, flood fill, BFS, DFS, beam search, shortest-path search, limited action-sequence search, or custom heuristics are all valid.\n"
-    "- Optimize for the shortest reliable sequence that advances the current goal as described by your world model. If confidence is low, program a discriminating probe and revise the world model from the result.\n"
     "- Once the important state variables and action effects are sufficiently understood, stop probing and search in the inferred state space.\n"
     "- Inspect current and history frames from Python instead of describing frames freehand.\n"
     "- Never print or echo full board frames. Return only compact derived summaries such as object lists, diffs, coordinates, counts, or tiny local crops.\n"
@@ -98,17 +223,124 @@ PYTHON_ADDENDUM = (
     "- `action(...)` accepts an ordered list of one or more actions. Once your code has selected a reliable sequence, it is often useful to batch it.\n"
     "- You can also call `action(...)` multiple times in one Python snippet, including inside loops. Each call updates the preloaded variables before execution continues.\n"
     "- If an action result reports `game_over`, `run_complete`, `level_completed`, or `done`, stop acting immediately and re-ground on the next turn.\n"
+    "- Flag semantics: `game_over` = this attempt FAILED (death or a limit ran out) — the level auto-resets to its initial state and completed levels are kept; it never means the run is won. `level_completed` = advanced one level. `run_complete`/`done` = the whole game is won. Deaths come either from the action itself (e.g., a hazard cell) or from a step/time budget depleting — watch for a bar at the grid border that shrinks each action, and count remaining budget into your plans.\n"
+)
+
+FRAME_DIFF_HINT_ADDENDUM = (
+    "- `frame_diff(before=None, after=None)` compares two frames. Omitted arguments default to `previous_frame` and `current_frame`, respectively.\n"
+    "- It returns a dictionary containing `changed_cell_count` (an integer) and lists named `moved`, `rotated`, `appeared`, `disappeared`, `changed_color`, and `resized`. Unchanged objects are omitted.\n"
+    "- Object-level changes are inferred by matching same-color, 4-connected components between frames: cells connect through shared edges, not diagonally. A game object may therefore be split into several components, or touching same-color objects may form one component.\n"
+    "- `moved` describes components matched at different positions. Matching does not establish persistent object identity: with multiple similar components, the reported correspondence may be ambiguous or incorrect. Movement combined with rotation includes `rotated_by` in degrees clockwise; `rotated` describes rotation in place.\n"
+    "- `changed_color` requires identical occupied cells with a different color. Entries contain `at`, `pixels`, `before_color`, `after_color`, `before_hash`, and `after_hash`.\n"
+    "- `resized` describes same-color components with overlapping footprints whose size changed. This can include backgrounds and budget bars. `at_before` and `at_after` can differ.\n"
+    "- `appeared` and `disappeared` entries include `bbox` in `[r0, c0, r1, c1]` order. Object hashes use the same rotation-invariant shape-and-color representation as segmentation; they are not unique object identities, and multiple objects can share a hash.\n"
+    "- Examples: `frame_diff()` compares the latest action’s before/after frames; `frame_diff(history[-3].frame, history[-2].frame)` compares two explicitly selected historical frames, when available.\n"
+)
+
+STEP_VERIFICATION_ADDENDUM = (
+    "- During a multi-action plan, verify important effects by comparing the actual game "
+    "state with your predictions—for example, changed cells, object properties, positions, "
+    "or counters. Make these comparisons in code so the batch can stop when a mismatch "
+    "undermines the remaining plan. Test uncertain interactions before committing to long "
+    "batches.\n"
+    "- If your search finds no solution under your current model of the game, remember "
+    "that the game is solvable. Reconsider your mechanics, goal, search implementation, "
+    "or search limits, including interactions with new elements. Take a targeted action "
+    "to test an uncertain rule or overlooked interaction, then update your model from "
+    "the result.\n"
+)
+
+WORLD_MODEL_ADDENDUM = (
+    "- Maintain a compact working world model: what entities or regions exist, what actions "
+    "seem to do, what the goal likely is, what remains uncertain, and what plan best fits the "
+    "evidence so far.\n"
+    "- Optimize for the shortest reliable sequence that advances the current goal as described "
+    "by your world model. If confidence is low, program a discriminating probe and revise the "
+    "world model from the result.\n"
+)
+
+WORLD_MODEL_FREE_ADDENDUM = (
+    "- Optimize for the shortest reliable sequence that advances the current goal. If "
+    "confidence is low, program a discriminating probe and revise your approach from the "
+    "result.\n"
+)
+
+DEATH_GUARD_ADDENDUM = (
+    "- An action that ended a previous attempt from this EXACT board state is refused before "
+    "execution: it raises `KnownDeathActionError`, nothing runs, and no attempt is lost. "
+    "`stop_reason` is `known_death`. The guard checks every position of a batch, not just the "
+    "first, so a fatal move is caught whichever route reached it. To repeat such an action "
+    "deliberately, issue it again as your very next action.\n"
 )
 
+NOOP_GUARD_ADDENDUM = (
+    "- An action already proven to change nothing in this EXACT board state is refused before "
+    "execution: it raises `KnownNoOpActionError`, which is NOT catchable and ends the snippet - "
+    "nothing runs, "
+    "and no action budget is spent. `stop_reason` is `known_noop`. If you have a specific reason "
+    "to believe the action will behave differently now, issue that same action again as your very "
+    "next action and it will execute.\n"
+)
+
+PREFER_TOOL_CALLS_LINE = (
+    # The closing clause used to warn that a long chain "wastes the whole turn".
+    # That overstated it - a truncated reply keeps its reasoning in history, and
+    # what is lost is only that nothing executed - and it aimed at the tail,
+    # since truncation touched about 4% of replies. The speed claim is the
+    # stronger one and it is true: decode runs at tens of tokens per second per
+    # stream, while a tool call is a short generation plus a mostly-cached
+    # prefill, so 2,000 tokens of further reasoning costs far more wall-clock
+    # than running the code that would settle the question.
+    "- Reading and computing cost nothing; only `action(...)` spends the level budget. When you "
+    "are unsure, prefer another tool call over more reasoning: the code answers what the "
+    "reasoning would only guess at, and running it is faster than thinking your way to the "
+    "same answer.\n"
+)
+
+EPHEMERAL_FUNCTIONS_LINE = (
+    "- The `python` tool code is not saved between calls, so rewrite any custom utility logic you still need.\n"
+)
+
+PERSISTENT_FUNCTIONS_LINE = (
+    "- Python variables reset between tool calls. Eligible functions you define are "
+    "retained automatically {lifetime}. Call your retained functions directly without "
+    "repeating their definitions. Redefine a function when you need to change it.\n"
+    "- For a function to be retained, its dependencies must also be available next call. "
+    "Pass snippet-specific data as arguments. You may use Python builtins, provided "
+    "globals such as `current_frame`, and other retained functions. Provided globals "
+    "are refreshed for each call. {import_guidance} Use plain "
+    "top-level definitions without decorators or annotations, and literal defaults. "
+    "Tool results report which functions were retained and explain any rejection.\n"
+)
+
+
 COMPACT_TOOL_SESSION_ADDENDUM = (
     "\n\nTool session rules:\n"
     "- You have exactly one tool: `python`.\n"
     f"- {TOOL_CALL_FORMAT_GUIDANCE}\n"
-    "- The `python` tool code is not saved between calls, so rewrite any custom utility logic you still need.\n"
+    "{persistent_functions}"
     "- You can call the `python` tool as many times as you want per step. Investigate until your code has a clear probe or plan.\n"
     "- Do not ration tool calls when the state is unclear. Spend extra tool calls to confirm what changed between frames and whether the last action affected gameplay state or only HUD elements such as countdown bars.\n"
-    "- After `action(...)` returns, the structured runtime state is refreshed before the next Python statement and before the next tool call. Inspection-only Python calls do not clear `last_action_result`.\n"
+    "{prefer_tool_calls}"
+    "- After `action(...)` returns, the structured runtime state is refreshed before the next Python statement and before the next tool call. Inspection-only Python calls do not clear `last_action_call_result`.\n"
     "- Each `python` tool call has a hard time limit of 30 seconds.\n"
     "- Tool responses are capped to about {tool_output_tokens} tokens. If a response is cut off, the tool result will tell you that.\n"
     "- Keep code snippets short and purpose-built rather than dumping large frameworks into one call.\n"
 )
+
+
+SUMMARY_REQUEST_PROMPT = (
+    "Summarize everything above: what you have established about this game, what you have "
+    "ruled out, what you were in the middle of, and anything else a later turn would need. "
+    "Keep uncertain ideas tentative, and preserve unresolved alternatives. "
+    "Do not act and do not call any tool - write the summary only.\n"
+    # No "the turns after this one stay visible": the summary is appended at
+    # the end of history, so nothing follows it when it is written. That
+    # sentence belonged to an earlier design where the summary was inserted
+    # ahead of a retained tail, and here it points at messages that do not
+    # exist yet.
+    "Write it to stand completely on its own. Everything above this point will eventually be "
+    "dropped and replaced by this summary, and earlier summaries go with it, so anything you "
+    "leave out is gone.\n"
+    "Length is not a problem here; leaving something out is."
+)
diff --git a/ARC3-Inference/inference/agent/python_tool_sandbox.py b/ARC3-Inference/inference/agent/python_tool_sandbox.py
index ac24747..58a94b3 100644
--- a/ARC3-Inference/inference/agent/python_tool_sandbox.py
+++ b/ARC3-Inference/inference/agent/python_tool_sandbox.py
@@ -3,6 +3,7 @@ from __future__ import annotations
 
 import inspect
 import json
+import logging
 import os
 import queue
 import signal
@@ -14,12 +15,15 @@ import textwrap
 import time
 from typing import Any, Callable
 
+from inference.utils import frame_diff as _frame_diff_module
 from inference.utils import segmentation as _segmentation
+from inference.utils import retained_functions as _retained_functions
 from inference.utils.grid_utils import ARC_COLOR_CHARS
 
 
 _SANDBOX_BOOTSTRAP = textwrap.dedent(
     r"""
+    import ast
     import builtins
     import contextlib
     import io
@@ -37,12 +41,22 @@ _SANDBOX_BOOTSTRAP = textwrap.dedent(
 
     __SEGMENTATION_SOURCE__
 
+    __FRAME_DIFF_SOURCE__
+
+    __RETAINED_FUNCTIONS_SOURCE__
+
     HOST_STDOUT = sys.stdout
 
     SAFE_MODULES = {
         "bisect",
         "collections",
         "copy",
+        # pure-Python sequence comparison, no I/O. Measured across 32
+        # transcripts it was imported 9 times and USED none of them - the model
+        # reaches for it on "compare two things" and then writes the comparison
+        # by hand. Allowing it therefore buys robustness, not capability: a
+        # stray import line no longer costs the whole snippet.
+        "difflib",
         "fractions",
         "functools",
         "heapq",
@@ -56,6 +70,25 @@ _SANDBOX_BOOTSTRAP = textwrap.dedent(
         "string",
     }
     SAFE_BUILTINS = {
+        # Exception types. Only four were present, so anything narrower than
+        # `except Exception` failed at the except clause and took the snippet
+        # with it. These are inert classes and expose nothing; SystemExit and
+        # KeyboardInterrupt stay out because the sandbox uses them for its own
+        # timeout and shutdown handling.
+        "ArithmeticError",
+        "AssertionError",
+        "AttributeError",
+        "EOFError",
+        "IndexError",
+        "KeyError",
+        "LookupError",
+        "NameError",
+        "NotImplementedError",
+        "OverflowError",
+        "RecursionError",
+        "StopIteration",
+        "UnboundLocalError",
+        "ZeroDivisionError",
         "abs",
         "all",
         "any",
@@ -147,9 +180,10 @@ _SANDBOX_BOOTSTRAP = textwrap.dedent(
 
 
     class HistoryEntryView:
-        def __init__(self, *, action, frame):
+        def __init__(self, *, action, frame, result=None):
             self.action = action
             self.frame = frame
+            self.result = dict(result) if isinstance(result, dict) else {}
 
         def __str__(self):
             return f"AsciiHistoryEntryView(action={self.action!r}, frame={self.frame})"
@@ -197,12 +231,13 @@ _SANDBOX_BOOTSTRAP = textwrap.dedent(
                 HistoryEntryView(
                     action=str(entry.get("action", "")),
                     frame=_frame_from_payload(entry.get("frame")),
+                    result=entry.get("result"),
                 )
             )
         return items
 
 
-    def _transitions_from_history(history, last_action_result):
+    def _transitions_from_history(history):
         transitions = []
         for index, entry in enumerate(history):
             action = str(getattr(entry, "action", "") or "").strip()
@@ -214,11 +249,9 @@ _SANDBOX_BOOTSTRAP = textwrap.dedent(
                     action=action,
                     before_frame=before_frame,
                     after_frame=entry.frame,
-                    result={},
+                    result=entry.result,
                 )
             )
-        if transitions and isinstance(last_action_result, dict):
-            transitions[-1].result = dict(last_action_result)
         return transitions
 
 
@@ -327,15 +360,38 @@ _SANDBOX_BOOTSTRAP = textwrap.dedent(
             "result": None,
         }
         runtime_globals["__builtins__"]["__import__"] = _safe_import
+        _animation_views = []
+
+        def frame_diff(before=None, after=None):
+            # Object-level diff between two frames. Defaults: previous_frame -> current_frame.
+            # Returns only what changed; unchanged objects are omitted.
+            if before is None:
+                before = runtime_globals.get("previous_frame")
+            if after is None:
+                after = runtime_globals.get("current_frame")
+            if before is None or after is None:
+                return {"error": "need two frames (no previous_frame available?)"}
+            if before.shape != after.shape:
+                return {"changed_cell_count": -1, "note": "frame shapes differ; full redraw",
+                        "before_shape": before.shape, "after_shape": after.shape}
+            return compute_frame_diff(
+                before._grid,
+                after._grid,
+                before.segmentation.get("nodes", []),
+                after.segmentation.get("nodes", []),
+            )
 
         def _refresh_state(state_payload):
             current_frame = _frame_from_payload(state_payload.get("current_frame"))
             history = _history_from_payload(state_payload.get("history"))
-            last_action_result = state_payload.get("last_action_result")
+            last_action_call_result = state_payload.get("last_action_call_result")
             action_result = (
-                dict(last_action_result) if isinstance(last_action_result, dict) else {}
+                dict(last_action_call_result) if isinstance(last_action_call_result, dict) else {}
             )
-            transitions = _transitions_from_history(history, action_result)
+            # action_echo is printed by action(), then removed from its return value.
+            # Keep the exposed call result identical across subsequent inspections.
+            action_result.pop("action_echo", None)
+            transitions = _transitions_from_history(history)
             last_transition = transitions[-1] if transitions else None
 
             runtime_globals["current_frame"] = current_frame
@@ -350,12 +406,98 @@ _SANDBOX_BOOTSTRAP = textwrap.dedent(
                 last_transition.after_frame if last_transition is not None else None
             )
             runtime_globals["last_action"] = last_transition.action if last_transition is not None else None
+            runtime_globals["frame_diff"] = frame_diff
             runtime_globals["valid_actions"] = [str(item) for item in state_payload.get("valid_actions", [])]
-            runtime_globals["last_action_result"] = action_result
+            runtime_globals["death_ledger"] = state_payload.get("death_ledger") or {"attempts": []}
+            runtime_globals["last_action_call_result"] = action_result
+
+        # Both refusals derive from BaseException, not Exception, so a
+        # defensive `except Exception:` around action(...) - which models write
+        # routinely - cannot swallow them. A refusal means the action did not
+        # run at all, so the snippet's assumed state is already wrong and every
+        # line after it rests on a false premise. One run treated 1500
+        # consecutive refusals as proof that every cell on the board was inert.
+        # StaleStateActionError below is deliberately catchable: it reports a
+        # belief that MAY be stale rather than a verdict on the action.
+        class KnownNoOpActionError(BaseException):
+            pass  # raised INSTEAD of executing an action already proven inert here
+
+        class KnownDeathActionError(BaseException):
+            pass  # raised INSTEAD of executing an action that ended a prior attempt
+
+        # Raised INSTEAD of executing an action this snippet has already run
+        # from this exact board. Buggy loop code can spend a whole life bar
+        # getting nowhere - a cursor that wraps turns RIGHT into a short cycle,
+        # a mistaken `while pos < j: LEFT` walks away from its target forever -
+        # and the terminal latch only stops that once the game actually ends,
+        # measured at 20 to 30 wasted actions. BaseException like the two above:
+        # the shape that produces these loops is `try: action(...) except
+        # Exception:` inside a while, which would otherwise swallow the verdict
+        # and keep going. No override slot: the snippet is not reading the
+        # refusal, its code is running, and writing a new snippet IS the
+        # response - the map is rebuilt per execution.
+        class RepeatedActionInStateError(BaseException):
+            pass
+
+        # Raised when a PREVIOUS action in this snippet changed nothing, and the
+        # snippet then tries to act again. Deliberately a plain Exception, not
+        # BaseException: unlike the two refusals above this is a warning about
+        # the snippet's beliefs rather than a verdict on the action, and a model
+        # that has re-read current_frame may legitimately want to proceed.
+        class StaleStateActionError(Exception):
+            pass
+
+        runtime_globals["KnownNoOpActionError"] = KnownNoOpActionError
+        runtime_globals["KnownDeathActionError"] = KnownDeathActionError
+        runtime_globals["StaleStateActionError"] = StaleStateActionError
+        runtime_globals["RepeatedActionInStateError"] = RepeatedActionInStateError
+        # armed when an action changes nothing; scoped to THIS snippet because
+        # the closure is rebuilt per execution, so no reset path can be missed
+        _stale_state: list[str] = []
+
+        # Raised when a snippet tries to act after the attempt or level ended.
+        # Derives from BaseException, not Exception, so a defensive
+        # `try: action(...) except Exception:` around the call - which models
+        # write routinely - cannot swallow it. Continuing to execute a path
+        # computed against a board that no longer exists is never correct.
+        class TerminalStateActionError(BaseException):
+            pass
+
+        runtime_globals["TerminalStateActionError"] = TerminalStateActionError
+        # latched when an action reports a terminal result; scoped to THIS
+        # snippet because these closures are rebuilt for every execution, so
+        # no reset path can be missed and the next snippet always starts clear
+        _terminal_state = []
+
+        _TERMINAL_DETAIL = {
+            "game_over": (
+                "the previous action reported game_over. The level has been reset to its "
+                "start, so the board you planned against is gone and this action was not "
+                "executed. The next turn begins from the new state."
+            ),
+            "level_completed": (
+                "the previous action reported level_completed. A new level has been loaded, "
+                "so the board you planned against is gone and this action was not executed. "
+                "The next turn begins from the new state."
+            ),
+            "run_complete": (
+                "the previous action reported run_complete. The run is finished and this "
+                "action was not executed."
+            ),
+        }
 
         def action(actions):
+            if _terminal_state:
+                raise TerminalStateActionError(_TERMINAL_DETAIL[_terminal_state[0]])
             normalized_actions = _normalize_actions(actions)
-            _send({"type": "action", "actions": normalized_actions})
+            _send({
+                "type": "action",
+                "actions": normalized_actions,
+                # the solver decides precedence: a known death or a known no-op
+                # is a sharper message than "your state may be stale", and it
+                # carries a concrete override
+                "stale_after": _stale_state[0] if _stale_state else None,
+            })
             reply = _recv()
             if reply.get("type") == "action_error":
                 raise RuntimeError(str(reply.get("error", "action failed")))
@@ -363,28 +505,225 @@ _SANDBOX_BOOTSTRAP = textwrap.dedent(
                 raise RuntimeError("Invalid action response from sandbox host.")
             action_result = reply.get("action_result") or {}
             action_results.append(action_result)
+            # Cached animations describe the previous action. Invalidate the
+            # existing views so aliases also refresh, even if a guard raises.
+            for view in _animation_views:
+                view.invalidate()
             _refresh_state(reply.get("state") or {})
+            if action_result.get("stop_reason") == "reset_not_first":
+                raise RuntimeError(str(action_result.get("stop_detail")))
+            if action_result.get("stop_reason") == "stale_state":
+                raise StaleStateActionError(
+                    str(action_result.get("stop_detail") or "stale state")
+                )
+            if action_result.get("stop_reason") == "known_death":
+                raise KnownDeathActionError(
+                    str(action_result.get("stop_detail") or "known fatal action was not executed.")
+                )
+            if action_result.get("stop_reason") == "repeated_action_in_state":
+                raise RepeatedActionInStateError(
+                    str(action_result.get("stop_detail") or "")
+                    or "this action was already run from this board in this snippet"
+                )
+            if action_result.get("stop_reason") == "known_noop":
+                # nothing executed and no action was spent; halt the snippet so
+                # the model re-plans instead of continuing on a false premise
+                raise KnownNoOpActionError(
+                    str(action_result.get("stop_detail") or "known no-op action was not executed.")
+                )
+            for _flag in ("run_complete", "done", "level_completed", "game_over"):
+                if action_result.get(_flag):
+                    # `done` means the run was won; report it as run_complete
+                    _terminal_state.append(
+                        "run_complete" if _flag == "done" else _flag
+                    )
+                    break
+            _echo = action_result.pop("action_echo", None)
+            if _echo:
+                # into the snippet's own stdout, so a loop can see what its
+                # call did without waiting for the next turn's opener
+                print(_echo)
+            gameplay_changed = action_result.get("gameplay_changed")
+            if gameplay_changed is True:
+                # a real change means the snippet's beliefs are current again
+                del _stale_state[:]
+            if gameplay_changed is False and action_result.get("executed"):
+                _executed = action_result.get("executed_actions")
+                _name = (
+                    _executed[-1]
+                    if isinstance(_executed, list) and _executed
+                    else action_result.get("action_display")
+                    or action_result.get("action_name")
+                    or "?"
+                )
+                # Arm rather than raise. The action executed and changed
+                # nothing, which is often exactly what the snippet went looking
+                # for - it printed the board before, and wants to print it
+                # after. Killing it here throws away the second half of the
+                # measurement AND the batch stop's own trace, which is computed
+                # and then discarded. So let the snippet finish observing, and
+                # stop it only if it tries to ACT on the stale belief.
+                if not _stale_state:
+                    _stale_state.append(str(_name))
             return action_result
 
+        def _animation_fetch(kind):
+            _send({"type": "animation", "request": {"kind": kind}})
+            reply = _recv()
+            if reply.get("type") == "animation_error":
+                raise RuntimeError(str(reply.get("error", "animation failed")))
+            if reply.get("type") != "animation_result":
+                raise RuntimeError("Invalid animation response from sandbox host.")
+            payload = reply.get("animation")
+            if kind != "frames":
+                return payload or {}
+            return [
+                FrameView(
+                    ascii=item.get("ascii", ""),
+                    step=item.get("step", 0),
+                    level=item.get("level", 0),
+                    shape=item.get("shape", (0, 0)),
+                    grid=item.get("grid", ()),
+                )
+                for item in (payload or [])
+            ]
+
         runtime_globals["action"] = action
+        if initial.get("animation_enabled"):
+            # Globals rather than attributes on last_transition: after a death
+            # the harness appends a RESET entry, so last_transition is the
+            # reset and not the action that animated. The record is simply
+            # "the last executed action that animated", and naming it that way
+            # removes the mismatch. Empty when the last executed action
+            # returned a single frame.
+            class _LazyAnimation:
+                def __init__(self, kind, empty):
+                    self._kind = kind
+                    self._empty = empty
+                    self._value = None
+
+                def _get(self):
+                    if self._value is None:
+                        self._value = _animation_fetch(self._kind) or self._empty
+                    return self._value
+
+                def invalidate(self):
+                    self._value = None
+
+                def __len__(self):
+                    return len(self._get())
+
+                def __iter__(self):
+                    return iter(self._get())
+
+                def __getitem__(self, key):
+                    return self._get()[key]
+
+                def __bool__(self):
+                    return bool(self._get())
+
+                def __contains__(self, key):
+                    return key in self._get()
+
+                def keys(self):
+                    return self._get().keys()
+
+                def get(self, key, default=None):
+                    return self._get().get(key, default)
+
+                def __repr__(self):
+                    return repr(self._get())
+
+            runtime_globals["last_animation_frames"] = _LazyAnimation("frames", [])
+            runtime_globals["last_animation_timeline"] = _LazyAnimation("timeline", {})
+            _animation_views.extend((runtime_globals["last_animation_frames"],
+                                     runtime_globals["last_animation_timeline"]))
+
         _refresh_state(initial.get("state") or {})
 
+        _persistence_enabled = bool(initial.get("persistence_enabled"))
+        _retention_options = {
+            "retain_imports": bool(initial.get("retain_imports")),
+            "repair_hints": bool(initial.get("repair_hints")),
+        }
+        _kept_pool = {}
+        _retention_rejected = []
+        _available_names = set(runtime_globals) | set(runtime_globals["__builtins__"])
+        if _persistence_enabled:
+            _before_restore = dict(runtime_globals)
+            try:
+                _kept_pool, _retention_rejected = restore_functions(
+                    initial.get("kept_functions") or [], runtime_globals, _available_names,
+                    **_retention_options)
+            except Exception:
+                # Optional cache maintenance must never prevent this snippet running.
+                runtime_globals.clear()
+                runtime_globals.update(_before_restore)
+                _kept_pool = {}
+                _retention_rejected = [{"name": "all", "reason": "restoration failed; redefine needed functions"}]
+
+        def _collect_retention_payload():
+            if not _persistence_enabled:
+                return {}
+            try:
+                kept, rejected = collect_functions(
+                    _code_text, _kept_pool, runtime_globals, _available_names,
+                    **_retention_options)
+                return {
+                    "keepable_functions": [{"name": n, "source": src} for n, src in kept.items()],
+                    "retention_rejected": _retention_rejected + rejected,
+                }
+            except Exception:
+                # Bookkeeping failures may drop helpers, but must never replace
+                # the snippet's original error/output or replay its actions.
+                return {"keepable_functions": [], "retention_rejected": [
+                    {"name": "all", "reason": "retention failed; redefine needed functions"}]}
+
         try:
-            compiled = compile(str(initial.get("code", "")), "<python_tool>", "exec")
+            _code_text = str(initial.get("code", ""))
+            _tree = ast.parse(_code_text, "<python_tool>")
+            _last_expr = None
+            if _tree.body and isinstance(_tree.body[-1], ast.Expr):
+                _last_expr = ast.Expression(_tree.body[-1].value)
+                ast.copy_location(_last_expr, _tree.body[-1])
+                _tree.body = _tree.body[:-1]
             with contextlib.redirect_stdout(stdout):
-                exec(compiled, runtime_globals, runtime_globals)
+                if _tree.body:
+                    exec(compile(_tree, "<python_tool>", "exec"), runtime_globals, runtime_globals)
+                if _last_expr is not None:
+                    _value = eval(compile(_last_expr, "<python_tool>", "eval"), runtime_globals, runtime_globals)
+                    if _value is not None:
+                        print(repr(_value))
+                        if runtime_globals.get("result") is None:
+                            runtime_globals["result"] = _value
             _send(
                 {
                     "type": "final",
+                    **_collect_retention_payload(),
                     "stdout": stdout.getvalue(),
                     "result": _json_safe(runtime_globals.get("result")),
                     "action_results": _json_safe(action_results),
                 }
             )
-        except Exception as exc:
+        except (
+            KnownNoOpActionError,
+            RepeatedActionInStateError,
+            KnownDeathActionError,
+            TerminalStateActionError,
+            Exception,
+        ) as exc:
+            # The three refusals derive from BaseException so a model's own
+            # `except Exception:` cannot swallow them - but THIS handler is the
+            # same construct, so without naming them explicitly they escaped it
+            # too, killed the child process, and the host reported a bare
+            # "Sandbox process exited unexpectedly". The refusal messages - the
+            # action name, that nothing was spent, how to override - were built
+            # and then never delivered. They stop the snippet either way; the
+            # point is that the model is told why.
             _send(
                 {
                     "type": "error",
+                    **_collect_retention_payload(),
                     "error": _sanitize_exception(exc),
                     "stdout": stdout.getvalue(),
                     "action_results": _json_safe(action_results),
@@ -395,12 +734,31 @@ _SANDBOX_BOOTSTRAP = textwrap.dedent(
     if __name__ == "__main__":
         main()
     """
-).replace("__SEGMENTATION_SOURCE__\n", inspect.getsource(_segmentation))
+).replace("__RETAINED_FUNCTIONS_SOURCE__\n", inspect.getsource(_retained_functions)).replace(
+    "__SEGMENTATION_SOURCE__\n", inspect.getsource(_segmentation)
+).replace(
+    "__FRAME_DIFF_SOURCE__\n", inspect.getsource(_frame_diff_module)
+)
+
+
+log = logging.getLogger(__name__)
 
 
 def _sanitize_host_error_text(text: str) -> str:
-    if not str(text or "").strip():
-        return "Sandbox process exited unexpectedly."
+    """What the MODEL is told when the sandbox process dies.
+
+    Deliberately fixed: the child's traceback carries host paths and internals
+    that have no business in the model's context. But the text was previously
+    read and thrown away entirely, so a crash left no trace anywhere - the one
+    thing that knows what went wrong said nothing. It now goes to the harness
+    log instead, where it is useful and nobody is prompted by it.
+    """
+    detail = str(text or "").strip()
+    if detail:
+        log.warning(
+            "sandbox process died; stderr follows:\n%s",
+            detail[-4000:],
+        )
     return "Sandbox process exited unexpectedly."
 
 
@@ -450,7 +808,11 @@ def run_sandboxed_python(
     code: str,
     timeout_seconds: int,
     initial_state: dict[str, Any],
-    action_handler: Callable[[list[dict[str, Any]]], dict[str, Any]],
+    action_handler: Callable[..., dict[str, Any]],
+    animation_handler: Callable[[dict[str, Any]], dict[str, Any]] | None = None,
+    kept_functions: list[str] | None = None,
+    retain_imports: bool = False,
+    repair_hints: bool = False,
 ) -> dict[str, Any]:
     with tempfile.TemporaryDirectory(prefix="rgb_python_tool_") as sandbox_dir:
         host_action_results: list[dict[str, Any]] = []
@@ -489,6 +851,11 @@ def run_sandboxed_python(
             process.stdin,
             {
                 "code": code,
+                "kept_functions": list(kept_functions or []),
+                "persistence_enabled": kept_functions is not None,
+                "retain_imports": retain_imports,
+                "repair_hints": repair_hints,
+                "animation_enabled": animation_handler is not None,
                 "timeout_seconds": timeout_seconds,
                 "sandbox_cwd": sandbox_dir,
                 "state": initial_state,
@@ -534,9 +901,37 @@ def run_sandboxed_python(
                 }
 
             msg_type = str(message.get("type", "")).strip()
+            if msg_type == "animation":
+                if animation_handler is None:
+                    _send_json_line(
+                        process.stdin,
+                        {"type": "animation_error",
+                         "error": "Animation frames are not available in this session."},
+                    )
+                    continue
+                try:
+                    animation_payload = animation_handler(dict(message.get("request") or {}))
+                except Exception:  # noqa: BLE001
+                    _send_json_line(
+                        process.stdin,
+                        {"type": "animation_error",
+                         "error": "animation failed in sandbox host."},
+                    )
+                    continue
+                _send_json_line(
+                    process.stdin,
+                    {"type": "animation_result", "animation": animation_payload},
+                )
+                continue
             if msg_type == "action":
                 try:
-                    action_result_payload = action_handler(list(message.get("actions") or []))
+                    # the message carries more than the action list: stale_after
+                    # tells the solver a previous action in this snippet changed
+                    # nothing, which it weighs after its own guards
+                    action_result_payload = action_handler(
+                        list(message.get("actions") or []),
+                        stale_after=message.get("stale_after"),
+                    )
                 except Exception:  # noqa: BLE001
                     _send_json_line(
                         process.stdin,
@@ -566,6 +961,9 @@ def run_sandboxed_python(
                     "result": message.get("result"),
                     "error": str(message.get("error", "") or ""),
                     "action_results": list(message.get("action_results") or host_action_results),
+                    "keepable_functions": list(message.get("keepable_functions") or []),
+                    **({"retention_rejected": message.get("retention_rejected", [])}
+                       if kept_functions is not None else {}),
                 }
 
             _wait_for_process_exit(process)
diff --git a/ARC3-Inference/inference/agent/runtime_state.py b/ARC3-Inference/inference/agent/runtime_state.py
index 03edebb..5a00139 100644
--- a/ARC3-Inference/inference/agent/runtime_state.py
+++ b/ARC3-Inference/inference/agent/runtime_state.py
@@ -2,7 +2,7 @@
 from __future__ import annotations
 
 import json
-from dataclasses import dataclass
+from dataclasses import dataclass, field
 from pathlib import Path
 from typing import Any
 
@@ -42,6 +42,7 @@ class Frame:
 class HistoryEntry:
     action: str
     frame: Frame
+    result: dict[str, Any] = field(default_factory=dict)
 
 
 def normalize_grid(raw: Any) -> tuple[tuple[int, ...], ...]:
@@ -95,13 +96,17 @@ def history_entry_from_payload(payload: Any) -> HistoryEntry | None:
     frame = frame_from_payload(payload.get("frame"))
     if frame is None:
         return None
-    return HistoryEntry(action=str(payload.get("action", "")).strip(), frame=frame)
+    return HistoryEntry(
+        action=str(payload.get("action", "")).strip(), frame=frame,
+        result=dict(payload["result"]) if isinstance(payload.get("result"), dict) else {},
+    )
 
 
 def history_entry_to_payload(entry: HistoryEntry) -> dict[str, Any]:
     return {
         "action": entry.action,
         "frame": frame_to_payload(entry.frame),
+        "result": dict(getattr(entry, "result", {}) or {}),
     }
 
 
diff --git a/ARC3-Inference/inference/agent/tool_agent.py b/ARC3-Inference/inference/agent/tool_agent.py
index 59655e9..84a9cdd 100644
--- a/ARC3-Inference/inference/agent/tool_agent.py
+++ b/ARC3-Inference/inference/agent/tool_agent.py
@@ -1,11 +1,16 @@
 """Direct OpenAI-compatible tool-calling analyzer for ARC puzzle runs."""
 from __future__ import annotations
 
+import base64
+import heapq
+import threading
 import json
 import logging
 import os
+import random
 import re
 import time
+from functools import lru_cache
 from dataclasses import dataclass
 from pathlib import Path
 from typing import Any, Callable, Optional
@@ -13,25 +18,61 @@ from urllib.parse import urlparse, urlunparse
 
 import requests
 
-from inference.agent.action_names import to_engine_action, to_model_action
+from inference.agent.noop_repeat_guard import NoopRepeatGuard, action_signature
+from inference.agent.priority_scheduler import (
+    ProgressPace, PrioritySnapshot, parse_total_levels, priority_value,
+)
+from inference.utils.animation import (
+    _transient_cells as _animation_transient_cells,
+    build_animation_frames,
+    build_animation_view,
+    describe_animation,
+)
+from inference.agent.action_names import reset_exposed, to_engine_action, to_model_action, undo_exposure_mode
+from inference.utils.retained_functions import function_signature
 from inference.agent.prompts import (
+    apply_level_transfer_system_guidance,
+    LEVEL_START_USER_PROMPT,
     COMPACT_TOOL_SESSION_ADDENDUM,
+    PERSISTENT_FUNCTIONS_LINE,
+    EPHEMERAL_FUNCTIONS_LINE,
+    SUMMARY_REQUEST_PROMPT,
+    PREFER_TOOL_CALLS_LINE,
     GAME_OVERVIEW_ADDENDUM,
-    PYTHON_ADDENDUM,
+    WORLD_MODEL_ADDENDUM,
+    WORLD_MODEL_FREE_ADDENDUM,
+    FRAME_DIFF_HINT_ADDENDUM,
+    STEP_VERIFICATION_ADDENDUM,
+    PYTHON_ADDENDUM_HEAD,
+    PYTHON_ADDENDUM_TAIL,
+    GAMEPLAY_CHANGED_ADDENDUM,
     STRUCTURED_RUNTIME_STATE_ADDENDUM,
+    ANIMATION_ADDENDUM,
+    ANIMATION_ADDENDUM_TIMELINE,
     MULTIMODAL_CONTEXT_ADDENDUM,
     TOOL_CALL_FORMAT_GUIDANCE,
     VISUAL_GAME_ADDENDUM,
+    ACTION_INFO_ADDENDUM,
+    UNDO_INFO_ADDENDUM,
+    RESET_INFO_ADDENDUM,
 )
 
 from inference.agent.vision_context import (
+    animation_composite_part,
+    animation_death_image_mode,
+    animation_image_mode,
+    animation_peak_part,
     current_grid_image_enabled,
     current_grid_image_part,
 )
 
 from inference.agent.python_tool_sandbox import run_sandboxed_python
 from inference.agent.runtime_state import Frame, HistoryEntry, RUNTIME_STATE_FILENAME, load_runtime_state
-from inference.utils.openai_compat import build_chat_payload, build_headers
+from inference.utils.openai_compat import (
+    assemble_streamed_chat_response,
+    build_chat_payload,
+    build_headers,
+)
 
 log = logging.getLogger(__name__)
 
@@ -73,6 +114,14 @@ def _get_env_bool(name: str, default: bool) -> bool:
     return default
 
 
+def _level_transfer_guidance_enabled() -> bool:
+    """Opt-in level-transfer prompts; set before constructing the agent.
+
+    ARC3_LEVEL_TRANSFER_GUIDANCE defaults off, preserving the original prompts.
+    """
+    return _get_env_bool("ARC3_LEVEL_TRANSFER_GUIDANCE", False)
+
+
 def _contains_tool_call_markup(*chunks: str) -> bool:
     for chunk in chunks:
         lowered = chunk.lower()
@@ -141,6 +190,13 @@ _LOCAL_ANALYZER_TOOL_STEPS = _get_env_int("LOCAL_ANALYZER_TOOL_STEPS", 12)
 _LOCAL_ANALYZER_TOOL_TIMEOUT = _get_env_int("LOCAL_ANALYZER_TOOL_TIMEOUT", 30)
 _LOCAL_ANALYZER_TOOL_OUTPUT_TOKENS = _get_env_int("LOCAL_ANALYZER_TOOL_OUTPUT_TOKENS", 1024)
 _LOCAL_ANALYZER_YIELD_SECONDS = _get_env_float("LOCAL_ANALYZER_YIELD_SECONDS", 0.0)
+# A wall-clock budget is really a token budget with the throughput baked in:
+# the same 45 seconds bought 2,140 generated tokens on one backend and 3,829 on
+# a faster one, so every serving change silently retuned it. Measured in
+# tokens the setting means the same thing whatever the hardware does. Zero
+# disables, as with the seconds; if both are set, whichever comes first ends
+# the turn.
+_LOCAL_ANALYZER_YIELD_TOKENS = _get_env_int("LOCAL_ANALYZER_YIELD_TOKENS", 0)
 _LOCAL_ANALYZER_ENABLE_THINKING = _get_env_bool("LOCAL_ANALYZER_ENABLE_THINKING", True)
 _LOCAL_ANALYZER_TEMPERATURE = _get_env_float("LOCAL_ANALYZER_TEMPERATURE", 0.6)
 _LOCAL_ANALYZER_TOP_P = _get_env_float("LOCAL_ANALYZER_TOP_P", 0.95)
@@ -148,15 +204,102 @@ _LOCAL_ANALYZER_TOP_K = _get_env_int("LOCAL_ANALYZER_TOP_K", 20)
 _LOCAL_ANALYZER_SEED = _get_env_int("LOCAL_ANALYZER_SEED", -1)
 _REQUEST_SAFETY_MARGIN_TOKENS = 512
 _CONTEXT_OVERFLOW_RETRY_TRIM_TOKENS = 512
-_PERSISTENT_HISTORY_ASSISTANT_TURNS = 30
+def _context_drain_floor_mode() -> bool:
+    """Whether the drain stops ABOVE the target rather than below it.
+
+    History is dropped in whole blocks, so trimming "until under the target"
+    normally overshoots by most of a block. With a 40k budget and a 12k drain
+    the band is nominally 28k-40k, but simulation with ~6k blocks gives an
+    effective 14k-38k: the average sits ~3k below the nominal midpoint and the
+    minimum is far below the target.
+
+    Floor mode drops the mandatory blocks to get under budget, then stops
+    before a drop would cross below the target - so the retained content keeps
+    a real floor and the band means what it says. The cost is trim frequency:
+    the overshoot is what lengthens the interval between trims, so removing it
+    trims about half again as often, which shortens the stable prefix and cuts
+    the prefix-cache hit rate. Off by default for that reason; the same average
+    depth is available by raising the budget, which does not touch trim
+    frequency."""
+    return _get_env_bool("ARC3_CONTEXT_DRAIN_FLOOR", False)
+
+
+def _context_drain_tokens() -> int:
+    """Hysteresis for context trimming, in tokens.
+
+    Without it, trimming drops the oldest exchange the moment the estimate
+    exceeds the budget and stops the instant it fits - so once history reaches
+    the ceiling, nearly every turn evicts a little. Each eviction is cheap in
+    tokens but total in cache cost: prefix caching keys on the token sequence
+    from position zero, so removing anything from the FRONT shifts every
+    surviving token and invalidates the whole cached prefix, not just the part
+    removed. Frequent small evictions are therefore the worst case.
+
+    With a drain set, trimming still TRIGGERS at the budget but keeps dropping
+    until it is `drain` tokens below it, so evictions become rare and large:
+    the same tokens discarded overall, a fraction of the invalidations. The
+    cost is average history depth - the window sits below the budget most of
+    the time rather than pressed against it.
+
+    Size it against tokens per turn: at ~4-5k per turn including the opener,
+    8-12k buys two or three clean turns between evictions. 0 (default) keeps
+    the original drop-until-it-fits behavior."""
+    return max(0, _get_env_int("ARC3_CONTEXT_DRAIN_TOKENS", 0))
+
+
+def _history_turn_drain() -> int:
+    """Hysteresis for the assistant-turn cap, in assistant messages.
+
+    The token trimmer got hysteresis in ARC3_CONTEXT_DRAIN_TOKENS, but the
+    turn cap runs AFTER it and had none: once history sits at the cap, every
+    commit shaves back to exactly N, so the prefix is invalidated on every
+    single turn. With a 46k window and a cap of 30 assistant messages the cap
+    is the binding limit, which is why a token drain alone changed nothing.
+
+    Set this and the cap triggers at N but drains to N - drain, so evictions
+    become periodic instead of continuous."""
+    return max(0, _get_env_int("ARC3_HISTORY_TURN_DRAIN", 0))
+
+
+def _history_drain_coalesce() -> bool:
+    """Take the deep turn cut whenever the token trimmer already dropped.
+
+    Invalidation is all-or-nothing: removing anything from the front of
+    history shifts every surviving token, so a commit that has already
+    dropped blocks for token reasons has already paid the full cache cost.
+    Dropping turns in that same commit is therefore free, whereas letting the
+    cap fire on its own a turn later costs a second invalidation. This makes
+    the two mechanisms evict together rather than alternately."""
+    return _get_env_bool("ARC3_HISTORY_DRAIN_COALESCE", False)
+
+
+def _persistent_history_assistant_turns() -> int:
+    """Cap on assistant messages kept in persistent history.
+
+    Counts ASSISTANT MESSAGES, not analyzer turns: a turn that inspects, then
+    computes, then acts spends three of them, so 30 is closer to ten or
+    fifteen turns of real history than to thirty. The cap is independent of
+    the token budget, so with a large context window it can be the binding
+    constraint while thousands of tokens sit unused - which is what the
+    default 30 does at 98k, having been chosen when the window was 32k.
+
+    0 or negative disables the cap entirely, leaving the token budget as the
+    only limit."""
+    return _get_env_int("ARC3_HISTORY_ASSISTANT_TURNS", 30)
+_WM_NUDGE_TURNS = _get_env_int("ARC3_WM_NUDGE_TURNS", 0)
+_WM_WIPE_ON_GAME_OVER = _get_env_int("ARC3_WM_WIPE_ON_GAME_OVER", 1)
+_AUTO_FRAME_DIFF = _get_env_bool("ARC3_AUTO_FRAME_DIFF", False)
+_AUTO_FRAME_DIFF_BUDGET = _get_env_int("ARC3_AUTO_FRAME_DIFF_BUDGET", 300)
+_AUTO_FRAME_DIFF_MAX_GROUP = 40
+_DIFF_IMAGE = _get_env_bool("ARC3_DIFF_IMAGE", False)
 _RESPONSE_META_MAX_CHARS = 4000
 
 _PYTHON_TOOL_DESCRIPTION = (
     "Run one ephemeral Python snippet against preloaded ASCII game state. Available globals: "
     "`current_frame`, `previous_frame`, `history`, `transitions`, `last_transition`, "
-    "`valid_actions`, `last_action_result`, "
+    "`valid_actions`, `last_action_call_result`, "
     "and `action(actions)` for executing one or more real environment actions. "
-    "`current_frame` and each `history[*].frame` expose only `.ascii`, `.segmentation`, `.step`, and `.level`; "
+    "`current_frame` and each `history[*].frame` expose only `.ascii`, `.segmentation`, `.step`, `.level`, and `.shape` (a `(rows, cols)` tuple); "
     "`history[-1].frame` is the current post-action frame, not the previous frame. "
     "For before/after diffs, compare `previous_frame` to `current_frame` or use `last_transition.before_frame` and `.after_frame`. "
     "For MOUSE, pass `row` and `col` integer fields; legacy x/y fields are rejected. "
@@ -198,8 +341,10 @@ def _terminal_action_stop_detail(reason: str | None) -> str:
         return "No further actions were executed because the run is already complete."
     if reason == "game_over":
         return (
-            "No further actions were executed because the previous action reached GAME_OVER; "
-            "the runner will auto-reset before the next analyzer turn."
+            "No further actions were executed because the previous action reached GAME_OVER "
+            "(attempt failed; the run is NOT finished). The runner will auto-reset the level "
+            "before your next turn. Cause is either the action itself (hazard) or a depleted "
+            "step/time budget (usually a shrinking bar at the grid border)."
         )
     if reason == "level_completed":
         return (
@@ -215,6 +360,1324 @@ def _display_action_number(action_num: int) -> int:
     return max(1, int(action_num) + 1)
 
 
+def _stale_summary_lines(summary: dict[str, Any]) -> list[str]:
+    """Rendering for a step summary carried over from an earlier exchange of
+    the same turn. States that explicitly instead of re-narrating the old
+    outcome as if it had just happened. Two ways a turn can reach here:
+    a snippet ran and executed nothing (default wording), or the exchange
+    produced no tool call at all (stale_reason='no_tool_call') - in which case
+    claiming a snippet ran would be false."""
+    if summary.get("stale_reason") == "no_tool_call":
+        lines = [
+            "Your previous exchange this turn produced no tool call and "
+            "executed no actions."
+        ]
+    else:
+        lines = [
+            "Your last python snippet executed NO actions (it only computed or "
+            "inspected state - that is fine)."
+        ]
+    executed_actions = summary.get("executed_actions")
+    rendered = ""
+    if isinstance(executed_actions, list):
+        rendered = ", ".join(
+            str(name).strip() for name in executed_actions[:10] if str(name).strip()
+        )
+    if summary.get("game_over"):
+        fatal = str(summary.get("fatal_action") or "")
+        lines.append(
+            "Reminder of the still-pending outcome from earlier this turn: the "
+            f"executed sequence ({rendered or 'see earlier message'}) ended in "
+            "GAME OVER"
+            + (f" after '{fatal}'" if fatal else "")
+            + "; the level was auto-reset and you are planning the fresh attempt "
+            "now. Do NOT re-submit that sequence - the full diagnosis "
+            "instructions were given in an earlier message this turn."
+        )
+    elif summary.get("level_transition"):
+        lines.append(
+            "Reminder: the earlier sequence this turn completed the level; you "
+            "are now on the new level."
+        )
+    elif rendered:
+        lines.append(
+            "For reference, the most recent executed sequence earlier this turn "
+            f"was: {rendered}."
+        )
+    return lines
+
+
+def _state_identity_border() -> int:
+    """Border cropped when deciding whether two boards are THE SAME STATE.
+
+    Distinct from ARC3_NOOP_GUARD_BORDER, which asks a different question. The
+    batch stop asks "did this action accomplish anything meaningful", and there
+    cropping the HUD is right: a timer ticking down is not progress. The repeat
+    guards and the death ledger ask "have I been here before", and there
+    cropping is a liability - two boards with identical interiors but different
+    HUD are treated as one state, so a mechanic gated on anything the HUD
+    encodes (a countdown, a phase, a collected count) can be mis-recorded. The
+    resulting block is permanent and self-sealing: the action never executes,
+    so no evidence can arrive to correct it.
+
+    Defaults to 0 - exact whole-frame matching, which is safe by construction
+    and matches the reference implementation. Raise it to trade that guarantee
+    for more matches."""
+    return max(0, _get_env_int("ARC3_STATE_IDENTITY_BORDER", 0))
+
+
+def _interior_state_hash(grid: list[list[int]]) -> str:
+    """Hash identifying whether two boards are the same state, for the repeat
+    guards and the death ledger. Crops ARC3_STATE_IDENTITY_BORDER cells, which
+    defaults to 0 - the whole frame, HUD included. Small grids hash whole."""
+    import hashlib
+
+    border = _state_identity_border()
+    rows = len(grid)
+    cols = max((len(row) for row in grid), default=0)
+    if rows > 2 * border and cols > 2 * border:
+        core = [tuple(row[border:cols - border]) for row in grid[border:rows - border]]
+    else:
+        core = [tuple(row) for row in grid]
+    return hashlib.sha1(repr(core).encode("utf-8")).hexdigest()
+
+
+_LEAKED_TOOLCALL_RE = None
+
+
+def _extract_leaked_tool_calls(message: dict[str, Any]) -> dict[str, Any]:
+    """Fallback for servers without a tool-call parser for the model's
+    dialect: when a response carries no structured tool_calls but the content
+    contains Nemotron-style <TOOLCALL>[{...}]</TOOLCALL> markup, parse it and
+    synthesize the OpenAI-shaped tool_calls the harness expects. Malformed
+    payloads leave the message untouched. Disable with
+    ARC3_TOOLCALL_TEXT_FALLBACK=0."""
+    global _LEAKED_TOOLCALL_RE
+    if not _get_env_bool("ARC3_TOOLCALL_TEXT_FALLBACK", True):
+        return message
+    if not isinstance(message, dict) or message.get("tool_calls"):
+        return message
+    content = message.get("content")
+    if not isinstance(content, str) or "<TOOLCALL>" not in content:
+        return message
+    import json as _json
+    import re as _re
+    import uuid as _uuid
+
+    if _LEAKED_TOOLCALL_RE is None:
+        _LEAKED_TOOLCALL_RE = _re.compile(
+            r"<TOOLCALL>\s*(\[.*?\])\s*</TOOLCALL>", _re.DOTALL
+        )
+    match = _LEAKED_TOOLCALL_RE.search(content)
+    if match is None:
+        return message
+    try:
+        parsed = _json.loads(match.group(1))
+    except _json.JSONDecodeError:
+        return message
+    if not isinstance(parsed, list):
+        parsed = [parsed]
+    calls: list[dict[str, Any]] = []
+    for item in parsed:
+        if not isinstance(item, dict) or not item.get("name"):
+            continue
+        arguments = item.get("arguments", {})
+        if not isinstance(arguments, str):
+            arguments = _json.dumps(arguments, ensure_ascii=False)
+        calls.append(
+            {
+                "id": f"call_{_uuid.uuid4().hex}",
+                "type": "function",
+                "function": {
+                    "name": str(item["name"]),
+                    "arguments": arguments,
+                },
+            }
+        )
+    if not calls:
+        return message
+    remainder = (content[: match.start()] + content[match.end():]).strip()
+    updated = dict(message)
+    updated["tool_calls"] = calls
+    updated["content"] = remainder or None
+    return updated
+
+
+def _env_int(name: str, default: int) -> int:
+    """A numeric override that cannot take the run down on a typo."""
+    try:
+        return int(os.environ.get(name, "") or default)
+    except (TypeError, ValueError):
+        return default
+
+
+def _env_float(name: str, default: float) -> float:
+    try:
+        return float(os.environ.get(name, "") or default)
+    except (TypeError, ValueError):
+        return default
+
+
+_RETRYABLE_HTTP_STATUSES = {429, 500, 502, 503, 504, 520, 522, 524}
+# Establishing a TCP connection is fast or it is not happening: a server that
+# has exited refuses instantly, and one that is unreachable will not become
+# reachable inside the read budget. Separate from the read timeout so a dead
+# endpoint fails in seconds rather than waiting out the generation budget.
+_CONNECT_TIMEOUT_SECONDS = 10.0
+
+
+def _http_retry_delay_seconds(
+    attempt: int,
+    retry_after: str | None,
+    base_seconds: float,
+    max_seconds: float = 60.0,
+) -> float:
+    """Exponential backoff with jitter, or the server's Retry-After when it
+    parses as a number and asks for longer. Capped at 60s per wait.
+
+    Retry-After raises the wait, never lowers it. A provider under load can
+    answer `Retry-After: 1` to every request - observed at 100 consecutive
+    retries roughly 1.3s apart - and honouring that literally means hammering
+    an endpoint that has already said no, which neither clears the rate limit
+    nor helps the run. The configured base is the floor; the server can still
+    ask for more.
+    """
+    backoff = min(max_seconds, base_seconds * (2 ** attempt)) + random.uniform(0.0, 1.0)
+    if retry_after:
+        try:
+            # max_seconds caps the exponential, not an explicit request from the
+            # server: a Retry-After of 30 against a 5s ceiling still waits 30,
+            # because the endpoint said so and hammering it earlier helps no one
+            return max(backoff, min(60.0, float(retry_after)))
+        except ValueError:
+            pass
+    return backoff
+
+
+def _post_with_retries(
+    post_fn,
+    *,
+    retries: int,
+    base_seconds: float,
+    max_seconds: float = 60.0,
+    initial_seconds: float = 0.0,
+    sleep=time.sleep,
+    monotonic=time.monotonic,
+):
+    """Retry transient HTTP failures (rate limits, gateway errors) in place,
+    BEFORE they escalate to turn abort + history rollback. Returns the final
+    response either way; non-retryable statuses return immediately and flow
+    into the existing error handling. Rate-limit retries here preserve the
+    turn's partial tool work, which the abort path deliberately discards."""
+    # A negative count means keep trying. Zero keeps its obvious meaning - do
+    # not retry - so a typo disables retries rather than hanging the run.
+    unlimited = retries < 0
+    budget = max(0, retries)
+    # A floor on total retry time, used once. On Kaggle the benchmark is
+    # released before the server has finished loading, so the first request of
+    # every game meets a refused connection and has to wait it out - minutes,
+    # not the seconds the ordinary budget allows. Expressed as time rather than
+    # a count because the count that covers a 15-minute load would be absurd at
+    # any sensible delay, and would then apply to every later failure too.
+    started_at = monotonic()
+
+    def _budget_label(attempt_index: int) -> str:
+        """What the log should call the limit. Only reached outside a grace
+        period now - inside one the retry lines are suppressed entirely, since
+        "1/3" describes a budget that is not the one in force."""
+        if unlimited:
+            return "unlimited"
+        if attempt_index >= budget:
+            return f"{initial_seconds:.0f}s grace"
+        return str(budget)
+
+    def _exhausted(attempt_index: int) -> bool:
+        if unlimited:
+            return False
+        if attempt_index < budget:
+            return False
+        return (monotonic() - started_at) >= initial_seconds
+
+    def _announce_grace(reason: str) -> None:
+        """Said once, on the first failure, so the grace is visible from the
+        start. Without it the log shows three ordinary-looking retries before
+        the "Ns grace" label appears, and a reader watching a cold start cannot
+        tell a long wait from a broken one."""
+        nonlocal announced
+        if announced or initial_seconds <= 0:
+            return
+        announced = True
+        log.warning(
+            "first analyzer request failed (%s); retrying for up to %.0fs while "
+            "the endpoint comes up, then falling back to %d retries per request",
+            reason, initial_seconds, budget,
+        )
+
+    announced = False
+    response = None
+    attempt = -1
+    while True:
+        attempt += 1
+        try:
+            response = post_fn()
+        except requests.exceptions.RequestException as exc:
+            # The server was never reached - refused, DNS, a socket error, or a
+            # connect timeout. Without this the exception leaves the turn
+            # entirely and the analyzer-level retry starts it over, discarding
+            # whatever tool work it had already done; retrying the request in
+            # place keeps it. A ReadTimeout is deliberately not caught: the
+            # response body stalled, which has its own path that can yield with
+            # history preserved.
+            if _is_read_timeout(exc) or _exhausted(attempt):
+                raise
+            _announce_grace(type(exc).__name__)
+            delay = _http_retry_delay_seconds(attempt, None, base_seconds, max_seconds)
+            # Silent while the grace is carrying the loop: at 14 games polling a
+            # cold endpoint every 5s for 15 minutes this line alone would be
+            # ~2,500 entries, burying everything else. The announcement above
+            # already said a wait is in progress, and the outcome is logged when
+            # the loop ends either way.
+            if initial_seconds <= 0:
+                log.warning(
+                    "analyzer endpoint unreachable (%s); retry %d/%s in %.1fs",
+                    type(exc).__name__, attempt + 1, _budget_label(attempt), delay,
+                )
+            sleep(delay)
+            continue
+        status = getattr(response, "status_code", 200)
+        if status not in _RETRYABLE_HTTP_STATUSES or _exhausted(attempt):
+            return response
+        _announce_grace(f"HTTP {status}")
+        retry_after = None
+        headers = getattr(response, "headers", None)
+        if headers is not None:
+            retry_after = headers.get("Retry-After")
+        delay = _http_retry_delay_seconds(attempt, retry_after, base_seconds, max_seconds)
+        if initial_seconds <= 0:
+            log.warning(
+                "analyzer endpoint returned HTTP %s; retry %d/%s in %.1fs",
+                status, attempt + 1, _budget_label(attempt), delay,
+            )
+        sleep(delay)
+    return response
+
+
+_CONTROL_MESSAGE_KEY = "_arc3_control"
+
+_YIELD_RESUME_PROMPT_WITH_TOOLS = (
+    "You yielded control on the turn time budget. Your tool results above are "
+    "still valid - do NOT restart your analysis or re-inspect the board from "
+    "scratch. Continue from them and call `action(actions)` with your best "
+    "next action or short batch."
+)
+
+_YIELD_RESUME_PROMPT_WITH_TOOLS_NEUTRAL = (
+    "You yielded control on the turn time budget. This is a scheduling "
+    "interruption, not a signal that your analysis is complete. Your tool "
+    "results above are still valid - do NOT restart or re-inspect the board "
+    "from scratch. Pick up where you left off: keep investigating if you still "
+    "need to, and call `action(actions)` once your evidence supports a choice."
+)
+
+
+def _yield_resume_tone() -> str:
+    """Wording for the commonest resumption message.
+
+    'commit' (default) is the original text, which reads as though the model
+    has finished: it asks for the best next action. At a 60s budget that fires
+    after very little deliberation, so it can push a commitment the evidence
+    does not yet support.
+
+    'neutral' keeps the load-bearing part - do not restart, your results are
+    valid - while naming the interruption for what it is and leaving the
+    decision to continue investigating open. The anti-stall backstops are
+    untouched: the no-tool-call nudge, the tool-step-exhaustion prompt and the
+    timeout prompt all still push to act, and they fire on exhausted work
+    rather than on elapsed time.
+
+    'minimal' goes further and does not mention the yield at all. A yield that
+    executed nothing and a snippet that only inspected are the same situation
+    from the model's side - nothing happened, carry on - and the scheduling
+    reason is harness detail it cannot act on. The wording echoes the mid-turn
+    stale opener so a resumption reads as a continuation rather than as its own
+    kind of event, keeping only the part that is load-bearing: the earlier
+    results are still good.
+
+    Read the arm as actions performed against tokens generated: the change
+    should raise tokens per action if it is working as intended, and the
+    question is whether the actions get better.
+    """
+    tone = os.environ.get("ARC3_YIELD_RESUME_TONE", "").strip().lower()
+    return tone if tone in ("neutral", "minimal") else "commit"
+
+_YIELD_RESUME_PROMPT_NO_TOOLS = (
+    "You spent the entire turn budget reasoning without calling a tool. Stop "
+    "deliberating and call the `python` tool NOW: inspect "
+    "`current_frame.segmentation`, then call `action(actions)` with the best "
+    "available action. Acting on a partial model beats another turn of "
+    "analysis."
+)
+
+_YIELD_RESUME_PROMPT_WITH_TOOLS_MINIMAL = (
+    # Says nothing about yielding. From the model's side a yield with nothing
+    # executed and an inspect-only snippet are the same situation - nothing
+    # happened, carry on - and the scheduling reason is harness detail it
+    # cannot act on. Deliberately echoes the mid-turn stale opener it already
+    # sees often, so a resumption reads as a continuation rather than as a
+    # distinct kind of message. Kept: the earlier results are still good, so
+    # do not start over.
+    "Your last python snippet executed no actions (it only computed or "
+    "inspected state - that is fine). Your tool results above are still "
+    "valid: continue from them rather than re-inspecting the board from "
+    "scratch."
+)
+
+_YIELD_RESUME_PROMPT_NO_TOOLS_MINIMAL = (
+    # The with-tools wording would be false here: there was no snippet at all,
+    # so there are no results to continue from and the useful instruction is
+    # where to start instead.
+    "Your previous exchange produced no tool call and executed no actions. "
+    "Start with the `python` tool: inspect `current_frame.segmentation`, then "
+    "call `action(actions)` when your evidence supports a choice."
+)
+
+_YIELD_RESUME_PROMPT_NO_TOOLS_NEUTRAL = (
+    # The same edit as the with-tools pair: keep what is load-bearing - that
+    # nothing executed, and the concrete first step - and drop the pressure to
+    # commit. This fires when the yield budget expired on a turn that never
+    # called a tool, so the model really did spend the turn reasoning; what is
+    # unwarranted is telling it that acting on a partial model is better, when
+    # the budget expiring says nothing about whether its analysis was close.
+    "You yielded control on the turn time budget without calling a tool. This "
+    "is a scheduling interruption, not a signal that your analysis is complete "
+    "- but nothing has been executed and the board is unchanged. Start with "
+    "the `python` tool: inspect `current_frame.segmentation`, then call "
+    "`action(actions)` once your evidence supports a choice."
+)
+
+_DEAD_REASONING_STUB = "(previous attempt produced no tool call)"
+
+_DEGENERATE_CONTENT_STUB = (
+    "(previous reply was highly repetitive and has been omitted; do not "
+    "continue that pattern - write concise plain prose)"
+)
+
+_STATE_ONLY_CAPTION = (
+    # Leads with the assurance rather than with "Current board state":
+    # _build_user_message appends its own "Current grid image:" immediately
+    # below, so opening with a second caption put two labels for one picture
+    # next to each other. The load-bearing part is that the opener has not
+    # gone stale, not that a board follows.
+    "Nothing has been executed since the turn opener above, so the step, "
+    "level and valid actions stated there still hold."
+)
+
+
+def _yield_resume_prompt_mode() -> str:
+    """How a resumed turn re-opens.
+
+    'full'       (default) rebuild the complete opener - 2-3k tokens restating
+                 state the retained opener already carries.
+    'short'      a brief continuation sentence and nothing else.
+    'state_only' the continuation sentence plus a one-line caption and the
+                 CURRENT board image(s). Measurements showed 'short' holding up
+                 while images were attached and degrading without them, which
+                 is consistent with the omitted text (batch summary, state
+                 line, diffs, ledger) being what the image was covering for.
+                 This mode makes that explicit: drop the boilerplate, keep the
+                 state, in the cheapest form available.
+    """
+    mode = os.environ.get("ARC3_YIELD_RESUME_PROMPT", "").strip().lower()
+    return mode if mode in ("short", "state_only") else "full"
+
+
+_RESUME_PROMPT_TIMEOUT = (
+    "The previous request to the model timed out and was abandoned. Your tool "
+    "results above are still valid - do NOT restart your analysis. Continue "
+    "from them and call `action(actions)` with your best next action, keeping "
+    "your reasoning short so the next request completes."
+)
+
+_RESUME_PROMPT_TOOL_STEPS_EXHAUSTED = (
+    "You used every available tool step this turn without calling "
+    "`action(actions)`. Your tool results above are still valid - do NOT begin "
+    "a new investigation. Act NOW with the best option you already have."
+)
+
+
+def _yield_resume_prompt(reason: str) -> str:
+    """Continuation text for a resumed turn, chosen by why it ended.
+
+    'timeout'      - the request exceeded the analyzer read timeout: the work
+                     already done this turn is intact, the model just needs to
+                     wrap up quickly.
+    'tool_steps'   - the step budget ran out mid-investigation: results exist,
+                     the missing thing is the action.
+    'yield_tools'  - the time budget ran out with tool work done: same premise,
+                     softer framing.
+    'yield_bare'   - the turn produced only reasoning, so there is nothing
+                     above to continue from and the useful instruction is to
+                     stop deliberating.
+    """
+    if reason == "timeout":
+        return _RESUME_PROMPT_TIMEOUT
+    if reason == "tool_steps":
+        return _RESUME_PROMPT_TOOL_STEPS_EXHAUSTED
+    tone = _yield_resume_tone()
+    if reason == "yield_tools":
+        if tone == "minimal":
+            return _YIELD_RESUME_PROMPT_WITH_TOOLS_MINIMAL
+        return (
+            _YIELD_RESUME_PROMPT_WITH_TOOLS_NEUTRAL
+            if tone == "neutral"
+            else _YIELD_RESUME_PROMPT_WITH_TOOLS
+        )
+    if tone == "minimal":
+        return _YIELD_RESUME_PROMPT_NO_TOOLS_MINIMAL
+    return (
+        _YIELD_RESUME_PROMPT_NO_TOOLS_NEUTRAL
+        if tone == "neutral"
+        else _YIELD_RESUME_PROMPT_NO_TOOLS
+    )
+
+
+def _fatal_lookahead_enabled() -> bool:
+    """Block a batch that replays a recorded fatal path BEFORE spending any of
+    its actions, rather than running the harmless prefix and refusing only the
+    final move.
+
+    Default off keeps the per-position behaviour: the prefix executes and the
+    model ends up at the brink, which costs those actions but leaves it
+    somewhere it can still choose differently. With lookahead on, nothing is
+    spent and the model must resubmit a shorter batch to reach that point."""
+    return _get_env_bool("ARC3_DEATH_GUARD_LOOKAHEAD", False)
+
+
+def _yield_on_timeout_enabled() -> bool:
+    """Treat an analyzer read timeout as a yield instead of a failed turn.
+
+    The default path clears preserve_history, so the whole invocation's tool
+    calls and results are rolled back and the retry re-derives them from
+    scratch - on Kaggle that is a ~900s round trip discarded. Nothing can have
+    executed when a timeout fires (an executing dispatch breaks the loop before
+    another request is issued), so preserving the exchanges is safe: history
+    ends on a tool message and the resumed invocation appends a user message,
+    exactly the shape every yielded turn already produces.
+
+    Scoped to read timeouts only. Connection errors and HTTP failures keep the
+    rollback path, and context-overflow handling is untouched - preserving an
+    over-long context would just reproduce the failure."""
+    return _get_env_bool("ARC3_YIELD_ON_TIMEOUT", True)
+
+
+def _is_read_timeout(exc: BaseException) -> bool:
+    """requests raises ReadTimeout for a stalled response body; ConnectTimeout
+    (a sibling of the same base) means the server was never reached, so it is
+    deliberately excluded."""
+    return isinstance(exc, requests.exceptions.ReadTimeout)
+
+
+def _stub_dead_reasoning_enabled() -> bool:
+    """Replace reasoning-only assistant replies (no tool calls, no content)
+    with a short stub IN-TURN. Without this the dead trace is re-sent on every
+    later request of the same turn - observed at ~35k characters, truncated
+    mid-sentence by finish_reason=length, i.e. a broken thought replayed as
+    context."""
+    return _get_env_bool("ARC3_STUB_DEAD_REASONING", False)
+
+
+_HISTORY_IMAGE_PLACEHOLDER = "[grid image omitted from history]"
+
+
+def _history_image_keep() -> int | None:
+    """How many of the most recent image-bearing history MESSAGES keep their
+    images. None (unset) preserves today's behavior - images accumulate for as
+    long as their message is retained, ~400 tokens each at
+    MULTIMODAL_UPSCALE=10. 0 strips every image from history, so only the
+    current turn is visual. Counting per message rather than per image keeps a
+    multi-image turn (game-over visuals) together in one slot."""
+    raw = os.environ.get("ARC3_HISTORY_IMAGE_KEEP", "").strip()
+    if not raw:
+        return None
+    try:
+        return max(0, int(raw))
+    except ValueError:
+        return None
+
+
+def _resume_prompt_images_enabled() -> bool:
+    """Whether a resumption message re-attaches the grid image. A resumption
+    only happens when nothing executed, so the board is identical to the one
+    the retained opener already showed; skipping it saves an image per
+    resumption and stops resumptions from occupying history retention slots.
+    Default True = today's behavior."""
+    return _get_env_bool("ARC3_RESUME_PROMPT_IMAGES", True)
+
+
+def _message_has_image(message: dict[str, Any]) -> bool:
+    content = message.get("content")
+    if not isinstance(content, list):
+        return False
+    return any(
+        isinstance(part, dict) and part.get("type") == "image_url"
+        for part in content
+    )
+
+
+def _strip_message_images(message: dict[str, Any]) -> dict[str, Any]:
+    """Replace image parts with a short text marker. The marker keeps the
+    message well formed (a message whose only part was an image would
+    otherwise be empty) and tells the model a visual was there rather than
+    silently showing a turn that never had one."""
+    content = message.get("content")
+    if not isinstance(content, list):
+        return message
+    parts: list[Any] = []
+    for part in content:
+        if isinstance(part, dict) and part.get("type") == "image_url":
+            parts.append({"type": "text", "text": _HISTORY_IMAGE_PLACEHOLDER})
+        else:
+            parts.append(part)
+    return {**message, "content": parts}
+
+
+def _apply_history_image_window(
+    messages: list[dict[str, Any]], keep: int
+) -> list[dict[str, Any]]:
+    """Keep images on the `keep` most recent image-bearing messages, strip the
+    rest. Walks backwards so recency wins; originals are never mutated."""
+    seen = 0
+    result: list[dict[str, Any]] = []
+    for message in reversed(messages):
+        if _message_has_image(message):
+            seen += 1
+            if seen > keep:
+                message = _strip_message_images(message)
+        result.append(message)
+    result.reverse()
+    return result
+
+
+# Tag for the pair of messages a rolling summary is made of: the request that
+# asks for it and the reply that is it. Deliberately absent from
+# _CONTROL_KIND_ENV, so no pruning knob can remove them - the trimmer's
+# summary-aware drain navigates by these, and losing one would silently return
+# it to ordinary block dropping.
+_SUMMARY_CONTROL_KIND = "summary"
+# Below this a reply that also called a tool is a preamble rather than a
+# summary. Real ones measured 4,385 to 6,248 characters; an introduction to a
+# tool call measured nine.
+_SUMMARY_MIN_USABLE_CHARS = 500
+
+
+def _summary_interval_tokens() -> int:
+    """New context tokens between summary attempts, successful or not. 0 disables.
+
+    Count newly appended prompts, assistant messages (including reasoning), tool
+    results and images, excluding the summary exchange itself. Re-sending or
+    evicting existing history does not change this counter.
+
+    The model is asked for a summary on a schedule rather than when it feels
+    like one. Measured across a run where the prompt asked for a note every
+    turn: one game in fourteen complied consistently, and the rest wrote
+    something on roughly half their replies - so a mechanism that depends on
+    the model choosing to summarise does not get summaries.
+
+    Each summary is generated from live context, which keeps both the KV prefix
+    and the recurrent state intact - a summary built from a truncated prefix
+    would force a full prefill, and recurrent state cannot be rewound the way a
+    KV prefix can be reused.
+
+    The summary is appended when it is made, so every later turn is generated
+    with it already present. Nothing is ever inserted underneath a message that
+    was produced without it.
+    """
+    return max(0, _get_env_int("ARC3_SUMMARY_INTERVAL_TOKENS", 0))
+
+
+def _drain_stop_at_summaries() -> bool:
+    """Whether the drain may stop early on a summary.
+
+    The drain normally evicts from the budget down to budget-minus-drain in one
+    go. With this on it stops sooner when the head is already a summary request
+    and the request is back under budget - so the retained history starts with a
+    digest of everything dropped, without evicting a token more than the
+    ordinary drain would.
+
+    The earlier design made summaries the ONLY boundary, which could not
+    overshoot in the good case but removed two intervals at once whenever a
+    summary had failed - 38% of a 42k window at an 8k interval. Here the drain
+    target still bounds the drop, so the worst case is exactly today's
+    behaviour and the summary is an early exit rather than the sole exit.
+
+    The trade is coverage: when no summary falls inside the drain range the drop
+    stops mid-span with the previous summary already gone, and that stretch has
+    nothing standing in for it. So ARC3_SUMMARY_INTERVAL_TOKENS wants to be
+    comfortably SMALLER than ARC3_CONTEXT_DRAIN_TOKENS - several candidates per
+    drop - which is the opposite of sizing the interval to replace the drain.
+    """
+    return _get_env_bool("ARC3_DRAIN_STOP_AT_SUMMARIES", False)
+
+
+def _hide_followup_summaries() -> bool:
+    """Send only the first summary in history, hiding the rest.
+
+    Each summary covers everything before it, so a later one restates an
+    earlier one's material - measured at 1 or 2 carried per request, 4,496
+    characters each, so roughly 6% of a 42k window at worst.
+
+    Applied to the wire and to the estimate, never to the stored history: the
+    hidden ones stay so that when eviction removes the first, the next becomes
+    the first and appears. Dropping them at commit would delete what it meant
+    to hide.
+
+    The cost is the thing the rolling design otherwise avoids. A hidden summary
+    is absent while the turns after it are generated and becomes their head
+    later, so those messages end up under something that was not there when
+    they were written - which is exactly the rearrangement the append-only
+    scheme was built to prevent.
+    """
+    return _get_env_bool("ARC3_HIDE_FOLLOWUP_SUMMARIES", False)
+
+
+def _apply_summary_visibility(
+    messages: list[dict[str, Any]], *, evicted: bool
+) -> list[dict[str, Any]]:
+    """Hide completed summaries, preserving a pending final summary request.
+
+    Before the first eviction no completed summaries are shown. What they
+    cover is still in context verbatim, so a summary there is pure duplication.
+    Hiding them all means appending one does not change the prompt, leaving the fill
+    phase a perfect append with a prefix that never moves. The first summary
+    appears when eviction starts, which invalidates the prefix anyway.
+    """
+    if not _hide_followup_summaries():
+        return messages
+
+    # The request currently asking for a summary has no reply yet. It must
+    # reach the model even when completed summaries are hidden. Split it off
+    # without mutating the input; only stored pairs participate in visibility.
+    pending_request: list[dict[str, Any]] = []
+    history = messages
+    if (
+        messages
+        and _message_is_summary(messages[-1])
+        and str(messages[-1].get("role", "")).strip() == "user"
+    ):
+        pending_request = messages[-1:]
+        history = messages[:-1]
+    if not evicted:
+        return [m for m in history if not _message_is_summary(m)] + pending_request
+    kept: list[dict[str, Any]] = []
+    seen_pair = False
+    in_first_pair = False
+    for message in history:
+        if not _message_is_summary(message):
+            kept.append(message)
+            continue
+        is_request = str(message.get("role", "")).strip() == "user"
+        if is_request:
+            # the request opens a pair; the first one and its reply stay
+            in_first_pair = not seen_pair
+            seen_pair = True
+        if in_first_pair:
+            kept.append(message)
+    return kept + pending_request
+
+
+def _summary_replaces_history() -> bool:
+    """Whether a successful summary discards the history it summarised.
+
+    The opposite end of the design from drain sizing: instead of evicting a
+    fixed span and hoping a summary falls inside it, the summary IS the
+    history. Everything above it goes the moment it lands, leaving system
+    prompt, request, summary, and whatever the turn appends after.
+
+    Safe in the sense that matters - nothing is dropped unless a usable summary
+    exists, so a failed or rejected one leaves ordinary eviction in charge and
+    the next turn tries again. But it is the more aggressive option, not the
+    safer one: a large drain still leaves a turn or two of verbatim history,
+    where this leaves none.
+
+    Worth trying because the summaries turn out to be good. One measured at
+    7,107 characters carried the board layout, the movement lattice, which
+    actions did nothing and why, an unresolved two-branch interpretation of the
+    last result, and the next probes to run - which is most of what the evicted
+    turns were holding. And the raw frames are not in the history anyway:
+    current_frame, previous_frame, history and frame_diff are runtime globals
+    rebuilt every turn, so what eviction destroys is the interpretation, which
+    is precisely what the summary keeps.
+
+    Costs: the prefix is invalidated on every summary rather than on every
+    eviction, and the turn immediately after has no verbatim record of the turn
+    before it beyond what the opener carries.
+    """
+    return _get_env_bool("ARC3_SUMMARY_REPLACES_HISTORY", False)
+
+
+def _summary_turn_context() -> bool:
+    """Whether the summary request is told what just happened.
+
+    The request goes out at turn start, BEFORE this turn's opener exists, so
+    the newest thing in context is the previous turn's tool results. A level-up
+    is visible there only as a `level_completed` flag inside a JSON payload -
+    so a summary written at that moment can describe the level just finished as
+    if it were still current, which is exactly the material a later turn needs
+    to be right about.
+
+    What goes in: what executed, whether the board changed, why a sequence
+    stopped early, whether the level changed, and the step and level now.
+
+    What stays out: the valid actions, which are state for the next decision
+    and go stale the moment the summary outlives the turn; the tool inventory
+    and the world-model demands, which are instructions that compete with
+    "write the summary"; and the animation line, which is a pointer to go and
+    read an accessor that will describe a different action by the time anyone
+    reads the summary.
+    """
+    return _get_env_bool("ARC3_SUMMARY_TURN_CONTEXT", False)
+
+
+def _summary_enable_thinking() -> bool:
+    """Whether the summary request lets the model think first.
+
+    OFF by default. Thinking about a summary is not obviously worth it - the
+    material is all in front of the model - and it is expensive twice over:
+    observed at 18,303 characters of reasoning that reached the output limit
+    with no summary at all, and it doubles the generation for a request that
+    already runs once per interval.
+
+    The template must be able to suppress it without moving the prefix. In the
+    Qwen3-family templates the `enable_thinking` conditional wraps BOTH the
+    empty think block at the tail AND the "Reasoning effort is set to ..."
+    sentence at the top of the system message, so the stock template diverges
+    from the first tokens and every summary prefills the whole context -
+    measured as prefix reuse falling from 75% to 45%. Forcing that conditional
+    true (`{%- if true %}`) keeps the sentence unconditional and leaves only
+    the tail block varying, which restores a byte-identical prefix. Check that
+    before assuming this default is free.
+    """
+    return _get_env_bool("ARC3_SUMMARY_ENABLE_THINKING", False)
+
+
+def _summary_max_gen_tokens(default: int | None) -> int:
+    """Output limit: the normal cap, or 8192 when normal output is unlimited.
+
+    The summary is asked for everything worth keeping, so it legitimately wants
+    more room than a turn reply - and it is one request per interval, so a
+    larger ceiling costs little. Raising this keeps the thinking;
+    ARC3_SUMMARY_ENABLE_THINKING=0 removes it.
+    """
+    fallback = default if default is not None and default > 0 else 8192
+    return max(1, _get_env_int("ARC3_SUMMARY_MAX_GEN", fallback))
+
+
+def _message_is_summary(message: dict[str, Any]) -> bool:
+    return message.get(_CONTROL_MESSAGE_KEY) == _SUMMARY_CONTROL_KIND
+
+
+_CONTROL_KIND_ENV = {
+    "nudge": "ARC3_PRUNE_NUDGES",
+    "resume": "ARC3_PRUNE_RESUME_PROMPTS",
+    "stub": "ARC3_PRUNE_DEAD_STUBS",
+}
+
+
+def _prune_control_context_enabled() -> bool:
+    """Master switch: drop every kind of tagged control message when
+    committing turn messages to persistent history. The per-kind knobs
+    (ARC3_PRUNE_NUDGES / _RESUME_PROMPTS / _DEAD_STUBS) turn individual kinds
+    on without it, so an arm can attribute each effect separately."""
+    return _get_env_bool("ARC3_PRUNE_CONTROL_CONTEXT", False)
+
+
+def _prune_control_kind_enabled(kind: str) -> bool:
+    """A kind is pruned if the master switch is on OR its own knob is set.
+
+    - 'nudge'  : the "you have not acted yet" corrective (a few hundred tokens)
+    - 'resume' : the message appended when re-entering a non-executing turn
+                 (a full opener in `full` mode, the short continuation in
+                 `short` mode) - a restatement of unchanged state
+    - 'stub'   : the placeholder that ARC3_STUB_DEAD_REASONING leaves behind
+    """
+    if _prune_control_context_enabled():
+        return True
+    env_name = _CONTROL_KIND_ENV.get(kind)
+    return bool(env_name) and _get_env_bool(env_name, False)
+
+
+def _any_control_pruning_enabled() -> bool:
+    return _prune_control_context_enabled() or any(
+        _get_env_bool(name, False) for name in _CONTROL_KIND_ENV.values()
+    )
+
+
+def _mark_control_message(message: dict[str, Any], kind: str = "resume") -> dict[str, Any]:
+    message[_CONTROL_MESSAGE_KEY] = kind
+    return message
+
+
+def _strip_control_keys(messages: list[dict[str, Any]]) -> list[dict[str, Any]]:
+    """Wire-safe copy: the private tag never leaves the process."""
+    cleaned: list[dict[str, Any]] = []
+    for message in messages:
+        if _CONTROL_MESSAGE_KEY in message:
+            copy = dict(message)
+            copy.pop(_CONTROL_MESSAGE_KEY, None)
+            cleaned.append(copy)
+        else:
+            cleaned.append(message)
+    return cleaned
+
+
+def _prune_control_messages(messages: list[dict[str, Any]]) -> list[dict[str, Any]]:
+    """Remove tagged control messages, then strip trailing user messages.
+
+    The trailing-user strip is alternation repair, not a preference: pruning a
+    spin-out turn (opener + dead reasoning, no tool calls) would otherwise
+    leave the opener dangling while the next invocation appends another user
+    message. A turn with no tool calls executed nothing, so the next opener
+    regenerates identical state."""
+    kept = [
+        m
+        for m in messages
+        if not (
+            m.get(_CONTROL_MESSAGE_KEY)
+            and _prune_control_kind_enabled(str(m.get(_CONTROL_MESSAGE_KEY)))
+        )
+    ]
+    while kept and str(kept[-1].get("role", "")).strip() == "user":
+        kept.pop()
+    return kept
+
+
+def _reasoning_history_keys() -> tuple[str, ...]:
+    """Message key(s) under which prior reasoning is sent back to the server.
+
+    Backends disagree: OpenRouter returns and reads `reasoning`, while
+    llama.cpp returns `reasoning_content` and its chat template reads ONLY
+    that - so reasoning stored under `reasoning` is silently discarded there
+    (measured: 69 prompt tokens vs 3670 for the same text under the two keys).
+    A dropped reasoning block is invisible except as prompt growth that falls
+    short of the tokens generated.
+
+    Default `reasoning` preserves existing behaviour. Set to
+    `reasoning_content` for llama.cpp, or to a comma list to send both - which
+    is backend-agnostic but risks a template that renders both keys duplicating
+    the text."""
+    raw = os.environ.get("ARC3_REASONING_HISTORY_KEY", "").strip()
+    keys = tuple(k.strip() for k in raw.split(",") if k.strip())
+    return keys or ("reasoning",)
+
+
+def _level_inventory_enabled() -> bool:
+    """The START-vs-START object inventory shown on the first turn of a new
+    level.
+
+    It has its own switch because it answers a different question from the
+    turn-to-turn diff: not "what did my last actions do" but "what is new
+    about this level", with `appeared` types flagged as likely new mechanics.
+    Previously it rode on the auto-diff machinery and was suppressed only when
+    BOTH ARC3_AUTO_FRAME_DIFF and ARC3_DIFF_IMAGE were off - a coupling that
+    was invisible from the knob names."""
+    return _get_env_bool("ARC3_LEVEL_INVENTORY", False)
+
+
+def _resume_diff_image_enabled() -> bool:
+    """Re-attach the current turn's diff image to resumption messages.
+
+    The diff image is produced only while building a full turn opener, so a
+    resumption (short/state_only) carries the current grid but no diff. That
+    is normally harmless - a resumption happens when nothing executed, so the
+    diff would be identical to the one the turn's original opener already
+    showed. It stops being harmless when ARC3_HISTORY_IMAGE_KEEP has stripped
+    that opener's images: the model has then lost the diff entirely, and the
+    resumption is its only chance to see what changed.
+
+    Costs roughly one image per resumption. Off by default."""
+    return _get_env_bool("ARC3_RESUME_DIFF_IMAGE", False)
+
+
+def _reasoning_effort_ladder() -> list[str]:
+    """Reduced reasoning-effort levels to step down through after a truncated
+    generation, e.g. "medium,low".
+
+    A reply cut off at the output ceiling is wasted twice over: the tokens are
+    spent, and if the cut lands inside a tool call the turn produces nothing
+    callable. Rather than accommodating long generations with a bigger ceiling,
+    this asks the model for less deliberation until it completes a turn -
+    stepping one rung down per truncation and resetting once an action
+    executes, which is the evidence that the turn recovered.
+
+    Empty (default) disables the behaviour entirely."""
+    raw = os.environ.get("ARC3_REASONING_EFFORT_LADDER", "").strip()
+    return [level.strip() for level in raw.split(",") if level.strip()]
+
+
+def _preserve_thinking_kwarg() -> bool | None:
+    """Value for the `preserve_thinking` chat-template kwarg, or None to omit.
+
+    The payload builder sets chat_template_kwargs wholesale for vLLM, which
+    replaces rather than merges - so a server-side
+    --default-chat-template-kwargs carrying preserve_thinking can be silently
+    dropped by a request. Defaults to on so the request asserts it rather than
+    depending on server-side merge semantics; set to an empty string to omit
+    the key entirely and leave the server default in charge."""
+    raw = os.environ.get("ARC3_PRESERVE_THINKING", "1").strip().lower()
+    if not raw:
+        return None
+    return raw in ("1", "true", "yes", "on")
+
+
+def _openrouter_provider_prefs(provider: str) -> dict[str, Any] | None:
+    """OpenRouter request-level provider pinning. ARC3_OPENROUTER_PROVIDER
+    holds a provider name (or a comma-separated preference order); routing is
+    then restricted to that list with fallbacks disabled, so every request in
+    a run is served by a known engine. Under :nitro-style routing, providers
+    differ in media handling (some silently drop video/image parts),
+    quantization, and sampling behavior - a silent confound for any controlled
+    comparison. Unset or non-OpenRouter: no change to the request."""
+    if str(provider or "").strip().lower() != "openrouter":
+        return None
+    raw = os.environ.get("ARC3_OPENROUTER_PROVIDER", "").strip()
+    if not raw:
+        return None
+    order = [name.strip() for name in raw.split(",") if name.strip()]
+    if not order:
+        return None
+    return {"order": order, "allow_fallbacks": False}
+
+
+_SLIM_MEMORY_SECTIONS = ("World model", "Plan")
+_MEDIUM_MEMORY_SECTIONS = ("World model", "Plan", "Action model")
+
+_MEMORY_SECTION_GUIDANCE_FULL = (
+                "If you include assistant text before a tool call, use it to update your persistent memory sections. Section meanings: `World model:` = durable RULES of the environment that will still be true many actions from now (mechanics, what objects/colors mean, movement rules, win conditions) — never current positions, events, or what just happened. `Goal model:` = what winning requires. `Action model:` = what each action does mechanically. `Recent findings:` = this turn's observations and events (positions, action results, surprises). `Open questions:` = unresolved hypotheses. `Plan:` = intended next steps. `Cross-level notes:` = knowledge that carries across levels. Example — GOOD: `World model: Red cells are walls; each move shifts the player 2 cells; yellow keys open doors of the same color.` BAD: `World model: Batch stopped after 2 DOWNs, player now at (4,2).` (that belongs in `Recent findings:`). IMPORTANT: each labeled section you write REPLACES the stored version shown under 'Working world model carried from earlier turns' — write the COMPLETE updated section; any fact you omit is permanently lost."
+)
+
+_MEMORY_SECTION_MEANINGS = {
+    "World model": (
+        "durable RULES of the environment that will still be true many actions from now "
+        "(mechanics, what objects/colors mean, movement rules, win conditions) - never "
+        "current positions or what just happened"
+    ),
+    "Goal model": "what winning requires",
+    "Action model": "what each action does mechanically",
+    "Recent findings": "this turn's observations",
+    "Open questions": "unresolved hypotheses",
+    "Plan": "intended next steps",
+    "Cross-level notes": "knowledge that carries across levels",
+}
+
+
+def _advertised_section_labels() -> tuple[str, ...]:
+    """The sections the model should be TOLD about: exactly those the current
+    mode carries.
+
+    Advertising all seven while a mode drops five invites the model to write
+    into sections it will never see again - the write is parsed, stored, and
+    silently discarded from the prompt. It also confounds a slim/medium arm,
+    because a poor result could mean either that fewer sections hurt or that
+    output was wasted on sections that went nowhere."""
+    labels = _memory_section_labels()
+    if labels is None:
+        return tuple(_MEMORY_SECTION_MEANINGS)
+    # canonical order, so the advertisement matches the order the carried block
+    # renders them in
+    chosen = set(labels)
+    return tuple(name for name in _MEMORY_SECTION_MEANINGS if name in chosen)
+
+
+def _section_list_phrase(labels: tuple[str, ...], conjunction: str = "and") -> str:
+    """Render the labels as prose. The conjunction is a parameter because the
+    two call sites word it differently: the opener lists what is available
+    ("and"), the no-tool-call followup offers alternatives ("or")."""
+    quoted = [f"`{name}:`" for name in labels]
+    if len(quoted) == 1:
+        return quoted[0]
+    if len(quoted) == 2:
+        # no serial comma before a two-item conjunction
+        return f" {conjunction} ".join(quoted)
+    return ", ".join(quoted[:-1]) + f", {conjunction} " + quoted[-1]
+
+
+def _memory_section_guidance_upstream() -> str:
+    return (
+        "If you include assistant text before a tool call, keep it short and use it to "
+        "update the world model. Helpful optional prefixes are "
+        f"{_section_list_phrase(_advertised_section_labels())}."
+    )
+
+
+def _memory_section_guidance_empty() -> str:
+    labels = _advertised_section_labels()
+    meanings = " ".join(
+        f"`{name}:` = {_MEMORY_SECTION_MEANINGS[name]}." for name in labels
+    )
+    return (
+        "You have no stored memory sections yet. If you include assistant text before a "
+        f"tool call, use it to start them: {meanings}"
+    )
+
+
+_MEMORY_SECTION_GUIDANCE_UPSTREAM = (
+    "If you include assistant text before a tool call, keep it short and use it to update "
+    "the world model. Helpful optional prefixes are `World model:`, `Goal model:`, "
+    "`Action model:`, `Recent findings:`, `Open questions:`, `Plan:`, and "
+    "`Cross-level notes:`."
+)
+
+
+def _wm_section_explanation_mode() -> str:
+    """Which wording explains the memory sections in the turn opener.
+
+    'tufa' (also spelled 'upstream', the default) is the terse original: one
+    sentence naming the prefixes, framed
+    as optional. 'full' (default) is ours - a glossary of what each section
+    means, a GOOD/BAD example, and the warning that a written section REPLACES
+    the stored one so an omitted fact is lost. Roughly 1100 characters against
+    250, every turn, and it asks for complete rewrites rather than short
+    updates - so it is a candidate for longer generations as well as better
+    memory."""
+    mode = os.environ.get("ARC3_WM_SECTION_EXPLANATION", "tufa").strip().lower()
+    return "full" if mode == "full" else "upstream"
+
+
+_MEMORY_SECTION_GUIDANCE_EMPTY = (
+    "You have no stored memory sections yet. If you include assistant text before a tool "
+    "call, use it to start them: `World model:` = durable RULES of the environment that will "
+    "still be true many actions from now (mechanics, what objects/colors mean, movement "
+    "rules, win conditions) - never current positions or what just happened. `Goal model:` = "
+    "what winning requires. `Action model:` = what each action does mechanically. `Recent "
+    "findings:` = this turn's observations. `Open questions:` = unresolved hypotheses. "
+    "`Plan:` = intended next steps. `Cross-level notes:` = knowledge that carries across "
+    "levels."
+)
+
+_MEMORY_SECTION_GUIDANCE_SLIM = (
+                "If you include assistant text before a tool call, use it to update your persistent memory sections. Section meanings: `World model:` = durable RULES of the environment that will still be true many actions from now (mechanics, what objects/colors mean, movement rules, win conditions) \u2014 never current positions, events, or what just happened. `Plan:` = intended next steps. Example \u2014 GOOD: `World model: Red cells are walls; each move shifts the player 2 cells; yellow keys open doors of the same color.` BAD: `World model: Batch stopped after 2 DOWNs, player now at (4,2).` (that is a passing event, not a rule). IMPORTANT: each labeled section you write REPLACES the stored version shown under 'Working world model carried from earlier turns' \u2014 write the COMPLETE updated section; any fact you omit is permanently lost."
+)
+
+
+def _wm_revise_nudges_enabled() -> bool:
+    """The sticky 'LEVEL CHANGED / ATTEMPT RESET - revise your sections' hint.
+
+    Built for an observed failure (the champion drifting without revising),
+    but a later model maintained its sections diligently with no prompting at
+    all, and the mechanism has needed two corrections since. Turning it off
+    changes ONLY the pressure: sections are still parsed, stored and
+    re-injected, so a null result means the nagging was not earning its
+    prompt weight."""
+    return not _memory_sections_disabled() and _get_env_bool(
+        "ARC3_WM_REVISE_NUDGES", False
+    )
+
+
+def _wm_rebuild_nudge_enabled() -> bool:
+    """The post-wipe "Rebuild them NOW: write fresh World model:, Goal model:,
+    Action model:" demand, which repeats every turn until the model complies.
+
+    An unmodified run wipes the sections silently - the carried block simply
+    vanishes and reappears when the model next writes something. This nudge
+    applies its strongest pressure exactly when the model has least to say: it
+    has just lost its notes and has not yet explored the new level. Default off
+    for that reason; the wipe itself is unaffected."""
+    return not _memory_sections_disabled() and _get_env_bool(
+        "ARC3_WM_REBUILD_NUDGE", False
+    )
+
+
+def _memory_sections_disabled() -> bool:
+    """ARC3_MEMORY_SECTIONS=off removes the persistent-memory mechanic entirely:
+    no carried block, no section guidance, no nudges, no parsing, and the
+    world-model lines drop out of the system prompt too. The model is left to
+    reason from history alone, which is what an unmodified run does before any
+    of this machinery existed."""
+    return os.environ.get("ARC3_MEMORY_SECTIONS", "").strip().lower() == "off"
+
+
+def _memory_section_labels() -> tuple[str, ...] | None:
+    """Which memory sections are advertised and carried, or None for all.
+
+    'slim' keeps World model and Plan - the two the model actually maintains.
+    Measured over 25 games: World model 351 writes, Plan 255, Open questions
+    25, Recent findings 11, Action model 10, Goal model 9, Cross-level notes
+    ZERO. The dropped sections are also the stale ones: a section written once
+    and never revised keeps being re-injected as current, which was observed
+    holding an answered question open for a whole game.
+
+    'medium' additionally keeps Action model. It is written in a third of
+    games and describes MECHANICS rather than transient state, so it ages far
+    better than `Recent findings:` or `Open questions:` - on a game whose
+    difficulty is working out what each control does, it is the section least
+    worth dropping."""
+    mode = os.environ.get("ARC3_MEMORY_SECTIONS", "").strip().lower()
+    if mode == "slim":
+        return _SLIM_MEMORY_SECTIONS
+    if mode == "medium":
+        return _MEDIUM_MEMORY_SECTIONS
+    return None
+
+
+def _message_display_text(message: dict[str, Any]) -> str:
+    """Transcript rendering of a built user message. Content-parts messages
+    carry their text in parts; images are noted rather than dumped."""
+    content = message.get("content")
+    if isinstance(content, str):
+        return content
+    if not isinstance(content, list):
+        return ""
+    pieces: list[str] = []
+    for part in content:
+        if not isinstance(part, dict):
+            continue
+        if part.get("type") == "image_url":
+            pieces.append("[image attached]")
+        else:
+            text = str(part.get("text") or "").strip()
+            if text:
+                pieces.append(text)
+    return "\n\n".join(pieces)
+
+
+def _wm_revise_hint(reason: str, already_shown: bool) -> str:
+    """Text for the sticky revise-nudge. Full event text on first showing;
+    accurate reminder wording on repeats (the event is no longer 'just now',
+    and the adjacent step summary may correctly say the level is unchanged)."""
+    if reason == "level":
+        if not already_shown:
+            return (
+                "LEVEL CHANGED - your memory sections were KEPT, not wiped. Learned "
+                "mechanics, action semantics, and the general goal usually carry over "
+                "to the new level; the map/layout is new, and levels often introduce "
+                "new elements or mechanics. On this first turn, scan the grid for "
+                "unfamiliar elements (compare objects against your World/Action "
+                "model), then revise your `World model:` and `Plan:` sections - "
+                "update the map, keep what still holds, and note anything new."
+            )
+        return (
+            "Reminder: a level change occurred earlier and you have STILL not "
+            "revised your `World model:` or `Plan:` sections since. Do it now - "
+            "update the map for this level, keep what still holds, and note "
+            "anything new."
+        )
+    if reason == "death":
+        if not already_shown:
+            return (
+                "ATTEMPT RESET - your memory sections were KEPT, not wiped. The "
+                "level is unchanged, so your map and mechanics knowledge still "
+                "hold. Amend your sections NOW with what this death taught you "
+                "(the fatal action or sequence and its cause), and revise your "
+                "`Plan:` to avoid repeating it."
+            )
+        return (
+            "Reminder: this attempt was reset by an earlier death and you have "
+            "STILL not amended your memory sections with what it taught you. Do "
+            "it now - record the fatal action or sequence and its cause, and "
+            "revise your `Plan:` to avoid repeating it."
+        )
+    return ""
+
+
+def _suppress_board_dumps(text: str) -> tuple[str, int]:
+    """Collapse runs of full-width board rows in tool output. Enforces the
+    existing 'never print full boards' policy mechanically for models that
+    ignore it: keeps the first rows of each run as orientation, replaces the
+    rest with a stub, and appends a compute-don't-count nudge. Small crops
+    (fewer than 6 rows) pass through untouched. ARC3_BOARD_DUMP_GUARD=0
+    disables."""
+    if not _get_env_bool("ARC3_BOARD_DUMP_GUARD", False):
+        return text, 0
+
+    def _board_like(line: str) -> bool:
+        stripped = line.strip()
+        return (
+            len(stripped) >= 24
+            and stripped.isalnum()
+            and len(set(stripped)) <= 10
+        )
+
+    lines = text.split("\n")
+    out: list[str] = []
+    suppressed = 0
+    i = 0
+    while i < len(lines):
+        if _board_like(lines[i]):
+            j = i
+            while j < len(lines) and _board_like(lines[j]):
+                j += 1
+            run = j - i
+            if run >= 6:
+                out.extend(lines[i:i + 2])
+                out.append(f"[... {run - 2} full-width board rows suppressed ...]")
+                suppressed += run - 2
+            else:
+                out.extend(lines[i:j])
+            i = j
+        else:
+            out.append(lines[i])
+            i += 1
+    if suppressed:
+        out.append(
+            "[BOARD PRINT GUARD] Printing full boards is disallowed and "
+            "character counting is unreliable for you. Compute over the grid "
+            "in python (segmentation nodes, slicing, run-length checks, BFS) "
+            "and print only coordinates, counts, and small crops."
+        )
+    return "\n".join(out), suppressed
+
+
+def _memory_section_max_chars() -> int:
+    """Cap on a single stored memory section, in characters.
+
+    These sections are re-injected into EVERY later turn opener, so an
+    oversized one is paid for repeatedly and grows the prompt without bound.
+    The extraction path used to store them uncapped, which let a degenerate
+    75k-character `World model:` into every subsequent prompt until the
+    request exceeded the model's context entirely. Legitimate sections run a
+    few hundred characters, so the default leaves ample room while making the
+    runaway case impossible. 0 disables the cap (the old behaviour)."""
+    return max(0, _get_env_int("ARC3_MEMORY_SECTION_MAX_CHARS", 8000))
+
+
+def _degeneracy_ratio_threshold() -> float:
+    """Compression ratio below which text is treated as degenerate.
+
+    Measured on representative text: varied prose 0.20-0.28, a coordinate
+    list 0.28, an ASCII board map 0.016, one sentence repeated 300 times
+    0.008, the field mojibake spiral 0.0016. The default sits below the board
+    map so a pasted grid is not falsely rejected, while still catching genuine
+    repetition loops by a wide margin. Rejecting on compressibility catches
+    those generally rather than one encoding artifact, and does not penalise
+    legitimate unicode. 0 disables the check."""
+    try:
+        return max(0.0, float(os.environ.get("ARC3_DEGENERACY_RATIO", "0.01")))
+    except ValueError:
+        return 0.01
+
+
+def _degeneracy_min_chars() -> int:
+    """Length floor for the degeneracy check: short strings compress badly for
+    uninteresting reasons, so testing them would produce false positives."""
+    return max(1, _get_env_int("ARC3_DEGENERACY_MIN_CHARS", 500))
+
+
+def _is_degenerate_text(value: str) -> bool:
+    threshold = _degeneracy_ratio_threshold()
+    if threshold <= 0:
+        return False
+    text = str(value or "")
+    if len(text) < _degeneracy_min_chars():
+        return False
+    try:
+        import zlib
+
+        encoded = text.encode("utf-8", "replace")
+        return len(zlib.compress(encoded, 6)) / max(1, len(encoded)) < threshold
+    except Exception:
+        return False
+
+
 def _normalize_summary_text(value: Any, *, max_chars: int | None = 280) -> str:
     text = " ".join(str(value or "").split())
     if max_chars is None or max_chars <= 0 or len(text) <= max_chars:
@@ -223,9 +1686,100 @@ def _normalize_summary_text(value: Any, *, max_chars: int | None = 280) -> str:
     return f"{text[:max_chars].rstrip()}... [{omitted} chars omitted]"
 
 
+_HEADER_PARENTHETICAL_RE = re.compile(r"\([^)]*\)")
+# Only SHORT decorations are stripped. A long parenthetical or emphasised aside
+# means the line is a sentence rather than a header, and removing it could turn
+# prose into a spurious match - so the header is left intact and simply fails to
+# match, which loses the section rather than mis-capturing it.
+_HEADER_DECORATION_MAX_CHARS = 30
+
+
+def _normalize_section_header(head: str) -> str:
+    """Reduce a candidate section header to its bare label.
+
+    Models decorate these constantly. Observed in a single game: `Revised
+    world model`, `World model update`, `World model updates`, `World model
+    revision`, `Updated world model`, `World model (revised)`, `World model
+    (L4)`, `Action model (from run 1)`, `Plan (19 moves)`, and `Revised world
+    model - **important finding**`. Under exact matching, thirteen of
+    twenty-seven memory writes in that game were silently discarded, including
+    the one that correctly derived the level's kill rule.
+    """
+    def drop_short(match: "re.Match[str]") -> str:
+        return " " if len(match.group(0)) <= _HEADER_DECORATION_MAX_CHARS else match.group(0)
+
+    text = _HEADER_PARENTHETICAL_RE.sub(drop_short, head)
+    # an emphasised trailing aside is dropped on the same terms; the em-dash is
+    # deliberately NOT a cut point - it would discard text AFTER the label, which
+    # can change what the header means, and it earned nothing in practice
+    if "**" in text:
+        head_part, _, rest = text.partition("**")
+        if len(rest) <= _HEADER_DECORATION_MAX_CHARS:
+            text = head_part
+    text = text.replace("*", " ").replace("_", " ").replace("#", " ")
+    # dangling dashes left behind by a removed decoration are punctuation, not
+    # content: stripping them discards nothing, unlike cutting AT a dash
+    return " ".join(text.split()).strip(" \u2014\u2013-")
+
+
+def _tolerant_section_headers_enabled() -> bool:
+    """Accept decorated section headers (`Updated world model:`, `Plan (19
+    moves):`) rather than only the exact label.
+
+    Doubled memory capture on a measured game (12 sections -> 24), which also
+    doubles the carried world-model block the model is asked to rewrite in
+    full each turn - so it is a candidate for longer generations, not only for
+    better memory. Off restores exact matching."""
+    return _get_env_bool("ARC3_TOLERANT_SECTION_HEADERS", False)
+
+
+def _match_section_label(head: str, labels: list[str]) -> str | None:
+    """The canonical label a header refers to, or None.
+
+    Accepts the bare label, or the label with at most ONE extra word before or
+    after it. One word rather than more is deliberate: it covers every
+    decoration seen in practice while keeping prose from matching. Labels are
+    tried longest-first so `Cross-level notes` is not shadowed by `Plan`."""
+    tolerant = _tolerant_section_headers_enabled()
+    # with the knob off, compare the RAW head: normalising would still strip
+    # parentheticals and emphasis, which is itself part of the tolerance
+    normalized = (
+        _normalize_section_header(head).lower() if tolerant else " ".join(head.split()).lower()
+    )
+    if not normalized:
+        return None
+    if tolerant:
+        # A trailing parenthetical is decoration, not part of the label. The
+        # model versions and annotates its own sections - "World model
+        # (revised)", "(major revision)", "World model v34 (level 3)" - and
+        # those are the same section wearing a hat. Measured on one run: the
+        # one-extra-word rule alone caught 89% of section writes, stripping a
+        # trailing parenthetical first took it to 98%.
+        #
+        # Stripping rather than widening the word allowance to two: a
+        # parenthetical is a bounded construct, whereas two free words start
+        # matching prose. The 60-character cap in _extract_labeled_blocks is
+        # applied to the UNDECORATED head, so a long decoration is rejected
+        # before it can be stripped down to a match.
+        without_paren = re.sub(r"\s*\([^)]*\)\s*$", "", normalized).strip()
+        if without_paren:
+            normalized = without_paren
+    for label in sorted(labels, key=len, reverse=True):
+        lowered = label.lower()
+        if normalized == lowered:
+            return label
+        if not tolerant:
+            continue
+        prefix = lowered + " "
+        if normalized.startswith(prefix) and len(normalized[len(prefix):].split()) == 1:
+            return label
+        suffix = " " + lowered
+        if normalized.endswith(suffix) and len(normalized[: -len(suffix)].split()) == 1:
+            return label
+    return None
+
+
 def _extract_labeled_blocks(content: str, labels: list[str]) -> dict[str, str]:
-    normalized_labels = {label.lower(): label for label in labels}
-    targets = tuple(f"{label.lower()}:" for label in labels)
     extracted: dict[str, list[str]] = {label: [] for label in labels}
     current_label: str | None = None
 
@@ -234,15 +1788,17 @@ def _extract_labeled_blocks(content: str, labels: list[str]) -> dict[str, str]:
         candidate = stripped
         while candidate.startswith(("-", "*")):
             candidate = candidate[1:].lstrip()
-        lowered = candidate.lower()
 
         matched_label: str | None = None
         inline_value = ""
-        for target in targets:
-            if lowered.startswith(target):
-                matched_label = normalized_labels[target[:-1]]
-                inline_value = candidate[len(target):].strip()
-                break
+        if ":" in candidate:
+            head, _, rest = candidate.partition(":")
+            # a header is a short line-initial phrase, not a sentence that
+            # happens to contain a colon
+            if len(head) <= 60:
+                matched_label = _match_section_label(head, labels)
+                if matched_label is not None:
+                    inline_value = rest.strip()
 
         if matched_label is not None:
             current_label = matched_label
@@ -254,13 +1810,641 @@ def _extract_labeled_blocks(content: str, labels: list[str]) -> dict[str, str]:
             extracted[current_label].append(stripped)
 
     return {
-        label: _normalize_summary_text("\n".join(lines).strip(), max_chars=None)
+        label: _normalize_summary_text(
+            "\n".join(lines).strip(), max_chars=_memory_section_max_chars() or None
+        )
         for label, lines in extracted.items()
         if "\n".join(lines).strip()
     }
 
 
-def _extract_scientist_note(content: str) -> dict[str, str]:
+def _opener_commit_hint() -> bool:
+    """Ask the turn opener for a commitment, the way a resumption already does.
+
+    Every piece of commitment pressure in the harness lives in the resumption
+    path - "call action(actions) with your best next action or short batch",
+    "Stop deliberating", "Act NOW". The fresh opener says nothing about acting
+    at all. Measured across two runs, sequences following a resumption batch
+    6.73 and 6.91 actions; those following a fresh opener batch 5.43 and 3.91,
+    and the opener figure is what collapsed when resumptions became rarer.
+
+    Deliberately asymmetric: probing stays available but has to be justified,
+    while batching is the default. An earlier prompt that told the model to
+    "re-issue actions one at a time" produced a 200-iteration single-step loop,
+    so the wording avoids prescribing a size while naming the batch.
+    """
+    return _get_env_bool("ARC3_OPENER_COMMIT_HINT", False)
+
+
+def _new_noop_wording() -> bool:
+    """Whether the patch-110 rewordings are used.
+
+    They replaced "changed nothing" with "no change inside the board area"
+    across every guard message. Off restores the wording of the run that scored
+    4.05, so the rewordings can be judged apart from the outcome line they
+    shipped with."""
+    return _get_env_bool("ARC3_NEW_CHANGED_PROMPTS", False)
+
+
+def _animation_timeline_enabled() -> bool:
+    """Whether the diff timeline is offered alongside the raw frames.
+
+    The timeline is a cheap first look, but a summary that reads coherently
+    invites stopping at it: on one level it produced a confident and wrong
+    mechanic that the model only corrected after going to the frames. Off, the
+    model computes its own comparison from `animation_frames` - a few lines it
+    writes readily - at the cost of a snippet before it learns anything, which
+    is wasted on animations that turn out to be irrelevant.
+
+    Only an experiment settles which costs more.
+    """
+    return _get_env_bool("ARC3_ANIMATION_TIMELINE", True)
+
+
+def _dedupe_multicall_line() -> bool:
+    """Drop the duplicated multi-call permission and its stale caveat.
+
+    Upstream states "you may call action() more than once" twice per opener,
+    about thirty lines apart, the second time without the caveat - so the last
+    word the model gets is the permission stripped of its qualifier.
+
+    The caveat itself is now enforced three ways: the batch loop breaks on a
+    terminal result and skips the rest, the per-snippet latch refuses the next
+    action() call, and an executed action ends the turn so the next snippet
+    always opens on a fresh board. On is the change; off keeps upstream.
+    """
+    return _get_env_bool("ARC3_DEDUPE_MULTICALL_LINE", False)
+
+
+def _prefer_tool_calls() -> bool:
+    """State the cost asymmetry: tool calls are free, actions are not.
+
+    The prompt already says not to ration tool calls, but only ever compares
+    them with other tool calls - never with reasoning, which is the trade the
+    model actually makes when it writes twelve thousand characters of
+    hypothesis before its first snippet. Observed: a turn spent 18,012
+    characters reasoning, hit the output cap, and executed nothing.
+    """
+    return _get_env_bool("ARC3_PREFER_TOOL_CALLS", False)
+
+
+# Three bands, a million apart, so no calculated value can cross between them.
+# The maximum _game_priority is about 1,325.
+#
+#   ~2,000,000   never trimmed: the original prefix is whole and this game has
+#                never paid a prefill, so it is the cheapest of all to continue
+#   ~1,000,000   warm again: trimmed once, prefix rebuilt. Ordered within the
+#                band by the estimate at that trim, so a game that was doing
+#                well sits above one that was stalling
+#      0..1,325  awaiting prefill: the trim invalidated the prefix and the next
+#                request has to pay for it anyway, which is the one moment
+#                yielding the slot is free
+#
+# The old scheme sent a flat 999999 while the prefix held, so every game tied at
+# startup and the server broke ties by arrival. Subtracting the queue position
+# gives a deterministic order instead.
+# ---------------------------------------------------------------- TEMPORARY
+# Diagnostic for "the backend never shows more than 13 concurrent requests with
+# 14 games". Records where each game thread is and prints a census once a
+# second, so the missing one can be located rather than guessed at.
+# Remove once the cause is known.
+_GATE_STATS = {"enqueued": 0, "blocked": 0, "blocked_ms": 0, "handover_ms": 0}
+_DIAG_LOCK = threading.Lock()
+_DIAG_STATE: dict[str, str] = {}
+_DIAG_LAST = [0.0]
+
+
+def _diag(agent: Any, where: str) -> None:
+    if not _get_env_bool("ARC3_DIAG_CONCURRENCY", False):
+        return
+    name = getattr(agent, "_diag_name", "?")
+    now = time.monotonic()
+    with _DIAG_LOCK:
+        _DIAG_STATE[name] = where
+        if now - _DIAG_LAST[0] < 1.0:
+            return
+        _DIAG_LAST[0] = now
+        counts: dict[str, int] = {}
+        for value in _DIAG_STATE.values():
+            counts[value] = counts.get(value, 0) + 1
+        total = len(_DIAG_STATE)
+        inflight = counts.get("request", 0)
+        summary = "  ".join(
+            f"{k}={v}" for k, v in sorted(counts.items(), key=lambda kv: -kv[1])
+        )
+        log.warning(
+            "DIAG %d games, %d in flight | %s | gate: %d handovers, %d blocked, "
+            "%dms blocked, %dms total",
+            total, inflight, summary,
+            _GATE_STATS["enqueued"], _GATE_STATS["blocked"],
+            _GATE_STATS["blocked_ms"], _GATE_STATS["handover_ms"],
+        )
+        if inflight < total:
+            others = sorted(
+                f"{g}:{w}" for g, w in _DIAG_STATE.items() if w != "request"
+            )
+            log.warning("DIAG   elsewhere: %s", ", ".join(others))
+
+
+def _priority_refresh_enabled() -> bool:
+    return any(_get_env_bool(name, False) for name in (
+        "ARC3_PRIORITY_REFRESH_QUEUE", "ARC3_PRIORITY_PACE", "ARC3_PRIORITY_TAIL_FADE",
+    ))
+
+
+def _priority_action_count(summary: dict[str, Any]) -> int:
+    return int(summary.get("end_action_num") or 0)
+
+
+@lru_cache(maxsize=8)
+def _load_pace_reference(path: str) -> dict[str, Any]:
+    data = json.loads(Path(path).read_text())
+    values = [data["fallback_tokens"], *data["by_level"].values()]
+    if not all(isinstance(v, (int, float)) and 0 < v < float("inf") for v in values):
+        raise ValueError("Pace reference costs must be finite positive numbers")
+    return data
+
+
+def _pace_reference_tokens(level: int) -> float:
+    path = os.environ.get("ARC3_PRIORITY_PACE_REFERENCE", "").strip()
+    if not path:
+        path = str(Path(__file__).with_name("priority_pace_reference.json"))
+    reference = _load_pace_reference(path)
+    return float(reference["by_level"].get(str(level), reference["fallback_tokens"]))
+
+
+class _PriorityGate:
+    """Harness-side admission control, for when the server ignores `priority`.
+
+    The request field is a hint a backend may drop; this enforces the same
+    ordering locally by limiting how many games hold a slot at once and
+    choosing who gets a freed one.
+
+    A game keeps its slot while its prefix is intact, because interrupting it
+    there throws away a cache entry about to be reused. When the trimmer drops
+    a message the prefix is invalid anyway and the next request must be
+    prefilled from the divergence point - the one moment when yielding is free -
+    so that is where the handover happens.
+
+    Release and re-enqueue are one critical section. Releasing first would leave
+    a window in which a lower-priority waiter takes the slot the caller was
+    about to compete for.
+    """
+
+    def __init__(self, slots: int) -> None:
+        self._cond = threading.Condition()
+        self._free = max(1, int(slots))
+        self._waiting: list[tuple[int, int]] = []
+        self._seq = 0
+        self._admitted: set[int] = set()
+        self._snapshots: dict[int, PrioritySnapshot] = {}
+        self._clock_start: float | None = None
+        self._clock_deadline: float | None = None
+        self._tail_fade_announced = False
+
+    def configure_clock(self, started_at: float, deadline: float | None) -> None:
+        """One shared monotonic clock, configured before initial admission.
+
+        Earliest session start/deadline wins; appropriate for simultaneously
+        launched sessions. No clock is inferred from the last action payload.
+        """
+        if _get_env_bool("ARC3_PRIORITY_TAIL_FADE", False) and deadline is None:
+            raise ValueError("ARC3_PRIORITY_TAIL_FADE requires a finite runtime or soft deadline")
+        with self._cond:
+            self._clock_start = min(self._clock_start, started_at) if self._clock_start is not None else started_at
+            if deadline is not None:
+                self._clock_deadline = min(self._clock_deadline, deadline) if self._clock_deadline is not None else deadline
+
+    def _snapshot_priority(self, snapshot: PrioritySnapshot, now: float) -> int:
+        elapsed = max(0.0, now - self._clock_start) if self._clock_start is not None else 0.0
+        endgame = _endgame_start_minutes() > 0 and elapsed >= 60 * _endgame_start_minutes()
+        tail_fraction = None
+        if _get_env_bool("ARC3_PRIORITY_TAIL_FADE", False):
+            if self._clock_deadline is None or self._clock_start is None:
+                raise ValueError("ARC3_PRIORITY_TAIL_FADE requires a finite runtime or soft deadline")
+            fraction_setting = os.environ.get("ARC3_PRIORITY_TAIL_FADE_FRACTION", "").strip()
+            if fraction_setting:
+                try:
+                    fraction = float(fraction_setting)
+                except ValueError:
+                    raise ValueError("ARC3_PRIORITY_TAIL_FADE_FRACTION must be > 0 and <= 1") from None
+                if not 0.0 < fraction <= 1.0:
+                    raise ValueError("ARC3_PRIORITY_TAIL_FADE_FRACTION must be > 0 and <= 1")
+                # Use the full shared run duration, not the shrinking time left.
+                window = fraction * max(0.0, self._clock_deadline - self._clock_start)
+            else:
+                window = max(1.0, 60 * _get_env_float("ARC3_PRIORITY_TAIL_FADE_MINUTES", 90.0))
+            remaining = max(0.0, self._clock_deadline - now)
+            tail_fraction = remaining / window if window > 0.0 else 0.0
+            if remaining <= window:
+                # Snapshot scoring also happens outside _pump's lock. Announce
+                # once per shared gate, even when several games trim together.
+                with self._cond:
+                    if not self._tail_fade_announced:
+                        self._tail_fade_announced = True
+                        log.warning(
+                            "tail fade phase after %.1f minutes: %.1f minutes "
+                            "remaining; fading tail B to zero over the final "
+                            "%.1f minutes, retaining hazard C",
+                            elapsed / 60.0, remaining / 60.0, window / 60.0,
+                        )
+        return priority_value(
+            snapshot, endgame=endgame, tail_fraction=tail_fraction,
+            human_actions=_priority_human_actions(), action_scale=_priority_action_scale(),
+            token_scale=_priority_token_scale(), action_weight=_priority_action_weight(),
+            tail_base=_priority_tail_base(), tail_cap=_priority_tail_cap(),
+            **_priority_level_options(),
+        )
+
+    def acquire(self, priority: int) -> None:
+        with self._cond:
+            self._enqueue_and_wait(priority)
+
+    def release(self) -> None:
+        with self._cond:
+            self._free += 1
+            self._pump()
+
+    def handover(self, priority: int, snapshot: PrioritySnapshot | None = None) -> None:  # noqa: D401
+        """Give up the slot and immediately compete for it again.
+
+        The caller may win it straight back, which is the right outcome when
+        nothing else is waiting or everything else is further along.
+        """
+        _t0 = time.monotonic()
+        with self._cond:
+            self._free += 1
+            self._enqueue_and_wait(priority, snapshot)
+        _GATE_STATS["handover_ms"] += int(1000 * (time.monotonic() - _t0))
+
+    def _enqueue_and_wait(self, priority: int, snapshot: PrioritySnapshot | None = None) -> None:
+        # TEMPORARY: with slots == games nobody should ever block here
+        _GATE_STATS["enqueued"] += 1
+        self._seq += 1
+        token = self._seq
+        if snapshot is not None:
+            self._snapshots[token] = snapshot
+        # negated so the heap pops the HIGHEST priority; the sequence number
+        # breaks ties by arrival
+        heapq.heappush(self._waiting, (-int(priority), token))
+        self._pump()
+        if token not in self._admitted:
+            _GATE_STATS["blocked"] += 1
+            t0 = time.monotonic()
+            while token not in self._admitted:
+                self._cond.wait()
+            _GATE_STATS["blocked_ms"] += int(1000 * (time.monotonic() - t0))
+        self._admitted.discard(token)
+
+    def _pump(self) -> None:
+        if _priority_refresh_enabled() and self._free > 0:
+            now = time.monotonic()
+            self._waiting = [
+                (-self._snapshot_priority(self._snapshots[token], now), token)
+                if token in self._snapshots else (priority, token)
+                for priority, token in self._waiting
+            ]
+            heapq.heapify(self._waiting)
+        woke = False
+        while self._free > 0 and self._waiting:
+            _, token = heapq.heappop(self._waiting)
+            self._snapshots.pop(token, None)
+            self._admitted.add(token)
+            self._free -= 1
+            woke = True
+        if woke:
+            self._cond.notify_all()
+
+
+def _max_active_streams() -> int:
+    """Games allowed to hold a slot at once. 0 disables the gate entirely.
+
+    Below the game count this staggers the start - a slot only frees when a
+    game's context is first trimmed, which measurements put 30 to 50 turns in -
+    so early games run alone for a while and late ones begin much later. That is
+    the point: a prefix is reused for as long as it lasts, instead of fourteen
+    games each progressing at a fourteenth of the rate.
+    """
+    return max(0, _get_env_int("ARC3_MAX_ACTIVE_STREAMS", 0))
+
+
+_PRIORITY_GATE: _PriorityGate | None = None
+_PRIORITY_GATE_LOCK = threading.Lock()
+
+
+def _priority_gate() -> _PriorityGate | None:
+    """Process-wide, built on first use. Games run as threads in one process,
+    so a module-level gate needs no plumbing and nothing to pickle."""
+    global _PRIORITY_GATE
+    slots = _max_active_streams()
+    if slots <= 0:
+        return None
+    with _PRIORITY_GATE_LOCK:
+        if _PRIORITY_GATE is None:
+            _PRIORITY_GATE = _PriorityGate(slots)
+            log.warning("priority gate active: %d concurrent streams", slots)
+        return _PRIORITY_GATE
+
+
+_PRIORITY_BAND = 1_000_000
+_PRIORITY_UNTRIMMED_BASE = 2_000_000
+
+
+def _priority_level_options() -> dict[str, Any]:
+    # Keep boolean lookup settings backward compatible. The explicit
+    # "remaining" mode selects the 8/7/5/0 tail, already at its final scale.
+    lookup = os.environ.get("ARC3_PRIORITY_TAIL_LOOKUP", "").strip().lower()
+    return {
+        "normalize_score": _get_env_bool("ARC3_PRIORITY_SCORE_NORMALIZATION", False),
+        "tail_lookup": (
+            "remaining" if lookup == "remaining"
+            else _get_env_bool("ARC3_PRIORITY_TAIL_LOOKUP", False)
+        ),
+        "tail_efficiency": _get_env_float("ARC3_PRIORITY_TAIL_EFFICIENCY", 0.8),
+    }
+
+
+def _priority_action_scale() -> float:
+    return max(1.0, _get_env_float("ARC3_PRIORITY_ACTION_SCALE", 115.0))
+
+
+def _priority_token_scale() -> float:
+    return max(1.0, _get_env_float("ARC3_PRIORITY_TOKEN_SCALE", 62000.0))
+
+
+def _priority_action_weight() -> float:
+    """How much of `c` comes from the action curve rather than the token one.
+
+    A blend rather than max(actions, tokens): max sits below both curves, so it
+    demotes harder than either measure alone warrants. Replaying four runs
+    against their own level-up costs, max was the weakest of every combination
+    tried, while the weight itself barely mattered - 0.25 to 1.0 all landed
+    within a few percent of each other.
+    """
+    return min(1.0, max(0.0, _get_env_float("ARC3_PRIORITY_ACTION_WEIGHT", 0.25)))
+
+
+def _priority_tail_cap() -> float:
+    """Ceiling on `b`, the value of every level after the current one.
+
+    min(cap, level + 1) rather than a constant. Chaining the measured
+    completion rates makes `b` nearly flat, which is why a constant looked
+    right - but that chain prices expected future score, and what the scheduler
+    needs is the term that lets depth outrank freshness. Replayed, a flat tail
+    was the worst option tested at every constrained budget; the cap matters
+    more than the slope, and a lower cap is better because it leaves more of
+    the ordering to `a`.
+    """
+    return max(0.0, _get_env_float("ARC3_PRIORITY_TAIL_CAP", 4.0))
+
+
+def _priority_tail_base() -> float:
+    """Value of `b` at level 1, before the per-level rise.
+
+    b = min(cap, base + level - 1). Base 2 with cap 4 gives the fitted form,
+    2 3 4 4 4...; base 6 with cap 6 gives a flat tail, which is what this
+    replaced and is worth being able to get back for a comparison.
+    """
+    return max(0.0, _get_env_float("ARC3_PRIORITY_TAIL_BASE", 2.0))
+
+
+def _priority_human_actions() -> float:
+    """Denominator offset in the efficiency term.
+
+    The benchmark scores a level by min(1, (human/used)^2). The level is not
+    finished yet, so the term belongs at the expected completion point rather
+    than the current count - hence an offset instead of a bare division.
+    """
+    return max(1.0, _get_env_float("ARC3_PRIORITY_HUMAN_ACTIONS", 25.0))
+
+
+def _endgame_start_minutes() -> float:
+    """Wall-clock minutes from a game's start before it switches phase.
+
+    0 disables the endgame phase entirely.
+    """
+    return max(0.0, _get_env_float("ARC3_ENDGAME_START_MINUTES", 0.0))
+
+
+# Endgame priorities sit in the upper half of whichever warmth band the game is
+# in: the cache argument for the bands is orthogonal to the phase, so a game
+# about to pay a prefill still yields to one that has a warm prefix. The widest
+# value either phase can produce is under 2,000 even at level 15, so half a
+# band is ample.
+_PRIORITY_ENDGAME_OFFSET = 500_000
+
+
+def _game_priority(
+    level: int,
+    actions_on_level: int,
+    tokens_on_level: int = 0,
+    endgame: bool = False,
+    total_levels: int | None = None,
+) -> int:
+    """Expected remaining score for a game on `level` after this much spend.
+
+    Two phases. In the main one, (a + b) * c:
+
+      a  what finishing THIS level banks - the score becomes `level` levels
+         completed, discounted by the benchmark's efficiency term
+      b  every level after it, min(cap, base + level - 1)
+      c  whether this level still yields anything, as a blend of two measured
+         decay curves
+
+    In the endgame there is no time to reach a further level, so the tail and
+    the hazard stop meaning anything and what remains is simply what completing
+    this level would bank. The 0.25 floor lets efficiency separate games at the
+    same depth without letting a shallower one overtake a deeper one.
+
+    Replaying four runs against their own level-up costs, the endgame switch was
+    worth 8 to 12% of total score at constrained budgets and nothing at all once
+    the budget covered what the games actually spent - which is where real runs
+    sit, so this matters only if the competition budget is tighter.
+    """
+    level = max(1, int(level))
+    actions = max(0, int(actions_on_level))
+    tokens = max(0, int(tokens_on_level))
+    options = _priority_level_options()
+    if options["normalize_score"] or options["tail_lookup"]:
+        # Same scoring as the refreshed queue, retaining this legacy path's
+        # phase offset. With both flags off, execute the original path below.
+        value = priority_value(
+            PrioritySnapshot(level, actions, tokens, total_levels=total_levels),
+            endgame=endgame, human_actions=_priority_human_actions(),
+            action_scale=_priority_action_scale(), token_scale=_priority_token_scale(),
+            action_weight=_priority_action_weight(), tail_base=_priority_tail_base(),
+            tail_cap=_priority_tail_cap(), **options,
+        )
+        return value + (_PRIORITY_ENDGAME_OFFSET if endgame else 0)
+    human = _priority_human_actions()
+    efficiency = min(1.0, (human / (human + actions)) ** 2)
+
+    if endgame:
+        value = level * (0.25 + efficiency)
+        return _PRIORITY_ENDGAME_OFFSET + max(1, int(value * 100))
+
+    a = level * efficiency
+    b = min(_priority_tail_cap(), _priority_tail_base() + level - 1.0)
+    weight = _priority_action_weight()
+    by_actions = actions / _priority_action_scale()
+    by_tokens = tokens / _priority_token_scale()
+    c = (
+        weight * 0.5 ** (by_actions * by_actions)
+        + (1.0 - weight) * 0.5 ** (by_tokens * by_tokens)
+    )
+    return max(1, int((a + b) * c * 100))
+
+
+def _persistent_functions_scope() -> str:
+    """How long a kept helper lives: 'levelup' (default), 'turn', or 'game'.
+
+    'turn' clears at every analyzer invocation that is not a resumption, so a
+    turn that inspects, acts and yields loses its toolkit even though the
+    investigation continues. That is conservative but costly: the helpers the
+    keepability check lets through are pure transforms taking the board as an
+    argument, and those do not care that an action happened.
+
+    'levelup' keeps them until the level changes, which is where a helper can
+    genuinely go wrong - not by holding stale data, which the check already
+    prevents, but by encoding an assumption in its code, like a glyph parser
+    with row offsets from the previous layout.
+
+    'game' keeps helpers across turns and level transitions. The model is
+    reminded to recheck level-specific assumptions. New game sessions still
+    start with an empty pool; snippet variables remain ephemeral in every scope.
+    """
+    scope = os.environ.get("ARC3_PERSISTENT_FUNCTIONS_SCOPE", "").strip().lower()
+    return scope if scope in ("turn", "levelup", "game") else "levelup"
+
+
+def _repeat_state_guard_enabled() -> bool:
+    """Refuse an action this snippet has already run from this exact board.
+
+    Buggy loop code can spend a whole life bar getting nowhere: a cursor that
+    wraps turns RIGHT into a five-state cycle, and `while pos < j: LEFT` walks
+    away from its target forever. The terminal latch does stop these, but only
+    once the game ends - 20 to 30 actions later.
+
+    Keyed on (board, action) rather than on repetition counts, because the
+    loops mostly alternate: LEFT, RIGHT, LEFT, RIGHT defeats a same-action rule
+    but revisits its first state on the third action.
+
+    Per snippet, and no override slot. A repeat ACROSS snippets is the model
+    choosing, having seen the board and the results; a repeat inside one is
+    code executing a plan fixed before any of it ran, which is the only case
+    that can be a bug the model has not had a chance to notice. It is also not
+    reading the refusal - its code is running - so writing a new snippet is the
+    response, and that clears the map.
+    """
+    return _get_env_bool("ARC3_REPEAT_STATE_GUARD", False)
+
+
+def _guards_active(level: Any) -> bool:
+    """Whether the per-action guards apply at this level.
+
+    Level 1 pays the smallest share of the score - one part in N - and it is
+    where the model knows least, so it probes hardest: repeating an action to
+    see whether the repeat differs, retrying what killed it to learn what
+    killed it. The guards read that as waste. ARC3_GUARDS_FROM_LEVEL=2 lets
+    level 1 run unguarded.
+
+    Covers the stale block and both repeat guards. NOT the batch no-op stop,
+    which costs one action of a planned sequence rather than refusing a
+    deliberate probe, and not the terminal latch, which stops a snippet acting
+    against a board that no longer exists.
+    """
+    try:
+        floor = max(1, int(os.environ.get("ARC3_GUARDS_FROM_LEVEL", "1") or 1))
+    except ValueError:
+        floor = 1
+    try:
+        return int(level) >= floor
+    except (TypeError, ValueError):
+        return True  # unknown level: guard rather than not
+
+
+def _persistent_functions() -> bool:
+    """Retain eligible function sources across calls; off by default.
+
+    Scope is controlled by ARC3_PERSISTENT_FUNCTIONS_SCOPE. State and variables
+    are never serialized. Definitions are validated and recreated in each fresh
+    sandbox, against its current harness globals.
+    """
+    return _get_env_bool("ARC3_PERSISTENT_FUNCTIONS", False)
+
+
+def _animation_enabled() -> bool:
+    """Whether intermediate animation frames are captured and offered.
+
+    Off by default: this is new information for the model rather than a fix to
+    something broken, so it needs to be measurable on its own."""
+    return _get_env_bool("ARC3_ANIMATION", False)
+
+
+def _report_gameplay_changed() -> bool:
+    """Tell the model WHICH kind of change an action produced.
+
+    The harness computes two flags - `board_changed` over the whole frame and
+    `gameplay_changed` over the interior only - but the turn opener reports the
+    first and the system prompt names only the first. On a game with a
+    remaining-steps bar the whole frame differs after every action, so the
+    opener says a change occurred every time and hedges: "verify that it
+    affected gameplay objects rather than only HUD elements". Meanwhile the
+    per-action trace, built from the second flag, can say the same action
+    changed nothing - so the two lines of the same message contradict each
+    other, and on a single-action turn only the misleading one appears.
+
+    On (default) the opener reports the three states that actually occur and
+    the system prompt explains both flags. The wording stays geometric -
+    "cells inside the board area" rather than "gameplay objects" - because the
+    test is a border crop, not a semantic one."""
+    return _get_env_bool("ARC3_REPORT_GAMEPLAY_CHANGED", False)
+
+
+def _explain_gameplay_changed() -> bool:
+    """Whether the system prompt explains the two change flags.
+
+    Split from the per-turn line because the two halves have very different
+    reach and very different arguments. The sentence is paid once and states
+    something the model cannot work out for itself: `last_action_call_result`
+    exposes both flags and nothing says what distinguishes them, so a game with
+    a timer bar makes `board_changed` true after every action.
+
+    The opener line is paid on every executing turn - on one measured run,
+    thousands of times against thirteen batch-guard firings - and only
+    summarises a comparison the model can make with frame_diff. It is also the
+    half with evidence against it: the one run that carried it scored 2.30 on
+    competition where the run without it scored 4.05, though that run changed
+    guard wording at the same time.
+
+    Defaults to following ARC3_REPORT_GAMEPLAY_CHANGED so the pair behaves as
+    before unless it is set explicitly.
+    """
+    raw = os.environ.get("ARC3_EXPLAIN_GAMEPLAY_CHANGED", "").strip()
+    if not raw:
+        return _report_gameplay_changed()
+    return raw.lower() not in ("0", "false", "no", "off")
+
+
+def _alias_respects_stored() -> bool:
+    """Whether `Hypothesis:` may replace a world model that already exists.
+
+    The alias condition checks only THIS reply's extraction: a reply carrying a
+    hypothesis and no `World model:` has its hypothesis stored as the world
+    model, overwriting whatever was there. That is right on a first turn and
+    destructive later - `Hypothesis` connotes untested, so a speculation
+    silently replaces tested content. Measured on 25 games: 7 such overwrites
+    against 351 world-model writes with tolerant headers on, 16 against 311 in
+    an unpatched run.
+
+    On (default) the aliases apply only when the STORED value is also empty.
+    Off restores the original behaviour."""
+    return _get_env_bool("ARC3_ALIAS_RESPECTS_STORED", True)
+
+
+def _extract_scientist_note(
+    content: str, stored: dict[str, Any] | None = None
+) -> dict[str, str]:
     if not content.strip():
         return {}
     extracted = _extract_labeled_blocks(
@@ -287,12 +2471,19 @@ def _extract_scientist_note(content: str) -> dict[str, str]:
         "current_plan": extracted.get("Plan", ""),
         "cross_level_notes": extracted.get("Cross-level notes", ""),
     }
-    if not result["world_model"]:
-        result["world_model"] = extracted.get("Hypothesis", "")
-    if not result["recent_findings"]:
-        result["recent_findings"] = extracted.get("History check", "")
-    if not result["current_plan"]:
-        result["current_plan"] = extracted.get("Next test", "")
+    respect_stored = _alias_respects_stored()
+    existing = stored or {}
+
+    def apply_alias(key: str, label: str) -> None:
+        if result[key]:
+            return
+        if respect_stored and str(existing.get(key) or "").strip():
+            return
+        result[key] = extracted.get(label, "")
+
+    apply_alias("world_model", "Hypothesis")
+    apply_alias("recent_findings", "History check")
+    apply_alias("current_plan", "Next test")
     return result
 
 
@@ -309,7 +2500,17 @@ def _empty_world_model() -> dict[str, str]:
 
 
 def _request_tool_choice(tools: list[dict[str, Any]] | None) -> str | None:
-    return "auto" if tools else None
+    """Tool-choice sent with tool-bearing requests. LOCAL_ANALYZER_TOOL_CHOICE
+    overrides the default "auto": set "omit"/"none" to drop the field for
+    servers whose validation rejects "auto" without a tool-call parser (tools
+    are still sent, so chat templates still render tool instructions); any
+    other value (e.g. "required") passes through verbatim."""
+    if not tools:
+        return None
+    choice = os.environ.get("LOCAL_ANALYZER_TOOL_CHOICE", "auto").strip().lower()
+    if choice in ("", "omit", "none"):
+        return None
+    return choice
 
 
 def _trim_log_text(text: str, *, max_chars: int = _RESPONSE_META_MAX_CHARS) -> str:
@@ -323,6 +2524,7 @@ def _trim_log_text(text: str, *, max_chars: int = _RESPONSE_META_MAX_CHARS) -> s
 def _format_model_response_meta(
     *,
     finish_reason: str,
+    served_by: str,
     reasoning: str,
     content: str,
     tool_calls: list[dict[str, Any]],
@@ -332,6 +2534,7 @@ def _format_model_response_meta(
 ) -> str:
     lines = [
         f"finish_reason: {finish_reason or '(empty)'}",
+        f"served_by: {served_by or '(unknown)'}",
         f"tool_call_count: {len(tool_calls)}",
         f"content_chars: {len(content)}",
         f"reasoning_chars: {len(reasoning)}",
@@ -347,15 +2550,203 @@ def _format_model_response_meta(
     return "\n".join(lines)
 
 
+def _estimate_diff_tokens(text: str) -> int:
+    return len(text) // 3
+
+
+def _diff_entry_line(bucket: str, entry: dict[str, Any]) -> str:
+    color = entry.get("color")
+    pixels = entry.get("pixels")
+    if bucket == "moved":
+        line = f"moved: color {color} {pixels}px {entry.get('from')}->{entry.get('to')}"
+        if entry.get("rotated_by"):
+            line += f" rotated_by {entry['rotated_by']}"
+        return line
+    if bucket == "rotated":
+        return (
+            f"rotated: color {color} {pixels}px by {entry.get('rotated_by')} "
+            f"at {entry.get('at')}"
+        )
+    if bucket == "changed_color":
+        return (
+            f"recolored at {entry.get('at')}: color "
+            f"{entry.get('before_color')}->{entry.get('after_color')} "
+            f"({entry.get('pixels')}px)"
+        )
+    if bucket == "resized":
+        return (
+            f"resized: color {entry.get('color')} "
+            f"{entry.get('before_pixels')}->{entry.get('after_pixels')}px "
+            f"at {entry.get('at_before')}"
+        )
+    return f"{bucket}: color {color} {pixels}px at {entry.get('at')}"
+
+
+_DIFF_BUCKETS = ("moved", "rotated", "changed_color", "resized", "appeared", "disappeared")
+
+
+def _render_auto_frame_diff(
+    diff: dict[str, Any], budget_tokens: int, total_cells: int | None = None
+) -> list[str]:
+    """Render a frame diff as prompt lines, degrading to stay under budget:
+    full listing -> aggregated by (bucket, color, pixels) -> counts only.
+    Resized entries covering most of the grid (the background reshaping as
+    collateral of other objects moving) are elided as noise; the callable
+    frame_diff() keeps them."""
+    if total_cells:
+        resized = diff.get("resized") or []
+        kept = [e for e in resized if (e.get("before_pixels") or 0) * 2 < total_cells]
+        if len(kept) != len(resized):
+            diff = dict(diff)
+            diff["resized"] = kept
+    count = diff.get("changed_cell_count")
+    if count == -1:
+        return [f"grid shape changed ({diff.get('before_shape')} -> {diff.get('after_shape')}); per-object diff skipped."]
+    header = f"{count} cells changed."
+    # rung 1: full listing
+    full = [header]
+    for bucket in _DIFF_BUCKETS:
+        for entry in diff.get(bucket) or []:
+            full.append("- " + _diff_entry_line(bucket, entry))
+    if _estimate_diff_tokens("\n".join(full)) <= budget_tokens:
+        return full
+    # rung 2: aggregate identical entries, largest objects first, capped
+    lines = [header]
+    for bucket in _DIFF_BUCKETS:
+        entries = list(diff.get(bucket) or [])
+        if not entries:
+            continue
+        entries.sort(key=lambda e: -(e.get("pixels") or e.get("after_pixels") or 0))
+        groups: dict[tuple, int] = {}
+        for entry in entries:
+            key = (entry.get("color"), entry.get("pixels"))
+            groups[key] = groups.get(key, 0) + 1
+        shown = 0
+        for entry in entries:
+            if shown >= 3:
+                break
+            key = (entry.get("color"), entry.get("pixels"))
+            if groups.get(key, 0) > 3:
+                continue
+            lines.append("- " + _diff_entry_line(bucket, entry))
+            shown += 1
+        swarm = {k: n for k, n in groups.items() if n > 3}
+        for (color, pixels), n in sorted(swarm.items(), key=lambda kv: -kv[1]):
+            lines.append(f"- {bucket}: {n}x color {color} {pixels}px (positions elided)")
+        remaining = len(entries) - shown - sum(swarm.values())
+        if remaining > 0:
+            lines.append(f"- {bucket}: +{remaining} more")
+    lines.append("(call frame_diff() in python for the full object list)")
+    if _estimate_diff_tokens("\n".join(lines)) <= budget_tokens:
+        return lines
+    # rung 3: counts only, cannot be large
+    parts = [
+        f"{len(diff.get(bucket) or [])} {bucket}"
+        for bucket in _DIFF_BUCKETS
+        if diff.get(bucket)
+    ]
+    return [
+        header + " " + ", ".join(parts) + " (details omitted; call frame_diff() in python)."
+    ]
+
+
 def _build_system_prompt(*, tool_output_tokens: int) -> str:
     prompt = "You are a coding agent solving a grid-based puzzle game."
+    # ARC3_SYSTEM_PROMPT_PREFIX (read at call time): prepended verbatim as the
+    # first line(s) of the system prompt. Intended for model-specific control
+    # directives that live in the system prompt, e.g. Nemotron's "/think"
+    # reasoning toggle. Empty/unset leaves the prompt unchanged.
+    _prefix = os.environ.get("ARC3_SYSTEM_PROMPT_PREFIX", "").strip()
+    if _prefix:
+        prompt = f"{_prefix}\n\n{prompt}"
     prompt += GAME_OVERVIEW_ADDENDUM
     prompt += STRUCTURED_RUNTIME_STATE_ADDENDUM
+    if _explain_gameplay_changed():
+        prompt += GAMEPLAY_CHANGED_ADDENDUM.format(
+            border=f"{_get_env_int('ARC3_NOOP_GUARD_BORDER', 4)} cells"
+        )
+    if _animation_enabled():
+        prompt += ANIMATION_ADDENDUM
+        if _animation_timeline_enabled():
+            prompt += ANIMATION_ADDENDUM_TIMELINE
     if current_grid_image_enabled():
         prompt += MULTIMODAL_CONTEXT_ADDENDUM
+    if _AUTO_FRAME_DIFF:
+        prompt += (
+            " Each turn's message includes an 'Auto frame diff' section summarizing "
+            "object-level changes since your previous analyzer turn (it spans all "
+            "actions executed in between, not just the last one). Treat it as ground "
+            "truth for what changed, consult it before acting, and call `frame_diff()` "
+            "in python when you need per-action or full object detail."
+        )
+    if _DIFF_IMAGE and current_grid_image_enabled():
+        prompt += (
+            " Alongside the current grid image you receive a diff image: cells that "
+            "changed since your previous analyzer turn are shown in their new true "
+            "color; unchanged cells are dimmed to a dark navy that is not a game "
+            "color. Use it to locate change at a glance."
+        )
     prompt += VISUAL_GAME_ADDENDUM
-    prompt += PYTHON_ADDENDUM
-    prompt += COMPACT_TOOL_SESSION_ADDENDUM.format(tool_output_tokens=tool_output_tokens)
+    if _get_env_bool("ARC3_ACTION_INFO", False):
+        # The action-info section replaces the earlier coordinate-only guidance.
+        prompt = prompt.replace(
+            "- For `MOUSE`, pass `row` and `col` integer arguments. `row` is vertical position, `col` is horizontal position.\n",
+            "",
+        )
+        prompt += ACTION_INFO_ADDENDUM
+        if undo_exposure_mode() == "on":
+            prompt += UNDO_INFO_ADDENDUM
+        if reset_exposed():
+            prompt += RESET_INFO_ADDENDUM
+    prompt += (
+        PYTHON_ADDENDUM_HEAD.replace(
+            "- Every `python` tool call starts fresh. Re-import modules or re-define any custom utility logic you need.\n",
+            "- Python variables reset between tool calls. Re-import modules as needed; "
+            "eligible functions are retained as described in the tool session rules below.\n",
+        ) if _persistent_functions() else PYTHON_ADDENDUM_HEAD
+    )
+    prompt += (
+        WORLD_MODEL_FREE_ADDENDUM if _memory_sections_disabled() else WORLD_MODEL_ADDENDUM
+    )
+    prompt += PYTHON_ADDENDUM_TAIL
+    if _get_env_bool("ARC3_FRAME_DIFF_HINT", False):
+        # documents `frame_diff(before, after)`. The function stays callable
+        # either way - this only controls whether the model is told about it,
+        # so the arm measures whether knowing costs more than it returns.
+        prompt += FRAME_DIFF_HINT_ADDENDUM
+    if _get_env_bool("ARC3_STEP_VERIFICATION_HINT", False):
+        # General checks during action sequences and targeted probes when search fails.
+        prompt += STEP_VERIFICATION_ADDENDUM
+    # only describe the guards that are actually armed: prompt weight spent on a
+    # mechanism that cannot fire is weight the model reads and cannot use, and an
+    # exception it is told about but never sees is worse than silence
+    # The guards are no longer described up front. Every claim the addenda made -
+    # not executed, nothing spent, the batch is checked throughout, how to
+    # override - is repeated in the refusal message itself, which arrives with
+    # the concrete action attached and only when it is relevant. Describing a
+    # mechanism the model may never meet is prompt weight spent on nothing.
+    prompt += COMPACT_TOOL_SESSION_ADDENDUM.format(
+        tool_output_tokens=tool_output_tokens,
+        prefer_tool_calls=PREFER_TOOL_CALLS_LINE if _prefer_tool_calls() else "",
+        persistent_functions=(
+            PERSISTENT_FUNCTIONS_LINE.format(lifetime=(
+                "for later calls throughout this game, including across level changes; "
+                "they are cleared when a new game starts"
+                if _persistent_functions_scope() == "game" else
+                "for later calls on the same level; they are cleared when the level changes"
+                if _persistent_functions_scope() == "levelup" else
+                "for later calls in this analyzer turn, including yield resumptions; "
+                "they are cleared at the start of the next turn"
+            ), import_guidance=(
+                "Supported explicit top-level imports used by retained functions are restored "
+                "with them; imports inside a function also work."
+                if _get_env_bool("ARC3_PERSISTENT_FUNCTIONS_IMPORTS", False) else
+                "Put needed imports inside the function."
+            )) if _persistent_functions() else EPHEMERAL_FUNCTIONS_LINE
+        ),
+    )
+    if _level_transfer_guidance_enabled():
+        prompt = apply_level_transfer_system_guidance(prompt)
     return prompt
 
 
@@ -378,6 +2769,12 @@ class AnalyzerTurnResult:
 class _ToolDispatchResult:
     content: str
     step_executed: bool = False
+    # True when an executed action changed the board outside the HUD border, or
+    # ended the level or the attempt. "Executed" alone is too weak a signal for
+    # the reasoning-effort ladder: bumping a wall executes and achieves nothing,
+    # and resetting on it lets the model oscillate between truncating at full
+    # effort and acting inertly at reduced effort, never settling.
+    made_progress: bool = False
 
 
 @dataclass(frozen=True)
@@ -398,6 +2795,7 @@ class _AsciiFrameView:
 class _AsciiHistoryEntryView:
     action: str
     frame: _AsciiFrameView
+    result: dict[str, Any]
 
     def __str__(self) -> str:
         return f"AsciiHistoryEntryView(action={self.action!r}, frame={self.frame})"
@@ -422,7 +2820,10 @@ def _to_ascii_history_views(history_entries: list[HistoryEntry]) -> list[_AsciiH
         frame_view = _to_ascii_frame_view(entry.frame)
         if frame_view is None:
             continue
-        views.append(_AsciiHistoryEntryView(action=entry.action, frame=frame_view))
+        views.append(_AsciiHistoryEntryView(
+            action=entry.action, frame=frame_view,
+            result=dict(getattr(entry, "result", {}) or {}),
+        ))
     return views
 
 
@@ -445,26 +2846,167 @@ def _ascii_history_view_payload(history_entries: list[HistoryEntry]) -> list[dic
         frame_payload = _ascii_frame_view_payload(entry.frame)
         if frame_payload is None:
             continue
-        payload.append({"action": entry.action, "frame": frame_payload})
+        payload.append({
+            "action": entry.action, "frame": frame_payload,
+            "result": dict(getattr(entry, "result", {}) or {}),
+        })
     return payload
 
 
-def _format_action_span(start_action_num: int | None, end_action_num: int | None) -> str | None:
-    if start_action_num is None or end_action_num is None:
+_VISION_PATCH_PIXELS = 32          # Qwen3.6: 16px patches, 2x2 merge
+_VISION_SENTINEL_TOKENS = 2        # vision start/end markers
+_IMAGE_TOKENS_FALLBACK = 402       # 64-cell board at MULTIMODAL_UPSCALE=10
+_PNG_SIGNATURE = b"\x89PNG\r\n\x1a\n"
+
+
+def _text_token_chars() -> int:
+    """Characters per token for the text heuristic - the SEED value only.
+
+    Measured against a server-reported prompt_tokens on a real request: 25,189
+    characters of system prompt plus opener tokenised to 5,892, i.e. about 4.3
+    characters per token. The 3 default therefore over-counts text by roughly
+    40%, so the trimmer evicts history the budget could have held. This value
+    is used until the first response supplies a measurement; after that
+    ARC3_CALIBRATE_TEXT_TOKENS drives the divisor from the server's own
+    figures."""
+    return max(1, _get_env_int("ARC3_TEXT_TOKEN_CHARS", 3))
+
+
+def _calibrate_text_tokens() -> bool:
+    """Derive the characters-per-token divisor from the server's reported
+    prompt_tokens rather than a fixed guess.
+
+    Every response carries prompt_tokens for the request that produced it - an
+    exact count including the chat template. Subtracting the vision cost of the
+    request's images leaves the text, and dividing by the text characters gives
+    the divisor for the next estimate. No extra requests, no tokenizer, and it
+    tracks content: code-heavy turns tokenise differently from prose.
+
+    The last measurement is used rather than an average: history carries
+    forward, so the prompt just measured is mostly the prompt about to be
+    built, and averaging would blend in content already evicted."""
+    return _get_env_bool("ARC3_CALIBRATE_TEXT_TOKENS", True)
+
+
+# Asymmetric on purpose. Over-counting trims history and degrades one game;
+# under-counting overflows the context, and an overflowing request holds a
+# slot and can pressure the others. Measured across 22 games: 2.75-3.69,
+# mean 3.12. The 3.3 ceiling therefore binds on the sparsest games by
+# design - a deliberate haircut - while the floor is set low enough that
+# only an arithmetic failure reaches it, and such a game is likely lost
+# anyway.
+def _text_token_chars_min() -> float:
+    """Floor for the calibrated divisor. Low enough that only an arithmetic
+    failure reaches it, and a game in that state is likely lost anyway."""
+    # a divisor of zero would make every estimate infinite and trim the whole
+    # history on the first request
+    return max(0.1, _get_env_float("ARC3_TEXT_TOKEN_CHARS_MIN", 1.0))
+
+
+def _text_token_chars_max() -> float:
+    """Ceiling for the calibrated divisor.
+
+    Exposed because 3.3 is a fitted constant, not a property of the tokeniser:
+    it came from 22 games measuring 2.75-3.69, so it binds on the sparsest
+    boards by design. A run whose prompts are denser than that sample - a
+    different game set, a longer system prompt, another model - is being
+    trimmed harder than its own measurements justify, and raising this is how
+    that is tested.
+    """
+    # never below the floor: an inverted pair would clamp every measurement to
+    # the ceiling and silently invert the guard's meaning
+    return max(_text_token_chars_min(),
+               _get_env_float("ARC3_TEXT_TOKEN_CHARS_MAX", 3.3))
+
+
+def _image_token_estimate_enabled() -> bool:
+    """Count image parts by their real vision-token cost instead of by the
+    length of their base64 payload. Without this an image is charged roughly
+    one token per three base64 characters - thousands for a board render whose
+    true cost is a few hundred - so enabling images silently evicts real
+    history."""
+    return _get_env_bool("ARC3_IMAGE_TOKEN_ESTIMATE", True)
+
+
+def _png_dimensions(data_url: str) -> tuple[int, int] | None:
+    """Width/height from a base64 PNG data URL, read from the IHDR header
+    without decoding the image. Only the first 24 bytes are needed."""
+    marker = "base64,"
+    index = data_url.find(marker)
+    if index < 0:
+        return None
+    head = data_url[index + len(marker):index + len(marker) + 32]
+    if len(head) < 32:
+        return None
+    try:
+        raw = base64.b64decode(head, validate=True)
+    except Exception:
         return None
-    if start_action_num <= 0 or end_action_num <= 0:
+    if len(raw) < 24 or not raw.startswith(_PNG_SIGNATURE) or raw[12:16] != b"IHDR":
         return None
-    if start_action_num == end_action_num:
-        return f"{start_action_num}"
-    return f"{start_action_num}-{end_action_num}"
+    width = int.from_bytes(raw[16:20], "big")
+    height = int.from_bytes(raw[20:24], "big")
+    if width <= 0 or height <= 0:
+        return None
+    return width, height
+
+
+def _image_part_tokens(part: dict[str, Any]) -> int:
+    """Vision tokens for one image content part: one token per merged patch
+    plus the two sentinel tokens. ARC3_IMAGE_TOKENS_FLAT overrides the
+    computation (useful for non-Qwen backends) and is also the fallback when
+    the header cannot be read."""
+    flat = _get_env_int("ARC3_IMAGE_TOKENS_FLAT", 0)
+    if flat > 0:
+        return flat
+    url = ""
+    image_url = part.get("image_url")
+    if isinstance(image_url, dict):
+        url = str(image_url.get("url") or "")
+    dims = _png_dimensions(url) if url else None
+    if dims is None:
+        return _IMAGE_TOKENS_FALLBACK
+    width, height = dims
+    cells = -(-width // _VISION_PATCH_PIXELS) * -(-height // _VISION_PATCH_PIXELS)
+    return cells + _VISION_SENTINEL_TOKENS
+
+
+def _split_messages_for_estimate(
+    messages: list[dict[str, Any]],
+) -> tuple[list[dict[str, Any]], int]:
+    """Return (messages with image payloads replaced by a short placeholder,
+    total vision tokens). Originals are never mutated."""
+    image_tokens = 0
+    rewritten: list[dict[str, Any]] = []
+    for message in messages:
+        content = message.get("content")
+        if not isinstance(content, list):
+            rewritten.append(message)
+            continue
+        parts: list[Any] = []
+        for part in content:
+            if isinstance(part, dict) and part.get("type") == "image_url":
+                image_tokens += _image_part_tokens(part)
+                parts.append({"type": "image_url", "image_url": {"url": "<image>"}})
+            else:
+                parts.append(part)
+        rewritten.append({**message, "content": parts})
+    return rewritten, image_tokens
 
 
-def _estimate_tokens(value: Any) -> int:
+def _estimate_tokens(value: Any, chars_per_token: float | None = None) -> int:
     try:
-        rendered = json.dumps(value, ensure_ascii=True, sort_keys=True, default=str)
+        # ensure_ascii=False is deliberate: with escaping ON, every non-ASCII
+        # character renders as a six-character \uXXXX sequence, so any prompt
+        # containing arrows, box-drawing glyphs or a multiplication sign is
+        # counted at up to 6x its real size. The trimmer decides when to evict
+        # in these units, so the inflation makes it discard history it did not
+        # need to - and the error grows with the share of non-ASCII text.
+        rendered = json.dumps(value, ensure_ascii=False, sort_keys=True, default=str)
     except TypeError:
         rendered = str(value)
-    return max(1, (len(rendered) + 2) // 3)
+    chars = chars_per_token or _text_token_chars()
+    return max(1, int(len(rendered) / chars) + (1 if len(rendered) % chars else 0))
 
 
 def _host_accessible_base_url(base_url: str) -> str:
@@ -785,20 +3327,29 @@ def _append_request_snapshot(
     event: str | None = None,
     tool_choice: str | None = None,
     finish_reason: str | None = None,
+    served_by: str | None = None,
     analysis_step: int | None = None,
     action: int | None = None,
     request_index_within_turn: int | None = None,
+    usage: dict[str, Any] | None = None,
+    chat_template_kwargs: dict[str, Any] | None = None,
 ) -> None:
     payload = {
         "messages": messages,
         "tools": tools or [],
     }
+    if isinstance(chat_template_kwargs, dict) and chat_template_kwargs:
+        payload["chat_template_kwargs"] = dict(chat_template_kwargs)
+    if isinstance(usage, dict) and usage:
+        payload["usage"] = usage
     if event:
         payload["event"] = event
     if tool_choice:
         payload["tool_choice"] = tool_choice
     if finish_reason is not None:
         payload["finish_reason"] = str(finish_reason)
+    if served_by:
+        payload["served_by"] = str(served_by)
     if analysis_step is not None:
         payload["analysis_step"] = analysis_step
     if action is not None:
@@ -885,9 +3436,13 @@ def _extract_reasoning_text(message: dict[str, Any]) -> str:
 
 
 def _is_context_length_error(exc: BaseException) -> bool:
-    message = str(exc).lower()
+    message = str(exc).lower().replace("\u2019", "'")
     return (
         "maximum context length" in message
+        # SGLang reports an oversized prompt with this wording instead of
+        # "maximum context length". Route it through the same drain/retry path.
+        or "is longer than the model's context length" in message
+        or "context_length_exceeded" in message
         or "reduce the length of the input prompt" in message
         or "parameter=input_tokens" in message
         or '"param":"input_tokens"' in message
@@ -899,6 +3454,7 @@ class _ChatCompletionResult:
     message: dict[str, Any]
     finish_reason: str = ""
     usage: dict[str, Any] | None = None
+    served_by: str = ""
 
 
 class ToolAgent:
@@ -910,6 +3466,7 @@ class ToolAgent:
         model: str = _DEFAULT_ANALYZER_MODEL,
         timeout: Optional[float] = None,
         save_request_logs: bool = False,
+        dispatch_index: int = 0,
         api_key: str | None = None,
         base_url: str | None = None,
         provider: str | None = None,
@@ -932,6 +3489,10 @@ class ToolAgent:
         self._tool_steps = None if _LOCAL_ANALYZER_TOOL_STEPS <= 0 else max(1, _LOCAL_ANALYZER_TOOL_STEPS)
         self._python_timeout = min(30, max(1, _LOCAL_ANALYZER_TOOL_TIMEOUT))
         self._yield_seconds = None if _LOCAL_ANALYZER_YIELD_SECONDS <= 0 else float(_LOCAL_ANALYZER_YIELD_SECONDS)
+        self._yield_tokens = (
+            None if _LOCAL_ANALYZER_YIELD_TOKENS <= 0 else int(_LOCAL_ANALYZER_YIELD_TOKENS)
+        )
+        self._turn_generated_tokens = 0
         configured_max_output = _LOCAL_ANALYZER_MAX_OUTPUT
         self._max_output_tokens = None if configured_max_output <= 0 else max(1, configured_max_output)
         self._reply_reserve_tokens = self._max_output_tokens or 512
@@ -946,6 +3507,75 @@ class ToolAgent:
             1024,
             _LOCAL_ANALYZER_CONTEXT_WINDOW - self._reply_reserve_tokens - self._request_safety_margin_tokens,
         )
+        self._turns_without_wm_update = 0
+        self._pending_wm_rebuild_reason = ""
+        self._wm_absorbed_in_turn = False
+        self._auto_diff_prev_frame: Frame | None = None
+        self._pending_diff_image: dict[str, Any] | None = None
+        self._context_was_trimmed = False
+        self._has_evicted = False
+        self._actions_at_level_start = 0
+        self._tokens_at_level_start = 0
+        self._progress_pace = ProgressPace()
+        self._pace_last_completed_level = 0
+        # queue position spreads the untrimmed band so games do not tie
+        self._diag_name = f"g{int(dispatch_index):02d}"
+        self._game_elapsed_seconds: float | None = None
+        self._endgame_announced = False
+        self._priority_current = _PRIORITY_UNTRIMMED_BASE - max(0, int(dispatch_index))
+        self._priority_next = self._priority_current
+        self._kept_functions: dict[str, str] = {}
+        self._last_animation_chain: list[Any] = []
+        self._last_action_gameplay_changed: bool | None = None
+        self._pending_animation_image: dict[str, Any] | None = None
+        self._pending_animation_image_label: str = ""
+        self._pending_diff_image_label: str = ""
+        self._turn_diff_image: dict[str, Any] | None = None
+        self._turn_diff_image_label: str = ""
+        self._pending_gameover_images: list[tuple[str, dict[str, Any]]] = []
+        self._noop_guard_pre_hash: str = ""
+        # (interior board hash, action signature) already run in THIS snippet.
+        # Written in _record_guard_observations, read in _action_guard_hook,
+        # cleared before each sandbox call - three sites, one lifetime.
+        self._summary_context_tokens_since_attempt = 0
+        self._http_initial_grace_used = False
+        self._repeat_state_seen: set[tuple[str, str]] = set()
+        # (state_hash, action_sig) for each action the guard hook waved through,
+        # in execution order - the only place the intermediate board states of a
+        # batch are visible to the agent, since the solver reports outcomes per
+        # position but not the boards they ran against.
+        self._guard_hook_trace: list[tuple[str, str]] = []
+        self._rejected_section_updates: list[str] = []
+        # characters per token, measured from the last response; None until one
+        # arrives, and reported in the transcript whenever it changes
+        self._text_chars_per_token: float | None = None
+        # index into the reasoning-effort ladder; -1 means "server default"
+        self._reasoning_effort_rung: int = -1
+        self._guard_death_override_batch: bool = False
+        self._noop_guard_deaths_enabled: bool = _get_env_bool(
+            "ARC3_DEATH_REPEAT_GUARD", False
+        )
+        self._noop_repeat_guard: NoopRepeatGuard | None = (
+            NoopRepeatGuard()
+            if (
+                _get_env_bool("ARC3_NOOP_REPEAT_GUARD", False)
+                or self._noop_guard_deaths_enabled
+            )
+            else None
+        )
+        # only the no-op half is gated separately; the shared store is created
+        # when either half is enabled
+        self._noop_guard_noops_enabled: bool = _get_env_bool(
+            "ARC3_NOOP_REPEAT_GUARD", False
+        )
+        self._resume_after_yield: bool = False
+        self._resume_reason: str = ""
+        self._prev_level_start_frame: Frame | None = None
+        self._cur_level_start_frame: Frame | None = None
+        self._death_ledger_level: Any = None
+        self._death_ledger_attempts: list[dict[str, Any]] = []
+        self._death_ledger_index: dict[str, list[tuple[int, int]]] = {}
+        self._death_ledger_total: int = 0
         self._history_messages: list[dict[str, Any]] = []
         self._session_runtime_dir: Path | None = None
         self._session_total_tokens = 0
@@ -953,7 +3583,7 @@ class ToolAgent:
         self._step_env_callback: Callable[[dict[str, Any]], dict[str, Any]] | None = None
         self._current_valid_actions: list[str] = []
         self._last_step_summary: dict[str, Any] | None = None
-        self._last_action_result: dict[str, Any] | None = None
+        self._last_action_call_result: dict[str, Any] | None = None
         self._summarized_knowledge = _empty_world_model()
 
     def _headers(self) -> dict[str, str]:
@@ -976,12 +3606,20 @@ class ToolAgent:
         runtime_dir = state_path.parent
         if self._session_runtime_dir != runtime_dir:
             self._session_runtime_dir = runtime_dir
+            self._priority_level_count_warned = False
+            if _persistent_functions():
+                self._kept_functions = {}
+                self._retained_clear_notice = ""
             self._history_messages = []
             self._session_total_tokens = 0
             self._session_generated_tokens = 0
+            self._summary_context_tokens_since_attempt = 0
             self._last_step_summary = None
-            self._last_action_result = None
+            self._last_action_call_result = None
             self._summarized_knowledge = _empty_world_model()
+            self._turns_without_wm_update = 0
+            self._pending_wm_rebuild_reason = ""
+            self._pending_wm_revise_reason = ""
 
     @property
     def total_tokens(self) -> int:
@@ -991,7 +3629,19 @@ class ToolAgent:
     def generated_tokens(self) -> int:
         return max(0, int(self._session_generated_tokens))
 
-    def _accumulate_usage_tokens(self, usage: dict[str, Any] | None) -> None:
+    def _accumulate_usage_tokens(
+        self, usage: dict[str, Any] | None, *, count_toward_turn: bool = True
+    ) -> None:
+        """Add a response's usage to the counters.
+
+        count_toward_turn=False for generation that is not part of the turn's
+        deliberation - the rolling summary. Its tokens are real GPU time and
+        belong in the session and per-game totals, or a summary-heavy run looks
+        cheaper than it is and gets more compute than
+        max_generated_tokens_per_game intends. But the per-turn yield budget
+        measures how long the model has been working on THIS turn, and charging
+        it for a summary would shorten whichever turn happened to follow one.
+        """
         if not isinstance(usage, dict):
             return
         generated_token_count = 0
@@ -1003,6 +3653,8 @@ class ToolAgent:
             except (TypeError, ValueError):
                 continue
         self._session_generated_tokens += generated_token_count
+        if count_toward_turn:
+            self._turn_generated_tokens += generated_token_count
 
         total_tokens = usage.get("total_tokens")
         try:
@@ -1021,6 +3673,41 @@ class ToolAgent:
                 continue
         self._session_total_tokens += token_count
 
+    @staticmethod
+    def _turn_action_trace(executed_results: list[dict[str, Any]]) -> list[str]:
+        """Stitch per-call action traces into one sequentially numbered
+        turn-level trace; synthesize entries for calls without a trace."""
+        entries: list[str] = []
+        for item in executed_results:
+            item_trace = item.get("action_trace")
+            if isinstance(item_trace, list) and item_trace:
+                entries.extend(
+                    re.sub(r"^\d+:\s*", "", str(line)) for line in item_trace
+                )
+                continue
+            names = item.get("executed_actions")
+            if not isinstance(names, list) or not names:
+                display = str(item.get("action_display") or "").strip()
+                names = [display] if display else []
+            if item.get("game_over"):
+                verdict = "GAME OVER (attempt ended immediately after this action)"
+            elif item.get("run_complete") or item.get("done"):
+                verdict = "RUN COMPLETE"
+            elif item.get("level_completed"):
+                verdict = "LEVEL COMPLETED"
+            elif item.get("gameplay_changed") is False:
+                verdict = (
+                    "NO-OP (executed, no change inside the board area)"
+                    if _new_noop_wording()
+                    else "NO-OP (executed, changed nothing)"
+                )
+            elif item.get("gameplay_changed") is True:
+                verdict = "effect"
+            else:
+                verdict = "executed"
+            entries.extend(f"{name} -> {verdict}" for name in names)
+        return [f"{position}: {entry}" for position, entry in enumerate(entries, 1)]
+
     def _summarize_step_sequence(self, action_results: list[dict[str, Any]]) -> dict[str, Any] | None:
         if not action_results:
             return None
@@ -1063,58 +3750,161 @@ class ToolAgent:
             "level_transition": any(bool(item.get("level_completed")) for item in executed_results),
             "run_complete": any(bool(item.get("run_complete")) for item in executed_results),
             "game_over": any(bool(item.get("game_over")) for item in executed_results),
+            "game_over_trace": (
+                ToolAgent._turn_action_trace(executed_results)
+                if any(item.get("game_over") for item in executed_results)
+                else []
+            ),
+            "fatal_action": next(
+                (
+                    str(item.get("fatal_action") or item.get("action_display") or "")
+                    for item in executed_results
+                    if item.get("game_over")
+                ),
+                "",
+            ),
             "board_changed": any(bool(item.get("board_changed")) for item in executed_results),
+            # the interior-only flag, so the opener can say WHICH kind of change
+            # happened instead of hedging over the broader one
+            "gameplay_changed": any(
+                bool(item.get("gameplay_changed")) for item in executed_results
+            ),
             "stop_reason": last.get("stop_reason"),
+            "no_op_stops": [
+                {
+                    "action": item.get("no_op_action"),
+                    "index": item.get("no_op_action_index"),
+                    "skipped": item.get("skipped_actions") or [],
+                    "trace": item.get("action_trace") or [],
+                }
+                for item in executed_results
+                if item.get("stop_reason") == "no_op_action"
+            ],
+            # the animation of the last executed action, which is what
+            # last_animation_frames / last_animation_timeline will return when asked
+            "animation": (executed_results[-1].get("animation") if executed_results else None),
+            "animation_action": (
+                executed_results[-1].get("action_display") if executed_results else None
+            ),
+            "guard_stops": [
+                {
+                    "kind": item.get("stop_reason"),
+                    "action": item.get("blocked_action"),
+                    "detail": item.get("stop_detail"),
+                    "skipped": item.get("skipped_actions") or [],
+                    "trace": item.get("action_trace") or [],
+                }
+                # scans ALL results, not just executed ones: a refusal has
+                # executed=False by construction and is filtered out of
+                # executed_results before this point
+                for item in action_results
+                if item.get("stop_reason") in ("known_noop", "known_death", "stale_state")
+            ],
         }
 
-    def _describe_last_outcome(self, summary: dict[str, Any] | None) -> str:
-        if not summary:
-            return ""
-        span = _format_action_span(
-            summary.get("start_action_num"),
-            summary.get("end_action_num"),
-        )
-        count = summary.get("executed_count")
-        prefix = "Last executed sequence"
-        if span and count:
-            prefix = f"Actions {span} ({count} total)"
-        elif span:
-            prefix = f"Action span {span}"
-        elif count:
-            prefix = f"Last executed sequence ({count} total)"
+    # _describe_last_outcome removed. It built an outcome sentence and was
+    # called from nowhere - in this harness AND upstream - so neither its
+    # wording nor the fallback it carried ever reached a transcript. Two
+    # patches were spent revising text no model ever read. What it uniquely
+    # provided is now rendered inline in _build_user_prompt.
 
-        level = summary.get("level")
-        if summary.get("level_transition"):
-            level_text = f" to level {level}" if level is not None else ""
-            return f"{prefix} triggered a level transition{level_text}; re-ground on the new scene."
-        if summary.get("run_complete"):
-            return f"{prefix} completed the run."
-        if summary.get("game_over"):
-            return f"{prefix} reached GAME_OVER."
-
-        pieces = [prefix]
-        if summary.get("board_changed"):
-            pieces.append("produced a board change; verify that it affected gameplay objects rather than only HUD elements.")
-        else:
-            pieces.append("did not show a confirmed board change; treat this as weak evidence until verified.")
-        stop_reason = _normalize_summary_text(summary.get("stop_reason"))
-        if stop_reason:
-            pieces.append(f"stop_reason={stop_reason}.")
-        return " ".join(pieces)
-
-    def _update_summarized_knowledge_from_assistant(self, content: str) -> None:
-        note = _extract_scientist_note(content)
+    def _update_summarized_knowledge_from_assistant(self, content: str) -> list[str]:
+        if _memory_sections_disabled():
+            # nothing is carried or shown, so parsing would only build state
+            # that never reaches the model
+            return []
+        note = _extract_scientist_note(content, self._summarized_knowledge)
         if not note:
-            return
+            return []
+        updated: list[str] = []
+        rejected: list[str] = []
         for key, value in note.items():
-            if value:
-                self._summarized_knowledge[key] = value
+            if not value:
+                continue
+            if _is_degenerate_text(value):
+                # keep the previous version rather than truncating: a truncated
+                # degenerate section is still degenerate, and it would be
+                # re-injected into every later prompt
+                rejected.append(key)
+                continue
+            self._summarized_knowledge[key] = value
+            updated.append(key)
+        if rejected:
+            self._rejected_section_updates = sorted(set(rejected))
+            log.warning(
+                "rejected degenerate memory section update(s): %s",
+                ", ".join(self._rejected_section_updates),
+            )
+        if updated:
+            self._turns_without_wm_update = 0
+            self._wm_absorbed_in_turn = True
+            if "world_model" in updated:
+                self._pending_wm_rebuild_reason = ""
+            if "world_model" in updated or "current_plan" in updated:
+                self._pending_wm_revise_reason = ""
+        return updated
 
     def _update_summarized_knowledge_from_step_summary(self) -> None:
         summary = self._last_step_summary
         if not summary:
             return
-        if summary.get("level_transition") or summary.get("run_complete") or summary.get("game_over"):
+        wipe_reason = ""
+        revise_reason = ""
+        if summary.get("level_transition"):
+            # the priority estimate is per level, so both counts restart here
+            if _get_env_bool("ARC3_PRIORITY_PACE", False):
+                completed_level = max(1, int(summary.get("level") or 2) - 1)
+                if completed_level > self._pace_last_completed_level:
+                    self._progress_pace.record_completion(
+                        self._session_generated_tokens - self._tokens_at_level_start,
+                        _pace_reference_tokens(completed_level),
+                    )
+                    self._pace_last_completed_level = completed_level
+            self._actions_at_level_start = _priority_action_count(summary)
+            self._tokens_at_level_start = self._session_generated_tokens
+        if summary.get("level_transition") and _persistent_functions_scope() == "levelup":
+            # a helper can encode the old layout in its code even when it holds
+            # no stale data, so the level boundary is where it stops being safe
+            self._kept_functions = {}
+            if _persistent_functions():
+                self._retained_clear_notice = (
+                    "Your retained functions were cleared at this level transition. "
+                    "Redefine any helpers you still need after checking their assumptions "
+                    "against the new board."
+                )
+        elif (
+            summary.get("level_transition")
+            and _persistent_functions_scope() == "game"
+            and _persistent_functions()
+            and getattr(self, "_kept_functions", None)
+        ):
+            self._retained_clear_notice = (
+                "Your retained functions remain available from previous levels. "
+                "Check level-specific assumptions against the new board before reusing them."
+            )
+        if summary.get("level_transition"):
+            # read at call time (not a module constant) so this patch's hunks
+            # stay clear of the env-knob block that other patches extend
+            if _get_env_int("ARC3_WM_WIPE_ON_LEVEL", 1):
+                wipe_reason = "a level transition"
+            else:
+                revise_reason = "level"
+        elif summary.get("run_complete"):
+            wipe_reason = "run completion"
+        elif summary.get("game_over"):
+            if _WM_WIPE_ON_GAME_OVER:
+                wipe_reason = "a game over"
+            else:
+                revise_reason = "death"
+        if revise_reason:
+            self._pending_wm_revise_reason = revise_reason
+            self._wm_revise_hint_shown = False
+            # a newer keep-event supersedes any stale rebuild nag (and vice
+            # versa below): at most one pending nudge, always for the latest event
+            self._pending_wm_rebuild_reason = ""
+        if wipe_reason:
+            self._pending_wm_rebuild_reason = wipe_reason
+            self._pending_wm_revise_reason = ""
             for key in (
                 "world_model",
                 "goal_model",
@@ -1135,7 +3925,24 @@ class ToolAgent:
             ("Plan", self._summarized_knowledge.get("current_plan", "")),
             ("Cross-level notes", self._summarized_knowledge.get("cross_level_notes", "")),
         ]
+        if _memory_sections_disabled():
+            return []
+        rejected = list(getattr(self, "_rejected_section_updates", []))
+        self._rejected_section_updates = []
+        allowed = _memory_section_labels()
+        if allowed is not None:
+            entries = [(label, value) for label, value in entries if label in allowed]
         lines = [f"- {label}: {value}" for label, value in entries if value]
+        if rejected:
+            # without this the model sees its section unchanged and is likely to
+            # emit the same degenerate text again
+            lines.append(
+                "- NOTE: your last update to "
+                + ", ".join(f"`{name.replace('_', ' ')}`" for name in rejected)
+                + " was REJECTED as degenerate (highly repetitive or corrupted "
+                "text) and the previous version above was kept. Rewrite it "
+                "concisely in plain prose."
+            )
         if not lines:
             return []
         return [
@@ -1144,19 +3951,857 @@ class ToolAgent:
             "- Revise any item above immediately if `current_frame` or `history` contradicts it.",
         ]
 
+    def _prepare_auto_diff(
+        self,
+        current_frame: Frame | None,
+        previous_step_summary: dict[str, Any] | None,
+    ) -> list[str]:
+        """Track the frame shown at the previous analyzer turn and, when
+        enabled, produce the auto-injected frame-diff lines and/or the diff
+        image for the outgoing user message. Suppressed on the first turn and
+        on the turn after a game over (the auto-RESET makes that span
+        meaningless)."""
+        if not (_AUTO_FRAME_DIFF or _DIFF_IMAGE or _level_inventory_enabled()):
+            return []
+        # Imported lazily so this patch's hunks stay clear of the module
+        # import header, which other optional patches also modify.
+        from inference.agent.vision_context import diff_image_part
+        from inference.utils.frame_diff import compute_frame_diff
+        from inference.utils.grid_utils import ARC_COLOR_CHARS
+        from inference.utils.segmentation import segment_layer
+        self._pending_diff_image = None
+        self._pending_diff_image_label = ""
+        # a new opener supersedes the previous turn's diff whether or not it
+        # produces one of its own
+        self._turn_diff_image = None
+        self._turn_diff_image_label = ""
+        if current_frame is None:
+            return []
+        previous_frame = self._auto_diff_prev_frame
+        self._auto_diff_prev_frame = current_frame
+        if previous_frame is None:
+            self._cur_level_start_frame = current_frame
+            return []
+        prev_level = getattr(previous_frame, "level", None)
+        cur_level = getattr(current_frame, "level", None)
+        level_changed = (
+            prev_level is not None
+            and cur_level is not None
+            and prev_level != cur_level
+        )
+        if level_changed:
+            # shift the per-level start frames even when a game over follows:
+            # the current frame is this level's (near-)initial board
+            self._prev_level_start_frame = self._cur_level_start_frame
+            self._cur_level_start_frame = current_frame
+        if previous_step_summary and previous_step_summary.get("game_over"):
+            return []
+        if level_changed:
+            if not _level_inventory_enabled():
+                # still clear the diff image: a cross-level diff has no meaning
+                self._pending_diff_image = None
+                self._pending_diff_image_label = ""
+                return []
+            return self._level_transition_diff_lines(
+                current_frame, previous_step_summary
+            )
+        if not (_AUTO_FRAME_DIFF or _DIFF_IMAGE):
+            # reached only when the inventory knob alone kept us here
+            return []
+        prev_step = getattr(previous_frame, "step", None)
+        cur_step = getattr(current_frame, "step", None)
+        span = f"step {prev_step} -> {cur_step}"
+        if _DIFF_IMAGE:
+            part = diff_image_part(previous_frame, current_frame)
+            if part is not None and previous_frame.grid != current_frame.grid:
+                label = (
+                    f"Diff image ({span}): cells that changed since your previous "
+                    "analyzer turn in their new true color; unchanged cells dimmed "
+                    "to dark navy (an off-palette color)."
+                )
+                self._pending_diff_image = part
+                self._pending_diff_image_label = label
+                # kept beyond the single use above so a resumption can re-show
+                # it; replaced whenever a new opener produces a fresh diff, so
+                # a stale one from an earlier turn can never leak in
+                self._turn_diff_image = part
+                self._turn_diff_image_label = label
+        if not _AUTO_FRAME_DIFF:
+            return []
+        executed = 0
+        if previous_step_summary and not previous_step_summary.get("stale"):
+            # a stale summary describes an earlier exchange whose changes were
+            # already diffed then; nothing has executed since, so don't claim
+            # "no cell changed despite N executed actions"
+            try:
+                executed = int(previous_step_summary.get("executed_count") or 0)
+            except (TypeError, ValueError):
+                executed = 0
+        if previous_frame.grid == current_frame.grid:
+            if executed > 0:
+                return [
+                    f"Auto frame diff ({span}): no cell changed since your previous "
+                    f"analyzer turn despite {executed} executed action(s)."
+                ]
+            return []
+        prev_rows = len(previous_frame.grid)
+        cur_rows = len(current_frame.grid)
+        prev_cols = max((len(r) for r in previous_frame.grid), default=0)
+        cur_cols = max((len(r) for r in current_frame.grid), default=0)
+        if (prev_rows, prev_cols) != (cur_rows, cur_cols):
+            return [
+                f"Auto frame diff ({span}): grid shape changed "
+                f"({prev_rows}x{prev_cols} -> {cur_rows}x{cur_cols}); diff skipped."
+            ]
+        diff = compute_frame_diff(
+            previous_frame.grid,
+            current_frame.grid,
+            segment_layer(previous_frame.grid, ARC_COLOR_CHARS).get("nodes", []),
+            segment_layer(current_frame.grid, ARC_COLOR_CHARS).get("nodes", []),
+            max_group_match=_AUTO_FRAME_DIFF_MAX_GROUP,
+        )
+        rendered = _render_auto_frame_diff(
+            diff, _AUTO_FRAME_DIFF_BUDGET, total_cells=cur_rows * cur_cols
+        )
+        lines = [
+            f"Auto frame diff ({span}; covers everything since your previous "
+            "analyzer turn, not just the last action):"
+        ]
+        lines.extend(rendered)
+        return lines
+
+    def _death_ledger_record(self, history_entries: list["HistoryEntry"]) -> None:
+        entries = list(history_entries or [])
+        if len(entries) < 2 or str(entries[-1].action or "").upper() != "RESET":
+            return
+        death_idx = len(entries) - 2
+        death = entries[death_idx]
+        if death.frame is None or str(death.action or "").upper() == "RESET":
+            return
+        death_level = getattr(death.frame, "level", None)
+        if (
+            self._death_ledger_level is not None
+            and death_level is not None
+            and death_level != self._death_ledger_level
+        ):
+            self._death_ledger_attempts = []
+            self._death_ledger_index = {}
+        if death_level is not None:
+            self._death_ledger_level = death_level
+        start_idx = None
+        for idx in range(death_idx - 1, -1, -1):
+            entry = entries[idx]
+            if str(entry.action or "").upper() == "RESET":
+                start_idx = idx
+                break
+            level = (
+                getattr(entry.frame, "level", None)
+                if entry.frame is not None
+                else None
+            )
+            if (
+                death_level is not None
+                and level is not None
+                and level != death_level
+            ):
+                start_idx = idx + 1
+                break
+        partial = start_idx is None
+        if partial:
+            start_idx = 0
+        actions = [
+            str(entries[i].action) for i in range(start_idx + 1, death_idx + 1)
+        ]
+        frames = [entries[i].frame for i in range(start_idx, death_idx)]
+        trim = 0
+        for i in range(len(frames) - 1, -1, -1):
+            if frames[i] is None:
+                trim = i + 1
+                break
+        if trim:
+            frames = frames[trim:]
+            actions = actions[trim:]
+            partial = True
+        if not actions or len(frames) != len(actions):
+            return
+        for item in self._death_ledger_attempts:
+            if item["actions"] == actions:
+                self._death_ledger_total += 1
+                item["ids"].append(self._death_ledger_total)
+                return
+        self._death_ledger_total += 1
+        uidx = len(self._death_ledger_attempts)
+        hashes = [_interior_state_hash(f.grid) for f in frames]
+        self._death_ledger_attempts.append(
+            {
+                "actions": actions,
+                "hashes": hashes,
+                "ids": [self._death_ledger_total],
+                "partial": partial,
+            }
+        )
+        for pos, state_hash in enumerate(hashes):
+            self._death_ledger_index.setdefault(state_hash, []).append((uidx, pos))
+
+    def _death_ledger_payload(self) -> dict[str, Any]:
+        return {
+            "level": self._death_ledger_level,
+            "attempts": [
+                {
+                    "actions": list(item["actions"]),
+                    "state_hashes": list(item["hashes"]),
+                    "attempt_ids": list(item["ids"]),
+                    "partial": bool(item.get("partial")),
+                }
+                for item in self._death_ledger_attempts
+            ],
+        }
+
+    def _death_ledger_observe(
+        self,
+        history_entries: list["HistoryEntry"],
+        previous_step_summary: dict[str, Any] | None,
+    ) -> None:
+        """Record a death, if the previous sequence ended in one.
+
+        Called unconditionally rather than from the rendering path. The ledger
+        has TWO consumers - the advisory lines under ARC3_DEATH_LEDGER, and the
+        death guard's fatal lookup under ARC3_DEATH_REPEAT_GUARD - and the
+        recorder used to live inside the first. So enabling only the guard left
+        it consulting an index nothing ever wrote to: armed, checked, always
+        empty, every repeat of a fatal route allowed through. Observed as three
+        identical deaths in a row with the guard on.
+        """
+        if not (
+            previous_step_summary
+            and previous_step_summary.get("game_over")
+            and not previous_step_summary.get("stale")
+        ):
+            return
+        if not (
+            _get_env_bool("ARC3_DEATH_LEDGER", False)
+            or _get_env_bool("ARC3_DEATH_REPEAT_GUARD", False)
+        ):
+            # nothing reads it; skip the work rather than accumulate state
+            return
+        self._death_ledger_record(history_entries)
+
+    def _death_ledger_lines(
+        self,
+        current_frame: Frame | None,
+        history_entries: list["HistoryEntry"],
+        previous_step_summary: dict[str, Any] | None,
+    ) -> list[str]:
+        """Advisory display of recorded fatal continuations matching the
+        CURRENT interior state. Broad by design (bar/items may not be visible
+        in the interior), hence advisory phrasing; exact-sequence enforcement
+        is a separate concern."""
+        if current_frame is None:
+            return []
+        level = getattr(current_frame, "level", None)
+        if (
+            self._death_ledger_level is not None
+            and level is not None
+            and level != self._death_ledger_level
+        ):
+            self._death_ledger_attempts = []
+            self._death_ledger_index = {}
+            self._death_ledger_level = level
+        if not self._death_ledger_attempts:
+            return []
+        state_hash = _interior_state_hash(current_frame.grid)
+        matches: dict[int, int] = {}
+        for uidx, pos in self._death_ledger_index.get(state_hash, []):
+            matches[uidx] = max(matches.get(uidx, -1), pos)
+        if not matches:
+            return []
+        by_suffix: dict[tuple, list[int]] = {}
+        for uidx, pos in matches.items():
+            item = self._death_ledger_attempts[uidx]
+            suffix = tuple(item["actions"][pos:])
+            by_suffix.setdefault(suffix, []).extend(item["ids"])
+        ordered = sorted(by_suffix.items(), key=lambda kv: len(kv[0]))
+        total_deaths = sum(
+            len(item["ids"]) for item in self._death_ledger_attempts
+        )
+        lines = [
+            "Known fatal continuations FROM THE CURRENT BOARD STATE (this "
+            f"level has {total_deaths} failed attempt(s) recorded; the game "
+            "is deterministic):"
+        ]
+        for suffix, ids in ordered[:3]:
+            shown = ", ".join(suffix[:12]) + (", ..." if len(suffix) > 12 else "")
+            id_text = ", ".join(str(i) for i in sorted(set(ids)))
+            lines.append(
+                f"- {len(suffix)} action(s): {shown} -> GAME OVER "
+                f"(attempt(s) {id_text})"
+            )
+        if len(ordered) > 3:
+            lines.append(
+                f"- +{len(ordered) - 3} more matching fatal continuation(s)"
+            )
+        lines.append(
+            "Do NOT replay a listed continuation exactly - vary the route or "
+            "action at or before its first step. (Advisory: fatality can "
+            "depend on state not visible in the board interior, e.g. bar "
+            "level or held items; if you are certain the situation differs, "
+            "state why before proceeding.) The full ledger is available in "
+            "python as `death_ledger`."
+        )
+        return lines
+
+    def _level_transition_diff_lines(
+        self,
+        current_frame: Frame | None,
+        previous_step_summary: dict[str, Any] | None,
+    ) -> list[str]:
+        """First prompt of a new level: acknowledge the completing action and
+        render a START-vs-START inventory against the previous level instead
+        of the meaningless end-of-old-board vs new-board diff. The diff image
+        is suppressed for this prompt (no cross-level semantics)."""
+        from inference.utils.frame_diff import compute_frame_diff
+        from inference.utils.grid_utils import ARC_COLOR_CHARS
+        from inference.utils.segmentation import segment_layer
+
+        self._pending_diff_image = None
+        self._pending_diff_image_label = ""
+        lines: list[str] = []
+        final_action = ""
+        if previous_step_summary and not previous_step_summary.get("stale"):
+            actions = previous_step_summary.get("executed_actions")
+            if isinstance(actions, list) and actions:
+                final_action = str(actions[-1]).strip()
+        lines.append(
+            "Level completed - the board was replaced, so no object diff is "
+            "shown across the boundary"
+            + (f" (the final action {final_action!r} finished the level)" if final_action else "")
+            + "."
+        )
+        prev_start = self._prev_level_start_frame
+        cur_start = self._cur_level_start_frame
+        if prev_start is None or cur_start is None:
+            return lines
+        grid_a, grid_b = prev_start.grid, cur_start.grid
+        rows = len(grid_b)
+        cols = max((len(r) for r in grid_b), default=0)
+        if len(grid_a) != rows or max(
+            (len(r) for r in grid_a), default=0
+        ) != cols:
+            lines.append(
+                "New level inventory skipped (boards have different shapes)."
+            )
+            return lines
+        diff = compute_frame_diff(
+            grid_a,
+            grid_b,
+            segment_layer(grid_a, ARC_COLOR_CHARS).get("nodes", []),
+            segment_layer(grid_b, ARC_COLOR_CHARS).get("nodes", []),
+            max_group_match=_AUTO_FRAME_DIFF_MAX_GROUP,
+        )
+        lines.append(
+            "New level inventory (previous level START vs this level START; "
+            "read it as a TYPE comparison, not motion: 'moved' = object types "
+            "recurring from the previous level at a new spot, 'appeared' = "
+            "types NOT present on the previous level - likely new mechanics, "
+            "investigate these first, 'disappeared' = previous types absent "
+            "here; identical objects at identical positions are not listed):"
+        )
+        lines.extend(
+            _render_auto_frame_diff(
+                diff, _AUTO_FRAME_DIFF_BUDGET, total_cells=rows * cols
+            )
+        )
+        return lines
+
+    def _prepare_gameover_images(
+        self,
+        history_entries: list["HistoryEntry"],
+        previous_step_summary: dict[str, Any] | None = None,
+    ) -> None:
+        """Queue the death-turn visuals (ARC3_GAMEOVER_DIFF_IMAGE).
+
+        The auto-diff - and with it the diff image - is suppressed after a game
+        over because the auto-RESET makes that span meaningless, so the visual
+        slot is idle on exactly the turn where the death is invisible: the
+        board the model sees has already been reset. Two parts fill it:
+
+          1. the death frame itself (the budget bar in it is what the BAR RULE
+             asks about, and it exists nowhere else in the message), and
+          2. a change-highlight diff of the killing action (last alive frame ->
+             death frame), the spatial form of the textual 'Fatal step diff'.
+
+        Anchoring matches _game_over_diff_lines: history ends with the RESET
+        marker, entries[-2] is the death frame, entries[-3] the last alive one.
+        """
+        self._pending_gameover_images = []
+        if not _get_env_bool("ARC3_GAMEOVER_DIFF_IMAGE", False):
+            return
+        from inference.agent.vision_context import (
+            current_grid_image_part,
+            diff_image_part,
+        )
+
+        entries = list(history_entries or [])
+        if len(entries) < 3 or str(entries[-1].action or "").upper() != "RESET":
+            return
+        death = entries[-2]
+        last_alive = entries[-3]
+        if death.frame is None or str(death.action or "").upper() == "RESET":
+            return
+        death_part = current_grid_image_part(death.frame)
+        if death_part is not None:
+            self._pending_gameover_images.append(
+                (
+                    "Death frame image (the board at the moment the attempt "
+                    "ended, before the automatic reset): inspect the budget bar "
+                    "here for the BAR RULE.",
+                    death_part,
+                )
+            )
+        self._append_fatal_animation_image(previous_step_summary)
+        if last_alive.frame is None:
+            return
+        fatal_part = diff_image_part(last_alive.frame, death.frame)
+        if fatal_part is not None and last_alive.frame.grid != death.frame.grid:
+            self._pending_gameover_images.append(
+                (
+                    "Fatal step diff image (last alive frame -> death frame): "
+                    "cells changed by the killing action in their new true "
+                    "color; unchanged cells dimmed to dark navy (an off-palette "
+                    "color).",
+                    fatal_part,
+                )
+            )
+
+    def _game_over_diff_lines(
+        self,
+        history_entries: list["HistoryEntry"],
+        previous_step_summary: dict[str, Any] | None = None,
+    ) -> list[str]:
+        """After a game over (fresh rendering only), push two diffs over the
+        fatal sequence itself: sequence progress (frame before the sequence's
+        first action -> last alive frame, omitted when fewer than two actions
+        executed) and the fatal step (last alive -> death frame). Anchored by
+        the summary's executed_count - the turns before the fatal sequence
+        already delivered their own diffs while the model was alive."""
+        from inference.utils.frame_diff import compute_frame_diff
+        from inference.utils.grid_utils import ARC_COLOR_CHARS
+        from inference.utils.segmentation import segment_layer
+
+        entries = list(history_entries or [])
+        if len(entries) < 2 or str(entries[-1].action or "").upper() != "RESET":
+            return []
+        death_idx = len(entries) - 2
+        death = entries[death_idx]
+        if death.frame is None or str(death.action or "").upper() == "RESET":
+            return []
+        executed = 0
+        if previous_step_summary:
+            try:
+                executed = int(previous_step_summary.get("executed_count") or 0)
+            except (TypeError, ValueError):
+                executed = 0
+        if executed < 1:
+            executed = 1
+        start_idx = death_idx - executed
+        partial = start_idx < 0
+        if partial:
+            start_idx = 0
+        origin = "earliest retained frame" if partial else "sequence start"
+        death_level = getattr(death.frame, "level", None)
+        while start_idx < death_idx - 1:
+            frame = entries[start_idx].frame
+            level = getattr(frame, "level", None) if frame is not None else None
+            if (
+                frame is None
+                or (
+                    death_level is not None
+                    and level is not None
+                    and level != death_level
+                )
+            ):
+                # the sequence itself crossed a level-up (or a frame is
+                # missing); anchor at the first same-level frame within it
+                start_idx += 1
+                origin = "level start within the sequence"
+                continue
+            break
+        start = entries[start_idx]
+        last_alive = entries[death_idx - 1]
+        if start.frame is None or last_alive.frame is None:
+            return []
+
+        def _diff_lines(frame_a, frame_b, label: str) -> list[str]:
+            grid_a, grid_b = frame_a.grid, frame_b.grid
+            rows = len(grid_b)
+            cols = max((len(r) for r in grid_b), default=0)
+            if len(grid_a) != rows or max(
+                (len(r) for r in grid_a), default=0
+            ) != cols:
+                return [f"{label} skipped (grid shape changed)."]
+            diff = compute_frame_diff(
+                grid_a,
+                grid_b,
+                segment_layer(grid_a, ARC_COLOR_CHARS).get("nodes", []),
+                segment_layer(grid_b, ARC_COLOR_CHARS).get("nodes", []),
+                max_group_match=_AUTO_FRAME_DIFF_MAX_GROUP,
+            )
+            rendered = _render_auto_frame_diff(
+                diff, _AUTO_FRAME_DIFF_BUDGET, total_cells=rows * cols
+            )
+            return [label, *rendered]
+
+        out: list[str] = []
+        pre_death_actions = (death_idx - 1) - start_idx
+        if pre_death_actions >= 1:
+            out.extend(
+                _diff_lines(
+                    start.frame,
+                    last_alive.frame,
+                    f"Sequence progress diff ({origin} step "
+                    f"{getattr(start.frame, 'step', '?')} -> step "
+                    f"{getattr(last_alive.frame, 'step', '?')}, what the first "
+                    f"{pre_death_actions} action(s) of the fatal sequence "
+                    "changed - the reset reverted all of it):",
+                )
+            )
+        out.extend(
+            _diff_lines(
+                last_alive.frame,
+                death.frame,
+                f"Fatal step diff (step {getattr(last_alive.frame, 'step', '?')} "
+                f"-> step {getattr(death.frame, 'step', '?')}, what the killing "
+                f"action {str(death.action)!r} changed):",
+            )
+        )
+        return out
+
+    def _append_fatal_animation_image(self, summary: dict[str, Any] | None) -> None:
+        """An image of what the fatal action's animation hid, on a game over.
+
+        Gated on transient pixels alone, NOT on gameplay_changed: after a death
+        the board has already reset, so gameplay_changed compares the
+        pre-action board against the restarted level and is essentially always
+        true. The justification differs from the ordinary case too - there the
+        board shows nothing, here it shows a different board entirely, and
+        either way the fatal moment is unreachable.
+        """
+        if not _animation_enabled():
+            return
+        mode = animation_death_image_mode()
+        if mode == "off" or not summary:
+            return
+        animation = summary.get("animation") or {}
+        if not int(animation.get("transient_pixels") or 0):
+            return
+        chain = self._last_animation_chain or []
+        cells = _animation_transient_cells(chain)
+        if not cells:
+            return
+        if mode == "composite":
+            part = animation_composite_part(
+                chain, cells, border=_get_env_int("ARC3_NOOP_GUARD_BORDER", 4)
+            )
+            label = (
+                "Fatal action animation image: the cells that changed during the fatal "
+                "action's animation, each in the last colour it held before reverting; "
+                "everything else, including the border, is dimmed to dark navy (an "
+                "off-palette colour). This is a composite of all cells that changed "
+                "during the animation, not a single board state that the game "
+                "displayed. A cell is drawn nearer its true colour the longer it held "
+                "it, so a bright trail marks where something stayed and a uniformly "
+                "dim one marks something that passed through. Neither the death frame "
+                "nor the last alive frame contains these cells - they existed only "
+                "between the two."
+            )
+        else:
+            part, index = animation_peak_part(chain, cells)
+            label = (
+                f"Fatal action animation image (frame {index} of {len(chain) - 1}): the "
+                "board partway through the fatal action's animation, at the point where "
+                "the most cells were showing something they later reverted. This is a "
+                "real state the game displayed, between the last alive frame and the "
+                "death frame."
+            )
+        if part is not None:
+            self._pending_gameover_images.append((label, part))
+
+    def _prepare_animation_image(self, summary: dict[str, Any] | None) -> None:
+        """Attach an image of what the animation hid, when the board did not move.
+
+        Gated on gameplay_changed being false: that is exactly when the board
+        image the model receives is uninformative, because the action did
+        something and no frame it can reach shows what. On any other turn the
+        current board already carries the outcome.
+        """
+        if not _animation_enabled():
+            # ARC3_ANIMATION_IMAGE needs ARC3_ANIMATION: without it no chain is
+            # captured, so this would silently do nothing. Stated rather than
+            # left implicit - a knob that quietly has no effect is the failure
+            # mode that costs the most time to diagnose.
+            return
+        mode = animation_image_mode()
+        if mode == "off" or not summary:
+            return
+        if self._last_action_gameplay_changed is not False:
+            return
+        animation = summary.get("animation") or {}
+        if not int(animation.get("transient_pixels") or 0):
+            return
+        chain = self._last_animation_chain or []
+        cells = _animation_transient_cells(chain)
+        if not cells:
+            return
+        if mode == "composite":
+            part = animation_composite_part(
+                chain, cells, border=_get_env_int("ARC3_NOOP_GUARD_BORDER", 4)
+            )
+            label = (
+                "Animation image: the cells that changed during the animation, each in "
+                "the last colour it held before reverting; everything else, including "
+                "the border, is dimmed to dark navy (an off-palette colour). This is a "
+                "composite of all cells that changed during the animation, not a single "
+                "board state that the game displayed. A cell is drawn nearer its true "
+                "colour the longer it held it, so a bright trail marks where something "
+                "stayed and a uniformly dim one marks something that passed through. "
+                "The action left the board area unchanged, so none of these cells "
+                "appear in `current_frame`."
+            )
+        else:
+            part, index = animation_peak_part(chain, cells)
+            label = (
+                f"Animation image (frame {index} of {len(chain) - 1}): the board partway "
+                "through the animation, at the point where the most cells were showing "
+                "something they later reverted. This is a real state the game displayed, "
+                "but not one that survives into `current_frame`."
+            )
+        if part is not None:
+            self._pending_animation_image = part
+            self._pending_animation_image_label = label
+
+    def _retained_function_context(self) -> str:
+        """Optional prompt metadata must never interrupt game execution."""
+        if not _persistent_functions():
+            return ""
+        try:
+            notice = getattr(self, "_retained_clear_notice", "")
+            signatures = [function_signature(src) for src in self._kept_functions.values()]
+            parts = [notice] if notice else []
+            if signatures:
+                scope = {"levelup": "on this level", "turn": "in this turn", "game": "in this game"}[_persistent_functions_scope()]
+                parts.append(
+                    "Your retained functions from earlier Python calls " + scope + ": "
+                    + ", ".join(signatures) + ". These are your previous definitions, "
+                    "available to call directly. Their presence does not mean they are "
+                    "correct; revise them if new evidence contradicts their assumptions."
+                )
+            self._retained_clear_notice = ""
+            return "\n\n" + "\n".join(parts) if parts else ""
+        except Exception:
+            self._kept_functions = {}
+            self._retained_clear_notice = ""
+            return "\n\nYour retained functions were cleared after a retention error; redefine any you need."
+
+    def _retained_sources(self):
+        if not _persistent_functions():
+            return None
+        try:
+            return list(self._kept_functions.values())
+        except Exception:
+            self._kept_functions = {}
+            return []
+
+    def _record_retained_functions(self, sandbox_result, payload) -> None:
+        if not _persistent_functions():
+            return
+        try:
+            previous = self._kept_functions
+            kept = {}
+            signatures = {}
+            for item in sandbox_result.get("keepable_functions") or []:
+                name, source = item["name"], item["source"]
+                signature = function_signature(source)
+                if signature.split("(", 1)[0] != name:
+                    raise ValueError("invalid retained function metadata")
+                kept[name] = source
+                signatures[name] = signature
+            notes = []
+            scope = {"levelup": "on this level", "turn": "in this turn", "game": "in this game"}[_persistent_functions_scope()]
+            for name, source in kept.items():
+                if previous.get(name) != source:
+                    notes.append("Retained your function " + signatures[name] + " for later Python calls " + scope + ".")
+            rejected = sandbox_result.get("retention_rejected") or []
+            repair_hints = _get_env_bool("ARC3_PERSISTENT_FUNCTIONS_REPAIR_HINTS", False)
+            rejected_names = set()
+            seen_rejections = set()
+            for item in rejected:
+                name, reason = str(item["name"]), str(item["reason"])
+                hint = (str(item.get("hint") or "").strip() if repair_hints else
+                        "Pass snippet-specific data as arguments or redefine it if needed.")
+                rejection_key = (name, reason, hint)
+                if rejection_key in seen_rejections:
+                    continue
+                seen_rejections.add(rejection_key)
+                rejected_names.add(name)
+                if name == "all":
+                    notes.append("Your retained functions were cleared: " + reason + ".")
+                    continue
+                notes.append("Your function " + name + " was not retained: " + reason + "."
+                             + (" " + hint if hint else ""))
+            for name in previous.keys() - kept.keys() - rejected_names:
+                notes.append("Your previous function " + name + " is no longer retained; redefine it if needed.")
+            self._kept_functions = kept
+            if notes:
+                payload["function_retention"] = "\n".join(notes)
+        except Exception:
+            self._kept_functions = {}
+            payload["function_retention"] = "Your retained functions were cleared after a retention error; redefine any you need."
+
     def _build_user_message(self, user_prompt: str, current_frame: Frame | None) -> dict[str, Any]:
+        user_prompt += self._retained_function_context()
         image_part = current_grid_image_part(current_frame)
+        diff_part = self._pending_diff_image
+        diff_label = self._pending_diff_image_label
+        animation_part = self._pending_animation_image
+        animation_label = self._pending_animation_image_label
+        gameover_images = list(self._pending_gameover_images)
+        self._pending_diff_image = None
+        self._pending_diff_image_label = ""
+        self._pending_animation_image = None
+        self._pending_animation_image_label = ""
+        self._pending_gameover_images = []
+        if animation_part is not None:
+            # the animation image only ever fires when nothing inside the board
+            # moved, which is precisely when the diff image shows the ticking
+            # budget bar and nothing else - two images, one of them noise
+            diff_part = None
         if image_part is None:
             return {"role": "user", "content": user_prompt}
 
-        return {
-            "role": "user",
-            "content": [
-                {"type": "text", "text": f"{user_prompt}\n\nCurrent grid image:"},
-                image_part,
-            ],
-        }
+        content: list[dict[str, Any]] = [
+            {"type": "text", "text": f"{user_prompt}\n\nCurrent grid image (the board AFTER the automatic reset - this is what you act on now):" if gameover_images else f"{user_prompt}\n\nCurrent grid image:"},
+            image_part,
+        ]
+        if diff_part is not None:
+            content.append({"type": "text", "text": diff_label})
+            content.append(diff_part)
+        if animation_part is not None:
+            content.append({"type": "text", "text": animation_label})
+            content.append(animation_part)
+        for label, part in gameover_images:
+            content.append({"type": "text", "text": label})
+            content.append(part)
+        return {"role": "user", "content": content}
+
+
+    def _memory_section_guidance(self, block_shown: bool) -> list[str]:
+        """The section explanation, in upstream's slot just before the tool-call
+        format guidance.
+
+        Returns nothing when the mechanic is off. In 'full' mode the wording
+        depends on whether a stored block was shown: the REPLACES warning is
+        meaningless when there is nothing stored to replace, so a first turn or
+        a post-wipe turn gets the format glossary without it."""
+        if _memory_sections_disabled():
+            return []
+        if _wm_section_explanation_mode() == "upstream":
+            return [_memory_section_guidance_upstream()]
+        if not block_shown:
+            return [_memory_section_guidance_empty()]
+        if _memory_section_labels() is None:
+            return [_MEMORY_SECTION_GUIDANCE_FULL]
+        # a mode carrying a subset needs the subset described, not the fixed
+        # two-section text: `medium` carries three
+        return [
+            _memory_section_guidance_empty()
+            + " IMPORTANT: each labeled section you write REPLACES the stored version shown "
+            "under 'Working world model carried from earlier turns' - write the COMPLETE "
+            "updated section; any fact you omit is permanently lost."
+        ]
+
+    @staticmethod
+    def _manual_reset_context_line(summary: dict[str, Any]) -> str | None:
+        if not reset_exposed() or any(summary.get(flag) for flag in (
+            "stale", "game_over", "run_complete", "done", "level_transition", "level_completed",
+        )):
+            return None
+        actions = [str(name).strip().upper() for name in summary.get("executed_actions") or []]
+        if "RESET" not in actions:
+            return None
+        if actions == ["RESET"]:
+            return (
+                "You deliberately reset the current level. The current frame shows "
+                "the state returned by RESET."
+            )
+        return (
+            "You deliberately reset the current level, then executed the subsequent "
+            "actions listed above. The current frame shows their resulting state."
+        )
+
+    def _summary_turn_context_lines(self) -> list[str]:
+        """What just happened, for the summary request.
 
+        Deliberately not _build_user_prompt: that consumes _pending_diff_image,
+        _pending_animation_image and the wipe-nudge state, so calling it twice
+        would swallow images the real opener should carry. This reads
+        _last_step_summary and mutates nothing - the stale flag is set by tool
+        dispatch, never by building a prompt, so an extra read is safe.
+        """
+        summary = self._last_step_summary
+        if not summary or summary.get("stale"):
+            # nothing executed since the last opener; the block would say
+            # "Executed actions: none" and repeat an unchanged state line
+            return []
+        lines: list[str] = []
+        count = summary.get("executed_count")
+        try:
+            normalized = int(count) if count is not None else None
+        except (TypeError, ValueError):
+            normalized = None
+        if not normalized:
+            return []
+        label = "action" if normalized == 1 else "actions"
+        lines.append(f"The code executed {normalized} {label} in the previous sequence.")
+        rendered = summary.get("executed_actions") or []
+        if rendered:
+            prefix = (
+                "Executed actions:" if len(rendered) <= 10
+                else "Executed actions (first 10):"
+            )
+            lines.append(f"{prefix} {', '.join(str(a) for a in rendered[:10])}.")
+        changed = summary.get("gameplay_changed")
+        if _report_gameplay_changed():
+            if changed is True:
+                lines.append("That sequence changed cells inside the board area.")
+            elif changed is False and summary.get("board_changed"):
+                lines.append(
+                    "That sequence changed only cells near the edge - nothing inside "
+                    "the board area moved."
+                )
+            elif changed is False:
+                lines.append("That sequence produced no change anywhere on the board.")
+        for stop in (summary.get("no_op_stops") or [])[:3]:
+            lines.append(
+                f"A batched sequence was stopped by the no-op guard: "
+                f"{stop.get('action')!r} executed but changed nothing, and "
+                f"{stop.get('skipped', 0)} later action(s) were skipped."
+            )
+        for stop in (summary.get("guard_stops") or [])[:3]:
+            lines.append(
+                f"The sequence was stopped before {stop.get('action')!r}: "
+                f"{stop.get('reason', 'a guard refused it')}."
+            )
+        if summary.get("level_completed"):
+            lines.append("You have progressed to a new level!")
+        elif summary.get("game_over"):
+            lines.append("The attempt ended and the level was reset.")
+        else:
+            lines.append(self._manual_reset_context_line(summary) or "You are still on the same level.")
+        step = summary.get("action_num")
+        level = summary.get("level")
+        if step is not None and level is not None:
+            lines.append(f"Current state: step {step}, level {level}.")
+        return lines
 
     def _build_user_prompt(
         self,
@@ -1183,7 +4828,9 @@ class ToolAgent:
             default=current_level,
         )
         lines: list[str] = []
-        if previous_step_summary:
+        if previous_step_summary and previous_step_summary.get("stale"):
+            lines.extend(_stale_summary_lines(previous_step_summary))
+        elif previous_step_summary:
             count = previous_step_summary.get("executed_count")
             try:
                 normalized_count = int(count) if count is not None else None
@@ -1200,18 +4847,236 @@ class ToolAgent:
                 lines.append(f"{action_prefix} {', '.join(rendered_actions[:10])}.")
             else:
                 lines.append("Executed actions: none.")
+            if _report_gameplay_changed() and normalized_count:
+                # Whether the sequence MOVED anything. The harness computes this
+                # for every action - it drives the batch stop - and until now
+                # said nothing about it: `_describe_last_outcome` produces the
+                # sentence but is called nowhere, so neither its wording nor the
+                # older one it replaced has ever reached a transcript. On a
+                # single-action turn no per-action trace is built either, so the
+                # model had no account of the outcome from any source.
+                border = _get_env_int("ARC3_NOOP_GUARD_BORDER", 4)
+                if previous_step_summary.get("gameplay_changed"):
+                    lines.append("That sequence changed cells inside the board area.")
+                elif previous_step_summary.get("board_changed"):
+                    lines.append(
+                        f"That sequence changed only cells within {border} of the edge "
+                        "(usually a timer or remaining-steps bar) - nothing inside the "
+                        "board area moved."
+                    )
+                else:
+                    lines.append("That sequence produced no change anywhere on the board.")
+            if _animation_enabled() and not previous_step_summary.get("game_over"):
+                animation_line = describe_animation(
+                    previous_step_summary.get("animation"),
+                    action_display=previous_step_summary.get("animation_action"),
+                    # the animated action's own flag, not the sequence's: an
+                    # inert SPACE at the end of a batch that moved other things
+                    # still hid everything it did
+                    gameplay_changed=self._last_action_gameplay_changed,
+                    timeline=_animation_timeline_enabled(),
+                )
+                if animation_line:
+                    lines.append(animation_line)
+                self._prepare_animation_image(previous_step_summary)
+            no_op_stops = previous_step_summary.get("no_op_stops") or []
+            for stop in no_op_stops[:3]:
+                skipped = stop.get("skipped") or []
+                skipped_text = (
+                    f" The remaining {len(skipped)} action(s) {skipped!r} were skipped."
+                    if skipped
+                    else ""
+                )
+                lines.append(
+                    f"Note: a batched sequence was stopped by the no-op guard: action "
+                    f"{stop.get('action')!r} (position {stop.get('index')} in that batch) "
+                    + (f"executed but produced no change inside the board area.{skipped_text} "
+                       if _new_noop_wording() else
+                       f"executed but changed nothing outside the border.{skipped_text} ")
+                    +
+                    "That action IS included in the executed list above, so it is the "
+                    f"LAST of them - only the first {max(0, int(stop.get('index') or 1) - 1)} "
+                    "executed action(s) moved anything. Do not count submitted or executed "
+                    "actions to update your position; re-locate from current_frame."
+                )
+                trace = stop.get("trace") or []
+                if trace:
+                    lines.append("Per-action trace: " + "; ".join(str(t) for t in trace) + ".")
+            if len(no_op_stops) > 3:
+                lines.append(
+                    f"({len(no_op_stops) - 3} further no-op guard stop(s) occurred this sequence.)"
+                )
+            guard_stops = previous_step_summary.get("guard_stops") or []
+            for stop in guard_stops[:3]:
+                skipped = stop.get("skipped") or []
+                skipped_text = (
+                    f" The remaining {len(skipped)} action(s) ({skipped!r}) were never executed."
+                    if skipped
+                    else ""
+                )
+                if stop.get("kind") == "stale_state":
+                    # the exception said this in the turn where it happened;
+                    # restated here because the plan being void is worth
+                    # carrying into the turn that has to replace it
+                    lines.append(
+                        f"Note: the sequence was stopped before {stop.get('action')!r} - an "
+                        + ("earlier action in that snippet produced no change inside the "
+                           "board area, so the positions and "
+                           if _new_noop_wording() else
+                           "earlier action in that snippet changed nothing, so the positions and ")
+                        +
+                        f"paths it had computed may have been wrong.{skipped_text} Re-ground on "
+                        "the current board before planning again."
+                    )
+                elif stop.get("kind") == "known_death":
+                    lines.append(
+                        f"Note: the sequence was stopped before {stop.get('action')!r} - that "
+                        "action ended a previous attempt from that exact board state, so it was "
+                        f"NOT executed and the attempt is intact.{skipped_text} Plan a different "
+                        "move, or repeat it deliberately if you now believe it is safe."
+                    )
+                else:
+                    lines.append(
+                        f"Note: the sequence was stopped before {stop.get('action')!r} - that "
+                        + ("action already produced no change inside the board area in that "
+                           "exact board state, so it was "
+                           if _new_noop_wording() else
+                           "action already changed nothing in that exact board state, so it was ")
+                        +
+                        f"NOT executed and no action was spent.{skipped_text}"
+                    )
+                trace = stop.get("trace") or []
+                if trace:
+                    lines.append("Per-action trace: " + "; ".join(str(t) for t in trace) + ".")
+            if len(guard_stops) > 3:
+                lines.append(
+                    f"({len(guard_stops) - 3} further guard stop(s) occurred this sequence.)"
+                )
             if previous_step_summary.get("run_complete"):
                 lines.append("You have completed the run!")
             elif previous_step_summary.get("level_transition"):
-                lines.append("You have progressed to a new level!")
+                lines.append(
+                    LEVEL_START_USER_PROMPT
+                    if _level_transfer_guidance_enabled()
+                    else "You have progressed to a new level!"
+                )
             else:
-                lines.append("You are still on the same level.")
+                lines.append(
+                    self._manual_reset_context_line(previous_step_summary)
+                    or "You are still on the same level."
+                )
             if previous_step_summary.get("game_over"):
-                lines.append("The game is over.")
+                lines.append(
+                    "GAME OVER occurred during the previous sequence. This means the "
+                    "attempt FAILED (it does NOT mean the run is finished). The game has "
+                    "already been automatically RESET: the current level restarted from "
+                    "its initial state and previously completed levels are kept. A game "
+                    "over is caused either by the specific action taken (e.g., moving "
+                    "onto a hazard cell) or by a step/time budget running out, which is "
+                    "usually indicated by a bar at the grid border that shrinks with each "
+                    "action. Before acting again, diagnose the cause by inspecting the "
+                    "board right before the death. `history[-1].action` is 'RESET', "
+                    "`history[-2].action` is the fatal action, `history[-2].frame` is the "
+                    "death frame (the board after the fatal action), and "
+                    "`history[-3].frame` is the board immediately before "
+                    "the fatal action — check what cell the fatal move targeted there, and "
+                    "inspect the budget bar in the death frame. BAR RULE: the bar "
+                    "shrinking with each action is NORMAL — a slightly shorter bar is "
+                    "NOT evidence of the cause of death. The budget caused the death "
+                    "ONLY if the bar is fully (or almost fully) depleted in the death "
+                    "frame (`history[-2].frame`). If it IS depleted: you must reach the "
+                    "goal in fewer actions, or look for a way to restore the bar (e.g., "
+                    "collecting an item that refills it). If the bar clearly had budget "
+                    "remaining: the fatal action itself caused the death (hazard or "
+                    "forbidden move) — do NOT retry the same path; a DIFFERENT approach "
+                    "must be tried. Record the diagnosed cause in your world model."
+                )
+                if _animation_enabled():
+                    # after the BAR RULE, which tells the model how to read
+                    # history[-2] and history[-3]; this says what those two
+                    # frames cannot contain
+                    death_animation_line = describe_animation(
+                        previous_step_summary.get("animation"),
+                        action_display=previous_step_summary.get("animation_action"),
+                        # the animated action's own flag, not the sequence's: an
+                    # inert SPACE at the end of a batch that moved other things
+                    # still hid everything it did
+                    gameplay_changed=self._last_action_gameplay_changed,
+                        game_over=True,
+                        timeline=_animation_timeline_enabled(),
+                    )
+                    if death_animation_line:
+                        lines.append(death_animation_line)
+                fatal_action = str(previous_step_summary.get("fatal_action") or "")
+                if fatal_action:
+                    lines.append(
+                        f"The game over occurred immediately after action {fatal_action!r}. "
+                        "Apply the BAR RULE above to decide: bar fully depleted in the "
+                        "death frame means budget death (shorter route or bar restore "
+                        "needed); bar with budget remaining means this action itself was "
+                        "fatal and a DIFFERENT approach is required."
+                    )
+                game_over_trace = previous_step_summary.get("game_over_trace") or []
+                if game_over_trace:
+                    lines.append(
+                        "Per-action trace of the previous sequence: "
+                        + "; ".join(str(t) for t in game_over_trace)
+                        + "."
+                    )
+                self._prepare_gameover_images(
+                    history_entries, previous_step_summary
+                )
+                if _get_env_bool("ARC3_GAMEOVER_DIFF", False):
+                    # read at call time so this patch's hunks stay clear of the
+                    # env-knob block that other patches extend
+                    lines.extend(
+                        self._game_over_diff_lines(
+                            history_entries, previous_step_summary
+                        )
+                    )
+                lines.append(
+                    "Do NOT re-submit this same series of moves: the game is "
+                    "deterministic, so replaying the sequence that just killed you "
+                    "will kill you again. Change the plan BEFORE the fatal point - a "
+                    "different route, action, or timing. If you believe the same "
+                    "moves 'should' work, that belief is exactly what this death "
+                    "falsified; record the correction in your world model instead "
+                    "of retesting it."
+                )
         elif (current_frame is not None and current_frame.step > 0) or action_num > 0:
             lines.append("No previous action sequence was captured.")
         else:
             lines.append("No previous sequence has been executed yet.")
+        if self._pending_wm_rebuild_reason and _wm_rebuild_nudge_enabled():
+            lines.append(
+                f"IMPORTANT: your memory sections were reset by {self._pending_wm_rebuild_reason} "
+                "(cross-level notes were kept). Rebuild them NOW: write fresh "
+                # naming a section a restricted mode drops would ask for a
+                # rebuild that is parsed and then thrown away
+                f"{_section_list_phrase(_advertised_section_labels())} sections for this "
+                "level in your next message, before or alongside your tool call."
+            )
+        elif (
+            _wm_revise_nudges_enabled()
+            and getattr(self, "_pending_wm_revise_reason", "") in ("level", "death")
+        ):
+            _revise_reason = getattr(self, "_pending_wm_revise_reason", "")
+            lines.append(
+                _wm_revise_hint(
+                    _revise_reason,
+                    getattr(self, "_wm_revise_hint_shown", False),
+                )
+            )
+            self._wm_revise_hint_shown = True
+        elif _WM_NUDGE_TURNS > 0 and self._turns_without_wm_update >= _WM_NUDGE_TURNS:
+            lines.append(
+                f"You have not updated any memory section for "
+                f"{self._turns_without_wm_update} analysis steps. Review what recent "
+                "action traces, stops, and observations taught you, and write updated "
+                "sections now - knowledge that is not written down is eventually lost "
+                "to context eviction."
+            )
+        lines.extend(self._prepare_auto_diff(current_frame, previous_step_summary))
         state_line = f"Current state: step {current_step}, level {current_level}"
         if observed_max_level > current_level:
             state_line += f" out of observed max level {observed_max_level} so far"
@@ -1220,21 +5085,65 @@ class ToolAgent:
             [
                 state_line,
                 f"Valid actions right now: {_format_valid_action_line(valid_actions)}.",
-                "Only tool: `python`. It receives `current_frame`, `previous_frame`, `history`, `transitions`, `last_transition`, `valid_actions`, `last_action_result`, and `action(actions)`.",
+            ]
+        )
+        # recording is independent of display: the guard needs the ledger even
+        # when the advisory lines are switched off
+        self._death_ledger_observe(history_entries, previous_step_summary)
+        if _get_env_bool("ARC3_DEATH_LEDGER", False):
+            # read at call time so this patch's hunks stay clear of the
+            # env-knob block that other patches extend; placed directly after
+            # the valid-actions line so the fatal continuations sit adjacent
+            # to the action choice they constrain
+            lines.extend(
+                self._death_ledger_lines(
+                    current_frame, history_entries, previous_step_summary
+                )
+            )
+        lines.extend(
+            [
+                "Only tool: `python`. It receives `current_frame`, `previous_frame`, `history`, `transitions`, `last_transition`, `valid_actions`, `last_action_call_result`, `frame_diff(before, after)`, and `action(actions)`.",
                 "Only letter-coded board views and lightweight metadata are exposed; raw numeric color IDs are not available.",
                 "Keep tool output compact: use `current_frame.segmentation` as the primary view, and `current_frame.ascii` only for a small specific region; never print full boards.",
                 "For the most recent change, compare `previous_frame` to `current_frame`, or `last_transition.before_frame` to `last_transition.after_frame`; `history[-1].frame` is the current frame, not the previous one.",
-                "Use Python to inspect the evidence, refine that world model from the newest history, and search or score candidate actions or short sequences against the current goal as you currently understand it.",
-                "Maintain a compact working world model of what the current level seems to contain, what actions appear to do, what the goal seems to be, what is still uncertain, and what plan currently looks best.",
-                "Below you are provided with the current world model from the previous turn. The default behavior is to copy it and add or remove things based on the evidence that you gathered. BEFORE EXECUTING NEW ACTIONS YOU MUST ALWAYS GIVE THE REVISED VERSION OF THE WORLD MODEL.",
             ]
         )
-        lines.append(
-            "You may call `action(actions)` more than once in one Python snippet if your search or control loop needs it, "
-            "but stop immediately if a result reports `game_over`, `run_complete`, `level_completed`, or `done`."
-        )
-        lines.extend(self._summarized_knowledge_lines())
-        lines.append("end of world model. ")
+        # Upstream states four things about the world model here, and with
+        # ARC3_MEMORY_SECTIONS=off three of them are false or unanswerable: the
+        # block they refer to is suppressed, so "Below you are provided with the
+        # current world model" is followed by nothing, and the model is told in
+        # capitals to always give a revised version with nowhere to put it.
+        # Observed as a drop in replies carrying any assistant text at all,
+        # 76% to 43% - a broken instruction, not an absent one, which is why
+        # `off` was never a clean control.
+        #
+        # The first sentence loses only its middle clause: "from the newest
+        # history" attaches to inspecting the evidence just as well, so the
+        # remainder still says use Python, inspect, and score against the goal.
+        if _memory_sections_disabled():
+            lines.append(
+                "Use Python to inspect the evidence from the newest history, and search or score candidate actions or short sequences against the current goal as you currently understand it."
+            )
+        else:
+            lines.extend(
+                [
+                    "Use Python to inspect the evidence, refine that world model from the newest history, and search or score candidate actions or short sequences against the current goal as you currently understand it.",
+                    "Maintain a compact working world model of what the current level seems to contain, what actions appear to do, what the goal seems to be, what is still uncertain, and what plan currently looks best.",
+                    "Below you are provided with the current world model from the previous turn. The default behavior is to copy it and add or remove things based on the evidence that you gathered. BEFORE EXECUTING NEW ACTIONS YOU MUST ALWAYS GIVE THE REVISED VERSION OF THE WORLD MODEL.",
+                ]
+            )
+        if not _dedupe_multicall_line():
+            # kept mid-opener only for the upstream wording; the surviving copy
+            # in the closing block is where a note about calling action() twice
+            # belongs, next to the instruction to call it at all
+            lines.append(
+                "You may call `action(actions)` more than once in one Python snippet if your search or control loop needs it, "
+                "but stop immediately if a result reports `game_over`, `run_complete`, `level_completed`, or `done`."
+            )
+        knowledge_lines = self._summarized_knowledge_lines()
+        if knowledge_lines:
+            lines.extend(knowledge_lines)
+            lines.append("end of world model. ")
         if action_num == 0:
             lines.append(
                 "Ground yourself in `current_frame` before acting, but start with a compact structural summary rather than restating the full frame."
@@ -1247,12 +5156,21 @@ class ToolAgent:
             [
                 "When ready, call `action(actions)` from inside the `python` tool with the best valid action or ordered batch selected by your code. If your code has found a reliable short sequence, prefer batching it in one call.",
                 "You may call `action(actions)` more than once in one Python snippet if your search or control loop needs it.",
-                "If you include assistant text before a tool call, keep it short and use it to update the world model. Helpful optional prefixes are `World model:`, `Goal model:`, `Action model:`, `Recent findings:`, `Open questions:`, `Plan:`, and `Cross-level notes:`.",
+                *self._memory_section_guidance(bool(knowledge_lines)),
                 TOOL_CALL_FORMAT_GUIDANCE,
             ]
         )
         if "MOUSE" in _normalize_valid_actions(valid_actions):
             lines.append("If you use MOUSE, include integer row and col arguments.")
+        if _opener_commit_hint():
+            # last line of the opener, where the resumption prompt sits relative
+            # to the reply it influences
+            lines.append(
+                "When your evidence supports a choice, call `action(actions)` with the "
+                "sequence it supports. Probe with a single action when you genuinely "
+                "need to distinguish two possibilities; otherwise batch the route you "
+                "have."
+            )
         return "\n".join(lines)
 
     def _tools(self, state_path: Path) -> list[dict[str, Any]]:
@@ -1279,35 +5197,105 @@ class ToolAgent:
             }
         ]
 
+    def _harness_template_kwargs(self) -> dict[str, Any]:
+        """Chat-template kwargs the HARNESS sets per request.
+
+        Shared by the request builder and the request log so the log records
+        what was actually sent rather than a reconstruction - which matters
+        for the effort ladder, where the value changes mid-run, and for
+        preserve_thinking, where the question is whether it reaches the server
+        at all."""
+        kwargs: dict[str, Any] = {}
+        preserve_thinking = _preserve_thinking_kwarg()
+        if preserve_thinking is not None:
+            kwargs["preserve_thinking"] = preserve_thinking
+        ladder = _reasoning_effort_ladder()
+        rung = getattr(self, "_reasoning_effort_rung", -1)
+        if ladder and 0 <= rung < len(ladder):
+            kwargs["reasoning_effort"] = ladder[rung]
+        return kwargs
+
     def _chat_completion(
         self,
         messages: list[dict[str, Any]],
         *,
         tools: list[dict[str, Any]] | None,
         request_timeout_seconds: float | None = None,
+        max_tokens: int | None = None,
+        enable_thinking: bool | None = None,
     ) -> _ChatCompletionResult:
         payload = build_chat_payload(
             provider=self._model.provider,
             model=self._model.model_id,
             messages=messages,
-            max_tokens=self._max_output_tokens,
+            max_tokens=self._max_output_tokens if max_tokens is None else max_tokens,
             temperature=_LOCAL_ANALYZER_TEMPERATURE,
             top_p=_LOCAL_ANALYZER_TOP_P,
             top_k=_LOCAL_ANALYZER_TOP_K,
-            thinking=bool(_LOCAL_ANALYZER_ENABLE_THINKING),
+            thinking=(
+                bool(_LOCAL_ANALYZER_ENABLE_THINKING)
+                if enable_thinking is None
+                else enable_thinking
+            ),
             tools=tools,
             tool_choice=_request_tool_choice(tools),
             seed=_LOCAL_ANALYZER_SEED,
         )
+        provider_prefs = _openrouter_provider_prefs(self._model.provider)
+        if provider_prefs:
+            payload["provider"] = provider_prefs
+        # merge rather than assign: the payload builder may already have set
+        # chat_template_kwargs (enable_thinking), and replacing the dict would
+        # drop it - as a request-level dict can also drop a server-side default
+        template_kwargs = dict(payload.get("chat_template_kwargs") or {})
+        template_kwargs.update(self._harness_template_kwargs())
+        if enable_thinking is not None:
+            # after the merge: _harness_template_kwargs does not set
+            # enable_thinking today, but an override that a later edit could
+            # silently undo is the bug that keeps recurring here
+            template_kwargs["enable_thinking"] = bool(enable_thinking)
+        if template_kwargs:
+            payload["chat_template_kwargs"] = template_kwargs
+        payload["messages"] = _strip_control_keys(
+            _apply_summary_visibility(
+                payload.get("messages") or [],
+                evicted=getattr(self, "_has_evicted", False),
+            )
+        )
         def post_chat(request_payload: dict[str, Any]) -> requests.Response:
             return requests.post(
                 f"{self._model.base_url.rstrip('/')}/chat/completions",
                 headers=self._headers(),
                 json=request_payload,
-                timeout=request_timeout_seconds if request_timeout_seconds is not None else self._timeout,
+                # (connect, read): one scalar gives the connect phase the whole
+                # read budget, so an unreachable host stalls a single attempt
+                # for the rest of the game
+                timeout=(
+                    _CONNECT_TIMEOUT_SECONDS,
+                    request_timeout_seconds if request_timeout_seconds is not None else self._timeout,
+                ),
+                stream=bool(request_payload.get("stream")),
             )
 
-        response = post_chat(payload)
+        # Per agent, not per process: every game meets the cold server on its
+        # own first request, so one shared flag would grant the grace to
+        # whichever game happened to go first and let the rest fail fast.
+        initial_grace = 0.0
+        if not self._http_initial_grace_used:
+            initial_grace = _env_float("ARC3_HTTP_RETRY_INITIAL_SECONDS", 0.0)
+        _diag(self, "request")
+        try:
+            response = _post_with_retries(
+                lambda: post_chat(payload),
+                retries=_env_int("ARC3_HTTP_RETRIES", 3),
+                base_seconds=_env_float("ARC3_HTTP_RETRY_BASE_SECONDS", 5.0),
+                max_seconds=_env_float("ARC3_HTTP_RETRY_MAX_SECONDS", 5.0),
+                initial_seconds=initial_grace,
+            )
+        finally:
+            # consumed whether the call succeeded or failed: the grace covers
+            # the server coming up, and by now it either did or will not
+            self._http_initial_grace_used = True
         try:
             response.raise_for_status()
         except requests.HTTPError as exc:
@@ -1322,22 +5310,63 @@ class ToolAgent:
             if detail:
                 message += f" | response: {detail}"
             raise requests.RequestException(message)
-        payload = response.json()
+        if payload.get("stream"):
+            # decode_unicode=True decodes each transport chunk INDEPENDENTLY, so a
+            # multi-byte UTF-8 sequence straddling a chunk boundary is split into
+            # two broken halves - an em-dash (E2 80 94) becomes "\u00e2" plus
+            # fragments. Most characters survive, which is why short probes look
+            # clean and only long streamed replies are corrupted. The assembler
+            # already decodes bytes itself, and a complete SSE line is always a
+            # whole UTF-8 sequence, so hand it raw bytes instead.
+            payload = assemble_streamed_chat_response(
+                response.iter_lines(decode_unicode=False)
+            )
+        else:
+            payload = response.json()
         choices = payload.get("choices", [])
         if not choices:
-            raise requests.RequestException("server returned no choices")
+            detail = payload.get("error") or {
+                k: v for k, v in payload.items() if k not in ("choices",)
+            }
+            raise requests.RequestException(
+                f"server returned no choices | body: {str(detail)[:600]}"
+            )
         choice = choices[0]
         return _ChatCompletionResult(
-            message=choice.get("message", {}),
+            message=_extract_leaked_tool_calls(choice.get("message", {})),
             finish_reason=str(choice.get("finish_reason", "") or ""),
             usage=payload.get("usage"),
+            served_by=str(payload.get("provider", "") or ""),
         )
 
     def _trim_tool_text(self, text: str) -> tuple[str, bool]:
-        if len(text) <= self._tool_output_chars:
+        """Keep both ends and drop the middle.
+
+        Truncating the tail loses the half that matters. An `error` field ends
+        with `SomeError: <the explanation>` after the traceback, so a deep
+        traceback pushed the explanation out entirely. And a snippet's `stdout`
+        typically prints the state, acts, then prints the state again - so the
+        tail holds the result and the head holds the setup.
+
+        Weighted toward the HEAD: a snippet's output is mostly sequential
+        reasoning that reads from the top, so two thirds there. The remaining
+        third is ample for what has to survive at the end - the longest refusal
+        message measures ~550 characters and a user-frame traceback ~250, well
+        inside a 1365-character tail at the default budget.
+        """
+        limit = self._tool_output_chars
+        if len(text) <= limit:
             return text, False
-        omitted = len(text) - self._tool_output_chars
-        return f"{text[:self._tool_output_chars]}\n... [truncated {omitted} chars]", True
+        omitted = len(text) - limit
+        if not _get_env_bool("ARC3_MIDDLE_TRUNCATION", False):
+            return f"{text[:limit]}\n... [truncated {omitted} chars]", True
+        tail = limit // 3
+        head = limit - tail
+        return (
+            f"{text[:head]}\n... [truncated {omitted} chars from the middle] ...\n"
+            f"{text[-tail:]}",
+            True,
+        )
 
     def _summarize_planned_actions(self, value: Any) -> Any:
         if isinstance(value, dict):
@@ -1366,6 +5395,9 @@ class ToolAgent:
         for field in truncate_fields:
             value = result.get(field)
             if isinstance(value, str):
+                value, suppressed_rows = _suppress_board_dumps(value)
+                if suppressed_rows:
+                    result["board_print_suppressed_rows"] = suppressed_rows
                 result[field], field_truncated = self._trim_tool_text(value)
                 truncated = truncated or field_truncated
         if truncated:
@@ -1413,13 +5445,230 @@ class ToolAgent:
             raise TypeError(f"Action {index} must be a string or a dict.")
         return normalized
 
+    def _fatal_prefix_match(
+        self, state_hash: str, remaining_sigs: list[str]
+    ) -> tuple[int, str] | None:
+        """Does a recorded fatal sequence continue from here?
+
+        Returns ``(offset, action)`` when executing ``remaining_sigs`` from
+        this board would replay a recorded attempt into its death - offset 0
+        meaning the very next action is the fatal one. Returns None otherwise.
+
+        Generalizes the single-action check: an attempt recorded as actions
+        ``A[0..n-1]`` with pre-action hashes ``H[0..n-1]`` is fatal from here
+        if some ``H[k]`` equals this board AND ``A[k:]`` is a prefix of what
+        the model is about to run. The environment is deterministic and a
+        death resets to level start, so replaying ``A[k:]`` from ``H[k]``
+        reproduces ``H[k+1] ...`` and ends the attempt.
+
+        Catching it at the FIRST position of such a batch is what makes the
+        block worth having: the per-position check alone would let the harmless
+        prefix run and refuse only the final move, spending those actions to
+        arrive somewhere the model has already been. A model that genuinely
+        wants the prefix can submit it without the fatal tail, which no longer
+        matches and is not blocked.
+        """
+        index = getattr(self, "_death_ledger_index", None)
+        attempts = getattr(self, "_death_ledger_attempts", None)
+        if not index or not attempts or not remaining_sigs:
+            return None
+        for attempt_idx, position in index.get(state_hash, ()):  # type: ignore[union-attr]
+            try:
+                actions = attempts[attempt_idx]["actions"]
+            except (IndexError, KeyError, TypeError):
+                continue
+            tail = [action_signature(a) for a in actions[position:]]
+            if not tail or len(tail) > len(remaining_sigs):
+                continue
+            if remaining_sigs[: len(tail)] == tail:
+                return len(tail) - 1, tail[-1]
+        return None
+
+    def _action_guard_hook(
+        self,
+        grid: Any,
+        action_name: str,
+        action_data: dict[str, Any],
+        level: Any = None,
+        remaining: list[tuple[str, dict[str, Any]]] | None = None,
+    ) -> tuple[str, str] | None:
+        """Called by the solver before each action of a batch. Returns
+        ``(stop_reason, detail)`` to refuse, or None to proceed."""
+        guard = self._noop_repeat_guard
+        if guard is None or not grid:
+            return None
+        if not _guards_active(level):
+            # level 1 is where the model probes hardest and where an action is
+            # worth least, so the repeat guards can be told to start later. The
+            # guard still WATCHES - note_level and the state hash below run
+            # either way - so that by the time it starts refusing it already
+            # knows what has been tried.
+            guard.note_level(level)
+            self._noop_guard_pre_hash = _interior_state_hash(
+                [list(row) for row in grid]
+            )
+            return None
+        rows = [list(row) for row in grid]
+        state_hash = _interior_state_hash(rows)
+        guard.note_level(level)
+        # the solver passes ENGINE names (ACTION2); every other consumer - the
+        # death ledger, the transcript, the model itself - speaks model names
+        # (DOWN), so translate here or the guard both mismatches the ledger and
+        # reports "ACTION2" to a model that has never seen that label
+        action_sig = action_signature(
+            {"action": to_model_action(action_name), **(action_data or {})}
+        )
+        self._noop_guard_pre_hash = state_hash
+        if self._noop_guard_deaths_enabled and not self._guard_death_override_batch:
+            # with lookahead off, only THIS action is considered, so a match
+            # can be found solely at offset 0 - the per-position behaviour
+            window = (
+                (remaining or [(action_name, action_data)])
+                if _fatal_lookahead_enabled()
+                else [(action_name, action_data)]
+            )
+            remaining_sigs = [
+                action_signature({"action": to_model_action(name), **(data or {})})
+                for name, data in window
+            ]
+            match = self._fatal_prefix_match(state_hash, remaining_sigs)
+            if guard.should_block_death(
+                state_hash, action_sig, lambda _h, _a: match is not None
+            ):
+                offset, fatal_action = match  # type: ignore[misc]
+                if offset == 0:
+                    detail = (
+                        f"{fatal_action} ended a previous attempt on this level from this "
+                        "exact board state, so it was NOT executed."
+                    )
+                else:
+                    detail = (
+                        "this sequence replays a path that ended a previous attempt: "
+                        f"{fatal_action} is fatal {offset} action(s) from here, so NOTHING "
+                        "was executed and no actions were spent. Submit a different "
+                        "continuation, or the same prefix without the fatal tail."
+                    )
+                return (
+                    "known_death",
+                    detail
+                    + " This refusal is not catchable and ended the snippet, so anything "
+                    "after the call did not run. Issuing the same action again as the "
+                    "very next action will override the block and execute the action. "
+                    "That is only worth doing if you have a specific reason to believe "
+                    "it will not end the attempt this time. Otherwise, plan a different "
+                    "continuation from this state.",
+                )
+            # the override fired: let the rest of THIS call run without further
+            # death checks, or the model would have to insist again at every
+            # position of a sequence it has already deliberately confirmed
+            if match is not None:
+                self._guard_death_override_batch = True
+        if (
+            self._noop_guard_noops_enabled
+            and not (reset_exposed() and action_sig == "RESET")
+            and guard.should_block(state_hash, action_sig)
+        ):
+            return (
+                "known_noop",
+                (f"{action_sig} already produced no change inside the board area in this "
+                   "exact board state "
+                   if _new_noop_wording() else
+                   f"{action_sig} already changed nothing in this exact board state ")
+                +
+                "earlier on this level, so it was NOT executed and no action was "
+                "spent. This refusal is not catchable and ended the snippet, so "
+                "anything after the call did not run - a retry loop cannot probe "
+                "past it. It applies to that one action from this one board state; "
+                "it is not a constraint on sequences or batch size. Issuing the same "
+                "action again as the very next action will override the block and "
+                "execute the action. That is only worth doing if you have a specific "
+                "reason to believe it will behave differently now. Otherwise, plan a "
+                "different continuation from this state.",
+            )
+        if (
+            _repeat_state_guard_enabled()
+            and (state_hash, action_sig) in self._repeat_state_seen
+        ):
+            # after known_noop deliberately: a repeated no-op is better reported
+            # as a no-op, which also keeps its override intact. Reaching here
+            # means the action DID change the board and still returned to a
+            # state this snippet has already acted from.
+            return (
+                "repeated_action_in_state",
+                f"{action_sig} was already run from this exact board state earlier "
+                "in this snippet, so what it does here is known and it was NOT "
+                # not "no action budget was spent": by construction this fires
+                # after the snippet has already spent budget on the actions that
+                # got it here, and the model reads that phrase as covering the
+                # whole snippet
+                "executed. A sequence that returns to "
+                # examples, not a diagnosis: the harness knows the board
+                # repeated, not why, and stating the cause would have the model
+                # looking for a wrapping cursor it may not have
+                "a board it has already acted from is not advancing (e.g. a cursor "
+                "that wraps around, or a loop stepping the wrong way past its "
+                "target). "
+                "Nothing after the call ran. Re-read `current_frame`, work out why "
+                "the sequence is not converging, and write a different snippet - "
+                "this block is per snippet, so the next one starts with no history.",
+            )
+        guard.note_executed(state_hash, action_sig)
+        self._guard_hook_trace.append((state_hash, action_sig))
+        return None
+
+    def _record_guard_observations(
+        self,
+        guard: NoopRepeatGuard,
+        raw_payload: dict[str, Any],
+        compact_payload: dict[str, Any],
+    ) -> None:
+        """Record every executed position of this call, not just the first.
+
+        The guard hook fires once per position immediately before that action
+        runs, so `_guard_hook_trace` holds the board hash each action actually
+        executed against - including the intermediate states of a batch, which
+        appear nowhere in the returned payload. The solver reports outcomes as
+        `gameplay_changed_per_action`, index-aligned with `executed_actions`
+        and therefore with the trace; a single-action call has no such list, so
+        the aggregate verdict is used instead.
+
+        Zipping to the shorter of the two is deliberate: a batch stopped early
+        (blocked, fatal, invalid) leaves one more trace entry than there are
+        outcomes, and an unmatched entry must not be recorded against a verdict
+        that belongs to a different action.
+        """
+        trace = list(self._guard_hook_trace)
+        self._guard_hook_trace = []
+        if not trace:
+            return
+        verdicts = raw_payload.get("gameplay_changed_per_action")
+        if not isinstance(verdicts, list) or not verdicts:
+            if len(trace) != 1 or not compact_payload.get("executed"):
+                return
+            verdicts = [compact_payload.get("gameplay_changed")]
+        for (state_hash, action_sig), changed in zip(trace, verdicts):
+            if reset_exposed() and action_sig == "RESET":
+                # Retain its position in the trace so following verdicts stay
+                # aligned, but do not learn that a bar-only reset is ineffective.
+                continue
+            # unconditional: known_noop reaches a repeated no-op first, so
+            # filtering on `changed` here would buy nothing
+            self._repeat_state_seen.add((state_hash, action_sig))
+            guard.observe(state_hash, action_sig, gameplay_changed=changed)
+
     def _compact_action_result(self, payload: dict[str, Any]) -> dict[str, Any]:
         compact = {
             "executed": bool(payload.get("executed")),
             "action_num": payload.get("action_num"),
             "level": payload.get("level"),
             "score": payload.get("score"),
-            "reward": payload.get("reward"),
+            # `reward` is not carried: it is (levels_completed - previous) /
+            # number_of_levels, so it reads 0.0 on every action that does not
+            # finish a level. A model seeing 0.0 on almost every action can
+            # reasonably take it for a verdict on the action, which it is not,
+            # and `level_completed` in this same dict already says the thing it
+            # was trying to say. The solver still computes it for the viewer
+            # and the raw payload.
             "state": payload.get("state"),
             "valid_actions": payload.get("valid_actions", []),
             "board_changed": bool(payload.get("board_changed")),
@@ -1429,6 +5678,8 @@ class ToolAgent:
             "run_complete": bool(payload.get("run_complete")),
             "action_display": payload.get("action_display") or payload.get("action_name"),
         }
+        if "gameplay_changed" in payload:
+            compact["gameplay_changed"] = bool(payload.get("gameplay_changed"))
         executed_actions = payload.get("executed_actions")
         if isinstance(executed_actions, list) and executed_actions:
             compact["executed_actions"] = [str(action).strip() for action in executed_actions if str(action).strip()]
@@ -1439,6 +5690,11 @@ class ToolAgent:
             compact["requested_count"] = payload.get("requested_count", batch_size)
             compact["executed_count"] = payload.get("executed_count", batch_size)
             compact["stopped_early"] = bool(payload.get("stopped_early"))
+        # blocked_action names the refused action for the next turn's opener;
+        # without it the model was told "the sequence was stopped before None"
+        for passthrough_key in ("no_op_action_index", "no_op_action", "skipped_actions", "gameplay_changed_per_action", "action_trace", "no_op", "fatal_action", "blocked_action", "action_echo", "animation"):
+            if passthrough_key in payload:
+                compact[passthrough_key] = payload.get(passthrough_key)
         if payload.get("stop_reason"):
             compact["stop_reason"] = payload.get("stop_reason")
         if payload.get("stop_detail"):
@@ -1446,6 +5702,14 @@ class ToolAgent:
         for timing_key in ("run_elapsed_seconds", "time_remaining_seconds"):
             if timing_key in payload:
                 compact[timing_key] = payload.get(timing_key)
+        # the session's own clock, which is also what decides when the game
+        # stops - so the endgame phase cannot drift away from the deadline it
+        # is meant to anticipate. It starts at session construction, so a game
+        # waiting on the gate is already ageing, which is the same convention
+        # max_runtime_s_per_game uses.
+        elapsed = payload.get("run_elapsed_seconds")
+        if isinstance(elapsed, (int, float)):
+            self._game_elapsed_seconds = float(elapsed)
         if payload.get("error"):
             compact["error"] = payload.get("error")
         return compact
@@ -1466,7 +5730,7 @@ class ToolAgent:
         def _serialized_runtime_state(
             *,
             next_valid_actions: list[str] | None = None,
-            last_action_result: dict[str, Any] | None = None,
+            last_action_call_result: dict[str, Any] | None = None,
         ) -> dict[str, Any]:
             refreshed_frame, refreshed_history = load_runtime_state(state_path)
             current_frame_payload = _ascii_frame_view_payload(refreshed_frame)
@@ -1475,15 +5739,20 @@ class ToolAgent:
             else:
                 sanitized_actions = list(valid_actions)
             persisted_action_result = (
-                last_action_result
-                if isinstance(last_action_result, dict)
-                else self._last_action_result
+                last_action_call_result
+                if isinstance(last_action_call_result, dict)
+                else self._last_action_call_result
             )
             return {
                 "current_frame": current_frame_payload,
                 "history": _ascii_history_view_payload(refreshed_history),
                 "valid_actions": sanitized_actions,
-                "last_action_result": (
+                "death_ledger": (
+                    self._death_ledger_payload()
+                    if _get_env_bool("ARC3_DEATH_LEDGER", False)
+                    else None
+                ),
+                "last_action_call_result": (
                     dict(persisted_action_result)
                     if isinstance(persisted_action_result, dict)
                     else {}
@@ -1491,9 +5760,24 @@ class ToolAgent:
             }
 
         terminal_action_result: dict[str, Any] | None = None
+        snippet_executed_actions = 0
 
-        def _handle_action(actions: list[dict[str, Any]]) -> dict[str, Any]:
-            nonlocal terminal_action_result
+        def _handle_animation(request: dict[str, Any]) -> dict[str, Any]:
+            if self._step_env_callback is None:
+                return {"error": "Animation frames are not available in this session."}
+            raw = self._step_env_callback({"query": "animation"})
+            record = raw.get("record") if isinstance(raw, dict) else None
+            record = record if isinstance(record, dict) else None
+            if str(request.get("kind") or "") == "frames":
+                return build_animation_frames(record)
+            if not _animation_timeline_enabled():
+                return {}
+            return build_animation_view(record)
+
+        def _handle_action(
+            actions: list[dict[str, Any]], *, stale_after: str | None = None
+        ) -> dict[str, Any]:
+            nonlocal terminal_action_result, snippet_executed_actions
             if self._step_env_callback is None:
                 raise RuntimeError("action(actions) is not available in this session.")
             normalized_actions = self._normalize_python_actions(actions)
@@ -1504,7 +5788,6 @@ class ToolAgent:
                     "action_num": terminal_action_result.get("action_num"),
                     "level": terminal_action_result.get("level"),
                     "score": terminal_action_result.get("score"),
-                    "reward": 0.0,
                     "state": terminal_action_result.get("state"),
                     "valid_actions": [],
                     "board_changed": False,
@@ -1518,37 +5801,107 @@ class ToolAgent:
                     "stop_reason": f"previous_{reason}",
                     "stop_detail": _terminal_action_stop_detail(reason),
                 }
-                self._last_action_result = dict(compact_payload)
+                self._last_action_call_result = dict(compact_payload)
                 return {
                     "action_result": compact_payload,
                     "state": _serialized_runtime_state(
                         next_valid_actions=[],
-                        last_action_result=compact_payload,
+                        last_action_call_result=compact_payload,
                     ),
                 }
-            raw_payload = self._step_env_callback({"actions": normalized_actions})
+            if reset_exposed():
+                reset_positions = [
+                    index for index, item in enumerate(normalized_actions)
+                    if to_engine_action(item.get("action")) == "RESET"
+                ]
+                if reset_positions and (snippet_executed_actions or reset_positions != [0]):
+                    detail = (
+                        "RESET must be the first executed game action in a Python snippet. "
+                        "This call executed no actions. Start a new snippet with RESET "
+                        "as its first action; inspection or computation may come before it."
+                    )
+                    refusal = {
+                        "executed": False,
+                        "stop_reason": "reset_not_first",
+                        "stop_detail": detail,
+                        "error": detail,
+                        "valid_actions": list(self._current_valid_actions),
+                    }
+                    self._last_action_call_result = dict(refusal)
+                    return {
+                        "action_result": refusal,
+                        "state": _serialized_runtime_state(last_action_call_result=refusal),
+                    }
+            guard = self._noop_repeat_guard
+            self._guard_hook_trace = []
+            self._guard_death_override_batch = False
+            # No pre-check here: the solver's per-position hook covers every
+            # action of every call, including the first. Checking in both
+            # places consumed the override slot twice - the pre-check armed it,
+            # the retry cleared it, and the hook then blocked the very action
+            # the model had been invited to repeat.
+            step_arguments: dict[str, Any] = {"actions": normalized_actions}
+            if stale_after:
+                # forwarded as an argument so the solver can weigh it AFTER its
+                # own per-position guards, which name a specific action and
+                # carry an override
+                step_arguments["stale_after"] = stale_after
+            raw_payload = self._step_env_callback(step_arguments)
             if not isinstance(raw_payload, dict):
                 raise RuntimeError("action(actions) did not return a JSON-like payload.")
+            if reset_exposed() and raw_payload.get("executed"):
+                # Use actual execution, not submitted batch size: a refused
+                # call does not consume the snippet's first-action allowance.
+                snippet_executed_actions += max(1, int(raw_payload.get("executed_count") or 1))
+            if _animation_enabled():
+                # Held on the agent, not routed through the sandbox: the chain
+                # is dropped by compaction so the raw grids never become text,
+                # which also meant it never reached the summary. It is only
+                # ever used to render an image here, so it has no business
+                # crossing the pipe twice.
+                self._last_animation_chain = raw_payload.get("animation_chain") or []
+                # Whether the LAST executed action moved anything, which is not
+                # the same as whether the sequence did: a batch of UP, DOWN,
+                # LEFT, SPACE reports gameplay_changed for the batch, so an
+                # inert SPACE at the end was hidden behind three moves that
+                # were not. The image exists for the action that animated, so
+                # it is that action's flag that decides.
+                per_action = raw_payload.get("gameplay_changed_per_action")
+                if isinstance(per_action, list) and per_action:
+                    self._last_action_gameplay_changed = bool(per_action[-1])
+                else:
+                    self._last_action_gameplay_changed = raw_payload.get(
+                        "gameplay_changed"
+                    )
             compact_payload = self._compact_action_result(raw_payload)
+            if guard is not None and normalized_actions:
+                self._record_guard_observations(guard, raw_payload, compact_payload)
             next_valid_actions = raw_payload.get("valid_actions")
             if isinstance(next_valid_actions, list):
                 self._current_valid_actions = _normalize_valid_actions(next_valid_actions)
             if compact_payload.get("executed") and _terminal_action_reason(compact_payload):
                 terminal_action_result = compact_payload
-            self._last_action_result = dict(compact_payload)
+            self._last_action_call_result = dict(compact_payload)
             return {
                 "action_result": compact_payload,
                 "state": _serialized_runtime_state(
                     next_valid_actions=next_valid_actions if isinstance(next_valid_actions, list) else None,
-                    last_action_result=compact_payload,
+                    last_action_call_result=compact_payload,
                 ),
             }
 
+        # per snippet: a repeat across snippets is the model choosing, not a
+        # loop failing to converge
+        self._repeat_state_seen.clear()
         sandbox_result = run_sandboxed_python(
             code=code,
             timeout_seconds=self._python_timeout,
             initial_state=_serialized_runtime_state(),
             action_handler=_handle_action,
+            animation_handler=_handle_animation if _animation_enabled() else None,
+            kept_functions=self._retained_sources(),
+            retain_imports=_get_env_bool("ARC3_PERSISTENT_FUNCTIONS_IMPORTS", False),
+            repair_hints=_get_env_bool("ARC3_PERSISTENT_FUNCTIONS_REPAIR_HINTS", False),
         )
 
         action_results = [
@@ -1557,6 +5910,7 @@ class ToolAgent:
             if isinstance(item, dict)
         ]
         payload: dict[str, Any] = {"tool": "python"}
+        self._record_retained_functions(sandbox_result, payload)
         rendered_stdout = str(sandbox_result.get("stdout", "") or "")
         rendered_error = str(sandbox_result.get("error", "") or "")
         if rendered_error:
@@ -1575,16 +5929,28 @@ class ToolAgent:
                 else:
                     payload["result"] = {
                         "action_calls": len(action_results),
-                        "last_action_result": action_results[-1],
+                        "last_action_call_result": action_results[-1],
                     }
 
         step_executed = any(bool(item.get("executed")) for item in action_results)
+        made_progress = any(
+            bool(item.get("gameplay_changed"))
+            or bool(item.get("level_completed"))
+            or bool(item.get("game_over"))
+            or bool(item.get("run_complete"))
+            for item in action_results
+        )
         if step_executed:
             self._last_step_summary = self._summarize_step_sequence(action_results)
             self._update_summarized_knowledge_from_step_summary()
+        elif self._last_step_summary and not self._last_step_summary.get("stale"):
+            # carried over across an exchange that executed nothing: mark it so
+            # the next prompt reminds instead of re-narrating it as fresh
+            self._last_step_summary = {**self._last_step_summary, "stale": True}
         return _ToolDispatchResult(
             self._render_tool_payload(payload, truncate_fields=("stdout", "error", "result")),
             step_executed=step_executed,
+            made_progress=made_progress,
         )
 
     def _dispatch_tool(self, state_path: Path, name: str, arguments: dict[str, Any]) -> _ToolDispatchResult:
@@ -1593,22 +5959,284 @@ class ToolAgent:
             return self._run_python_tool(state_path, arguments)
         return _ToolDispatchResult(json.dumps({"error": f"Unknown tool: {name}"}, indent=2))
 
+    def _calibrate_from_usage(
+        self,
+        messages: list[dict[str, Any]],
+        tools: list[dict[str, Any]] | None,
+        usage: dict[str, Any] | None,
+    ) -> str:
+        """Update the characters-per-token divisor from a served request.
+
+        The images are subtracted using the SAME formula that will re-add them,
+        so the two cancel and the divisor measures text alone. Clamped because
+        an under-count risks context overflow, which is the dangerous
+        direction; a bad measurement should cost accuracy, not a rejected
+        request."""
+        if not _calibrate_text_tokens() or not isinstance(usage, dict):
+            return ""
+        try:
+            prompt_tokens = int(usage.get("prompt_tokens") or 0)
+        except (TypeError, ValueError):
+            return ""
+        if prompt_tokens <= 0:
+            return ""
+        scrubbed, image_tokens = _split_messages_for_estimate(list(messages))
+        payload: dict[str, Any] = {"messages": scrubbed}
+        if tools:
+            payload["tools"] = tools
+            payload["tool_choice"] = _request_tool_choice(tools)
+        try:
+            rendered = json.dumps(payload, ensure_ascii=False, sort_keys=True, default=str)
+        except TypeError:
+            return ""
+        text_tokens = prompt_tokens - image_tokens
+        if text_tokens <= 0 or not rendered:
+            return ""
+        measured = len(rendered) / text_tokens
+        clamped = min(_text_token_chars_max(), max(_text_token_chars_min(), measured))
+        previous = self._text_chars_per_token
+        self._text_chars_per_token = clamped
+        if previous is None or abs(clamped - previous) >= 0.05:
+            note = "" if clamped == measured else f" (clamped from {measured:.2f})"
+            # returned rather than written here: append_transcript is a closure
+            # inside analyze(), not reachable from a method
+            return (
+                f"text_token_calibration: {clamped:.2f} chars/token{note} "
+                f"[prompt_tokens {prompt_tokens}, images {image_tokens}, "
+                f"text chars {len(rendered)}]"
+            )
+        return ""
+
+    def _append_context_message(
+        self, messages: list[dict[str, Any]], message: dict[str, Any]
+    ) -> None:
+        """Append and count new context once, before any history can be evicted.
+
+        Use the context estimator for all roles, including reasoning and image
+        tokens. Subtract the empty request envelope; static system/tool schema
+        tokens and re-sent history are not new context. The summary exchange
+        itself bypasses this helper and starts a fresh interval.
+        """
+        messages.append(message)
+        if _summary_interval_tokens() <= 0:
+            return
+        added = max(
+            0,
+            self._estimate_request_input_tokens([message])
+            - self._estimate_request_input_tokens([]),
+        )
+        self._summary_context_tokens_since_attempt = (
+            getattr(self, "_summary_context_tokens_since_attempt", 0) + added
+        )
+
+    def _maybe_append_rolling_summary(
+        self, append_transcript, tools: list[dict[str, Any]] | None,
+        *, request_timeout_seconds: float | None = None,
+    ) -> None:
+        """Attempt a summary after another interval of newly added context.
+
+        Called at turn start, which is the only clean boundary: nothing is in
+        flight, and a yield resumption arrives here too so no separate handling
+        is needed. Crossing the threshold is not urgent - the summary is not
+        needed until eviction reaches it - so overshooting by a turn costs
+        nothing and interrupting an investigation would.
+
+        The request carries the current history unchanged, so it is a
+        continuation rather than a prefix: both the KV prefix and the recurrent
+        state stay warm. Both messages are tagged, and the trimmer's
+        summary-aware drain navigates by that tag.
+        """
+        interval = _summary_interval_tokens()
+        if interval <= 0:
+            return
+        history = self._history_messages
+        if not history:
+            return
+
+        since = getattr(self, "_summary_context_tokens_since_attempt", 0)
+        if since < interval:
+            return
+        # before the instruction: context first, then the ask, so the ask is
+        # still the last thing read. Stored exactly as sent - a request message
+        # that differs from the one in history would diverge from the cached
+        # prefix at that point and cost a prefill on every later turn.
+        context_lines = self._summary_turn_context_lines() if _summary_turn_context() else []
+        request_text = (
+            "\n".join([*context_lines, "", SUMMARY_REQUEST_PROMPT])
+            if context_lines
+            else SUMMARY_REQUEST_PROMPT
+        )
+        request = _mark_control_message(
+            {"role": "user", "content": request_text},
+            _SUMMARY_CONTROL_KIND,
+        )
+        messages = [
+            {"role": "system", "content": self._system_prompt},
+            *history,
+            request,
+        ]
+        # Every attempt starts a new interval, including HTTP errors, empty
+        # replies and rejected tool calls. Reset before dispatch so all exit
+        # paths share that behavior; the summary exchange is excluded from this clock.
+        self._summary_context_tokens_since_attempt = 0
+        try:
+            result = self._chat_completion(
+                messages,
+                tools=tools,
+                max_tokens=_summary_max_gen_tokens(self._max_output_tokens),
+                enable_thinking=_summary_enable_thinking(),
+                request_timeout_seconds=request_timeout_seconds,
+            )
+        except BaseException as exc:  # a failed summary must not end the turn
+            log.warning("rolling summary request failed: %s", exc)
+            return
+        # before any rejection path: the tokens were generated whatever the
+        # reply turned out to be, and a summary that is thrown away still cost
+        # the time
+        self._accumulate_usage_tokens(
+            getattr(result, "usage", None), count_toward_turn=False
+        )
+        dropped_call = ""
+        message = result.message or {}
+        text = _normalize_message_content(message.get("content", "")).strip()
+        if message.get("tool_calls"):
+            # A reply that calls a tool may still have written a real summary
+            # first - the instruction not to act is advisory, and the model
+            # sometimes summarises and then reaches for the tool anyway. Keeping
+            # a usable one is better than asking again for something it has
+            # already produced, so the text is judged on its length: real
+            # summaries here run 4,000 to 6,000 characters, while a preamble
+            # introducing a tool call runs to tens.
+            #
+            # The tool call itself is dropped either way. It was never going to
+            # execute: this request is outside the turn loop, with no sandbox
+            # and no action budget attached.
+            names = ", ".join(
+                str((c.get("function") or {}).get("name", "?"))
+                for c in message.get("tool_calls") or []
+                if isinstance(c, dict)
+            )
+            if len(text) < _SUMMARY_MIN_USABLE_CHARS:
+                log.warning(
+                    "rolling summary tried to call %s with only %d chars of text; "
+                    "skipping",
+                    names or "a tool", len(text),
+                )
+                if text:
+                    append_transcript(
+                        "ROLLING SUMMARY REJECTED",
+                        f"[tried to call {names}, text too short to use]\n{text}",
+                    )
+                return
+            log.warning(
+                "rolling summary tried to call %s but wrote %d chars first; "
+                "keeping the text, dropping the call",
+                names or "a tool", len(text),
+            )
+            dropped_call = names or "a tool"
+        if not text:
+            # A reply that deliberated to the output limit without reaching the
+            # content wants a different fix from one that simply said nothing -
+            # a larger max_output or a shorter interval - so the two are logged
+            # apart rather than both as "empty".
+            reasoning_chars = len(_extract_reasoning_text(message))
+            if reasoning_chars:
+                log.warning(
+                    "rolling summary produced %d chars of reasoning and no summary "
+                    "(finish_reason=%s); skipping",
+                    reasoning_chars,
+                    str(getattr(result, "finish_reason", "") or "?"),
+                )
+            else:
+                log.warning("rolling summary came back empty; skipping")
+            return
+        truncated = str(getattr(result, "finish_reason", "")).strip() == "length"
+        if truncated:
+            # Kept rather than rejected: a summary cut at the output limit still
+            # covers the early material, and the alternative is no summary at
+            # all, which means the same span is evicted with nothing standing in
+            # for it. But the tail is where "what I was in the middle of" lives,
+            # so the model is told the text is incomplete rather than being left
+            # to infer it from a sentence that stops.
+            log.warning(
+                "rolling summary hit the output limit at %d chars of summary, after "
+                "%d chars of reasoning; keeping it, marked as incomplete",
+                len(text),
+                len(_extract_reasoning_text(message)),
+            )
+            text = f"{text}\n[This summary was cut off at the output limit and is incomplete.]"
+        reply = _mark_control_message(
+            {"role": "assistant", "content": text}, _SUMMARY_CONTROL_KIND
+        )
+        if _summary_replaces_history():
+            # only now, with a usable summary in hand - a failed or rejected one
+            # returned above and left history untouched
+            replaced_count = len(self._history_messages)
+            self._history_messages = [request, reply]
+            # this discards history without going through the trimmer, so the
+            # prefix is invalid and the flag has to be set by hand - otherwise
+            # the next request keeps a priority that assumes a warm cache
+            self._note_history_evicted()
+        else:
+            self._history_messages.append(request)
+            self._history_messages.append(reply)
+            replaced_count = 0
+        reasoning_chars = len(_extract_reasoning_text(message))
+        append_transcript(
+            "ANALYZER STATUS",
+            f"rolling_summary: {len(text)} chars of summary after "
+            f"{reasoning_chars} chars of reasoning, {since} new context tokens since the previous attempt"
+            + (" [TRUNCATED]" if truncated else "")
+            + (f" [tool call dropped: {dropped_call}]" if dropped_call else "")
+            + (f" [replaced {replaced_count} messages of history]"
+               if replaced_count else ""),
+        )
+        # the text itself, so a run can be read back and judged rather than only
+        # counted - this is the artefact the whole mechanism produces. Salvaged
+        # ones get their own section name so they can be grepped and read apart
+        # from the clean ones, which is the comparison that says whether a reply
+        # that also reached for a tool wrote a worse summary.
+        append_transcript(
+            "ROLLING SUMMARY SALVAGED" if dropped_call else "ROLLING SUMMARY",
+            text,
+        )
+
     def _estimate_request_input_tokens(
         self,
         messages: list[dict[str, Any]],
         *,
         tools: list[dict[str, Any]] | None = None,
     ) -> int:
+        messages = _apply_summary_visibility(
+            messages, evicted=getattr(self, "_has_evicted", False)
+        )
+        image_tokens = 0
+        if _image_token_estimate_enabled():
+            messages, image_tokens = _split_messages_for_estimate(messages)
         payload: dict[str, Any] = {"messages": messages}
         if tools:
             payload["tools"] = tools
             payload["tool_choice"] = _request_tool_choice(tools)
-        return _estimate_tokens(payload)
+        return _estimate_tokens(payload, self._text_chars_per_token) + image_tokens
 
     def _drop_oldest_history_block(self, history: list[dict[str, Any]], *, preserve_recent: int) -> bool:
         removable = len(history) - preserve_recent
         if removable <= 0:
             return False
+        # Never drop the last user message. Dropping one and draining to the
+        # next is fine while another exists; when none does, the drain runs to
+        # preserve_recent and _drop_until_first_user_message then empties the
+        # history outright - the request becomes the system prompt alone and the
+        # server answers "No user query found in messages".
+        #
+        # Reachable mid-turn, where the list ends with a tool result rather than
+        # the opener, so preserve_recent protects the tool message and the
+        # opener is the first thing popped. Seen in a live request log as a
+        # 1-message request at request_index_within_turn=2. Pre-existing and
+        # independent of the summary drain, which only makes it easier to reach
+        # by evicting further in one step.
+        if not any(str(m.get("role", "")).strip() == "user" for m in history[1:]):
+            return False
         first = history.pop(0)
         first_role = str(first.get("role", "")).strip()
         if first_role in {"assistant", "tool"}:
@@ -1617,6 +6245,13 @@ class ToolAgent:
             return True
         while history and history[0].get("role") == "tool" and len(history) > preserve_recent:
             history.pop(0)
+        # Only navigate by summaries when there is one left to land on, and only
+        # one that is a REQUEST (a user message) and sits within the removable
+        # range. Checking merely that a summary exists somewhere let the loop
+        # skip every remaining user message when the last one was already at the
+        # head, stripping history down to preserve_recent with no user message
+        # in it at all - which the chat template rejects outright:
+        # "No user query found in messages", HTTP 400, turn lost.
         while history and history[0].get("role") != "user" and len(history) > preserve_recent:
             history.pop(0)
         return True
@@ -1626,9 +6261,28 @@ class ToolAgent:
         messages: list[dict[str, Any]],
         *,
         max_turns: int,
+        drain_turns: int = 0,
+        force_drain: bool = False,
     ) -> list[dict[str, Any]]:
-        if max_turns <= 0 or not messages:
+        if not messages:
             return []
+        if max_turns <= 0:
+            # explicit opt-out: leave the token budget as the only limit
+            return list(messages)
+
+        drain_turns = min(max(0, drain_turns), max(0, max_turns - 1))
+        if drain_turns:
+            present = sum(
+                1
+                for message in messages
+                if str(message.get("role", "")).strip() == "assistant"
+            )
+            # trigger at the cap unless this commit has already invalidated the
+            # prefix for token reasons, in which case take the deep cut now
+            if present > max_turns or force_drain:
+                max_turns = max_turns - drain_turns
+            elif not force_drain:
+                return list(messages)
 
         kept_reversed: list[dict[str, Any]] = []
         assistant_turns = 0
@@ -1650,14 +6304,34 @@ class ToolAgent:
             trimmed.pop(0)
         return trimmed
 
+    def _note_history_evicted(self) -> None:
+        # The scheduler consumes the first flag at the next handover. The
+        # second stays set so summaries remain visible after history is lost.
+        self._context_was_trimmed = True
+        self._has_evicted = True
+
     def _persistent_history_messages(self, messages: list[dict[str, Any]], *, tools: list[dict[str, Any]] | None = None) -> list[dict[str, Any]]:
+        original_history = messages[1:]
+        if _any_control_pruning_enabled():
+            messages = _prune_control_messages(messages)
+        keep_images = _history_image_keep()
+        if keep_images is not None:
+            # after pruning, so a dropped resumption opener cannot consume a
+            # retention slot
+            messages = _apply_history_image_window(messages, keep_images)
+        before_trim = len(messages)
         trimmed = self._trim_messages_for_context(messages, tools=tools)
         if not trimmed:
+            if original_history:
+                self._note_history_evicted()
             return []
+        token_trim_dropped = len(trimmed) < before_trim
         trimmed_history = trimmed[1:]
         history = self._keep_recent_history_turns(
             trimmed_history,
-            max_turns=_PERSISTENT_HISTORY_ASSISTANT_TURNS,
+            max_turns=_persistent_history_assistant_turns(),
+            drain_turns=_history_turn_drain(),
+            force_drain=token_trim_dropped and _history_drain_coalesce(),
         )
         if (
             history
@@ -1667,7 +6341,115 @@ class ToolAgent:
             previous_message = trimmed_history[len(trimmed_history) - len(history) - 1]
             if str(previous_message.get("role", "")).strip() == "user":
                 history = [previous_message, *history]
-        return self._drop_until_first_user_message(history)
+        history = self._drop_until_first_user_message(history)
+        # Include turn-cap eviction, control pruning and image removal, which
+        # can change the cached prefix without the token trimmer dropping any
+        # messages. Image removal can leave the message count unchanged.
+        if history != original_history:
+            self._note_history_evicted()
+        return history
+
+    def _priority_total_levels(self) -> int:
+        """Read public game metadata through the existing session callback.
+
+        No new solver state or prompt fields. A custom callback, old object,
+        or unavailable metadata can fall back without interrupting a game.
+        """
+        try:
+            callback = getattr(self, "_step_env_callback", None)
+            session = getattr(callback, "__self__", None)
+            game = getattr(session, "game", None)
+            count = parse_total_levels(getattr(game, "number_of_levels", None))
+        except Exception:
+            count = None
+        if count is not None:
+            return count
+        if not getattr(self, "_priority_level_count_warned", False):
+            self._priority_level_count_warned = True
+            log.warning("priority: total level count unavailable or invalid; using 10")
+        return 10
+
+    def _maybe_handover(self) -> None:
+        """Re-price this game, and yield the slot if a gate is running.
+
+        Called right after the trimmer and before the request goes out, which is
+        where the prefix has just been invalidated. Consumes the trim flag: it
+        describes the trim that produced THIS request, and leaving it set would
+        keep the game demoted long after the prefix had been rebuilt.
+        """
+        if not self._context_was_trimmed:
+            return
+        self._context_was_trimmed = False
+        summary = self._last_step_summary or {}
+        level = int(summary.get("level") or 1)
+        actions = max(
+            0, _priority_action_count(summary) - self._actions_at_level_start
+        )
+        # getattr: an agent unpickled from before this field existed would
+        # otherwise raise here on every turn
+        tokens = max(
+            0,
+            self._session_generated_tokens
+            - getattr(self, "_tokens_at_level_start", 0),
+        )
+        gate = _priority_gate()
+        snapshot = None
+        options = _priority_level_options()
+        total_levels = (
+            self._priority_total_levels()
+            if options["normalize_score"] or options["tail_lookup"] else None
+        )
+        if gate is not None and _priority_refresh_enabled():
+            pace = self._progress_pace.cost_multiplier() if _get_env_bool("ARC3_PRIORITY_PACE", False) else 1.0
+            snapshot = PrioritySnapshot(level, actions, tokens, pace, total_levels)
+            estimate = gate._snapshot_priority(snapshot, time.monotonic())
+        else:
+            estimate = _game_priority(
+                level, actions, tokens, endgame=self._in_endgame(),
+                total_levels=total_levels,
+            )
+        # this request pays the prefill, so it drops to the bottom band; the one
+        # after it, with the prefix rebuilt, returns to the warm band
+        self._priority_current = estimate
+        self._priority_next = estimate + _PRIORITY_BAND
+        if gate is not None:
+            _diag(self, "gate")
+            gate.handover(estimate, snapshot)
+
+    def _in_endgame(self) -> bool:
+        """Whether this game has been running long enough to switch phase.
+
+        Read from the session's run_elapsed_seconds rather than a clock of its
+        own: that is the timer max_runtime_s_per_game uses to end the game, so
+        the two cannot disagree about how old a game is.
+        """
+        minutes = _endgame_start_minutes()
+        if minutes <= 0:
+            return False
+        elapsed = getattr(self, "_game_elapsed_seconds", None)
+        if elapsed is None:
+            return False          # no step has reported the clock yet
+        if elapsed < minutes * 60.0:
+            return False
+        if not getattr(self, "_endgame_announced", False):
+            self._endgame_announced = True
+            log.warning(
+                "endgame phase after %.0f minutes: ranking on what completing "
+                "the current level would bank, with no tail and no hazard",
+                elapsed / 60.0,
+            )
+        return True
+
+    def _note_request_succeeded(self) -> None:
+        """Promote to the warm band once a request has actually landed.
+
+        Called on a completed response rather than on dispatch: a retry after a
+        connection failure has rebuilt nothing, so the game stays in the bottom
+        band until one lands.
+        """
+        nxt = getattr(self, "_priority_next", None)
+        if nxt is not None:
+            self._priority_current = nxt
 
     def _trim_messages_for_context(
         self,
@@ -1676,6 +6458,7 @@ class ToolAgent:
         tools: list[dict[str, Any]] | None = None,
         preserve_recent: int = 1,
         extra_safety_tokens: int = 0,
+        force_drain: bool = False,
     ) -> list[dict[str, Any]]:
         if not messages:
             return []
@@ -1683,10 +6466,83 @@ class ToolAgent:
         history = list(messages[1:])
         preserve_recent = max(0, preserve_recent)
         budget_tokens = max(1, self._context_budget_tokens - max(0, extra_safety_tokens))
-        while history and self._estimate_request_input_tokens([system_message, *history], tools=tools) > budget_tokens:
+        # capped at half the budget: a mis-set knob should cost history depth,
+        # not erase history
+        drain_tokens = min(_context_drain_tokens(), budget_tokens // 2)
+        target_tokens = max(1, budget_tokens - drain_tokens) if drain_tokens else budget_tokens
+        floor_mode = bool(drain_tokens) and _context_drain_floor_mode()
+        # Normally eviction starts only once the estimate exceeds the budget.
+        # After the server has rejected a request the estimate is known to be
+        # wrong, so waiting for it to admit the request is too big would drop
+        # nothing at all; force_drain enters the loop already draining and lets
+        # the existing target - budget minus one drain, computed against the
+        # real budget - decide where to stop.
+        draining = bool(force_drain and drain_tokens)
+        def _head_is_summary() -> bool:
+            """A summary REQUEST at the head, not its reply.
+
+            Landing on the assistant reply leaves history starting with an
+            assistant message, which _drop_until_first_user_message then strips -
+            so the exit would drop the very thing it stopped for.
+            """
+            return bool(
+                history
+                and _message_is_summary(history[0])
+                and str(history[0].get("role", "")).strip() == "user"
+            )
+
+        # `draining` means two different things. On the normal path it is the
+        # hysteresis phase - we crossed the budget and are heading for the
+        # target - and stopping early there is exactly the point. Under
+        # force_drain it means the server rejected the request and the estimate
+        # cannot be trusted, so a summary-shaped head says nothing about whether
+        # the request now fits; stopping early there risks another rejection and
+        # another round trip.
+        def _stop_early(used_tokens: int) -> bool:
+            return (
+                _drain_stop_at_summaries()
+                and draining
+                and not force_drain
+                and used_tokens <= budget_tokens
+                and _head_is_summary()
+            )
+
+        while history:
+            used = self._estimate_request_input_tokens([system_message, *history], tools=tools)
+            over_budget = used > budget_tokens
+            if not over_budget and not draining:
+                break
+            if floor_mode and not over_budget:
+                # the mandatory phase is done; keep draining only while the NEXT
+                # drop would still leave us at or above the target. Looking
+                # ahead rather than dropping and checking is what puts a real
+                # floor under the retained history.
+                if _stop_early(used):
+                    break
+                probe = list(history)
+                if not self._drop_oldest_history_block(probe, preserve_recent=preserve_recent):
+                    break
+                after = self._estimate_request_input_tokens(
+                    [system_message, *probe], tools=tools
+                )
+                if after < target_tokens:
+                    break
+                history[:] = probe
+                continue
+            if not floor_mode and used <= (target_tokens if draining else budget_tokens):
+                break
+            if _stop_early(used):
+                break
+            draining = True
             if not self._drop_oldest_history_block(history, preserve_recent=preserve_recent):
                 break
         history = self._drop_until_first_user_message(history)
+        # A dropped message invalidates the server's prefix for this
+        # conversation, so the next request has to be prefilled from the
+        # divergence point regardless. That makes it the one moment when
+        # yielding the slot to another game costs nothing.
+        if len(history) != len(messages) - 1:
+            self._note_history_evicted()
         return [system_message, *history]
 
     def _force_reduce_messages(
@@ -1701,6 +6557,7 @@ class ToolAgent:
         history = list(messages[1:])
         if not self._drop_oldest_history_block(history, preserve_recent=max(0, preserve_recent)):
             return list(messages)
+        self._note_history_evicted()
         return [system_message, *history]
 
     def analyze(
@@ -1719,6 +6576,10 @@ class ToolAgent:
             return None
         self._ensure_session(state_path)
         self._step_env_callback = step_env
+        if self._noop_repeat_guard is not None:
+            solver_obj = getattr(step_env, "__self__", None)
+            if solver_obj is not None:
+                solver_obj.action_guard_hook = self._action_guard_hook
         self._current_valid_actions = _normalize_valid_actions(valid_actions)
 
         analyzer_log = transcript_path or (state_path.parent / f"{state_path.stem}_analyzer.txt")
@@ -1732,6 +6593,7 @@ class ToolAgent:
             previous_step_summary=self._last_step_summary,
         )
         display_action_num = _display_action_number(action_num)
+        self._wm_absorbed_in_turn = False
 
         with open(analyzer_log, "a", encoding="utf-8") as f:
             step_label = f"analysis_step={analysis_step} | " if analysis_step is not None else ""
@@ -1749,22 +6611,108 @@ class ToolAgent:
                 transcript_updated("".join(transcript_parts))
 
         append_transcript("SYSTEM PROMPT", self._system_prompt)
-        append_transcript("USER PROMPT", user_prompt)
 
         previous_history_messages = list(self._history_messages)
+        # Mirrors the history snapshot: a turn abandoned on a request failure is
+        # rolled back, and the per-game token allowance has to roll back with it
+        # or the limit stops measuring the game and starts measuring how flaky
+        # the endpoint was. The tokens were really generated - the server spent
+        # them - but the point of the limit is that the same game gets the same
+        # budget whatever the infrastructure did.
+        tokens_at_turn_start = self._session_generated_tokens
+        # The tools go with it, even though the summary must not call one. The
+        # template renders their definitions into a system block near the start
+        # of the prompt, so a request without them diverges from the first
+        # tokens and prefills the whole context: measured at 15% shared prefix
+        # against 99% with them, and as prefix reuse falling from 75% to 45%
+        # across a run. The instruction not to act, and the harness rejecting a
+        # reply that calls a tool, are what keep it from acting - not withholding
+        # the definitions.
+        self._maybe_append_rolling_summary(
+            append_transcript, self._tools(state_path),
+            request_timeout_seconds=request_timeout_seconds,
+        )
         preserve_history = True
-        messages: list[dict[str, Any]] = self._trim_messages_for_context(
-            [{"role": "system", "content": self._system_prompt}, *self._history_messages, self._build_user_message(user_prompt, current_frame)],
+        resuming_after_yield = bool(getattr(self, "_resume_after_yield", False))
+        resume_reason = str(getattr(self, "_resume_reason", "") or "yield_bare")
+        self._resume_after_yield = False
+        self._resume_reason = ""
+        if not resuming_after_yield and _persistent_functions_scope() == "turn":
+            # A resumption is the same turn, so helpers survive it either way.
+            if _persistent_functions() and getattr(self, "_kept_functions", None):
+                self._retained_clear_notice = (
+                    "Your retained functions were cleared at the start of this new analyzer turn; "
+                    "redefine any you still need."
+                )
+            self._kept_functions = {}
+        resume_mode = _yield_resume_prompt_mode() if resuming_after_yield else "full"
+        # state_only always carries the board - that IS the mode - so the
+        # image knob does not apply to it
+        resume_frame = (
+            current_frame
+            if not resuming_after_yield
+            or resume_mode == "state_only"
+            or _resume_prompt_images_enabled()
+            else None
+        )
+        if (
+            resuming_after_yield
+            and resume_mode in ("state_only", "short")
+            and _resume_diff_image_enabled()
+            and self._turn_diff_image is not None
+        ):
+            # _build_user_message consumes the pending slot, so re-arm it from
+            # the turn cache for this one message
+            self._pending_diff_image = self._turn_diff_image
+            self._pending_diff_image_label = self._turn_diff_image_label
+        if resume_mode == "state_only":
+            opening_message = self._build_user_message(
+                _yield_resume_prompt(resume_reason) + "\n\n" + _STATE_ONLY_CAPTION,
+                current_frame,
+            )
+        elif resume_mode == "short":
+            opening_message = self._build_user_message(
+                _yield_resume_prompt(resume_reason), resume_frame
+            )
+        else:
+            opening_message = self._build_user_message(
+                user_prompt, resume_frame if resuming_after_yield else current_frame
+            )
+        # log what was actually SENT, not the argument that was passed in: on a
+        # resumed invocation the short/state_only substitution happens here, so
+        # transcribing `user_prompt` would report a full opener the model never
+        # saw - and at a 60s yield budget that is most turns
+        append_transcript("USER PROMPT", _message_display_text(opening_message))
+        if resuming_after_yield:
+            # Tag EVERY resumption message, not just the short form. A yield
+            # leaves this turn's original opener in history, so the message
+            # appended here is a duplicate restatement of unchanged state
+            # (same step, level, board, diffs, ledger - a no-execution yield
+            # changes nothing). It is needed for the request that resumes the
+            # turn and is pure duplication afterwards, which is precisely what
+            # ARC3_PRUNE_CONTROL_CONTEXT exists to drop.
+            _mark_control_message(opening_message, "resume")
+        messages: list[dict[str, Any]] = [
+            {"role": "system", "content": self._system_prompt}, *self._history_messages,
+        ]
+        self._append_context_message(messages, opening_message)
+        messages = self._trim_messages_for_context(
+            messages,
             tools=self._tools(state_path),
             preserve_recent=1,
         )
         step_executed = False
+        turn_tool_call_count = 0
         captured_reasoning = ""
         latest_request_messages: list[dict[str, Any]] | None = None
         latest_request_tools: list[dict[str, Any]] | None = None
         latest_request_tool_choice: str | None = None
         latest_request_index = 0
         turn_started_at = time.monotonic()
+        # per TURN, not per session: the budget is how much the model may
+        # generate before the turn ends, and a resumption continues the same
+        # turn, so this resets where the clock does
+        self._turn_generated_tokens = 0
         yielded_control_reason: str | None = None
 
         def control_yield_reason() -> str | None:
@@ -1776,6 +6724,11 @@ class ToolAgent:
                     log.warning("analyzer stop check failed at action %d: %s", display_action_num, exc)
             if self._yield_seconds is not None and (time.monotonic() - turn_started_at) >= self._yield_seconds:
                 return "turn_time_budget"
+            if (
+                self._yield_tokens is not None
+                and self._turn_generated_tokens >= self._yield_tokens
+            ):
+                return "turn_token_budget"
             return None
 
         try:
@@ -1788,6 +6741,7 @@ class ToolAgent:
                 tools = self._tools(state_path)
                 tool_choice = _request_tool_choice(tools)
                 messages = self._trim_messages_for_context(messages, tools=tools)
+                self._maybe_handover()
                 latest_request_messages = json.loads(json.dumps(messages))
                 latest_request_tools = json.loads(json.dumps(tools))
                 latest_request_tool_choice = tool_choice
@@ -1818,9 +6772,17 @@ class ToolAgent:
                             analysis_step=analysis_step,
                             action=display_action_num,
                             request_index_within_turn=latest_request_index,
+                            chat_template_kwargs=self._harness_template_kwargs(),
                         )
                     result = self._chat_completion(messages, **request_kwargs)
+                    _calibration_note = self._calibrate_from_usage(
+                        messages, request_kwargs.get("tools"), result.usage
+                    )
+                    if _calibration_note:
+                        append_transcript("ANALYZER STATUS", _calibration_note)
                     self._accumulate_usage_tokens(result.usage)
+                    self._note_request_succeeded()
+                    _diag(self, "local")
                     if self._save_request_logs:
                         _append_request_snapshot(
                             _resolve_request_log_path(state_path),
@@ -1832,14 +6794,25 @@ class ToolAgent:
                             action=display_action_num,
                             request_index_within_turn=latest_request_index,
                             finish_reason=result.finish_reason,
+                            served_by=result.served_by,
+                            usage=result.usage,
+                            chat_template_kwargs=self._harness_template_kwargs(),
                         )
                 except requests.RequestException as exc:
                     if not _is_context_length_error(exc):
                         raise
+                    # A rejected request means the estimate was wrong, and 512
+                    # tokens only helps when it was wrong by less than that -
+                    # otherwise nothing is dropped and the fallback removes one
+                    # block per rejected round trip. force_drain evicts down to
+                    # the drain target instead, in one step, which is what the
+                    # drain is for: land clear of the ceiling rather than just
+                    # under it.
                     trimmed_messages = self._trim_messages_for_context(
                         messages,
                         tools=tools,
                         extra_safety_tokens=_CONTEXT_OVERFLOW_RETRY_TRIM_TOKENS,
+                        force_drain=True,
                     )
                     if trimmed_messages == messages:
                         trimmed_messages = self._force_reduce_messages(messages)
@@ -1851,6 +6824,21 @@ class ToolAgent:
                     )
                     messages = trimmed_messages
                     continue
+                truncated_tool_call = False
+                if str(result.finish_reason or "").strip() == "length":
+                    ladder = _reasoning_effort_ladder()
+                    if ladder and self._reasoning_effort_rung < len(ladder) - 1:
+                        self._reasoning_effort_rung += 1
+                        log.warning(
+                            "generation hit the output ceiling; reducing reasoning_effort to %r",
+                            ladder[self._reasoning_effort_rung],
+                        )
+                        append_transcript(
+                            "ANALYZER STATUS",
+                            "reasoning_effort_reduced: generation was truncated at the output "
+                            f"limit; subsequent requests use reasoning_effort="
+                            f"{ladder[self._reasoning_effort_rung]!r} until an action executes.",
+                        )
                 raw_reasoning = _extract_reasoning_text(result.message)
                 raw_content = _normalize_message_content(result.message.get("content", ""))
                 tool_calls = json.loads(json.dumps(result.message.get("tool_calls") or []))
@@ -1871,8 +6859,35 @@ class ToolAgent:
                             json.loads(raw_arguments)
                         except json.JSONDecodeError as exc:
                             malformed_argument_errors.append(f"{tool_name}: invalid JSON arguments ({exc})")
+                if malformed_argument_errors:
+                    # Drop ALL calls from this reply rather than the bad one.
+                    #
+                    # An unparseable arguments string means the reply was cut
+                    # off mid-call (finish_reason=length is the usual cause),
+                    # so anything after the truncation point is missing and an
+                    # earlier "valid" call in the same message expresses only
+                    # part of an intent - dispatching half a plan is worse than
+                    # dispatching none.
+                    #
+                    # Critically, the message must also never reach history:
+                    # servers re-parse tool calls in the incoming message list
+                    # when rendering the chat template, so one malformed call
+                    # committed to history makes EVERY later request fail at
+                    # render time, before inference - an unbreakable loop that
+                    # the HTTP retry cannot help with. Content and reasoning
+                    # are kept; only the calls are dropped, which routes this
+                    # into the existing no-tool-call path.
+                    log.warning(
+                        "dropping %d malformed tool call(s) (finish_reason=%s): %s",
+                        len(tool_calls),
+                        result.finish_reason or "?",
+                        "; ".join(malformed_argument_errors),
+                    )
+                    tool_calls = []
+                    truncated_tool_call = str(result.finish_reason or "").strip() == "length"
                 response_meta = _format_model_response_meta(
                     finish_reason=result.finish_reason,
+                    served_by=result.served_by,
                     reasoning=reasoning,
                     content=content,
                     tool_calls=tool_calls,
@@ -1889,7 +6904,8 @@ class ToolAgent:
                 if reasoning:
                     captured_reasoning = reasoning
                     append_transcript("THINKING", reasoning)
-                    assistant_message["reasoning"] = reasoning
+                    for _reasoning_key in _reasoning_history_keys():
+                        assistant_message[_reasoning_key] = reasoning
 
                 if not tool_calls:
                     if content:
@@ -1900,11 +6916,87 @@ class ToolAgent:
                         assistant_message["content"] = None
 
                     if content or reasoning:
-                        messages.append(assistant_message)
+                        if (
+                            not content
+                            and reasoning
+                            and _is_degenerate_text(reasoning)
+                        ):
+                            # A reasoning-only reply that is one character
+                            # repeated to the output cap is a serving fault, not
+                            # thinking: observed as exactly 6144 "!" from the
+                            # first token, with content_chars 0 and no tool
+                            # call, under speculative decoding. Keeping it cost
+                            # far more than the lost turn - 12k characters of
+                            # noise pushed the trimmer from 31 history messages
+                            # to 12, evicting real history, and the model then
+                            # concluded "my replies keep failing to emit the
+                            # tool call" and started writing shorter answers to
+                            # fix a formatting error it had not made.
+                            # transcript AND log: the transcript records it
+                            # where it happened, but a serving fault should be
+                            # visible while the run is going rather than only
+                            # on a later grep
+                            log.warning(
+                                "degenerate reasoning dropped: %d chars, no content "
+                                "and no tool call - the reply was a repetition loop. "
+                                "This is a serving fault, not the model reasoning.",
+                                len(reasoning),
+                            )
+                            append_transcript(
+                                "ANALYZER STATUS",
+                                "degenerate_reasoning_dropped: the reply was a "
+                                f"repetition loop ({len(reasoning)} chars) with no "
+                                "content or tool call, and was replaced in context.",
+                            )
+                            assistant_message = _mark_control_message(
+                                {"role": "assistant", "content": _DEGENERATE_CONTENT_STUB},
+                                "stub",
+                            )
+                        elif content and _is_degenerate_text(content):
+                            # a repetition loop feeds on its own output, so the
+                            # garbage must not reach the next request's context
+                            append_transcript(
+                                "ANALYZER STATUS",
+                                "degenerate_content_stubbed: assistant text was highly "
+                                "repetitive and was replaced in context.",
+                            )
+                            assistant_message = {
+                                "role": "assistant",
+                                "content": _DEGENERATE_CONTENT_STUB,
+                            }
+                        elif (
+                            not content
+                            and reasoning
+                            and _stub_dead_reasoning_enabled()
+                        ):
+                            # reasoning-only reply: nothing to continue from,
+                            # and re-sending it biases the retry toward the
+                            # same dead end. Transcript keeps the full text.
+                            assistant_message = _mark_control_message(
+                                {"role": "assistant", "content": _DEAD_REASONING_STUB},
+                                "stub",
+                            )
+                        self._append_context_message(messages, assistant_message)
                     yielded_control_reason = control_yield_reason()
                     if yielded_control_reason is not None:
                         break
                     followup_prefix = "You have not acted yet. Investigate first. "
+                    if truncated_tool_call:
+                        # the only signal the model gets that LENGTH, not
+                        # formatting, is why its call vanished
+                        followup_prefix = (
+                            "Your previous reply was CUT OFF before the tool call finished, so the "
+                            "call was incomplete and could not be executed. Keep your reasoning "
+                            "much shorter this time and emit the `python` tool call early enough "
+                            "that it completes within the output limit. Prefer one small snippet "
+                            "over a long analysis. "
+                        )
+                    elif malformed_argument_errors:
+                        followup_prefix = (
+                            "Your previous tool call had arguments that were not valid JSON, so it "
+                            "was not executed. Emit exactly one `python` tool call whose arguments "
+                            "are a well-formed JSON object. "
+                        )
                     if tool_call_markup_in_text:
                         followup_prefix = (
                             "You did not call a tool. We detected `<tool_call>` markup inside your reasoning or assistant text, "
@@ -1912,10 +7004,31 @@ class ToolAgent:
                             "Emit exactly one `python` tool call directly as your next response. "
                             "Do not place `<tool_call>` markup inside reasoning, explanation, or notes. "
                         )
+                    # the clause names where transient facts SHOULD go, so it
+                    # holds only while that section is carried; a mode that
+                    # drops it would be pointing the model nowhere
+                    _transient_clause = (
+                        " \u2014 current positions and this turn's events go in "
+                        "`Recent findings:`. "
+                        if "Recent findings" in _advertised_section_labels()
+                        # "only" wants something to contrast against, and the
+                        # clause that supplied it names a section this mode has
+                        # dropped; keep the contrast, drop the destination
+                        else ", not current positions or what just happened. "
+                    )
                     followup_prompt = (
                         f"{followup_prefix}"
                         "Then investigate and revise your working world model of what the level contains, what actions appear to do, what the current goal seems to be, and what plan looks best. "
-                        "If helpful, include short world-model update lines such as `World model:`, `Goal model:`, `Action model:`, `Recent findings:`, `Open questions:`, `Plan:`, or `Cross-level notes:`. "
+                        # the same narrowing the opener applies: advertising a section a
+                        # restricted mode drops invites the model to write
+                        # where it will never read it back
+                        f"If helpful, include memory update lines such as "
+                        f"{_section_list_phrase(_advertised_section_labels(), 'or')}. "
+                        "`World model:` holds durable environment rules"
+                        + (" only" if "Recent findings" in _advertised_section_labels() else "")
+                        + f"{_transient_clause}"
+                        "Each section you write replaces the stored one entirely, "
+                        "so restate what is still true, not only what is new. "
                         "Call the `python` tool with code that inspects `current_frame`, `previous_frame`, `last_transition`, `history`, or `valid_actions` -- use `current_frame.segmentation` as the primary view, and `.ascii` only for a small specific region -- "
                         "compare `previous_frame` to `current_frame` for the most recent change, "
                         "derives a compact board summary, programs a small search or scorer over candidate actions or short sequences, "
@@ -1923,7 +7036,12 @@ class ToolAgent:
                         f"{TOOL_CALL_FORMAT_GUIDANCE}"
                     )
                     append_transcript("USER PROMPT", followup_prompt)
-                    messages.append({"role": "user", "content": followup_prompt})
+                    self._append_context_message(
+                        messages,
+                        _mark_control_message(
+                            {"role": "user", "content": followup_prompt}, "nudge"
+                        )
+                    )
                     continue
 
                 if content:
@@ -1931,7 +7049,8 @@ class ToolAgent:
                     append_transcript("ASSISTANT", content)
                     assistant_message["content"] = content
                 assistant_message["tool_calls"] = tool_calls
-                messages.append(assistant_message)
+                turn_tool_call_count += len(tool_calls)
+                self._append_context_message(messages, assistant_message)
 
                 for tool_index, tool_call in enumerate(tool_calls):
                     function = tool_call.get("function", {}) if isinstance(tool_call, dict) else {}
@@ -1954,8 +7073,17 @@ class ToolAgent:
                     dispatch = self._dispatch_tool(state_path, tool_name, arguments)
                     if dispatch.step_executed:
                         step_executed = True
+                        if self._reasoning_effort_rung >= 0 and dispatch.made_progress:
+                            # the turn achieved something, so restore the server
+                            # default; an inert action leaves the reduction in
+                            # place rather than inviting another truncation
+                            log.info(
+                                "action changed the board; restoring default reasoning_effort"
+                            )
+                            self._reasoning_effort_rung = -1
                     append_transcript(f"TOOL RESULT: {tool_name}", _render_tool_result_display(dispatch.content))
-                    messages.append(
+                    self._append_context_message(
+                        messages,
                         {
                             "role": "tool",
                             "tool_call_id": tool_call.get("id", ""),
@@ -1977,8 +7105,44 @@ class ToolAgent:
                     break
 
         except requests.RequestException as exc:
+            if _is_read_timeout(exc) and _yield_on_timeout_enabled():
+                # Keep this turn's completed exchanges and hand control back to
+                # the solver, which re-invokes on the same game state.
+                append_transcript(
+                    "ANALYZER STATUS",
+                    f"request_timeout (yielding, history preserved): {exc}",
+                )
+                log.warning(
+                    "analyzer request timed out at action %d; yielding with %d tool call(s) preserved: %s",
+                    display_action_num,
+                    turn_tool_call_count,
+                    exc,
+                )
+                self._resume_after_yield = True
+                self._resume_reason = "timeout"
+                if latest_request_messages is not None:
+                    _write_prompt_log_snapshot(
+                        prompt_log,
+                        model_id=self._model.model_id,
+                        base_url=self._model.base_url,
+                        display_action_num=display_action_num,
+                        analysis_step=analysis_step,
+                        request_index=latest_request_index,
+                        messages=latest_request_messages,
+                        tools=latest_request_tools,
+                        tool_choice=latest_request_tool_choice,
+                        transcript="".join(transcript_parts),
+                    )
+                # preserve_history stays True, so the finally block commits
+                # this turn's exchanges exactly as a normal yield would.
+                return AnalyzerTurnResult(
+                    step_executed=False,
+                    reasoning=captured_reasoning,
+                    yielded_control=True,
+                )
             append_transcript("ANALYZER STATUS", f"request_error: {exc}")
             preserve_history = False
+            self._session_generated_tokens = tokens_at_turn_start
             if latest_request_messages is not None:
                 _write_prompt_log_snapshot(
                     prompt_log,
@@ -1997,6 +7161,7 @@ class ToolAgent:
         except Exception as exc:
             append_transcript("ANALYZER STATUS", f"error: {exc}")
             preserve_history = False
+            self._session_generated_tokens = tokens_at_turn_start
             if latest_request_messages is not None:
                 _write_prompt_log_snapshot(
                     prompt_log,
@@ -2013,10 +7178,15 @@ class ToolAgent:
             log.warning("analyzer failed at action %d: %s", display_action_num, exc)
             return None
         finally:
+            if step_executed and not self._wm_absorbed_in_turn:
+                self._turns_without_wm_update += 1
             if preserve_history:
                 self._history_messages = self._persistent_history_messages(messages, tools=self._tools(state_path))
             else:
                 self._history_messages = previous_history_messages
+            solver_obj = getattr(self._step_env_callback, "__self__", None)
+            if solver_obj is not None and hasattr(solver_obj, "action_guard_hook"):
+                solver_obj.action_guard_hook = None
             self._step_env_callback = None
             self._current_valid_actions = []
 
@@ -2036,6 +7206,7 @@ class ToolAgent:
             f"request_safety_margin_tokens: {self._request_safety_margin_tokens}\n"
             f"tool_output_tokens: {self._tool_output_tokens}\n"
             f"yield_seconds: {self._yield_seconds if self._yield_seconds is not None else 'disabled'}\n"
+            f"yield_tokens: {self._yield_tokens if self._yield_tokens is not None else 'disabled'}\n"
             f"available_tools: python\n"
             f"python_timeout_seconds: {self._python_timeout}\n"
             f"history_messages: {len(self._history_messages)}\n"
@@ -2056,6 +7227,31 @@ class ToolAgent:
                 tool_choice=latest_request_tool_choice,
                 transcript="".join(transcript_parts),
             )
+        # An invocation that executed nothing will be re-invoked by the solver
+        # on the same game state, so whatever it appends next is a duplicate
+        # restatement. This covers BOTH non-execution exits: the time budget
+        # (yielded_control_reason set) and the tool-step budget (loop condition
+        # went false, no yield reason). Execution and yielding are mutually
+        # exclusive, so `not step_executed` is the whole condition.
+        self._resume_after_yield = not step_executed
+        if yielded_control_reason is not None:
+            self._resume_reason = (
+                "yield_tools" if turn_tool_call_count > 0 else "yield_bare"
+            )
+        else:
+            self._resume_reason = (
+                "tool_steps" if turn_tool_call_count > 0 else "yield_bare"
+            )
+        if not step_executed and turn_tool_call_count == 0 and self._last_step_summary:
+            # no dispatch ran, so _dispatch_tool never marked the summary
+            # stale; without this the next opener re-narrates an older
+            # execution as if it had just happened (or claims nothing has run
+            # yet) directly beneath this turn's own reasoning.
+            self._last_step_summary = {
+                **self._last_step_summary,
+                "stale": True,
+                "stale_reason": "no_tool_call",
+            }
         return AnalyzerTurnResult(
             step_executed=step_executed,
             reasoning=captured_reasoning,
diff --git a/ARC3-Inference/inference/agent/vision_context.py b/ARC3-Inference/inference/agent/vision_context.py
index bb8232f..e3dadff 100644
--- a/ARC3-Inference/inference/agent/vision_context.py
+++ b/ARC3-Inference/inference/agent/vision_context.py
@@ -3,6 +3,7 @@ from __future__ import annotations
 
 import base64
 import io
+from collections import defaultdict
 import os
 from typing import Any
 
@@ -39,6 +40,24 @@ def current_grid_image_enabled() -> bool:
     return multimodal_context() == "current_grid"
 
 
+_RESAMPLE_FILTERS = {
+    "nearest": Image.Resampling.NEAREST,
+    "bilinear": Image.Resampling.BILINEAR,
+    "bicubic": Image.Resampling.BICUBIC,
+    "lanczos": Image.Resampling.LANCZOS,
+}
+
+
+def multimodal_resample() -> Image.Resampling:
+    """Upscale filter for grid images. Nearest neighbor (the default) keeps
+    cells as flat exact-palette blocks; interpolating filters produce softer
+    edges that sit closer to a vision encoder's natural-image training
+    distribution, which has measurably helped on some models. Model-dependent:
+    test against the model actually served."""
+    raw = os.environ.get("MULTIMODAL_RESAMPLE", "").strip().lower()
+    return _RESAMPLE_FILTERS.get(raw, Image.Resampling.NEAREST)
+
+
 def current_grid_image_upscale() -> int:
     raw = os.environ.get("MULTIMODAL_UPSCALE", "").strip()
     if not raw:
@@ -63,7 +82,7 @@ def frame_to_png_data_url(frame: Frame, *, upscale: int | None = None) -> str:
             value = row[col_idx] if col_idx < len(row) else 0
             pixels[col_idx, row_idx] = ARC_COLOR_MAP.get(int(value), ARC_COLOR_MAP[0])
     if scale > 1:
-        image = image.resize((cols * scale, rows * scale), Image.Resampling.NEAREST)
+        image = image.resize((cols * scale, rows * scale), multimodal_resample())
 
     buffer = io.BytesIO()
     image.save(buffer, format="PNG")
@@ -80,3 +99,234 @@ def current_grid_image_part(frame: Frame | None) -> dict[str, Any] | None:
             "url": frame_to_png_data_url(frame),
         },
     }
+
+
+# Off-palette neutral for unchanged cells in diff images. The ARC palette's
+# indices 0-5 span a white-to-black grayscale ramp, so no gray (or black, or
+# white) can be used without colliding with real cell values; a dark
+# desaturated navy is unambiguous.
+DIFF_UNCHANGED_RGB: tuple[int, int, int] = (16, 24, 48)
+
+
+_ANIMATION_IMAGE_MODES = ("off", "peak", "composite")
+
+
+def _animation_image_mode(name: str) -> str:
+    """One of off, peak, composite. Anything else raises.
+
+    These two are the only enum knobs among a set of otherwise boolean ARC3_*
+    variables, so `=1` is the natural thing to write - and silently meant off,
+    which cost a whole run to notice. Failing at startup is better than a
+    feature that is configured and quietly absent.
+    """
+    mode = os.environ.get(name, "off").strip().lower()
+    if mode not in _ANIMATION_IMAGE_MODES:
+        raise ValueError(
+            f"{name}={mode!r} is not valid. Use one of "
+            f"{', '.join(_ANIMATION_IMAGE_MODES)} - this knob is not a boolean."
+        )
+    return mode
+
+
+def animation_death_image_mode() -> str:
+    """off | peak | composite, for the fatal action of a game over.
+
+    Separate from ARC3_ANIMATION_IMAGE because the two fire on different
+    conditions and a game-over turn already carries two images, so a third is
+    a cost worth deciding on its own.
+    """
+    return _animation_image_mode("ARC3_ANIMATION_DEATH_IMAGE")
+
+
+def animation_image_mode() -> str:
+    """off | peak | composite.
+
+    Only ever attached when the action left the board area unchanged, which is
+    exactly when the board image the model receives is uninformative: the
+    action did something, and no frame it can reach shows what.
+    """
+    return _animation_image_mode("ARC3_ANIMATION_IMAGE")
+
+
+def animation_composite_part(
+    chain: list[tuple[tuple[int, ...], ...]],
+    transient_cells: list[tuple[int, int]],
+    *,
+    border: int,
+) -> dict[str, Any] | None:
+    """Every transient cell in the last colour it held before reverting.
+
+    Not a board state: cells from different moments composited onto one grid.
+    The alternative - a single real frame - misses a sweep that touches
+    different regions at different times, which has no peak moment.
+
+    Everything else is dimmed to the diff image's off-palette navy, so the
+    image carries only what no reachable frame contains. That includes cells
+    that changed just once (visible in current_frame anyway) and the border,
+    where the budget bar ticks on every action and would otherwise be the
+    brightest thing in the picture.
+    """
+    if not chain or not transient_cells:
+        return None
+    if not current_grid_image_enabled():
+        return None
+    final = chain[-1]
+    rows = len(final)
+    cols = max((len(row) for row in final), default=0)
+    if rows <= 0 or cols <= 0:
+        return None
+
+    wanted = set(transient_cells)
+    last_seen: dict[tuple[int, int], int] = {}
+    held: dict[tuple[int, int], int] = defaultdict(int)
+    for grid in chain[:-1]:
+        for row_idx, row in enumerate(grid):
+            final_row = final[row_idx] if row_idx < len(final) else ()
+            for col_idx, value in enumerate(row):
+                cell = (row_idx, col_idx)
+                if cell not in wanted:
+                    continue
+                if row_idx < border or row_idx >= rows - border:
+                    continue
+                if col_idx < border or col_idx >= cols - border:
+                    continue
+                final_value = final_row[col_idx] if col_idx < len(final_row) else None
+                if value == final_value:
+                    continue
+                value = int(value)
+                if last_seen.get(cell) != value:
+                    # a cell that flickers between colours restarts the count,
+                    # so the duration belongs to the colour actually painted
+                    held[cell] = 0
+                # later frames overwrite earlier ones, so this ends up holding
+                # the LAST non-final colour the cell showed
+                last_seen[cell] = value
+                held[cell] += 1
+    if not last_seen:
+        return None
+
+    # Brightness carries DURATION: a cell blended toward its true colour in
+    # proportion to how many frames it held it. This separates two mechanics
+    # that are otherwise identical in a flat composite - a 4x4 block falling
+    # down a column touches each cell for a frame or two and renders uniformly
+    # dim, while a column filling from the top holds its highest cell for the
+    # whole animation and its lowest for a moment, so it renders as a gradient
+    # pointing back at the source.
+    span = max(1, len(chain) - 1)
+    image = Image.new("RGB", (cols, rows), DIFF_UNCHANGED_RGB)
+    pixels = image.load()
+    for (row_idx, col_idx), value in last_seen.items():
+        colour = ARC_COLOR_MAP.get(value, ARC_COLOR_MAP[0])
+        # floored so a single-frame cell stays legible rather than vanishing
+        weight = max(0.35, min(1.0, held[(row_idx, col_idx)] / span))
+        pixels[col_idx, row_idx] = tuple(
+            int(round(base + weight * (target - base)))
+            for base, target in zip(DIFF_UNCHANGED_RGB, colour)
+        )
+    scale = current_grid_image_upscale()
+    if scale > 1:
+        image = image.resize((cols * scale, rows * scale), multimodal_resample())
+    buffer = io.BytesIO()
+    image.save(buffer, format="PNG")
+    encoded = base64.b64encode(buffer.getvalue()).decode("ascii")
+    return {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{encoded}"}}
+
+
+def animation_peak_part(
+    chain: list[tuple[tuple[int, ...], ...]],
+    transient_cells: list[tuple[int, int]],
+) -> tuple[dict[str, Any] | None, int]:
+    """The real frame showing the most transient cells at once, and its index.
+
+    Unlike the composite this is a state the game actually displayed, so
+    nothing in it is fabricated - at the cost of missing anything that was
+    hidden at a different moment.
+    """
+    if not chain or not transient_cells or len(chain) <= 2:
+        return None, -1
+    if not current_grid_image_enabled():
+        return None, -1
+    final = chain[-1]
+    wanted = set(transient_cells)
+    best_index, best_count = -1, -1
+    for index, grid in enumerate(chain[1:]):
+        count = 0
+        for row_idx, col_idx in wanted:
+            row = grid[row_idx] if row_idx < len(grid) else ()
+            final_row = final[row_idx] if row_idx < len(final) else ()
+            value = row[col_idx] if col_idx < len(row) else None
+            final_value = final_row[col_idx] if col_idx < len(final_row) else None
+            if value != final_value:
+                count += 1
+        if count > best_count:
+            best_index, best_count = index, count
+    if best_index < 0 or best_count <= 0:
+        return None, -1
+
+    grid = chain[1:][best_index]
+    rows = len(grid)
+    cols = max((len(row) for row in grid), default=0)
+    image = Image.new("RGB", (cols, rows), ARC_COLOR_MAP[0])
+    pixels = image.load()
+    for row_idx, row in enumerate(grid):
+        for col_idx in range(cols):
+            value = row[col_idx] if col_idx < len(row) else 0
+            pixels[col_idx, row_idx] = ARC_COLOR_MAP.get(int(value), ARC_COLOR_MAP[0])
+    scale = current_grid_image_upscale()
+    if scale > 1:
+        image = image.resize((cols * scale, rows * scale), multimodal_resample())
+    buffer = io.BytesIO()
+    image.save(buffer, format="PNG")
+    encoded = base64.b64encode(buffer.getvalue()).decode("ascii")
+    return (
+        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{encoded}"}},
+        best_index,
+    )
+
+
+def diff_image_part(
+    previous_frame: Frame | None, current_frame: Frame | None
+) -> dict[str, Any] | None:
+    """Render a change-highlight image: cells that differ between the two
+    frames appear in their (new) true palette color; unchanged cells are
+    dimmed to an off-palette dark navy. Returns None when disabled, frames
+    are missing, or shapes differ (a full redraw makes a diff meaningless)."""
+    if previous_frame is None or current_frame is None:
+        return None
+    if not current_grid_image_enabled():
+        return None
+    prev_grid, cur_grid = previous_frame.grid, current_frame.grid
+    rows = len(cur_grid)
+    cols = max((len(row) for row in cur_grid), default=0)
+    if rows <= 0 or cols <= 0:
+        return None
+    if len(prev_grid) != rows or max(
+        (len(row) for row in prev_grid), default=0
+    ) != cols:
+        return None
+
+    scale = current_grid_image_upscale()
+    image = Image.new("RGB", (cols, rows), DIFF_UNCHANGED_RGB)
+    pixels = image.load()
+    for row_idx in range(rows):
+        prev_row = prev_grid[row_idx]
+        cur_row = cur_grid[row_idx]
+        for col_idx in range(cols):
+            prev_value = prev_row[col_idx] if col_idx < len(prev_row) else 0
+            cur_value = cur_row[col_idx] if col_idx < len(cur_row) else 0
+            if prev_value != cur_value:
+                pixels[col_idx, row_idx] = ARC_COLOR_MAP.get(
+                    int(cur_value), ARC_COLOR_MAP[0]
+                )
+    if scale > 1:
+        image = image.resize((cols * scale, rows * scale), multimodal_resample())
+
+    buffer = io.BytesIO()
+    image.save(buffer, format="PNG")
+    encoded = base64.b64encode(buffer.getvalue()).decode("ascii")
+    return {
+        "type": "image_url",
+        "image_url": {
+            "url": f"data:image/png;base64,{encoded}",
+        },
+    }
diff --git a/ARC3-Inference/inference/framework/run.py b/ARC3-Inference/inference/framework/run.py
index 4d4accf..fc7d80c 100644
--- a/ARC3-Inference/inference/framework/run.py
+++ b/ARC3-Inference/inference/framework/run.py
@@ -419,6 +419,7 @@ def _make_solver(
         analyzer_timeout=getattr(args, "analyzer_timeout", 120),
         max_actions_per_game=args.max_actions,
         max_runtime_s_per_game=max_runtime_minutes_per_game * 60.0,
+        max_generated_tokens_per_game=args.max_generated_tokens_per_game,
         concurrency=effective_concurrency,
         save_request_logs=bool(args.analyzer_save_request_logs),
         start_local_server=start_local_server,
@@ -1222,6 +1223,15 @@ def main() -> None:
     parser.add_argument("--experiment-dir", dest="experiment_dir", default="")
     parser.add_argument("--max-actions", type=int, default=None)
     parser.add_argument("--max-runtime-minutes", type=float, default=None)
+    parser.add_argument(
+        "--max-generated-tokens-per-game",
+        type=int,
+        default=None,
+        help="Give up on a game once it has generated this many tokens. A "
+             "wall-clock limit means something different on every backend and "
+             "moves hour to hour on a hosted API; this is what makes an "
+             "offline run comparable with a Kaggle one.",
+    )
     parser.add_argument("--max-experiment-runtime-minutes", type=float, default=None)
     parser.add_argument("--max-experiment-runtime-hours", type=float, default=None)
     parser.add_argument("--n-passes", dest="n_passes", type=int, default=1)
diff --git a/ARC3-Inference/inference/framework/solver.py b/ARC3-Inference/inference/framework/solver.py
index 0b69d62..e40976c 100644
--- a/ARC3-Inference/inference/framework/solver.py
+++ b/ARC3-Inference/inference/framework/solver.py
@@ -7,6 +7,7 @@ import contextlib
 import copy
 import functools
 import html
+import logging
 import json
 import os
 import re
@@ -23,7 +24,13 @@ import arcengine
 import taaf.game
 from taaf.solver import Solver
 
+from inference.utils.animation import (
+    collapse_frames,
+    normalize_frames,
+    summarize_animation,
+)
 from inference.agent.action_names import (
+    reset_exposed,
     to_engine_action,
     to_model_action,
     to_model_actions,
@@ -34,7 +41,11 @@ from inference.agent.runtime_state import (
     RUNTIME_STATE_FILENAME,
     write_runtime_state,
 )
-from inference.agent.tool_agent import ToolAgent
+from inference.agent.tool_agent import (
+    _PRIORITY_UNTRIMMED_BASE,
+    ToolAgent,
+    _priority_gate,
+)
 from inference.framework.kaggle import (
     DEFAULT_QWEN_MODEL_DATASET_SOURCE,
     DEFAULT_SERVED_MODEL_NAME,
@@ -53,10 +64,242 @@ from inference.utils.viewer_artifacts import (
     reset_raw_events_sidecar,
 )
 
+log = logging.getLogger(__name__)
+
 AnalyzerFactory = Callable[[taaf.game.Game, int], Any]
 
-ANALYZER_RETRY_BACKOFF_SECONDS = 1.0
+def _env_number(name: str, default: float, cast=float) -> float:
+    """A numeric env override that cannot take the run down at import time."""
+    try:
+        return cast(os.environ.get(name, "") or default)
+    except (TypeError, ValueError):
+        return cast(default)
+
+
+ANALYZER_RETRY_BACKOFF_SECONDS = max(
+    0.0, _env_number("ARC3_ANALYZER_RETRY_BACKOFF_SECONDS", 10.0)
+)
+# A request that fails outright - a dead endpoint, a refused connection - is
+# reported as retryable and the solver goes back to the top of the loop. With
+# no limit that is an infinite spin: nothing monitors the local server, so if
+# it exits mid-run every game retries once a second until its runtime budget is
+# gone, and the run finishes reporting whatever each game had reached with no
+# crash marker anywhere. Giving up after a run of consecutive failures makes
+# that visible - the run ends in seconds rather than at the wall clock, and the
+# games carry a note saying why. 0 restores the unlimited behaviour.
+# Negative means keep retrying; 0 means give up on the first failure. No
+# max(0, ...) clamp here - it would flatten a negative to zero and turn
+# "never give up" into "give up at once".
+# Kaggle requires an action within 15 minutes of the run starting. SGLang
+# needs about 10.5 of those to load weights - 6 on the main model, 3 more on the
+# MTP draft head, which is a second full pass over the same 206 shards - and the
+# notebook blocks on /health before playing anything, so a slow load spends the
+# whole allowance before a single action is issued and the submission scores
+# zero. With this set, the first N games to start execute one RESET straight
+# against the engine before touching the analyzer, which satisfies the clock
+# while the server is still loading; the HTTP retry loop then covers the rest of
+# the wait. 0 disables.
+WARMUP_ACTION_GAMES = _env_number("ARC3_WARMUP_ACTION_GAMES", 0, int)
+
+
+MAX_CONSECUTIVE_ANALYZER_FAILURES = _env_number("ARC3_MAX_ANALYZER_FAILURES", 10, int)
 DEFAULT_CANCEL_DRAIN_TIMEOUT_SECONDS = 120.0
+
+# No-op guard: width of the border frame treated as HUD (progress bars,
+# timers). 0 disables the guard. Simple v1 heuristic; see step_env.
+# arcengine renders a frame after every internal step(), so one action can come
+# back as a short animation whose intermediate frames the harness has always
+# discarded. Off by default: new information for the model, not a fix.
+ANIMATION_ENABLED = os.environ.get("ARC3_ANIMATION", "0").strip().lower() in (
+    "1",
+    "true",
+    "yes",
+    "on",
+)
+NOOP_GUARD_BORDER_WIDTH = max(0, int(os.environ.get("ARC3_NOOP_GUARD_BORDER", "4")))
+# Print one line per action(...) call into the snippet's stdout. Costs a line of
+# tool output per call, which is charged against tool_output_tokens.
+# The patch-110 rewordings, behind one switch. They replaced "changed nothing"
+# with "no change inside the board area" across every guard message, because the
+# old phrasing was false in the case that matters - the HUD usually DID change.
+# The reasoning still looks right, but they shipped alongside the outcome line
+# in a run that lost 43% of the previous competition score, so the two need to
+# be separable before either can be judged. Off restores the wording of the run
+# that scored 4.05.
+NEW_NOOP_WORDING = os.environ.get(
+    "ARC3_NEW_CHANGED_PROMPTS", "0"
+).strip().lower() in ("1", "true", "yes", "on")
+ACTION_ECHO = os.environ.get("ARC3_ACTION_ECHO", "0").strip().lower() in (
+    "1",
+    "true",
+    "yes",
+    "on",
+)
+# When an action changes nothing, the sandbox arms a per-snippet latch and the
+# NEXT action(...) call is refused. Off restores the older behaviour of letting
+# the snippet keep acting on a state its own beliefs no longer describe.
+STALE_STATE_BLOCK = os.environ.get(
+    "ARC3_STALE_STATE_BLOCK", "0"
+).strip().lower() in ("1", "true", "yes", "on")
+
+
+# Stops a batch at the first action that changes nothing inside the board, so a
+# wrong movement model costs one action rather than the whole sequence. It used
+# to be implied by ARC3_NOOP_GUARD_BORDER being positive, which conflated the
+# crop width with whether the guard runs at all - the border is also what
+# gameplay_changed means, and that is wanted independently. Off by default:
+# the guard converts actions into turns, which is a good trade only when tokens
+# are cheap relative to the action budget.
+# Level 1 pays the smallest share of the score - one part in N - and it is
+# where the model knows least, so it probes hardest: repeating an action to see
+# whether the repeat differs, retrying something that killed it to learn what
+# killed it. The guards read that as waste. Set to 2 to let level 1 run
+# unguarded and start guarding at level 2, where an action is worth more and
+# the model has a working model of the game.
+#
+# The batch no-op stop is deliberately NOT covered: it costs one action of a
+# planned sequence rather than refusing a deliberate probe, and a wrong
+# movement model is exactly as expensive on level 1 as anywhere. Nor is the
+# terminal latch, which stops a snippet acting after a level-up or a death -
+# that is a board that no longer exists, not a judgement about the action.
+GUARDS_FROM_LEVEL = max(1, int(os.environ.get("ARC3_GUARDS_FROM_LEVEL", "1") or 1))
+
+
+def _guards_active(level: Any) -> bool:
+    """Whether the per-action guards apply at this level."""
+    try:
+        return int(level) >= GUARDS_FROM_LEVEL
+    except (TypeError, ValueError):
+        return True  # unknown level: guard rather than not
+
+
+BATCH_NOOP_BLOCK = os.environ.get(
+    "ARC3_BATCH_NOOP_BLOCK", "0"
+).strip().lower() in ("1", "true", "yes", "on")
+
+
+_BLOCKED_VERDICTS = {
+    "known_noop": "BLOCKED (known no-op in this exact board state; never executed)",
+    "known_death": "BLOCKED (ended a previous attempt from this exact board state; never executed)",
+}
+
+
+def _action_verdict(item: dict) -> str:
+    """The one-word account of what an executed action did.
+
+    Shared with the per-action trace so the echo introduces no second
+    vocabulary for the model to learn."""
+    if item.get("game_over"):
+        return "GAME OVER (attempt ended immediately after this action)"
+    if item.get("run_complete") or item.get("done"):
+        return "RUN COMPLETE"
+    if item.get("level_completed"):
+        return "LEVEL COMPLETED"
+    changed = item.get("gameplay_changed")
+    if changed is False:
+        return (
+            "NO-OP (executed, no change inside the board area)"
+            if NEW_NOOP_WORDING
+            else "NO-OP (executed, changed nothing)"
+        )
+    if changed is True:
+        return "effect"
+    return "executed"
+
+
+def _build_action_echo(
+    requested_displays: list[str],
+    executed_payloads: list[dict],
+    stop_reason: str | None,
+) -> str:
+    """One line per action(...) call, for the snippet's own stdout.
+
+    A single action produces no per-action trace at all - the trace is built
+    only for batches, game overs and refusals - so the snippet's only account
+    of it arrives a turn later in the opener. Inside a loop that is too late to
+    branch on. This fills the gap at the moment of the call, in the trace's own
+    words, without repeating the trace for batches that already get one.
+    """
+    if not requested_displays:
+        return ""
+    if len(requested_displays) == 1:
+        if executed_payloads:
+            return f"[action] {requested_displays[0]} -> {_action_verdict(executed_payloads[0])}"
+        reason = f" | stop_reason: {stop_reason}" if stop_reason else ""
+        return f"[action] {requested_displays[0]} -> not executed{reason}"
+    ran = ", ".join(
+        str(item.get("action_display") or item.get("action_name") or "?")
+        for item in executed_payloads
+    )
+    line = (
+        f"[action] batch of {len(requested_displays)} actions, "
+        f"{len(executed_payloads)} executed"
+    )
+    if ran:
+        line += f": {ran}"
+    if stop_reason:
+        line += f" | stop_reason: {stop_reason}"
+    return line
+
+
+def _build_action_trace(
+    requested_displays: list[str],
+    executed_payloads: list[dict],
+    blocked_kind: str | None = None,
+) -> list[str]:
+    """One self-describing line per submitted action: index, name, verdict.
+
+    A guard refusal is called out at its own position rather than lumped in
+    with the generic SKIPPED verdict: the prompt tells the model to treat this
+    trace as the authoritative account of what happened, so a refusal that
+    reads as "skipped" would send it looking for an upstream cause that does
+    not exist. Positions AFTER the refusal are genuinely skipped and keep the
+    plain verdict.
+    """
+    trace: list[str] = []
+    blocked_position = len(executed_payloads) + 1 if blocked_kind else None
+    for position, name in enumerate(requested_displays, 1):
+        if blocked_position is not None and position == blocked_position:
+            trace.append(
+                f"{position}: {name} -> "
+                + _BLOCKED_VERDICTS.get(blocked_kind or "", "BLOCKED (never executed)")
+            )
+            continue
+        if position <= len(executed_payloads):
+            verdict = _action_verdict(executed_payloads[position - 1])
+        else:
+            verdict = "SKIPPED (never executed)"
+        trace.append(f"{position}: {name} -> {verdict}")
+    return trace
+
+
+def _interior_gameplay_changed(
+    previous_grid: list[list[int]] | None,
+    new_grid: list[list[int]] | None,
+    *,
+    border: int = NOOP_GUARD_BORDER_WIDTH,
+) -> bool:
+    """True when any cell outside the border frame differs between grids.
+
+    Falls back to a plain inequality check (treat as changed) when the guard
+    is disabled, grids are missing, shapes differ, or the border swallows the
+    whole grid — the guard must fail open.
+    """
+    if border <= 0 or previous_grid is None or new_grid is None:
+        return previous_grid != new_grid
+    rows = len(previous_grid)
+    cols = len(previous_grid[0]) if rows else 0
+    if rows != len(new_grid) or any(len(r) != cols for r in (previous_grid[0:1] + new_grid[0:1])):
+        return True
+    if rows <= 2 * border or cols <= 2 * border:
+        return previous_grid != new_grid
+    for r in range(border, rows - border):
+        prev_row = previous_grid[r]
+        new_row = new_grid[r]
+        if prev_row[border:cols - border] != new_row[border:cols - border]:
+            return True
+    return False
+
 _LOCAL_SERVER_PROCESS_ENV_KEYS = (
     "LOCAL_ANALYZER_API_KEY",
     "OPENAI_API_KEY",
@@ -90,6 +333,24 @@ def artifact_stem(value: str) -> str:
     return re.sub(r"[^A-Za-z0-9_.-]+", "_", value)
 
 
+def _animation_chain(
+    previous_grid: tuple[tuple[int, ...], ...],
+    state: taaf.game.GameState | None,
+) -> list[tuple[tuple[int, ...], ...]]:
+    """The pre-action board, then each distinct state the action passed through.
+
+    `state.frame` is only `raw.frame[-1]`; the rest of that list is the
+    animation. Runs of identical grids collapse here rather than at render
+    time, because a single action can return dozens of frames and they stay in
+    memory until the next action replaces them.
+    """
+    if state is None:
+        return []
+    return collapse_frames(
+        previous_grid, normalize_frames(getattr(state.raw, "frame", None))
+    )
+
+
 def _grid_from_state(state: taaf.game.GameState | None) -> tuple[tuple[int, ...], ...]:
     if state is None:
         return ()
@@ -113,7 +374,7 @@ def _engine_action_names(game: taaf.game.Game) -> list[str]:
             name = arcengine.GameAction.from_id(int(action_id)).name
         except Exception:
             continue
-        if name == "RESET":
+        if name == "RESET" and not reset_exposed():
             continue
         if name not in names:
             names.append(name)
@@ -127,6 +388,46 @@ def _model_mouse_action_data(
     return {"row": int(data.get("y", 0)), "col": int(data.get("x", 0))}
 
 
+def _guard_action_data(action: Any) -> dict[str, Any]:
+    """Action data in the row/col form the guard signature expects.
+
+    ACTION6 carries x/y from the engine; `_format_action_display` already
+    translates it the same way, so the ledger and the guard agree on what
+    identifies a click."""
+    data = dict(getattr(action, "data", None) or {})
+    if getattr(getattr(action, "id", None), "name", "") == "ACTION6":
+        return _model_mouse_action_data(data)
+    return data
+
+
+def _stale_repeat(stale_after: str, action: Any) -> bool:
+    """Is this the same action that just changed nothing?
+
+    Compared on the DISPLAY form, which carries mouse coordinates - a click
+    somewhere else after a no-op click is a different action, and the repeat
+    guard's override is meaningful for it.
+    """
+    return str(stale_after).strip() == _format_action_display(
+        action.id.name, dict(action.data)
+    ).strip()
+
+
+def _stale_detail(action: Any, stale_after: str) -> str:
+    return (
+        f"{_format_action_display(action.id.name, dict(action.data))} was not "
+        f"executed. The action immediately before it, {stale_after}, was a "
+        + ("no-op - it produced no change inside the board area - so positions "
+           if NEW_NOOP_WORDING else
+           "no-op - it changed nothing outside the board border - so positions ")
+        +
+        "and paths computed since then may be wrong, which is why execution "
+        "has been halted here. Re-read current_frame and last_action_call_result, "
+        "work out why nothing moved, then act on what you find. "
+        "(StaleStateActionError can be caught if a snippet has good reason to "
+        "keep acting through no-ops.)"
+    )
+
+
 def _format_action_display(
     action_name: str, action_data: dict[str, Any] | None = None
 ) -> str:
@@ -182,6 +483,14 @@ class _HarnessGameSession:
     analysis_step: int = 0
     last_engine_action: str | None = None
     token_baseline: int = 0
+    # token_baseline is a cursor: it is dragged forward after every step so the
+    # engine can be told what that step cost, and so cannot answer "how much
+    # has this game spent". This one is captured once and never moved.
+    game_token_baseline: int = 0
+    # The last executed action's animation, or None. One record, not a
+    # history: the model reads it via animation() in the turn that follows,
+    # and older animations are almost never what it is asking about.
+    animation_record: dict[str, Any] | None = None
     _viewer_events_flushed: int = field(default=0, init=False, repr=False)
 
     def current_frame(self) -> Frame:
@@ -243,6 +552,30 @@ class _HarnessGameSession:
             return None
         return max(0.1, min(candidates))
 
+    def token_limit_reached(self) -> bool:
+        """Whether this game has generated its allowance.
+
+        Read from the analyzer rather than summed per step: should_stop is
+        passed into the analyzer and runs at every loop boundary, so reading a
+        counter that only advances when an action executes would leave a game
+        that deliberates without acting running past its allowance. The
+        analyzer's accumulator advances on every response.
+
+        Measured against a baseline captured at game start, which is 0 for the
+        per-game analyzer the solver builds but need not be if an
+        analyzer_factory hands back a shared one.
+        """
+        # getattr, not attribute access: the notebook unpickles a solver whose
+        # __dict__ was captured before this field existed, and __setstate__ does
+        # a plain dict update - it applies no dataclass defaults. A bare access
+        # would raise AttributeError here on the first turn of every game, since
+        # should_stop() runs this every time.
+        limit = getattr(self.solver, "max_generated_tokens_per_game", None)
+        if limit is None or limit <= 0:
+            return False
+        spent = _analyzer_reported_tokens(self.analyzer) - self.game_token_baseline
+        return spent >= limit
+
     def should_stop(self) -> bool:
         run = self.game.game_run
         if run is None or run.state != "playing":
@@ -258,8 +591,15 @@ class _HarnessGameSession:
             and self.action_count >= self.solver.max_actions_per_game
         ):
             return True
+        if self.token_limit_reached():
+            return True
         return False
 
+    def _queue_position(self) -> int:
+        """Position in the dispatch order, used only to break ties in the
+        untrimmed priority band so games do not all queue at one value."""
+        return max(0, int(self.game_index)) * 100 + max(0, int(self.pass_index))
+
     def play(self) -> None:
         run = self.game.game_run
         assert run is not None, "TAAF starts games before invoking the solver."
@@ -267,12 +607,51 @@ class _HarnessGameSession:
         self.transcript_path.parent.mkdir(parents=True, exist_ok=True)
         self.transcript_path.touch(exist_ok=True)
         self.token_baseline = _analyzer_reported_tokens(self.analyzer)
-        self.seed_initial_history()
-        self.write_runtime_state()
-        self._append_initial_viewer_event()
-        self.write_viewer_payload()
+        self.game_token_baseline = self.token_baseline
+        # A game that has not started yet sits in the top band, so every game
+        # gets going in queue order before any that has already trimmed comes
+        # back for a second turn.
+        gate = _priority_gate()
+        if gate is not None:
+            deadline = (
+                self.started_at + self.solver.max_runtime_s_per_game
+                if self.solver.max_runtime_s_per_game is not None else None
+            )
+            soft_remaining = self.solver.soft_time_remaining_seconds()
+            if soft_remaining is not None:
+                soft_deadline = time.monotonic() + soft_remaining
+                deadline = min(deadline, soft_deadline) if deadline is not None else soft_deadline
+            gate.configure_clock(self.started_at, deadline)
+            gate.acquire(_PRIORITY_UNTRIMMED_BASE - self._queue_position())
+        try:
+            self.seed_initial_history()
+            self.write_runtime_state()
+            self._append_initial_viewer_event()
+            self.write_viewer_payload()
+            self._play_inner()
+        finally:
+            if gate is not None:
+                gate.release()
+
+    def _play_inner(self) -> None:
+        run = self.game.game_run
+        if self.solver._claim_warmup_slot():
+            # Straight to the engine - _execute_auto_reset takes the same path
+            # and needs no analyzer, so this works with the endpoint still
+            # loading. One action off this game's budget buys the whole
+            # submission a deadline it would otherwise miss.
+            # warning, not info: _configure_logging only runs under run.py, so
+            # the notebook has no handler and falls back to logging's lastResort,
+            # which emits WARNING and above. An info line would vanish exactly
+            # where this message matters most.
+            log.warning(
+                "%s: warmup RESET issued before the analyzer is reachable",
+                run.game_id,
+            )
+            self._execute_auto_reset()
         try:
             retry_analysis_step: int | None = None
+            consecutive_failures = 0
             while not self.should_stop():
                 if (
                     _is_engine_game_over(self.game)
@@ -310,15 +689,42 @@ class _HarnessGameSession:
                 if result is None:
                     raise RuntimeError("Analyzer did not return a result.")
                 if result.retryable_failure:
+                    consecutive_failures += 1
+                    if (
+                        MAX_CONSECUTIVE_ANALYZER_FAILURES >= 0
+                        and consecutive_failures > MAX_CONSECUTIVE_ANALYZER_FAILURES
+                    ):
+                        log.error(
+                            "%s: analyzer failed %d times in a row; giving up on this "
+                            "game. The endpoint is probably down - nothing here can "
+                            "restart it.",
+                            run.game_id,
+                            consecutive_failures,
+                        )
+                        run.solver_note = (
+                            f"analyzer failed {consecutive_failures} times consecutively"
+                        )
+                        break
                     retry_analysis_step = analysis_step
                     if self.should_stop():
                         break
                     time.sleep(ANALYZER_RETRY_BACKOFF_SECONDS)
                     continue
 
+                # a turn that came back at all clears the run: the limit is for
+                # an endpoint that has stopped answering, not for a game that
+                # hits an occasional blip
+                consecutive_failures = 0
                 retry_analysis_step = None
                 if getattr(result, "yielded_control", False):
-                    retry_analysis_step = analysis_step
+                    # Reuse the step id only for a pure thinking continuation.
+                    # If this invocation already executed actions, the resumed
+                    # invocation opens with a fresh batch-summary prompt - a new
+                    # logical turn - and must get a fresh analysis_step, or the
+                    # viewer merges two turns into one card (and sums their
+                    # token usage).
+                    if not result.step_executed:
+                        retry_analysis_step = analysis_step
                     continue
                 if not result.step_executed:
                     continue
@@ -586,6 +992,14 @@ class _HarnessGameSession:
         }
 
     def step_env(self, arguments: dict[str, Any]) -> dict[str, Any]:
+        if str(arguments.get("query") or "").strip() == "animation":
+            # Routed through step_env so no analyzer-protocol signature has to
+            # change. Executes nothing and spends no action budget.
+            return {
+                "executed": False,
+                "query": "animation",
+                "record": self.animation_record if ANIMATION_ENABLED else None,
+            }
         requested_actions, error = self._normalize_actions(arguments)
         if error is not None or requested_actions is None:
             return self._error_payload(error or "Could not parse action request.")
@@ -601,10 +1015,84 @@ class _HarnessGameSession:
             for action in requested_actions
         ]
 
+        guard_hook = getattr(self, "action_guard_hook", None)
+        blocked_detail: str | None = None
+        blocked_kind: str | None = None
+
+        # Checked AFTER the per-position guards below, so a known death or a
+        # known no-op takes precedence: both name a specific action and offer an
+        # override, which is more actionable than a general staleness warning.
+        stale_after = (
+            arguments.get("stale_after")
+            if STALE_STATE_BLOCK and _guards_active(_level_number(self.game))
+            else None
+        )
+
         for batch_index, action in enumerate(requested_actions, start=1):
             if self.should_stop():
                 stop_reason = "stopped"
                 break
+            if (
+                stale_after
+                and batch_index == 1
+                and _stale_repeat(stale_after, action)
+            ):
+                # the model is re-issuing the action that just changed nothing;
+                # the staleness message is the correct one here, so take it
+                # before the repeat guard can claim the turn
+                blocked_kind = "stale_state"
+                blocked_detail = _stale_detail(action, stale_after)
+                stop_reason = blocked_kind
+                break
+            if guard_hook is not None:
+                # Per-position check: the hash is recomputed from the CURRENT
+                # board, so position N of a batch is judged against the state
+                # its predecessors actually produced - not the state the batch
+                # started in. This is what lets a fatal move be caught no
+                # matter which route reached it.
+                # The engine stores mouse coordinates as x/y; every consumer
+                # that identifies an action - the ledger, the action display,
+                # the guard signature - works in row/col. Passing action.data
+                # raw made every click share the signature "MOUSE" with no
+                # coordinates, so one no-op click poisoned clicking ANYWHERE
+                # from that board state. On a mouse-only game that is terminal:
+                # observed as 175 replies and one executed action in a run.
+                verdict = guard_hook(
+                    _grid_from_state(self.game.current_state),
+                    action.id.name,
+                    _guard_action_data(action),
+                    _level_number(self.game),
+                    [
+                        (later.id.name, _guard_action_data(later))
+                        for later in requested_actions[batch_index - 1:]
+                    ],
+                )
+                if verdict:
+                    blocked_kind, blocked_detail = verdict
+                    stop_reason = blocked_kind
+                    break
+            if stale_after and batch_index == 1:
+                # Ordering against the repeat guards above:
+                #
+                # Different action - the repeat guard wins. It names the action
+                # and its override is meaningful: the model may have reason to
+                # believe circumstances changed since that earlier visit.
+                #
+                # SAME action as the one that just no-opped - this message wins,
+                # and the check is placed before the guards for that case (see
+                # the early branch above). Telling a model that just watched X
+                # do nothing to "issue it again and it will execute" is advice
+                # to repeat a failure; worse, the guard's own override is
+                # already satisfied by this call, so without the swap the action
+                # would simply run with no message at all.
+                blocked_kind = "stale_state"
+                blocked_detail = _stale_detail(action, stale_after)
+                stop_reason = blocked_kind
+                break
+            # anything that reaches execution clears the flag: it can only
+            # happen when the check above declined to fire, and leaving a stale
+            # marker set behind an executed action would be a latent surprise
+            stale_after = None
             if action.id.value not in self.game.current_state.available_actions:
                 message = f"{_format_action_display(action.id.name, dict(action.data))} is not valid right now."
                 if executed_payloads:
@@ -636,8 +1124,35 @@ class _HarnessGameSession:
             if payload.get("level_completed"):
                 stop_reason = "level_completed"
                 break
+            if (
+                BATCH_NOOP_BLOCK
+                and NOOP_GUARD_BORDER_WIDTH > 0
+                and batch_size >= 2
+                and not payload.get("gameplay_changed", True)
+            ):
+                stop_reason = "no_op_action"
+                break
 
         if not executed_payloads:
+            if blocked_detail is not None:
+                # Nothing ran at all: report the refusal itself rather than a
+                # generic error, so the sandbox can raise the right exception.
+                return {
+                    **self._error_payload(blocked_detail),
+                    "executed": False,
+                    "stop_reason": blocked_kind,
+                    "stop_detail": blocked_detail,
+                    "requested_count": batch_size,
+                    "executed_count": 0,
+                    "stopped_early": True,
+                    "requested_actions": requested_displays,
+                    "skipped_actions": list(requested_displays),
+                    # the refused action is position 1 here, since nothing ran.
+                    # Omitting it left the next turn's opener saying "stopped
+                    # before None" - the later assignment for the partial-batch
+                    # case is never reached on this path.
+                    "blocked_action": requested_displays[0] if requested_displays else None,
+                }
             return self._error_payload("No action was executed.")
 
         final_payload = dict(executed_payloads[-1])
@@ -657,12 +1172,64 @@ class _HarnessGameSession:
         final_payload["stopped_early"] = len(executed_payloads) < batch_size
         if stop_reason is not None:
             final_payload["stop_reason"] = stop_reason
+        if final_payload["stopped_early"]:
+            final_payload["skipped_actions"] = requested_displays[len(executed_payloads):]
+        for item in executed_payloads:
+            if item.get("game_over"):
+                final_payload["fatal_action"] = str(item.get("action_display") or "")
+                break
+        if batch_size > 1 or final_payload.get("game_over") or blocked_kind:
+            final_payload["action_trace"] = _build_action_trace(
+                requested_displays, executed_payloads, blocked_kind
+            )
+            final_payload["gameplay_changed_per_action"] = [
+                bool(item.get("gameplay_changed", True)) for item in executed_payloads
+            ]
+        if ACTION_ECHO:
+            echo = _build_action_echo(requested_displays, executed_payloads, stop_reason)
+            if echo:
+                # consumed by the sandbox, which prints it and pops it: leaving
+                # it in the returned dict shows the model the same line twice,
+                # once on stdout and once in the payload it reads back
+                final_payload["action_echo"] = echo
+        if blocked_detail is not None:
+            final_payload["stop_detail"] = blocked_detail
+            final_payload["blocked_action"] = requested_displays[len(executed_payloads)]
+        if stop_reason == "no_op_action":
+            noop_index = len(executed_payloads)
+            final_payload["no_op_action_index"] = noop_index
+            final_payload["no_op_action"] = final_payload["executed_actions"][-1]
+            skipped = requested_displays[noop_index:]
+            final_payload["stop_detail"] = (
+                f"Action {noop_index} of {batch_size} "
+                + (
+                    f"({final_payload['executed_actions'][-1]!r}) EXECUTED but produced no "
+                    f"change inside the board area - only the outer {NOOP_GUARD_BORDER_WIDTH} "
+                    "cells, where a timer or remaining-steps bar usually sits, could have moved. "
+                    if NEW_NOOP_WORDING else
+                    f"({final_payload['executed_actions'][-1]!r}) EXECUTED but changed nothing "
+                    f"outside the {NOOP_GUARD_BORDER_WIDTH}-cell border (treated as HUD). "
+                )
+                +
+                f"Actions {noop_index + 1}-{batch_size} ({skipped!r}) were SKIPPED and did "
+                "not execute. Indices are 1-based positions in your submitted list. "
+                f"Net effect: only the first {max(0, noop_index - 1)} executed action(s) "
+                "changed the board; the final executed action moved nothing. Do NOT "
+                "update your position estimate by counting submitted moves - re-locate "
+                "the player from current_frame before planning. "
+                "The per-action verdicts are listed in action_trace. "
+                "This stop is about one action from one board state, not about batch "
+                "size - batching is still the right approach, and the stop is what keeps "
+                "a wrong movement model from costing the whole sequence."
+            )
         self.write_viewer_payload()
         return final_payload
 
     def _execute_auto_reset(self) -> None:
         action = arcengine.ActionInput(id=arcengine.GameAction.RESET, data={})
-        self._execute_action(action, batch_index=1, batch_size=1, generated_tokens=0)
+        self._execute_action(
+            action, batch_index=1, batch_size=1, generated_tokens=0, automatic=True
+        )
 
     def _execute_action(
         self,
@@ -672,6 +1239,7 @@ class _HarnessGameSession:
         batch_size: int,
         generated_tokens: int | None = None,
         flush_viewer_payload: bool = True,
+        automatic: bool = False,
     ) -> dict[str, Any]:
         previous_grid = _grid_from_state(self.game.current_state)
         previous_completed = int(self.game.current_state.levels_completed)
@@ -690,17 +1258,40 @@ class _HarnessGameSession:
             step=self.action_count,
             level=_level_number(self.game),
         )
-        self.history_entries.append(
-            HistoryEntry(action=action_display, frame=current_frame)
-        )
-        self.write_runtime_state()
 
         completed = int(new_state.levels_completed)
         reward = float(completed - previous_completed) / max(
             1.0, float(self.game.number_of_levels)
         )
         raw_state = new_state.raw.state
-        board_changed = previous_grid != _grid_from_state(new_state)
+        new_grid = _grid_from_state(new_state)
+        board_changed = previous_grid != new_grid
+        gameplay_changed = _interior_gameplay_changed(previous_grid, new_grid)
+        no_op = gameplay_changed is False
+        animation_summary = None
+        if ANIMATION_ENABLED and action.id.name != "RESET":
+            # The auto-reset after a death runs through here too. Letting it
+            # touch the record would clear the fatal action's animation before
+            # the model ever reads it - and a death is the case where those
+            # frames matter most, since the board it now sees is the restarted
+            # level and the death exists nowhere else.
+            chain = _animation_chain(previous_grid, new_state)
+            animation_summary = summarize_animation(chain)
+            # ONE record, replaced by every executed action - animated or not.
+            # A no-op that animated is the case this exists for, and the guards
+            # refuse the NEXT action without executing it, so the record still
+            # describes the animated one when the model reads the opener.
+            self.animation_record = (
+                {
+                    "action_display": action_display,
+                    "chain": chain,
+                    "summary": animation_summary,
+                    "step": self.action_count,
+                    "level": _level_number(self.game),
+                }
+                if animation_summary
+                else None
+            )
         level_completed = bool(
             new_state.just_won_level and raw_state != arcengine.GameState.WIN
         )
@@ -713,6 +1304,8 @@ class _HarnessGameSession:
             "state": raw_state.name,
             "valid_actions": to_model_actions(_engine_action_names(self.game)),
             "board_changed": board_changed,
+            "gameplay_changed": gameplay_changed,
+            "no_op": no_op,
             "done": raw_state == arcengine.GameState.WIN,
             "level_completed": level_completed,
             "game_over": raw_state == arcengine.GameState.GAME_OVER,
@@ -728,6 +1321,25 @@ class _HarnessGameSession:
             "batch_size": batch_size,
             **self.timing_payload(),
         }
+        if animation_summary:
+            payload["animation"] = animation_summary
+            # the raw chain, for rendering an image in the opener. Dropped by
+            # _compact_action_result, so it never reaches the model as text.
+            payload["animation_chain"] = chain
+        # Store the individual action before step_env folds results into a batch.
+        # Keep only compact metadata, without grids, animation chains, or clocks.
+        payload["automatic"] = automatic
+        result_keys = (
+            "executed", "action_num", "level", "score", "state", "valid_actions",
+            "board_changed", "gameplay_changed", "no_op", "done", "level_completed",
+            "game_over", "run_complete", "action_name", "action_data", "action_display",
+            "animation", "automatic",
+        )
+        individual_result = copy.deepcopy({k: payload[k] for k in result_keys if k in payload})
+        self.history_entries.append(
+            HistoryEntry(action=action_display, frame=current_frame, result=individual_result)
+        )
+        self.write_runtime_state()
         self._append_action_viewer_event(payload, current_frame)
         if flush_viewer_payload:
             self.write_viewer_payload()
@@ -743,6 +1355,12 @@ class HarnessSolver(Solver):
     analyzer_timeout: float | None = 120.0
     max_actions_per_game: int | None = None
     max_runtime_s_per_game: float | None = None
+    # Generated tokens a game may spend before it gives up, counted from the
+    # analyzer's own accumulator so it measures what the model produced rather
+    # than what was billed. A wall-clock limit means something different on
+    # every backend - and on a hosted API it moves hour to hour - so this is
+    # what makes an offline run comparable with a Kaggle one. None disables.
+    max_generated_tokens_per_game: int | None = None
     concurrency: int = 16
     save_request_logs: bool = False
     start_local_server: bool = False
@@ -816,6 +1434,7 @@ class HarnessSolver(Solver):
         state.pop("_local_servers", None)
         state.pop("_local_server_original_env", None)
         state.pop("_worker_pool", None)
+        state.pop("_warmup_lock", None)
         return state
 
     def __setstate__(self, state: dict[str, Any]) -> None:
@@ -829,6 +1448,7 @@ class HarnessSolver(Solver):
         self._local_servers = []
         self._local_server_original_env = {}
         self._worker_pool = None
+        self._warmup_lock = threading.Lock()
 
     def __deepcopy__(self, memo: dict[int, Any]) -> "HarnessSolver":
         cls = type(self)
@@ -845,6 +1465,8 @@ class HarnessSolver(Solver):
                 object.__setattr__(new, key, {})
             elif key == "_worker_pool":
                 object.__setattr__(new, key, None)
+            elif key == "_warmup_lock":
+                object.__setattr__(new, key, threading.Lock())
             else:
                 object.__setattr__(new, key, copy.deepcopy(value, memo))
         return new
@@ -879,6 +1501,12 @@ class HarnessSolver(Solver):
         )
 
     def _setup(self) -> None:
+        # Created here rather than in __init__ because the notebook unpickles a
+        # solver whose __dict__ predates these attributes, and __setstate__
+        # applies no dataclass defaults. _setup runs after unpickling, so the
+        # attributes always exist by the time a game asks for a slot.
+        self._warmup_remaining = WARMUP_ACTION_GAMES
+        self._warmup_lock = threading.Lock()
         if self.start_local_server:
             self._start_local_servers()
         self._worker_pool = ThreadPoolExecutor(
@@ -1178,6 +1806,23 @@ class HarnessSolver(Solver):
             return None
         return self._local_servers[int(game_index) % len(self._local_servers)]
 
+    def _claim_warmup_slot(self) -> bool:
+        """Whether this game should spend one action before the analyzer is up.
+
+        Claimed by whichever runners start first rather than by index: task
+        creation order and semaphore fairness both happen to give indices 0..N,
+        but neither is guaranteed by the API, and the cost of being wrong is a
+        game that never fires the action the deadline depends on.
+        """
+        lock = getattr(self, "_warmup_lock", None)
+        if lock is None:
+            return False
+        with lock:
+            if getattr(self, "_warmup_remaining", 0) <= 0:
+                return False
+            self._warmup_remaining -= 1
+            return True
+
     def _make_analyzer(
         self,
         game: taaf.game.Game,
@@ -1190,6 +1835,9 @@ class HarnessSolver(Solver):
             model=self.model,
             timeout=self.analyzer_timeout,
             save_request_logs=self.save_request_logs,
+            # queue position, so the untrimmed priority band has a deterministic
+            # order instead of every game tying at the same value
+            dispatch_index=index,
             api_key=(
                 local_server.api_key
                 if local_server is not None
diff --git a/ARC3-Inference/inference/utils/animation.py b/ARC3-Inference/inference/utils/animation.py
new file mode 100644
index 0000000..d2a1484
--- /dev/null
+++ b/ARC3-Inference/inference/utils/animation.py
@@ -0,0 +1,333 @@
+"""Animation awareness.
+
+``arcengine`` renders a frame after every internal ``step()``, so a single
+action can come back as a short animation. ``GameState.frame`` is only the last
+of those, and the harness consumed nothing else - so every intermediate frame
+was discarded before the agent could see it. On some games the whole effect of
+an action lives there: the board before and the board after are identical, and
+reading the final frame cannot possibly reveal what happened.
+
+Two shapes occur. Where cells change and change BACK, the information is
+genuinely hidden and worth spending prompt tokens on. Where every cell changes
+at most once, the frames are motion interpolation toward a state the final
+frame already shows, and there is nothing to find - those animations are
+reported nowhere, because a line about them would cost tokens to say that
+nothing is missing.
+"""
+from __future__ import annotations
+
+from collections import defaultdict
+from typing import Any, Sequence
+
+from inference.utils.grid_utils import ARC_COLOR_CHARS, format_grid_ascii
+
+Grid = tuple[tuple[int, ...], ...]
+
+# Retrieval budget.
+#
+# Why a diff timeline rather than the frames themselves: one 64x64 grid renders
+# as roughly 4,100 characters, and a single action can return dozens of frames.
+# Whole frames are never affordable against a ~1,024-token tool budget, so
+# retrieval collapses identical frames and reports only the cells that changed
+# between the rest. A single frame can still be read verbatim, cropped.
+# No step limit: the timeline crosses the pipe as data and the model prints
+# only what it chooses, so dropping steps served a prompt budget that no longer
+# applies. It also cost a real misreading - a step count reported as omitted
+# was read as "those steps had no changes", the exact inverse.
+ANIMATION_MAX_CELLS_PER_STEP = 24
+ANIMATION_MAX_TOTAL_CELLS = 80
+
+
+def normalize_frames(raw_frames: Any) -> list[Grid]:
+    """Engine frame list to hashable, comparable grids."""
+    frames: list[Grid] = []
+    for frame in raw_frames or ():
+        rows = frame.tolist() if hasattr(frame, "tolist") else frame
+        frames.append(tuple(tuple(int(cell) for cell in row) for row in rows or ()))
+    return frames
+
+
+def collapse_frames(before: Grid | None, frames: Sequence[Grid]) -> list[Grid]:
+    """The pre-action board, then each distinct state the action passed through.
+
+    Runs of identical grids collapse to one entry; non-adjacent repeats stay
+    separate, because a grid at positions 2 and 7 with something between them
+    is two distinct moments and the timeline should show both transitions.
+
+    ``before`` is prepended and is NOT optional. A cell showing X before the
+    action, Y during it and X again at the end arrives from the engine as
+    [Y, Y, X] - one transition, which would not register as transient. With the
+    board before it is [X, Y, Y, X], which gives two. Without this the very
+    case the feature exists for is invisible.
+    """
+    collapsed: list[Grid] = []
+    if before is not None:
+        collapsed.append(before)
+    for grid in frames:
+        if collapsed and collapsed[-1] == grid:
+            continue
+        collapsed.append(grid)
+    return collapsed
+
+
+def _transient_cells(chain: Sequence[Grid]) -> list[tuple[int, int]]:
+    """Cells whose value changes two or more times across the chain.
+
+    A cell that changes once and stays changed is visible in the final frame,
+    so the agent loses nothing by never seeing the frames between. A cell that
+    changes twice has shown something that is no longer there - a rejected
+    click, a consumed attempt, a sprite crossing a background cell. Those are
+    exactly the pixels no frame the agent can reach will ever contain.
+    """
+    changes: defaultdict[tuple[int, int], int] = defaultdict(int)
+    for before, after in zip(chain, chain[1:]):
+        rows = max(len(before), len(after))
+        for row in range(rows):
+            before_row = before[row] if row < len(before) else ()
+            after_row = after[row] if row < len(after) else ()
+            for col in range(max(len(before_row), len(after_row))):
+                old = before_row[col] if col < len(before_row) else None
+                new = after_row[col] if col < len(after_row) else None
+                if old != new:
+                    changes[(row, col)] += 1
+    return sorted(cell for cell, count in changes.items() if count >= 2)
+
+
+def summarize_animation(chain: Sequence[Grid]) -> dict[str, Any] | None:
+    """Compact metadata for one action, or ``None`` if it did not animate.
+
+    ``None`` for the single-frame case means ordinary actions carry no extra
+    tokens at all - the key is simply absent from the action result.
+
+    ``frames`` counts engine frames after collapsing and excludes the prepended
+    pre-action board, so it answers "how many distinct states did this action
+    pass through" rather than describing the internal representation.
+    """
+    if len(chain) <= 2:
+        return None
+    transient = _transient_cells(chain)
+    summary: dict[str, Any] = {
+        "frames": len(chain) - 1,
+        "transient_pixels": len(transient),
+    }
+    if transient:
+        rows = [cell[0] for cell in transient]
+        cols = [cell[1] for cell in transient]
+        # Inclusive [top, left, bottom, right]: enough to point at the region
+        # without spending tokens on a coordinate list.
+        summary["transient_bbox"] = [min(rows), min(cols), max(rows), max(cols)]
+    return summary
+
+
+def _bbox_phrase(bbox: Sequence[int] | None) -> str:
+    """Where the transient cells are, collapsing degenerate ranges.
+
+    A single cell reads as "at (20,30)", a single row or column as a span in
+    the other axis only; "rows 20-20, cols 30-30" is noise the model has to
+    parse before noticing it means one cell.
+    """
+    if not bbox:
+        return ""
+    top, left, bottom, right = bbox
+    if top == bottom and left == right:
+        return f" at ({top},{left})"
+    if top == bottom:
+        return f" in row {top}, cols {left}-{right}"
+    if left == right:
+        return f" in col {left}, rows {top}-{bottom}"
+    return f" around rows {top}-{bottom}, cols {left}-{right}"
+
+
+def describe_animation(
+    summary: dict[str, Any] | None,
+    *,
+    action_display: str | None,
+    gameplay_changed: bool | None,
+    game_over: bool = False,
+    timeline: bool = True,
+) -> str:
+    """The opener line, or "" when there is nothing worth saying.
+
+    Silent unless cells changed and changed back. Where every cell changed at
+    most once the final frame already shows the outcome, and a line saying so
+    would spend tokens to report that nothing is missing.
+    """
+    if not summary:
+        return ""
+    transient = int(summary.get("transient_pixels") or 0)
+    if transient <= 0:
+        return ""
+    frames = summary.get("frames")
+    cells = "1 cell changed and changed back" if transient == 1 else (
+        f"{transient} cells changed and changed back"
+    )
+    where = _bbox_phrase(summary.get("transient_bbox"))
+    name = f" (`{action_display}`)" if action_display else ""
+    # point at whichever accessor is actually available
+    where_to_look = (
+        "`last_animation_timeline`"
+        if timeline
+        else "`last_animation_frames`"
+    )
+
+    if game_over:
+        # Follows the BAR RULE block, which has just walked the model through
+        # history[-3] and history[-2]. This says what those two frames cannot
+        # contain, which is precisely what "transient" means.
+        return (
+            f"That fatal action animated over {frames} frames, and inside it "
+            f"{cells}{where}. Those cells are in neither "
+            f"`history[-3].frame` nor `history[-2].frame` - read {where_to_look} to see "
+            "what happened between them."
+        )
+    if gameplay_changed is False:
+        return (
+            f"The last executed action{name} animated over {frames} frames and left the "
+            f"board area unchanged, but it was not a no-op: {cells}{where}. "
+            "Whatever this action did is visible only there - "
+            f"read {where_to_look} before concluding it failed."
+        )
+    return (
+        f"The last executed action{name} animated over {frames} frames. "
+        f"{cells[0].upper()}{cells[1:]}{where}. {where_to_look} shows them."
+    )
+
+
+def _color_char(value: int | None) -> str:
+    if value is None:
+        return "?"
+    return ARC_COLOR_CHARS[max(0, min(15, int(value)))]
+
+
+def _cell_at(grid: Grid, row: int, col: int) -> int | None:
+    if row >= len(grid):
+        return None
+    line = grid[row]
+    return line[col] if col < len(line) else None
+
+
+def _diff_cells(before: Grid, after: Grid) -> list[tuple[int, int, int | None, int | None]]:
+    rows = max(len(before), len(after))
+    changes: list[tuple[int, int, int | None, int | None]] = []
+    for row in range(rows):
+        cols = max(
+            len(before[row]) if row < len(before) else 0,
+            len(after[row]) if row < len(after) else 0,
+        )
+        for col in range(cols):
+            old = _cell_at(before, row, col)
+            new = _cell_at(after, row, col)
+            if old != new:
+                changes.append((row, col, old, new))
+    return changes
+
+
+def _bbox_text(cells: Sequence[tuple[int, int, Any, Any]]) -> str:
+    rows = [cell[0] for cell in cells]
+    cols = [cell[1] for cell in cells]
+    return f"rows {min(rows)}-{max(rows)}, cols {min(cols)}-{max(cols)}"
+
+
+def _format_changes(
+    cells: Sequence[tuple[int, int, int | None, int | None]], budget: int
+) -> list[str]:
+    """Changed cells grouped by colour transition, one compact line each.
+
+    Strings rather than nested coordinate lists because the caller renders with
+    json.dumps(indent=2), which would put every [row, col] pair on four lines.
+    """
+    grouped: dict[str, list[str]] = {}
+    for row, col, old, new in cells[:budget]:
+        key = f"{_color_char(old)}>{_color_char(new)}"
+        grouped.setdefault(key, []).append(f"({row},{col})")
+    lines = [f"{key} @ {' '.join(points)}" for key, points in grouped.items()]
+    if len(cells) > budget:
+        lines.append(f"... {len(cells) - budget} further changed cells omitted")
+    return lines
+
+
+def build_animation_frames(record: dict[str, Any] | None) -> list[dict[str, Any]]:
+    """Every distinct state the action passed through, as FrameView payloads.
+
+    The pre-action board is dropped here: the model already has it as
+    `previous_frame`, and including it would make index 0 mean something
+    different from the step numbers in the timeline.
+
+    Sends the grid as well as the ascii so `segmentation` works on the far
+    side. That doubles the payload, but it crosses a pipe rather than the
+    prompt - what reaches the model is only what its own code prints.
+    """
+    if not record:
+        return []
+    chain = list(record.get("chain") or ())
+    if len(chain) <= 2:
+        return []
+    step = record.get("step") or 0
+    level = record.get("level") or 0
+    return [
+        {
+            "ascii": format_grid_ascii(grid),
+            "step": step,
+            "level": level,
+            "shape": (len(grid), max((len(row) for row in grid), default=0)),
+            "grid": [list(row) for row in grid],
+        }
+        for grid in chain[1:]
+    ]
+
+
+def build_animation_view(record: dict[str, Any] | None) -> dict[str, Any]:
+    """The diff timeline: which cells changed at each step.
+
+    The cheap first look. Reading a frame verbatim is no longer this
+    function's job - `last_transition.frames()` hands back FrameView objects
+    and the model crops their ascii the same way it crops current_frame, with
+    its own helpers rather than a region parameter it has to guess.
+    """
+    empty = {
+        "error": (
+            "No animation is available: the last executed action returned a single frame."
+        )
+    }
+    if not record:
+        return empty
+    chain: list[Grid] = list(record.get("chain") or ())
+    if len(chain) <= 2:
+        return empty
+
+    header: dict[str, Any] = {
+        "action": record.get("action_display"),
+        # excludes the prepended pre-action board, matching the metadata
+        "frames": len(chain) - 1,
+    }
+
+    steps: list[dict[str, Any]] = []
+    for position in range(1, len(chain)):
+        cells = _diff_cells(chain[position - 1], chain[position])
+        if cells:
+            steps.append({"_position": position, "_cells": cells})
+
+    rendered: list[dict[str, Any]] = []
+    remaining = ANIMATION_MAX_TOTAL_CELLS
+    for step in steps:
+        cells = step["_cells"]
+        entry: dict[str, Any] = {
+            "step": step["_position"],
+            "changed": len(cells),
+            "bbox": _bbox_text(cells),
+        }
+        budget = min(ANIMATION_MAX_CELLS_PER_STEP, max(0, remaining))
+        if len(cells) > ANIMATION_MAX_CELLS_PER_STEP or budget == 0:
+            # Too broad to enumerate usefully: a colour-transition census says
+            # more per token than an arbitrarily truncated cell list.
+            census: dict[str, int] = {}
+            for _, _, old, new in cells:
+                key = f"{_color_char(old)}>{_color_char(new)}"
+                census[key] = census.get(key, 0) + 1
+            entry["transitions"] = census
+        else:
+            entry["changes"] = _format_changes(cells, budget)
+            remaining -= min(len(cells), budget)
+        rendered.append(entry)
+
+    return {**header, "steps": rendered}
diff --git a/ARC3-Inference/inference/utils/frame_diff.py b/ARC3-Inference/inference/utils/frame_diff.py
new file mode 100644
index 0000000..a49b067
--- /dev/null
+++ b/ARC3-Inference/inference/utils/frame_diff.py
@@ -0,0 +1,307 @@
+"""Object-level frame diff core, shared by the sandbox runtime and the host.
+
+This module must stay dependency-free (stdlib only, no imports): its source is
+injected verbatim into the sandbox bootstrap template (see
+python_tool_sandbox.py), which runs under ``python -I -S`` and cannot import
+project modules. The host imports it normally for the auto frame-diff feature.
+"""
+
+
+def _node_topleft(node):
+    points = node.get("boundary") or []
+    if not points:
+        return (0, 0)
+    return (min(p[0] for p in points), min(p[1] for p in points))
+
+def _hungarian(cost):
+    # Minimum-cost assignment (Jonker-Volgonant style with potentials).
+    # cost: n x m matrix with n <= m. Returns list row -> column.
+    n = len(cost)
+    m = len(cost[0]) if n else 0
+    INF = float("inf")
+    u = [0.0] * (n + 1)
+    v = [0.0] * (m + 1)
+    p = [0] * (m + 1)      # p[j] = row assigned to column j (1-based; 0 = none)
+    way = [0] * (m + 1)
+    for i in range(1, n + 1):
+        p[0] = i
+        j0 = 0
+        minv = [INF] * (m + 1)
+        used = [False] * (m + 1)
+        while True:
+            used[j0] = True
+            i0 = p[j0]
+            delta = INF
+            j1 = 0
+            for j in range(1, m + 1):
+                if used[j]:
+                    continue
+                cur = cost[i0 - 1][j - 1] - u[i0] - v[j]
+                if cur < minv[j]:
+                    minv[j] = cur
+                    way[j] = j0
+                if minv[j] < delta:
+                    delta = minv[j]
+                    j1 = j
+            for j in range(m + 1):
+                if used[j]:
+                    u[p[j]] += delta
+                    v[j] -= delta
+                else:
+                    minv[j] -= delta
+            j0 = j1
+            if p[j0] == 0:
+                break
+        while True:
+            j1 = way[j0]
+            p[j0] = p[j1]
+            j0 = j1
+            if j0 == 0:
+                break
+    assignment = [-1] * n
+    for j in range(1, m + 1):
+        if p[j]:
+            assignment[p[j] - 1] = j - 1
+    return assignment
+
+def _node_bbox(node):
+    points = node.get("boundary") or []
+    if not points:
+        return (0, 0, 0, 0)
+    rows = [p[0] for p in points]; cols = [p[1] for p in points]
+    return (min(rows), min(cols), max(rows), max(cols))
+
+def _node_center2(node):
+    # bbox center in half-cell units (ints): (r0+r1, c0+c1)
+    points = node.get("boundary") or []
+    if not points:
+        return (0, 0)
+    rows = [p[0] for p in points]; cols = [p[1] for p in points]
+    return (min(rows) + max(rows), min(cols) + max(cols))
+
+def compute_frame_diff(grid_a, grid_b, nodes_a, nodes_b, max_group_match=None):
+    # Object-level diff between two same-shaped grids with precomputed
+    # segmentation nodes. Returns only what changed; unchanged objects are
+    # omitted. When max_group_match is set, identical-hash groups larger than
+    # it skip pairwise matching and report plain appeared/disappeared entries
+    # (guards against O(n^3) assignment on e.g. hundreds of identical pixels).
+    changed = set()
+    for r, (row_a, row_b) in enumerate(zip(grid_a, grid_b)):
+        if row_a == row_b:
+            continue
+        for c, (va, vb) in enumerate(zip(row_a, row_b)):
+            if va != vb:
+                changed.add((r, c))
+    if not changed:
+        return {"changed_cell_count": 0, "moved": [], "rotated": [],
+                "appeared": [], "disappeared": [],
+                "changed_color": [], "resized": []}
+
+    def touched_nodes(nodes):
+        touched = []
+        for node in nodes:
+            points = node.get("boundary") or []
+            if not points:
+                continue
+            r0 = min(p[0] for p in points); r1 = max(p[0] for p in points)
+            c0 = min(p[1] for p in points); c1 = max(p[1] for p in points)
+            # dilate by 1 so objects that shrank/grew still register as
+            # touched by the cells they vacated/claimed
+            if any(r0 - 1 <= r <= r1 + 1 and c0 - 1 <= c <= c1 + 1 for (r, c) in changed):
+                touched.append(node)
+        return touched
+
+    before_nodes = touched_nodes(nodes_a)
+    after_nodes = touched_nodes(nodes_b)
+
+    moved, appeared, disappeared = [], [], []
+    rotated = []
+
+    def _classify_match(node_a, node_b):
+        rot_a = node_a.get("rotation")
+        rot_b = node_b.get("rotation")
+        symmetry = node_a.get("rotational_symmetry") or 1
+        rotated_by = None
+        if rot_a is not None and rot_b is not None:
+            modulus = 180 if symmetry == 2 else 360
+            delta = (rot_b - rot_a) % modulus
+            if delta:
+                rotated_by = delta
+        pa, pb = _node_topleft(node_a), _node_topleft(node_b)
+        base_entry = {"color": node_a.get("color"),
+                      "pixels": node_a.get("pixels"),
+                      "hash": node_a.get("hash")}
+        if rotated_by is not None:
+            ca, cb = _node_center2(node_a), _node_center2(node_b)
+            in_place = (abs(ca[0] - cb[0]) <= 2 and abs(ca[1] - cb[1]) <= 2)
+            base_entry["rotated_by"] = rotated_by
+            if in_place:
+                base_entry["at"] = list(pb)
+                rotated.append(base_entry)
+            else:
+                base_entry["from"] = list(pa); base_entry["to"] = list(pb)
+                moved.append(base_entry)
+        elif pa != pb:
+            base_entry["from"] = list(pa); base_entry["to"] = list(pb)
+            moved.append(base_entry)
+        # same hash, same position, same rotation: unchanged -> omit
+
+    groups_before = {}
+    for node in before_nodes:
+        groups_before.setdefault(node.get("hash"), []).append(node)
+    groups_after = {}
+    for node in after_nodes:
+        groups_after.setdefault(node.get("hash"), []).append(node)
+
+    for key, group_a in groups_before.items():
+        group_b = groups_after.pop(key, [])
+        if (
+            max_group_match is not None
+            and max(len(group_a), len(group_b)) > max_group_match
+        ):
+            # too many identical objects to match affordably
+            for node_a in group_a:
+                disappeared.append({"color": node_a.get("color"),
+                                    "pixels": node_a.get("pixels"),
+                                    "hash": node_a.get("hash"),
+                                    "shape_hash": node_a.get("shape_hash"),
+                                    "at": list(_node_topleft(node_a)),
+                                    "bbox": list(_node_bbox(node_a))})
+            for node_b in group_b:
+                appeared.append({"color": node_b.get("color"),
+                                 "pixels": node_b.get("pixels"),
+                                 "hash": node_b.get("hash"),
+                                 "shape_hash": node_b.get("shape_hash"),
+                                 "at": list(_node_topleft(node_b)),
+                                 "bbox": list(_node_bbox(node_b))})
+            continue
+        if not group_b:
+            for node_a in group_a:
+                disappeared.append({"color": node_a.get("color"),
+                                    "pixels": node_a.get("pixels"),
+                                    "hash": node_a.get("hash"),
+                                    "shape_hash": node_a.get("shape_hash"),
+                                    "at": list(_node_topleft(node_a)),
+                                    "bbox": list(_node_bbox(node_a))})
+            continue
+        # optimal assignment over bbox-center distances; rows = smaller side
+        transposed = len(group_a) > len(group_b)
+        rows, cols = (group_b, group_a) if transposed else (group_a, group_b)
+        cost = []
+        for row_node in rows:
+            cr = _node_center2(row_node)
+            cost.append([
+                abs(cr[0] - _node_center2(col_node)[0])
+                + abs(cr[1] - _node_center2(col_node)[1])
+                for col_node in cols
+            ])
+        assignment = _hungarian(cost)
+        matched_cols = set()
+        for row_index, col_index in enumerate(assignment):
+            if col_index < 0:
+                continue
+            matched_cols.add(col_index)
+            if transposed:
+                _classify_match(cols[col_index], rows[row_index])
+            else:
+                _classify_match(rows[row_index], cols[col_index])
+        if transposed:
+            for col_index, node_a in enumerate(cols):
+                if col_index not in matched_cols:
+                    disappeared.append({"color": node_a.get("color"),
+                                        "pixels": node_a.get("pixels"),
+                                        "hash": node_a.get("hash"),
+                                    "shape_hash": node_a.get("shape_hash"),
+                                        "at": list(_node_topleft(node_a)),
+                                        "bbox": list(_node_bbox(node_a))})
+        else:
+            for col_index, node_b in enumerate(cols):
+                if col_index not in matched_cols:
+                    appeared.append({"color": node_b.get("color"),
+                                     "pixels": node_b.get("pixels"),
+                                     "hash": node_b.get("hash"),
+                             "shape_hash": node_b.get("shape_hash"),
+                                     "at": list(_node_topleft(node_b)),
+                                     "bbox": list(_node_bbox(node_b))})
+            continue
+        # transposed: every after-node (a row) got matched; nothing appeared
+    for group_b in groups_after.values():
+        for node_b in group_b:
+            appeared.append({"color": node_b.get("color"),
+                             "pixels": node_b.get("pixels"),
+                             "hash": node_b.get("hash"),
+                             "shape_hash": node_b.get("shape_hash"),
+                             "at": list(_node_topleft(node_b)),
+                             "bbox": list(_node_bbox(node_b))})
+    # Phase A - changed_color: EXACT pairs only. Same position, byte-
+    # identical shape (color-independent pose signature), different
+    # color. This asserts "these exact cells changed color" and nothing
+    # more; fuzzy transformation claims are deliberately not made.
+    changed_color = []
+    used_gone, used_new = set(), set()
+    for gi, gone in enumerate(disappeared):
+        for ni, new in enumerate(appeared):
+            if ni in used_new:
+                continue
+            if (gone["at"] == new["at"]
+                    and gone.get("shape_hash")
+                    and gone.get("shape_hash") == new.get("shape_hash")
+                    and gone["color"] != new["color"]):
+                changed_color.append({"at": gone["at"],
+                                      "pixels": gone["pixels"],
+                                      "before_color": gone["color"],
+                                      "after_color": new["color"],
+                                      "before_hash": gone.get("hash"),
+                                      "after_hash": new.get("hash")})
+                used_gone.add(gi); used_new.add(ni)
+                break
+    disappeared = [entry for index, entry in enumerate(disappeared) if index not in used_gone]
+    appeared = [entry for index, entry in enumerate(appeared) if index not in used_new]
+
+    # Phase B - resized: same-color pairs whose bboxes substantially
+    # overlap (overlap area / smaller bbox area >= 0.5); greedy
+    # one-to-one by descending overlap then size similarity.
+    resized = []
+
+    def _bbox_area(b):
+        return max(0, b[2] - b[0] + 1) * max(0, b[3] - b[1] + 1)
+
+    def _overlap_ratio(b1, b2):
+        inter_r = min(b1[2], b2[2]) - max(b1[0], b2[0]) + 1
+        inter_c = min(b1[3], b2[3]) - max(b1[1], b2[1]) + 1
+        if inter_r <= 0 or inter_c <= 0:
+            return 0.0
+        smaller = min(_bbox_area(b1), _bbox_area(b2))
+        return (inter_r * inter_c) / smaller if smaller else 0.0
+
+    candidates = []
+    for gi, gone in enumerate(disappeared):
+        for ni, new in enumerate(appeared):
+            if gone["color"] != new["color"]:
+                continue
+            bg = gone.get("bbox") or (0, 0, -1, -1)
+            bn = new.get("bbox") or (0, 0, -1, -1)
+            ratio = _overlap_ratio(bg, bn)
+            if ratio >= 0.5:
+                area_g, area_n = _bbox_area(bg), _bbox_area(bn)
+                similarity = (min(area_g, area_n) / max(area_g, area_n)) if max(area_g, area_n) else 0.0
+                candidates.append((ratio, similarity, gi, ni))
+    candidates.sort(key=lambda item: (-item[0], -item[1]))
+    used_gone, used_new = set(), set()
+    for ratio, similarity, gi, ni in candidates:
+        if gi in used_gone or ni in used_new:
+            continue
+        used_gone.add(gi); used_new.add(ni)
+        gone, new = disappeared[gi], appeared[ni]
+        resized.append({"at_before": gone["at"], "at_after": new["at"],
+                        "color": gone["color"],
+                        "before_pixels": gone["pixels"],
+                        "after_pixels": new["pixels"],
+                        "before_hash": gone.get("hash"),
+                        "after_hash": new.get("hash")})
+    disappeared = [entry for index, entry in enumerate(disappeared) if index not in used_gone]
+    appeared = [entry for index, entry in enumerate(appeared) if index not in used_new]
+    return {"changed_cell_count": len(changed), "moved": moved,
+            "rotated": rotated,
+            "appeared": appeared, "disappeared": disappeared,
+            "changed_color": changed_color, "resized": resized}
diff --git a/ARC3-Inference/inference/utils/openai_compat.py b/ARC3-Inference/inference/utils/openai_compat.py
index 157e113..2fadb51 100644
--- a/ARC3-Inference/inference/utils/openai_compat.py
+++ b/ARC3-Inference/inference/utils/openai_compat.py
@@ -1,6 +1,8 @@
 """Helpers for provider-specific OpenAI-compatible requests."""
 from __future__ import annotations
 
+import json
+
 from typing import Any
 
 
@@ -46,22 +48,46 @@ def build_chat_payload(
     tools: list[dict[str, Any]] | None = None,
     tool_choice: str | None = None,
     seed: int | None = None,
+    priority: int | None = None,
+    stream: bool = False,
 ) -> dict[str, Any]:
     payload: dict[str, Any] = {
         "model": model,
         "messages": messages,
-        "stream": False,
+        "stream": bool(stream),
         "temperature": temperature,
         "top_p": top_p,
     }
     if max_tokens is not None:
         payload["max_tokens"] = max_tokens
+    if priority is not None:
+        # scheduling hint, ignored by servers that do not implement it
+        payload["priority"] = int(priority)
     if tools:
         payload["tools"] = tools
         if tool_choice:
             payload["tool_choice"] = tool_choice
 
     normalized = normalize_provider(provider)
+    if stream:
+        payload["stream_options"] = {"include_usage": True}
+    if normalized == "openrouter":
+        import os
+        if not payload["stream"] and os.environ.get(
+            "OPENROUTER_STREAM", ""
+        ).strip().lower() not in ("0", "false", "no", "off"):
+            payload["stream"] = True
+        if payload["stream"]:
+            payload["stream_options"] = {"include_usage": True}
+        payload["reasoning"] = {"enabled": bool(thinking)}
+        _order = os.environ.get("OPENROUTER_PROVIDER_ORDER", "").strip()
+        if _order:
+            payload["provider"] = {
+                "order": [s.strip() for s in _order.split(",") if s.strip()],
+                "allow_fallbacks": os.environ.get(
+                    "OPENROUTER_ALLOW_FALLBACKS", ""
+                ).strip().lower() in ("1", "true", "yes"),
+            }
     if normalized == "vllm":
         if top_k > 0:
             payload["top_k"] = top_k
@@ -70,3 +96,101 @@ def build_chat_payload(
             payload["seed"] = seed
 
     return payload
+
+
+def assemble_streamed_chat_response(lines) -> dict[str, Any]:
+    """Consume an SSE chat-completion stream and rebuild the non-streaming
+    response shape: {"choices": [{"message", "finish_reason"}], "usage"}.
+
+    Tolerates keep-alive comments (": ..."), event: lines, usage-only chunks,
+    and mid-stream error events. If an error arrives and no assistant content
+    was produced, returns {"error": ..., "choices": []} so callers surface it.
+    """
+    content_parts: list[str] = []
+    reasoning_parts: list[str] = []
+    tool_calls_by_index: dict[int, dict[str, Any]] = {}
+    finish_reason = ""
+    usage: dict[str, Any] | None = None
+    error: Any = None
+    provider_name = [""]
+
+    for raw_line in lines:
+        if raw_line is None:
+            continue
+        line = raw_line.decode("utf-8", "replace") if isinstance(raw_line, bytes) else str(raw_line)
+        line = line.strip()
+        if not line or line.startswith(":") or line.startswith("event:"):
+            continue
+        if line.startswith("data:"):
+            line = line[5:].strip()
+        if line == "[DONE]":
+            break
+        try:
+            chunk = json.loads(line)
+        except json.JSONDecodeError:
+            continue
+        if not isinstance(chunk, dict):
+            continue
+        if chunk.get("error"):
+            error = chunk["error"]
+            continue
+        if isinstance(chunk.get("usage"), dict):
+            usage = chunk["usage"]
+        if chunk.get("provider") and not provider_name[0]:
+            provider_name[0] = str(chunk["provider"])
+        for choice in chunk.get("choices") or []:
+            if not isinstance(choice, dict):
+                continue
+            if choice.get("finish_reason"):
+                finish_reason = str(choice["finish_reason"])
+            delta = choice.get("delta") or {}
+            if not isinstance(delta, dict):
+                continue
+            if isinstance(delta.get("content"), str):
+                content_parts.append(delta["content"])
+            if isinstance(delta.get("reasoning"), str):
+                reasoning_parts.append(delta["reasoning"])
+            elif isinstance(delta.get("reasoning_content"), str):
+                reasoning_parts.append(delta["reasoning_content"])
+            elif isinstance(delta.get("reasoning_details"), list):
+                for detail in delta["reasoning_details"]:
+                    if isinstance(detail, dict) and isinstance(detail.get("text"), str):
+                        reasoning_parts.append(detail["text"])
+            for fragment in delta.get("tool_calls") or []:
+                if not isinstance(fragment, dict):
+                    continue
+                index = int(fragment.get("index", 0) or 0)
+                entry = tool_calls_by_index.setdefault(
+                    index,
+                    {"id": "", "type": "function", "function": {"name": "", "arguments": ""}},
+                )
+                if fragment.get("id"):
+                    entry["id"] = str(fragment["id"])
+                if fragment.get("type"):
+                    entry["type"] = str(fragment["type"])
+                function = fragment.get("function") or {}
+                if isinstance(function, dict):
+                    if function.get("name"):
+                        entry["function"]["name"] = str(function["name"])
+                    if isinstance(function.get("arguments"), str):
+                        entry["function"]["arguments"] += function["arguments"]
+
+    produced = bool(content_parts or reasoning_parts or tool_calls_by_index)
+    if error is not None and not produced:
+        return {"error": error, "choices": []}
+
+    message: dict[str, Any] = {"role": "assistant", "content": "".join(content_parts)}
+    if reasoning_parts:
+        message["reasoning"] = "".join(reasoning_parts)
+    if tool_calls_by_index:
+        message["tool_calls"] = [tool_calls_by_index[i] for i in sorted(tool_calls_by_index)]
+    result: dict[str, Any] = {
+        "choices": [{"message": message, "finish_reason": finish_reason}],
+    }
+    if usage is not None:
+        result["usage"] = usage
+    if error is not None:
+        result["error"] = error
+    if provider_name[0]:
+        result["provider"] = provider_name[0]
+    return result
diff --git a/ARC3-Inference/inference/utils/retained_functions.py b/ARC3-Inference/inference/utils/retained_functions.py
new file mode 100644
index 0000000..66aa5a2
--- /dev/null
+++ b/ARC3-Inference/inference/utils/retained_functions.py
@@ -0,0 +1,301 @@
+"""Conservative source-only function retention; no snippet state is serialized.
+
+Also embedded into the isolated Python runner. Keep this module stdlib-only.
+"""
+import ast
+import dis
+import sys
+import types
+
+
+def _import_statements(nodes):
+    """One explicit absolute import per binding; never replay wildcard imports."""
+    statements = {}
+    for node in nodes:
+        if isinstance(node, ast.Import):
+            for alias in node.names:
+                binding = alias.asname or alias.name.split('.')[0]
+                statements[binding] = ast.unparse(ast.Import(names=[alias]))
+        elif isinstance(node, ast.ImportFrom) and not node.level and node.module:
+            for alias in node.names:
+                if alias.name == '*':
+                    raise ValueError('wildcard imports are not retained; use explicit imports inside the function')
+                statements[alias.asname or alias.name] = ast.unparse(
+                    ast.ImportFrom(module=node.module, names=[alias], level=0))
+        else:
+            raise ValueError('only explicit absolute imports can accompany retained functions')
+    return statements
+
+
+def _source_parts(source, retain_imports=True):
+    tree = ast.parse(source, '<retained>', 'exec')
+    if (not tree.body or not isinstance(tree.body[-1], ast.FunctionDef)
+            or (not retain_imports and len(tree.body) != 1)):
+        raise ValueError('only top-level function definitions can be retained')
+    imports = _import_statements(tree.body[:-1])
+    return tree.body[-1], imports
+
+
+def function_signature(source):
+    node, _ = _source_parts(source)
+    return node.name + '(' + ast.unparse(node.args) + ')'
+
+
+def _function_record(source, retain_imports=False):
+    node, imports = _source_parts(source, retain_imports)
+    if getattr(node, "type_params", []):
+        raise ValueError("generic type parameters are not retained")
+    if node.decorator_list:
+        raise ValueError('decorated functions are not retained')
+    args = [*node.args.posonlyargs, *node.args.args, *node.args.kwonlyargs]
+    args += [a for a in (node.args.vararg, node.args.kwarg) if a is not None]
+    if node.returns is not None or any(a.annotation is not None for a in args):
+        raise ValueError('use an unannotated definition for retention')
+    # Restoration must never evaluate model calls or snapshot a frame via defaults.
+    for default in [*node.args.defaults, *node.args.kw_defaults]:
+        if default is not None:
+            try:
+                ast.literal_eval(default)
+            except Exception:
+                raise ValueError('defaults must be literals; pass changing data as arguments') from None
+    tree = ast.Module(body=[node], type_ignores=[])
+    code = next(c for c in compile(tree, '<retained>', 'exec').co_consts
+                if isinstance(c, types.CodeType))
+    reads = set()
+    pending = [code]
+    while pending:
+        current = pending.pop()
+        for instruction in dis.get_instructions(current):
+            if instruction.opname in ('LOAD_GLOBAL', 'LOAD_NAME'):
+                reads.add(instruction.argval)
+            elif instruction.opname in ('STORE_GLOBAL', 'DELETE_GLOBAL'):
+                raise ValueError('functions that assign global variables are not retained')
+        pending.extend(c for c in current.co_consts if isinstance(c, types.CodeType))
+    return node.name, source, reads - imports.keys()
+
+
+def _rejection(name, reason, hint=None, **metadata):
+    result = {'name': name, 'reason': reason, **metadata}
+    if hint:
+        result['hint'] = hint
+    return result
+
+
+def _validation_hint(reason):
+    if 'decorat' in reason:
+        return 'Use a plain top-level def without decorators if you want to reuse it.'
+    if 'annotat' in reason:
+        return 'Remove parameter and return annotations if you want to retain this function.'
+    if 'default' in reason:
+        return 'Use literal defaults and pass changing values explicitly as arguments.'
+    if 'global variables' in reason:
+        return 'Return updated values instead of assigning global variables.'
+    if 'import' in reason:
+        return 'Put explicit, sandbox-supported imports inside the function.'
+    return 'Use a plain top-level def if you want to retain this function.'
+
+
+def prepare_functions(sources, available, *, retain_imports=False):
+    """Validate sources without executing them; drop invalid dependency chains."""
+    records, rejected = {}, []
+    for source in sources:
+        name = '<unknown>'
+        try:
+            parsed = ast.parse(source)
+            if (parsed.body and isinstance(parsed.body[-1], ast.FunctionDef)
+                    and (retain_imports or len(parsed.body) == 1)):
+                name = parsed.body[-1].name
+                records.pop(name, None)
+            name, source, reads = _function_record(source, retain_imports)
+            _, imports = _source_parts(source)
+            if imports.keys() & set(available):
+                raise ValueError('import alias conflicts with a provided global or builtin')
+            records[name] = (source, reads)
+        except Exception as exc:
+            reason = str(exc)[:180]
+            rejected.append(_rejection(name, reason, _validation_hint(reason)))
+    # Import bundles must not overwrite another retained function at restore.
+    function_names = set(records)
+    for name, (source, _) in list(records.items()):
+        _, imports = _source_parts(source)
+        if imports.keys() & function_names:
+            del records[name]
+            rejected.append(_rejection(name, 'import alias conflicts with a retained function',
+                                       'Choose a different import alias or function name.'))
+    # A wrapper must not survive its unavailable dependency. Re-run to a fixed point.
+    while True:
+        dropped = []
+        for name, (source, reads) in records.items():
+            missing = reads - set(available) - records.keys()
+            if missing:
+                dropped.append(name)
+                names = ', '.join(sorted(missing))
+                rejected.append(_rejection(
+                    name, 'unavailable dependencies: ' + names,
+                    'Pass ' + names + ' as arguments, or put their imports inside the function.',
+                    missing_dependencies=sorted(missing)))
+        if not dropped:
+            break
+        for name in dropped:
+            del records[name]
+    return {name: value[0] for name, value in records.items()}, rejected
+
+
+def restore_functions(sources, namespace, available, *, retain_imports=False,
+                      repair_hints=False):
+    """Restore definitions and explicit imports through the sandbox importer.
+
+    Never replay assignments, expressions, decorators, or previous actions.
+    """
+    kept, rejected = prepare_functions(sources, available, retain_imports=retain_imports)
+    failed = set()
+    for name, source in kept.items():
+        before = dict(namespace)
+        try:
+            if name in available:
+                raise ValueError('name conflicts with a provided global or builtin')
+            exec(compile(source, '<retained>', 'exec'), namespace, namespace)
+        except Exception:
+            # A multi-import bundle can fail partway through. Restore bindings
+            # so one failed helper does not replace another helper's globals.
+            namespace.clear()
+            namespace.update(before)
+            failed.add(name)
+            reason = ('could not restore this definition or its imports' if repair_hints else
+                      'could not restore this definition; redefine it if needed')
+            rejected.append(_rejection(name, reason,
+                                       'Check its imports are supported by the sandbox, then redefine it if needed.'))
+    survivors, dependencies = prepare_functions(
+        [source for name, source in kept.items() if name not in failed], available,
+        retain_imports=retain_imports)
+    rejected.extend(dependencies)
+    for name in kept.keys() - survivors.keys():
+        if name not in available:
+            namespace.pop(name, None)
+    return survivors, rejected
+
+
+def _matching_imports(nodes, namespace):
+    """Recognize live import bindings without importing or evaluating anything.
+
+    Only module bindings and callable/immutable exports are eligible. Ordinary
+    snippet assignments and mutable imported data are never serialized. Looking
+    in module dictionaries avoids invoking __getattr__ during bookkeeping.
+    """
+    matches, declared = {}, {}
+    for node in nodes:
+        if not isinstance(node, (ast.Import, ast.ImportFrom)):
+            continue
+        try:
+            statements = _import_statements([node])
+        except ValueError:
+            continue
+        declared.update(statements)
+        for binding, statement in statements.items():
+            single = ast.parse(statement).body[0]
+            alias = single.names[0]
+            if isinstance(single, ast.Import):
+                # Even unaliased dotted imports must already have executed.
+                if alias.name not in sys.modules:
+                    continue
+                module_name = alias.name if alias.asname else alias.name.split('.')[0]
+                expected = sys.modules.get(module_name)
+            else:
+                module = sys.modules.get(single.module)
+                expected = vars(module).get(alias.name) if type(module) is types.ModuleType else None
+            eligible = type(expected) in (
+                types.ModuleType, types.FunctionType, types.BuiltinFunctionType, types.MethodType,
+                type, str, bytes, int, float, complex, bool,
+            )
+            if eligible and binding in namespace and namespace[binding] is expected:
+                matches[binding] = statement
+    return matches, declared
+
+
+def collect_functions(code_text, previous, namespace, available, *, retain_imports=False,
+                      repair_hints=False):
+    """Collect from the actual namespace, including after partial execution.
+
+    A later def may never have run. Match the live function's source location
+    rather than choosing the last definition in the submitted snippet.
+    """
+    candidates = {}
+    rejected = []
+    try:
+        tree = ast.parse(code_text, '<python_tool>')
+    except SyntaxError:
+        # Parsing failed before execution; restored functions are still valid.
+        tree = ast.Module(body=[], type_ignores=[])
+    fresh = {}
+    for node in tree.body:
+        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
+            fresh.setdefault(node.name, []).append(node)
+    import_nodes = []
+    for source in previous.values():
+        # Old source-only records remain valid; no migration is necessary.
+        import_nodes.extend(ast.parse(source).body[:-1])
+    import_nodes.extend(tree.body)
+    imports, declared_imports = (_matching_imports(import_nodes, namespace)
+                                if retain_imports or repair_hints else ({}, {}))
+    for name in dict.fromkeys([*previous, *fresh]):
+        value = namespace.get(name)
+        if not repair_hints and (name in available or type(value) is not types.FunctionType
+                                 or value.__name__ != name):
+            rejected.append(_rejection(name, 'definition did not execute, was removed/replaced, or conflicts with a provided name'))
+            continue
+        if name in available:
+            rejected.append(_rejection(name, 'name conflicts with a provided global or builtin',
+                                       'Choose a different function name.'))
+            continue
+        if name not in namespace:
+            reason = 'function was removed' if name in previous else 'definition was not reached or was removed'
+            rejected.append(_rejection(name, reason, 'Define it before calling it if it is still needed.'))
+            continue
+        if type(value) is not types.FunctionType or value.__name__ != name:
+            rejected.append(_rejection(name, 'function name was replaced by another value',
+                                       'Use a separate variable name; redefine the function only if needed.'))
+            continue
+        if name in previous and value.__code__.co_filename == '<retained>':
+            # Preserve the restored definition when execution stopped before a
+            # replacement (or while evaluating that replacement's defaults).
+            candidates[name] = previous[name]
+            continue
+        node = next((node for node in fresh.get(name, [])
+                     if isinstance(node, ast.FunctionDef) and not node.decorator_list
+                     and value.__code__.co_filename == '<python_tool>'
+                     and value.__code__.co_firstlineno == node.lineno), None)
+        if node is None:
+            rejected.append(_rejection(name, 'definition was decorated, replaced outside a top-level def, or is unsupported',
+                                       'Use a plain top-level def without decorators if you want to reuse it.'))
+            continue
+        candidates[name] = ast.get_source_segment(code_text, node)
+    # Rebuild every bundle from the LIVE namespace. An old helper must not
+    # silently restore an obsolete import when that alias was rebound/deleted.
+    for name, source in (list(candidates.items()) if retain_imports else []):
+        try:
+            node, _ = _source_parts(source)
+            definition = ast.get_source_segment(source, node)
+            _, _, reads = _function_record(definition)
+            statements = [imports[n] for n in sorted(reads & imports.keys()) if n not in available]
+            candidates[name] = '\n'.join([*statements, definition])
+        except Exception:
+            # Validation below produces the specific rejection and prunes callers.
+            pass
+    kept, invalid = prepare_functions(candidates.values(), available, retain_imports=retain_imports)
+    for item in invalid:
+        missing = item.get('missing_dependencies', [])
+        if not missing:
+            continue
+        hints = []
+        missing_imports = [n for n in missing if n in declared_imports]
+        missing_helpers = [n for n in missing if n in candidates and n not in declared_imports]
+        missing_data = sorted(set(missing) - set(missing_imports) - set(missing_helpers))
+        if missing_imports:
+            statements = '; '.join(declared_imports[n] for n in missing_imports)
+            hints.append('Put `' + statements + '` inside this function if you want to reuse it.')
+        if missing_helpers:
+            hints.append('Make the helper dependencies retainable too: ' + ', '.join(missing_helpers) + '.')
+        if missing_data:
+            hints.append('Pass ' + ', '.join(missing_data) + ' as arguments; snippet variables are not retained.')
+        item['hint'] = ' '.join(hints)
+    return kept, rejected + invalid
diff --git a/ARC3-Inference/inference/utils/segmentation.py b/ARC3-Inference/inference/utils/segmentation.py
index 20f10ed..33f8848 100644
--- a/ARC3-Inference/inference/utils/segmentation.py
+++ b/ARC3-Inference/inference/utils/segmentation.py
@@ -62,17 +62,61 @@ def _corner_points(contour):
     return corners
 
 
-def _object_hash(cells, color):
-    """Translation-invariant signature of an object: its color plus its cell shape,
-    normalized so the top-left of its bounding box is the origin. Same shape + color
-    => same hash regardless of position, so objects can be matched across frames."""
+def _normalize_cells(cells):
     min_r = min(r for r, _ in cells)
     min_c = min(c for _, c in cells)
-    norm = sorted((r - min_r, c - min_c) for r, c in cells)
-    payload = repr((color, norm)).encode()
+    return sorted((r - min_r, c - min_c) for r, c in cells)
+
+
+def _rotate_cells_90(norm_cells):
+    """Rotate a normalized cell set 90 degrees clockwise; returns normalized."""
+    max_r = max(r for r, _ in norm_cells)
+    return sorted((c, max_r - r) for r, c in norm_cells)
+
+
+def _cells_hash(norm_cells, color):
+    payload = repr((color, norm_cells)).encode()
     return hashlib.sha1(payload).hexdigest()[:16]
 
 
+def _object_signature(cells, color):
+    """Rotation-canonical identity of an object.
+
+    Hashes the shape in all four 90-degree rotations (extend the transform list
+    with reflections for full D4 canonicalization later) and takes the lowest
+    hash as the canonical id. Returns (canonical_hash, rotation, symmetry,
+    pose_hash) where:
+      - canonical_hash: identical for all rotated copies of the same shape+color.
+      - rotation: degrees CLOCKWISE to rotate the canonical form to obtain the
+        observed object. None for 4-fold symmetric shapes (all rotations equal);
+        reduced mod 180 (i.e. 0 or 90) for 2-fold symmetric shapes.
+      - symmetry: rotational symmetry order (1, 2, or 4).
+      - pose_hash: the classic rotation-SENSITIVE hash of the observed pose.
+    """
+    norm = _normalize_cells(cells)
+    variants = [norm]
+    for _ in range(3):
+        variants.append(_rotate_cells_90(variants[-1]))
+    hashes = [_cells_hash(v, color) for v in variants]
+    # The reference orientation is picked from the COLOUR-FREE hash. Choosing it
+    # from the colour-bearing one made the argmin depend on the colour, so the
+    # same shape in the same orientation reported a different rotation for a red
+    # object than for a blue one - and a model comparing rotations across
+    # objects was reading noise.
+    shape_hashes = [_cells_hash(v, "__shape__") for v in variants]
+    k = shape_hashes.index(min(shape_hashes))
+    canonical = hashes[k]
+    distinct = len(set(shape_hashes))
+    symmetry = {1: 4, 2: 2, 4: 1}.get(distinct, 1)
+    rotation = ((4 - k) % 4) * 90  # so observed = canonical rotated this much CW
+    if symmetry == 4:
+        rotation = None
+    elif symmetry == 2:
+        rotation = rotation % 180
+    shape_hash = _cells_hash(norm, "__shape__")
+    return canonical, rotation, symmetry, hashes[0], shape_hash
+
+
 def segment_layer(layer, color_chars):
     """Segment one frame layer into connected-component nodes.
 
@@ -189,11 +233,16 @@ def segment_layer(layer, color_chars):
         comp = components[cid]
         color = color_chars[max(0, min(15, comp["value"]))]
         boundary = _corner_points(_trace_outer_contour(comp["cells"], comp["start"]))
+        signature = _object_signature(comp["cells"], color)
         nodes.append(
             {
                 "id": cid,
                 "color": color,
-                "hash": _object_hash(comp["cells"], color),
+                "hash": signature[0],
+                "rotation": signature[1],
+                "rotational_symmetry": signature[2],
+                "pose_hash": signature[3],
+                "shape_hash": signature[4],
                 "pixels": len(comp["cells"]),
                 "boundary": [[r, c] for r, c in boundary],
                 "children": children[cid],

## 1. Environment and submission mode

Detect whether this is a real competition rerun (which minimises diagnostics), set the
framework's environment flags, and put the CUDA libraries on the linker path.

In [ ]:
import json
import os
import pickle
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

# Kaggle might cancel the run in no action is done in some timeframe.
# To prevent this, harness will start up even if backend is not yet ready,
# and the first started game will perform a single reset action.
NOTEBOOK_START_TIME = time.time()
SERVER_STARTUP_TIMEOUT = 12 * 60
os.environ['ARC3_WARMUP_ACTION_GAMES'] = '1'
os.environ['ARC3_HTTP_RETRY_INITIAL_SECONDS'] = '2400'  # v32: 40 min (was 15) for a slow model-server start

# model and dataset paths
WHEELHOUSE_DIR    = '/kaggle/input/datasets/dfranzen/pennyroyal-v253'
MODEL_DIR         = '/kaggle/input/models/dfranzen/intel-qwen3.8-flash-next-w4a16-autoround/transformers/default/1'
DRAFT_MODEL_DIR   = '/kaggle/input/models/dfranzen/albucino-qwen3-8-flash-next-drafter/transformers/default/1'
ORIG_BUNDLE_DIR   = '/kaggle/input/datasets/dfranzen/taaf-kaggle-source-bundle-copy'
SERVED_MODEL_NAME = 'flashnext'
# ---------------- v35: model server ----------------
SERVER_ENGINE                = 'vllm'   # v35: 'vllm' = lordhansolo's vLLM runtime + NVFP4/FP8 weights, 14 games at once | 'sglang' = v34 exactly
VLLM_FALLBACK_TO_SGLANG      = True     # v35: if vLLM cannot start (or its watchdog gives up), start v34's SGLang server instead
VLLM_RUNTIME_DIR             = '/kaggle/input/datasets/lordhansolo/vllm-main-e975732-arc3'
VLLM_MODEL_DIR               = '/kaggle/input/models/lordhansolo/qwen3-8-flash-next-mixed-nvfp4-fp8/pytorch/hf-mixed-mtp-nvfp4/1'
VLLM_MAX_NUM_SEQS            = 18       # v47: games generating at once on vLLM: 18 running generate ~11% more tokens than 16 (v40's fit; flat above 18) | 16 = v42-v46 (v40/v41: 20 x 68K; v38/v39: 16; v35-v37: 14)
VLLM_REAP_KEEP               = 448      # v42: REAP-at-load - the target model keeps 448 of 512 routed experts per MoE layer (+~7.9 GiB KV) | 384 (stronger, coding-calibrated list) | None = full model
VLLM_HISTORY_BUDGET_K_BY_REAP = {None: 88, 448: 104, 384: 104}   # v47: history trim point (K tokens) for the KV pool each model leaves: 18 x 104K -> est. peak ~84% (v46 measured 83% at 16 x 116K); full model: v39's 88K at 16 games | v42-v46: 448: 116
VLLM_HISTORY_BUDGET_K        = VLLM_HISTORY_BUDGET_K_BY_REAP[VLLM_REAP_KEEP]   # v42: 116 = v36's history (v39: 88; v40/v41: 68)
VLLM_FALLBACK_SEQS_AFTER_RELEASE = 14  # v42: if the pruned server dies after the benchmark release, the full model runs 14 games (v36's shape for 116K)
VLLM_FALLBACK_SEQS_BEFORE_RELEASE = 16  # v47: if the pruned model fails before the release, the full model runs v39's tested shape: 16 games x 88K (seats and harness both)
VLLM_SPEC_RELAX_T            = None     # v43: exact acceptance (v42's t = 0.6 made the model loop: 34 repeating thinking blocks in 3 games)
VLLM_PREFIX_RETENTION_INTERVAL = 0      # v37: 0 = v35 = his production setting (v36's -1 reused no reply tokens; see the intro)
assert SERVER_ENGINE in ('vllm', 'sglang'), SERVER_ENGINE

# ---------------- merged add-on switches (see the intro; every one can be turned off) ----------------
ENABLE_SOLVED_LEVEL_MEMORY   = True   # Duck v18 M85 (adapted): pin 'Rule for level N' + winning tail per solved level
ENABLE_COMPACT_RUNTIME_STATE = True   # Duck v18 M86 idea: same state JSON, written without indent (~7x smaller)
ENABLE_SERVER_WATCHDOG       = True   # relaunch SGLang if it exits mid-run (max 2 restarts)
ENABLE_DEADLINE_GUARD        = True   # submission: end play before the Kaggle runtime limit
KAGGLE_RUNTIME_LIMIT_S       = 9 * 3600   # competition notebook limit the deadline guard protects
DEADLINE_SAFETY_MARGIN_S     = 8 * 60     # left for cancel-drain + result writing after play stops
# ---------------- Save & Run game set, at COMPETITION-MATCHED compute per game (N games -> 532*N/110 minutes) ----------------
LOCAL_GAME_SET               = 'all25'     # v38: all 25 public games (fills the 16 vLLM seats, 121 min) | 'mixed14' (v35-v37, 68 min) | 'mixed10' | 'demo10'
_ALL25_GAMES = ['ar25', 'bp35', 'cd82', 'cn04', 'dc22', 'ft09', 'g50t', 'ka59', 'lf52', 'lp85', 'ls20', 'm0r0', 'r11l',
                're86', 's5i5', 'sb26', 'sc25', 'sk48', 'sp80', 'su15', 'tn36', 'tr87', 'tu93', 'vc33', 'wa30']
_LOCAL_SETS = {
    'all25': _ALL25_GAMES,
    'demo10': ['ar25', 'ft09', 'lp85', 'r11l', 're86', 'sb26', 'sc25', 'tr87', 'tu93', 'vc33'],
    # stratified by da-fr's 4-pass competition-compute means (87, 82, 80, 63, 54, 46, 28, 24, 19, 9)
    'mixed10': ['ft09', 'r11l', 'vc33', 're86', 'tn36', 'm0r0', 'su15', 's5i5', 'ka59', 'g50t'],
    'mixed14': ['ft09', 'r11l', 'vc33', 're86', 'tn36', 'm0r0', 'su15', 's5i5', 'ka59', 'g50t', 'lp85', 'ls20', 'ar25', 'sb26'],
}
if LOCAL_GAME_SET not in _LOCAL_SETS:
    raise ValueError(f'LOCAL_GAME_SET must be one of {sorted(_LOCAL_SETS)}, got {LOCAL_GAME_SET!r}')
LOCAL_EVAL_ALL_25_GAMES      = LOCAL_GAME_SET == 'all25'
LOCAL_PLAY_MINUTES           = round(532 * len(_LOCAL_SETS[LOCAL_GAME_SET]) / 110)   # same GPU time per game as the competition
ENABLE_DPRIME_SCHEDULER      = True   # v5: affectify's D' slot priority (False = da-fr's priority)
DPRIME_OVERRIDES             = {'level_pow': 0.75, 'token_scale': 700_000.0}   # v36: token_scale 700K for 14 slots (v34/v35: 400K; {} = v33)
ENABLE_ACTION_HINTS          = True   # v6: harness-computed per-level summary of what each action did
ENABLE_TOOL_NUDGE            = True   # v6: remind about check_rules/plan_route at 8 and 25 moves per level
ENABLE_BUDGET_METER          = True   # v7: measured HUD move-budget bar (cells left, rate per move, moves left)
ENABLE_STUCK_FACTS           = True   # v23: on levels past 40K tokens, facts on what was tried / never tried (no ask)
ENABLE_KEEP_FUNCTIONS_ON_TIMEOUT = True  # v23: a timed-out python call no longer wipes the retained functions
ENABLE_GAME_CATALOGUE        = False  # v22's design-pattern knowledge (off in v23)
ENABLE_SECOND_OPINION        = True   # v24: on a stalled level, a fresh model instance proposes hypotheses (needs ENABLE_STUCK_FACTS)
ENABLE_SO_SHARED_RULES       = True   # v25: the second opinion sees the rules OTHER games of this run discovered
ENABLE_SO_CATALOGUE          = True   # v25: the second opinion's system prompt carries the design catalogue
ENABLE_FIRST_IMPRESSIONS     = False  # v60: OFF - v59c's clean full test (advisers could not see the played game) cleared 10-17% fewer levels per token at 40-80K tokens than v45-v47 and only caught up by 120-200K: no gain on unseen games, ~4.6% of tokens spent | True = v55-v59 behaviour
V60_EFFORT_LADDER            = True   # v60: after a reply cut off at the 12,288-token output cap, the next requests think at 'medium' until an action changes the board (Franzen's built-in ARC3_REASONING_EFFORT_LADDER, off in his public notebook; 33-41 cut-off replies per 2-hour run = ~7% of tokens)
V60_LIGHT_TRIMS              = True   # v60: a history trim keeps ~V60_TRIM_KEEP_K tokens instead of half the budget (~52K): re-reading after trims about halves (measured: trims cost ~6.5% of generation); average history depth ~-12%
V60_TRIM_KEEP_K              = 35     # v60: K tokens of history kept by a trim when V60_LIGHT_TRIMS
V60_NO_DIFF_IMAGE            = False  # v60 submission: KEEP the per-turn difference image - it is the player's only automatic view of what changed, and changes to what the player sees cost points on the hidden games (Goal Lens v49; first impressions v55 24.0 / v59b 24.21) | True = drop it (~400 tokens a turn)
ENABLE_CROSS_GAME_RULES      = True   # v44: on a stalled level (when the level-progress facts fire) the player sees how other games of this run worked - level 1: their level-1 rules; later levels: what their later levels added (needs ENABLE_STUCK_FACTS) | False = v43
ENABLE_XG_LIBRARY            = True   # v47: the cross-game pool starts with 70 rules our players stated on the 25 public games (never a game's own; Save & Run: matched by game id) | False = v46
ENABLE_HISTORY_DEDUP         = True   # v31: on together with exact replay, as in v20 (30.37 on the leaderboard)
ENABLE_CACHE_DIAG            = True   # v19: Save & Run only - log why the server re-reads previous replies
ENABLE_EXACT_REPLAY          = True   # v31: on, with de-dup (v21/v28/v29 had replay WITHOUT de-dup: 26.79/26.01/26.87)
ENABLE_TRIM_MEMORY           = False  # v42: off (v36's play); v41: at a history trim, keep verbatim excerpts of the dropped turns' reasoning at the head of the history
ENABLE_GATEWAY_PATIENCE      = True   # v32: game-server calls get GATEWAY_TIMEOUT_S instead of arc_agi's 10 s
GATEWAY_TIMEOUT_S            = 60
ENABLE_SCORECARD_KEEPALIVE   = True   # v32: submission only - after KEEPALIVE_IDLE_S without a game command, RESET one game at the start of level 1
KEEPALIVE_IDLE_S             = 240    # the game server closes a scorecard that has been idle for 15 min
KEEPALIVE_LAST_RESORT_IDLE_S = 480    # v45: after 8 min without any game command and no game at a level start, restart the current level of one game (level 1 with no points first)
ENABLE_FIRST_LIGHT           = True   # v46: a game still on level 1 after FL_MIN_LEVEL1_TOKENS gets a CPU search of level 1 while it waits for its GPU seat | False = v45
FL_MIN_LEVEL1_TOKENS         = 180000 # v48: generated tokens on level 1 before a wait is used - over 236 game-runs the model cleared every level 1 it had time for, 10% after 60K (v46/v47's trigger; v47 lost m0r0 that way), none after 180K
FL_MAX_MOVES_PER_GAME        = 12000  # search moves per game at most (level 1 of m0r0 took 11,741 in tests)
FL_MAX_MOVES_TOTAL           = 60000  # search moves per run at most (about twice what a contest run plays itself)
FL_MAX_CONCURRENT            = 2      # games searching at the same time
ENABLE_SURVIVAL_SMOKE        = True   # v32: Save & Run only - test both on the real arc_agi server code after the run
ENABLE_GAME_COUNT_GUARD      = True   # v33: contest run - concurrency >= number of games (every game starts at once)
COMMIT_PLAY_MINUTES          = 30     # v60: 30-min commit test for the submission notebook (v60-test: None = the full 121-min test) | never affects the contest
if COMMIT_PLAY_MINUTES:
    LOCAL_PLAY_MINUTES = int(COMMIT_PLAY_MINUTES)

# ---------------- v3 strategy profiles: built-in Duck-harness strategies da-fr's config leaves off ----------------
PROFILE = 'v2'   # one of PROFILES below; applies to Save & Run AND to a real submission
_ACT = {'ARC3_OPENER_COMMIT_HINT': '1', 'ARC3_PREFER_TOOL_CALLS': '1', 'ARC3_STUB_DEAD_REASONING': '1'}
_DEATH = {'ARC3_DEATH_LEDGER': '1', 'ARC3_DEATH_REPEAT_GUARD': '1', 'ARC3_REPEAT_STATE_GUARD': '1'}
_INVENTORY = {'ARC3_LEVEL_INVENTORY': '1'}
PROFILES = {
    'v2': {},
    'act': _ACT,
    'death': _DEATH,
    'inventory': _INVENTORY,
    'act_death': {**_ACT, **_DEATH},
    'act_death_inventory': {**_ACT, **_DEATH, **_INVENTORY},
}
EXTRA_ENV = {}          # optional one-off harness env overrides on top of the profile, e.g. {'ARC3_LEVEL_INVENTORY': '1'}
SERVER_SPEC_STEPS = 3   # speculative-decoding draft steps (da-fr: 3). 4 is an experiment: validate boot + memory first
SERVER_MAMBA_CACHE = 60  # v29: back to da-fr's 60 (v21-v28 used 50; both leaderboard scores with 50 were low)
assert SERVER_MAMBA_CACHE // 5 >= 10, 'the server keeps 5 Mamba slots per running request: need >= 50 for 10 games'
MEMORY_PLACEMENT             = 'history'  # v2: 'history' (no prefix-cache misses) | 'system' (v1 behavior, A/B only)
ENABLE_SERVER_REPORT         = True   # Save & Run: print cache reuse / throughput / tokens-per-action after the run
ENABLE_WORLD_MODEL_TOOLS     = True   # v4: check_rules / plan_route in the python sandbox (False = v3 behavior)

# ---------------- input discovery (TAAF launcher): accept both Kaggle mount layouts ----------------
def _resolve_input(path: str) -> str:
    p = Path(path)
    if p.exists():
        return str(p)
    parts = p.parts
    candidates = []
    if len(parts) > 5 and parts[3] in ('datasets', 'models'):
        # /kaggle/input/<datasets|models>/<owner>/<rest...>  ->  /kaggle/input/<rest...>
        candidates.append(Path('/kaggle/input', *parts[5:]))
    for c in candidates:
        if c.exists():
            print(f'input resolved: {path} -> {c}')
            return str(c)
    print(f'WARNING: input not found: {path} (tried {[str(c) for c in candidates]})')
    return path

WHEELHOUSE_DIR  = _resolve_input(WHEELHOUSE_DIR)
MODEL_DIR       = _resolve_input(MODEL_DIR)
DRAFT_MODEL_DIR = _resolve_input(DRAFT_MODEL_DIR)
ORIG_BUNDLE_DIR = _resolve_input(ORIG_BUNDLE_DIR)

# Apply harness patch
BUNDLE_DIR = Path('/kaggle/taaf-kaggle-source-share')
!rm -Rf $BUNDLE_DIR
!cp -a $ORIG_BUNDLE_DIR $BUNDLE_DIR
_patch = subprocess.run(["git", "apply", "--include=ARC3-Inference/*", "-v", "/kaggle/harness-changes.patch"],
                        cwd=f"{BUNDLE_DIR}/src", capture_output=True, text=True)
_patch_files = (_patch.stdout + _patch.stderr).count('Applied patch')
if _patch.returncode != 0:
    print(_patch.stdout[-6000:], _patch.stderr[-6000:])
    import hashlib as _hl
    _sha = _hl.sha256(open('/kaggle/harness-changes.patch', encoding='utf-8').read().rstrip('\n').encode()).hexdigest()
    if _sha != '07fd859e7ce7ad8ac8a2f8f726d862660c1be1abc97560a63b5985703ddf7f4f':
        print('DIAGNOSIS: the patch cell (cell 2) was edited - a stray space or changed line is enough. '
              'Re-upload the untouched notebook file; do not open or edit the big patch cell.')
    else:
        print('DIAGNOSIS: the patch is intact, so the attached source bundle differs from the expected one. '
              'Check that dfranzen/taaf-kaggle-source-bundle-copy is attached (not another bundle or version).')
    raise RuntimeError(f'harness patch failed (git apply exit {_patch.returncode})')
print(f'harness patch applied successfully ({_patch_files} files)')
if _patch_files != 16:
    print('WARNING: expected 16 patched files; full git output follows:')
    print(_patch.stdout + _patch.stderr)

# True only inside a real competition rerun; switches diagnostics + soft deadline.
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
if not TRUE_SUBMISSION:
    # v59b/v59c: a test run releases the games only once the model server is up (at most 75 min), so its
    # minutes are all play (Oct 24: a slow machine was ready at 2,567 s, after v59's 30-minute window had closed -
    # nothing played). The contest rerun keeps the 12-minute release.
    SERVER_STARTUP_TIMEOUT = 75 * 60
NOTEBOOK_START_EPOCH = time.time()

# Non-interactive matplotlib backend: diagnostics render plots with no display attached.
os.environ["MPLBACKEND"] = "Agg"
# Marks the run as a (real or emulated) submission so the framework + solver can adjust.
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
# In submission, disable the periodic JSON/HTML diagnostics writes and per-frame logging.
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"
# Pin arc_agi's cached level_reset_only before its client is built (RESET keeps the level).
os.environ["ONLY_RESET_LEVELS"] = "true"

# Everything the run produces is written here.
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print(f"taaf.kaggle: TRUE_SUBMISSION={TRUE_SUBMISSION}")
SERVED_MODEL_HOST = '127.0.0.1'
SERVED_MODEL_PORT = 8001
SERVER_BASE_URL = f'http://{SERVED_MODEL_HOST}:{SERVED_MODEL_PORT}/v1'
USE_PRIORITY_SCHEDULING = True

setup_env = {
    ########## transformers config ##########
    'USE_TF': '0',
    'TRANSFORMERS_NO_TF': '1',
    'TRANSFORMERS_NO_TORCHVISION': '1',

    ########## serving config ##########
    'LOCAL_ANALYZER_BASE_URL': SERVER_BASE_URL,
    'OPENAI_BASE_URL': SERVER_BASE_URL,
    'LOCAL_ANALYZER_PROVIDER': 'vllm',
    'OPENAI_PROVIDER': 'vllm',
    'LOCAL_ANALYZER_MODEL_ID': SERVED_MODEL_NAME,
    'INFERENCE_ANALYZER_MODEL': SERVED_MODEL_NAME,
    'LOCAL_ANALYZER_APP_NAME': 'ARC3 Agent Harness',
    
    ########## analyzer config ##########
    'ARC3_REASONING_HISTORY_KEY': 'reasoning' if SERVER_ENGINE == 'vllm' else 'reasoning_content',  # v35: on vLLM checked at start by a probe (launcher)
    'LOCAL_ANALYZER_ENABLE_THINKING': 'true',
    'LOCAL_ANALYZER_TOOL_STEPS': '0',
    'LOCAL_ANALYZER_TOOL_TIMEOUT': '30',
    'LOCAL_ANALYZER_YIELD_SECONDS': '0',
    'LOCAL_ANALYZER_YIELD_TOKENS': '2048',
    'LOCAL_ANALYZER_TEMPERATURE': '0.7',   # v59: back to v48 / da-fr (v54-v58 ran 1.0: decode ~7% slower at the same load, MTP acceptance 2.43 vs 2.70; v55 LB 24); first impressions keep 1.0 (FI_TEMPERATURE)
    'LOCAL_ANALYZER_TOP_P': '0.95',
    'LOCAL_ANALYZER_TOP_K': '20',
    'ARC3_REASONING_EFFORT_LADDER': 'medium' if V60_EFFORT_LADDER else '',   # v60 (empty = off, the harness default)

    ########## context pruning ##########
    'LOCAL_ANALYZER_CONTEXT_WINDOW': ((VLLM_HISTORY_BUDGET_K if SERVER_ENGINE == 'vllm' else 116)+12)*1024,  # v38
    'LOCAL_ANALYZER_MAX_OUTPUT': 12*1024,
    'ARC3_CONTEXT_DRAIN_TOKENS': (((VLLM_HISTORY_BUDGET_K if SERVER_ENGINE == 'vllm' else 116) - V60_TRIM_KEEP_K)*1024 if V60_LIGHT_TRIMS
                                  else (VLLM_HISTORY_BUDGET_K if SERVER_ENGINE == 'vllm' else 116)//2*1024),  # v38: half the budget; v60: keep V60_TRIM_KEEP_K
    'ARC3_HISTORY_ASSISTANT_TURNS': 150,
    'ARC3_HISTORY_TURN_DRAIN': 30,
    'ARC3_HISTORY_DRAIN_COALESCE': '1',

    ########## tool output ##########
    'LOCAL_ANALYZER_TOOL_OUTPUT_TOKENS': '3072',
    'ARC3_MIDDLE_TRUNCATION': '1',
    'ARC3_ACTION_ECHO': '1',
    
    ########## prompt changes ##########
    'ARC3_EXPLAIN_GAMEPLAY_CHANGED': '1',
    'ARC3_NEW_CHANGED_PROMPTS': '1',
    'ARC3_LEVEL_TRANSFER_GUIDANCE': '1',
    'ARC3_ACTION_INFO': '1',
    'ARC3_DEDUPE_MULTICALL_LINE': '1',
    'ARC3_FRAME_DIFF_HINT': '1',

    ########## vision & animations ##########
    'MULTIMODAL_CONTEXT': 'current_grid',
    'MULTIMODAL_UPSCALE': '10',
    'ARC3_DIFF_IMAGE': '0' if V60_NO_DIFF_IMAGE else '1',   # v60
    'ARC3_GAMEOVER_DIFF_IMAGE': '1',
    'ARC3_ANIMATION': '1',

    ########## guards ##########
    'ARC3_GUARDS_FROM_LEVEL': 2,
    'ARC3_BATCH_NOOP_BLOCK': '1',
    'ARC3_STALE_STATE_BLOCK': '1',

    ########## world model ##########
    'ARC3_MEMORY_SECTIONS': 'off',

    ########## persistent functions ##########
    'ARC3_PERSISTENT_FUNCTIONS': '1',
    'ARC3_PERSISTENT_FUNCTIONS_SCOPE': 'game',
    'ARC3_PERSISTENT_FUNCTIONS_IMPORTS': '1',

    ########## controls ##########
    'EXPOSE_UNDO': 'on',
}

if USE_PRIORITY_SCHEDULING: setup_env.update({
    'ARC3_MAX_ACTIVE_STREAMS': VLLM_MAX_NUM_SEQS if SERVER_ENGINE == 'vllm' else 10,  # v35
    'ARC3_PRIORITY_REFRESH_QUEUE': '1',
    'ARC3_PRIORITY_PACE': '0',
    'ARC3_PRIORITY_TAIL_FADE': '1',
    'ARC3_PRIORITY_TAIL_FADE_FRACTION': '0.2',
    'ARC3_PRIORITY_TAIL_LOOKUP': 'remaining',
    'ARC3_PRIORITY_SCORE_NORMALIZATION': '1',
})

if PROFILE not in PROFILES:
    raise ValueError(f'PROFILE must be one of {sorted(PROFILES)}, got {PROFILE!r}')
setup_env.update(PROFILES[PROFILE])
setup_env.update(EXTRA_ENV)
print(f'v3 profile {PROFILE!r}: {dict(PROFILES[PROFILE], **EXTRA_ENV)} | spec steps {SERVER_SPEC_STEPS}')

if ENABLE_SERVER_WATCHDOG:
    # Must be set before the solver module is imported (benchmark unpickle). The stock limit (10 failed
    # requests, ~4 min) abandons every game during a server relaunch. v32: 240 is ~100 min of patience (60 was
    # ~25 min); a relaunch with the 886 s weight load seen on 2026-10-13 takes ~20 min.
    # Any successful turn resets the count, so a healthy server never sees a difference.
    setup_env['ARC3_MAX_ANALYZER_FAILURES'] = 240

os.environ.update({k: str(v) for k,v in setup_env.items()})

## 2. Precaching

Precache wheelhouse for faster startup.

In [ ]:
import os
import threading
import time

from concurrent.futures import as_completed, ThreadPoolExecutor

def precache(*paths, times=1, delay=0, threads=16) -> None:
    time.sleep(delay)
    print(f'Precaching: Started...')
    start_time = time.time()

    CHUNK_SIZE = 32 * 1024 * 1024  # 128 MB

    chunks = []  # list of (path, offset, length)
    total_size = 0
    file_count = 0

    def _add_file(i: int, path: str) -> None:
        nonlocal total_size, file_count
        size = os.path.getsize(path)
        total_size += size
        file_count += 1
        for offset in range(0, max(size, 1), CHUNK_SIZE):
            length = min(CHUNK_SIZE, size - offset)
            chunks.append((i, path, offset, length))

    for i, path in enumerate(paths):
        try:
            if os.path.isfile(path):
                _add_file(i, path)
                continue
            for root, _, files in os.walk(path):
                for file_name in files:
                    file_path = os.path.join(root, file_name)
                    if os.path.isfile(file_path):
                        _add_file(i, file_path)
        except: pass

    chunks.sort()

    _local = threading.local()

    def _read_chunk(chunk: tuple) -> None:
        try:
            buf = _local.buf
        except AttributeError:
            _local.buf = buf = bytearray(CHUNK_SIZE)
        i, path, offset, _ = chunk
        try:
            with open(path, 'rb') as f:
                f.seek(offset)
                remaining = CHUNK_SIZE
                for i in range(1024):
                    n = f.readinto(buf)
                    if not n:
                        break
                    remaining -= n
                    if remaining <= 0: break
        except: pass

    with ThreadPoolExecutor(max_workers=threads) as pool:
        for _ in range(times):
            list(pool.map(_read_chunk, chunks))

    elapsed = time.time() - start_time
    print(f'Precaching: loaded {file_count} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')

if SERVER_ENGINE == 'sglang':   # v35: the vLLM runtime brings its own files
    precache_wheels_thread = threading.Thread(target=precache, args=(WHEELHOUSE_DIR,))
    precache_wheels_thread.start()
    precache_wheels_thread.join()

precache_model_thread = threading.Thread(target=precache, args=(MODEL_DIR, DRAFT_MODEL_DIR), kwargs={'delay': 60, 'threads': 3})

## 3. Install the ARC runtime

Install `arc-agi` from the offline competition wheelhouse (the Kaggle submission environment
has no internet).

In [ ]:
# Install the ARC runtime from the bundled competition wheels.
# Quiet: stdout is discarded; stderr (and a non-zero exit) still surface real failures.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        str(next(iter(sorted(Path("/kaggle/input").glob("**/arc_agi_3_wheels"))),
                 Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels"))),
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)

## 4. Import the bundled source

Put the snapshotted repositories on the path (this process and any child processes).

In [ ]:
# Each bundled repo exposes its importable tree at <repo>/src or <repo>.
def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


# Environment handed to each setup command (paths + any keys it has persisted).
def _command_env() -> dict:
    env = os.environ.copy()
    # "$PYTHON" in a command resolves to this notebook's interpreter.
    env["PYTHON"] = sys.executable
    # Absolute path to the mounted source bundle.
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    # The writable /kaggle/working directory.
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    # A command writes a JSON object here to persist env keys to later commands + the run.
    return env


# Make the bundled repos importable here (sys.path) and in child processes (.pth).
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in source_entries))
print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)")

## 5. Start serving (v35: vLLM, with v34's SGLang as the fallback)

The next cells write the two server launchers to files and then start the one chosen by `SERVER_ENGINE` in the config cell. The SGLang launcher is v34's serve cell, unchanged. The vLLM launcher is adapted from lordhansolo's public notebook (see the intro).

In [ ]:
%%writefile /tmp/arc_sglang_serve.py
# Paste this entire file into the launcher cell.
# Required notebook variables: MODEL_DIR, DRAFT_MODEL_DIR, WHEELHOUSE_DIR,
# WORKING_DIR, SERVED_MODEL_NAME, SERVED_MODEL_PORT.
# MODEL_DIR: Intel AutoRound INT4 target (BF16 PLE or albucino FP8 PLE).
# DRAFT_MODEL_DIR: downloaded albucino runtime/mtp-int4-g32, or its dataset root.
import os, sys, glob, time, shutil, subprocess, urllib.request, urllib.error
import copy, hashlib, json, re, shlex, socket, zipfile
from pathlib import Path

PREFIX = "/tmp/sgl-intel"       # Separate from the previous fork's /tmp/sgl venv.
LOG = str(Path(WORKING_DIR) / "serve.log")
CFG = dict(
    CTX=(116+12+8)*1024,
    MEMFRAC=0.96,
    MAXREQ=10,
    CUDAGRAPH_MAXBS=10,
    MAMBA_CACHE=SERVER_MAMBA_CACHE,
    CHUNK=8192,
    MAX_PREFILL=16384,
    SPEC=True,
    FRSPEC=True,
    AUTOTUNE=True,
    # Preserve the original Kaggle checkpoint prefetch/cache behavior.
    PREFETCH_CHECKPOINTS=True,
    DROP_CACHE_AFTER_LOAD=True,
    LINEAR_BACKEND="flashinfer",
    KVDTYPE="fp8_e4m3",
    SSM_DTYPE="bfloat16",
    MAMBA_RADIX="extra_buffer",
    GDN_MTP_CACHE_MODE="none",
    SERVED_NAME=SERVED_MODEL_NAME,
    SPEC_ACCEPT_SINGLE=1.0,
    SPEC_ACCEPT_ACC=1.0,
    SPEC_STEPS=SERVER_SPEC_STEPS,
)
VENV = f"{PREFIX}/venv"
PYTHON = f"{VENV}/bin/python"
SGLANG = f"{VENV}/bin/sglang"
TOKEN_MAP_SHA = "becfa41d394b86c26c632bea8f3c6ea64bbb76d7b238d8673c06afae21269f25"
TOKENIZER_SHA = "06b9509352d2af50381ab2247e083b80d32d5c0aba91c272ca9ff729b6a0e523"
EXPERT_TARGET = r"re:^mtp\.layers\.0\.mlp\.experts\.[0-9]+\.(gate_proj|up_proj|down_proj)$"
DENSE_IGNORE = r"re:^(?!mtp\.layers\.0\.mlp\.experts(?:\.|$)).*"


def run(cmd, *, check=True, env=None):
    cmd = list(map(str, cmd))
    print("$", shlex.join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True, env=env)
    if result.stdout.strip(): print(result.stdout.strip()[-6000:])
    if result.stderr.strip() and result.returncode: print(result.stderr.strip()[-6000:])
    if check and result.returncode:
        raise RuntimeError(f"Command failed ({result.returncode}): {shlex.join(cmd)}")
    return result


def find_unique(root, name, *, directory=False, required=True):
    root = Path(root)
    direct = root if root.name == name else root / name
    valid = lambda p: p.is_dir() if directory else p.is_file()
    if valid(direct): return direct
    hits = sorted(p for p in root.rglob(name) if valid(p))
    if len(hits) > 1:
        raise RuntimeError(f"Multiple {name} matches under {root}; use a more specific directory: {hits}")
    if hits: return hits[0]
    if required: raise FileNotFoundError(f"{name} not found under {root}")
    return None


def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(8*1024*1024), b""): h.update(block)
    return h.hexdigest()


def indexed_shards(root):
    root = Path(root)
    index = json.loads((root / "model.safetensors.index.json").read_text())
    names = sorted(set(index["weight_map"].values()))
    if not names: raise RuntimeError(f"Empty weight index: {root}")
    for name in names:
        rel = Path(name)
        if rel.is_absolute() or ".." in rel.parts:
            raise RuntimeError(f"Invalid shard path: {name}")
        p = root / rel
        if not p.is_file() or p.stat().st_size < 8:
            raise RuntimeError(f"Missing/empty shard: {p}")
    return index, names


def prepare_draft_view(target, source_root, work_root):
    """Adapt metadata only; link original dataset weights without copying them."""
    target, source_root, work_root = map(Path, (target, source_root, work_root))
    configs = [source_root / "config.json"] if (source_root / "config.json").is_file() else []
    configs += sorted(p for p in source_root.rglob("config.json") if p not in configs)
    candidates = []
    for path in configs:
        cfg = json.loads(path.read_text())
        q = cfg.get("quantization_config", {})
        if q.get("quant_method") == "compressed-tensors" and "mtp_routed_experts" in q.get("config_groups", {}):
            candidates.append((path.parent, cfg))
    if len(candidates) != 1:
        raise RuntimeError(f"Expected one INT4 g32 MTP checkpoint under DRAFT_MODEL_DIR={source_root}; found {len(candidates)}")
    source, cfg = candidates[0]
    index, shards = indexed_shards(source)
    if not any(k.startswith("mtp.layers.0.mlp.experts.") for k in index["weight_map"]):
        raise RuntimeError("Draft index does not contain the expected MTP experts")
    cfg = copy.deepcopy(cfg)
    q = cfg["quantization_config"]
    group = q["config_groups"]["mtp_routed_experts"]
    w = group["weights"]
    if (w["num_bits"], w["group_size"], w["symmetric"]) != (4, 32, True):
        raise RuntimeError("Expected symmetric INT4 group32 draft experts")
    if group["targets"] not in (["RoutedExperts"], [EXPERT_TARGET]):
        raise RuntimeError(f"Unexpected draft target rules: {group['targets']}")
    if q.get("ignore", []) not in ([], [DENSE_IGNORE]):
        raise RuntimeError("Unexpected draft ignore rules")
    group["targets"] = [EXPERT_TARGET]
    q["ignore"] = [DENSE_IGNORE]
    # Content/path-specific directory makes notebook reruns idempotent without
    # overwriting an unrelated view. We never modify the read-only datasets.
    identity = json.dumps({"source": str(source.resolve()), "target": str(target.resolve()), "config": cfg}, sort_keys=True)
    suffix = hashlib.sha256(identity.encode()).hexdigest()[:16]
    view = work_root / f"draft-view-{suffix}"
    view.mkdir(parents=True, exist_ok=True)
    config_path = view / "config.json"
    config_text = json.dumps(cfg, indent=2) + "\n"
    if config_path.exists() or config_path.is_symlink():
        if config_path.is_symlink() or config_path.read_text() != config_text:
            raise RuntimeError(f"Unexpected existing draft-view config: {config_path}")
    else:
        config_path.write_text(config_text)
    links = {name: source / name for name in shards}
    links["model.safetensors.index.json"] = source / "model.safetensors.index.json"
    for name in ("tokenizer.json", "tokenizer_config.json", "chat_template.jinja", "preprocessor_config.json", "generation_config.json"):
        if (target / name).is_file(): links[name] = target / name
    for name, src in links.items():
        dest = view / name
        dest.parent.mkdir(parents=True, exist_ok=True)
        if dest.exists() or dest.is_symlink():
            if not dest.is_symlink() or dest.resolve() != src.resolve():
                raise RuntimeError(f"Unexpected existing draft-view file: {dest}")
        else:
            dest.symlink_to(src.resolve())
    print(f"Draft source: {source}\nSGLang draft view: {view} ({len(shards)} linked weight shards)")
    return view


# ---- validate notebook inputs / identify the offline bundle ----
MODEL_DIR = str(Path(MODEL_DIR).expanduser().resolve())
assert Path(MODEL_DIR).is_dir(), f"model dir not found: {MODEL_DIR}"
assert CFG["CUDAGRAPH_MAXBS"] >= CFG["MAXREQ"], "CUDAGRAPH_MAXBS must cover MAXREQ"
assert CFG["CHUNK"] > 0 and CFG["CHUNK"] % 64 == 0
# Refuse to start a second server on an occupied port; do not kill other jobs.
with socket.socket() as probe:
    if probe.connect_ex(("127.0.0.1", int(SERVED_MODEL_PORT))) == 0:
        raise RuntimeError(f"Port {SERVED_MODEL_PORT} is occupied; stop the existing server before rerunning")
Path(PREFIX).mkdir(parents=True, exist_ok=True)
Path(LOG).parent.mkdir(parents=True, exist_ok=True)
WHEELS = find_unique(WHEELHOUSE_DIR, "wheels", directory=True)
LOCK = find_unique(WHEELHOUSE_DIR, "requirements.lock")
sglang_wheels = sorted(WHEELS.glob("sglang-*.whl"))
assert len(sglang_wheels) == 1, f"Expected one SGLang wheel, found {sglang_wheels}"
wheel = sglang_wheels[0]
install_marker = Path(PREFIX) / "installed-bundle.txt"

# ---- one-time offline install, isolated from the old mratsim environment ----
install_env = dict(os.environ, UV_OFFLINE="1", UV_PYTHON_DOWNLOADS="never", PYTHONNOUSERSITE="1")
install_env.pop("PYTHONPATH", None)
if not Path(SGLANG).exists() or not install_marker.is_file():
    print("=== installing Pennyroyal (offline) ===")
    # Bootstrap uv directly from its wheel; no global pip changes/ensurepip.
    uv_wheels = sorted(WHEELS.glob("uv-*.whl"))
    assert len(uv_wheels) == 1, "Expected one bundled uv wheel"
    uv_binary = Path(PREFIX) / "bin/uv"
    uv_binary.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(uv_wheels[0]) as archive:
        members = [n for n in archive.namelist() if Path(n).name == "uv" and not n.endswith("/")]
        assert len(members) == 1, "Cannot identify the Linux uv executable in its wheel"
        binary = archive.read(members[0])
        assert binary.startswith(b"\x7fELF"), "Bundled uv executable is not a Linux ELF binary"
        uv_binary.write_bytes(binary)
    uv_binary.chmod(0o755)
    uv = [str(uv_binary)]
    if not Path(PYTHON).exists():
        run(uv + ["venv", "--python", sys.executable, VENV], env=install_env)
    install = uv + ["pip", "install", "--python", PYTHON, "--no-index", "--find-links", str(WHEELS)]
    run(install + ["-r", str(LOCK)], env=install_env)
    run(install + ["--reinstall", "--no-deps", str(wheel)], env=install_env)
    install_marker.write_text('install')
else:
    print(f"Matching offline bundle already installed at {VENV}")

# ---- bundled CUDA toolkit and linker symlinks ----
cu13 = sorted(Path(VENV).glob("lib/python*/site-packages/nvidia/cu13"))
if not cu13 or not (cu13[0] / "bin/nvcc").is_file():
    raise RuntimeError("Bundled CUDA 13 nvcc missing; check that the complete Pennyroyal wheelhouse was uploaded")
CUDA_HOME = str(cu13[0])
libdir = Path(CUDA_HOME) / "lib"
lib64 = Path(CUDA_HOME) / "lib64"
if not lib64.exists() and not lib64.is_symlink(): lib64.symlink_to("lib")
for so in sorted(libdir.glob("*.so.*")):
    link = libdir / re.sub(r"\.so\..*$", ".so", so.name)
    if not link.exists() and not link.is_symlink(): link.symlink_to(so.name)
assert (libdir / "libcudart.so").exists(), "libcudart.so missing: JIT linker cannot find CUDA runtime"
if not (libdir / "libcuda.so").exists():
    candidates = []
    for directory in ("/usr/local/nvidia/lib64", "/usr/local/nvidia/lib", "/usr/lib/x86_64-linux-gnu", "/usr/lib64"):
        candidates += sorted(p for p in Path(directory).glob("libcuda.so*") if p.is_file())
    if not candidates:
        code = 'import torch; torch.cuda.init(); print(open("/proc/self/maps").read())'
        maps = run([PYTHON, "-c", code], env=install_env).stdout
        candidates = [Path(p) for p in re.findall(r"(/[^\s]*libcuda\.so[^\s]*)", maps)]
    if not candidates: raise RuntimeError("GPU driver libcuda.so not found")
    link = libdir / "libcuda.so"
    if link.is_symlink(): link.unlink()
    link.symlink_to(candidates[0].resolve())
CXX = next((c for c in ("g++-15", "g++-14", "g++-13", "g++-12", "g++-11", "g++") if shutil.which(c)), None)
assert CXX, "Host C++ compiler not found"
CC = CXX.replace("g++", "gcc")
assert shutil.which(CC), f"Matching C compiler missing: {CC}"

# ---- runtime environment ----
env = dict(os.environ)
env.pop("PYTHONPATH", None)
cache = f"{PREFIX}/cache"
for name in ("huggingface", "torch", "torchinductor", "triton", "flashinfer", "cuda", "sglang/jit"):
    Path(cache, name).mkdir(parents=True, exist_ok=True)
env.update({
    "PYTHONNOUSERSITE": "1", "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
    "HF_DATASETS_OFFLINE": "1", "UV_OFFLINE": "1", "UV_PYTHON_DOWNLOADS": "never",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "CUDA_HOME": CUDA_HOME, "CUDACXX": f"{CUDA_HOME}/bin/nvcc",
    "PATH": f"{CUDA_HOME}/bin:{VENV}/bin:" + env.get("PATH", ""),
    "LD_LIBRARY_PATH": f"{CUDA_HOME}/lib64:{CUDA_HOME}/lib:" + env.get("LD_LIBRARY_PATH", ""),
    "CC": CC, "CXX": CXX, "CUDAHOSTCXX": CXX,
    "TORCH_CUDA_ARCH_LIST": "12.0",
    "HF_HOME": f"{cache}/huggingface", "XDG_CACHE_HOME": cache,
    "TORCH_HOME": f"{cache}/torch", "CUDA_CACHE_PATH": f"{cache}/cuda",
    "TORCHINDUCTOR_CACHE_DIR": f"{cache}/torchinductor", "TRITON_CACHE_DIR": f"{cache}/triton",
    "FLASHINFER_WORKSPACE_BASE": f"{cache}/flashinfer",
    "SGLANG_CACHE_DIR": f"{cache}/sglang", "SGLANG_JIT_CACHE_DIR": f"{cache}/sglang/jit",
    "SGLANG_ALLOW_OVERWRITE_LONGER_CONTEXT_LEN": "1",
    "OMP_NUM_THREADS": "8", "MKL_NUM_THREADS": "8", "TOKENIZERS_PARALLELISM": "false",
    "MAX_JOBS": "8", "CMAKE_BUILD_PARALLEL_LEVEL": "8",
    "FLASHINFER_NINJA_JOBS": "8", "FLASHINFER_NVCC_THREADS": "2", "TORCHINDUCTOR_COMPILE_THREADS": "8",
    "SGLANG_ENABLE_SM120_LOWM_BF16_GEMM": "1", "SGLANG_SM120_ONLINE_MXFP8": "0",
    "SGLANG_SM120_LOWM_FP8_WEIGHT": "0", "SGLANG_SM120_LM_HEAD_FP8": "0",
    "SGLANG_MAMBA_CONV_DTYPE": "bfloat16", "SGLANG_NUMA_BIND_V2": "false",
    "SGLANG_MM_PREPROCESS_DEVICE": "cpu", "NUMPY_MADVISE_HUGEPAGE": "0",
})
run([str(Path(CUDA_HOME) / "bin/nvcc"), "--version"], env=env)
print("Host compiler:", CXX)
# Report the optional later caching patch. This cell does not modify wheel code.
run([PYTHON, "-c", '''import pathlib, sglang, torch
print("sglang", sglang.__version__, "| torch", torch.__version__)
assert torch.cuda.is_available(), "CUDA unavailable"
print("GPU:", torch.cuda.get_device_name(0), "capability:", torch.cuda.get_device_capability(0))
'''], env=env)

# ---- validate target and create the read-only draft's SGLang view ----
target_cfg = json.loads(Path(MODEL_DIR, "config.json").read_text())
q = target_cfg.get("quantization_config", {})
assert q.get("quant_method") == "auto-round" and q.get("bits") == 4, "MODEL_DIR must be the Intel/albucino AutoRound INT4 checkpoint"
_, target_shards = indexed_shards(MODEL_DIR)
print("Target:", MODEL_DIR, "| shards:", len(target_shards), "| PLE dtype:", target_cfg.get("text_config", {}).get("ple_embedding_dtype", "checkpoint default (typically BF16)"))
DRAFT_VIEW = None
if CFG["SPEC"]:
    assert Path(DRAFT_MODEL_DIR).is_dir(), f"draft dir not found: {DRAFT_MODEL_DIR}"
    DRAFT_VIEW = prepare_draft_view(MODEL_DIR, DRAFT_MODEL_DIR, PREFIX)

# ---- assemble Pennyroyal / AutoRound launch arguments ----
graph_bs = sorted({1, 2, 4, 7, 8, 9, 10, CFG["MAXREQ"], CFG["CUDAGRAPH_MAXBS"]})
graph_bs = [n for n in graph_bs if n <= CFG["CUDAGRAPH_MAXBS"]]
args = [SGLANG, "serve", "--model-path", MODEL_DIR, "--load-format", "safetensors",
    "--model-loader-extra-config", '{"enable_multithread_load":false}',
    "--served-model-name", str(CFG["SERVED_NAME"]), "--host", "127.0.0.1",
    "--port", str(SERVED_MODEL_PORT), "--tensor-parallel-size", "1",
    "--dtype", "bfloat16", "--quantization", "auto-round",
    "--kv-cache-dtype", CFG["KVDTYPE"], "--mem-fraction-static", str(CFG["MEMFRAC"]),
    "--context-length", str(CFG["CTX"]), "--page-size", "64",
    "--max-running-requests", str(CFG["MAXREQ"]), "--chunked-prefill-size", str(CFG["CHUNK"]),
    "--max-prefill-tokens", str(CFG["MAX_PREFILL"]),
    "--cuda-graph-max-bs-decode", str(CFG["CUDAGRAPH_MAXBS"]),
    "--cuda-graph-bs-decode", *map(str, graph_bs),
    "--mamba-ssm-dtype", CFG["SSM_DTYPE"], "--max-mamba-cache-size", str(CFG["MAMBA_CACHE"]),
    "--mamba-radix-cache-strategy", CFG["MAMBA_RADIX"], "--mamba-track-interval", "64",
    "--mamba-backend", CFG["LINEAR_BACKEND"],
    "--linear-attn-decode-backend", CFG["LINEAR_BACKEND"],
    "--linear-attn-prefill-backend", CFG["LINEAR_BACKEND"],
    "--moe-runner-backend", "auto", "--ple-offload-embedding", "--trust-remote-code",
    "--mm-feature-transport", "cpu", "--image-processor-backend", "pil",
    "--reasoning-parser", "qwen3", "--tool-call-parser", "qwen3_coder",
    "--default-chat-template-kwargs", '{"preserve_thinking":true}',
    "--watchdog-timeout", "1800", "--schedule-policy", "lpm", "--warmups", "structured_output",
    "--enable-cache-report", "--enable-metrics", "--enable-request-time-stats-logging"]
if CFG["PREFETCH_CHECKPOINTS"]: args += ["--weight-loader-prefetch-checkpoints"]
if CFG["DROP_CACHE_AFTER_LOAD"]: args += ["--weight-loader-drop-cache-after-load"]
tmpl = Path(MODEL_DIR, "chat_template.jinja")
if tmpl.is_file(): args += ["--chat-template", str(tmpl)]
if not CFG["AUTOTUNE"]: args += ["--disable-flashinfer-autotune"]
if CFG["GDN_MTP_CACHE_MODE"]: args += ["--gdn-mtp-cache-mode", CFG["GDN_MTP_CACHE_MODE"]]
if CFG["SPEC"]:
    args += ["--speculative-algorithm", "NEXTN", "--speculative-num-steps", str(CFG["SPEC_STEPS"]),
        "--speculative-eagle-topk", "1", "--speculative-num-draft-tokens", str(CFG["SPEC_STEPS"]+1),
        "--speculative-draft-model-path", str(DRAFT_VIEW),
        "--speculative-draft-model-quantization", "compressed-tensors", "--speculative-moe-runner-backend", "auto",
        "--speculative-draft-kv-cache-dtype", CFG["KVDTYPE"],
        "--speculative-accept-threshold-single", str(CFG["SPEC_ACCEPT_SINGLE"]),
        "--speculative-accept-threshold-acc", str(CFG["SPEC_ACCEPT_ACC"])]
    if CFG["FRSPEC"]:
        tok = find_unique(WHEELHOUSE_DIR, "hot_tokens_64k.pt", required=False)
        if tok is None: tok = find_unique(WHEELHOUSE_DIR, "flash-next-64k.pt")
        assert sha256(tok) == TOKEN_MAP_SHA, "Wrong FR-Spec map; use the map from the pinned Pennyroyal bundle"
        assert sha256(Path(MODEL_DIR, "tokenizer.json")) == TOKENIZER_SHA, "Tokenizer differs from the FR-Spec map; set CFG['FRSPEC']=False"
        args += ["--speculative-token-map", str(tok)]

# ---- v19: prefix-cache diagnostics (Save & Run only; measurement, never changes serving) ----
ARC_CACHE_DIAG_SOURCE = r'''# ARC v19 prefix-cache diagnostics for the SGLang scheduler (measurement only, never changes serving).
# Written into the installed sglang package by the notebook's serve cell, only in Save & Run (never in a competition
# rerun), and active only when ARC_CACHE_DIAG_PATH is set. Question it answers: when a game's next request re-reads
# the model's previous reply instead of reusing it from cache, is it because
#   (a) the server never kept a Mamba checkpoint at the end of the reply (finish record: last_track <= prompt), or
#   (b) the next prompt differs from prompt+reply somewhere inside the reply (new record: lcp < previous length)?
# Every hook is wrapped: any exception is counted and the original method result is returned unchanged.
import json
import os
import threading
import time

_PATH = os.environ.get("ARC_CACHE_DIAG_PATH", "")
_LOCK = threading.Lock()
_RECENT = []            # newest last: dict(rid, seq, P, O, kv, last_track, t)
_MAX_RECENT = 24
_ERRORS = [0]
_MAX_ERRORS = 50        # after this many failures the hooks switch themselves off


def _np():
    import numpy
    return numpy


def _arr(x):
    np = _np()
    return np.fromiter(x, dtype=np.int64, count=len(x))


def _write(rec):
    try:
        line = json.dumps(rec)
        with _LOCK:
            with open(_PATH, "a") as f:
                f.write(line + "\n")
    except Exception:
        _ERRORS[0] += 1


def _active():
    return bool(_PATH) and _ERRORS[0] < _MAX_ERRORS


def _on_finish(req, kv_len, is_insert):
    P = len(req.origin_input_ids)
    O = len(req.output_ids)
    kv = int(kv_len) if kv_len is not None else P + O
    seq = _arr((list(req.origin_input_ids) + list(req.output_ids))[:kv])
    rec = {"ev": "finish", "rid": str(req.rid), "P": P, "O": O, "kv": kv,
           "last_track": getattr(req, "mamba_last_track_seqlen", None),
           "protected": getattr(req, "cache_protected_len", None),
           "insert": bool(is_insert), "t": time.time()}
    with _LOCK:
        _RECENT.append({"rid": rec["rid"], "seq": seq, "P": P, "O": O, "kv": kv,
                        "last_track": rec["last_track"], "t": rec["t"]})
        del _RECENT[:-_MAX_RECENT]
    _write(rec)


def _on_new(req):
    np = _np()
    x = _arr(req.full_untruncated_fill_ids)
    matched = len(req.prefix_indices) if getattr(req, "prefix_indices", None) is not None else None
    best, best_lcp = None, -1
    with _LOCK:
        recent = list(_RECENT)
    for r in recent:
        s = r["seq"]
        n = min(len(s), len(x))
        neq = np.flatnonzero(s[:n] != x[:n])
        lcp = int(neq[0]) if len(neq) else n
        if lcp > best_lcp:
            best, best_lcp = r, lcp
    rec = {"ev": "new", "rid": str(req.rid), "N": int(len(x)), "matched": matched,
           "branch": getattr(req, "mamba_branching_seqlen", None), "t": time.time()}
    if best is not None:
        rec.update({"prev_rid": best["rid"], "lcp": best_lcp, "prev_P": best["P"], "prev_O": best["O"],
                    "prev_kv": best["kv"], "prev_last_track": best["last_track"],
                    "gap": round(rec["t"] - best["t"], 3)})
        if best_lcp < min(best["kv"], len(x)):
            lo = max(0, best_lcp - 8)
            rec["new_tok"] = [int(v) for v in x[lo:best_lcp + 8]]
            rec["prev_tok"] = [int(v) for v in best["seq"][lo:best_lcp + 8]]
            rec["tok_from"] = lo
    _write(rec)


def install_cache(cache_cls):
    if not _PATH or getattr(cache_cls.cache_finished_req, "_arc_cd", False):
        return False
    inner = cache_cls.cache_finished_req

    def cache_finished_req(self, req, is_insert=True, **kwargs):
        if _active():
            try:
                _on_finish(req, kwargs.get("kv_len_to_handle"), is_insert)
            except Exception:
                _ERRORS[0] += 1
        return inner(self, req, is_insert, **kwargs)

    cache_finished_req._arc_cd = True
    cache_cls.cache_finished_req = cache_finished_req
    _write({"ev": "installed", "what": "cache_finished_req", "t": time.time()})
    return True


def install_req(req_cls):
    if not _PATH or getattr(req_cls.init_next_round_input, "_arc_cd", False):
        return False
    inner = req_cls.init_next_round_input

    def init_next_round_input(self, *args, **kwargs):
        out = inner(self, *args, **kwargs)
        if _active() and not getattr(self, "_arc_cd_seen", False):
            try:
                self._arc_cd_seen = True
                if len(self.output_ids) == 0:
                    _on_new(self)
            except Exception:
                _ERRORS[0] += 1
        return out

    init_next_round_input._arc_cd = True
    req_cls.init_next_round_input = init_next_round_input
    _write({"ev": "installed", "what": "init_next_round_input", "t": time.time()})
    return True
'''
CACHE_DIAG_FILE = None
if ENABLE_CACHE_DIAG and not TRUE_SUBMISSION:
    try:
        _cd_srt = sorted(Path(VENV).glob('lib/python*/site-packages/sglang/srt'))
        assert len(_cd_srt) == 1, _cd_srt
        _cd_srt = _cd_srt[0]
        (_cd_srt / '_arc_cache_diag.py').write_text(ARC_CACHE_DIAG_SOURCE)
        for _cd_rel, _cd_cls, _cd_fn in (('managers/schedule_batch.py', 'Req', 'install_req'),
                                         ('mem_cache/unified_radix_cache.py', 'UnifiedRadixCache', 'install_cache')):
            _cd_file = _cd_srt / _cd_rel
            _cd_txt = _cd_file.read_text()
            assert f'\nclass {_cd_cls}(' in _cd_txt, (_cd_rel, _cd_cls)
            if '# ARC-CACHE-DIAG-HOOK' not in _cd_txt:
                _cd_file.write_text(_cd_txt.rstrip('\n') + '\n\n\n# ARC-CACHE-DIAG-HOOK\ntry:\n'
                                    f'    from sglang.srt import _arc_cache_diag as _arc_cd\n'
                                    f'    _arc_cd.{_cd_fn}({_cd_cls})\nexcept Exception:\n    pass\n')
        CACHE_DIAG_FILE = str(Path(WORKING_DIR) / 'cache_diag.jsonl')
        env['ARC_CACHE_DIAG_PATH'] = CACHE_DIAG_FILE
        print('v19 cache diagnostics hooked ->', CACHE_DIAG_FILE, flush=True)
    except Exception as _cd_exc:
        print('v19 cache diagnostics skipped:', repr(_cd_exc)[:300], flush=True)

# ---- launch detached and wait for health ----
precache_model_thread.start()
print("\n" + shlex.join(args) + "\n")
log_offset = Path(LOG).stat().st_size if Path(LOG).exists() else 0
with open(LOG, "ab", buffering=0) as logf:
    proc = subprocess.Popen(args, env=env, stdout=logf, stderr=subprocess.STDOUT, start_new_session=True)
print(f"pid {proc.pid} -> {LOG}")

def show_log_tail():
    with open(LOG, "rb") as f:
        f.seek(0, 2)
        f.seek(max(log_offset, f.tell()-20000))
        print(f.read().decode(errors="replace"))

url, t0 = f"http://127.0.0.1:{SERVED_MODEL_PORT}/health", time.time()
last_report = -30
while time.time() - NOTEBOOK_START_TIME < SERVER_STARTUP_TIMEOUT:
    if proc.poll() is not None:
        show_log_tail()
        break
    try:
        with urllib.request.urlopen(url, timeout=5) as r:
            if r.status == 200:
                print(f"\nREADY after {int(time.time()-t0)}s -> {url.replace('/health','/v1')}")
                break
    except Exception:
        pass
    elapsed = int(time.time()-t0)
    if elapsed-last_report >= 30:
        print(f"  {elapsed}s ...", flush=True); last_report=elapsed
    time.sleep(3)
else:
    print(f"\nDEADLINE at {int(time.time()-NOTEBOOK_START_TIME)}s from notebook start; "
          f"releasing the benchmark with the server still loading")

## 5a. vLLM launcher (v35)

lordhansolo's vLLM runtime: it unpacks the vLLM main e975732 image layers to /tmp (about 19.5 GB), applies his hash-checked ARC3 overlay and starts the OpenAI server through his watchdog, which restarts the server up to 5 times and prefetches the checkpoint shards. The cell after it starts the server and falls back to SGLang if it cannot start.

In [ ]:
%%writefile /tmp/arc_vllm_serve.py
# ===== v35: vLLM model server (lordhansolo's public Kaggle runtime), written to a file by its cell =====
# v42: REAP-at-load (VLLM_REAP_KEEP, config cell): the target model loads only the kept routed experts of every MoE
# layer (public REAP saliency lists, verified against this checkpoint's routers), freeing their GPU memory for the KV
# cache; the MTP draft keeps all 512. If the pruned server dies before it is ready, the full model starts instead.
# Relaxed MTP acceptance (VLLM_SPEC_RELAX_T): a drafted token is accepted with probability min(1, p/t) instead of p.
# v36: VLLM_PREFIX_RETENTION_INTERVAL (config cell) sets --prefix-cache-retention-interval; -1 leaves the flag out
# (his runtime default, dense Mamba checkpoints). v36's Save & Run showed that no setting lets a turn resume inside
# the previous reply on this runtime, so v37 is back to 0 = v35 = his production setting. The watchdog config also
# names the host-memory log that his test-run monitor thread writes (v35 left it out; the thread stopped on a KeyError).
# Adapted from the setup script of lordhansolo's public notebook "ARC-AGI-3 Milestone 2" (Kaggle
# lordhansolo/arc-agi-3-milestone-2): vLLM main e975732 image runtime + his hash-pinned ARC3 overlay (dataset
# lordhansolo/vllm-main-e975732-arc3), the mixed NVFP4/FP8 Qwen3.8-Flash-Next checkpoint with its built-in MTP head
# (model lordhansolo/qwen3-8-flash-next-mixed-nvfp4-fp8, from primitive-ai), his server watchdog and GPU shard
# prefetcher (WATCHDOG_SCRIPT_TEXT, verbatim) and his 32K draft vocabulary (embedded below, sha256-checked). All credit
# for the runtime, the overlay, the watchdog and the draft vocabulary is his.
# Changes for this notebook: our served name, port and harness context; 640-pixel board images (our
# MULTIMODAL_UPSCALE 10; he used 256); a non-blocking start (the benchmark is released at SERVER_STARTUP_TIMEOUT, as in
# v34); the runtime's patch identity is checked against the dataset's own PATCH_IDENTITY.json (the applier verifies every
# file hash) instead of one fixed overlay hash; no test-only cache-diagnostics plugin, profiler or smoke request.
# Defines functions only; the serve cell calls vllm_start_and_wait().
import base64 as _vl_b64
import hashlib as _vl_hashlib
import json as _vl_json
import math as _vl_math
import os
import shutil as _vl_shutil
import subprocess as _vl_sp
import sys as _vl_sys
import threading as _vl_threading
import time as _vl_time
import urllib.request as _vl_urlreq
import zlib as _vl_zlib
from datetime import datetime as _vl_datetime
from pathlib import Path as _VlPath

VLLM_STATE = {"engine": "vllm", "status": None, "ready": False, "error": None, "runtime_unpack_s": None,
              "launch_s": None, "ready_s": None, "overlay_sha256": None, "tested_overlay": None,
              "vllm_version": None, "kv_cache_tokens": None, "reasoning_probe": None, "reasoning_key": None,
              "watchdog_exit": None, "fallback": None, "retention_interval": None,
              "reap": None, "spec_relax": None}
_VL_TESTED_OVERLAY = "e3a6fe0d9f010bc1fb66e53e7dc6fde43272824d5200da5a49b5c4456065e0cb"
_VL_TESTED_VLLM = "0.29.1rc1.dev573+ge97573215"
# the overlay files that make prefix caching safe for this hybrid Mamba + MTP model (his fail-closed check)
_VL_PREFIX_CACHE_CRITICAL = (
    "vllm/v1/core/kv_cache_coordinator.py", "vllm/v1/core/kv_cache_utils.py",
    "vllm/v1/core/single_type_kv_cache_manager.py", "vllm/v1/worker/gpu/model_states/mamba_hybrid.py",
    "vllm/v1/worker/gpu_model_runner.py", "vllm/v1/worker/mamba_utils.py")
VLLM_HOST = "127.0.0.1"
VLLM_PORT = int(SERVED_MODEL_PORT)
VLLM_BASE_URL = f"http://{VLLM_HOST}:{VLLM_PORT}/v1"
VLLM_MAX_MODEL_LEN = 147072
VLLM_GPU_MEMORY_UTILIZATION = "0.98"
VLLM_MTP_TOKENS = 3
VLLM_CUDAGRAPH_CAPTURE_STEP = 8
VLLM_MAX_NUM_BATCHED_TOKENS = 2048
VLLM_IMAGE_MAX_PIXELS = 640 * 640          # our board image: 64 cells x MULTIMODAL_UPSCALE 10 (measured: all 640x640)
VLLM_READY_TIMEOUT_S = 3600                # his limit for a cold start (NFS reads of a 168 GiB checkpoint)
VLLM_SERVER_LOG = _VlPath(WORKING_DIR) / "serve.log"
VLLM_SERVER_PID = _VlPath(WORKING_DIR) / "vllm-openai-server.pid"
VLLM_SERVER_READY = _VlPath(WORKING_DIR) / "vllm-openai-server.ready"
VLLM_WATCHDOG_SCRIPT = _VlPath(WORKING_DIR) / "vllm-watchdog.py"
VLLM_WATCHDOG_CONFIG = _VlPath(WORKING_DIR) / "vllm-watchdog.json"
VLLM_WATCHDOG_LOG = _VlPath(WORKING_DIR) / "vllm-watchdog.log"
VLLM_SHARD_PREFETCH_REPORT = _VlPath(WORKING_DIR) / "vllm-shard-prefetch.json"
VLLM_HOST_MEMORY_LOG = _VlPath(WORKING_DIR) / "vllm-host-memory.jsonl"   # v36: his monitor writes it (test runs only)
VLLM_IMAGE_RUNTIME_ROOT = _VlPath("/tmp/vllm-image-runtime")
VLLM_IMAGE_RUNTIME_CACHE_ROOT = _VlPath("/tmp/vllm-image-runtime-cache")
VLLM_DRAFT_VOCAB_FILE = _VlPath("/tmp/arc_draft_vocab_32k.json")
_VL_DRAFT_VOCAB_SHA256 = "55176ffc7ae9545d174118477354fe660adfeed9530430a639eda1633f18f4a3"
_VL_DRAFT_VOCAB_B64 = '...'
WATCHDOG_SCRIPT_TEXT = '\'\'\'Keep the vLLM OpenAI server alive for the whole harness run.\n\nStarted by the notebook setup step with the path of a JSON config holding the server command and\nthe server log, pid and ready-marker paths. The server is spawned as a child and awaited with a\nblocking wait, so a dead server is noticed without polling. Every exit is logged with its code,\nuptime and the tail of the server log. The server is restarted after a short pause, which lets the\nengine process that the dying API server kills release the GPU and the port. It is not restarted\nwhen it died before the setup step marked it ready, so a broken launch fails the setup step, and\nnot more than MAX_RESTARTS times, so a crash that comes back deterministically ends in a readable\n"giving up" line instead of a restart loop for the rest of the run.\n\nEach server start runs the GPU shard prefetcher from that start\'s log offset, so a restart never\nmistakes the previous load\'s completed progress bar for its own. Prefetch reports are kept\nseparately for the initial start and every restart. The prefetch runs in a separate process,\nwith bounded termination waits so a blocked NFS read cannot hold up server restarts.\n\'\'\'\nimport concurrent.futures\nimport json\nimport os\nimport re\nimport signal\nimport subprocess\nimport sys\nimport threading\nimport time\nfrom contextlib import contextmanager\nfrom datetime import datetime\nfrom pathlib import Path\n\n\ndef is_test_run() -> bool:\n    \'\'\'Return true only for an explicit noncompetition test run.\'\'\'\n\n    return (\n        os.environ.get(\'TAAF_RUN_AS_SUBMISSION\', \'\').strip().lower() == \'0\'\n        and os.environ.get(\'KAGGLE_IS_COMPETITION_RERUN\', \'\').strip().lower() in {\'\', \'0\', \'false\'}\n    )\n\n\nMAX_RESTARTS = 5\nRESTART_DELAY_SECONDS = 10\nHOST_MEMORY_SAMPLE_INTERVAL_SECONDS = 15\nGPU_SHARD_PREFETCH_THREADS = 4\nGPU_SHARD_PREFETCH_WINDOW_BYTES = 8 * 1024**3\nGPU_SHARD_PREFETCH_BLOCK_BYTES = 16 * 1024**2\nGPU_SHARD_PREFETCH_POLL_SECONDS = 0.5\nGPU_SHARD_PREFETCH_STOP_TIMEOUT_SECONDS = 2\nPLE_SHARD_NAME_PREFIX = \'ple-\'\nGPU_SHARD_PREFETCH_ABORT = threading.Event()\nWATCHDOG_EXIT_DEFERRED = False\nWATCHDOG_EXIT_REQUESTED = False\n\nconfig = json.loads(Path(sys.argv[1]).read_text(encoding=\'utf-8\'))\nserver_command = config[\'command\']\nserver_log = Path(config[\'server_log\'])\nserver_pid = Path(config[\'server_pid\'])\nserver_ready = Path(config[\'server_ready\'])\n\n\ndef read_integer_file(path: str) -> int | None:\n    \'\'\'Read a cgroup counter, returning None for unreadable or nonnumeric values.\'\'\'\n    try:\n        return int(Path(path).read_text(encoding=\'ascii\').strip())\n    except (OSError, ValueError):\n        return None\n\n\ndef read_host_memory() -> dict[str, int | None]:\n    \'\'\'Report available RAM, including the cgroup limit used by vLLM\'s loader.\'\'\'\n    host_available_bytes = None\n    for line in Path(\'/proc/meminfo\').read_text(encoding=\'ascii\').splitlines():\n        if line.startswith(\'MemAvailable:\'):\n            host_available_bytes = int(line.split()[1]) * 1024\n            break\n    if host_available_bytes is None:\n        raise ValueError(\'MemAvailable is missing from /proc/meminfo\')\n\n    cgroup_limit_bytes = read_integer_file(\'/sys/fs/cgroup/memory.max\')\n    if cgroup_limit_bytes is not None:\n        cgroup_usage_bytes = read_integer_file(\'/sys/fs/cgroup/memory.current\')\n    else:\n        cgroup_limit_bytes = read_integer_file(\'/sys/fs/cgroup/memory/memory.limit_in_bytes\')\n        if cgroup_limit_bytes is not None and cgroup_limit_bytes >= 1 << 62:\n            cgroup_limit_bytes = None\n        cgroup_usage_bytes = (\n            read_integer_file(\'/sys/fs/cgroup/memory/memory.usage_in_bytes\')\n            if cgroup_limit_bytes is not None else None\n        )\n\n    cgroup_available_bytes = None\n    if cgroup_limit_bytes is not None:\n        cgroup_available_bytes = (\n            cgroup_limit_bytes\n            if cgroup_usage_bytes is None else max(0, cgroup_limit_bytes - cgroup_usage_bytes)\n        )\n    available_bytes = (\n        min(host_available_bytes, cgroup_available_bytes)\n        if cgroup_available_bytes is not None else host_available_bytes\n    )\n\n    return {\n        \'available_bytes\': available_bytes,\n        \'host_available_bytes\': host_available_bytes,\n        \'cgroup_available_bytes\': cgroup_available_bytes,\n        \'cgroup_limit_bytes\': cgroup_limit_bytes,\n        \'cgroup_usage_bytes\': cgroup_usage_bytes,\n    }\n\n\ndef log_host_memory() -> None:\n    \'\'\'Sample host RAM throughout server startup, inference, and restarts.\'\'\'\n    output_path = Path(config[\'host_memory_log\'])\n    try:\n        output_path.parent.mkdir(parents=True, exist_ok=True)\n        with output_path.open(\'w\', encoding=\'utf-8\') as output:\n            while True:\n                record = {\'ts\': round(time.time(), 3), **read_host_memory()}\n                output.write(json.dumps(record) + \'\\n\')\n                output.flush()\n                time.sleep(HOST_MEMORY_SAMPLE_INTERVAL_SECONDS)\n    except (OSError, ValueError) as error:\n        log_event(f\'host memory monitor stopped: {error!r}\')\n\n\ndef log_event(message: str) -> None:\n    print(f\'{datetime.now().isoformat(timespec="seconds")} watchdog: {message}\', flush=True)\n\n\ndef read_server_log_tail(lines: int = 40) -> str:\n    if not server_log.exists():\n        return \'\'\n\n    return \'\\n\'.join(server_log.read_text(encoding=\'utf-8\', errors=\'replace\').splitlines()[-lines:])\n\n\ndef build_natural_sort_key(path: Path) -> list:\n    \'\'\'Split a file name into text and integer parts, the key vLLM\'s safetensors loader sorts by.\'\'\'\n    return [int(part) if part.isdigit() else part for part in re.split(r\'(\\d+)\', path.name)]\n\n\ndef list_loader_shards(model_path: Path) -> list[Path]:\n    \'\'\'Return the checkpoint shards of the index in the order vLLM\'s safetensors loader reads them.\'\'\'\n    index = json.loads((model_path / \'model.safetensors.index.json\').read_text(encoding=\'utf-8\'))\n    shards = [model_path / name for name in set(index[\'weight_map\'].values())]\n\n    return sorted(shards, key=build_natural_sort_key)\n\n\ndef read_file_into_page_cache(path: Path, stop_event: threading.Event) -> dict:\n    \'\'\'Read a file in large blocks so its pages sit in the page cache, until the end or a stop.\n\n    The read stops early when its stop_event or GPU_SHARD_PREFETCH_ABORT is set. Both are checked\n    between blocks, so a stop takes effect only after an in-flight read returns. Returns the\n    monotonic start and end of the read and whether it was stopped early.\n    \'\'\'\n    started_at = time.monotonic()\n    buffer = memoryview(bytearray(GPU_SHARD_PREFETCH_BLOCK_BYTES))\n    with path.open(\'rb\', buffering=0) as handle:\n        while not (stopped := stop_event.is_set() or GPU_SHARD_PREFETCH_ABORT.is_set()) and handle.readinto(buffer):\n            pass\n\n    return {\'started_at\': started_at, \'finished_at\': time.monotonic(), \'stopped\': stopped}\n\n\ndef drop_file_from_page_cache(path: Path) -> None:\n    \'\'\'Ask the kernel to evict a file\'s clean cached pages. The kernel treats this as a hint.\n\n    An eviction that fails only leaves the pages in the cache, so it is reported and the prefetch\n    carries on with a window that is smaller in practice than it is on paper.\n    \'\'\'\n    try:\n        file_descriptor = os.open(path, os.O_RDONLY)\n        try:\n            os.posix_fadvise(file_descriptor, 0, 0, os.POSIX_FADV_DONTNEED)\n        finally:\n            os.close(file_descriptor)\n    except OSError as error:\n        print(f\'GPU shard prefetch could not evict {path.name}: {error!r}\', flush=True)\n\n\nclass LoaderProgressFollower:\n    \'\'\'Follow the completed-shard count of the target load\'s progress bar in the vLLM server log.\n\n    The bar prints `| N/<shard_count> [` after the N-th shard, one update per line. The MTP draft\'s\n    second bar has a different total, so it never matches. The unterminated tail of each read is\n    kept and completed by the next one, so a line cut by a read is still counted. The count is -1\n    until the bar\'s first line, which the loader prints once the pinned PLE table is allocated.\n    \'\'\'\n\n    def __init__(self, log_handle, shard_count: int) -> None:\n        self._log_handle = log_handle\n        self._pattern = re.compile(rf\'\\| (\\d+)/{shard_count} \\[\')\n        self._unterminated_text = \'\'\n        self.loaded_shard_count = -1\n\n    def update_loaded_shard_count(self) -> int:\n        \'\'\'Read the new log text and return the highest completed count seen so far.\'\'\'\n        complete_text, _, self._unterminated_text = (self._unterminated_text + self._log_handle.read()).rpartition(\'\\n\')\n        for match in self._pattern.finditer(complete_text):\n            self.loaded_shard_count = max(self.loaded_shard_count, int(match.group(1)))\n\n        return self.loaded_shard_count\n\n\nclass GpuShardPrefetcher:\n    \'\'\'Keep the GPU shards a bounded window ahead of vLLM\'s single-stream loader in the page cache.\n\n    The loader reads the GPU shards through one NFS stream at 44-153 MiB/s, while the PLE shards\n    already arrive at 615-923 MiB/s through its multi-threaded copy, so only the GPU shards are\n    prefetched. vLLM\'s own prefetch (v264, v265) read ahead with no bound, and after the pinned PLE\n    table the page cache holds about 20 GiB, so pages far ahead were evicted before the loader got\n    there. Nothing is read before the loader prints its first progress line, so the window never\n    competes with the PLE allocation. From then on shards are submitted in load order while the\n    bytes submitted and not yet passed by the loader fit the window, and the shard being loaded is\n    always submitted.\n\n    A shard the loader has passed gets its read stopped, and once that read has ended its pages\n    are evicted. Evicting while the read still runs would let it refill the cache behind the loader.\n    Pages read twice (the prefetch, then the loader) sit on the active LRU list and would\n    otherwise push the window\'s unread pages out first.\n\n    The loop returns once every GPU shard is passed and evicted, and writes a JSON report with the\n    per-shard prefetch and loader times, relative to the loader\'s first progress line. If the bar\n    never appears, nothing is read and no report is written. Shards the loader crossed inside one\n    poll carry the timestamp of the next printed update, so complete_before_loader_reached is an\n    upper bound and the report counts them. The log offset isolates each server start\'s progress.\n    \'\'\'\n\n    def __init__(self, shards: list[Path], server_log: Path, report_path: Path, log_offset: int = 0) -> None:\n        self._shards = shards\n        self._server_log = server_log\n        self._report_path = report_path\n        self._log_offset = log_offset\n        self._gpu_shard_indices = [\n            index for index, shard in enumerate(shards) if not shard.name.startswith(PLE_SHARD_NAME_PREFIX)\n        ]\n        self._shard_sizes = {index: shards[index].stat().st_size for index in self._gpu_shard_indices}\n        self._futures: dict[int, concurrent.futures.Future] = {}\n        self._stop_events: dict[int, threading.Event] = {}\n        self._progress_reached_at: dict[int, float] = {}\n        self._submitted_position = 0\n        self._passed_position = 0\n        self._dropped_position = 0\n\n    def run(self) -> None:\n        \'\'\'Follow the loader until every GPU shard is passed and evicted, then write the report.\n\n        GPU_SHARD_PREFETCH_ABORT ends the loop as well as the reads, and a pass that ends early\n        writes no report.\n        \'\'\'\n        while not self._server_log.exists():\n            if GPU_SHARD_PREFETCH_ABORT.is_set():\n                return\n            time.sleep(GPU_SHARD_PREFETCH_POLL_SECONDS)\n        with (\n            self._server_log.open(encoding=\'utf-8\', errors=\'replace\') as log_handle,\n            concurrent.futures.ThreadPoolExecutor(\n                max_workers=GPU_SHARD_PREFETCH_THREADS, thread_name_prefix=\'gpu-shard-prefetch\'\n            ) as executor,\n        ):\n            log_handle.seek(self._log_offset)\n            follower = LoaderProgressFollower(log_handle, len(self._shards))\n            while self._dropped_position < len(self._gpu_shard_indices) and not GPU_SHARD_PREFETCH_ABORT.is_set():\n                loaded_shard_count = follower.update_loaded_shard_count()\n                if loaded_shard_count >= 0:\n                    self._record_progress(loaded_shard_count)\n                    self._stop_passed_shards(loaded_shard_count)\n                    self._drop_passed_shards()\n                    self._submit_window(executor)\n                time.sleep(GPU_SHARD_PREFETCH_POLL_SECONDS)\n        if self._dropped_position == len(self._gpu_shard_indices) and self._progress_reached_at:\n            self._write_report()\n\n    def _record_progress(self, loaded_shard_count: int) -> None:\n        \'\'\'Stamp the first time each completed count was seen, including counts a poll skipped.\n\n        tqdm drops the updates that fall inside its minimum interval, so counts it never printed\n        carry the timestamp of the next printed one, which is later than the real crossing. Those\n        shards show equal reached and passed times in the report.\n        \'\'\'\n        now = time.monotonic()\n        for count in range(len(self._progress_reached_at), loaded_shard_count + 1):\n            self._progress_reached_at[count] = now\n\n    def _stop_passed_shards(self, loaded_shard_count: int) -> None:\n        \'\'\'Stop the reads of GPU shards the loader has finished and cancel those still queued.\'\'\'\n        while (\n            self._passed_position < len(self._gpu_shard_indices)\n            and self._gpu_shard_indices[self._passed_position] < loaded_shard_count\n        ):\n            shard_index = self._gpu_shard_indices[self._passed_position]\n            if shard_index in self._futures:\n                self._stop_events[shard_index].set()\n                self._futures[shard_index].cancel()\n            self._passed_position += 1\n\n    def _drop_passed_shards(self) -> None:\n        \'\'\'Evict passed GPU shards in order, each only after its read has ended.\'\'\'\n        while self._dropped_position < self._passed_position:\n            shard_index = self._gpu_shard_indices[self._dropped_position]\n            if shard_index in self._futures and not self._futures[shard_index].done():\n                return\n            drop_file_from_page_cache(self._shards[shard_index])\n            self._dropped_position += 1\n\n    def _submit_window(self, executor: concurrent.futures.ThreadPoolExecutor) -> None:\n        \'\'\'Submit the next GPU shards while the bytes still held in the page cache fit the window.\n\n        The window counts from the first shard not yet evicted, not from the loader, so shards the\n        loader has passed while their read winds down keep their bytes in the budget.\n        \'\'\'\n        self._submitted_position = max(self._submitted_position, self._passed_position)\n        ahead_bytes = sum(\n            self._shard_sizes[index]\n            for index in self._gpu_shard_indices[self._dropped_position:self._submitted_position]\n        )\n        while self._submitted_position < len(self._gpu_shard_indices) and (\n            self._submitted_position == self._passed_position\n            or ahead_bytes + self._shard_sizes[self._gpu_shard_indices[self._submitted_position]]\n            <= GPU_SHARD_PREFETCH_WINDOW_BYTES\n        ):\n            shard_index = self._gpu_shard_indices[self._submitted_position]\n            self._stop_events[shard_index] = threading.Event()\n            self._futures[shard_index] = executor.submit(\n                read_file_into_page_cache, self._shards[shard_index], self._stop_events[shard_index]\n            )\n            ahead_bytes += self._shard_sizes[shard_index]\n            self._submitted_position += 1\n\n    def _describe_shard(self, shard_index: int, loader_started_at: float) -> dict:\n        \'\'\'Return one shard\'s report row, with times in seconds after the loader\'s first progress line.\'\'\'\n        row = {\n            \'name\': self._shards[shard_index].name,\n            \'gib\': round(self._shard_sizes[shard_index] / 1024**3, 3),\n            \'loader_reached\': round(self._progress_reached_at[shard_index] - loader_started_at, 1),\n            \'loader_passed\': round(self._progress_reached_at[shard_index + 1] - loader_started_at, 1),\n        }\n        future = self._futures.get(shard_index)\n        if future is None or future.cancelled():\n            row[\'prefetch\'] = \'not started\'\n        elif future.exception() is not None:\n            row[\'prefetch\'] = f\'failed: {future.exception()!r}\'\n        else:\n            read = future.result()\n            row[\'prefetch\'] = \'stopped\' if read[\'stopped\'] else \'complete\'\n            row[\'prefetch_started\'] = round(read[\'started_at\'] - loader_started_at, 1)\n            row[\'prefetch_finished\'] = round(read[\'finished_at\'] - loader_started_at, 1)\n\n        return row\n\n    def _write_report(self) -> None:\n        \'\'\'Write the window settings and one row per GPU shard to the report file.\'\'\'\n        loader_started_at = self._progress_reached_at[0]\n        rows = [self._describe_shard(index, loader_started_at) for index in self._gpu_shard_indices]\n        report = {\n            \'threads\': GPU_SHARD_PREFETCH_THREADS,\n            \'window_gib\': GPU_SHARD_PREFETCH_WINDOW_BYTES / 1024**3,\n            \'gpu_shards\': len(self._gpu_shard_indices),\n            \'gpu_shard_gib\': round(sum(self._shard_sizes.values()) / 1024**3, 2),\n            \'other_shards\': len(self._shards) - len(self._gpu_shard_indices),\n            \'complete_before_loader_reached\': sum(\n                row[\'prefetch\'] == \'complete\' and row[\'prefetch_finished\'] <= row[\'loader_reached\'] for row in rows\n            ),\n            \'shards_crossed_within_one_poll\': sum(row[\'loader_reached\'] == row[\'loader_passed\'] for row in rows),\n            \'shards\': rows,\n        }\n        self._report_path.write_text(json.dumps(report, indent=1), encoding=\'utf-8\')\n\n\ndef prefetch_gpu_shards(log_offset: int, report_path: Path) -> None:\n    \'\'\'Build the GPU shard prefetcher for the mounted model and run it.\n\n    It runs in a separate process, so a failure while reading the checkpoint index\n    only ends the prefetch and leaves the server launch alone.\n    \'\'\'\n    GpuShardPrefetcher(\n        list_loader_shards(Path(config[\'model_path\'])), server_log, report_path, log_offset\n    ).run()\n\n\ndef start_gpu_shard_prefetch(log_offset: int, report_path: Path) -> subprocess.Popen | None:\n    \'\'\'Launch an isolated prefetch process, leaving the server running if spawning fails.\'\'\'\n    try:\n\n        return subprocess.Popen([\n            sys.executable, __file__, sys.argv[1], \'--prefetch\', str(log_offset), str(report_path)\n        ])\n    except OSError as error:\n        log_event(f\'could not start GPU shard prefetch: {error!r}\')\n\n        return None\n\n\ndef stop_gpu_shard_prefetch(prefetch_process: subprocess.Popen) -> None:\n    \'\'\'Terminate prefetch without waiting indefinitely for a blocked filesystem read.\'\'\'\n    if prefetch_process.poll() is not None:\n\n        return\n    prefetch_process.terminate()\n    try:\n        prefetch_process.wait(timeout=GPU_SHARD_PREFETCH_STOP_TIMEOUT_SECONDS)\n\n        return\n    except subprocess.TimeoutExpired:\n        log_event(f\'GPU shard prefetch did not stop within {GPU_SHARD_PREFETCH_STOP_TIMEOUT_SECONDS}s; sending SIGKILL\')\n    prefetch_process.kill()\n    try:\n        prefetch_process.wait(timeout=GPU_SHARD_PREFETCH_STOP_TIMEOUT_SECONDS)\n    except subprocess.TimeoutExpired:\n        log_event(\'GPU shard prefetch is still blocked after SIGKILL; continuing without waiting for it\')\n\n\n@contextmanager\ndef defer_watchdog_exit():\n    \'\'\'Finish registering or stopping prefetch before honoring a watchdog exit request.\'\'\'\n    global WATCHDOG_EXIT_DEFERRED\n    WATCHDOG_EXIT_DEFERRED = True\n    try:\n        yield\n    finally:\n        WATCHDOG_EXIT_DEFERRED = False\n        if WATCHDOG_EXIT_REQUESTED:\n            raise SystemExit(0)\n\n\ndef exit_watchdog(signal_number: int, frame) -> None:\n    \'\'\'Request shutdown once, deferring it while prefetch ownership changes or cleanup runs.\'\'\'\n    global WATCHDOG_EXIT_REQUESTED\n    if WATCHDOG_EXIT_REQUESTED:\n\n        return\n    WATCHDOG_EXIT_REQUESTED = True\n    if not WATCHDOG_EXIT_DEFERRED:\n        raise SystemExit(0)\n\n\ndef run_watchdog() -> None:\n    \'\'\'Spawn the server, wait for it to exit, log the exit and start it again while that is allowed.\'\'\'\n    restart_count = 0\n    prefetch_process = None\n    while True:\n        with server_log.open(\'a\' if restart_count else \'w\', encoding=\'utf-8\') as server_log_handle:\n            if restart_count:\n                server_log_handle.write(\n                    f\'\\n==== watchdog restart {restart_count} at {datetime.now().isoformat(timespec="seconds")} ====\\n\'\n                )\n                server_log_handle.flush()\n            log_offset = server_log_handle.tell()\n            started_at = time.monotonic()\n            server_process = subprocess.Popen(\n                server_command, stdout=server_log_handle, stderr=subprocess.STDOUT, text=True\n            )\n            server_pid.write_text(str(server_process.pid), encoding=\'utf-8\')\n            log_event(f\'vLLM server started (pid {server_process.pid}, restart {restart_count})\')\n            report_path = Path(config[\'shard_prefetch_report\'])\n            if restart_count:\n                report_path = report_path.with_name(f\'{report_path.stem}-restart-{restart_count}{report_path.suffix}\')\n            try:\n                with defer_watchdog_exit():\n                    if prefetch_process is None or prefetch_process.poll() is not None:\n                        prefetch_process = start_gpu_shard_prefetch(log_offset, report_path)\n                    else:\n                        log_event(\'previous GPU shard prefetch is still exiting; skipping another prefetch\')\n                exit_code = server_process.wait()\n                log_event(f\'vLLM server exited with code {exit_code} after {time.monotonic() - started_at:.0f}s\')\n            finally:\n                with defer_watchdog_exit():\n                    if prefetch_process is not None:\n                        stop_gpu_shard_prefetch(prefetch_process)\n        log_event(\'last server log lines:\\n\' + read_server_log_tail())\n        if not server_ready.exists():\n            log_event(\'server died before it was ready; not restarting\')\n            sys.exit(1)\n        if restart_count >= MAX_RESTARTS:\n            log_event(f\'giving up after {MAX_RESTARTS} restarts; the server stays down\')\n            sys.exit(1)\n        restart_count += 1\n        log_event(f\'restarting vLLM server in {RESTART_DELAY_SECONDS}s (restart {restart_count} of {MAX_RESTARTS})\')\n        time.sleep(RESTART_DELAY_SECONDS)\n\n\nif len(sys.argv) > 2 and sys.argv[2] == \'--prefetch\':\n    prefetch_gpu_shards(int(sys.argv[3]), Path(sys.argv[4]))\n    sys.exit(0)\nsignal.signal(signal.SIGTERM, exit_watchdog)\nif is_test_run():\n    threading.Thread(target=log_host_memory, name=\'host-memory-monitor\', daemon=True).start()\nrun_watchdog()\n'
_VL_GPU_NAME_PATTERNS = {"rtx-pro-6000": ("rtx pro 6000",), "h100": ("h100",), "l4": ("l4",)}


# ---------------- v42: REAP-at-load and relaxed MTP acceptance (switches in the config cell) ----------------
# REAP masks: per MoE layer a 512-bit mask (hex) of the routed experts each public REAP build keeps, and the sha256 of
# every router of the checkpoint we serve (primitive-ai mixed NVFP4/FP8 = lordhansolo's Kaggle model). Both builds' routers
# are bit-exact rows of these routers, which is how the kept ids were recovered and checked (448: lee-chang-93
# Qwen3.8-Flash-Next-NVFP4-REAP-k448, REAP saliency over 16.49M tokens of agentic traffic incl. images; 384: sh0wie
# Qwen3.8-Flash-Next-REAP-384-bf16, agentic-coding calibration).
_VL_REAP_MASKS = '{"router_sha256":["966e4d1ca7c18994a9311e82f0074584fe09e65e81eb8ddefe0a4dceebfc6ac6","0721144c6eed2441e5cee395d2aa2700d8f9365f0b05d7dbd553003ffc4caab6","1a16dafde1a408268028e6f3118b7d3d31d2000dc9a162da7f7f4f4a7cd131d3","b3f123bb49874bade319d7e32a769fd1c32acadce05b07410257b895e327c1a0","bd0bee66e0a620accfed0ab417e48faf26a3a14ce71c33a09bc0b03f9b877cb6","c90180c94e3a3c7355d7464f463038d07e9b14caa73ee74e4d35071748ffd4c8","7de6e13d2b6f680fad0fe76018bb9c1b04b66f8d304079b52eb5a38336854e6f","c26d58cf31e907cd0506e3d5ffe56023df1a5dd84191457b215a31afc638a653","dc704ea74fb6f27b27eb9b673f89d9a8944354efa2c60745603e60c34db0bee2","8556487138975295619957d9ece22b042e4b8790810edac60c8e08eae4f59f4a","e323ad1226a0b71111bd2a0c60a8a8056ec3acc8736b983ccf6f6db874cd69e0","8f41100e6b565bac4e094095cf35e4243800d9da972b55f8b6fcf9a4bb047fe3","dfcf0966365ab4116a103b609d21ff7fa167f6326b7c0fd9db754c5ee5d6645a","bcff93fea0edcb06dbb482c32abfb60b4cd3343a50be797d5799368b3bbb44a4","02de4640bf7ff452130884cdb8399dee67cf35c299457fc7d8d45b457ea56719","37395c5c6e31be9460d0865753c64fbe704e899428d56df15ede1ae722d8f992","b575959cd3ca396faaf51885ae2e51f78b73222befc8bb7c579057bfa03d9e03","aa818cd313a84e3b18553eab7c4a11bf957649adce253c59679e7ce477bd241c","9b574cee838a52b1b9a05232926fbe3bda382b892f1ed06111c25323b4b58d93","941f42d680d22173cd26acc7a31c1bb1bb815405484225a5bc95187be1742bcf","ac293c17daa82ab23e7563213249cd0f938befc2879eb6f9c79bc95da73a7ef9","c6d2a815a4845abf5471d767432496a30fd1c9a2cfb91effb2b42ca17bf3c9f9","f192d5d220b7c36a03b06944f096f00f3fed2391b757f78beff4993a3f0b22da","d2b556ad8a96ec7850eb664dd168ca6c5f39689dd5590f0620cfe9671008bf97","2d47ad8729068dd4916c1053346c59cbaf3afcd1992a9b224b36623f4442397c","4580b6ef10fc5d128eceae2ba14934c9c5465e10120ef825c5ad67e9bfb72534","f1546d4f3a72b76a9cb8822f43ef056d532633c7f14b00ea0fd94256f7538286","6e7184b20326f9e6fbfe088c9c3d3665ffb6d36973b7128629bb3028245179c6","98d02026a77e26889cce6aa134a6619fd5ee65beb1caa3d37e9b607ff3962d54","206df6ab22454a9cc438297f39dae52a1568f82f500447103534efea39748a0c","0044c151393108bed930b05bf652effd7c4d50aa90058c5dee81423e2c5c543c","50da9380f9347a900b790976a4fbad72d3a70f7fb0537445eaaa44e397bfff47","18bb280af63184a3c228b55fec353e33ae756866d269633746e902ebdd7a4f06","101eca6571f54ecd14c7de091e1bd9dfcefb45c945e889f819954f62ef6d6a86","d3cbba9e46030673c68e40422876895f971da54e57ea58048c79299b03bc74ce","b57c3f60056e17617a5d010ae2991a80b7c3a66a088507f79bb121641cf5ee89","b7856db65a890f658bd2d530567d58b8819728fa3db6d1697c05f335cace7076","22e8bf4b247cdcc128ea397bb4f58e03a8014ded2eb7562fd620813d82df3748","fd16f13a006c3f02ebaf5cdccd495267cbf3500578f6207c1972603381cb4208","bd80a966af8a1a315186854a0f10e165a9d39e96a14261648b3bfeb59186552e","3f53a93d8113ef0577b2b358dc8e192164bd4be53cb08d3583a76bc4f7ab3b7f","0bdacc8e5cca10c4643d75bfa40289d24118b29a6c5048346809d77269973826","6926fc5258647eddce8aa9a22b078c8cc27b3a65520e6dca5f0d0ca9c5f823aa","725d8b7b5cb7d51d5a4fefdbc52f9c8191f4fa94ac99f2f24998dd6e33ab0211","30564b155c430051eae623197867bf048e51dce0e9ce00b44eca8db9754c5fa8","144f087ba93ccbd5140588324ad1e104f842bbc521707330bfa222e5999667b2","7a037ec32f53119f1d91f3172ee2030872ca27cb63900bac8c98e44c2c5f0ce2","a074847671d9ef0699a5a542bbd4c5a76046543e76fdb499d62bd7be1fd08c50"],"masks":{"448":["bffffb7ff2dfffffff9fffffdd5bdf6fbf1fffd7fd7effcffff7effebd6f7f7d6bcfefffdbffefffffddfbffbeffdf7fefd7fffefffbdffffefffffffdbffff7","5f7fefdfbfff7defffffedfff7ffffffffff7cfffeffc3f7bf7ff7ecfedeffefdfff6fffdeff5fbf67fffb7fffafeffffdefbfeafe7df7fffffdfefff7bfdd7f","97fffeffffefff7ff9fffeefffff9fdbdefefdfeffeffcffbfcffffffdbffb9f7ff7f7f5faffdfffdffff77f6fff7ffdbfa71ff3bff9f7ffbffeffbf7effffbf","ffffffffbcef2fffffffffdfbd5fffeefff7ddbff7fffffa1ff7fff7ddffffef7fffabff7ffb7ff2feadfffdd7fff5ffddfffff7fff5e3fffefefbdf7ddfffdf","ffabffffefebefff67fffffb77fff77d7fff5fe4feff6ffb5ff8dff7ffff9b9ffdfff3d7efffdffffffddbffef7abfdffffef7ff3dffbffffff7fefffffffffd","fff7fcff7ffbffdfffddffffbffdfdfeffefbfeefefeef77fffffffffbee7d7ebeefffbfffeb7cfffc7dfebbdfffffff7bebeffdbdfffffdff5afefffdbdfaff","ffffbefffdfffffdf3ffb8fffbdfffdffbdbfb6ffbbdfffefd77fffbffff6bfffffffeeffbdfe6ffffffdfd677ffdfdbffffb7fbfcf7affffff7edbeffff37fa","bffff7e7bfefe3fffdf3ffffffdeffffefdbebefedfcfffeedffcfdffe7f3bffff77bffffffffdef7ffbfff7fd5bffffeffffeeffe7f5fdffdfbfbffb8ffff7d","f6dfeffdffffffffefb7ff9fbcfedf4dbfcfebffdf9ffffdff7ff5d7feeff777fffffb7bffffffeefffffa7ffdffffefffbaffbfbfe7fb7ffdffffffddbb6fff","9f7effff7fddbf9efffffffffdbfffffffffdbffbf7fffbff27dfbfbf9fe7fffd7fffffbfd7ffcdffffb3fff7f6ffbfff6dfbbefeffcf7fcffeffffffc9fdff5","7fefffff77fbf4d7ff3ff7ffafff7ffff6ffee5ffeb7fdff7fffeffffd7ffeff7fffffbff7fbe7fcbbfffd7f7fdfff7dafffffff3fbeefbfbbfff6edfdff7f7f","eeff7dffcfffdadebdbf3fffeffffffffff723ffffffffbef7bfffdbffeff6dfff9fabe7f2effffbbbffffffff7feffffffff7ffebfefffffebffbff1757feff","fffff7fddffbffffffef7fdf7f7bdfefdffeffdbfedffff9effffb7ffdf77eaffbcffef7bfaefff5fbbffdffefffffcebff3fcdfffffffdffa7df7e77fff7bff","dfbfbefeffaeeadfeff9befbfefefdfb7ffffffffd6bffbffffffa7ddbfe77b6fffbeffffffbeffedffffffbfffb7fffff7bffffbffff7ebff9dfdf7efefff5f","bfdf7afb97fffdfffffd3ffffb7a7dfffbffffbfbbf37f79fcffffffbfff7fdd6ffffefffcffffffffeefed7dffffb7dfbffebeffafdfffffffdfefff7fff56b","7ffffff7baffefeff7ffbffbd7ddfffffefcfff7fdf7aaff8ffffbfddfedffe9d7fefbfffdbffffffbf7f7ffffff1deffffeffdc77fff7fffdfddfffd6ffbbff","9bfbffdfdefedb7fdfdf7f7effdffdaffdbffffefffeff7ffef59dffffdfdeddbeffdffffbf7ffffffffffffefbfddbffbdffcffffdfbef7e7fffd9ebeff5fff","ffffdff569ffffff5fbbfefdeefffdefffdfb7fefe9f7fbfbfe7fff3ffffdffbfbfdfdffffefffffddf2ff3f96fff7f6bffff7ffffefdfffeffebffefeefddfb","efff6dffebffffafffffbbffcbbffedfff67ffdfbdbffff7ff7fdf7bfdffbffffb5ffffffffffff353fd7ffdbff5ff7ff7febfb7fdbf9fffffefde7ef57fbfff","fffcffffa7fbf775fdefbdfffbe7dbeff7fffeff7defdfffffffffff77feebfefeff77fad7ffeffbebfbf7ff7f7ff5fffdffcffeff57ffbfffaffff7bffdefdf","ffeffff3ef3fffbf7fff5fbf77febfffef67ffdffffffbfffef3b7ff3b6f46ff3afff9ffbfffdffffffffdd5ffffff3ffeff3fdfffeddef7fffffff6ffafefff","cfcfcf7fffff7fffff5fb7fefabfdfbfbf7eddfdcb1ffefdffefffffffffff7fbfffb67ffdffffddff8dfffdffffedde4ffbffff7fdfffff6ffffffbffef9dff","effdfeb6effff3f7ff7ff79fefffb76ffffbcefe7fff57f7f7fffff7bfbe6f2ffedfefebfffeffdffefefefff7f7fdfd7f7bff7ffeffffff3fffffdbfdfffffe","fdff777ffffff7ffeffeffffdffff7ffff15eff7bfff5fecffbffffff7fdffdbedfdfffffaeff7f7fef9dfffbfaf7efbf7ffffffd6dfffffdffd3abfef5d9ffd","f7fffff5efffefffefb7ff9f3efefffca79feb7effdfffffff7ff5fdffbbf7fffefff77bbfeeffefffffff7fedffddeffebbfbbfbfe7fb7ffeffffffdfbe7efb","b7fffbbf5ffdffdfeffffff77fbbfeffff7fdbfbff5fbfbffe79f9bbf1ff7fffd6fffffbfd7ffedffffb3fff7f6ffbffe7dfbfdfeff7f7ffdfeffffffcbefffd","3deafffd7f7b3bf5ff7b7fffefff7fffffffeffffebdfdff7fffff7f7dfff79fffffffdffffbe7fcbffffdff7baeffef7fafffdffbbee7bffbeffcfff7df9fff","eefffeffdfffdedb7de73f7ff7ffeffffff32bffffffffbeffdbffdfffbff4d7ffffbfe7ffeefebbffffdffffb7dffebff7ff7ffdaeffffffdfff8de9f5fffff","fffffff5dbf9fffbffff6fbffffbfffddefeeffbeffffffd6f9efbdfffff3feeffcffff3bfeaf77eff2ff7bfffffffc4ff7bfdfffffffecbf3fdf7f7fffffbff","dfbbbffeefcffbffeffbfefbf4fefffbfffffffbff6bfb7fd7fffa3ddffffebefbdbeffff7fbfefefffffbdbffbb7ffffffbffffffbfffe3ff9dfdf7efe7f77f","bfdf7afaffffedefdfffbffbfb7f7ffffbfff7bdfbf377ffdef7fcfffbff79f56fdffefeeeffffffffeefed7dfeeff7dfbffefeffffffffff7fdfeffdffffd6b","fffbffd7defffbeee7f7fffffd7dbff5f9fffb7bfcffd7ffcbfffbffffef9ff5f7f77ebdfffffaffef2ffe6ffbff7ffff7ffd67d5fffff7ff7fffffffffffbdf","dfebffdffafffbff5fd7ff7bfdffffe7ffb72fffdffeffffffd59efffdfdffddfefffffffff7bbfffffdffffffbfddffff59f5ffdfbfbec7f7ffbd8fbeff5fff","f7dffff7bdfffff75f77fefdfefffdefffdff7fcfedfefbfbfefffebfffbfffffaff7ddbffcff6ffffb7ff7dd6fff779fffbffefefffdfffefffbffececf5ffd","fff775ff6fffff3fffff9fefdfffffdffff77feffdb77ff7dd5fdffdfdffbf3fbb7ffffffffdd9fb7bfffffdffddfef9e7feffffffbf9ffedfeffef677febfef","fff5fbfdbfbfff75ff7fbdaeffeff9ffff7bf7f67befffdffffd7fdff7bebffff7ff57daf7fffff3ffe7fbefffbffffffeff6ffb7f57ffbfffbff5fffffdffeb","fdefefebecbfff6f77fffee7fbfabeffbf77ffdfbffff9ffeed7dfeff8fd8fd7fffbffffbffffffffffffdfffdbfff7fff6d7fffff6effb7bffffff6fffdeffd","dfedf33fffffffffffdffffefabfddbfbf3ffdfdfbdfffbf57bfeffff7f9dfbaffedbb3fffffffdd9faff7fdffff7dde17fefff7fffffdffefff7ffbffffdfff","efffeebeaffff776ff7fffbfefffffffbbfb7fff7fbfd7fffbfffffefffbff3ffe5fffffe5d67ffffefffe7de7f6f7fdffcfff7bbcfeffbfdb7fefbff7fffdfe","bdffe77fff7fff7efffcff3fffbffe7f7fbdeffebffdeedfbffdafff7ffffdffefffffefdf77efdfffffdfffdebfbbffbfedfbfffedecffffffdffdfcf7d1fdd","fbffffeffefffdbffbbffff70fdffffbffb7ffbffff9feeffdfff7ffafddffffffddffdfdfdfb9fbfd7fff6efff7ffbdcffefebffff7bfdfbffffffbf7bf13d9","ffff7fffffdfffff7fefffffdffdffef7f8fbff7dbf7fffbdfedfffbfb7f3ffdfdfffe77fffff7d77dffdffdffafdff7f5ffbddfffdfbebf6daffffbffebd52f","fd7fffffffefcfe7b7ffffffbf1ebffffdfffffffffdf7adaf6bfffcfbfb7fbfdf7effffd7fdfd657fdffffdf79fff7dffdfffdbffffeffb7ff7fffecfffdf7e","ff3fcf7effeffffff6fedffbfdffefbfff7fffddff7fffefef57fbfdf7ffdfbffcffdf7dbabfcecf7effbe9f5bffffefffef7ffffff3fffffff7ffe7f7efb7ff","fefdfffbffffdbfffffb9fffaff7feeffffffadbfffffbfffffeff7ffffff7ff3fff6ff7f77fdfdffe7ffabf7afbfbfbdffc7fffdffbff7f4fcfbfbffff6c85f","f7f7fff7efff5ffffdee7ffefffbbdefedffeceffefdf7ff5ffffbefffedeffdfeb7ffeffeffbfdebffffbffd7fff7fefedffffe67bdff77ff7ffff7f7c6fffd","7ffffe7b27fffffd7ebfffbffffc6fffbd7ffff7fefffbfdfdd7dfffdffdfffffffef7ff7fffd5ffff7fdeff7dffefaf7f9ffff7ffeffbff3b79fffcf3f7ff6e","fffffff3ffefffff7fcfffffffdfffcfff577ffdeffff3ffffffbebf7ffffbff65ff7ff7ff3efff5dfff9ffeb7ffe7f9ffeb7adfffb5ffe77bfffff2fdb7dfbf"],"384":["3fbffa3fd2d7ffaff79f5ecfd95bdb67bf3f97d7fc7eff47bb33ef6ea96f7b6d7bcfe75fd9fbeff57dddf3ffbfebdd7f6bd67efeffbb9fffe67774fff9bbd7b7","553f6ede3effe5fffb76ecb6e6ffcefdf5bd5cdbfe5fd373ff7377e8fe96bffeefff67ffd8fe5ebb65d9bb7ffbefe6ff7dafb4eafe75f5dfffbdf4ff63bffd7f","f7fbfcfbffefbf7b79ffccaff7fe9fdbdebefbecfc6bf83ffbcff3ffcdb2fb9f7f35fdf47affc77bd7fea75f6ffe7ee0bc276ff3bf78b7f79fceffbb7e7737b5","ff97ff3ed6692adfff37df4dbd5dfd8ff9fd9c7fb7dbfdfc2776f727cdcfbbec7fffabfffbf53fb9feedff3e75df7c5f9d9f7fff67e7f3ff76faa3dd7f9fffdf","ff29f3f7cd8f24ff659bff72f53fd377cbbe1fe2ee7d6bbb5ff8fdf6febb9fb77fbff69fede7cffdfdacdfffcb78bdff6ffed77f3dfea6bffdf6fcfdbffdffef","eee7fdff7adf7fd7f418ff3bbf2db7fedee795fcfefc2d557ddffbfef9ee7c67fcafbeb55eff7cbbfc3c7efbdf2edffe79cbeff5f5bfdefbff56f7fff77ebafe","ebaf9e779df7aad9927cbcefd9deefdfeb5b7f6ffa9dbbfee577efdffaf18bbdffbff2fdfff7ef6effb9dfdedfbf7fdbffdf97db9e9ab79f3ff7edeacff135ff","feff77a7ae67a35f35f1e7c7ffdeffefc6d3eb2ff1fc5cfdfdbfe5ddee7fbbfffc37adbfacd7d5ef7e79fff7fd5bffdffffffaeda97f1bd3f4ffdbef98f7febd","7a9ffff438f6771befaaff9d2e7edf6cbfefe9bfdb4fdfffffdf6dd6dfebdff57dfeb163bff5ffe4ef7eeeffecdef4ecfbbafbacbdeff37ffcff777fdd3b6aff","977ff9ff6f85bb8eafdfbd3d7fb9cffef33fdbbfbd5bbfa9b26cf9eb751c7fffd2ffb7bffb6f7ccfbfd99fff6f5f7bfff6bfb3f2677df7fcddaffffffd9f5fd5","3ceb5fad2e3bf4c7ff33dfe5e6df7eebc6ffee6ffe91bcffffdfb7ff6ddfe29d7aff3f6cf6fbfdf4b9ffbcff671fff43ebafeffebb9eefff39fff5fffd775fdf","afcd3d7fcefffc5a4d673acf67daef7fffdd275fcfffdbbfedd76fddb9efe6c7dedcafe7facdf939bbffdf8fffdfeeaf7f6fe7fbdbeeffd5fca7faffb9f7fabf","feeef7b0dbf0ffbffcc7751e7f5f9f679ed6efbbeed5defdef1efb77d7662f267b4ffffdbfeeff75fb7fffa7aff7fdefa9b2fcbffcdffc8ff87deff73fffbbfb","ccbb9cfaefedcab2eff93ebbe6fefdfb7fff7ffbbd4bfb7affef9a7ddaf67ebe7bfbafebc8fbad3d5f7eebd9ffb37fefdfebdef770fff7abff9dfd77bffb7657","3fceaaf257ffadeddef9fefdcb725dcbeb77e0bdb3f37779fdcffbfd9b7e7fddeff2deec2cbfffbfdfaafad7df4fd63dfdfd6bcffbfdffffffedfeeffdfef17b","f97fefb7ba7c8cb74e9adcdd576c79f1febffe733db6aaffacb7dff7cb6bffe91df6ffffff2ed7ebffff92ffeefd9cefef9def5ef37ffffffdeefebfd7eebdef","8bf9b25ddfeedbfd9fd33d4cdbfffeae3fa731eebf36fd7d73e59dffffff8e94d7ffdefffba7dfdfbbf5fb7ee3bfdceffbfbf47fff9faff7e6fb3d8ffeff5f75","d69beff720fefafb5ff3e6dc7eff7dffedd8b7b8fcd5ebbfafe7fbe1ffdbdfbafcfdfd99ebcff6efdcf2f73d9cdff779bffbf7e3ffffd7ffffeebefece674dcd","67bb60ff6afdfb2f9ffd9bdf6beffcfdfd277fd78cb71fb2f57fdf7cfdffbf77bb5fef3977ffdff343ff77f5bfc5fe75a5eeffffffff9f7ddfeffc76355fb6e5","7fc5afdf87b3e7f579ff35e7dba7bbefe7faf63279feff873fffffdff7eeb3fefced4653d33ebbd3ffe2f6efff7bf5fedc67ddfc3b6ffefeffbffbff9f7fee5b","bfe66fea6cf74faf27b77cd7f6ffbedeea67c7debf3dfffaeed6572e727fafb7bbdabbfeffffffcf7df7fd9fb49e7f3fbfad77cfff6e7fb7bfefbef43feeeff5","45e9403bf4ff7ffec75f0f6e7fbfd7f5bf3e45f5ce17fe8d33dfeff7ffebf5fbfdedb23ff5f3f3f5beb4bbffdfff1ddfcffaaff7fffbfdfd5f6fffdbfffe9fcf","adbcefb683fef277ff7df7dfed7fb6ffbffbec7a7e9c55f7f3fffffcbffbeb2bfe5eeb63e59c7adf7eeefc7df7f6f5f47f5bff79bf6e7eaf3bdfefdb71f7ff7e","be3fe7777d3bee3fe9cdff1f94df6f7ffbfdcfffdffd4aea7ebdebdf7ebddfdbadf5ffdfd8d327d7f7f9ecdfde3fbbf9bbedcffbdafeeffedfb56bffef7d16d9","7a9f7ff428fee78befabff9e367c9f6ce7efe9bedacfdfffff9fbdd69fe9fff63cfef36bbff5ffe5ffeffeffec8ed4ecd8fafbbdbfdffb7f7eff7fffdd3a6aff","977f79ff7f859bceafdfbd3f77f9cfffe77f5bbfbd1abfadb36bf9ebf57e7fdfc2ffb77ffb6f738fbfdd9fdf6f57fbefe6cff5e36f75f3fcdda7ffff7d9fff55","3de67badec3b62e5ff335fb3efcff2ffc6efef2fbe95fff7befef77f6dffe29ffefdff4ed6f9fdfc3bffbcbf670fff69ebafeffebbdee7ffbbffddebfdf717de","bfeddc7f86fabc9a4d757f4ff7daeb7fffff275fcfffdbbfcff72ffdb9a6f4d7fedcafe7faeef9bcbbefdf6effddeeaeff6fe2bfdbfefff7cca77adf7bb7faff","feeff3b153f9ffbbfdd76d7f7f5bbbe59ede6fbbeed5dcfd6fbffa57f7762fa67b4f7ffbb9eef579fe77f7efbff3fdedcd32f8bffbdd7ecfda7deff77de77fff","cd9a9cfee3cdfef2efd93ebbe4fafdeb7ef7fffbbd5afb7ad5af927ddbfe7ebe7ffbefefccfb687eff7fdbdbffb37feedfeb5ef36efff7efff9dfd77efdb765e","3f4e2af277f9adeddaf9beffdb7e5dcfe357f0bcb3f3777bfc87fdfd9b7e7bdd6fdffece6cbffbffffa8eed7df4ed73dfddfebefff7dfffffffdfee7dbfff97a","dbfbed93be9c8dbbcbfedc9d3d7cfbfffef7be53fdfbdaeeccf7dff7db2ddfed57b3ffffff3edbabfbc7d7effefd95efe6d4ff5cd7fffe7ffd78febdd7eebdcf","9ff5b35dd8efdbfc1fd33dcedffffeab35a73bee9f36ff7ddbe59df7effb9e94defddebfffe79bfdbafdfbffe3bfdde7ffdb707fdd9f8f77f7fbbd8fbfff3f7f","d69fedf738fffaf35f57e6dcfeff6defedddffb8acd5ebbeafe7fbe3fefb5fbff4fffd9bebcff6efdc72f67dd49ff771bf7bffe3fdffdfffeffebffece4755c5","efb760ff6bfffb2f9ffd9bcfcbeffdff7de74fd2ccb72ef7cb7f5e78fdfbbf37bb5fff3d7f7fdbf263ff77fdb7c5fe7de5eafffffbff9f7cd7effcf6375e77e5","7fe5af9d27b3a7fd697f35aedfafbfafe77bf73679deff873ffd7fdff7be23de7fef46d2f337fbd3ffe2f5efffbff5ffdc774df93f77feffffbff1ffbffdee6b","fdee6f6becf7afafa3b7ecc7f6be3efea86bcfde3f3dfdfaeed456aff07daf57fbdafbfedfffffdfedfffd9fb49f7f77beed77efff6e7fafbfefbef43fedff75","fde950bff5ef3f7defdf2f7e7fbfc5f53f3e07e5de9ffe9d5fbf6ffff7a1f5f2fdedf32ef771fffd98f7b3fddffe195f07feaff5fdfbfffd4fffffd3ffff9e5f","adbcef9e83fef336ff7df7dfed7fb7ffbbfbf9fc7fbc51f7fbfff7fcbefbef3bb64fef77e1967a7f7eeffc3df7f6f5e47f1bef7bbd6efeaf5b5fef9b73f7fdfe","be3fe7f7793b3b3ec9fdef9b143f6e7f7ffdafbffff1eace3efd8bdb7eb7fddbeff6efefde53a7dfd7ffccdfde3fbbfdbbedeeffd8feceffdfb5ffdfef7d1edd","77ffbfe72ecdb95f9bde3fcf0dfffeebbf27fdaff571dfad6fbdefcdafdffffff6f457dffed3bffb7d5f7d6ef7b7bfbd8ffef796fbb7ff568ffe7978f7fd23fb","bf7b7ef53f5e6c7f7feffbfeff98bdcf7fc7be7fdbdefff5dfc5dbfaba9f3ffd78fffb755fffb3d64d979ff9f7fedfbfb1dfccd5efdbaebbecafff7babfd572f","f58eef9ff7c7c7c7b95ff5dfafbc17ff5cffffbfbbfcd1eca5fbef6cdbfb3fbfff3c7ddf77fcb565eddadbdee392ff7dbdbff76ffdffebba6fb7efffdfffda7b","ef6e5e7af9efffe5b7f9bff37ffdaf9d3e7f7fcde6f7bff78b55fef8f75edf37dadfce779afed8c666f6f2df5a7ffbaddfeffffffdfebfdfdb26f7f7e7afb7be","fabdfea1dffecfd0ffeb9fe78efe7cfd0fdf93db7e7fbaffabb0ef7fbfb7e5df7ffbef27f37f5fdfee6ff2bf5abb73e6bffef7fffdbaff774fdfbebfdff2c95e","f7f7f577abe7d3f7cc0b77ec9ffbbcefedfc7c85bcf5f73f567dfb7d5f9beff97e17fdfffefcbfd7b7fffbffd77ffffcfddfcfc866b59f77fb7ffbb7f7c2f6bf","6ffabe5b22fdeed27ebfbdfff6e42b7fac763bbef4bf7bfdfde1bfff9f97bfffa5fed3dd7dfbb5ffde36daff7faff5a33fafffe2fffffbfdbb7bbefdf3c3ffeb","31cfdbf3e9ed9f6f7cdfdbf7ffc7efcf3d567ff5f7bff2dfff7fae3f52fbf99745fa7ff3fe3ffff55ffd5f7e77bfef59eff932fbbfa5ffe77abfd5f3fd975f7f"]}}'
_VL_REAP_HOOK_SOURCE = '"""ARC3 v42 REAP-at-load: drop low-saliency routed experts while the TARGET model loads, so their GPU memory goes to the\nKV cache. Written as arc3_reap.py into the vLLM runtime\'s dist-packages by the notebook\'s launcher, which also appends\none guarded install() call to vllm/models/qwen4_exp/nvidia/model.py. Active only when ARC3_REAP_KEPT names a kept-expert\nfile (set in the server\'s environment); without it install() does nothing.\n\nWhat it does to the checkpoint stream of the target model (the MTP draft loads separately and keeps its 512 experts):\n- routed-expert tensors `...layers.L.mlp.experts.E.<rest>`: kept experts are renumbered to their rank in the layer\'s\n  sorted kept list (0..keep-1); pruned experts are skipped;\n- routers `...layers.L.mlp.gate.weight` ([512, hidden]): sha256-checked against the checkpoint the kept lists were\n  verified on, then reduced to the kept rows (same order) - so top-k routing simply never sees a pruned expert;\n- everything else (and every `mtp.` tensor) passes through untouched.\nThe server is started with --hf-overrides \'{"text_config": {"num_experts": keep}}\' (a dict override reaches the target\nmodel config only; vLLM builds the draft config without it). At the end of loading it checks the counts (every layer\nhad a router, keep experts kept and 512-keep skipped with the same tensors per expert) and writes ARC3_REAP_STATUS\n(JSON). Any problem raises, so a wrong file can never load silently; the launcher then restarts without REAP."""\nimport functools\nimport hashlib\nimport json\nimport os\nimport re\nimport threading\nimport time\n\nENV_KEPT = "ARC3_REAP_KEPT"\nENV_STATUS = "ARC3_REAP_STATUS"\nNUM_EXPERTS = 512\n_EXPERT = re.compile(r"^(?P<pre>(?:.*\\.)?layers\\.(?P<layer>\\d+)\\.mlp\\.experts\\.)(?P<eid>\\d+)(?P<rest>\\..+)$")\n_ROUTER = re.compile(r"^(?:.*\\.)?layers\\.(?P<layer>\\d+)\\.mlp\\.gate\\.weight$")\n_LOCAL = threading.local()\nCLASSES = ("Qwen4ExpForConditionalGeneration", "Qwen4ExpForCausalLM")\n\n\ndef write_status(**fields):\n    path = os.environ.get(ENV_STATUS)\n    if not path:\n        return\n    fields.update(pid=os.getpid(), at=round(time.time(), 1))\n    tmp = f"{path}.{os.getpid()}.tmp"\n    with open(tmp, "w", encoding="utf-8") as f:\n        json.dump(fields, f)\n    os.replace(tmp, path)\n\n\ndef load_spec(path):\n    """{\'keep\': K, \'kept\': {layer: [K sorted original ids]}, \'router_sha256\': [one per layer]}"""\n    with open(path, encoding="utf-8") as f:\n        spec = json.load(f)\n    keep = int(spec["keep"])\n    kept = {int(layer): [int(e) for e in ids] for layer, ids in spec["kept"].items()}\n    shas = list(spec["router_sha256"])\n    if not 0 < keep < NUM_EXPERTS or not kept or len(shas) != len(kept):\n        raise ValueError(f"bad REAP spec: keep {keep}, {len(kept)} layers, {len(shas)} router hashes")\n    for layer, ids in kept.items():\n        if len(ids) != keep or len(set(ids)) != keep or ids != sorted(ids) or ids[0] < 0 or ids[-1] >= NUM_EXPERTS:\n            raise ValueError(f"bad REAP kept list for layer {layer}")\n        if not 0 <= layer < len(shas):\n            raise ValueError(f"REAP kept list names layer {layer} without a router hash")\n    return keep, kept, shas\n\n\ndef _sha256(tensor):\n    import torch\n    t = tensor.detach()\n    if t.device.type != "cpu":\n        t = t.cpu()\n    t = t.contiguous()\n    if t.dtype in (torch.bfloat16, torch.float16):\n        t = t.view(torch.int16)\n    return hashlib.sha256(t.numpy().tobytes()).hexdigest()\n\n\ndef new_counters(layers):\n    return {"routers": 0, "kept": 0, "skipped": 0, "kept_by_layer": {l: 0 for l in layers},\n            "skipped_by_layer": {l: 0 for l in layers}, "router_layers": []}\n\n\ndef remap(weights, keep, kept, shas, counters):\n    """The checkpoint stream with pruned experts skipped, kept experts renumbered and routers reduced."""\n    slot = {layer: {e: s for s, e in enumerate(ids)} for layer, ids in kept.items()}\n    index = {}\n    for name, tensor in weights:\n        if name.startswith("mtp.") or ".mtp." in name:\n            yield name, tensor\n            continue\n        m = _EXPERT.match(name)\n        if m:\n            layer, eid = int(m.group("layer")), int(m.group("eid"))\n            if layer not in slot or eid >= NUM_EXPERTS:\n                raise RuntimeError(f"ARC3 REAP: unexpected routed-expert tensor {name}")\n            s = slot[layer].get(eid)\n            if s is None:\n                counters["skipped"] += 1\n                counters["skipped_by_layer"][layer] += 1\n                continue\n            counters["kept"] += 1\n            counters["kept_by_layer"][layer] += 1\n            yield f"{m.group(\'pre\')}{s}{m.group(\'rest\')}", tensor\n            continue\n        m = _ROUTER.match(name)\n        if m:\n            layer = int(m.group("layer"))\n            if layer not in slot:\n                raise RuntimeError(f"ARC3 REAP: router of unknown layer {name}")\n            if tuple(tensor.shape)[:1] != (NUM_EXPERTS,):\n                raise RuntimeError(f"ARC3 REAP: router {name} has shape {tuple(tensor.shape)}, expected {NUM_EXPERTS} rows")\n            sha = _sha256(tensor)\n            if sha != shas[layer]:\n                raise RuntimeError(f"ARC3 REAP: router of layer {layer} is not the checkpoint the kept lists were "\n                                   f"checked on (sha256 {sha[:16]} != {shas[layer][:16]})")\n            if layer not in index:\n                import torch\n                index[layer] = torch.tensor(kept[layer], dtype=torch.long, device=tensor.device)\n            counters["routers"] += 1\n            counters["router_layers"].append(layer)\n            yield name, tensor.index_select(0, index[layer])\n            continue\n        yield name, tensor\n\n\ndef check(counters, keep, kept):\n    problems = []\n    layers = sorted(kept)\n    if sorted(counters["router_layers"]) != layers:\n        problems.append(f"routers seen for {len(counters[\'router_layers\'])} of {len(layers)} layers")\n    per_expert = set()\n    for layer in layers:\n        k, s = counters["kept_by_layer"][layer], counters["skipped_by_layer"][layer]\n        if k % keep or s % (NUM_EXPERTS - keep) or not k or k // keep != s // (NUM_EXPERTS - keep):\n            problems.append(f"layer {layer}: {k} kept / {s} skipped expert tensors")\n        else:\n            per_expert.add(k // keep)\n    if len(per_expert) > 1:\n        problems.append(f"tensors per expert differ between layers: {sorted(per_expert)}")\n    return problems\n\n\ndef _wrap(cls, keep, kept, shas):\n    original = cls.load_weights\n\n    @functools.wraps(original)\n    def load_weights(self, weights, *args, **kwargs):\n        if getattr(_LOCAL, "active", False):          # a nested model\'s load_weights inside the outer one\n            return original(self, weights, *args, **kwargs)\n        counters = new_counters(kept)\n        _LOCAL.active = True\n        try:\n            loaded = original(self, remap(weights, keep, kept, shas, counters), *args, **kwargs)\n            problems = check(counters, keep, kept)\n            if problems:\n                raise RuntimeError("ARC3 REAP: " + "; ".join(problems))\n        except BaseException as exc:\n            write_status(status="error", keep=keep, error=repr(exc)[:2000], routers=counters["routers"],\n                         kept=counters["kept"], skipped=counters["skipped"])\n            raise\n        finally:\n            _LOCAL.active = False\n        per = counters["kept"] // (keep * len(kept))\n        write_status(status="ok", keep=keep, layers=len(kept), routers=counters["routers"], kept=counters["kept"],\n                     skipped=counters["skipped"], tensors_per_expert=per, model_class=type(self).__name__)\n        print(f"ARC3 REAP: kept {keep} of {NUM_EXPERTS} routed experts in each of {len(kept)} layers "\n              f"({counters[\'kept\']} expert tensors loaded, {counters[\'skipped\']} pruned tensors skipped, "\n              f"{per} per expert); {counters[\'routers\']} routers sha256-checked and reduced to the kept rows", flush=True)\n        return loaded\n\n    load_weights._arc3_reap = True\n    cls.load_weights = load_weights\n\n\ndef install(namespace):\n    """Called at the end of vllm/models/qwen4_exp/nvidia/model.py with its globals()."""\n    path = os.environ.get(ENV_KEPT)\n    if not path:\n        return False\n    keep, kept, shas = load_spec(path)\n    wrapped = []\n    for name in CLASSES:\n        cls = namespace.get(name)\n        if cls is None or getattr(getattr(cls, "load_weights", None), "_arc3_reap", False):\n            continue\n        _wrap(cls, keep, kept, shas)\n        wrapped.append(name)\n    if not wrapped:\n        raise RuntimeError(f"ARC3 REAP: none of {CLASSES} found to wrap")\n    return wrapped\n'
_VL_REAP_MODEL_FILE = "vllm/models/qwen4_exp/nvidia/model.py"
_VL_REAP_MARKER = "# ARC3 v42: REAP-at-load hook"
_VL_REAP_APPEND = (_VL_REAP_MARKER + " (does nothing unless ARC3_REAP_KEPT is set in the server environment)\n"
                   "try:\n"
                   "    import arc3_reap as _arc3_reap\n"
                   "    _arc3_reap.install(globals())\n"
                   "except Exception:\n"
                   "    import os as _arc3_os\n"
                   "    if _arc3_os.environ.get('ARC3_REAP_KEPT'):\n"
                   "        raise\n")
VLLM_REAP_KEPT_FILE = _VlPath("/tmp/arc3_reap_kept.json")
VLLM_REAP_STATUS_FILE = _VlPath(WORKING_DIR) / "arc3_reap_status.json"
VLLM_REAP_FAILED_LOG = _VlPath(WORKING_DIR) / "serve-reap-failed.log"
_VL_RELAX_FILE = "vllm/v1/worker/gpu/spec_decode/rejection_sampler_utils.py"
_VL_RELAX_STOCK_SHA256 = "d9f29ad1ce4fe40a078a78b21d73fab97bbbe018775acb480c0ad14dea0c5c35"   # vLLM e9757321 (the image)
_VL_RELAX_LINE = "                    accepted = target_logprob > tl.log(u) + draft_logprob\n"
_VL_REAP_ACTIVE = False       # True while the server is started with the pruned model
_VL_SEQS_OVERRIDE = None      # games at once for a full-model restart after the benchmark release


def _vl_reap_keep():
    keep = globals().get("VLLM_REAP_KEEP")
    if keep in (None, 0, False):
        return None
    keep = int(keep)
    if str(keep) not in _vl_json.loads(_VL_REAP_MASKS)["masks"]:
        raise ValueError(f"VLLM_REAP_KEEP must be one of {sorted(_vl_json.loads(_VL_REAP_MASKS)['masks'])} or None")
    return keep


def _vl_spec_relax_t():
    t = globals().get("VLLM_SPEC_RELAX_T")
    if t is None:
        return None
    t = float(t)
    if not 0.3 <= t <= 1.0:
        raise ValueError(f"VLLM_SPEC_RELAX_T must be in [0.3, 1.0] or None, got {t}")
    return t


def _vl_max_num_seqs():
    return int(_VL_SEQS_OVERRIDE or VLLM_MAX_NUM_SEQS)


def vllm_reap_spec(keep):
    """The kept-expert file the hook reads: sorted original expert ids per layer + the router hashes."""
    data = _vl_json.loads(_VL_REAP_MASKS)
    kept = {}
    for layer, mask in enumerate(data["masks"][str(keep)]):
        bits = int(mask, 16)
        kept[str(layer)] = [e for e in range(512) if bits >> e & 1]
        if len(kept[str(layer)]) != keep:
            raise ValueError(f"REAP mask of layer {layer} keeps {len(kept[str(layer)])} experts, not {keep}")
    return {"keep": keep, "kept": kept, "router_sha256": data["router_sha256"]}


def _vl_drop_pyc(path):
    cache = path.parent / "__pycache__"
    for pyc in cache.glob(path.stem + ".*.pyc") if cache.is_dir() else ():
        pyc.unlink(missing_ok=True)


def _vl_reap_status():
    try:
        return _vl_json.loads(VLLM_REAP_STATUS_FILE.read_text(encoding="utf-8"))
    except Exception:
        return None


def _vl_reap_disable(reason, during_run):
    """Stop using the pruned model: the next start is the full model. Before the benchmark release the harness still
    reads its history budget, so it gets the full model's; after it the server runs fewer games at once instead."""
    global _VL_REAP_ACTIVE, _VL_SEQS_OVERRIDE
    _VL_REAP_ACTIVE = False
    state = VLLM_STATE.get("reap") or {"keep": _vl_reap_keep()}
    fb = {"reason": str(reason)[:600], "during_run": during_run, "at_s": int(_vl_time.time() - NOTEBOOK_START_TIME)}
    budgets = globals().get("VLLM_HISTORY_BUDGET_K_BY_REAP") or {}
    full_k = budgets.get(None)
    if not during_run and full_k:
        os.environ["LOCAL_ANALYZER_CONTEXT_WINDOW"] = str((int(full_k) + 12) * 1024)
        os.environ["ARC3_CONTEXT_DRAIN_TOKENS"] = str((int(full_k) - int(globals().get("V60_TRIM_KEEP_K", 35))) * 1024
                                                      if globals().get("V60_LIGHT_TRIMS") else int(full_k) // 2 * 1024)  # v60
        fb["history_budget_k"] = int(full_k)
        seqs = globals().get("VLLM_FALLBACK_SEQS_BEFORE_RELEASE")   # v47: the full model's tested shape (16 x 88K)
        if seqs and int(seqs) < int(VLLM_MAX_NUM_SEQS):
            _VL_SEQS_OVERRIDE = int(seqs)
            os.environ["ARC3_MAX_ACTIVE_STREAMS"] = str(int(seqs))
            fb["max_num_seqs"] = int(seqs)
    elif during_run:
        _VL_SEQS_OVERRIDE = int(globals().get("VLLM_FALLBACK_SEQS_AFTER_RELEASE") or 14)
        fb["max_num_seqs"] = _VL_SEQS_OVERRIDE
    state["fallback"] = fb
    VLLM_STATE["reap"] = state
    _vl_log(f"REAP OFF ({reason}); the full model starts instead: {fb}")


def _vl_apply_arc3_patches(manifest):
    """v42 edits to the unpacked runtime (after his overlay): the REAP hook and the relaxed acceptance line."""
    global _VL_REAP_ACTIVE
    dist = VLLM_IMAGE_RUNTIME_ROOT / manifest["dist_packages"]
    keep = _vl_reap_keep()
    if keep:
        VLLM_STATE["reap"] = {"keep": keep, "installed": False, "load": None, "fallback": None}
        try:
            VLLM_REAP_KEPT_FILE.write_text(_vl_json.dumps(vllm_reap_spec(keep)), encoding="utf-8")
            (dist / "arc3_reap.py").write_bytes(_VL_REAP_HOOK_SOURCE.encode("utf-8"))
            model_py = dist / _VL_REAP_MODEL_FILE
            text = model_py.read_text(encoding="utf-8")
            if _VL_REAP_MARKER not in text:
                model_py.write_bytes((text.rstrip("\n") + "\n\n\n" + _VL_REAP_APPEND).encode("utf-8"))
            _vl_drop_pyc(model_py)
            VLLM_REAP_STATUS_FILE.unlink(missing_ok=True)
            VLLM_STATE["reap"]["installed"] = True
            _VL_REAP_ACTIVE = True
            _vl_log(f"REAP-at-load installed: the target model keeps {keep} of 512 routed experts per MoE layer")
        except Exception as exc:
            VLLM_STATE["reap"]["error"] = repr(exc)[:600]
            _vl_reap_disable(f"REAP hook could not be installed: {exc!r}", during_run=False)
    t = _vl_spec_relax_t()
    if t is not None and t < 1.0:
        target = dist / _VL_RELAX_FILE
        state = {"t": t, "patched": False}
        try:
            data = target.read_bytes()
            sha = _vl_hashlib.sha256(data).hexdigest()
            text = data.decode("utf-8")
            bonus = -_vl_math.log(t)
            new_line = (f"                    accepted = target_logprob + {bonus!r} > tl.log(u) + draft_logprob"
                        f"  # ARC3 v42: relaxed acceptance, min(1, p/{t})\n")
            if text.count(new_line) == 1 and _VL_RELAX_LINE not in text:   # this runtime is already patched
                state.update(patched=True, log_bonus=round(bonus, 6))
            elif sha != _VL_RELAX_STOCK_SHA256:
                state["why"] = f"stock file differs (sha256 {sha[:16]}); acceptance left exact"
            elif text.count(_VL_RELAX_LINE) != 1:
                state["why"] = f"acceptance line found {text.count(_VL_RELAX_LINE)} times; acceptance left exact"
            else:
                target.write_bytes(text.replace(_VL_RELAX_LINE, new_line).encode("utf-8"))
                _vl_drop_pyc(target)
                state.update(patched=True, log_bonus=round(bonus, 6))
        except Exception as exc:
            state["why"] = f"patch failed: {exc!r}"[:300]
        VLLM_STATE["spec_relax"] = state
        _vl_log(f"relaxed MTP acceptance t={t}: {state}")


def _vl_keep_failed_log():
    try:
        if VLLM_SERVER_LOG.exists():
            VLLM_REAP_FAILED_LOG.write_text(VLLM_SERVER_LOG.read_text(encoding="utf-8", errors="replace"),
                                            encoding="utf-8")
    except Exception:
        pass


def _vl_wait_gpu_free(timeout_s=180, below_mib=4000):
    """Before starting the full model after a failed pruned start: wait until the dead server's processes have
    released the GPU (nvidia-smi memory.used), at most timeout_s."""
    if not _vl_shutil.which("nvidia-smi"):
        return None
    started = _vl_time.time()
    used = None
    while _vl_time.time() - started < timeout_s:
        try:
            res = _vl_sp.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, timeout=30)
            used = max(int(x) for x in res.stdout.split() if x.strip().isdigit())
        except Exception:
            used = None
        if used is not None and used < below_mib:
            break
        _vl_time.sleep(5)
    _vl_log(f"GPU memory in use before the restart: {used} MiB after {_vl_time.time() - started:.0f}s")
    return used


def _vl_log(msg):
    print(f"[vllm {int(_vl_time.time() - NOTEBOOK_START_TIME)}s] {msg}", flush=True)


def _vl_resolve_model_path(path):
    """A Kaggle model mount: /kaggle/input/models/<owner>/<model>/<framework>/<instance>/<version> or the old
    /kaggle/input/<model>/<framework>/<instance>/<version>; the framework folder may be lower- or mixed-case."""
    p = _VlPath(path)
    if p.exists():
        return p
    parts = p.parts
    if len(parts) >= 9 and parts[3] == "models":
        owner, model, framework, instance, version = parts[4:9]
        for fw in dict.fromkeys((framework.lower(), framework, "PyTorch", "pytorch")):
            for cand in (_VlPath("/kaggle/input/models", owner, model, fw, instance, version),
                         _VlPath("/kaggle/input", model, fw, instance, version)):
                if cand.exists():
                    _vl_log(f"model resolved: {path} -> {cand}")
                    return cand
    return p


def _vl_assert_gpu():
    if not _VlPath("/kaggle/input").exists():
        return
    assert _vl_shutil.which("nvidia-smi"), "CUDA GPU check failed: nvidia-smi is not available."
    res = _vl_sp.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True)
    assert res.returncode == 0, f"nvidia-smi failed: {res.stderr.strip()}"
    names = [line.strip() for line in res.stdout.splitlines() if line.strip()]
    assert names, "nvidia-smi did not report any CUDA GPUs."
    expected = os.getenv("KAGGLE_GPU_TYPE", "rtx-pro-6000").strip().lower()
    patterns = _VL_GPU_NAME_PATTERNS.get(expected, (expected.replace("-", " "),))
    bad = [n for n in names if not any(pt in n.lower() for pt in patterns)]
    assert not bad, f"Expected GPU type {expected!r}, found {names}"
    _vl_log(f"GPU check passed: {names}")


def _vl_write_draft_vocab():
    raw = _vl_zlib.decompress(_vl_b64.b64decode(_VL_DRAFT_VOCAB_B64))
    if _vl_hashlib.sha256(raw).hexdigest() != _VL_DRAFT_VOCAB_SHA256:
        raise RuntimeError("embedded draft vocabulary does not match its sha256 (notebook damaged?)")
    VLLM_DRAFT_VOCAB_FILE.write_bytes(raw)


def _vl_read_manifest(runtime):
    path = runtime / "runtime-manifest.json"
    if not path.exists():
        raise FileNotFoundError(f"Missing image runtime manifest: {path}")
    return _vl_json.loads(path.read_text(encoding="utf-8"))


def _vl_validate_runtime(runtime, manifest):
    """His fail-closed rule, keyed to the attached dataset version: the runtime's patch must be the one the dataset's
    PATCH_IDENTITY.json describes (the applier then checks every overlay file hash) and must carry the overlay files
    that make prefix caching safe for this hybrid model."""
    patch = manifest.get("patch") or {}
    ident_path = runtime / "PATCH_IDENTITY.json"
    ident =_vl_json.loads(ident_path.read_text(encoding="utf-8")) if ident_path.is_file() else {}
    for key in ("artifact", "applier", "overlay_sha256", "vllm_commit"):
        if ident and patch.get(key) != ident.get(key):
            raise RuntimeError(f"runtime-manifest.json and PATCH_IDENTITY.json disagree on {key}: "
                               f"{patch.get(key)!r} vs {ident.get(key)!r}")
    targets = [o.get("target") for o in patch.get("overlay_files", [])]
    missing = [t for t in _VL_PREFIX_CACHE_CRITICAL if t not in targets]
    if missing:
        raise RuntimeError("runtime overlay lacks the Mamba/MTP prefix-cache fixes: " + ", ".join(missing))
    VLLM_STATE["overlay_sha256"] = patch.get("overlay_sha256")
    VLLM_STATE["tested_overlay"] = patch.get("overlay_sha256") == _VL_TESTED_OVERLAY
    VLLM_STATE["vllm_version"] = manifest.get("vllm_version")
    if not VLLM_STATE["tested_overlay"] or manifest.get("vllm_version") != _VL_TESTED_VLLM:
        _vl_log(f"NOTE: runtime {manifest.get('vllm_version')} / overlay {str(patch.get('overlay_sha256'))[:12]} differs "
                f"from the version this notebook was built against ({_VL_TESTED_VLLM} / {_VL_TESTED_OVERLAY[:12]})")


def _vl_python_paths(manifest):
    dist = VLLM_IMAGE_RUNTIME_ROOT / manifest["dist_packages"]
    return [dist / "nvidia_cutlass_dsl" / "dsl_packages", dist]


def _vl_cuda_home(manifest):
    return VLLM_IMAGE_RUNTIME_ROOT / _VlPath(manifest["nvcc"]).parent.parent


def _vl_library_paths(manifest):
    dist = VLLM_IMAGE_RUNTIME_ROOT / manifest["dist_packages"]
    nvidia_libs = sorted(dist.glob("nvidia/*/lib")) + sorted(dist.glob("nvidia/cu13/*/lib"))
    return [_vl_cuda_home(manifest) / "targets" / "x86_64-linux" / "lib", dist / "torch" / "lib", *nvidia_libs,
            _VlPath("/usr/local/nvidia/lib64")]


def _vl_prepend(existing, entries):
    return os.pathsep.join([str(e) for e in entries] + ([existing] if existing else []))


def _vl_runtime_env(manifest):
    """His server environment for the extracted image runtime (VLLM_ARC3_GDN_RECOVERSSM=1 frees 9 KV blocks per
    request; the draft vocabulary is handed to the runtime through VLLM_QWEN4_EXP_DRAFT_VOCAB)."""
    e = os.environ.copy()
    cuda_home = _vl_cuda_home(manifest)
    caches = {"VLLM_CACHE_ROOT": VLLM_IMAGE_RUNTIME_CACHE_ROOT / "vllm",
              "TRITON_CACHE_DIR": VLLM_IMAGE_RUNTIME_CACHE_ROOT / "triton",
              "FLASHINFER_WORKSPACE_BASE": VLLM_IMAGE_RUNTIME_CACHE_ROOT / "flashinfer",
              "B12X_COMPILE_CACHE_DIR": VLLM_IMAGE_RUNTIME_CACHE_ROOT / "b12x" / "compile",
              "TMPDIR": VLLM_IMAGE_RUNTIME_CACHE_ROOT / "tmp"}
    for path in caches.values():
        path.mkdir(parents=True, exist_ok=True)
    for key in ("PYTORCH_ALLOC_CONF", "VLLM_PLE_CPU_OFFLOAD", "VLLM_PLE_OFFLOAD_READY_TIMEOUT",
                "VLLM_PREFIX_CACHE_RETENTION_INTERVAL", "VLLM_QWEN4_EXP_DRAFT_VOCAB"):
        e.pop(key, None)
    e["PYTHONPATH"] = _vl_prepend(e.get("PYTHONPATH", ""), _vl_python_paths(manifest))
    e["PATH"] = _vl_prepend(e.get("PATH", ""), [cuda_home / "bin", VLLM_IMAGE_RUNTIME_ROOT / "usr" / "local" / "bin"])
    e["LD_LIBRARY_PATH"] = _vl_prepend(e.get("LD_LIBRARY_PATH", ""),
                                       [p for p in _vl_library_paths(manifest) if p.is_dir()])
    e.update({k: str(v) for k, v in caches.items()})
    e.update({"USE_TF": "0", "TRANSFORMERS_NO_TF": "1", "TRANSFORMERS_NO_TORCHVISION": "1", "VLLM_NO_USAGE_STATS": "1",
              "CUDA_HOME": str(cuda_home), "CUDACXX": str(cuda_home / "bin" / "nvcc"), "TORCH_CUDA_ARCH_LIST": "12.0",
              "VLLM_ENABLE_CUDA_COMPATIBILITY": "0", "VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS": "1",
              "VLLM_WORKER_MULTIPROC_METHOD": "spawn", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:False",
              "HF_HUB_OFFLINE": "1", "HF_DATASETS_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
              "TOKENIZERS_PARALLELISM": "false", "VLLM_GDN_DECODE_KERNEL": "triton",
              "VLLM_QWEN4_EXP_DRAFT_VOCAB": str(VLLM_DRAFT_VOCAB_FILE)})
    e.setdefault("VLLM_ARC3_GDN_RECOVERSSM", "1")
    e.pop("ARC3_REAP_KEPT", None)
    e.pop("ARC3_REAP_STATUS", None)
    if _VL_REAP_ACTIVE:   # v42
        e["ARC3_REAP_KEPT"] = str(VLLM_REAP_KEPT_FILE)
        e["ARC3_REAP_STATUS"] = str(VLLM_REAP_STATUS_FILE)
    return e


def _vl_install_runtime(runtime):
    """Unpack the image layers in index order, replay whiteouts and apply the runtime overlay (his code)."""
    manifest = _vl_read_manifest(runtime)
    _vl_validate_runtime(runtime, manifest)
    _vl_shutil.rmtree(VLLM_IMAGE_RUNTIME_ROOT, ignore_errors=True)
    VLLM_IMAGE_RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
    started = _vl_time.monotonic()
    decompress = ["--use-compress-program=pigz"] if _vl_shutil.which("pigz") else ["-z"]
    for layer in sorted(manifest["selected_layers"], key=lambda layer: int(layer["index"])):
        blob = runtime / layer["file"]
        if not blob.exists():
            raise FileNotFoundError(f"Missing runtime layer blob: {blob}")
        if blob.stat().st_size != int(layer["size"]):
            raise RuntimeError(f"Runtime layer {layer['index']} does not match its manifest size: {blob.name}")
        for wo in layer.get("whiteouts", []):
            if wo["kind"] == "opaque" and wo["target"] not in ("", "."):
                _vl_shutil.rmtree(VLLM_IMAGE_RUNTIME_ROOT / wo["target"], ignore_errors=True)
        _vl_sp.run(["tar", "-xf", str(blob), "-C", str(VLLM_IMAGE_RUNTIME_ROOT), "--no-same-owner", *decompress],
                   check=True)
        for wo in layer.get("whiteouts", []):
            (VLLM_IMAGE_RUNTIME_ROOT / wo["marker"]).unlink(missing_ok=True)
            if wo["kind"] == "file":
                target = VLLM_IMAGE_RUNTIME_ROOT / wo["target"]
                if target.is_dir() and not target.is_symlink():
                    _vl_shutil.rmtree(target)
                else:
                    target.unlink(missing_ok=True)
        _vl_log(f"extracted runtime layer {layer['index']} ({blob.name}) after {_vl_time.monotonic() - started:.0f}s")
    patch = manifest.get("patch")
    if patch:
        dist = VLLM_IMAGE_RUNTIME_ROOT / manifest["dist_packages"]
        res = _vl_sp.run([_vl_sys.executable, str(runtime / patch["applier"]), str(dist)], capture_output=True, text=True)
        if res.returncode != 0:
            raise RuntimeError(f"Runtime patch failed:\n{res.stderr.strip()[-3000:]}")
        _vl_log(f"runtime patch {patch['artifact']}: {res.stdout.strip()[-300:]}")
    _vl_log(f"image runtime ready: vLLM {manifest.get('vllm_version')}, torch {manifest.get('torch_version')}")
    return manifest


def vllm_server_command(model_path):
    """His Kaggle-verified launch with our served name, port and image size."""
    capture = _vl_math.ceil(_vl_max_num_seqs() * (1 + VLLM_MTP_TOKENS) / VLLM_CUDAGRAPH_CAPTURE_STEP) \
        * VLLM_CUDAGRAPH_CAPTURE_STEP
    speculative = {"method": "mtp", "num_speculative_tokens": VLLM_MTP_TOKENS, "use_local_argmax_reduction": True}
    cmd = [_vl_sys.executable, "-m", "vllm.entrypoints.cli.main", "serve", str(model_path),
           "--served-model-name", SERVED_MODEL_NAME, "--host", VLLM_HOST, "--port", str(VLLM_PORT),
           "--load-format", "safetensors", "--dtype", "bfloat16", "--tensor-parallel-size", "1",
           "--distributed-executor-backend", "mp", "--max-model-len", str(VLLM_MAX_MODEL_LEN),
           "--max-num-seqs", str(_vl_max_num_seqs()), "--max-num-batched-tokens", str(VLLM_MAX_NUM_BATCHED_TOKENS),
           "--async-scheduling", "--enable-chunked-prefill", "--max-cudagraph-capture-size", str(capture),
           "--enable-prefix-caching", "--enable-auto-tool-choice", "--tool-call-parser", "qwen3_coder",
           "--reasoning-parser", "qwen3", "--generation-config", "vllm",
           "--engram-config", _vl_json.dumps({"cpu_offload": True}), "--cpu-offload-params", "embed_tokens",
           "--mm-processor-kwargs", _vl_json.dumps({"max_pixels": VLLM_IMAGE_MAX_PIXELS}),
           "--limit-mm-per-prompt", _vl_json.dumps({"video": 0}),
           "--default-chat-template-kwargs", '{"preserve_thinking": true, "reasoning_effort": "xhigh"}',
           "--quantization", "compressed-tensors",
           "--prefix-match-unit", "128",
           "--speculative-config", _vl_json.dumps(speculative),
           "--gpu-memory-utilization", VLLM_GPU_MEMORY_UTILIZATION, "--kv-cache-dtype", "fp8_e4m3",
           "--attention-config", _vl_json.dumps({"indexer_kv_dtype": "fp8"}), "--mamba-ssm-cache-dtype", "bfloat16"]
    retention = int(VLLM_PREFIX_RETENTION_INTERVAL)   # v36: -1 = his runtime default (dense Mamba checkpoints)
    if retention >= 0:   # v37: at v35's place in the command, so 0 gives v35's command exactly
        at = cmd.index("--prefix-match-unit") + 2
        cmd[at:at] = ["--prefix-cache-retention-interval", str(retention)]
    VLLM_STATE["retention_interval"] = retention
    if _VL_REAP_ACTIVE:   # v42: the target model is built with the kept experts only (dict overrides skip the draft)
        at = cmd.index("--quantization") + 2
        cmd[at:at] = ["--hf-overrides", _vl_json.dumps({"text_config": {"num_experts": _vl_reap_keep()}})]
    template = _VlPath(model_path) / "chat_template.jinja"
    if template.exists():
        cmd += ["--chat-template", str(template)]
    cmd += ["--disable-log-stats", "--disable-uvicorn-access-log"] if TRUE_SUBMISSION else ["--enable-prompt-tokens-details"]
    return cmd


def _vl_tail(path, lines=60):
    try:
        return "\n".join(_VlPath(path).read_text(encoding="utf-8", errors="replace").splitlines()[-lines:])
    except OSError:
        return ""


def _vl_poll_ready(watchdog, until_epoch):
    """'ready' once /v1/models answers, 'dead' if the watchdog exited first, 'loading' at the deadline."""
    url = f"{VLLM_BASE_URL}/models"
    while _vl_time.time() < until_epoch:
        if watchdog.poll() is not None:
            return "dead"
        try:
            with _vl_urlreq.urlopen(url, timeout=5) as r:
                if r.status == 200:
                    return "ready"
        except Exception:
            pass
        _vl_time.sleep(5)
    return "loading"


def _vl_mark_ready():
    VLLM_SERVER_READY.write_text(_vl_datetime.now().isoformat(timespec="seconds"), encoding="utf-8")
    VLLM_STATE["ready"] = True
    VLLM_STATE["ready_s"] = int(_vl_time.time() - NOTEBOOK_START_TIME)
    try:
        import re as _vl_re
        text = VLLM_SERVER_LOG.read_text(encoding="utf-8", errors="replace")
        m = _vl_re.search(r"GPU KV cache size:\s*([\d,]+)\s*tokens", text)
        if m:
            VLLM_STATE["kv_cache_tokens"] = int(m.group(1).replace(",", ""))
        for line in text.splitlines():
            low = line.lower()
            if any(k in low for k in ("gpu kv cache size", "maximum concurrency", "attention backend", "kernel for")):
                print(line[:300], flush=True)
    except Exception:
        pass
    if VLLM_STATE.get("reap"):   # v42
        VLLM_STATE["reap"]["load"] = _vl_reap_status() if _VL_REAP_ACTIVE else None
        _vl_log(f"REAP: {VLLM_STATE['reap']}")
        if _VL_REAP_ACTIVE and (VLLM_STATE["reap"]["load"] or {}).get("status") != "ok":
            _vl_log("WARNING: the server is up but the REAP hook reported no successful load - check serve.log")
    _vl_log(f"READY -> {VLLM_BASE_URL} (KV cache {VLLM_STATE['kv_cache_tokens']} tokens)")


_VL_PROBE_REASONING = "Probe reasoning. " + " ".join(f"Step {i}: the blue block moves one cell." for i in range(30))


def _vl_prompt_tokens(messages):
    payload = {"model": SERVED_MODEL_NAME, "messages": messages, "max_tokens": 1, "temperature": 0.0,
               "chat_template_kwargs": {"enable_thinking": True}}
    req = _vl_urlreq.Request(f"{VLLM_BASE_URL}/chat/completions", data=_vl_json.dumps(payload).encode("utf-8"),
                             headers={"Content-Type": "application/json"})
    with _vl_urlreq.urlopen(req, timeout=120) as r:
        return int(_vl_json.loads(r.read().decode("utf-8"))["usage"]["prompt_tokens"])


def vllm_probe_reasoning_key():
    """Which assistant-message field carries the model's past reasoning into this server's prompt? The harness stores
    reasoning under ARC3_REASONING_HISTORY_KEY (one key: its token estimate counts every field). A key the server
    ignores would silently drop the model's own reasoning from its history (da-fr: a large regression). Three tiny
    requests (one output token each) decide; the harness reads the variable on every request."""
    def convo(key):
        assistant = {"role": "assistant", "content": "OK"}
        if key:
            assistant[key] = _VL_PROBE_REASONING
        return [{"role": "user", "content": "Reply with OK."}, assistant, {"role": "user", "content": "Reply with OK again."}]
    try:
        base = _vl_prompt_tokens(convo(None))
        deltas = {key: _vl_prompt_tokens(convo(key)) - base for key in ("reasoning", "reasoning_content")}
    except Exception as exc:
        VLLM_STATE["reasoning_probe"] = f"failed: {exc!r}"[:300]
        _vl_log(f"reasoning-field probe failed ({exc!r}); keeping {os.environ.get('ARC3_REASONING_HISTORY_KEY')!r}")
        return None
    VLLM_STATE["reasoning_probe"] = deltas
    chosen = next((key for key in ("reasoning", "reasoning_content") if deltas[key] >= 100), None)
    VLLM_STATE["reasoning_key"] = chosen
    if chosen:
        os.environ["ARC3_REASONING_HISTORY_KEY"] = chosen
        _vl_log(f"past reasoning reaches the prompt under {chosen!r} (prompt growth per field: {deltas})")
    else:
        _vl_log(f"WARNING: neither 'reasoning' nor 'reasoning_content' reaches the prompt (growth {deltas}); "
                f"the model would lose its past reasoning - check before submitting")
    return chosen


def _vl_on_dead(where):
    VLLM_STATE["status"] = "dead"
    VLLM_STATE["error"] = VLLM_STATE["error"] or f"server exited {where}"
    _vl_log(f"vLLM server is down ({where}); watchdog log tail:\n{_vl_tail(VLLM_WATCHDOG_LOG, 40)}")
    print(_vl_tail(VLLM_SERVER_LOG, 60), flush=True)


def _vl_background(watchdog, launch_epoch, fallback, relaunch=None):
    """Keeps waiting after the benchmark was released, marks the server ready (so the watchdog restarts later
    crashes), and starts the fallback server if the vLLM watchdog gives up at any time. v42: a pruned (REAP) server
    that dies while loading is replaced once by the full model (relaunch) before that."""
    try:
        if not VLLM_STATE["ready"]:
            status = _vl_poll_ready(watchdog, launch_epoch + VLLM_READY_TIMEOUT_S)
            if status == "dead" and _VL_REAP_ACTIVE and relaunch is not None:
                _vl_reap_disable(f"pruned server died while loading, after the release (hook: {_vl_reap_status()})",
                                 during_run=True)
                _vl_keep_failed_log()
                _vl_wait_gpu_free()
                watchdog, status = relaunch(launch_epoch + VLLM_READY_TIMEOUT_S)
            if status == "ready":
                _vl_mark_ready()
                VLLM_STATE["status"] = "ready"
                vllm_probe_reasoning_key()
            elif status == "loading":
                VLLM_STATE["error"] = f"not ready after {VLLM_READY_TIMEOUT_S}s"
                _vl_log(VLLM_STATE["error"])
        code = watchdog.wait()
        VLLM_STATE["watchdog_exit"] = {"code": code, "at_s": int(_vl_time.time() - NOTEBOOK_START_TIME)}
        _vl_on_dead("after the benchmark was released" if VLLM_STATE["ready"] else "while loading")
    except Exception as exc:
        VLLM_STATE["error"] = f"background monitor: {exc!r}"[:300]
        return
    if fallback is not None:
        fallback("vLLM watchdog exited during the run")


def vllm_start_and_wait(fallback=None):
    """Unpack the runtime, start his watchdog (which spawns the server), wait until ready or until the notebook's
    startup deadline. Returns 'ready', 'loading' (keeps loading in the background) or 'dead'."""
    launch_epoch = _vl_time.time()
    _vl_assert_gpu()
    model_path = _vl_resolve_model_path(VLLM_MODEL_DIR)
    runtime = _VlPath(_resolve_input(VLLM_RUNTIME_DIR))
    missing = [str(p) for p in (runtime, model_path) if not p.exists()]
    if missing:
        raise FileNotFoundError("missing vLLM input(s): " + ", ".join(missing)
                                + " - attach lordhansolo/vllm-main-e975732-arc3 and the model "
                                  "lordhansolo/qwen3-8-flash-next-mixed-nvfp4-fp8 (hf-mixed-mtp-nvfp4)")
    if not (model_path / "model.safetensors.index.json").is_file():
        raise FileNotFoundError(f"no model.safetensors.index.json in {model_path}")
    _vl_write_draft_vocab()
    manifest = _vl_install_runtime(runtime)
    VLLM_STATE["runtime_unpack_s"] = int(_vl_time.time() - launch_epoch)
    _vl_apply_arc3_patches(manifest)   # v42: REAP hook + relaxed acceptance

    def launch(until_epoch):
        global proc, args, env, LOG
        server_env = _vl_runtime_env(manifest)
        cmd = vllm_server_command(model_path)
        for path in (VLLM_SERVER_LOG, VLLM_SHARD_PREFETCH_REPORT, VLLM_SERVER_PID, VLLM_SERVER_READY):
            path.unlink(missing_ok=True)
        _vl_log("starting vLLM OpenAI server: " + " ".join(cmd))
        VLLM_WATCHDOG_SCRIPT.write_text(WATCHDOG_SCRIPT_TEXT, encoding="utf-8")
        VLLM_WATCHDOG_CONFIG.write_text(_vl_json.dumps({
            "command": cmd, "server_log": str(VLLM_SERVER_LOG), "server_pid": str(VLLM_SERVER_PID),
            "server_ready": str(VLLM_SERVER_READY), "model_path": str(model_path),
            "shard_prefetch_report": str(VLLM_SHARD_PREFETCH_REPORT),
            "host_memory_log": str(VLLM_HOST_MEMORY_LOG)}), encoding="utf-8")
        watchdog_log = VLLM_WATCHDOG_LOG.open("w", encoding="utf-8")
        watchdog = _vl_sp.Popen([_vl_sys.executable, str(VLLM_WATCHDOG_SCRIPT), str(VLLM_WATCHDOG_CONFIG)],
                                env=server_env, stdout=watchdog_log, stderr=_vl_sp.STDOUT, text=True)
        VLLM_STATE["launch_s"] = int(_vl_time.time() - NOTEBOOK_START_TIME)
        # the names the rest of the notebook reads (watchdog cell, report): the process, the command, its env, its log
        proc, args, env, LOG = watchdog, cmd, server_env, str(VLLM_SERVER_LOG)
        return watchdog, _vl_poll_ready(watchdog, until_epoch)

    watchdog, status = launch(max(NOTEBOOK_START_TIME + SERVER_STARTUP_TIMEOUT, _vl_time.time() + 5))
    if status == "dead" and _VL_REAP_ACTIVE:   # v42: the pruned model did not come up -> the full model, once
        _vl_reap_disable(f"pruned server died before it was ready (hook: {_vl_reap_status()})", during_run=False)
        _vl_keep_failed_log()
        _vl_wait_gpu_free()
        watchdog, status = launch(max(NOTEBOOK_START_TIME + SERVER_STARTUP_TIMEOUT, _vl_time.time() + 5))
    VLLM_STATE["status"] = status
    if status == "ready":
        _vl_mark_ready()
        vllm_probe_reasoning_key()
    elif status == "dead":
        _vl_on_dead("before it was ready")
        return status
    else:
        _vl_log(f"still loading at the startup deadline ({SERVER_STARTUP_TIMEOUT}s from notebook start); releasing "
                f"the benchmark, games wait for the server")
    _vl_threading.Thread(target=_vl_background, args=(watchdog, launch_epoch, fallback, launch),
                         name="vllm-ready-monitor", daemon=True).start()
    return status


In [ ]:
# ===== v35: start the model server chosen in the config cell =====
# SERVER_ENGINE = 'vllm': lordhansolo's vLLM runtime (file written by the cell above). If it cannot start (an input is
# missing, the runtime patch fails, the server dies while loading) and VLLM_FALLBACK_TO_SGLANG is on, v34's SGLang
# server starts instead, with v34's 10 games at once; the same happens later in the run if the vLLM watchdog gives up
# after its 5 restarts (games keep retrying meanwhile). SERVER_ENGINE = 'sglang': v34's server, exactly.
import threading as _sv_threading

SERVER_ENGINE_USED = None
CACHE_DIAG_FILE = None
_SV_FALLBACK_LOCK = _sv_threading.Lock()


def _serve_exec(path):
    exec(compile(open(path, encoding="utf-8").read(), path, "exec"), globals())


def _serve_sglang(reason="", *, during_run=False):
    global SERVER_ENGINE_USED
    with _SV_FALLBACK_LOCK:
        if SERVER_ENGINE_USED == "sglang":
            return
        if reason:
            print(f"v35: starting v34's SGLang server instead ({reason})", flush=True)
        if isinstance(globals().get("VLLM_STATE"), dict):
            VLLM_STATE["fallback"] = {"reason": reason[:300], "during_run": during_run,
                                      "at_s": int(time.time() - NOTEBOOK_START_TIME)}
        # read per request by the harness; the slot count only when the first game starts
        os.environ["ARC3_REASONING_HISTORY_KEY"] = "reasoning_content"
        if not during_run:
            os.environ["ARC3_MAX_ACTIVE_STREAMS"] = "10"
        SERVER_ENGINE_USED = "sglang"
        _serve_exec("/tmp/arc_sglang_serve.py")


def _serve_sglang_during_run(reason):
    try:
        _serve_sglang(reason, during_run=True)
    except Exception as exc:
        print(f"v35: SGLang fallback failed: {exc!r}"[:600], flush=True)


if SERVER_ENGINE == "vllm":
    _vl_status = "error"
    try:
        _serve_exec("/tmp/arc_vllm_serve.py")
        _vl_status = vllm_start_and_wait(_serve_sglang_during_run if VLLM_FALLBACK_TO_SGLANG else None)
    except Exception as _vl_exc:
        if isinstance(globals().get("VLLM_STATE"), dict):
            VLLM_STATE["status"], VLLM_STATE["error"] = "error", repr(_vl_exc)[:800]
        print(f"v35: vLLM launch failed: {_vl_exc!r}"[:2000], flush=True)
    if _vl_status in ("ready", "loading"):
        with _SV_FALLBACK_LOCK:   # the background monitor may already have switched to SGLang
            if SERVER_ENGINE_USED is None:
                SERVER_ENGINE_USED = "vllm"
    elif VLLM_FALLBACK_TO_SGLANG:
        _serve_sglang(f"vLLM status {_vl_status}")
    else:
        raise RuntimeError(f"vLLM server did not start ({_vl_status}) and VLLM_FALLBACK_TO_SGLANG is off")
else:
    _serve_sglang()
print(f"SERVER_ENGINE_USED={SERVER_ENGINE_USED} | harness: {os.environ.get('ARC3_MAX_ACTIVE_STREAMS')} games at once, "
      f"reasoning sent as {os.environ.get('ARC3_REASONING_HISTORY_KEY')}", flush=True)


## 5b. Model-server watchdog (SGLang)

v35: the vLLM server has its own watchdog (lordhansolo's, up to 5 restarts), so this one is armed only when SGLang serves.

If the SGLang process exits (OOM, a CUDA fault, a crash during warm-up), this kills its process group, waits for the GPU memory to be released, and relaunches it with exactly the same arguments. It allows at most two restarts. While it relaunches, games keep retrying because `ARC3_MAX_ANALYZER_FAILURES` was raised in the config cell. It does nothing while the server is healthy.

In [ ]:
import signal as _wd_signal
import socket as _wd_socket
import subprocess as _wd_sp
import threading as _wd_threading

SERVER_WATCHDOG = {"restarts": 0, "events": []}
_WD_ARGS, _WD_ENV, _WD_LOG = list(args), dict(env), LOG   # snapshot the exact launch command
_WD_MAX_RESTARTS, _WD_POLL_S = 2, 15


def _wd_gpu_used_mib():
    try:
        out = _wd_sp.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True, timeout=10).stdout.split()
        return float(out[0]) if out else None
    except Exception:
        return None


def _wd_event(msg):
    msg = f"[watchdog {int(time.time() - NOTEBOOK_START_TIME)}s] {msg}"
    SERVER_WATCHDOG["events"].append(msg)
    print(msg, flush=True)


def _server_watchdog():
    while True:
        time.sleep(_WD_POLL_S)
        try:
            if not _wd_check_once():
                return
        except Exception as exc:   # a watchdog that dies silently is worse than none
            _wd_event(f"watchdog error {type(exc).__name__}: {exc}")


def _wd_check_once():
    # returns False once the restart budget is spent and the server is down for good
    global proc
    if proc.poll() is None:
        return True
    _wd_event(f"model server pid {proc.pid} exited with code {proc.returncode}")
    try:
        with open(_WD_LOG, "rb") as f:
            f.seek(0, 2); f.seek(max(0, f.tell() - 3000))
            print(f.read().decode(errors="replace"), flush=True)
    except OSError:
        pass
    if SERVER_WATCHDOG["restarts"] >= _WD_MAX_RESTARTS:
        _wd_event("restart limit reached; not relaunching")
        return False
    try:
        os.killpg(proc.pid, _wd_signal.SIGKILL)   # scheduler/detokenizer children share its session
    except (OSError, AttributeError):
        pass
    t0 = time.time()
    while time.time() - t0 < 300:   # the dead server's children must release the GPU first
        used = _wd_gpu_used_mib()
        if used is not None and used < 4096:
            break
        time.sleep(5)
    t1 = time.time()
    while time.time() - t1 < 120:   # ...and its port, or the new server fails to bind and dies at once
        with _wd_socket.socket() as probe:
            if probe.connect_ex(("127.0.0.1", int(SERVED_MODEL_PORT))) != 0:
                break
        time.sleep(3)
    _wd_event(f"relaunching after {int(time.time() - t0)}s wait (GPU used {_wd_gpu_used_mib()} MiB)")
    with open(_WD_LOG, "ab", buffering=0) as logf:
        proc = _wd_sp.Popen(_WD_ARGS, env=_WD_ENV, stdout=logf, stderr=_wd_sp.STDOUT, start_new_session=True)
    SERVER_WATCHDOG["restarts"] += 1
    _wd_event(f"relaunched model server as pid {proc.pid} (restart {SERVER_WATCHDOG['restarts']}/{_WD_MAX_RESTARTS})")
    return True


if ENABLE_SERVER_WATCHDOG and SERVER_ENGINE_USED == "sglang":
    _wd_threading.Thread(target=_server_watchdog, name="sglang-watchdog", daemon=True).start()
    print("server watchdog armed", flush=True)
elif SERVER_ENGINE_USED == "vllm":
    print("vLLM serves: restarts are handled by its own watchdog process (up to 5)", flush=True)


## 6. Load the benchmark

Unpickle the deployment target and the benchmark, stamping the real submission state onto the
target and pointing the benchmark's outputs at the Kaggle working directory.

In [ ]:
# Restore the deployment target and record the real submission state on it.
with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION

# Restore the benchmark and point its outputs at the Kaggle working dir.
with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR

## 6b. Merged harness add-ons (new)

These are installed on top of the patched harness after it has been imported by the benchmark unpickle:
* **Solved-level memory** (adapted from Duck v18's M85). This is the only cross-level carrier in this configuration, because memory sections are off. In v2 the block goes into the normal user prompt, which is stored in history exactly as sent, so it never invalidates the prefix cache. It is re-appended only when no message still in history carries its current version.
* **Compact runtime state.** The JSON content is the same, written without the one-line-per-cell indentation.

* **History de-duplication (v18, v19)**: see cell 6c; installed after the add-ons, wraps only the history trimmer.
* **Level-progress facts (v23)**: on a level past ~40K generated tokens (then 80K, 120K), the turn ends with the harness's interaction coverage for that level - actions used / never tried, colors clicked / never clicked. Information only; installed before the solved-level memory.
* **Background second opinion (v28)**: the game's own request goes out at once; the second opinion is fetched by a background thread and shown on the game's next turn (dropped if the level was cleared meanwhile), so the game's cached prompt is not evicted while it waits.
* **Bounded second opinion (v26)**: the side request thinks briefly (reasoning_effort 'low', 3,072 tokens) and falls back to one direct answer (700 tokens), so it always costs < 4K tokens and nearly always answers.
* **Cross-game rules (v44)**: when the level-progress facts fire on a stalled level, the player itself also gets how up to 5 other games of this run worked, as their players stated it when they cleared a level - on level 1 their level-1 rules, on later levels what their later levels added - games with the most similar controls first, never its own game, rules it has already seen last. The second opinion gets the same selection (before: the newest rule of up to 10 other games). Installed after the second opinion, before the solved-level memory.
* **Collective memory (v25)**: rules that games state when they clear a level are pooled run-wide; another game's second opinion sees the newest rule of up to 10 other games, plus the design catalogue in its system prompt. The playing model's prompt is unchanged.
* **Second opinion (v24)**: when the level-progress facts fire (40K / 80K tokens), a fresh model instance that has not seen the player's reasoning gets the level-start and current boards plus the measured facts and proposes up to 3 hypotheses with their cheapest tests; shown after the facts as information. Installed after the facts, before the solved-level memory.
* **Keep functions on timeout (v10)**: a python call stopped by the 30 s limit no longer drops the model's retained functions.
* **Move-budget meter (v7)**: the HUD bar in the screen border measured from the frames (cells left, rate per move, moves left), linked to earlier GAME OVERs; installed before the solved-level memory.
* **Action-effect hints + tool reminder (v6)**: a harness-computed summary of what each action did on the current level, plus a reminder to use the world-model tools. Installed before the solved-level memory so that memory's request stays last.
* **World-model tools (v4)**: `check_rules` and `plan_route`, spliced into the python sandbox (standard library only; the installer refuses if the sandbox code would exceed 120 KB, under Linux's 128 KB per-argument limit), plus a fixed system-prompt section and usage counters.

All were tested locally against the da-fr-patched harness (Duck harness commit `7652836` with the patch above). The world-model tools were tested inside the real sandbox process: rule checking, wrong-rule detection, search, and a route executed through `action()`.

In [ ]:
# ===== Merged harness add-ons (installed on top of the patched da-fr harness) =====
# A. SOLVED-LEVEL MEMORY (from the Tufa/Duck v18 notebook, M85, adapted).
#    This build runs with ARC3_MEMORY_SECTIONS=off, so once the history drain evicts a cleared level nothing
#    carries the rule that solved it. After each solved level the model is asked (in the next prompt, no extra
#    request) for a "Rule for level N:" line; that rule plus the winning action tail and the clearing turn's
#    reasoning are kept in front of the model for the rest of the game.
#    v2 placement (MEMORY_PLACEMENT = "history", default): the block is appended to the NORMAL user prompt, which
#    the harness stores in history exactly as sent, so the server's prefix cache is never invalidated by it. The
#    block is re-appended only when no message still in history carries the current version -- after a new rule
#    was captured, or after the oldest-first history drain evicted it (a drain re-prefills the context anyway).
#    v1 placement (MEMORY_PLACEMENT = "system"): pinned at the end of the system prompt; every update forces a full
#    re-prefill of that game's context (v1 run: 39 updates, 503 vs 588 tok/s). Kept only for A/B tests.
# B. COMPACT RUNTIME STATE (idea from Duck v18 M86). The solver rewrites the whole game history to
#    tool_runtime_state.json before every analyzer turn with json.dumps(indent=2): one line per grid cell. The
#    file content is identical JSON written without indentation (7.7x smaller in tests), which matters because
#    every game shares this one notebook process.
# C. WORLD-MODEL TOOLS (v4, see below): check_rules / plan_route in the python sandbox.
# D. ACTION-EFFECT HINTS + TOOL REMINDER (v6, see below): per-level summary of what each action did.
# E. MOVE-BUDGET METER (v7, see below): measured HUD bar - cells left, rate per move, moves left.
# G. LEVEL-PROGRESS FACTS (v23, see below): only on a level past ~40K tokens - what was tried and never tried there.
# H. KEEP RETAINED FUNCTIONS ON TIMEOUT (v10, see below): a timed-out python call no longer wipes the model's helpers.
# S. SECOND OPINION + COLLECTIVE MEMORY (v24-v28, see below): when G fires (40K / 80K tokens), a fresh model
#    instance that has not seen the player's reasoning proposes up to 3 hypotheses from the two boards, the measured
#    facts, the design catalogue and the rules OTHER games of this run discovered; fetched in the background and
#    shown on the game's next turn.
# X. CROSS-GAME RULES (v44, see below): when G fires, the player itself sees how other games of this run worked
#    (level 1: their level-1 rules; later levels: what their later levels added), most similar controls first;
#    the second opinion gets the same selection.
# K. GAME-DESIGN CATALOGUE (v22, see below): what 15 other public games look like and how they work (knowledge only), in the system prompt.
# All are wrapped so that any internal failure falls back to the original behavior and is only counted.
import json as _json_mx
import re as _re_mx
import threading as _th_mx

_LK_MX = _th_mx.Lock()
ADDON_STATS = {
    "m85_levels_recorded": 0, "m85_rules_captured": 0, "m85_rules_missing": 0, "m85_asks": 0,
    "m85_pins": 0, "m85_injections": 0, "m85_injected_chars": 0, "m85_block_chars_max": 0, "m85_errors": 0, "m85_last_error": "",
    "compact_writes": 0, "compact_fallbacks": 0, "compact_last_error": "",
}


def _mx_inc(key, n=1):
    with _LK_MX:
        ADDON_STATS[key] += n


def _mx_err(prefix, exc):
    with _LK_MX:
        ADDON_STATS[prefix + "_last_error"] = f"{type(exc).__name__}: {exc}"[:300]


# ---------------------------------------------------------------- A. solved-level memory
M85_ACTIONS_KEPT = 30
M85_REASONING_CHARS = 900
M85_RULE_CHARS = 600
M85_FULL_LEVELS = 3
M85_ASK_TURNS = 2
M85_HEADER = "=== SOLVED LEVELS (facts from this game + your own conclusions; kept for the whole game) ==="
M85_FOOTER = "=== END SOLVED LEVELS ==="


def _m85_text(msg):
    parts = []
    for key in ("content", "reasoning_content", "reasoning"):
        value = msg.get(key) if isinstance(msg, dict) else None
        if isinstance(value, str):
            parts.append(value)
        elif isinstance(value, list):
            parts += [p.get("text", "") for p in value if isinstance(p, dict) and isinstance(p.get("text"), str)]
    return "\n".join(p for p in parts if p)


def m85_level_actions(history_entries, level):
    """Actions taken while on `level`. history[i].frame is the frame AFTER history[i].action, so an action belongs
    to the level of the frame before it; the seeded first entry has no action and is skipped."""
    out, prev = [], None
    for entry in history_entries or []:
        act = str(getattr(entry, "action", "") or "").strip()
        if act and prev == level:
            out.append(act)
        lv = getattr(getattr(entry, "frame", None), "level", None)
        if isinstance(lv, int):
            prev = lv
    return out


def m85_find_rule(text, level):
    """The model's 'Rule for level N:' statement: its own line if >= 40 chars, else up to 5 following lines."""
    # v3: also accepts the variants models drift into: '**Rule for level 2:**', 'Rule for level 2 -',
    # 'Rule (level 2):', 'Level 2 rule:'. A different number ('level 12' for 1) never matches.
    n = int(level)
    m = _re_mx.search(r"(?:Rule\s*(?:for\s*)?\(?\s*level\s*%d\s*\)?|Level\s*%d\s*rule)\s*\**\s*[:\-–—]\s*\**[ \t]*"
                      % (n, n), text or "", _re_mx.I)
    if not m:
        return None
    lines = text[m.end():].split("\n")
    first = lines[0].strip()
    if len(first) >= 40:
        rule = first
    else:
        out = [first] if first else []
        for ln in lines[1:6]:
            if not ln.strip():
                if out:
                    break
                continue
            out.append(ln.strip())
        rule = " ".join(out)
    rule = " ".join(rule.split())[:M85_RULE_CHARS]
    return rule if len(rule) >= 20 else None


def m85_block(levels):
    if not levels:
        return ""
    lines = [M85_HEADER]
    full_from = max(0, len(levels) - M85_FULL_LEVELS)
    for i, rec in enumerate(levels):
        lines.append(f"Level {rec['level']} - cleared after {rec['n_actions']} actions.")
        lines.append(f"  Your rule: {rec['rule'] if rec.get('rule') else 'not stated'}")
        if i >= full_from:
            tail = rec["actions"]
            skipped = rec["n_actions"] - len(tail)
            lines.append("  Last actions before it cleared" + (f" ({skipped} earlier omitted)" if skipped > 0 else "")
                         + ": " + (", ".join(tail) if tail else "none recorded"))
            if rec.get("reasoning"):
                lines.append("  Your reasoning in the turn that cleared it: " + rec["reasoning"])
    lines.append("Reuse these where the new level has the same controls; verify with one probe before relying on them.")
    lines.append(M85_FOOTER)
    return "\n".join(lines)


def _m85_apply_block(agent):
    """Idempotent: strip any previous block, then append the current pinned block."""
    block = m85_block([r for r in agent._m85_levels if r.get("final")])
    sp = getattr(agent, "_system_prompt", "") or ""
    old = getattr(agent, "_m85_block_text", "")
    if old and old in sp:
        sp = sp.replace("\n\n" + old, "").replace(old, "")
    if block:
        sp = sp.rstrip("\n") + "\n\n" + block
    if sp != getattr(agent, "_system_prompt", None):
        agent._system_prompt = sp
        _mx_inc("m85_pins")
    agent._m85_block_text = block
    with _LK_MX:
        ADDON_STATS["m85_block_chars_max"] = max(ADDON_STATS["m85_block_chars_max"], len(block))


def _m85_message_has(message, block):
    content = message.get("content") if isinstance(message, dict) else None
    if isinstance(content, str):
        return block in content
    if isinstance(content, list):
        return any(isinstance(p, dict) and isinstance(p.get("text"), str) and block in p["text"] for p in content)
    return False


def _m85_history_injection(agent):
    """Text to append to this user prompt, or "" when history still carries the current block."""
    block = m85_block([r for r in agent._m85_levels if r.get("final")])
    agent._m85_block_text = block
    if not block:
        return ""
    with _LK_MX:
        ADDON_STATS["m85_block_chars_max"] = max(ADDON_STATS["m85_block_chars_max"], len(block))
    for message in getattr(agent, "_history_messages", []) or []:
        if isinstance(message, dict) and message.get("role") == "user" and _m85_message_has(message, block):
            return ""
    _mx_inc("m85_injections")
    _mx_inc("m85_injected_chars", len(block))
    return "\n\n" + block


def _m85_capture(agent):
    """Finalize pending levels: capture 'Rule for level N:' from assistant replies (visible text or reasoning)."""
    changed = False
    pending = [r for r in agent._m85_levels if not r.get("final") and r["asks"] > r["checks"]]
    if not pending:
        return False
    texts = [_m85_text(m) for m in (getattr(agent, "_history_messages", []) or [])
             if isinstance(m, dict) and m.get("role") == "assistant"]
    for rec in pending:
        rec["checks"] += 1
        for text in reversed(texts):
            rule = m85_find_rule(text, rec["level"])
            if rule:
                rec["rule"] = rule
                break
        if rec.get("rule"):
            rec["final"] = True
            changed = True
            _mx_inc("m85_rules_captured")
        elif rec["checks"] >= M85_ASK_TURNS:
            rec["final"] = True
            changed = True
            _mx_inc("m85_rules_missing")
    return changed


def install_solved_level_memory():
    import inference.agent.tool_agent as _ta
    agent_cls = _ta.ToolAgent
    inner = agent_cls._build_user_prompt
    if getattr(inner, "_mx_m85_wrapped", False):
        return False

    def _build_user_prompt(self, action_num, *args, **kwargs):
        out = inner(self, action_num, *args, **kwargs)
        try:
            if not hasattr(self, "_m85_levels"):
                self._m85_levels = []
                self._m85_block_text = ""
            changed = _m85_capture(self)
            summary = kwargs.get("previous_step_summary") or {}
            level_now = getattr(kwargs.get("current_frame"), "level", None)
            if summary.get("level_transition") and not summary.get("run_complete") and isinstance(level_now, int):
                solved = level_now - 1
                if solved >= 1 and all(r["level"] != solved for r in self._m85_levels):
                    acts = m85_level_actions(kwargs.get("history_entries") or [], solved)
                    reasoning = ""
                    for m in reversed(getattr(self, "_history_messages", []) or []):
                        if isinstance(m, dict) and m.get("role") == "assistant":
                            text = " ".join(_m85_text(m).split())
                            if text:
                                reasoning = text[-M85_REASONING_CHARS:]
                                break
                    # an older level still waiting for its rule is finalized as-is
                    for rec in self._m85_levels:
                        if not rec.get("final"):
                            rec["final"] = True
                            changed = True
                    self._m85_levels.append(dict(level=solved, n_actions=len(acts), actions=acts[-M85_ACTIONS_KEPT:],
                                                 reasoning=reasoning, rule=None, asks=0, checks=0, final=False))
                    _mx_inc("m85_levels_recorded")
            ask = ""
            last = self._m85_levels[-1] if self._m85_levels else None
            if last is not None and not last.get("final") and last["asks"] < M85_ASK_TURNS:
                last["asks"] += 1
                _mx_inc("m85_asks")
                ask = ("\n\nLEVEL %d SOLVED. In your reply this turn, before acting, write one short paragraph that "
                       "starts exactly with 'Rule for level %d:' -- the goal, what each control/action did, and the "
                       "plan that won (at most 5 lines). It is kept for every later level."
                       % (last["level"], last["level"]))
            if MEMORY_PLACEMENT == "system":
                out += ask
                if changed or (self._m85_block_text and self._m85_block_text not in (self._system_prompt or "")):
                    _m85_apply_block(self)
            else:
                # block before the ask, so the ask is the last thing the model reads
                out += _m85_history_injection(self) + ask
        except Exception as exc:  # never break a turn because of the add-on
            _mx_inc("m85_errors")
            _mx_err("m85", exc)
        return out

    for name in dir(inner):
        if name.endswith("_wrapped"):
            setattr(_build_user_prompt, name, getattr(inner, name))
    _build_user_prompt._mx_m85_wrapped = True
    agent_cls._build_user_prompt = _build_user_prompt
    return True


# ---------------------------------------------------------------- B. compact runtime state
def install_compact_runtime_state():
    import inference.agent.runtime_state as _rs
    import inference.framework.solver as _solver
    original = _rs.write_runtime_state
    if getattr(original, "_mx_compact_wrapped", False):
        return False

    def write_runtime_state(path, *, current_frame, history):
        try:
            path.parent.mkdir(parents=True, exist_ok=True)
            payload = {
                "current_frame": _rs.frame_to_payload(current_frame),
                "history": [_rs.history_entry_to_payload(entry) for entry in history],
            }
            tmp_path = path.with_suffix(f"{path.suffix}.tmp")
            tmp_path.write_text(_json_mx.dumps(payload, separators=(",", ":")), encoding="utf-8")
            tmp_path.replace(path)
            _mx_inc("compact_writes")
        except Exception as exc:
            _mx_inc("compact_fallbacks")
            _mx_err("compact", exc)
            original(path, current_frame=current_frame, history=history)

    write_runtime_state._mx_compact_wrapped = True
    _rs.write_runtime_state = write_runtime_state
    if getattr(_solver, "write_runtime_state", None) is original:
        _solver.write_runtime_state = write_runtime_state
    return True


def addon_summary():
    with _LK_MX:
        return dict(ADDON_STATS)


# ---------------------------------------------------------------- D. automatic action-effect hints + tool reminder (v6)
# v5 showed the model never chose to call the world-model tools (wm_snippets = 0). v6 stops depending on its choice:
# the harness itself summarises what every action has done on the current level, from the model's own moves, and
# adds that summary to the turn's message. No game action is spent. The block is re-added only when its content
# changed or the history drain evicted it, so it does not repeat every turn. A one-line reminder to use
# check_rules / plan_route is added at most twice per level, and never once the model has used them on that level.
import collections as _ah_collections
import os as _ah_os
import re as _ah_re
import threading as _ah_threading

from inference.utils.grid_utils import ARC_COLOR_CHARS as _AH_COLORS

AH_MAX_LINES = 8            # actions listed (most used first)
AH_MIN_MOVES = 2            # moves on the level before a summary is shown
AH_COMPONENT_CAP = 400      # largest object treated as movable
NUDGE_AT_MOVES = (8, 25)    # reminder thresholds, per level
AH_HEADER = "ACTION EFFECTS ON THIS LEVEL (computed by the harness from your own moves on this level; the outer {b}-cell border is ignored):"
AH_FOOTER = "(These are observations, not guarantees - verify anything surprising. Count = how often that effect followed the action.)"
_AH_CLICK_RE = _ah_re.compile(r"^\s*([A-Z0-9_]+)\(\s*row\s*=\s*(-?\d+)\s*,\s*col\s*=\s*(-?\d+)\s*\)\s*$")
_AH_TL = _ah_threading.local()      # each game runs in its own worker thread
_AH_USED = set()                    # (agent id, level) where the model called check_rules / plan_route
_AH_USED_LOCK = _ah_threading.Lock()
ADDON_STATS.update({"ah_blocks": 0, "ah_chars": 0, "ah_moves_analysed": 0, "nudges": 0,
                    "ah_errors": 0, "ah_last_error": ""})


def _ah_color(v):
    try:
        return _AH_COLORS[int(v)]
    except (ValueError, IndexError, TypeError):
        return str(v)


def _ah_border(rows, cols):
    border = max(0, int(_ah_os.environ.get("ARC3_NOOP_GUARD_BORDER", "4") or 0))
    return border if rows > 2 * border and cols > 2 * border else 0


def _ah_component(grid, start, color, lo, hi_r, hi_c):
    """4-connected cells of `color` containing `start`, inside the interior; None if larger than the cap."""
    seen = {start}
    stack = [start]
    while stack:
        r, c = stack.pop()
        for nr, nc in ((r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)):
            if lo <= nr < hi_r and lo <= nc < hi_c and (nr, nc) not in seen and grid[nr][nc] == color:
                seen.add((nr, nc))
                if len(seen) > AH_COMPONENT_CAP:
                    return None
                stack.append((nr, nc))
    return seen


def ah_effect(before, after):
    """Effect of one move on the interior of the board: None = no visible change, else a short label."""
    rows = len(after)
    cols = len(after[0]) if rows else 0
    if len(before) != rows or (rows and len(before[0]) != cols):
        return "board size changed"
    b = _ah_border(rows, cols)
    hi_r, hi_c = rows - b, cols - b
    changed = []
    for r in range(b, hi_r):
        br, ar = before[r], after[r]
        if br == ar:
            continue
        for c in range(b, hi_c):
            if br[c] != ar[c]:
                changed.append((r, c))
    if not changed:
        return None
    left = _ah_collections.Counter(before[r][c] for r, c in changed)
    entered = _ah_collections.Counter(after[r][c] for r, c in changed)
    for color in sorted(set(left) & set(entered), key=lambda x: (-left[x], x)):
        if left[color] != entered[color]:
            continue
        lost = next(((r, c) for r, c in changed if before[r][c] == color and after[r][c] != color), None)
        gained = next(((r, c) for r, c in changed if after[r][c] == color and before[r][c] != color), None)
        if lost is None or gained is None:
            continue
        old = _ah_component(before, lost, color, b, hi_r, hi_c)
        new = _ah_component(after, gained, color, b, hi_r, hi_c) if old else None
        if not old or not new or len(old) != len(new):
            continue
        dr = min(r for r, _ in new) - min(r for r, _ in old)
        dc = min(c for _, c in new) - min(c for _, c in old)
        if (dr or dc) and {(r + dr, c + dc) for r, c in old} == new:
            label = f"the '{_ah_color(color)}' object moved ({dr:+d},{dc:+d})"
            if any(cell not in old and cell not in new for cell in changed):
                label += " + other cells changed"
            return label
    pairs = _ah_collections.Counter((before[r][c], after[r][c]) for r, c in changed)
    return "cells changed " + ", ".join(f"'{_ah_color(x)}'->'{_ah_color(y)}'" for (x, y), _ in pairs.most_common(2))


def _ah_action_key(action, before_grid):
    """Clicks are grouped by the color that was clicked; other actions by name."""
    m = _AH_CLICK_RE.match(str(action))
    if m:
        r, c = int(m.group(2)), int(m.group(3))
        try:
            return f"{m.group(1)} on '{_ah_color(before_grid[r][c])}' cells"
        except (IndexError, TypeError):
            return f"{m.group(1)} (off-board)"
    return str(action).strip()


def ah_level_summary(agent, history_entries, level):
    """(lines, signature, moves) for the current level, from cached per-move labels."""
    cache = agent.__dict__.setdefault("_ah_cache", {})
    per_action = _ah_collections.OrderedDict()
    moves = 0
    for prev, cur in zip(history_entries, history_entries[1:]):
        action = str(getattr(cur, "action", "") or "").strip()
        pf, cf = getattr(prev, "frame", None), getattr(cur, "frame", None)
        if not action or pf is None or cf is None or pf.level != level or cf.level != level:
            continue
        result = getattr(cur, "result", {}) or {}
        if action.upper() == "RESET" or result.get("automatic"):
            continue
        key = (cf.step, action)
        label = cache.get(key)
        if label is None:
            if result.get("game_over"):
                label = "GAME OVER"
            else:
                label = ah_effect(pf.grid, cf.grid)
                if label is None:
                    label = "no lasting change (it animates)" if result.get("animation") else "no visible change"
            cache[key] = label
            _mx_inc("ah_moves_analysed")
        per_action.setdefault(_ah_action_key(action, pf.grid), _ah_collections.Counter())[label] += 1
        moves += 1
    ranked = sorted(per_action.items(), key=lambda kv: -sum(kv[1].values()))[:AH_MAX_LINES]
    lines, signature = [], []
    for name, counts in ranked:
        total = sum(counts.values())
        top = counts.most_common(3)
        lines.append(f"- {name} x{total}: " + "; ".join(f"{lab} {n}/{total}" for lab, n in top))
        signature.append((name, tuple(lab for lab, _ in top[:2])))
    return lines, tuple(signature), moves


def _ah_in_history(agent, text):
    return any(isinstance(m, dict) and m.get("role") == "user" and _m85_message_has(m, text)
               for m in (getattr(agent, "_history_messages", []) or []))


def install_action_hints():
    import inference.agent.tool_agent as _ta
    agent_cls = _ta.ToolAgent
    inner = agent_cls._build_user_prompt
    if getattr(inner, "_ah_wrapped", False):
        return False

    def _build_user_prompt(self, action_num, *args, **kwargs):
        out = inner(self, action_num, *args, **kwargs)
        try:
            _AH_TL.agent = id(self)
            frame = kwargs.get("current_frame")
            level = getattr(frame, "level", None)
            if not isinstance(level, int):
                return out
            history = kwargs.get("history_entries") or []
            lines, signature, moves = ah_level_summary(self, history, level)
            state = self.__dict__.setdefault("_ah_state", {})
            if ENABLE_ACTION_HINTS and moves >= AH_MIN_MOVES and lines:
                rows = len(frame.grid)
                header = AH_HEADER.format(b=_ah_border(rows, len(frame.grid[0]) if rows else 0))
                block = "\n".join([header, *lines, AH_FOOTER])
                last = state.get(level)
                if last is None or last[0] != signature or not _ah_in_history(self, last[1]):
                    out += "\n\n" + block
                    state[level] = (signature, block)
                    _mx_inc("ah_blocks")
                    _mx_inc("ah_chars", len(block))
            if ENABLE_TOOL_NUDGE and ENABLE_WORLD_MODEL_TOOLS:
                shown = self.__dict__.setdefault("_ah_nudges", {}).get(level, 0)
                with _AH_USED_LOCK:
                    used = (id(self), level) in _AH_USED
                if not used and shown < len(NUDGE_AT_MOVES) and moves >= NUDGE_AT_MOVES[shown]:
                    out += (f"\n\nTIP (costs no game actions): this level already has {moves} recorded moves. Write "
                            f"step(board, action) for the rules you believe, run check_rules(step) to score it against "
                            f"all {moves} moves, then plan_route(step, goal) for the shortest route and play it with a "
                            f"single action() call.")
                    self._ah_nudges[level] = shown + 1
                    _mx_inc("nudges")
        except Exception as exc:  # never break a turn because of the add-on
            _mx_inc("ah_errors")
            _mx_err("ah", exc)
        return out

    for name in dir(inner):
        if name.endswith("_wrapped"):
            setattr(_build_user_prompt, name, getattr(inner, name))
    _build_user_prompt._ah_wrapped = True
    agent_cls._build_user_prompt = _build_user_prompt
    return True


if ENABLE_ACTION_HINTS or ENABLE_TOOL_NUDGE:
    # installed BEFORE the solved-level memory, so memory's block and its 'Rule for level' ask stay last
    print("action-effect hints / tool reminder installed:", install_action_hints())


# ---------------------------------------------------------------- E. move-budget meter (v7)
# Transcript analysis of the v5 run: many games show a HUD bar in the screen border (a full-width row such as row 0
# or row 63) that loses about one cell per move and ends the attempt with GAME OVER when empty. m0r0 died without
# realising it; s5i5 / vc33 spent a large share of their thinking measuring it. The harness measures it exactly from
# the frames and adds ONE line per turn - no game action, about 50 tokens.
ADDON_STATS.update({"budget_lines": 0, "budget_deaths_linked": 0, "budget_errors": 0, "budget_last_error": ""})
BUDGET_MIN_MOVES = 3        # moves in the current attempt before a rate is reported
BUDGET_MIN_CHANGE = 3       # cells the bar must have changed
BUDGET_MIN_SIZE = 8         # cells the bar must have had at some point


def _bg_lines(rows, cols):
    """Border rows / columns (the same border the harness's no-op guard treats as HUD)."""
    b = max(1, int(_ah_os.environ.get("ARC3_NOOP_GUARD_BORDER", "4") or 1))
    b = min(b, rows // 2, cols // 2)
    lines = [("row", r) for r in list(range(b)) + list(range(rows - b, rows))]
    lines += [("col", c) for c in list(range(b)) + list(range(cols - b, cols))]
    return lines


def _bg_counts(grid, lines):
    out = {}
    for kind, i in lines:
        cells = grid[i] if kind == "row" else [row[i] for row in grid]
        out[(kind, i)] = _ah_collections.Counter(cells)
    return out


def _bg_detect(series):
    """The border line + color that changed monotonically the most over this attempt, or None.
    Prefers a shrinking color (the bar) over the background that replaces it."""
    moves = len(series) - 1
    if moves < BUDGET_MIN_MOVES:
        return None
    best = None
    for line in series[0]:
        length = sum(series[0][line].values())
        for color in set(series[0][line]) | set(series[-1][line]):
            vals = [s[line].get(color, 0) for s in series]
            change = vals[-1] - vals[0]
            down = all(b <= a for a, b in zip(vals, vals[1:]))
            up = all(b >= a for a, b in zip(vals, vals[1:]))
            if abs(change) < BUDGET_MIN_CHANGE or max(vals) < BUDGET_MIN_SIZE or not (down or up):
                continue
            score = (abs(change), down, line[0] == "row", -line[1])
            if best is None or score > best[0]:
                rate = abs(change) / moves
                left = vals[-1] if down else length - vals[-1]
                best = (score, dict(line=line, color=color, vals=vals, down=down, rate=rate, left=left, moves=moves))
    return best[1] if best else None


def ah_budget(agent, history_entries, level):
    """One line about a shrinking/growing HUD bar in the current attempt on this level, or None."""
    cache = agent.__dict__.setdefault("_bg_cache", {})
    attempt, attempt_level, linked_deaths = [], None, 0
    for e in history_entries:
        f = getattr(e, "frame", None)
        if f is None:
            continue
        rows = len(f.grid)
        cols = len(f.grid[0]) if rows else 0
        if not rows or not cols:
            continue
        key = ("c", f.step)
        if key not in cache:
            cache[key] = _bg_counts(f.grid, _bg_lines(rows, cols))
        result = getattr(e, "result", {}) or {}
        action = str(getattr(e, "action", "") or "").strip().upper()
        if result.get("game_over"):
            det = _bg_detect([cache[("c", x.step)] for x in attempt])
            if det and det["down"] and det["left"] <= max(2.0, 2 * det["rate"]):
                linked_deaths += 1
            attempt, attempt_level = [], None
            continue
        if action == "RESET" or result.get("automatic") or f.level != attempt_level:
            attempt, attempt_level = [f], f.level
        else:
            attempt.append(f)
    if attempt_level != level:
        return None, linked_deaths
    det = _bg_detect([cache[("c", x.step)] for x in attempt])
    if det is None:
        return None, linked_deaths
    kind, idx = det["line"]
    text = (f"MOVE-BUDGET METER (measured by the harness): the '{_ah_color(det['color'])}' cells in {kind} {idx} went "
            f"{det['vals'][0]} -> {det['vals'][-1]} over the {det['moves']} moves of this attempt "
            f"({'-' if det['down'] else '+'}{det['rate']:.2f} per move)")
    if det["rate"] > 0:
        text += f"; at this rate it {'runs out' if det['down'] else 'fills up'} in about {int(det['left'] / det['rate'])} more moves"
    text += "."
    if linked_deaths:
        text += (f" Earlier in this game, {linked_deaths} GAME OVER(s) happened right when such a bar ran out"
                 f" - treat it as a hard move limit.")
    return text, linked_deaths


def install_budget_meter():
    import inference.agent.tool_agent as _ta
    agent_cls = _ta.ToolAgent
    inner = agent_cls._build_user_prompt
    if getattr(inner, "_bg_wrapped", False):
        return False

    def _build_user_prompt(self, action_num, *args, **kwargs):
        out = inner(self, action_num, *args, **kwargs)
        try:
            frame = kwargs.get("current_frame")
            level = getattr(frame, "level", None)
            if isinstance(level, int):
                line, linked = ah_budget(self, kwargs.get("history_entries") or [], level)
                if linked > self.__dict__.get("_bg_linked_seen", 0):
                    _mx_inc("budget_deaths_linked", linked - self.__dict__.get("_bg_linked_seen", 0))
                    self._bg_linked_seen = linked
                if line:
                    out += "\n\n" + line
                    _mx_inc("budget_lines")
        except Exception as exc:  # never break a turn because of the add-on
            _mx_inc("budget_errors")
            _mx_err("budget", exc)
        return out

    for name in dir(inner):
        if name.endswith("_wrapped"):
            setattr(_build_user_prompt, name, getattr(inner, name))
    _build_user_prompt._bg_wrapped = True
    agent_cls._build_user_prompt = _build_user_prompt
    return True


if ENABLE_BUDGET_METER:
    # installed right after the action hints and BEFORE the solved-level memory (memory's ask stays last)
    print("move-budget meter installed:", install_budget_meter())


# ---------------------------------------------------------------- G. level-progress facts (v23; v10's stuck check, information only)
# Analysis of 30 game runs (v7-v9): levels that needed >= 80K tokens hold 28-52% of all tokens, and the SAME level
# costs 20-40K in one run and 100-140K in another (m0r0 L1 21/19/131K, s5i5 L2 50/23/124K, vc33 L4 37/33/134K), so a
# stalled level is the model's path, not the level. Read in the transcripts: m0r0 assumed - never tested - that its two
# blocks cannot overlap, while overlapping them WAS the goal (110K tokens of wrong goals); vc33 saw a window turn
# orange and theorised for ~30 moves before clicking it, and the click solved the level. Only on a level that has
# already taken SB_FIRST_TOKENS without clearing (30% of levels in v7-v9 replays; levels going well are never touched), the turn gets
# the harness's interaction coverage for this level: actions used and never tried, colors clicked, colors never clicked
# (newly appeared ones first). v23: facts only. v10 also explained why levels stall and asked for 1-3 probe actions;
# on the leaderboard every version that told the model how to behave scored lower, so the advice and the ask are gone.
SB_FIRST_TOKENS = 40000     # generated tokens on the level before the first check (median level 21K, p75 39K)
SB_EVERY_TOKENS = 40000     # then again after this many more, at most SB_MAX_CHECKS per level
SB_MAX_CHECKS = 3
SB_MAX_COLORS = 6
_SB_SKIP_ACTIONS = {"RESET", "UNDO"}
_SB_CLICK_NAMES = {"MOUSE", "ACTION6", "CLICK"}
_SB_MODEL_NAMES = {"ACTION1": "UP", "ACTION2": "DOWN", "ACTION3": "LEFT", "ACTION4": "RIGHT", "ACTION5": "SPACE",
                   "ACTION6": "MOUSE", "ACTION7": "UNDO"}
ADDON_STATS.update({"sb_checks": 0, "sb_levels": 0, "sb_cleared_after": 0, "sb_tokens_after_check": 0,
                    "sb_functions_kept": 0, "sb_errors": 0, "sb_last_error": ""})


def _sb_tokens(agent):
    try:
        return int(getattr(agent, "_session_generated_tokens", 0) or 0)
    except (TypeError, ValueError):
        return 0


def _sb_name(action):
    m = _AH_CLICK_RE.match(str(action))
    return (m.group(1) if m else str(action).strip().split("(", 1)[0].strip()).upper()


def _sb_colors(cache, frame):
    """Cells per color inside the border, cached per frame."""
    key = ("sbc", frame.step, frame.level)
    got = cache.get(key)
    if got is None:
        rows = len(frame.grid)
        cols = len(frame.grid[0]) if rows else 0
        b = _ah_border(rows, cols)
        got = _ah_collections.Counter()
        for r in range(b, rows - b):
            got.update(frame.grid[r][b:cols - b])
        cache[key] = got
    return got


def sb_coverage(agent, history_entries, level, valid_actions):
    """Lines of facts: which actions / clicked colors were used on this level, and which colors were never clicked."""
    cache = agent.__dict__.setdefault("_sb_cache", {})
    tried = _ah_collections.Counter()
    clicked = _ah_collections.Counter()
    first_seen = {}
    last = None
    moves = 0
    for i, cur in enumerate(history_entries):
        pf = getattr(history_entries[i - 1], "frame", None) if i else None
        cf = getattr(cur, "frame", None)
        if cf is None or cf.level != level:
            continue
        if last is None:  # the level's first frame (game start, or the board right after the previous level)
            for color in _sb_colors(cache, cf):
                first_seen.setdefault(color, 0)
        last = cf
        action = str(getattr(cur, "action", "") or "").strip()
        result = getattr(cur, "result", {}) or {}
        if not action or pf is None or pf.level != level or action.upper() == "RESET" or result.get("automatic"):
            continue
        moves += 1
        tried[_sb_name(action)] += 1
        m = _AH_CLICK_RE.match(action)
        if m:
            try:
                clicked[pf.grid[int(m.group(2))][int(m.group(3))]] += 1
            except (IndexError, TypeError):
                pass
        if not result.get("game_over"):  # a death screen is not part of the level
            for color in _sb_colors(cache, cf):
                first_seen.setdefault(color, moves)
    if last is None:
        return [], 0
    names = []
    for a in valid_actions or []:  # the engine's names (ACTION1-7) -> the names the model uses (v10 said "ACTION6")
        n = _SB_MODEL_NAMES.get(str(a).strip().upper(), str(a).strip().upper())
        if n and n not in names:
            names.append(n)
    lines = []
    if tried:
        line = "- Actions you used on this level: " + ", ".join(f"{n} x{k}" for n, k in tried.most_common())
        never = [n for n in names if n not in tried and n not in _SB_SKIP_ACTIONS]
        lines.append(line + (f" (never tried on this level: {', '.join(never)})." if never else "."))
    elif names:
        lines.append("- You have not taken any action on this level yet.")
    if clicked or any(n in _SB_CLICK_NAMES for n in names):
        now = _sb_colors(cache, last)
        background = now.most_common(1)[0][0] if now else None
        if clicked:
            lines.append("- Colors you clicked on this level: "
                         + ", ".join(f"'{_ah_color(c)}' x{k}" for c, k in clicked.most_common(SB_MAX_COLORS)) + ".")
        untouched = [c for c in first_seen if c not in clicked and c != background]
        # objects that appeared during the level first (a new or changed thing is the classic missed probe), then rare colors
        untouched.sort(key=lambda c: (first_seen[c] == 0, now.get(c, 0), c))
        parts = []
        for c in untouched[:SB_MAX_COLORS]:
            n = now.get(c, 0)
            where = f"{n} cells now" if n else "not on the board now"
            if first_seen[c]:
                where = f"first appeared after move {first_seen[c]} of this level; " + where
            parts.append(f"'{_ah_color(c)}' ({where})")
        if parts:
            lines.append("- Colors on this level you never clicked: " + ", ".join(parts) + ".")
    return lines, moves


def sb_block(tokens_k, level, coverage_lines, moves=None):
    done = f" and {moves} actions" if moves else ""
    head = (f"LEVEL PROGRESS (measured by the harness): level {level} is not cleared yet after about {tokens_k}K "
            f"tokens of work{done} on it.")
    body = ("\nInteraction coverage on this level:\n" + "\n".join(coverage_lines)) if coverage_lines else ""
    return head + body


def install_stuck_check():
    import inference.agent.tool_agent as _ta
    agent_cls = _ta.ToolAgent
    inner = agent_cls._build_user_prompt
    if getattr(inner, "_sb_wrapped", False):
        return False

    def _build_user_prompt(self, action_num, *args, **kwargs):
        out = inner(self, action_num, *args, **kwargs)
        try:
            level = getattr(kwargs.get("current_frame"), "level", None)
            if not isinstance(level, int):
                return out
            now = _sb_tokens(self)
            st = self.__dict__.get("_sb_state")
            if st is not None and (now < st["start"] or level != st["level"]):
                if level > st["level"] and now >= st["start"] and st["first_check_at"] is not None:
                    _mx_inc("sb_cleared_after")
                    _mx_inc("sb_tokens_after_check", now - st["first_check_at"])
                st = None
            if st is None:
                st = {"level": level, "start": now, "checks": 0, "first_check_at": None}
                self._sb_state = st
            spent = now - st["start"]
            due = SB_FIRST_TOKENS + st["checks"] * SB_EVERY_TOKENS
            if st["checks"] < SB_MAX_CHECKS and spent >= due:
                lines, moves = sb_coverage(self, kwargs.get("history_entries") or [], level, kwargs.get("valid_actions"))
                out += "\n\n" + sb_block(max(1, spent // 1000), level, lines, moves)
                if st["checks"] == 0:
                    st["first_check_at"] = now
                    _mx_inc("sb_levels")
                st["checks"] += 1
                _mx_inc("sb_checks")
        except Exception as exc:  # never break a turn because of the add-on
            _mx_inc("sb_errors")
            _mx_err("sb", exc)
        return out

    for name in dir(inner):
        if name.endswith("_wrapped"):
            setattr(_build_user_prompt, name, getattr(inner, name))
    _build_user_prompt._sb_wrapped = True
    agent_cls._build_user_prompt = _build_user_prompt
    return True


# ---------------------------------------------------------------- H. keep retained functions when a python call times out (v10)
# The sandbox keeps the model's helper functions between python calls - but when a call hits the 30 s limit (or the
# sandbox process dies) the result carries no function list and the harness drops EVERY retained function. 11 such
# timeouts in the v7-v9 runs, all in BFS/search calls on hard levels; ka59 then abandoned its simulator ("All my
# retained functions were dropped") and planned by hand. A stopped call defined nothing new, so the functions kept
# before it are still exactly right: keep them and say so. Normal errors were never affected (they report their list).
def install_keep_functions_on_timeout():
    import inference.agent.tool_agent as _ta
    agent_cls = _ta.ToolAgent
    inner = getattr(agent_cls, "_record_retained_functions", None)
    if inner is None or getattr(inner, "_sb_wrapped", False):
        return False

    def _record_retained_functions(self, sandbox_result, payload):
        try:
            kept = getattr(self, "_kept_functions", None)
            if (isinstance(sandbox_result, dict) and "keepable_functions" not in sandbox_result
                    and sandbox_result.get("error") and isinstance(kept, dict) and kept):
                payload["function_retention"] = (
                    "Your retained functions were kept unchanged (" + ", ".join(sorted(kept)) + "): this call was "
                    "stopped before it finished, so nothing it defined was saved, and nothing earlier was lost.")
                _mx_inc("sb_functions_kept")
                return None
        except Exception as exc:
            _mx_inc("sb_errors")
            _mx_err("sb", exc)
        return inner(self, sandbox_result, payload)

    _record_retained_functions._sb_wrapped = True
    agent_cls._record_retained_functions = _record_retained_functions
    return True


if ENABLE_STUCK_FACTS:
    # installed after the hints and the budget meter and BEFORE the solved-level memory (memory's ask stays last)
    print("level-progress facts installed:", install_stuck_check())
if ENABLE_KEEP_FUNCTIONS_ON_TIMEOUT:
    print("keep functions on timeout installed:", install_keep_functions_on_timeout())


# ---------------------------------------------------------------- S. second opinion on a stuck level (v24) + collective memory (v25) + bounded answer (v26) + background (v28)
# Most lost points are levels the model never clears, and the SAME level costs 20-40K tokens in one run and 120-180K
# in another (vc33 level 4: 31K in v21, 123K in v22; tn36 level 2: 134K, 169K, 181K): the model locks onto one idea and
# its own long reasoning history keeps it there. A fresh model instance that has NOT read that history is not locked
# in. When the level-progress facts (section G) fire - a level past 40K, then 80K generated tokens without clearing -
# the harness asks one: it sees the level-start board, the current board, the action meanings and the measured facts
# (what each action did on this level, what was never tried, the budget bar, the rules of earlier levels), and answers
# with up to 3 alternative hypotheses, each with its cheapest test. The playing model gets that answer as information,
# right after the facts. Its own history and its prefix cache are untouched: the side request goes straight to the
# harness's unwrapped _chat_completion (never through the exact-replay wrapper, whose reply map it must not touch),
# and its generated tokens are charged to the game (they are real GPU time) but not to the turn's yield budget.
# v25 COLLECTIVE MEMORY: every rule a game states when it clears a level ('Rule for level N: ...', captured by the
# solved-level memory) goes into a run-wide pool, and the second opinion of a DIFFERENT game sees the newest rule of up
# to 10 other games. The hidden games come from one design team, so a mechanic one game had to discover is a fair
# guess for another; the pool adapts to the hidden set itself during the run. The second opinion also gets the
# knowledge-only design catalogue (v22's text) in its system prompt. Only the second opinion reads either: the
# playing model's own prompt is unchanged.
# v26 BOUNDED ANSWER: in the v24 Save & Run all 17 second opinions came back EMPTY - each spent the whole 8,192-token
# cap thinking (finish=length) for 133-174 s. Now the side request asks for short thinking (reasoning_effort 'low',
# 3,072 tokens) and, if that still ends without an answer, retries once without thinking (700 tokens).
# v27 GRID NOTE: one v25 answer gave pixel positions of the 10x-enlarged image ('(107,273)'); the request now states
# the grid size and the image scale and asks for grid (row, col) positions only.
# v28 BACKGROUND: waiting for the answer left the game idle 17-37 s, and in v24-v26 the request right after a second
# opinion re-read its whole cached prompt in 15-18 of ~24 cases (v23, same facts without a second opinion: 2 of 30).
# The game's own request now goes out at once; a background thread fetches the second opinion and the answer is
# appended to the game's NEXT turn (dropped if the level was cleared meanwhile). The background request is built from
# read-only agent fields and never changes the agent, so the player's concurrent requests keep their own settings.
import json as _so_json
import threading as _so_threading
import time as _so_time

SO_MAX_PER_LEVEL = 2
SO_THINK_TOKENS = 3072      # first try: thinking at reasoning_effort SO_EFFORT
SO_EFFORT = "low"
SO_DIRECT_TOKENS = 700      # fallback: no thinking, answer directly
SO_TIMEOUT_S = 300.0
SO_ANSWER_CHARS = 1500
SO_ANSWER_LINES = 10
SO_TRIGGER = "LEVEL PROGRESS (measured by the harness): level "
SO_HEADER = ("SECOND OPINION (a fresh model instance that has not seen your reasoning looked at the level-start board, "
             "the current board and the measured facts; possibilities, not facts):")
SO_SYSTEM = ("You are an expert player of grid-based puzzle video games (ARC-AGI-3). Each game has several levels; a "
             "level is cleared by reaching its goal, which the player has to discover by experimenting - nothing is "
             "explained. You give a short second opinion to a player who has been stuck on one level for a long time. "
             "You have not seen the player's reasoning, only the two boards and the facts the game harness measured. "
             "Think briefly, then answer.")
SO_ASK = ("What might the player be missing? Give up to 3 different, specific hypotheses about this level's goal or "
          "hidden mechanic that fit the facts, most likely first. For each, one line: the hypothesis (name objects by "
          "color and position) and the cheapest way to test it in 1-3 actions. The measured facts about this level "
          "outweigh any pattern from other games. At most 8 short lines, no preamble.")
ADDON_STATS.update({"so_asks": 0, "so_answers": 0, "so_empty": 0, "so_errors": 0, "so_tokens": 0,
                    "so_pool_rules": 0, "so_shared_shown": 0, "so_fallbacks": 0, "so_delivered": 0,
                    "so_discarded": 0, "so_dropped": 0, "so_last_error": ""})
SO_MAX_WAIT_S = 600.0       # an answer not back after this long is dropped (the thread may still finish; ignored)
_SO_PENDING_LOCK = _so_threading.Lock()
_SO_LOG_LOCK = _so_threading.Lock()
# Save & Run only: every question and answer goes to second_opinions.jsonl for analysis (never in a competition rerun)
SECOND_OPINION_LOG = (None if globals().get("TRUE_SUBMISSION", True)
                      else str(globals().get("WORKING_DIR", "/kaggle/working")) + "/second_opinions.jsonl")


SO_POOL_GAMES = 10          # the second opinion sees the newest rule of at most this many OTHER games
SO_POOL_RULE_CHARS = 300
SO_POOL_MAX = 500
_SO_POOL = []               # {"agent": id(agent), "level": n, "rule": text}, newest last
_SO_POOL_LOCK = _so_threading.Lock()
_so_m85_capture_inner = _m85_capture


def _m85_capture(agent):
    """The solved-level memory's capture (it calls this name at run time), plus: newly stated rules go to the pool."""
    changed = _so_m85_capture_inner(agent)
    try:
        if changed:
            pooled = agent.__dict__.setdefault("_so_pooled", set())
            for rec in getattr(agent, "_m85_levels", None) or []:
                level = rec.get("level")
                if rec.get("final") and rec.get("rule") and level not in pooled:
                    pooled.add(level)
                    rule = " ".join(str(rec["rule"]).split())[:SO_POOL_RULE_CHARS]
                    with _SO_POOL_LOCK:
                        _SO_POOL.append({"agent": id(agent), "level": level, "rule": rule})
                        del _SO_POOL[:-SO_POOL_MAX]
                    _mx_inc("so_pool_rules")
    except Exception as exc:  # never break a turn because of the add-on
        _mx_inc("so_errors")
        _mx_err("so", exc)
    return changed


def so_shared_rules(agent):
    """Newest pooled rule of up to SO_POOL_GAMES other games (one per game), newest first."""
    if not globals().get("ENABLE_SO_SHARED_RULES", True):
        return []
    with _SO_POOL_LOCK:
        pool = list(_SO_POOL)
    out, games = [], set()
    for rec in reversed(pool):
        if rec["agent"] == id(agent) or rec["agent"] in games:
            continue
        games.add(rec["agent"])
        out.append(f"- (another game, its level {rec['level']}) {rec['rule']}")
        if len(out) >= SO_POOL_GAMES:
            break
    return out


def so_system():
    text = SO_SYSTEM
    catalogue = gc_reader_text() if globals().get("ENABLE_SO_CATALOGUE", True) else None
    if catalogue:
        text += "\n\n" + catalogue
    return text


def _so_post(agent, messages, max_tokens, effort=None, thinking=True, temperature=None):
    """One side request -> (answer text, finish_reason, usage). Built only from read-only agent fields (model, base
    URL, headers, the class's template kwargs) so it is safe on a background thread while the player keeps playing."""
    import requests as _so_requests
    import inference.agent.tool_agent as _ta
    payload = _ta.build_chat_payload(
        provider=agent._model.provider, model=agent._model.model_id, messages=messages, max_tokens=max_tokens,
        temperature=_ta._LOCAL_ANALYZER_TEMPERATURE if temperature is None else float(temperature),
        top_p=_ta._LOCAL_ANALYZER_TOP_P, top_k=_ta._LOCAL_ANALYZER_TOP_K,
        thinking=bool(thinking), tools=None, seed=_ta._LOCAL_ANALYZER_SEED)
    payload["stream"] = False
    payload.pop("stream_options", None)
    kwargs = dict(payload.get("chat_template_kwargs") or {})
    kwargs.update(type(agent)._harness_template_kwargs(agent))      # e.g. preserve_thinking, as the player sends
    if effort:
        kwargs["reasoning_effort"] = effort
    if not thinking:
        kwargs["enable_thinking"] = False
    payload["chat_template_kwargs"] = kwargs
    response = _so_requests.post(f"{agent._model.base_url.rstrip('/')}/chat/completions", headers=agent._headers(),
                                 json=payload, timeout=(_ta._CONNECT_TIMEOUT_SECONDS, SO_TIMEOUT_S))
    response.raise_for_status()
    data = response.json()
    choices = data.get("choices") or []
    if not choices:
        raise RuntimeError(f"no choices: {str(data)[:200]}")
    msg = choices[0].get("message") or {}
    return _so_clean(msg.get("content") or ""), str(choices[0].get("finish_reason") or ""), data.get("usage")


def _so_action_meanings(agent):
    sp = getattr(agent, "_system_prompt", "") or ""
    i = sp.find("Action meanings (use only actions currently available):")
    if i < 0:
        return ""
    j = sp.find("\n\n", i)
    return sp[i:j if j > 0 else i + 1500].strip()


def so_facts(agent, history_entries, level, valid_actions, tokens_k, moves):
    """The text the second opinion sees (everything here is measured by the harness or stated by the game)."""
    parts = [f"The player has been on level {level} for about {tokens_k}K tokens of thinking and {moves} actions "
             "without clearing it."]
    meanings = _so_action_meanings(agent)
    if meanings:
        parts.append(meanings)
    names = []
    for a in valid_actions or []:
        n = _SB_MODEL_NAMES.get(str(a).strip().upper(), str(a).strip().upper())
        if n and n not in names:
            names.append(n)
    if names:
        parts.append("Actions available now: " + ", ".join(names) + ".")
    rules = [f"- Level {r.get('level')}: {r.get('rule')}" for r in (getattr(agent, "_m85_levels", None) or [])
             if isinstance(r, dict) and r.get("final") and r.get("rule")]
    if rules:
        parts.append("How the earlier levels of this game were cleared (the player's own rules):\n" + "\n".join(rules))
    shared = so_shared_rules(agent)
    if shared:
        _mx_inc("so_shared_shown")
        parts.append("Rules that OTHER games of this benchmark turned out to have, as stated by the players that cleared "
                     "those levels during this run (different games by the same designers; inspiration, not facts):\n"
                     + "\n".join(shared))
    effects, _sig, _n = ah_level_summary(agent, history_entries, level)
    if effects:
        parts.append("What each action did on this level (measured, most used first):\n" + "\n".join(effects))
    coverage, _m = sb_coverage(agent, history_entries, level, valid_actions)
    if coverage:
        parts.append("Interaction coverage on this level:\n" + "\n".join(coverage))
    budget, _deaths = ah_budget(agent, history_entries, level)
    if budget:
        parts.append(budget)
    colors = ", ".join(f"{k}={v}" for k, v in enumerate(_AH_COLORS))
    parts.append(f"Color letters used above: {colors}.")
    return "\n\n".join(parts)


def _so_clean(text):
    lines = [ln.rstrip() for ln in str(text or "").strip().splitlines() if ln.strip()]
    return "\n".join(lines[:SO_ANSWER_LINES])[:SO_ANSWER_CHARS].strip()


def _so_log(record):
    path = SECOND_OPINION_LOG
    if not path:
        return
    try:
        with _SO_LOG_LOCK:
            with open(path, "a", encoding="utf-8") as f:
                f.write(_so_json.dumps(record) + "\n")
    except Exception:
        pass


def so_grid_note(frame):
    """How the board images map to grid cells (the player acts on grid cells, never on image pixels)."""
    rows = len(frame.grid) if frame is not None else 64
    cols = len(frame.grid[0]) if frame is not None and rows else 64
    try:
        from inference.agent.vision_context import current_grid_image_upscale
        scale = int(current_grid_image_upscale())
    except Exception:
        scale = 10
    return (f"The two images below show the {rows}x{cols} game grid enlarged {scale}x: every grid cell is a "
            f"{scale}x{scale}-pixel square. Give every position as a grid (row, col) with row 0-{rows - 1} and col "
            f"0-{cols - 1}, never in image pixels.")


def so_build(agent, history_entries, level, valid_actions, frame, tokens_k, moves):
    """(messages, facts) of the side request - computed on the player's thread from its history."""
    from inference.agent.vision_context import current_grid_image_part
    start = next((h.frame for h in history_entries
                  if getattr(h, "frame", None) is not None and h.frame.level == level), None)
    facts = so_facts(agent, history_entries, level, valid_actions, tokens_k, moves)
    content = [{"type": "text", "text": facts + "\n\n" + so_grid_note(frame) + "\n\nBoard at the start of this level:"}]
    for label, f in (("", start), ("Board now:", frame)):
        if label:
            content.append({"type": "text", "text": label})
        part = current_grid_image_part(f) if f is not None else None
        if part:
            content.append(part)
    content.append({"type": "text", "text": SO_ASK})
    messages = [{"role": "system", "content": so_system()}, {"role": "user", "content": content}]
    return messages, facts


def _so_worker(agent, messages, pend):
    """Background thread: the side request (short thinking, then a direct fallback). Writes only into `pend`."""
    t0 = _so_time.time()
    answer, finish, usage, finish2, usage2, error = "", "", None, None, None, None
    try:
        answer, finish, usage = _so_post(agent, messages, SO_THINK_TOKENS, effort=SO_EFFORT, thinking=True)
        if not answer:                   # thinking ran out (or no text): one direct answer, no thinking
            _mx_inc("so_fallbacks")
            answer, finish2, usage2 = _so_post(agent, messages, SO_DIRECT_TOKENS, effort=None, thinking=False)
    except Exception as exc:
        error = exc
        _mx_inc("so_errors")
        _mx_err("so", exc)
    usages = [u for u in (usage, usage2) if isinstance(u, dict)]
    for u in usages:
        try:
            _mx_inc("so_tokens", int(u.get("completion_tokens") or 0))
        except (TypeError, ValueError):
            pass
    if answer:
        _mx_inc("so_answers")
    elif error is None:
        _mx_inc("so_empty")
    _so_log({"level": pend.get("level"), "tokens_k": pend.get("tokens_k"), "moves": pend.get("moves"),
             "facts": pend.get("facts"), "answer": answer, "finish": finish, "usage": usage,
             "fallback_finish": finish2, "fallback_usage": usage2, "error": repr(error)[:200] if error else None,
             "seconds": round(_so_time.time() - t0, 1)})
    with _SO_PENDING_LOCK:
        pend["answer"] = answer
        pend["usages"] = usages
        pend["done"] = True


def _so_settle(agent, pend):
    """On the player's thread: charge the side request's tokens to the game (real GPU time, not this turn's budget)."""
    for u in pend.get("usages") or []:
        try:
            agent._accumulate_usage_tokens(u, count_toward_turn=False)
        except Exception:
            pass


def install_second_opinion():
    import inference.agent.tool_agent as _ta
    agent_cls = _ta.ToolAgent
    inner = agent_cls._build_user_prompt
    if getattr(inner, "_so_wrapped", False):
        return False

    def _build_user_prompt(self, action_num, *args, **kwargs):
        out = inner(self, action_num, *args, **kwargs)
        try:
            st = self.__dict__.get("_sb_state")
            frame = kwargs.get("current_frame")
            level = getattr(frame, "level", None)
            # 1. an answer from an earlier turn: deliver it if the same level attempt is still running
            pend = self.__dict__.get("_so_pending")
            if pend is not None:
                with _SO_PENDING_LOCK:
                    done = bool(pend.get("done"))
                if done:
                    self._so_pending = None
                    _so_settle(self, pend)
                    if pend.get("answer") and pend.get("st") is st and level == pend.get("level"):
                        out += "\n\n" + SO_HEADER + "\n" + pend["answer"]
                        _mx_inc("so_delivered")
                    elif pend.get("answer"):
                        _mx_inc("so_discarded")              # the level was cleared (or a new game began) meanwhile
                elif _so_time.time() - pend.get("t0", 0.0) > SO_MAX_WAIT_S:
                    self._so_pending = None
                    _mx_inc("so_dropped")
            # 2. the level-progress facts fired this turn: start a background second opinion
            if (SO_TRIGGER in out and self.__dict__.get("_so_pending") is None and isinstance(st, dict)
                    and isinstance(level, int) and st.get("level") == level and int(st.get("so", 0)) < SO_MAX_PER_LEVEL):
                st["so"] = int(st.get("so", 0)) + 1      # counted first: a failure never triggers a retry storm
                hist = kwargs.get("history_entries") or []
                spent_k = max(1, (_sb_tokens(self) - int(st.get("start", 0))) // 1000)
                moves = sum(1 for p, c in zip(hist, hist[1:])
                            if getattr(c, "frame", None) is not None and getattr(p, "frame", None) is not None
                            and c.frame.level == level and p.frame.level == level
                            and str(getattr(c, "action", "") or "").strip().upper() != "RESET"
                            and not (getattr(c, "result", {}) or {}).get("automatic"))
                messages, facts = so_build(self, hist, level, kwargs.get("valid_actions"), frame, spent_k, moves)
                pend = {"level": level, "st": st, "t0": _so_time.time(), "done": False, "answer": "", "usages": [],
                        "tokens_k": spent_k, "moves": moves, "facts": facts}
                self._so_pending = pend
                _mx_inc("so_asks")
                _so_threading.Thread(target=_so_worker, args=(self, messages, pend), daemon=True,
                                     name="second-opinion").start()
        except Exception as exc:  # never break a turn because of the add-on
            _mx_inc("so_errors")
            _mx_err("so", exc)
        return out

    for name in dir(inner):
        if name.endswith("_wrapped"):
            setattr(_build_user_prompt, name, getattr(inner, name))
    _build_user_prompt._so_wrapped = True
    agent_cls._build_user_prompt = _build_user_prompt
    return True


if ENABLE_SECOND_OPINION:
    # installed after the level-progress facts (it answers them) and BEFORE the solved-level memory (memory stays last)
    print("second opinion installed:", install_second_opinion())


# ---------------------------------------------------------------- R. rival first impressions at every level start (v55; v57: asks only once the server is up; v58: on the level's first turn + what is new; v59: own temperature)
# The same model clears a level in one run and stalls on it for 100K+ tokens in another (26 of the 41 levels our runs
# never cleared were cleared by the same model in another run, often within 9-21 moves). Reading both runs of tn36
# level 2: the stuck run discovered nearly every mechanic but chose a complicated reading of it early and kept it for
# 359K tokens; the run that cleared it tested the simplest reading first ("the side panel shows the recipe - copy
# it"). The second opinion (section S) brings a fresh view only after 40K tokens on a level - after the lock-in. At
# the FIRST turn of every level, FI_SAMPLES fresh model instances (no history, sampled independently at the harness
# temperature) look at the level's first board, the controls and the rules of the earlier levels, and each gives up to
# 3 hypotheses about the goal with the cheapest test. v58: the level's first turn waits for them (v55's Save & Run:
# answers took 8-39 s, median 23 s, but arrived 1-3 turns late - after the player had formed its own reading - and
# were used in ~1 of 22 deliveries), and the fresh readers (first impressions + second opinion, NOT the player) are
# told which colours on the level's first board appeared on no board of the earlier levels. Tokens are charged to
# the game.
FI_SAMPLES = 2
FI_MAX_WAIT_S = 240.0       # deliver what came back by then; nothing back -> dropped
FI_SYNC_WAIT_S = 75.0       # v58: the level's first turn waits this long for the answers (v55: they came 1-3 turns
                            # late, after the player had its own reading; it used them in ~1 of 22 turns)
FI_NEW_MAX_LINES = 5        # v58: at most this many new colours are listed (largest first)
FI_NEW_HEADER = ("New on this level (measured by the harness: colours on this level's first board that appeared on no "
                 "board the player saw during the earlier levels of this game):")
FI_HEADER = ("FIRST IMPRESSIONS (fresh model instances that see only this level's first board, the controls and your "
             "rules of the earlier levels - not your reasoning; ideas to test cheaply, not facts):")
FI_SYSTEM = ("You are an expert player of grid-based puzzle video games (ARC-AGI-3). Each game has several levels; a "
             "level is cleared by reaching its goal, which the player has to discover by experimenting - nothing is "
             "explained. Levels build on the mechanics of the earlier ones and usually add something new. You give a "
             "quick first impression of a NEW level to a player who is about to start it. Think briefly, then answer.")
FI_ASK = ("Give up to 3 different, specific hypotheses about this level's goal and how to reach it, most likely first. "
          "For each, one line: the hypothesis (name objects by color and grid position) and the cheapest way to test "
          "it in 1-3 actions. Look for what the player controls, what is NEW compared with the earlier levels (new "
          "elements usually matter), and any part of the board that could be a target, a preview, a hint, a recipe or "
          "a control panel. Prefer the simplest explanation. At most 8 short lines, no preamble.")
ADDON_STATS.update({"fi_asks": 0, "fi_answers": 0, "fi_empty": 0, "fi_errors": 0, "fi_tokens": 0,
                    "fi_delivered": 0, "fi_discarded": 0, "fi_dropped": 0, "fi_waiting_server": 0, "fi_retries": 0,
                    "fi_sync_delivered": 0, "fi_wait_ms": 0, "fi_new_lists": 0, "fi_new_lines": 0,
                    "fi_last_error": ""})
_FI_LOCK = _so_threading.Lock()
FIRST_IMPRESSIONS_LOG = (None if globals().get("TRUE_SUBMISSION", True)
                         else str(globals().get("WORKING_DIR", "/kaggle/working")) + "/first_impressions.jsonl")



def _fi_track(agent, frame, history_entries):
    """v58: the colours of every board the player saw, per level, and each level's first board. Incremental: only the
    history entries added since the last call are read (all of them again if the history got shorter)."""
    seen = agent.__dict__.setdefault("_fi_seen", {})
    firsts = agent.__dict__.setdefault("_fi_first", {})
    entries = history_entries or []
    pos = agent.__dict__.get("_fi_scan_pos", 0)
    if pos > len(entries):
        pos = 0
    frames = [getattr(h, "frame", None) for h in entries[pos:]]
    agent._fi_scan_pos = len(entries)
    if frame is not None:
        frames.append(frame)
    for f in frames:
        lv = getattr(f, "level", None)
        grid = getattr(f, "grid", None)
        if not isinstance(lv, int) or grid is None or not len(grid):
            continue
        colors = seen.setdefault(lv, set())
        for row in grid:
            colors.update(int(v) for v in row)
        if lv not in firsts:
            firsts[lv] = grid


def _fi_components(grid, color):
    """4-connected objects of one colour on the whole board, largest first: [[(row, col), ...], ...]."""
    rows = len(grid)
    cols = len(grid[0]) if rows else 0
    seen, out = set(), []
    for r in range(rows):
        for c in range(cols):
            if (r, c) in seen or int(grid[r][c]) != color:
                continue
            stack, cells = [(r, c)], []
            seen.add((r, c))
            while stack:
                y, x = stack.pop()
                cells.append((y, x))
                for ny, nx in ((y + 1, x), (y - 1, x), (y, x + 1), (y, x - 1)):
                    if 0 <= ny < rows and 0 <= nx < cols and (ny, nx) not in seen and int(grid[ny][nx]) == color:
                        seen.add((ny, nx))
                        stack.append((ny, nx))
            out.append(cells)
    out.sort(key=len, reverse=True)
    return out


def _fi_box(cells):
    r0, r1 = min(p[0] for p in cells), max(p[0] for p in cells)
    c0, c1 = min(p[1] for p in cells), max(p[1] for p in cells)
    return f"rows {r0}-{r1}, cols {c0}-{c1}" if (r0, c0) != (r1, c1) else f"row {r0}, col {c0}"


def fi_new_elements(agent, level):
    """v58: the objects of the level's first board whose colour appeared on no board of the earlier levels (read
    from _fi_track's record). Cached per level once the record covers an earlier level."""
    if not isinstance(level, int) or level <= 1:
        return []
    cache = agent.__dict__.setdefault("_fi_new_cache", {})
    if level in cache:
        return cache[level]
    seen = agent.__dict__.get("_fi_seen") or {}
    grid = (agent.__dict__.get("_fi_first") or {}).get(level)
    earlier = [colors for lv, colors in seen.items() if lv < level]
    if grid is None or not earlier:
        return []
    old = set().union(*earlier)
    new = set()
    for row in grid:
        new.update(int(v) for v in row if int(v) not in old)
    found = []
    for color in new:
        objs = _fi_components(grid, color)
        found.append((sum(len(o) for o in objs), color, objs))
    found.sort(key=lambda t: (-t[0], t[1]))
    lines = []
    for total, color, objs in found[:FI_NEW_MAX_LINES]:
        name = f"'{_ah_color(color)}'"
        if len(objs) == 1:
            lines.append(f"- {name}: one object of {total} cells at {_fi_box(objs[0])}")
        elif len(objs) <= 3:
            lines.append(f"- {name}: {len(objs)} objects - " + "; ".join(f"{len(o)} cells at {_fi_box(o)}" for o in objs))
        else:
            spread = _fi_box([q for o in objs for q in o])
            lines.append(f"- {name}: {len(objs)} objects ({total} cells) spread over {spread}; the largest "
                         f"{len(objs[0])} cells at {_fi_box(objs[0])}")
    if len(found) > FI_NEW_MAX_LINES:
        lines.append(f"- ... and {len(found) - FI_NEW_MAX_LINES} more new colours")
    cache[level] = lines
    if lines:
        _mx_inc("fi_new_lists")
        _mx_inc("fi_new_lines", len(lines))
    return lines


def fi_facts(agent, level, valid_actions, new_lines=None):
    parts = [f"The player is about to start level {level} of this game."]
    meanings = _so_action_meanings(agent)
    if meanings:
        parts.append(meanings)
    names = []
    for a in valid_actions or []:
        n = _SB_MODEL_NAMES.get(str(a).strip().upper(), str(a).strip().upper())
        if n and n not in names:
            names.append(n)
    if names:
        parts.append("Actions available now: " + ", ".join(names) + ".")
    rules = [f"- Level {r.get('level')}: {r.get('rule')}" for r in (getattr(agent, "_m85_levels", None) or [])
             if isinstance(r, dict) and r.get("final") and r.get("rule")]
    if rules:
        parts.append("How the earlier levels of this game were cleared (the player's own rules):\n" + "\n".join(rules))
    shared = so_shared_rules(agent)
    if shared:
        parts.append("Rules that OTHER games of this benchmark turned out to have, as stated by the players that cleared "
                     "those levels during this run (different games by the same designers; inspiration, not facts):\n"
                     + "\n".join(shared))
    if new_lines:
        parts.append(FI_NEW_HEADER + "\n" + "\n".join(new_lines))
    colors = ", ".join(f"{k}={v}" for k, v in enumerate(_AH_COLORS))
    parts.append(f"Color letters used above: {colors}.")
    return "\n\n".join(parts)


def fi_build(agent, level, valid_actions, frame):
    from inference.agent.vision_context import current_grid_image_part
    new_lines = []
    try:
        new_lines = fi_new_elements(agent, level)
    except Exception as exc:
        _mx_inc("fi_errors")
        _mx_err("fi", exc)
    facts = fi_facts(agent, level, valid_actions, new_lines)
    content = [{"type": "text", "text": facts + "\n\n" + so_grid_note(frame) + "\n\nThe first board of this level:"}]
    part = current_grid_image_part(frame) if frame is not None else None
    if part:
        content.append(part)
    content.append({"type": "text", "text": FI_ASK})
    system = FI_SYSTEM
    catalogue = gc_reader_text() if globals().get("ENABLE_SO_CATALOGUE", True) else None
    if catalogue:
        system += "\n\n" + catalogue
    return [{"role": "system", "content": system}, {"role": "user", "content": content}], facts


def _fi_log(record):
    path = FIRST_IMPRESSIONS_LOG
    if not path:
        return
    try:
        with _FI_LOCK:
            with open(path, "a", encoding="utf-8") as f:
                f.write(_so_json.dumps(record) + "\n")
    except Exception:
        pass


FI_TEMPERATURE = 1.0         # v59: the player runs at 0.7 again; these short requests keep the temperature their
                             # answers were judged at in v55 (1.0, the model card's thinking setting: more varied)
FI_CONNECT_RETRIES = 3       # v57: a refused connection (server restarting) is retried this often
FI_CONNECT_SLEEP_S = 10.0


def _fi_server_up():
    """v57: False while the vLLM server is still loading (v55's Save & Run asked 36 times before it was up: the games
    start at the release, the server answered 14 minutes later). SGLang / unknown -> True (the worker retries)."""
    state = globals().get("VLLM_STATE")
    if globals().get("SERVER_ENGINE_USED") == "sglang" or not isinstance(state, dict) or not state:
        return True
    return state.get("status") == "ready" or bool(state.get("ready"))


def _fi_post(agent, messages, max_tokens, effort=None, thinking=True):
    """_so_post with a few retries on a refused connection (never on other errors)."""
    for attempt in range(FI_CONNECT_RETRIES + 1):
        try:
            return _so_post(agent, messages, max_tokens, effort=effort, thinking=thinking,
                            temperature=FI_TEMPERATURE)
        except Exception as exc:
            refused = ("ConnectionError" in type(exc).__name__ or "Connection refused" in str(exc)
                       or "NewConnectionError" in str(exc))
            if not refused or attempt >= FI_CONNECT_RETRIES:
                raise
            _mx_inc("fi_retries")
            _so_time.sleep(FI_CONNECT_SLEEP_S)


def _fi_worker(agent, messages, part, level):
    """Background thread: one first impression (short thinking, then a direct fallback). Writes only into `part`."""
    t0 = _so_time.time()
    answer, finish, usage, usage2, error = "", "", None, None, None
    try:
        answer, finish, usage = _fi_post(agent, messages, SO_THINK_TOKENS, effort=SO_EFFORT, thinking=True)
        if not answer:
            answer, _f2, usage2 = _fi_post(agent, messages, SO_DIRECT_TOKENS, effort=None, thinking=False)
    except Exception as exc:
        error = exc
        _mx_inc("fi_errors")
        _mx_err("fi", exc)
    usages = [u for u in (usage, usage2) if isinstance(u, dict)]
    for u in usages:
        try:
            _mx_inc("fi_tokens", int(u.get("completion_tokens") or 0))
        except (TypeError, ValueError):
            pass
    if answer:
        _mx_inc("fi_answers")
    elif error is None:
        _mx_inc("fi_empty")
    _fi_log({"level": level, "answer": answer, "finish": finish, "usage": usage, "fallback_usage": usage2,
             "error": repr(error)[:200] if error else None, "seconds": round(_so_time.time() - t0, 1)})
    with _FI_LOCK:
        part["answer"] = answer
        part["usages"] = usages
        part["done"] = True


def install_first_impressions():
    import inference.agent.tool_agent as _ta
    agent_cls = _ta.ToolAgent
    inner = agent_cls._build_user_prompt
    if getattr(inner, "_fi_wrapped", False):
        return False

    def _deliver(self, out, st, level):
        """Append the answers once all are back (or the wait is over); returns the prompt."""
        pend = self.__dict__.get("_fi_pending")
        if pend is not None:
            with _FI_LOCK:
                parts = [dict(p) for p in pend["parts"]]
            late = _so_time.time() - pend["t0"] > FI_MAX_WAIT_S
            answers = [p["answer"] for p in parts if p.get("done") and p.get("answer")]
            if all(p.get("done") for p in parts) or (late and answers):
                self._fi_pending = None
                for p in parts:
                    if p.get("done"):
                        _so_settle(self, p)
                if answers and pend.get("st") is st and level == pend.get("level"):
                    lines = [f"{chr(65 + i)}) " + a.replace("\n", "\n   ") for i, a in enumerate(answers)]
                    out += "\n\n" + FI_HEADER + "\n" + "\n".join(lines)
                    _mx_inc("fi_delivered")
                elif answers:
                    _mx_inc("fi_discarded")          # the level was cleared (or a new game began) meanwhile
            elif late:
                self._fi_pending = None
                _mx_inc("fi_dropped")
        return out

    def _build_user_prompt(self, action_num, *args, **kwargs):
        out = inner(self, action_num, *args, **kwargs)
        try:
            st = self.__dict__.get("_sb_state")
            frame = kwargs.get("current_frame")
            level = getattr(frame, "level", None)
            try:
                _fi_track(self, frame, kwargs.get("history_entries"))      # v58: what each level showed
            except Exception as exc:
                _mx_inc("fi_errors")
                _mx_err("fi", exc)
            # 1. answers from an earlier turn: deliver them once all are back (or the wait is over)
            out = _deliver(self, out, st, level)
            # 2. the first turn of a level: ask FI_SAMPLES fresh instances, and wait for them (v58)
            ready_to_ask = (isinstance(st, dict) and isinstance(level, int) and st.get("level") == level
                            and not st.get("fi") and self.__dict__.get("_fi_pending") is None and frame is not None)
            if ready_to_ask and not _fi_server_up():
                _mx_inc("fi_waiting_server")             # v57: not yet - a later turn of this level asks
            elif ready_to_ask:
                st["fi"] = True                          # set first: a failure never triggers a retry storm
                messages, facts = fi_build(self, level, kwargs.get("valid_actions"), frame)
                pend = {"level": level, "st": st, "t0": _so_time.time(),
                        "parts": [{"done": False, "answer": "", "usages": []} for _ in range(int(FI_SAMPLES))]}
                self._fi_pending = pend
                _mx_inc("fi_asks")
                for part in pend["parts"]:
                    _so_threading.Thread(target=_fi_worker, args=(self, messages, part, level), daemon=True,
                                         name="first-impression").start()
                # v58: this is the level's first look - wait for the answers so the player reads them now
                t_wait = _so_time.time()
                while _so_time.time() - t_wait < FI_SYNC_WAIT_S:
                    with _FI_LOCK:
                        if all(p.get("done") for p in pend["parts"]):
                            break
                    _so_time.sleep(0.25)
                _mx_inc("fi_wait_ms", int(1000 * (_so_time.time() - t_wait)))
                before = ADDON_STATS.get("fi_delivered", 0)
                out = _deliver(self, out, st, level)
                if ADDON_STATS.get("fi_delivered", 0) > before:
                    _mx_inc("fi_sync_delivered")
        except Exception as exc:  # never break a turn because of the add-on
            _mx_inc("fi_errors")
            _mx_err("fi", exc)
        return out

    for name in dir(inner):
        if name.endswith("_wrapped"):
            setattr(_build_user_prompt, name, getattr(inner, name))
    _build_user_prompt._fi_wrapped = True
    agent_cls._build_user_prompt = _build_user_prompt
    return True


_fi_so_facts_inner = so_facts


def so_facts(agent, history_entries, level, valid_actions, tokens_k, moves):
    """v58: the second opinion's facts + what is new on this level."""
    text = _fi_so_facts_inner(agent, history_entries, level, valid_actions, tokens_k, moves)
    if ENABLE_FIRST_IMPRESSIONS:
        try:
            _fi_track(agent, None, history_entries)
            lines = fi_new_elements(agent, level)
            if lines:
                text += "\n\n" + FI_NEW_HEADER + "\n" + "\n".join(lines)
        except Exception as exc:
            _mx_inc("fi_errors")
            _mx_err("fi", exc)
    return text


if ENABLE_FIRST_IMPRESSIONS:
    # installed after the second opinion (it reuses its request helpers and the level-progress state) and BEFORE the
    # solved-level memory (memory's ask stays last)
    try:
        print("first impressions installed:", install_first_impressions(), f"({FI_SAMPLES} per level start)")
    except Exception as _fi_exc:
        _mx_inc("fi_errors")
        _mx_err("fi", _fi_exc)
        print("first impressions NOT installed:", repr(_fi_exc)[:200])


# ---------------------------------------------------------------- X. cross-game rules for the player (v44)
# Every game of a run is played by this one process, and every level a game clears leaves that game's own 'Rule for
# level N:' paragraph (section A: the goal, what each control did, the plan that won). v25 pooled those rules, but only
# the second opinion read them, and it got the NEWEST rule of each other game - mostly a late level's details ('Same
# maze; goal = reach green. Guard dots show ...'): in v43 1 of 74 second-opinion answers used one. v44 shows the
# library to the player itself, only on a stalled level, on the turns where the level-progress facts (section G) fire
# (40K, 80K, 120K generated tokens on the level), as information with no instruction:
#  - stalled on level 1 (the game has no rule of its own yet): the level-1 rules of other games; each one describes a
#    game from scratch - its goal, its controls, what won;
#  - stalled on a later level: what later levels of other games ADDED (from the sentence that introduces the new
#    element: 'New mechanic: blue tokens are movable walls ...'), since a later level usually keeps its game's rules
#    and adds one twist.
# One rule per other game; games with the most similar controls first (MOUSE only, arrows only, arrows + SPACE, ...);
# rules this game has already been shown go last, so the next check shows others; never the game's own rules. The
# second opinion gets the same selection instead of the newest rules. In the contest the library is built from the
# hidden games themselves while they are played, so nothing in it is fitted to the public games.
# v47 PRACTICE-GAME LIBRARY (ENABLE_XG_LIBRARY): the pool also starts with the rules our own players stated when they
# cleared levels of the 25 public games (XG_LIBRARY: 25 level-1 rules + later-level additions, built by
# make_xg_library.py from nine Save & Runs). Early in a contest run few hidden games have cleared anything, so a stalled
# game used to get no block at all (xg_no_rules). Library records rank like any other rule (not shown yet first, then
# controls similarity) but count as the oldest, so an in-run rule wins a tie; their lines say 'a practice game'.
# Never a game's own rules: a game is matched by its game id (recorded per thread when _HarnessGameSession.play starts);
# in a Save & Run a game whose id is unknown gets no library rules at all.
import itertools as _xg_itertools
import json as _xg_json
import threading as _xg_threading

XG_SHOW = 5                 # rules per block for the player (one per other game)
XG_SO_SHOW = 6              # rules in the second opinion's facts (v25-v43: the newest rule of up to 10 other games)
XG_RULE_CHARS = 360         # a level-1 rule, clipped at a sentence (or word) boundary, its goal sentence first
XG_TWIST_CHARS = 260        # what a later level added: the sentence that says so and the next one
XG_MIN_CHARS = 60           # shorter captures are not rules ('as written above. Let me map level 5.')
XG_MAX_PER_LEVEL = 3        # blocks per level attempt (the facts fire at most SB_MAX_CHECKS = 3 times)
XG_POOL_MAX = 2000
XG_CONTROL_ORDER = ("UP", "DOWN", "LEFT", "RIGHT", "SPACE", "MOUSE")
XG_HEADER_L1 = ("OTHER GAMES IN THIS RUN - how level 1 of some other games of this benchmark worked, as stated by the "
                "players that cleared it (different games by the same designers; colors and positions refer to their "
                "boards; possibilities, not facts about this game):")
XG_HEADER_LN = ("OTHER GAMES IN THIS RUN - what later levels of some other games of this benchmark added, as stated by "
                "the players that cleared them (different games by the same designers; colors and positions refer to "
                "their boards; possibilities, not facts about this game):")
ADDON_STATS.update({"xg_pool": 0, "xg_pool_level1": 0, "xg_rejected": 0, "xg_blocks": 0, "xg_blocks_level1": 0,
                    "xg_rules_shown": 0, "xg_block_chars_max": 0, "xg_no_rules": 0, "xg_so_lists": 0,
                    "xg_errors": 0, "xg_last_error": "", "xg_lib_pool": 0, "xg_lib_shown": 0,
                    "xg_lib_blocks": 0, "xg_named_games": 0})
_XG_POOL = []               # {"game", "level", "rule", "twist", "controls", "seq"}, oldest first
_XG_LOCK = _xg_threading.Lock()
_XG_SEQ = _xg_itertools.count(1)
_XG_GAMES = _xg_itertools.count(1)
_XG_LOG_LOCK = _xg_threading.Lock()
_XG_TL = _xg_threading.local()    # v47: game id prefix ('ls20') of the session playing in this thread
XG_LIBRARY = [  # @@XG_LIBRARY@@ v47: practice-game rules (make_xg_library.py, runs v46, v45, v43, v42, v41, v40, v39, v36, v35)
    {'name': 'ar25', 'level': 1, 'rule': 'Goal: drive the reflection onto the yellow target shape. The light-blue vertical line is a mirror; arrow keys move the black dotted piece by 3 cells and its charcoal reflection moves mirror-symmetrically (LEFT/RIGHT invert across the line, UP/DOWN stay). SPACE does nothing; the yellow right-edge column is a step budget (-1/move).', 'twist': None, 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT', 'SPACE', 'MOUSE']},
    {'name': 'ar25', 'level': 2, 'rule': 'Goal stayed "make the charcoal mirror of the black piece cover the yellow target". New mechanic: the dotted divider slides, and SPACE swaps which object is ACTIVE (the one wearing white dots); arrows then move the active object in 3-cell steps (piece follows arrows, charcoal is its mirror across the divider axis, col\' = 2·axis − col).', 'twist': "New mechanic: the dotted divider slides, and SPACE swaps which object is ACTIVE (the one wearing white dots); arrows then move the active object in 3-cell steps (piece follows arrows, charcoal is its mirror across the divider axis, col' = 2·axis − col).", 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT', 'SPACE', 'MOUSE']},
    {'name': 'ar25', 'level': 6, 'rule': '... Then inspect. Key new mechanic in level 6 that I should carry forward: MOUSE click selects the object under the cursor directly (cheaper than cycling with SPACE); a "piece" can be a multi-component rigid group; the target can require both mirrors moved to the target\'s symmetry axes. For level 7, let me inspect the board. Plan: 1.', 'twist': 'Key new mechanic in level 6 that I should carry forward: MOUSE click selects the object under the cursor directly (cheaper than cycling with SPACE); a "piece" can be a multi-component rigid group;', 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT', 'SPACE', 'MOUSE']},
    {'name': 'bp35', 'level': 1, 'rule': 'goal = duck reaches the pink plus. duck moves ±6 with LEFT/RIGHT along the water surface; clicking a green 5x5 block turns it into water and floods the connecting thin black gaps (above/below), letting pools connect and the water level equalize (the duck and pink float on the surface); the duck escaping off the top just regenerates the board.', 'twist': None, 'controls': ['LEFT', 'RIGHT', 'MOUSE']},
    {'name': 'cd82', 'level': 1, 'rule': "The goal was to paint the white 10x10 block to match the top-left target panel (top half white, bottom half purple). The gray cup orbits the block; arrow keys roll it to the next orbit slot (the key must match the dominant tangential direction), clicking a top HUD slot sets the cup's ink color, SPACE presses the cup 6 cells toward the block and dyes the ...", 'twist': None, 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT', 'SPACE', 'MOUSE']},
    {'name': 'cd82', 'level': 3, 'rule': "Same goal — match the block to the top-left preview, but the preview can contain a small 3×4 stamp no half-plane pour makes. New mechanic: MOUSE on the bucket's cap (row 21, col 31) drops a 3×4 blob of the selected ink onto the block's top-centre (r0-2, c3-6) — it only works with the bucket upright (pose N).", 'twist': "New mechanic: MOUSE on the bucket's cap (row 21, col 31) drops a 3×4 blob of the selected ink onto the block's top-centre (r0-2, c3-6) — it only works with the bucket upright (pose N).", 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT', 'SPACE', 'MOUSE']},
    {'name': 'cd82', 'level': 4, 'rule': "The goal is still to match the block to the top-left preview. Key new discovery: the bucket's pose range is wider than I thought — 7 poses (SW, W, NW, N, NE, E, SE) spanning -135° to +135°; LEFT/RIGHT tilt within ±45°, and DOWN/UP swing outward/inward beyond ±45° (DOWN from NW goes to W, then SW; DOWN from NE goes to E, then SE).", 'twist': "Key new discovery: the bucket's pose range is wider than I thought — 7 poses (SW, W, NW, N, NE, E, SE) spanning -135° to +135°; LEFT/RIGHT tilt within ±45°, and DOWN/UP swing outward/inward beyond ±45° (DOWN from NW goes to W, then SW;", 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT', 'SPACE', 'MOUSE']},
    {'name': 'cn04', 'level': 1, 'rule': "Goal = plug the connector into the outlet so the plug's red prongs exactly cover the socket's red contact tabs, with the plug body flush against (not overlapping) the socket face. Controls: arrows move the whole plug 3 cells per press; SPACE rotates the plug 90° clockwise; MOUSE unused.", 'twist': None, 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT', 'SPACE', 'MOUSE']},
    {'name': 'cn04', 'level': 4, 'rule': "same goal as level 3 — **wire all pieces into one closed loop** so every wire end (red on the selected piece) meets another piece's end; joints turn gray. Controls unchanged (MOUSE select/toggle, arrows nudge 3 cells, SPACE rotate 90° CW about bbox top-left).", 'twist': 'New wrinkle learned: a piece can be moved *onto* another piece (overlap allowed) and clicking an overlapped cell selects the hidden piece. Won by solving a 4-cycle (spiral→U→bracket→bar): spiral right 3, U rotate 270° + down 2/left 1, bracket rotate 90° + up ...', 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT', 'SPACE', 'MOUSE']},
    {'name': 'cn04', 'level': 5, 'rule': "Same goal — mate every red foot with another piece's red foot. New mechanic: the active plug's white head-block can be clicked repeatedly to GROW the piece (it lengthens and gains feet) until the white block disappears, giving its final shape.", 'twist': "New mechanic: the active plug's white head-block can be clicked repeatedly to GROW the piece (it lengthens and gains feet) until the white block disappears, giving its final shape.", 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT', 'SPACE', 'MOUSE']},
    {'name': 'dc22', 'level': 1, 'rule': 'Goal: walk the player onto the yellow knob inside the gray socket — the toggles create the bridge needed. The green 2x2 knob is the player; arrow keys move it exactly 2 cells (its own size) and it may only land on non-background cells (solid objects; the dithered/checker pattern and plain background block it).', 'twist': None, 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT', 'MOUSE']},
    {'name': 'dc22', 'level': 2, 'rule': 'Same core: the green 2×2 player moves exactly 2 cells per arrow press and may only land on solid cells (background and dithered "ghost" cells block it); goal = land on the yellow knob in its gray socket. New mechanic: a small hat sitting in the left panel is a collectible — walking onto it removes it and adds that color\'s hat button to the right panel ...', 'twist': "New mechanic: a small hat sitting in the left panel is a collectible — walking onto it removes it and adds that color's hat button to the right panel, unlocking that color's toggle.", 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT', 'MOUSE']},
    {'name': 'dc22', 'level': 3, 'rule': 'Goal is still "walk the green 2×2 player (moves exactly 2 cells per arrow press, only onto solid non-dithered cells) onto the yellow knob in its gray socket". Buttons: red rotates the red bar pair 90° about the dark-red pivot, blue swaps each blue block with its dithered ghost twin, magenta teleports the player between the two magenta pads when standing on ...', 'twist': 'Standing on the hidden solid-purple pad (bottom-left, reachable only via the magenta teleport) collected the purple hat, and the new purple button solidified the purple ghost bridge over the yellow socket — cross it, drop into the socke', 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT', 'MOUSE']},
    {'name': 'dc22', 'level': 4, 'rule': 'Goal still = walk the green 2×2 player onto the yellow core (in the bottom box). New controls: the yellow-L podium slides both light-gray/orange strips +2 columns per click (they cycle, turning orange at the end then wrapping), and clicking while standing on a strip is refused unless the player\'s cells stay covered after the move — so you can "ride" a strip ...', 'twist': "New controls: the yellow-L podium slides both light-gray/orange strips +2 columns per click (they cycle, turning orange at the end then wrapping), and clicking while standing on a strip is refused unless the player's cells stay covered after the move — so you ...", 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT', 'MOUSE']},
    {'name': 'ft09', 'level': 1, 'rule': "The highlighted panel's center mini-pattern is a 3x3 target (mini white = red cell, mini gray = blue cell); clicking a panel cell toggled it blue↔red, and clicking the charcoal background just pulsed the corner brackets. Clicking the mini itself did nothing.", 'twist': None, 'controls': ['MOUSE']},
    {'name': 'ft09', 'level': 5, 'rule': "Same icon-target goal (white block = icon's center color, gray block = the other board color) but with a 2-color toggle (green↔purple) plus NEW tool cells drawn as pink/green checkerboards: clicking one toggles itself plus its 4 orthogonal neighbours (a plus-brush), while the ordinary patterned icons are inert.", 'twist': "Same icon-target goal (white block = icon's center color, gray block = the other board color) but with a 2-color toggle (green↔purple) plus NEW tool cells drawn as pink/green checkerboards: clicking one toggles itself plus its 4 orthogonal neighbours (a ...", 'controls': ['MOUSE']},
    {'name': 'g50t', 'level': 1, 'rule': 'the goal was to move the blue player block onto the blue socket; SPACE records the run and restarts, replaying a gray ghost; a ghost standing on the red head-button keeps the red bolt retracted so the player can pass the doorway.', 'twist': None, 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT', 'SPACE']},
    {'name': 'g50t', 'level': 2, 'rule': 'Goal unchanged — walk the blue block onto the blue ring socket; arrows move one cell, SPACE restarts the attempt. New twist: each red plug advances one cell toward its head only while its head cell is occupied, and a parked replay-ghost counts as an occupant.', 'twist': "New twist: each red plug advances one cell toward its head only while its head cell is occupied, and a parked replay-ghost counts as an occupant. Winning plan: end an attempt standing on a head so the next attempt's ghost parks there;", 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT', 'SPACE']},
    {'name': 'g50t', 'level': 3, 'rule': "Same goal/controls. New mechanic: the YELLOW plug+pad is a TOGGLE — stepping on its pad latches that gate open for the whole attempt (two pads pressed on the same tick cancel out). Also learned: echoes are never absorbed — the player can share/pass through an echo's cell (it's just hidden underneath).", 'twist': 'New mechanic: the YELLOW plug+pad is a TOGGLE — stepping on its pad latches that gate open for the whole attempt (two pads pressed on the same tick cancel out).', 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT', 'SPACE']},
    {'name': 'g50t', 'level': 4, 'rule': 'Goal = reach the blue goal box. Arrows move 1 cell; SPACE restarts and records moves as a gray ghost that freezes on the cell where SPACE was pressed, holding plates. New: purple pads are a linked teleport pair wired to a purple plate; when the plate is pressed (by a ghost landing on it) while the player stands on one pad, the player warps to the other pad.', 'twist': 'New: purple pads are a linked teleport pair wired to a purple plate; when the plate is pressed (by a ghost landing on it) while the player stands on one pad, the player warps to the other pad.', 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT', 'SPACE']},
    {'name': 'ka59', 'level': 1, 'rule': 'The goal is to fill each charcoal ring "socket" with a green token. The green token with a WHITE center is the active player; it moves exactly 3 cells per directional press (3x3 size) and can pass over charcoal rings.', 'twist': None, 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT', 'MOUSE']},
    {'name': 'ka59', 'level': 2, 'rule': 'Same goal — park every green square inside the charcoal ring of the same size (4 squares/4 rings this time); arrows move the white-dotted square 3 cells and clicking any green transfers control to it. New: a pushed box slides 3 cells at a time until it is blocked by a room wall, or until it has fully emerged from the purple wall it passed through (it stops ...', 'twist': 'Same goal — park every green square inside the charcoal ring of the same size (4 squares/4 rings this time); arrows move the white-dotted square 3 cells and clicking any green transfers control to it.', 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT', 'MOUSE']},
    {'name': 'ka59', 'level': 5, 'rule': 'goal = cover the charcoal socket with the green 3×3; the new element was a two-tone (orange/dark-red) plug blocking the U-shaped container: pressing into it pushed it (it slid far up, through the purple wall, and stopped near the top), and the orange/red split oscillates each move (a wave), which is decorative; MOUSE unused (only one green block).', 'twist': 'the new element was a two-tone (orange/dark-red) plug blocking the U-shaped container: pressing into it pushed it (it slid far up, through the purple wall, and stopped near the top), and the orange/red split oscillates each move (a wave), which is decorative;', 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT', 'MOUSE']},
    {'name': 'ka59', 'level': 6, 'rule': 'goal = fill every charcoal outline (yellow plus into the plus outline, player into the 3x3 ring). New mechanic: the orange/dark-red piston blocks are elevators — standing above one and pressing UP (or below one and pressing DOWN, or moving into the cell above/below one) carries the player through the purple wall until fully clear;', 'twist': 'New mechanic: the orange/dark-red piston blocks are elevators — standing above one and pressing UP (or below one and pressing DOWN, or moving into the cell above/below one) carries the player through the purple wall until fully clear;', 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT', 'MOUSE']},
    {'name': 'lf52', 'level': 1, 'rule': 'Goal: end with one plus. checkers-style jump puzzle. Click a plus (source), then click the landing cell 2 cells away in a straight line over an adjacent plus; the jumped plus is captured.', 'twist': None, 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT', 'MOUSE']},
    {'name': 'lf52', 'level': 2, 'rule': 'Goal = leave a single peg (peg-solitaire jumps: click peg → gray-plus landing → click it; jumped peg vanishes). NEW mechanic: the orange token rides a fixed corridor track — arrow keys slide it 1 station (6px) per press, following corners; it cannot enter panels.', 'twist': 'NEW mechanic: the orange token rides a fixed corridor track — arrow keys slide it 1 station (6px) per press, following corners; it cannot enter panels.', 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT', 'MOUSE']},
    {'name': 'lp85', 'level': 1, 'rule': 'goal = bring the yellow block into the yellow dashed marker slot (fixed at a corner). Ring of 20 blocks; red arrow click = rotate ring counterclockwise 1 step; green arrow click = rotate clockwise 1 step; Solved with 5 CCW clicks.', 'twist': None, 'controls': ['MOUSE']},
    {'name': 'ls20', 'level': 1, 'rule': 'Goal = slide the 5x5 orange/blue block into the black dock box at the top (a white "+" was an optional marker; covering it flipped the corner badge). Each of UP/DOWN/LEFT/RIGHT teleports the block exactly 5 cells (its own size) and is a full no-op if the 5x5 destination isn\'t clear; the yellow bottom bar is a move budget (-1/move).', 'twist': None, 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT']},
    {'name': 'ls20', 'level': 3, 'rule': 'Same goal (dock the block), but the dock requires the badge to match the dock icon in BOTH shape and color. The white "+" rotates the badge\'s shape (2 presses to match); the new 4-color pinwheel (a 3×3 key) recolors the badge (orange→blue) when the block covers it.', 'twist': 'the new 4-color pinwheel (a 3×3 key) recolors the badge (orange→blue) when the block covers it. Light-gray bars beside a corridor are launchers: entering the cell next to one slams the block along the corridor (the top-right one shoots you down into the dock).', 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT']},
    {'name': 'ls20', 'level': 4, 'rule': "Same core (5-cell steps, plus morphs the key, bump the framed picture when key color+shape match it to win), plus two new mechanics: the rainbow tile CYCLES the key color (b→N→R→O→b, one step per cover, reusable) and the light-gray 'w' bars are jets — stepping on the cell adjacent to a bar's face flings the block along that axis until blocked (chains don't ...", 'twist': "Same core (5-cell steps, plus morphs the key, bump the framed picture when key color+shape match it to win), plus two new mechanics: the rainbow tile CYCLES the key color (b→N→R→O→b, one step per cover, reusable) and the light-gray 'w' bars are jets — ...", 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT']},
    {'name': 'ls20', 'level': 5, 'rule': "Goal = same as before (make the bottom-left key glyph match the framed picture's glyph in colour AND shape, then press into the picture). New: the rainbow tile cycles the key colour (b→N→R→O→b, 3 covers needed from orange), plus1 is a 6-state shape cycler, and plus2 is a 90° CW rotator that OSCILLATES horizontally one 5-cell step per player move — a ...", 'twist': 'New: the rainbow tile cycles the key colour (b→N→R→O→b, 3 covers needed from orange), plus1 is a 6-state shape cycler, and plus2 is a 90° CW rotator that OSCILLATES horizontally one 5-cell step per player move — a rotation only fires when your footprint ...', 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT']},
    {'name': 'm0r0', 'level': 1, 'rule': "goal = swap the two blue blocks' positions (each ends in the other's start half/spot). Arrows move each block exactly 5 cells (its own size); LEFT/RIGHT mirror horizontally per block (leftmost goes left, rightmost goes right — or identity-based: the left-origin block goes left); UP/DOWN move both vertically; background is impassable; SPACE/MOUSE do nothing.", 'twist': None, 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT', 'SPACE', 'MOUSE']},
    {'name': 'm0r0', 'level': 2, 'rule': 'Goal = merge the two cyan (S) squares onto the same cell (same as level 1). Controls unchanged (UP/DOWN shared, LEFT/RIGHT mirrored about center). NEW: red/black checkerboard tiles are deadly hazards — moving a cyan onto one resets both cyan to start (no level fail, just costs a move).', 'twist': 'NEW: red/black checkerboard tiles are deadly hazards — moving a cyan onto one resets both cyan to start (no level fail, just costs a move). Won by routing both cyan around the walls and hazards down to the bottom bridge and merging.', 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT', 'SPACE', 'MOUSE']},
    {'name': 'm0r0', 'level': 3, 'rule': "Two 1-cell blocks in a thin black maze; UP/DOWN/LEFT/RIGHT move both blocks (the right one's horizontal axis mirrored); the goal is still to make them collide into one cell. New mechanic: blue tokens are movable walls — clicking a token turns it yellow and freezes the blocks, arrows then slide the token itself, and clicking an empty black cell unfreezes ...", 'twist': 'New mechanic: blue tokens are movable walls — clicking a token turns it yellow and freezes the blocks, arrows then slide the token itself, and clicking an empty black cell unfreezes them.', 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT', 'SPACE', 'MOUSE']},
    {'name': 'm0r0', 'level': 4, 'rule': 'Same goal as before — make the two 5×5 cyan blocks exchange places in one arrow press; UP/DOWN moved both blocks, LEFT/RIGHT moved the left-origin block literally and the right-origin block mirrored, each press shifting a block 5 cells into black floor only (red checkerboard = death → full level reset, so no parking on caps).', 'twist': "New mechanic: the blue 3×3 token is a movable wall — clicking it turns it into a steerable 3×3 yellow piece (blocks grey/frozen) and clicking a grey block drops the token at the yellow's spot.", 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT', 'SPACE', 'MOUSE']},
    {'name': 'm0r0', 'level': 5, 'rule': 'Same goal — the two 4×4 cyan blocks must pass through each other in one arrow press; UP/DOWN moved both, LEFT/RIGHT moved the left-origin block mirrored and the right-origin block literally, 4 cells per press, into black floor only.', 'twist': 'New mechanic: colored 4×12 bars are doors and the same-colored 4×4 patches are pressure-plate keys — a door opens only while a block stands exactly on its key patch (both blocks move every press, so one must be parked on a key while the other travels).', 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT', 'SPACE', 'MOUSE']},
    {'name': 'r11l', 'level': 1, 'rule': 'Goal: land the purple blob (middle) on the purple ring. click a cell → the white bead (tail) jumps there, the purple blob (middle) goes to the midpoint between the white and the gray head; clicking the gray head swaps head↔tail (relocating the head to the old tail spot); clicking the white tail = no-op.', 'twist': None, 'controls': ['MOUSE']},
    {'name': 'r11l', 'level': 3, 'rule': 'Same goal — seat each colored square on its matching ring (green square→green ring, purple→purple). Squares still sit at the floor-of-mean of their tethered diamonds; click a gray diamond to make it white, click land to move it.', 'twist': "New rule discovered: a move is REJECTED (animates, reverts) unless the square's 5x5 footprint and the diamond's footprint stay off water/gray — the rods themselves MAY cross water.", 'controls': ['MOUSE']},
    {'name': 'r11l', 'level': 4, 'rule': '3 chains, blobs are 2-3 colored; rings multi-colored; match by color set. Same move/swap mechanics; key new finding: the click hit-test for a bead is the 5x5 box (Chebyshev ≤ 2), so a move target within 2 cells of any bead center becomes a swap with that bead.', 'twist': 'key new finding: the click hit-test for a bead is the 5x5 box (Chebyshev ≤ 2), so a move target within 2 cells of any bead center becomes a swap with that bead.', 'controls': ['MOUSE']},
    {'name': 'r11l', 'level': 5, 'rule': 'New mechanic — "painter" tiles: 5×5 blob-shaped markers that are half-white/half-color. When a blob is moved onto a painter, the painter is consumed and paints the blob\'s non-white half with its color (white half = no change). The goal is still: put each blob on the ring whose left/right colors match the blob\'s left/right colors.', 'twist': 'New mechanic — "painter" tiles: 5×5 blob-shaped markers that are half-white/half-color. When a blob is moved onto a painter, the painter is consumed and paints the blob\'s non-white half with its color (white half = no change).', 'controls': ['MOUSE']},
    {'name': 're86', 'level': 1, 'rule': "Goal — collect all dots of each color by sweeping the matching cross's arms over them. SPACE toggles which cross carries the white cursor (active cross); arrow keys move the active cross 3 cells per press. Plan: toggle to each color, drive its cross so its row and column cover all four dots of that color.", 'twist': None, 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT', 'SPACE']},
    {'name': 're86', 'level': 2, 'rule': 'Same goal — cover every dot with the shape of its color; SPACE cycled the white head through three steerable shapes (orange X-cross, red diamond ring, blue plus-cross), each moving 3 cells per arrow press. The new mechanic was the diamond ring: it collects dots lying on its outline (Manhattan distance exactly 9 from its center), and shapes just occlude each ...', 'twist': 'The new mechanic was the diamond ring: it collects dots lying on its outline (Manhattan distance exactly 9 from its center), and shapes just occlude each other when overlapping.', 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT', 'SPACE']},
    {'name': 're86', 'level': 4, 'rule': "WIN = every dot covered by a line of a shape whose CURRENT color matches the dot's. Arrows slide the selected shape 3 cells; SPACE cycles the 2 shapes. NEW mechanic: 6 grey-framed palette squares repaint any shape whose line touches them (last touch wins).", 'twist': 'NEW mechanic: 6 grey-framed palette squares repaint any shape whose line touches them (last touch wins). Plan: X touched the green square then sat at (30,39) covering the 3 green dots;', 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT', 'SPACE']},
    {'name': 're86', 'level': 5, 'rule': "Same goal — every dot must be covered by an object of its own colour — with two dot colours (6 blue, 2 red) and recolour pads. Key new finding: a pad recolours an object when ANY pixel of the object touches the pad's fill (arm touch works, not just the centre); if two pads are touched at once the upper/left one seems to win.", 'twist': "Key new finding: a pad recolours an object when ANY pixel of the object touches the pad's fill (arm touch works, not just the centre); if two pads are touched at once the upper/left one seems to win.", 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT', 'SPACE']},
    {'name': 're86', 'level': 6, 'rule': "Goal = cover all 8 dots at once with a line of a shape whose color matches (dots aren't consumed). Arrows slide the white-marked shape 3 cells; SPACE cycles the 2 shapes. New: the gray plate is a solid obstacle, and a shape pressed into it **deforms** with conserved size — the ring's perimeter stays 72 (blocked axis −3, other axis +3), while the plus keeps ...", 'twist': "New: the gray plate is a solid obstacle, and a shape pressed into it **deforms** with conserved size — the ring's perimeter stays 72 (blocked axis −3, other axis +3), while the plus keeps its four arm *tips* anchored and its crossing shifts 3 toward the ...", 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT', 'SPACE']},
    {'name': 're86', 'level': 7, 'rule': "Goal — cover every dot simultaneously with same-colored shape pixels. SPACE cycled the white head between the two steerable shapes (the yellow T-plus and the square ring); arrows slid the steered shape 3 cells. New mechanic: paint buckets (gray-bordered color blocks along the top) repaint any overlapping shape instantly, and clamping a shape's edge against ...", 'twist': "New mechanic: paint buckets (gray-bordered color blocks along the top) repaint any overlapping shape instantly, and clamping a shape's edge against the gray obstacle while pressing toward it decouples the edges so the ring stretches/shrinks with perimeter ...", 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT', 'SPACE']},
    {'name': 's5i5', 'level': 1, 'rule': 'goal = extend each piston (colored bar with a red dot 1 cell behind its tip) so its red dot lands on the matching red cross; the two-cell gray panel next to each piston is a control: clicking the cell whose glyph spine points in a direction moves that piston 3 cells that way (opening faces the other way).', 'twist': None, 'controls': ['MOUSE']},
    {'name': 's5i5', 'level': 4, 'rule': "Same goal — the red dot must land on the red plus center. New mechanic: one panel per color controls ALL segments of that color at once (the yellow panel grew all five yellow bars simultaneously), while each segment's other-color upstream bar (green/blue/red/orange) repositions its yellow.", 'twist': "New mechanic: one panel per color controls ALL segments of that color at once (the yellow panel grew all five yellow bars simultaneously), while each segment's other-color upstream bar (green/blue/red/orange) repositions its yellow.", 'controls': ['MOUSE']},
    {'name': 'sb26', 'level': 1, 'rule': "goal = make the dots in the red box match the top row color sequence; clicking a bottom tray color selects it (white outline), clicking a dot swaps the selected color with the dot's content; SPACE submits/confirms.", 'twist': None, 'controls': ['SPACE', 'MOUSE']},
    {'name': 'sb26', 'level': 2, 'rule': 'Same core (click tray tile → click slot to swap in; placed tiles are selectable and can be swapped between slots; tray gray dots aren\'t selectable). New: two boxes joined by a colored "pipe"; the pipe/plug slot is fixed and NOT a slot.', 'twist': 'New: two boxes joined by a colored "pipe"; the pipe/plug slot is fixed and NOT a slot.', 'controls': ['SPACE', 'MOUSE']},
    {'name': 'sb26', 'level': 4, 'rule': "Same goal — fill the boxes so the expanded sequence matches the top row. New mechanic: a hollow colored square sitting in the swatch row is a pipe-mouth marker that must be clicked and placed into the parent box's mouth slot; once placed, that box is read by expanding the connected box in place of the mouth.", 'twist': "New mechanic: a hollow colored square sitting in the swatch row is a pipe-mouth marker that must be clicked and placed into the parent box's mouth slot; once placed, that box is read by expanding the connected box in place of the mouth.", 'controls': ['SPACE', 'MOUSE']},
    {'name': 'sc25', 'level': 1, 'rule': "Win = push the tip into the blue head. The plug tip must be slid fully into the head socket at the left end of the wire. Arrow keys: LEFT/RIGHT slide the tip along the wire (±4 cells, or ±2 when the tip is 2-wide), UP/DOWN rotate the tip's two-color halves.", 'twist': None, 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT', 'MOUSE']},
    {'name': 'sc25', 'level': 4, 'rule': "same as level 3 — drive the plug head into a socket; arrow keys move the head one head-width per press and turn it to face that way; clicking the lit keypad cells runs the panel's code. Level 4's new facts: the DIAMOND code shrinks the 4x4 head to 2x2 (so it can enter the narrow cord), the vertical-3-line code fires a surge along the cord in the facing ...", 'twist': "Level 4's new facts: the DIAMOND code shrinks the 4x4 head to 2x2 (so it can enter the narrow cord), the vertical-3-line code fires a surge along the cord in the facing direction with unlimited straight reach (destroys the dirty pink socket and the dark-red ...", 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT', 'MOUSE']},
    {'name': 'sk48', 'level': 1, 'rule': "goal = make the board match the bottom legend: the head's arm fully extended with the three blocks threaded on it in the legend's order (R nearest the head, then N, then b), spaced 6 apart. Controls: UP/DOWN move the head between 5 slots (6 rows); RIGHT extends the arm by 6 (or attaches a blocked block / pushes blocks right);", 'twist': None, 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT', 'MOUSE']},
    {'name': 'sp80', 'level': 1, 'rule': 'Goal = make the falling magenta goo land exactly inside a yellow notch (cup). LEFT/RIGHT move the blue bar (±4 cols); SPACE drops the magenta block, which spreads over the bar and drips off both bar ends (4-wide drips at bar_left-4..bar_left-1 and bar_right+1..bar_right+4); those drips must align with the notches.', 'twist': None, 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT', 'SPACE', 'MOUSE']},
    {'name': 'sp80', 'level': 4, 'rule': 'goal = fill all 4 bottom cups in one SPACE launch; ball falls down; splits at surfaces, exits 3 wide just outside each end; arrows move the blue paddle (and any connected blob) 3 cells; MOUSE swaps the paddle with a red bar (position+size, and the clicked bar becomes the new blue paddle);', 'twist': 'MOUSE swaps the paddle with a red bar (position+size, and the clicked bar becomes the new blue paddle); the charcoal 3x3 block is a HOLE the ball passes through (it moved with the connected paddle blob).', 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT', 'SPACE', 'MOUSE']},
    {'name': 'su15', 'level': 1, 'rule': "the goal is to roll the ball into that core. The purple 3×3 ball sits on a dotted 45° trail leading into the blue orb's gray core; MOUSE on a trail dot moves the ball along the trail to that dot, but only up to 3 dots ahead (farther clicks animate and fail). Plan: repeatedly click exactly 3 dots ahead along the trail until the core is reached.", 'twist': None, 'controls': ['MOUSE']},
    {'name': 'tn36', 'level': 1, 'rule': '5 DOWNs (all keys black) drove the plug into the cup and won. The yellow plug must be moved into the yellow cup on the checkerboard. The 5 bottom key glyphs each have a top bar and a bottom stem that toggle independently (black/gray) when clicked;', 'twist': None, 'controls': ['MOUSE']},
    {'name': 'tn36', 'level': 5, 'rule': "The movable piece is the small yellow block at the top; the fixed purple U is the cavity it must fill. New glyphs: bar@33+bar@39 (two bars, no stems) = ROTATE 90° CW; all 6 bits on = PAINT the block the cavity's color (required — a perfect yellow fit did NOT win, the painted one did).", 'twist': "New glyphs: bar@33+bar@39 (two bars, no stems) = ROTATE 90° CW; all 6 bits on = PAINT the block the cavity's color (required — a perfect yellow fit did NOT win, the painted one did).", 'controls': ['MOUSE']},
    {'name': 'tr87', 'level': 1, 'rule': 'Goal: make each column\'s pink glyph the key partner of the blue glyph above it. The top grid showed 6 blue→pink "key" pairs; the bottom had a fixed blue strip and an editable pink strip with a white bracket cursor. LEFT/RIGHT moved the cursor across pink tiles (wrapping), UP/DOWN cycled the selected pink glyph through a 7-shape ring (the 6 key pinks + one ...', 'twist': None, 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT']},
    {'name': 'tu93', 'level': 1, 'rule': 'Goal: jump the blue player across a checkerboard tile field onto the green tile. Each arrow key jumps the player exactly 2 tiles in that direction (over one adjacent floor tile) and swaps it with the landing tile; blocked if intermediate or landing tile is black/absent. Plan: BFS over even-parity tiles with 2-tile jumps, then batch the path.', 'twist': None, 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT']},
    {'name': 'tu93', 'level': 2, 'rule': 'Goal — reach the green tile by jumping exactly 2 tiles per arrow press (over one floor tile; blocked by black). New mechanic: a red guard tile with a facing dot — if you end your move exactly 2 tiles in front of it, it lunges forward and kills you, but landing ON it from another direction destroys it.', 'twist': 'New mechanic: a red guard tile with a facing dot — if you end your move exactly 2 tiles in front of it, it lunges forward and kills you, but landing ON it from another direction destroys it.', 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT']},
    {'name': 'tu93', 'level': 4, 'rule': 'Goal still reach the green tile with 2-tile jumps. New mechanic: an orange guard that PATROLS its column (moves 2 tiles per turn, bouncing at the ends) and kills on contact at its destination — it cannot be captured, only dodged; the red guard still kills at its front tile and dies if landed on.', 'twist': 'New mechanic: an orange guard that PATROLS its column (moves 2 tiles per turn, bouncing at the ends) and kills on contact at its destination — it cannot be captured, only dodged; the red guard still kills at its front tile and dies if landed on.', 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT']},
    {'name': 'tu93', 'level': 5, 'rule': 'Goal unchanged — land on the green tile with 2-tile arrow jumps. New mechanic: several ORANGE patrollers, each sliding 2 tiles/turn along its own floor run, bouncing at walls (and at each other); they kill on contact, so only their destination tiles are lethal and their vacated tiles are safe.', 'twist': 'New mechanic: several ORANGE patrollers, each sliding 2 tiles/turn along its own floor run, bouncing at walls (and at each other); they kill on contact, so only their destination tiles are lethal and their vacated tiles are safe.', 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT']},
    {'name': 'tu93', 'level': 6, 'rule': 'Goal unchanged — land on the green tile via 2-tile jumps. New mechanics: red guards are still capturable by landing on them from any non-front direction (front = the side they face), and a patrolling orange slides 2/turn along its whole row/column, passing THROUGH guards (which hide it visually) and killing on contact;', 'twist': 'New mechanics: red guards are still capturable by landing on them from any non-front direction (front = the side they face), and a patrolling orange slides 2/turn along its whole row/column, passing THROUGH guards (which hide it visually) and killing on ...', 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT']},
    {'name': 'tu93', 'level': 7, 'rule': 'Goal = land on green; same 2-block slides on a 9×13 board. New: the dark-red guard is a late-activating patroller (indicator turns yellow when it starts) that walks its corridor and kills on contact; red guards still charge their facing and die if landed on from side/back.', 'twist': 'New: the dark-red guard is a late-activating patroller (indicator turns yellow when it starts) that walks its corridor and kills on contact; red guards still charge their facing and die if landed on from side/back.', 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT']},
    {'name': 'tu93', 'level': 8, 'rule': 'Goal = land on green; same 2-block slides on an 11×11 corridor maze. New mechanic: the dark-red guard is a CHASER (dormant until you enter its sight line, then it walks 2 blocks/turn along the corridors toward you, killing on contact); red guards still charge their facing and die if landed on from the side.', 'twist': 'New mechanic: the dark-red guard is a CHASER (dormant until you enter its sight line, then it walks 2 blocks/turn along the corridors toward you, killing on contact); red guards still charge their facing and die if landed on from the side.', 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT']},
    {'name': 'vc33', 'level': 1, 'rule': 'Goal: slide the player\'s yellow block until it is column-aligned with the yellow slot in the wall. The gray "sheet" has a fixed-perimeter boundary with a player (arrow+yellow block) riding the gray/white edge; clicking the blue block on one side of the black wall slid the player +4 cols (and the opposite boundary -4), the other blue slid it -4.', 'twist': None, 'controls': ['MOUSE']},
    {'name': 'vc33', 'level': 4, 'rule': "Goal — float the piston block into the yellow gap's rows in the far wall. Controls: each blue tile at a wall's base pumps 3 rows of white from the opposite band into its own band (a band's cap = its wall's top; the piston's block rides its band's surface).", 'twist': 'New mechanic: a light-gray "sponge" in a wall is a gate — clicking it moves the piston into the neighbouring band, but only when the block is level with the gate\'s bottom row AND the two bands\' surfaces are within 1 row.', 'controls': ['MOUSE']},
    {'name': 'wa30', 'level': 1, 'rule': 'Goal = fill the blue "socket bar" with all the boxes. Arrows move the 4x4 player in 4-cell steps (facing = white strip); blocked moves just turn. SPACE facing an adjacent box picks it up (it sticks to the player at a fixed offset, drawn white) and SPACE again drops it. Win = all 3 boxes placed over the bar area.', 'twist': None, 'controls': ['UP', 'DOWN', 'LEFT', 'RIGHT', 'SPACE']},
]  # @@END_XG_LIBRARY@@
# Save & Run only: every pooled rule and every block shown goes to cross_game_rules.jsonl (never in a competition rerun)
CROSS_GAME_LOG = (None if globals().get("TRUE_SUBMISSION", True)
                  else str(globals().get("WORKING_DIR", "/kaggle/working")) + "/cross_game_rules.jsonl")
_XG_INNER_RULE = _re_mx.compile(r"Rule\s*(?:for\s*)?\(?\s*level\s*\d+\s*\)?\s*\**\s*[:\-–—]\s*\**[ \t]*", _re_mx.I)
_XG_JUNK = _re_mx.compile(r"^(?:\(?\s*write\b|as (?:written|stated|above)\b|see above\b|let me\b|i (?:need|will|'ll)\b|"
                          r"paragraph\b|now (?:level|the|let)\b|level \d+\s*(?:board|layout|:))", _re_mx.I)
_XG_SENTENCE = _re_mx.compile(r"(?<=[.;!])\s+(?=\S)")
_XG_NEW = _re_mx.compile(r"\bnew(?:ly)?\b(?!\s+(?:board|level|layout|frame|screen))|\bintroduc\w*|\badded\b|"
                         r"\btwist\b|\bthis time\b", _re_mx.I)
_XG_GOAL = _re_mx.compile(r"\b(?:goal|objective|win|won|wins|target)\b", _re_mx.I)
_XG_VALID = _re_mx.compile(r"Valid actions right now: ([A-Za-z0-9_ ,]+)\.")


def _xg_game(agent):
    """A run-unique number per game (id() of a finished game's agent can be reused by a new one)."""
    key = agent.__dict__.get("_xg_game")
    if key is None:
        key = agent.__dict__.setdefault("_xg_game", next(_XG_GAMES))
    return key


def xg_controls(valid_actions=None, prompt=None):
    names = set()
    for a in valid_actions or []:
        n = _SB_MODEL_NAMES.get(str(a).strip().upper(), str(a).strip().upper())
        if n in XG_CONTROL_ORDER:
            names.add(n)
    if not names and prompt:
        m = _XG_VALID.search(prompt)
        if m:
            names = {p.strip().upper() for p in m.group(1).split(",")} & set(XG_CONTROL_ORDER)
    return frozenset(names)


def xg_clean(rule):
    """The rule text, or None for a capture that is not a rule (the model writing about the paragraph it will write)."""
    text = " ".join(str(rule or "").split()).lstrip(" *:\"'`-–—").strip()
    if _XG_JUNK.match(text) or _re_mx.search(r"\blet me\b", text[:150], _re_mx.I):
        m = _XG_INNER_RULE.search(text)        # '... paragraph first. Rule for level 3: Goal - land each bar ...'
        text = text[m.end():].lstrip(" *:\"'`-–—").strip() if m else ""
        if _XG_JUNK.match(text) or _re_mx.search(r"\blet me\b", text[:150], _re_mx.I):
            text = ""
    return text if len(text) >= XG_MIN_CHARS else None


def xg_clip(text, limit=XG_RULE_CHARS):
    text = str(text or "").strip()
    if len(text) <= limit:
        return text
    cut = text[:limit]
    for sep in (". ", "; "):
        i = cut.rfind(sep)
        if i >= limit * 0.6:
            return cut[:i + 1].rstrip()
    i = cut.rfind(" ")
    return (cut[:i] if i >= limit * 0.6 else cut).rstrip(" ,;:-") + " ..."


def xg_twist(rule):
    """What a later level added: the first sentence that names something new ('New mechanic: ...', '... was added',
    'this time ...') and the sentence after it; None when the rule names nothing new (then it is not shown)."""
    parts = [p.strip() for p in _XG_SENTENCE.split(rule or "") if p.strip()]
    for i, p in enumerate(parts):
        if _XG_NEW.search(p):
            text = " ".join(parts[i:i + 2])
            return text if len(text) >= XG_MIN_CHARS else None
    return None


def xg_goal_first(rule):
    """A level-1 rule with its goal sentence moved to the front (clipping must not cut the goal)."""
    parts = [p.strip() for p in _XG_SENTENCE.split(rule or "") if p.strip()]
    for i, p in enumerate(parts):
        if _XG_GOAL.search(p):
            if i:
                parts = [p[:-1] + "." if p.endswith(";") else p] + parts[:i] + parts[i + 1:]
            break
    return " ".join(parts)


def _xg_log(record):
    path = CROSS_GAME_LOG
    if not path:
        return
    try:
        with _XG_LOG_LOCK:
            with open(path, "a", encoding="utf-8") as f:
                f.write(_xg_json.dumps(record, default=sorted) + "\n")
    except Exception:
        pass


def xg_add(agent, level, rule):
    """Pool one stated rule (called when section A finalizes it). Returns the pooled record or None."""
    text = xg_clean(rule)
    if text is None:
        _mx_inc("xg_rejected")
        _xg_log({"event": "rejected", "game": _xg_game(agent), "level": level, "rule": str(rule)[:300]})
        return None
    level = int(level)
    twist = xg_twist(text) if level >= 2 else None
    by_level = agent.__dict__.get("_xg_controls_by_level") or {}
    controls = frozenset(by_level.get(level) or agent.__dict__.get("_xg_controls") or ())
    rec = {"game": _xg_game(agent), "level": level,
           "rule": xg_clip(xg_goal_first(text) if level == 1 else text),
           "twist": xg_clip(twist, XG_TWIST_CHARS) if twist else None, "controls": controls}
    with _XG_LOCK:
        rec["seq"] = next(_XG_SEQ)
        _XG_POOL.append(rec)
        del _XG_POOL[:-XG_POOL_MAX]
    _mx_inc("xg_pool")
    if level == 1:
        _mx_inc("xg_pool_level1")
    _xg_log({"event": "pool", **rec})
    return rec


def xg_select(agent, level, controls, k=XG_SHOW, mark=True):
    """Up to k pooled records of OTHER games for a game stalled on `level`: level 1 -> their level-1 rules; later ->
    their later-level twists. One per game; not-yet-shown first, then most similar controls, then newest."""
    me = _xg_game(agent)
    with _XG_LOCK:
        pool = list(_XG_POOL)
    shown = agent.__dict__.setdefault("_xg_shown", set())
    per_game = {}
    for rec in pool:
        if rec["game"] == me:
            continue
        lib = rec.get("lib")
        if lib is not None:      # v47 practice-game rule: never the game's own; unknown game in a Save & Run -> none
            name = agent.__dict__.get("_xg_name")
            if name == lib or (name is None and not globals().get("TRUE_SUBMISSION", False)):
                continue
        if level <= 1:
            if rec["level"] != 1:
                continue
        elif rec["level"] < 2 or not rec.get("twist"):
            continue
        cur = per_game.get(rec["game"])
        if cur is None or (rec["seq"] in shown, -rec["seq"]) < (cur["seq"] in shown, -cur["seq"]):
            per_game[rec["game"]] = rec
    controls = frozenset(controls or ())

    def rank(rec):
        union = controls | rec["controls"]
        similarity = len(controls & rec["controls"]) / len(union) if union else 0.0
        return (rec["seq"] in shown, -similarity, -rec["seq"])

    picked = sorted(per_game.values(), key=rank)[:k]
    if mark:
        shown.update(rec["seq"] for rec in picked)
    return picked


def xg_line(rec, level):
    controls = ", ".join(n for n in XG_CONTROL_ORDER if n in rec["controls"]) or "not recorded"
    text = rec["rule"] if level <= 1 else rec["twist"]
    kind = "a practice game" if rec.get("lib") is not None else "another game"
    return f"- ({kind}, its level {rec['level']}; its controls: {controls}) {text}"


def xg_block(picked, level):
    if not picked:
        return ""
    header = XG_HEADER_L1 if level <= 1 else XG_HEADER_LN
    if any(r.get("lib") is not None for r in picked):     # v47: the block names both sources
        header = header.replace("OTHER GAMES IN THIS RUN", "OTHER GAMES (of this run, and practice games)", 1)
    return header + "\n" + "\n".join(xg_line(r, level) for r in picked)


def xg_seed_library(entries=None):
    """v47: the practice-game rules (XG_LIBRARY) as the oldest pool records - game 'lib:<id>', negative seq, 'lib' =
    the game id prefix they must never be shown to. Returns the number added (0 if already seeded)."""
    entries = XG_LIBRARY if entries is None else entries
    recs = []
    for i, e in enumerate(entries):
        name = str(e["name"]).strip().lower()
        level = int(e["level"])
        rule = str(e.get("rule") or "").strip()
        twist = str(e.get("twist") or "").strip() or None
        if not name or not rule or (level >= 2 and not twist):
            continue
        recs.append({"game": "lib:" + name, "level": level, "rule": rule, "twist": twist,
                     "controls": frozenset(c for c in e.get("controls") or () if c in XG_CONTROL_ORDER),
                     "seq": -(i + 1), "lib": name})
    with _XG_LOCK:
        if any(r.get("lib") is not None for r in _XG_POOL):
            return 0
        _XG_POOL[:0] = recs
    with _LK_MX:
        ADDON_STATS["xg_lib_pool"] = len(recs)
    return len(recs)


_xg_m85_capture_inner = _m85_capture


def _m85_capture(agent):
    """The solved-level memory's capture (it calls this name at run time; v25's pool wrapper inside), plus: every newly
    stated rule goes to the cross-game library."""
    changed = _xg_m85_capture_inner(agent)
    try:
        if changed and ENABLE_CROSS_GAME_RULES:
            pooled = agent.__dict__.setdefault("_xg_pooled", set())
            for rec in getattr(agent, "_m85_levels", None) or []:
                level = rec.get("level")
                if rec.get("final") and rec.get("rule") and level not in pooled:
                    pooled.add(level)
                    xg_add(agent, level, rec["rule"])
    except Exception as exc:  # never break a turn because of the add-on
        _mx_inc("xg_errors")
        _mx_err("xg", exc)
    return changed


_xg_so_shared_inner = so_shared_rules


def so_shared_rules(agent):
    """v44: the second opinion's 'rules of other games' = the cross-game selection for the stalled level (the facts it
    answers are about `agent._sb_state['level']`). Shown-marks are left to the player's block."""
    if not ENABLE_CROSS_GAME_RULES:
        return _xg_so_shared_inner(agent)
    if not globals().get("ENABLE_SO_SHARED_RULES", True):
        return []
    try:
        level = (agent.__dict__.get("_sb_state") or {}).get("level")
        if isinstance(level, int):
            picked = xg_select(agent, level, agent.__dict__.get("_xg_controls"), k=XG_SO_SHOW, mark=False)
            _mx_inc("xg_so_lists")
            return [xg_line(r, level) for r in picked]
    except Exception as exc:
        _mx_inc("xg_errors")
        _mx_err("xg", exc)
    return _xg_so_shared_inner(agent)


def _xg_install_game_names():
    """v47: record the game id prefix of the session playing in each thread (the D' scheduler and First Light wrap
    the same method later and call this one). False when the harness has no such hook (then no names are known)."""
    try:
        import inference.framework.solver as _xg_solver
        S = _xg_solver._HarnessGameSession
        play_inner = S.play
    except Exception:
        return False
    if getattr(play_inner, "_xg_wrapped", False):
        return True

    def play(self, *args, **kwargs):
        try:
            gid = str(getattr(getattr(getattr(self, "game", None), "game_run", None), "game_id", None) or "")
            _XG_TL.name = gid.split("-")[0].strip().lower() or None
        except Exception:
            _XG_TL.name = None
        try:
            return play_inner(self, *args, **kwargs)
        finally:
            _XG_TL.name = None

    for name in dir(play_inner):
        if name.endswith("_wrapped") or name.endswith("_d"):
            try:
                setattr(play, name, getattr(play_inner, name))
            except Exception:
                pass
    play._xg_wrapped = True
    S.play = play
    return True


def install_cross_game_rules():
    import inference.agent.tool_agent as _ta
    agent_cls = _ta.ToolAgent
    inner = agent_cls._build_user_prompt
    if getattr(inner, "_xg_wrapped", False):
        return False
    if globals().get("ENABLE_XG_LIBRARY", False):
        try:                     # a failure here leaves v46's cross-game rules exactly as they were
            names = _xg_install_game_names()
            seeded = xg_seed_library()
            print(f"cross-game practice library: {seeded} rules seeded, game names {'recorded' if names else 'UNKNOWN'}")
        except Exception as exc:
            _mx_inc("xg_errors")
            _mx_err("xg", exc)
            print("cross-game practice library NOT installed:", repr(exc)[:200])

    def _build_user_prompt(self, action_num, *args, **kwargs):
        out = inner(self, action_num, *args, **kwargs)
        try:
            if "_xg_name" not in self.__dict__:
                name = getattr(_XG_TL, "name", None)
                if name:
                    self._xg_name = name
                    _mx_inc("xg_named_games")
            level = getattr(kwargs.get("current_frame"), "level", None)
            controls = xg_controls(kwargs.get("valid_actions"), out)
            if controls:
                self._xg_controls = controls
                if isinstance(level, int):
                    self.__dict__.setdefault("_xg_controls_by_level", {})[level] = controls
            st = self.__dict__.get("_sb_state")
            if (SO_TRIGGER in out and isinstance(level, int) and isinstance(st, dict) and st.get("level") == level
                    and int(st.get("xg", 0)) < XG_MAX_PER_LEVEL):
                st["xg"] = int(st.get("xg", 0)) + 1      # counted first: a failure never repeats every turn
                picked = xg_select(self, level, self.__dict__.get("_xg_controls"))
                if picked:
                    block = xg_block(picked, level)
                    out += "\n\n" + block
                    _mx_inc("xg_blocks")
                    if level <= 1:
                        _mx_inc("xg_blocks_level1")
                    _mx_inc("xg_rules_shown", len(picked))
                    n_lib = sum(1 for r in picked if r.get("lib") is not None)
                    if n_lib:
                        _mx_inc("xg_lib_blocks")
                        _mx_inc("xg_lib_shown", n_lib)
                    with _LK_MX:
                        ADDON_STATS["xg_block_chars_max"] = max(ADDON_STATS["xg_block_chars_max"], len(block))
                    _xg_log({"event": "block", "game": _xg_game(self), "level": level, "check": st["xg"],
                             "tokens_k": max(0, (_sb_tokens(self) - int(st.get("start", 0))) // 1000),
                             "seqs": [r["seq"] for r in picked], "chars": len(block),
                             "name": self.__dict__.get("_xg_name"),
                             "lib": [r["lib"] for r in picked if r.get("lib") is not None]})
                else:
                    _mx_inc("xg_no_rules")
        except Exception as exc:  # never break a turn because of the add-on
            _mx_inc("xg_errors")
            _mx_err("xg", exc)
        return out

    for name in dir(inner):
        if name.endswith("_wrapped"):
            setattr(_build_user_prompt, name, getattr(inner, name))
    _build_user_prompt._xg_wrapped = True
    agent_cls._build_user_prompt = _build_user_prompt
    return True


if ENABLE_CROSS_GAME_RULES:
    # installed after the level-progress facts and the second opinion (it reads the facts' trigger) and BEFORE the
    # solved-level memory (memory's ask stays last)
    print("cross-game rules installed:", install_cross_game_rules())


if MEMORY_PLACEMENT not in ("history", "system"):
    raise ValueError(f"MEMORY_PLACEMENT must be 'history' or 'system', got {MEMORY_PLACEMENT!r}")
if ENABLE_SOLVED_LEVEL_MEMORY:
    print(f"solved-level memory installed ({MEMORY_PLACEMENT}):", install_solved_level_memory())
if ENABLE_COMPACT_RUNTIME_STATE:
    print("compact runtime state installed:", install_compact_runtime_state())

WM_TOOLS_SOURCE = r'''
# ---- world-model tools (v4): spliced into the python-tool sandbox; standard library only ----
import collections as _wm_collections
import re as _wm_re
import time as _wm_time

_WM_CLICK_RE = _wm_re.compile(r"^\s*([A-Za-z0-9_]+)\(\s*(?:row\s*=\s*)?(-?\d+)\s*,\s*(?:col\s*=\s*)?(-?\d+)\s*\)\s*$")
_WM_SKIP_PLAN = {"RESET", "UNDO", "ACTION7", "MOUSE", "ACTION6"}


def _wm_grid(board):
    """A fresh list-of-lists int copy of a frame view, grid, or tuple grid."""
    rows = getattr(board, "_grid", board)
    return [[int(v) for v in row] for row in rows]


def _wm_action_text(item):
    """The action exactly as `history` records it: 'UP', 'MOUSE(row=3, col=5)'."""
    if isinstance(item, dict):
        name = str(item.get("action", "")).strip().upper()
        if "row" in item or "col" in item:
            return f"{name}(row={item.get('row')}, col={item.get('col')})"
        return name
    match = _WM_CLICK_RE.match(str(item))
    if match:
        return f"{match.group(1).upper()}(row={int(match.group(2))}, col={int(match.group(3))})"
    return " ".join(str(item).split()).upper()


def _wm_action_obj(item):
    """The same action in the form action() accepts."""
    if isinstance(item, dict):
        return dict(item)
    match = _WM_CLICK_RE.match(str(item))
    if match:
        return {"action": match.group(1).upper(), "row": int(match.group(2)), "col": int(match.group(3))}
    return str(item).strip()


def _wm_apply(step, board, action_text):
    work = [row[:] for row in board]
    out = step(work, action_text)
    return work if out is None else _wm_grid(out)


def _wm_wrong_cells(pred, actual, border):
    rows = len(actual)
    cols = len(actual[0]) if rows else 0
    if len(pred) != rows or any(len(r) != cols for r in pred):
        return None
    wrong = []
    for r in range(border, rows - border):
        pr, ar = pred[r], actual[r]
        if border == 0 and pr == ar:
            continue
        for c in range(border, cols - border):
            if pr[c] != ar[c]:
                wrong.append((r, c, pr[c], ar[c]))
    return wrong


def _wm_check_rules(rg, step, border=0, level=None, show=3, time_limit=15.0):
    """Replay this level's recorded transitions through `step` and score it."""
    if not callable(step):
        raise TypeError("check_rules(step): step must be a function step(board, action) -> board")
    border = max(0, int(border))
    current = rg.get("current_frame")
    level = getattr(current, "level", None) if level is None else int(level)
    deadline = _wm_time.monotonic() + max(1.0, min(float(time_limit), 20.0))
    tested = correct = errors = 0
    mismatches = []
    stopped = ""
    for index, tr in enumerate(rg.get("transitions") or []):
        before, after = getattr(tr, "before_frame", None), getattr(tr, "after_frame", None)
        if before is None or after is None:
            continue
        if level is not None and (before.level != level or after.level != level):
            continue
        action_text = _wm_action_text(tr.action)
        if action_text == "RESET":
            continue
        if _wm_time.monotonic() > deadline:
            stopped = "time limit reached; later transitions not checked"
            break
        actual = _wm_grid(after)
        try:
            pred = _wm_apply(step, _wm_grid(before), action_text)
        except Exception as exc:
            errors += 1
            if len(mismatches) < show:
                mismatches.append({"transition": index, "action": action_text, "error": f"{type(exc).__name__}: {exc}"[:200]})
            continue
        tested += 1
        wrong = _wm_wrong_cells(pred, actual, border)
        if wrong == []:
            correct += 1
        elif len(mismatches) < show:
            if wrong is None:
                mismatches.append({"transition": index, "action": action_text, "error": "returned board has a different shape"})
            else:
                rs = [w[0] for w in wrong]
                cs = [w[1] for w in wrong]
                mismatches.append({"transition": index, "action": action_text, "wrong_cells": len(wrong),
                                   "bbox": [min(rs), min(cs), max(rs), max(cs)],
                                   "sample (row, col, predicted, actual)": wrong[:6]})
    result = {"level": level, "checked": tested, "correct": correct,
              "accuracy": round(correct / tested, 3) if tested else None,
              "step_errors": errors, "mismatches": mismatches}
    if stopped:
        result["note"] = stopped
    if not tested and not errors:
        result["note"] = "no recorded transitions on this level yet - act first, then check"
    return result


def _wm_plan_route(rg, step, goal, actions=None, key=None, max_states=20000, time_limit=10.0, start=None):
    """Breadth-first search over `step` from the current board until goal(board) is True."""
    if not callable(step) or not callable(goal):
        raise TypeError("plan_route(step, goal): both must be functions")
    board0 = _wm_grid(start if start is not None else rg.get("current_frame"))
    if actions is None:
        actions = [a for a in rg.get("valid_actions") or [] if str(a).strip().upper() not in _WM_SKIP_PLAN]
    actions = list(actions)
    if not actions:
        return {"found": False, "route": [], "reason": "no actions to search: pass actions=[...] (list clicks explicitly, e.g. 'MOUSE(row=3, col=5)')"}
    texts = [_wm_action_text(a) for a in actions]
    objs = [_wm_action_obj(a) for a in actions]
    max_states = max(1, min(int(max_states), 200000))
    deadline = _wm_time.monotonic() + max(0.5, min(float(time_limit), 20.0))
    rows = len(board0)
    cols = len(board0[0]) if rows else 0

    def pack(board):
        return bytes(v for row in board for v in row)

    def unpack(blob):
        return [list(blob[r * cols:(r + 1) * cols]) for r in range(rows)]

    keyf = key if callable(key) else pack
    try:
        if goal([row[:] for row in board0]):
            return {"found": True, "route": [], "length": 0, "explored": 0, "reason": "goal already true"}
        start_key = keyf(board0)
        packed0 = pack(board0)
    except ValueError:
        return {"found": False, "route": [], "reason": "board values must be 0..255 ints"}
    parent = {start_key: None}
    queue = _wm_collections.deque([(packed0, start_key)])
    explored = 0
    reason = "search space exhausted: no action sequence reaches the goal under this step()"
    while queue:
        if _wm_time.monotonic() > deadline:
            reason = "time limit reached"
            break
        packed, k = queue.popleft()
        explored += 1
        board = unpack(packed)
        for i, text in enumerate(texts):
            try:
                nxt = _wm_apply(step, board, text)
                nk = keyf(nxt)
            except Exception as exc:
                return {"found": False, "route": [], "explored": explored,
                        "reason": f"step/key raised {type(exc).__name__}: {exc} on action {text}"[:300]}
            if nk in parent:
                continue
            parent[nk] = (k, i)
            try:
                reached = goal([row[:] for row in nxt])
            except Exception as exc:
                return {"found": False, "route": [], "explored": explored,
                        "reason": f"goal raised {type(exc).__name__}: {exc}"[:300]}
            if reached:
                route_idx = []
                cur = nk
                while parent[cur] is not None:
                    prev, idx = parent[cur]
                    route_idx.append(idx)
                    cur = prev
                route_idx.reverse()
                return {"found": True, "route": [objs[j] for j in route_idx], "length": len(route_idx),
                        "explored": explored, "states": len(parent)}
            if len(parent) >= max_states:
                reason = f"max_states={max_states} reached (pass key=... to collapse equivalent boards)"
                queue.clear()
                break
            try:
                queue.append((pack(nxt), nk))
            except ValueError:
                return {"found": False, "route": [], "reason": "board values must be 0..255 ints"}
    return {"found": False, "route": [], "explored": explored, "states": len(parent), "reason": reason}


def _wm_bind(rg):
    def check_rules(step, border=0, level=None, show=3, time_limit=15.0):
        """Score your simulator step(board, action) against this level's recorded transitions."""
        return _wm_check_rules(rg, step, border=border, level=level, show=show, time_limit=time_limit)

    def plan_route(step, goal, actions=None, key=None, max_states=20000, time_limit=10.0, start=None):
        """Shortest action route from current_frame to goal(board) under your simulator; pass result['route'] to action()."""
        return _wm_plan_route(rg, step, goal, actions=actions, key=key, max_states=max_states,
                              time_limit=time_limit, start=start)

    return check_rules, plan_route
# ---- end world-model tools ----
'''


# ---------------------------------------------------------------- C. world-model tools (v4)
# The model already writes Python and is told to search. What it lacks is a way to TEST its guessed rules against
# the evidence without spending actions, and a reliable search it does not have to rewrite every turn. Two
# sandbox functions provide both; a fixed system-prompt section explains them (constant for the run, so it never
# invalidates the prefix cache). Installed by splicing standard-library-only source into the sandbox bootstrap.
WM_PROMPT = """WORLD-MODEL TOOLS (available in the python sandbox; they spend NO game actions):
- check_rules(step, border=0): write step(board, action) as your simulator of the game rules. It receives a copy of a board (list of rows of color ints) and one action exactly as `history` records it ('UP', 'MOUSE(row=3, col=5)', ...), and returns the next board (or edits the copy in place). check_rules replays every recorded transition of the current level through it and returns the accuracy plus the first mismatching cells. border=N ignores an N-cell frame such as a step counter.
- plan_route(step, goal, actions=None, key=None, max_states=20000, time_limit=10): breadth-first search over your simulator from current_frame until goal(board) is True. It returns {'found', 'route', ...}; pass result['route'] straight to action(). actions defaults to the non-click valid actions; list clicks explicitly, e.g. actions=['MOUSE(row=3, col=5)', ...]. key(board) can return a compact state (for example the player position) so equivalent boards are not searched twice.
Once you have a hypothesis about the rules: write step, run check_rules until accuracy is close to 1.0 on this level's history, then plan_route to the goal and execute the whole route in one action() call. If the real game then differs from your plan, check_rules shows which rule is wrong. Functions you define are kept between python calls, so step and goal can be refined instead of rewritten."""

ADDON_STATS.update({"wm_snippets": 0, "wm_check_rules": 0, "wm_plan_route": 0, "wm_routes_found": 0,
                    "wm_errors": 0, "wm_last_error": ""})


def install_world_model_tools():
    import inference.agent.python_tool_sandbox as _sb
    import inference.agent.tool_agent as _ta
    boot = _sb._SANDBOX_BOOTSTRAP
    if "_wm_bind" in boot:
        return False
    head = "HOST_STDOUT = sys.stdout\n"
    bind_after = '    runtime_globals["action"] = action\n'
    if boot.count(head) != 1 or boot.count(bind_after) != 1:
        raise RuntimeError("sandbox layout changed; world-model tools NOT installed")
    boot = boot.replace(head, WM_TOOLS_SOURCE + "\n\n" + head, 1)
    boot = boot.replace(bind_after, bind_after +
                        '    runtime_globals["check_rules"], runtime_globals["plan_route"] = _wm_bind(runtime_globals)\n', 1)
    compile(boot, "<sandbox-bootstrap>", "exec")   # refuse to install a bootstrap that does not even parse
    # the sandbox receives its bootstrap as ONE command-line argument (python -c ...); Linux caps a single
    # argument at 131,072 bytes, so stay well below it or every python call would fail to start
    if len(boot.encode("utf-8")) > 120_000:
        raise RuntimeError(f"sandbox bootstrap would be {len(boot.encode('utf-8'))} bytes; world-model tools NOT installed")
    _sb._SANDBOX_BOOTSTRAP = boot

    inner_prompt = _ta._build_system_prompt

    def _build_system_prompt(*args, **kwargs):
        return inner_prompt(*args, **kwargs).rstrip("\n") + "\n\n" + WM_PROMPT

    _ta._build_system_prompt = _build_system_prompt

    inner_run = _ta.run_sandboxed_python

    def run_sandboxed_python(*args, **kwargs):
        result = inner_run(*args, **kwargs)
        try:
            code = str(kwargs.get("code", ""))
            used_check, used_plan = "check_rules(" in code, "plan_route(" in code
            if used_check or used_plan:
                text = str(result.get("stdout", "")) + str(result.get("result", ""))
                with _LK_MX:
                    ADDON_STATS["wm_snippets"] += 1
                    ADDON_STATS["wm_check_rules"] += used_check
                    ADDON_STATS["wm_plan_route"] += used_plan
                    ADDON_STATS["wm_routes_found"] += text.count("'found': True")
                level = ((kwargs.get("initial_state") or {}).get("current_frame") or {}).get("level")
                with _AH_USED_LOCK:
                    _AH_USED.add((getattr(_AH_TL, "agent", None), level))
        except Exception as exc:
            _mx_inc("wm_errors")
            _mx_err("wm", exc)
        return result

    _ta.run_sandboxed_python = run_sandboxed_python
    return True


if ENABLE_WORLD_MODEL_TOOLS:
    print("world-model tools installed:", install_world_model_tools())


# ---------------------------------------------------------------- K. GAME-DESIGN CATALOGUE (v22: knowledge only)
# v22 = the v13 catalogue minus every instruction (the 'fast start' recipe, 'state the goal as', 'test it first',
# 'test quickly'): it only describes what these games tend to look like and how they tend to work.
# The model meets every game cold, and most of its tokens go into finding out what the goal is and what the controls
# do. These games are built by one team on one engine, so they share a design language. This section was written by
# reading the source code of 15 public games and playing them in the offline engine: ar25 bp35 cd82 cn04 dc22 lf52
# lp85 ls20 sb26 sc25 sk48 sp80 tr87 tu93 wa30. The 10 games of our local test set (ft09 g50t ka59 m0r0 r11l re86
# s5i5 su15 tn36 vc33) are deliberately NOT in it, so a local run measures transfer to unseen games, as on the
# leaderboard. It goes into the system prompt (the same for the whole game: prefix cache unaffected).
GC_HEADER = "DESIGN PATTERNS FROM OTHER GAMES OF THIS BENCHMARK"
GC_TEXT = """DESIGN PATTERNS FROM OTHER GAMES OF THIS BENCHMARK (all made by the same designers on the same engine). The current game is a different game: these are possibilities, not facts about it.

Screen conventions seen in most games:
- A bar along one edge of the screen (a full row or column, sometimes part of one) that shrinks or fills with every action is a move/energy budget. When it runs out the attempt fails, or a life is lost; small squares beside it are often lives. Some objects refill it when touched.
- The goal is almost always shown on screen: a target picture to reproduce, a key/pattern drawn on a door or in a corner, a target sequence of colors along the top, a rule book, or outlines/markers where pieces must end up.
- Controls usually split into: click an object to SELECT it (it becomes highlighted), arrows MOVE the selected object, SPACE rotates it, cycles the selection, or RUNS/RELEASES something whose result plays out as an animation. Clicking empty space usually does nothing.
- Objects often move in steps of their own size; walls block; a blocked move changes nothing (but may still cost budget).
- A level keeps the earlier rules and adds ONE new element or twist (a new tile type, fog that hides the board except near you, rules that can themselves be edited).
- Losing an attempt restarts the level; earlier levels stay solved.

Mechanics from 25 other games (one line each):
1. Key and doors: move through a maze; stepping on special tiles changes the shape, color or rotation of a key shown in a corner; a door opens only when the key matches the pattern drawn on it.
2. Mirrors: move pieces or mirror lines; every piece is also drawn reflected across the mirrors; cover all marked target cells with pieces or reflections.
3. Platformer (side view): left/right walk, gravity makes you fall, spikes kill, reach the gem; clicking some blocks removes them, clicking others makes them grow into neighboring cells.
4. Painting: pick a color by clicking a swatch, move a paint bucket around a canvas, SPACE throws paint onto one half or one triangle of it; reproduce the target picture shown nearby.
5. Connectors: select a piece, move it, SPACE rotates it; marked cells on piece edges must touch matching marked cells of another piece (they light up when matched).
6. Keys and bridges: walk to the exit; a key opens the door of its color; stepping off solid ground costs a large part of the budget.
7. Peg jumps: click a peg, then a direction arrow, to jump it over a neighboring peg of the same color into an empty hole; the jumped peg disappears.
8. Ring buttons (clicks only): each button rotates a ring or track of tiles one step left or right; bring the marked tiles onto their goal markers.
9. Program slots: fill empty slots with colored tiles from a palette, then SPACE runs a cursor through the slots; the run must produce the color sequence shown at the top; some tiles call another row like a subroutine.
10. Spell pad: arrows move a character; drawing a spell's pattern on a 3x3 pad of buttons casts it; a spell book shows the patterns.
11. Extending arms: select an arm; pressing toward its facing extends it and pushes blocks, the opposite direction retracts it, sideways slides it along its rail; push colored blocks onto their targets.
12. Liquid flow: move deflector pieces, then SPACE pours liquid from a source; it spreads and is redirected; fill every cup without touching hazards; only a few pours allowed.
13. Cipher: a rule book maps glyph sequences to glyph sequences; write the translation of the given input into an answer row (LEFT/RIGHT pick a slot, UP/DOWN change its glyph).
14. Followers: step through a maze; after each of your moves other creatures move, some repeating your moves; every one of your tokens must end on an exit.
15. Warehouse: walk around; SPACE picks up or drops an adjacent crate; helper characters carry crates on their own; get every crate onto a goal zone.
16. Pattern toggles: a small icon shows a target pattern; clicking a cell of a panel toggles its color; reproduce the icon's pattern in the panel; later, special tool cells toggle themselves and their four neighbours.
17. Replay ghosts: SPACE ends an attempt and restarts the level, and every earlier attempt replays as a gray ghost; a ghost standing on a button or plate holds a door or gate open for the next attempt, so plan attempts that help each other.
18. Token sockets: several same-shaped tokens; the marked one moves in steps of its own size, clicking another token gives it control; park every token in the ring of its size; later, pushable plugs and elevator blocks.
19. Mirror pair: two blocks move together - LEFT/RIGHT mirrored, UP/DOWN shared; bring them together or swap them; hazard tiles reset both; clicking a special token freezes the pair and lets you steer the token instead.
20. Bead chain: a chain of beads (head, middle, tail); clicking a cell sends the tail there and the middle jumps to the midpoint; clicking the head swaps head and tail; bring the colored middle onto the ring of its color; moves into water or obstacles are rejected; painter tiles recolor half of a bead.
21. Sweeping crosses: each colored cross or ring collects the dots of its color that its arms or outline cover; SPACE switches which shape the arrows move; colored pads repaint a shape that touches them; collect every dot.
22. Piston panels: each colored piston has a small two-cell control panel; clicking the cell whose glyph points a way moves the piston that way; land each piston's marker on its matching cross; one panel can drive every piece of its color.
23. Trails and spawning: a token moves along a dotted trail when you click a dot a few steps ahead; later, clicking a pair of marker dots spawns a small block, a block steps toward the edge or corner of it you click, and small blocks merge into bigger ones in the color/size order a legend shows.
24. Program glyphs: a row of glyphs whose bars toggle black/gray when clicked is a program; a run button executes it step by step, moving a plug toward the socket of its shape; a second panel can display a recipe or hint program and small cards beside it switch which hint is shown; later glyph combinations rotate or paint the plug.
25. Sliding sheets: clicking the colored blocks on either side of a wall slides a band and the player riding its edge; align the player's block with the slot in the wall; gates in the walls pass pieces between bands under conditions.

Lessons from many plays of these games:
- One click or key press that changed nothing is weak evidence: the same object often responds when clicked on another edge or corner, or after something else has changed.
- A second panel that mirrors the controls or the board usually displays a target, a recipe or a hint; small buttons next to it may change what it shows.
- If a level seems to lack an action it needs (a direction, a color, a way through), the level's NEW element is usually what provides it.
- When every control is understood and the level is still not solved, the reading of the goal is usually what is wrong: re-read what the board displays."""

# v59c CLEAN TEST: the line of GC_TEXT that describes each public game (lines 1-15 written in v13 from 15 games' source
# and play, 16-25 added in v56 in alphabetical order). In a test run on the public games the fresh readers would
# otherwise read the rule of the very game being played; the contest's hidden games have no line, so there the
# catalogue is unchanged.
GC_LINE_OF_GAME = {"ls20": 1, "ar25": 2, "bp35": 3, "cd82": 4, "cn04": 5, "dc22": 6, "lf52": 7, "lp85": 8, "sb26": 9,
                   "sc25": 10, "sk48": 11, "sp80": 12, "tr87": 13, "tu93": 14, "wa30": 15, "ft09": 16, "g50t": 17,
                   "ka59": 18, "m0r0": 19, "r11l": 20, "re86": 21, "s5i5": 22, "su15": 23, "tn36": 24, "vc33": 25}
ADDON_STATS.update({"gc_reader_hidden": 0, "gc_reader_unknown": 0})


def gc_reader_text():
    """The catalogue the fresh readers see (called on the player's thread, where section X's _XG_TL names the game).
    Contest: GC_TEXT unchanged. Test run: without the line of the game being played."""
    text = globals().get("GC_TEXT")
    if not text or globals().get("TRUE_SUBMISSION", True):
        return text
    n = GC_LINE_OF_GAME.get(getattr(globals().get("_XG_TL"), "name", None))
    if not n:
        _mx_inc("gc_reader_unknown")
        return text
    rows = text.split("\n")
    kept = [r for r in rows if not r.startswith(f"{n}. ")]
    if len(kept) != len(rows) - 1:
        _mx_inc("gc_reader_unknown")
        return text
    _mx_inc("gc_reader_hidden")
    return "\n".join(kept).replace("Mechanics from 25 other games", "Mechanics from 24 other games", 1)


ADDON_STATS.update({"gc_prompts": 0, "gc_missing_anchor": 0, "gc_errors": 0, "gc_last_error": ""})
GC_ANCHOR = "\n\nAction meanings (use only actions currently available):"
GC_FALLBACK = "\n\nWORLD-MODEL TOOLS"


def install_game_catalogue():
    import inference.agent.tool_agent as _ta
    inner = _ta._build_system_prompt
    if getattr(inner, "_gc_wrapped", False):
        return False

    def _build_system_prompt(*args, **kwargs):
        prompt = inner(*args, **kwargs)
        try:
            if GC_HEADER in prompt:
                return prompt
            block = "\n\n" + GC_TEXT
            if GC_ANCHOR in prompt:           # right before the action meanings: part of "understanding the game"
                prompt = prompt.replace(GC_ANCHOR, block + GC_ANCHOR, 1)
            elif GC_FALLBACK in prompt:       # layout changed: still deliver it, ahead of the world-model tools
                _mx_inc("gc_missing_anchor")
                prompt = prompt.replace(GC_FALLBACK, block + GC_FALLBACK, 1)
            else:
                _mx_inc("gc_missing_anchor")
                prompt = prompt.rstrip("\n") + block
            _mx_inc("gc_prompts")
        except Exception as exc:  # never break an agent because of the add-on
            _mx_inc("gc_errors")
            _mx_err("gc", exc)
        return prompt

    _build_system_prompt._gc_wrapped = True
    _ta._build_system_prompt = _build_system_prompt
    return True


if ENABLE_GAME_CATALOGUE:
    print("game-design catalogue installed:", install_game_catalogue(), f"({len(GC_TEXT)} chars)")


## 6c. History de-duplication at trim time (v18, v19)

When the context is full and old messages are about to be dropped anyway, older turn openers first lose their repeated instruction lines, stale retained-function lists and board images (the images of the last 3 openers stay). The drain then removes fewer real turns of the model's own reasoning, code and results. The newest opener and all assistant/tool messages are never modified; below the context limit nothing changes. `ENABLE_HISTORY_DEDUP = False` disables it.

v19: older openers also lose harness blocks that a later message carries again (solved-level memory, action-effect summary, move-budget meter, tool tip); the newest copy of each always stays. `DEDUP_SUPERSEDED_BLOCKS = False` below gives v18 exactly.

In [ ]:
# ===== v18/v19: history de-duplication at trim time (installed on top of the patched harness) =====
# Every turn opener repeats the same fixed instruction lines, a fresh copy of the retained-function list and 1-2 board
# images, and all of it stays in history: about 17% of a full prompt is stale copies (measured on the v7 run). The
# model's own reasoning, code and tool results are what it needs from history, and losing them costs a lot (v17).
# When the trimmer is about to drop old messages anyway - the one moment the cached prefix is lost regardless - older
# openers lose those copies first, so the drain then removes fewer real turns. The newest opener is never touched (the
# model sees exactly what it saw before), images of the newest DEDUP_KEEP_IMAGE_MESSAGES openers stay, and assistant /
# tool messages are never modified. Below the budget nothing changes, so the prefix cache behaves as before.
# v19: older openers also lose harness blocks that a LATER message carries again (solved-level memory, action-effect
# summary, move-budget meter, tool tip): those blocks are cumulative or a per-turn reading, so only the newest copy
# matters, and it always stays (in v18 such stale copies were still ~11% of a prompt right after a drain).
import threading as _dd_threading

DEDUP_KEEP_IMAGE_MESSAGES = 3
DEDUP_IMAGE_NOTE = "[older board image omitted to save context]"
DEDUP_FUNCS_PREFIX = "Your retained functions from earlier Python calls in this game:"
# exact lines of the harness turn opener (prompts.py); a line is removed only on an exact match, so a reworded
# harness prompt simply means nothing is removed
DEDUP_STATIC_LINES = frozenset([
    "Only tool: `python`. It receives `current_frame`, `previous_frame`, `history`, `transitions`, `last_transition`, "
    "`valid_actions`, `last_action_call_result`, `frame_diff(before, after)`, and `action(actions)`.",
    "Only letter-coded board views and lightweight metadata are exposed; raw numeric color IDs are not available.",
    "Keep tool output compact: use `current_frame.segmentation` as the primary view, and `current_frame.ascii` only for "
    "a small specific region; never print full boards.",
    "For the most recent change, compare `previous_frame` to `current_frame`, or `last_transition.before_frame` to "
    "`last_transition.after_frame`; `history[-1].frame` is the current frame, not the previous one.",
    "Use Python to inspect the evidence from the newest history, and search or score candidate actions or short "
    "sequences against the current goal as you currently understand it.",
    "Focus on what changed most recently in `history`, update the target environment change if needed, and separate "
    "gameplay-object changes from HUD-only changes.",
    "When ready, call `action(actions)` from inside the `python` tool with the best valid action or ordered batch "
    "selected by your code. If your code has found a reliable short sequence, prefer batching it in one call.",
    "You may call `action(actions)` more than once in one Python snippet if your search or control loop needs it.",
    "When calling `python`, emit exactly the tool-call format shown elsewhere in this prompt for this model. Use only "
    "that format; do not add markdown fences, prose wrappers, or alternate tool-call syntax. Do not quote or place "
    "tool-call markup inside explanatory text; when you decide to call the tool, emit the tool call itself.",
    "If you use MOUSE, include integer row and col arguments.",
])
# (kind, first line starts with, last line starts with | None for a one-line block); matched by exact prefixes of the
# add-on texts, so a reworded block is simply never removed
DEDUP_BLOCKS = (
    ("memory", "=== SOLVED LEVELS (facts from this game", "=== END SOLVED LEVELS ==="),
    ("effects", "ACTION EFFECTS ON THIS LEVEL (computed by the harness", "(These are observations, not guarantees"),
    ("budget", "MOVE-BUDGET METER (measured by the harness):", None),
    ("tip", "TIP (costs no game actions):", None),
)
DEDUP_SUPERSEDED_BLOCKS = True
_DD_LOCK = _dd_threading.Lock()
DEDUP_STATS = {"trims": 0, "messages_compacted": 0, "lines_removed": 0, "chars_removed": 0, "images_removed": 0,
               "blocks_removed": 0, "tokens_freed": 0, "errors": 0}


def _dd_inc(key, n=1):
    with _DD_LOCK:
        DEDUP_STATS[key] += n


def _dd_is_image(part):
    return isinstance(part, dict) and part.get("type") in ("image_url", "image", "input_image")


def _dd_has_image(message):
    content = message.get("content") if isinstance(message, dict) else None
    return isinstance(content, list) and any(_dd_is_image(p) for p in content)


def _dd_block_spans(lines):
    """[(kind, first, last)] of the complete harness blocks in these lines (an unterminated block is ignored)."""
    spans, i = [], 0
    while i < len(lines):
        hit = next(((kind, end) for kind, start, end in DEDUP_BLOCKS if lines[i].startswith(start)), None)
        if hit is None:
            i += 1
            continue
        kind, end = hit
        last = i if end is None else next((k for k in range(i + 1, len(lines)) if lines[k].startswith(end)), None)
        if last is None:
            i += 1
            continue
        spans.append((kind, i, last))
        i = last + 1
    return spans


def _dd_text_kinds(text):
    return {kind for kind, _, _ in _dd_block_spans(text.split("\n"))}


def _dd_strip_text(text, drop_kinds=frozenset()):
    """Remove the fixed opener lines, retained-function lists and the blocks of `drop_kinds` (with the blank line the
    add-on put in front of each). Returns (text, lines removed, chars removed, blocks removed)."""
    lines = text.split("\n")
    drop = [line in DEDUP_STATIC_LINES or line.startswith(DEDUP_FUNCS_PREFIX) for line in lines]
    blocks = 0
    if drop_kinds:
        for kind, first, last in _dd_block_spans(lines):
            if kind in drop_kinds:
                blocks += 1
                for k in range(first, last + 1):
                    drop[k] = True
                if first > 0 and lines[first - 1] == "":
                    drop[first - 1] = True
    kept = [line for line, d in zip(lines, drop) if not d]
    removed = [line for line, d in zip(lines, drop) if d]
    return "\n".join(kept), len(removed), sum(len(line) + 1 for line in removed), blocks


def _dd_message_kinds(message):
    content = message.get("content") if isinstance(message, dict) else None
    if isinstance(content, str):
        return _dd_text_kinds(content)
    kinds = set()
    if isinstance(content, list):
        for part in content:
            if isinstance(part, dict) and part.get("type") == "text" and isinstance(part.get("text"), str):
                kinds |= _dd_text_kinds(part["text"])
    return kinds


def dedup_history(messages, keep_image_messages=DEDUP_KEEP_IMAGE_MESSAGES):
    """Return (messages, counts). messages[0] is the system prompt. Only user messages before the newest user message
    change; changed ones are new dicts (every other key kept), all other entries are the same objects."""
    users = [i for i in range(1, len(messages))
             if isinstance(messages[i], dict) and str(messages[i].get("role", "")).strip() == "user"]
    counts = {"messages_compacted": 0, "lines_removed": 0, "chars_removed": 0, "images_removed": 0,
              "blocks_removed": 0}
    if len(users) < 2:
        return messages, counts
    image_users = [i for i in users if _dd_has_image(messages[i])]
    keep_images = set(image_users[-keep_image_messages:]) if keep_image_messages > 0 else set()
    # newest first: a block kind already carried by a later user message is stale here
    drop_for, seen = {}, set()
    for i in reversed(users):
        kinds = _dd_message_kinds(messages[i]) if DEDUP_SUPERSEDED_BLOCKS else set()
        drop_for[i] = frozenset(kinds & seen)
        seen |= kinds
    out = list(messages)
    for i in users[:-1]:
        content = messages[i].get("content")
        if isinstance(content, str):
            new_content, lines, chars, blocks = _dd_strip_text(content, drop_for[i])
            images = 0
        elif isinstance(content, list):
            new_content, lines, chars, images, blocks = [], 0, 0, 0, 0
            for part in content:
                if _dd_is_image(part) and i not in keep_images:
                    new_content.append({"type": "text", "text": DEDUP_IMAGE_NOTE})
                    images += 1
                elif isinstance(part, dict) and part.get("type") == "text" and isinstance(part.get("text"), str):
                    text, n_lines, n_chars, n_blocks = _dd_strip_text(part["text"], drop_for[i])
                    lines += n_lines
                    chars += n_chars
                    blocks += n_blocks
                    new_content.append({**part, "text": text} if n_lines else part)
                else:
                    new_content.append(part)
        else:
            continue
        if lines or images:
            out[i] = {**messages[i], "content": new_content}
            counts["messages_compacted"] += 1
            counts["lines_removed"] += lines
            counts["chars_removed"] += chars
            counts["images_removed"] += images
            counts["blocks_removed"] += blocks
    return out, counts


def install_history_dedup():
    import inference.agent.tool_agent as _ta
    agent_cls = _ta.ToolAgent
    inner = agent_cls._trim_messages_for_context
    if getattr(inner, "_dd_wrapped", False):
        return False

    def _trim_messages_for_context(self, messages, *, tools=None, preserve_recent=1, extra_safety_tokens=0,
                                   force_drain=False):
        try:
            if messages and len(messages) > 2:
                budget = max(1, self._context_budget_tokens - max(0, extra_safety_tokens))
                used = self._estimate_request_input_tokens(messages, tools=tools)
                if used > budget or force_drain:
                    compacted, counts = dedup_history(messages)
                    if counts["messages_compacted"]:
                        freed = used - self._estimate_request_input_tokens(compacted, tools=tools)
                        # a drain was due anyway: compact, then still drain to the usual target so the next prefix
                        # break comes no sooner than before
                        out = inner(self, compacted, tools=tools, preserve_recent=preserve_recent,
                                    extra_safety_tokens=extra_safety_tokens, force_drain=True)
                        self._note_history_evicted()
                        _dd_inc("trims")
                        for key, value in counts.items():
                            _dd_inc(key, value)
                        _dd_inc("tokens_freed", max(0, int(freed)))
                        return out
        except Exception:
            _dd_inc("errors")
        return inner(self, messages, tools=tools, preserve_recent=preserve_recent,
                     extra_safety_tokens=extra_safety_tokens, force_drain=force_drain)

    _trim_messages_for_context._dd_wrapped = True
    agent_cls._trim_messages_for_context = _trim_messages_for_context
    return True


if ENABLE_HISTORY_DEDUP:
    print(f"v19 history de-duplication installed: {install_history_dedup()} "
          f"(superseded harness blocks: {DEDUP_SUPERSEDED_BLOCKS})", flush=True)
else:
    print("v19 history de-duplication disabled", flush=True)


## 6d. Exact reply replay (v20)

The harness stores every model reply with its blank lines deleted, so the next prompt stopped matching the generated tokens a few lines into the reply and the server re-read the whole reply (v19 diagnostics: 62% of turns). This cell keeps each reply exactly as the server returned it and puts that text into the copy of the messages sent to the server. The harness's own history, trimming, token estimates, memory, transcripts and request logs are unchanged. `ENABLE_EXACT_REPLAY = False` (config cell) disables it.

In [ ]:
# ===== v20: exact replay of the model's own replies (prefix-cache fix; installed on top of the patched harness) =====
# v19's server diagnostics: in 62% of follow-up requests the server re-read the model's whole previous reply (2,267
# tokens on average, about a quarter of all prefill work). Cause: the harness stores every reply through
# _normalize_message_content, which deletes blank lines ("\n\n" -> "\n", 423 of 462 cases). The token count stays the
# same, so nothing else notices, but the next prompt no longer repeats the generated tokens a few lines into the reply
# and the server cannot reuse its cached state past the previous prompt.
# This wrapper keeps each reply exactly as the server returned it and, in the COPY of the message list that is sent to
# the server, renders those history messages with the original text. A message is replaced only when the harness's
# own normalisation of a returned reply gives exactly its stored text, and a message keeps the same text for as long
# as it stays in history. The harness's history, trimming, token estimates, memory, transcripts and request logs are
# unchanged. ENABLE_EXACT_REPLAY = False (config cell) disables it.
import threading as _xr_threading

XR_FIELDS = ("reasoning_content", "reasoning", "content")
XR_RECENT_MAX = 32          # replies kept even before they show up in history
XR_STATS = {"replies": 0, "replies_changed_by_harness": 0, "restored": 0, "requests_changed": 0, "chars_added": 0,
            "errors": 0}
_XR_LOCK = _xr_threading.Lock()


def _xr_inc(key, n=1):
    with _XR_LOCK:
        XR_STATS[key] += n


def _xr_record(state, message, normalize):
    """Remember the server's text for every field the harness will store normalised."""
    if not isinstance(message, dict):
        return
    _xr_inc("replies")
    changed = False
    raws = []
    reasoning = message.get("reasoning")
    if reasoning in (None, ""):
        reasoning = message.get("reasoning_content")
    raws.append(reasoning)
    raws.append(message.get("content"))
    for raw in raws:
        if not isinstance(raw, str) or not raw.strip():
            continue
        key = normalize(raw)
        if not key or key == raw.strip():   # the chat template trims both ends: nothing to restore
            continue
        changed = True
        # never re-map a stored text: a message already sent must render the same way on every later request
        state["map"].setdefault(key, raw)
        state["recent"].append(key)
    del state["recent"][:-XR_RECENT_MAX]
    if changed:
        _xr_inc("replies_changed_by_harness")


def _xr_restore(state, messages):
    """Copy of `messages` with stored replies replaced by the server's text, or None when nothing changes."""
    mapping = state["map"]
    out = None
    used = set()
    for i, msg in enumerate(messages):
        if not isinstance(msg, dict) or msg.get("role") != "assistant":
            continue
        fixed = None
        for field in XR_FIELDS:
            value = msg.get(field)
            if not isinstance(value, str) or not value:
                continue
            raw = mapping.get(value)
            if raw is None:
                continue
            used.add(value)
            if raw != value:
                if fixed is None:
                    fixed = dict(msg)
                fixed[field] = raw
                if value not in state["seen"]:
                    _xr_inc("restored")
                    _xr_inc("chars_added", len(raw) - len(value))
        if fixed is not None:
            if out is None:
                out = list(messages)
            out[i] = fixed
    # keep only what is still in history (dropped messages never come back) plus the newest replies
    keep = used.union(state["recent"])
    state["map"] = {k: v for k, v in mapping.items() if k in keep}
    state["seen"] = used
    return out


def install_exact_replay():
    import inference.agent.tool_agent as _ta
    agent_cls = _ta.ToolAgent
    inner = agent_cls._chat_completion
    if getattr(inner, "_xr_wrapped", False):
        return False
    normalize = _ta._normalize_message_content

    def _chat_completion(self, messages, *args, **kwargs):
        send, state = messages, None
        try:
            state = self.__dict__.get("_xr_state")
            if state is None:
                state = self.__dict__.setdefault("_xr_state", {"map": {}, "recent": [], "seen": set()})
            fixed = _xr_restore(state, messages)
            if fixed is not None:
                send = fixed
                _xr_inc("requests_changed")
        except Exception:
            _xr_inc("errors")
            send = messages
        result = inner(self, send, *args, **kwargs)
        try:
            if state is not None:
                _xr_record(state, getattr(result, "message", None), normalize)
        except Exception:
            _xr_inc("errors")
        return result

    _chat_completion._xr_wrapped = True
    agent_cls._chat_completion = _chat_completion
    return True


if ENABLE_EXACT_REPLAY:
    print(f"v20 exact reply replay installed: {install_exact_replay()}", flush=True)
else:
    print("v20 exact reply replay disabled", flush=True)


## 6e. Competition survival (v32)
In a real submission every game action is an HTTP call to Kaggle's game server; a Save & Run plays the games inside this process and never uses that path.

* **Game-server patience.** arc_agi's client gives every call 10 s and drops a slower answer (the model then sees an error for an action the server may already have applied). Calls now get `GATEWAY_TIMEOUT_S` (60 s).
* **Scorecard keep-alive (submission only).** arc_agi's server closes a scorecard after 15 minutes without a game command, and one scorecard holds every game. When no command has been sent for `KEEPALIVE_IDLE_S` (4 min) - the model server is loading or being relaunched - one RESET goes to a game standing at the very start of level 1. The server answers it with the unchanged board (the level's action counter is still 0); the scorecard's idle timer restarts. Games that have acted on their level, cleared a level or ended are never touched.

At the end of a Save & Run, `run_survival_smoke()` tests both on the real arc_agi server code (taaf's localhost competition simulator, three public games) and writes `survival_smoke.json`.

**v45 keep-alive ladder.** v32's keep-alive could only use a game still untouched at the start of level 1; hours into a 110-game run there may be none left, and a model-server outage of 15 minutes (a watchdog restart re-reads the weights) would close the scorecard while the notebook plays on. Now it falls back to a game at the start of a later level (unchanged board), and only after 8 minutes of silence to restarting the current level of one game (level 1 with no points first; cleared levels stay cleared; won games never). The Save & Run smoke test checks every tier on the real game-server code.


In [ ]:
# ===== v32: competition survival - game-server patience + scorecard keep-alive (installed before the benchmark runs) =====
# In a real submission every game action is an HTTP call to Kaggle's game server (the "gateway"). A Save & Run plays
# the games inside this process, so two failure modes of that path can never show up locally:
#   1. arc_agi's client gives every game-server call 10 s. A slower answer is dropped: the action comes back to the
#      model as an error although the server may already have applied it.
#      -> calls to the game server get GATEWAY_TIMEOUT_S (60 s) instead.
#   2. arc_agi's server closes a scorecard after 15 minutes without a game command (STALE_MINUTES, default 15), and in
#      competition mode ONE scorecard holds every game: once it is closed no game can act for the rest of the run.
#      Nothing is sent while the model server loads (the weight load took 886 s on 2026-10-13; 260-510 s before) or
#      while the watchdog relaunches it after a crash.
#      -> when the game server has seen no command for KEEPALIVE_IDLE_S (4 min), one RESET goes to a game that stands
#         at the very start of level 1. The server answers it with the unchanged board (the level's action counter is
#         still 0) and records it as one action of that level; the scorecard's idle timer restarts.
#      v45 LADDER: once every game has started (hours into a 110-game contest run) v32 had no such target left, and a
#         model-server outage (a watchdog restart re-reads the weights from network storage: ~10-12 min, plus the
#         first new reply) could close the scorecard; the notebook then plays on for hours while nothing counts (a
#         9-hour run with a low score). Now, if there is no game at the start of level 1: a game at the start of a
#         later level (unchanged board, one action counted there); and only after KEEPALIVE_LAST_RESORT_IDLE_S (8 min)
#         of silence, a RESET that restarts the current level of a game part-way through one - a level-1 game with no
#         points first, else the lowest level - which is what the game's own RESET does; cleared levels stay cleared.
#         Won games are never touched. One last-resort RESET per outage: the restarted game is a free target after.
# run_survival_smoke() checks both on the real arc_agi server code at the end of a Save & Run (survival_smoke.json).
import importlib as _sv_importlib
import threading as _sv_threading
import time as _sv_time
import types as _sv_types
import weakref as _sv_weakref

SURVIVAL_STATS = {'patched_modules': [], 'tracking': False, 'keepalive_thread': False, 'gw_cmds': 0, 'gw_errors': 0,
                  'gw_slow_10s': 0, 'gw_max_s': 0.0, 'games_tracked': 0, 'keepalives': 0, 'keepalive_no_target': 0,
                  'keepalive_errors': 0, 'keepalive_board_changed': 0, 'keepalive_tiers': {}, 'errors': 0}
_SV_LAST_CMD = [_sv_time.time()]   # wall time of the newest game command the game server answered (or failed)
_SV_GAMES = []                      # weakrefs to started competition-mode GameAPI objects
_SV_LOCK = _sv_threading.Lock()
_SV_STOP = _sv_threading.Event()


def _sv_timeout(t):
    floor = float(GATEWAY_TIMEOUT_S)
    if t is None:
        return floor
    if isinstance(t, (tuple, list)) and len(t) == 2:
        return (t[0], None if t[1] is None else max(float(t[1]), floor))
    try:
        return max(float(t), floor)
    except (TypeError, ValueError):
        return t


def _sv_note_cmd(url, t0, failed):
    if '/api/cmd/' not in str(url):
        return
    now = _sv_time.time()
    _SV_LAST_CMD[0] = now
    SURVIVAL_STATS['gw_cmds'] += 1
    SURVIVAL_STATS['gw_errors'] += int(failed)
    SURVIVAL_STATS['gw_slow_10s'] += int(now - t0 > 10.0)
    SURVIVAL_STATS['gw_max_s'] = max(SURVIVAL_STATS['gw_max_s'], round(now - t0, 2))


def _sv_patient_requests(real):
    """Stand-in for the `requests` module inside arc_agi: same attributes, longer timeout on every call."""
    class _PatientSession(real.Session):
        def request(self, method, url, *args, **kwargs):
            kwargs['timeout'] = _sv_timeout(kwargs.get('timeout'))
            t0, failed = _sv_time.time(), True
            try:
                out = super().request(method, url, *args, **kwargs)
                failed = False
                return out
            finally:
                _sv_note_cmd(url, t0, failed)

    def _call(name):
        fn = getattr(real, name)

        def call(*args, **kwargs):
            kwargs['timeout'] = _sv_timeout(kwargs.get('timeout'))
            return fn(*args, **kwargs)
        return call

    shim = _sv_types.ModuleType('requests')
    shim.__dict__.update(real.__dict__)   # exceptions, cookies, adapters, ... stay the real ones
    shim.Session = shim.session = _PatientSession
    for name in ('request', 'get', 'post', 'put', 'patch', 'delete', 'head'):
        if hasattr(real, name):
            setattr(shim, name, _call(name))
    shim._sv_patient = True
    return shim


def _sv_install_patience():
    import requests as _sv_real
    for name in ('arc_agi.remote_wrapper', 'arc_agi.base'):
        try:
            mod = _sv_importlib.import_module(name)
            cur = getattr(mod, 'requests', None)
            if cur is None or getattr(cur, '_sv_patient', False):
                continue
            mod.requests = _sv_patient_requests(_sv_real)
            SURVIVAL_STATS['patched_modules'].append(name)
        except Exception as exc:   # an unexpected arc_agi layout leaves the stock client in place
            SURVIVAL_STATS['errors'] += 1
            print('v32 survival: game-server patience not installed in', name, repr(exc)[:200], flush=True)


def _sv_remote(game):
    import arc_agi as _sv_arc
    spec = getattr(game, 'arcade_spec', None)
    return spec is not None and (bool(getattr(spec, 'competition_sim', False))
                                 or spec.operation_mode == _sv_arc.OperationMode.COMPETITION)


def _sv_install_tracking():
    """Mark, per competition-mode game, whether it stands at the very start of a level (server-side counter 0)."""
    import arcengine as _sv_ae
    import taaf.game_api as _sv_ga
    G = _sv_ga.GameAPI
    if getattr(G, '_sv_tracked', False):
        return
    start0, exec0 = G._start_game, G._execute_action
    reset_id = _sv_ae.GameAction.RESET
    ended = {_sv_ae.GameState.WIN, _sv_ae.GameState.GAME_OVER}

    def _start_game(self, session):
        state = start0(self, session)
        try:
            if _sv_remote(self):
                self.__dict__['_sv_fresh'] = True      # arcade.make ends with a RESET
                self.__dict__['_sv_last'] = _sv_time.time()
                with _SV_LOCK:
                    _SV_GAMES.append(_sv_weakref.ref(self))
                SURVIVAL_STATS['games_tracked'] += 1
        except Exception:
            SURVIVAL_STATS['errors'] += 1
        return state

    def _execute_action(self, action):
        if '_sv_last' not in self.__dict__:          # offline game (Save & Run): not tracked
            return exec0(self, action)
        try:
            before = int(self.env.observation_space.levels_completed)
        except Exception:
            before = None
        try:
            state = exec0(self, action)
        except Exception:
            self.__dict__['_sv_fresh'] = False         # server state unknown: never a keep-alive target
            self.__dict__['_sv_last'] = _sv_time.time()
            raise
        try:
            raw = state.raw
            if raw.state in ended:
                fresh = False
            elif action.id == reset_id:
                fresh = True
            else:   # a level transition zeroes the level's action counter
                fresh = before is not None and int(raw.levels_completed) > before
            self.__dict__['_sv_fresh'] = bool(fresh)
            self.__dict__['_sv_last'] = _sv_time.time()
        except Exception:
            self.__dict__['_sv_fresh'] = False
            SURVIVAL_STATS['errors'] += 1
        return state

    G._start_game, G._execute_action, G._sv_tracked = _start_game, _execute_action, True
    SURVIVAL_STATS['tracking'] = True


def _sv_last_board(obs):
    import numpy as _sv_np
    try:
        frames = list(obs.frame or [])
        return _sv_np.asarray(frames[-1]) if frames else None
    except Exception:
        return None


def _sv_keepalive_pick(now=None):
    """(game, tier) for the next keep-alive RESET, or (None, None). v45 ladder - the cheapest target first:
      1 = a game standing at the very start of level 1 (server counter 0: the RESET returns the unchanged board and
          counts one action of level 1) - v32's only target;
      2 = a game standing at the start of a later level, right after clearing one (counter 0 again: unchanged board,
          one action counted on that level);
      3 = a game still on level 1 with no level cleared, part-way through it (the RESET restarts its level 1 - the
          same thing its own RESET does - and it has no points yet);
      4 = any other game part-way through a level (the RESET restarts that level; cleared levels stay cleared).
    Tiers 1-2 change nothing on the board and are used after KEEPALIVE_IDLE_S without a game command; tiers 3-4 only
    after KEEPALIVE_LAST_RESORT_IDLE_S (the game server closes the scorecard for good at 15 min, and every game's
    progress after that is lost). Won games are never touched; within a tier the game idle longest goes first."""
    import arcengine as _sv_ae
    now = _sv_time.time() if now is None else now
    idle = now - _SV_LAST_CMD[0]
    ended = {_sv_ae.GameState.WIN, _sv_ae.GameState.GAME_OVER}
    best = {}
    with _SV_LOCK:
        _SV_GAMES[:] = [r for r in _SV_GAMES if r() is not None]
        for ref in _SV_GAMES:
            g = ref()
            env = getattr(g, 'env', None)
            obs = getattr(env, 'observation_space', None) if env is not None else None
            card = g.__dict__.get('_competition_scorecard')
            if obs is None or obs.state in ended or (card is not None and card.closed):
                continue
            levels = int(obs.levels_completed)
            if g.__dict__.get('_sv_fresh'):
                tier = 1 if levels == 0 else 2
            else:
                tier = 3 if levels == 0 else 4
            key = (levels, g.__dict__.get('_sv_last', 0.0))
            if tier not in best or key < best[tier][0]:
                best[tier] = (key, g)
    for tier in (1, 2, 3, 4):
        if tier in best and (tier <= 2 or idle >= KEEPALIVE_LAST_RESORT_IDLE_S):
            return best[tier][1], tier
    return None, None


def _sv_keepalive_tick(now=None):
    """If the game server has seen no command for KEEPALIVE_IDLE_S, send one RESET to the cheapest target of the
    ladder (_sv_keepalive_pick). Returns the game it reset, else None."""
    import numpy as _sv_np
    now = _sv_time.time() if now is None else now
    if now - _SV_LAST_CMD[0] < KEEPALIVE_IDLE_S:
        return None
    best, tier = _sv_keepalive_pick(now)
    if best is None:
        # nothing cheap to send yet: look again on the next poll (v32 waited a whole idle period here, which would also
        # have postponed the last-resort tiers past the 15-minute close)
        SURVIVAL_STATS['keepalive_no_target'] += 1
        return None
    try:
        board0 = _sv_last_board(best.env.observation_space)
        obs = best.env.reset()
        best.__dict__['_sv_last'] = _sv_time.time()
        if obs is None:
            SURVIVAL_STATS['keepalive_errors'] += 1
            best.__dict__['_sv_fresh'] = False
            return None
        SURVIVAL_STATS['keepalives'] += 1
        SURVIVAL_STATS['keepalive_tiers'][str(tier)] = SURVIVAL_STATS['keepalive_tiers'].get(str(tier), 0) + 1
        board1 = _sv_last_board(obs)
        changed = board0 is None or board1 is None or not _sv_np.array_equal(board0, board1)
        if tier <= 2 and changed:
            SURVIVAL_STATS['keepalive_board_changed'] += 1   # never expected at a level start
        # after any RESET the game stands at the start of its level (counter 0): a cheap target from now on, but only
        # if the board did what the tier expects
        best.__dict__['_sv_fresh'] = (not changed) if tier <= 2 else changed
        if changed:   # keep the harness in step with the board the game server now shows
            try:
                import taaf.game as _sv_tg
                best._current_state = _sv_tg.GameState(raw=obs)
            except Exception:
                SURVIVAL_STATS['errors'] += 1
        return best
    except Exception:
        SURVIVAL_STATS['keepalive_errors'] += 1
        best.__dict__['_sv_fresh'] = False
        return None


def _sv_keepalive_loop(poll_s=15.0):
    while not _SV_STOP.wait(poll_s):
        try:
            _sv_keepalive_tick()
        except Exception:
            SURVIVAL_STATS['errors'] += 1


def _sv_start_keepalive():
    if not SURVIVAL_STATS['keepalive_thread']:
        _sv_threading.Thread(target=_sv_keepalive_loop, name='scorecard-keepalive', daemon=True).start()
        SURVIVAL_STATS['keepalive_thread'] = True


def run_survival_smoke(env_dir, out_path=None):
    """Save & Run only (after the benchmark): drive the real arc_agi server code in competition mode on localhost
    (taaf's simulator, 3 public games) through the patched client. Measurement only; never raises."""
    res = {'ok': False}
    srv = None
    try:
        import arc_agi as _sv_arc
        import arcengine as _sv_ae
        import taaf.game as _sv_tg
        import taaf.game_api as _sv_ga
        from taaf.competition_arcade import CompetitionArcadeServer
        try:
            import importlib.metadata as _sv_md
            res['arc_agi'] = _sv_md.version('arc-agi')
        except Exception:
            res['arc_agi'] = getattr(_sv_arc, '__version__', '?')
        srv = CompetitionArcadeServer(game_ids=['ft09', 'ls20', 'vc33'], environments_dir=str(env_dir)).start()
        inner, slow = srv._app.wsgi_app, {'armed': False}

        def _slow_app(environ, start_response):
            if slow['armed'] and '/api/cmd/' in environ.get('PATH_INFO', ''):
                slow['armed'] = False
                _sv_time.sleep(12.0)   # longer than the stock 10 s client timeout
            return inner(environ, start_response)
        srv._app.wsgi_app = _slow_app
        session = _sv_tg.RunSession(record_intermediate_states=False)
        games = {g.split('-')[0]: _sv_ga.GameAPI(env_name=g, arcade_spec=srv.arcade_spec) for g in srv.exposed_game_ids}
        for g in games.values():
            g.start_game(session)
        import numpy as _sv_np
        start_boards = {k: _sv_last_board(g.env.observation_space) for k, g in games.items()}
        server_env = {w.environment_info.game_id.split('-')[0]: w for w in srv._api._environmentCache.values()}
        res['tracked_fresh_at_start'] = sum(bool(g.__dict__.get('_sv_fresh')) for g in games.values())
        # 1. a 12 s game-server answer is waited for
        mover = games['ls20']
        act = next(a for a in mover.current_state.raw.available_actions if a in (1, 2, 3, 4))
        slow['armed'] = True
        t0 = _sv_time.time()
        try:
            mover.execute_action(_sv_ae.ActionInput(id=_sv_ae.GameAction.from_id(act), data={}))
            res['slow_cmd_ok'] = True
        except Exception as exc:
            res['slow_cmd_ok'], res['slow_cmd_error'] = False, repr(exc)[:200]
        res['slow_cmd_s'] = round(_sv_time.time() - t0, 1)
        res['mover_fresh_after_move'] = bool(mover.__dict__.get('_sv_fresh'))
        # 2. a cleared level starts a fresh level: tracked fresh, server counter 0 (ft09 level 1, recorded clicks)
        clicker = games['ft09']
        for x, y in ((38, 38), (39, 47), (55, 47), (39, 55)):
            clicker.execute_action(_sv_ae.ActionInput(id=_sv_ae.GameAction.ACTION6, data={'x': x, 'y': y}))
        res['ft09_levels'] = int(clicker.current_state.raw.levels_completed)
        ft09_l2_board = _sv_last_board(clicker.env.observation_space)
        res['ft09_fresh_after_clear'] = bool(clicker.__dict__.get('_sv_fresh'))
        res['ft09_server_counter'] = int(server_env['ft09']._game._action_count)
        # 3. keep-alive after an idle spell: only the untouched level-1 game (vc33) qualifies
        card = srv._arcade.scorecard_manager.scorecards[games['vc33']._scorecard_id]
        lu0 = card.last_update
        _SV_LAST_CMD[0] = _sv_time.time() - KEEPALIVE_IDLE_S - 1
        n0, c0 = SURVIVAL_STATS['keepalives'], SURVIVAL_STATS['keepalive_board_changed']
        target = _sv_keepalive_tick()
        res['keepalive_sent'] = SURVIVAL_STATS['keepalives'] - n0
        res['keepalive_target'] = target.env.environment_info.game_id.split('-')[0] if target is not None else None
        res['keepalive_board_changed'] = SURVIVAL_STATS['keepalive_board_changed'] - c0
        res['scorecard_timer_refreshed'] = card.last_update > lu0
        res['vc33_server_counter'] = int(server_env['vc33']._game._action_count)
        res['ls20_server_counter'] = int(server_env['ls20']._game._action_count)
        res['no_keepalive_when_recent'] = _sv_keepalive_tick() is None
        # 4. v45 ladder: tier 2 (start of a later level), nothing before the last-resort idle, tier 3 (level 1
        #    part-way: level 1 restarts), tier 1 on the restarted game, tier 4 (later level part-way: level kept)
        lad = {}

        def _tick(idle_s):
            _SV_LAST_CMD[0] = _sv_time.time() - idle_s
            lu = card.last_update
            t = _sv_keepalive_tick()
            return t, card.last_update > lu

        games['vc33'].execute_action(_sv_ae.ActionInput(id=_sv_ae.GameAction.ACTION6, data={'x': 10, 'y': 10}))
        t, r = _tick(KEEPALIVE_IDLE_S + 60)
        lad['tier2'] = bool(t is clicker and r and int(clicker.env.observation_space.levels_completed) == 1
                            and _sv_np.array_equal(_sv_last_board(clicker.env.observation_space), ft09_l2_board)
                            and int(server_env['ft09']._game._action_count) == 0)
        clicker.execute_action(_sv_ae.ActionInput(id=_sv_ae.GameAction.ACTION6, data={'x': 20, 'y': 20}))
        t1, r1 = _tick(KEEPALIVE_IDLE_S + 60)
        t2, r2 = _tick(KEEPALIVE_LAST_RESORT_IDLE_S - 10)
        lad['nothing_before_last_resort'] = t1 is None and t2 is None and not r1 and not r2
        t, r = _tick(KEEPALIVE_LAST_RESORT_IDLE_S + 20)
        lad['tier3'] = bool(t is mover and r and int(server_env['ls20']._game._action_count) == 0
                            and _sv_np.array_equal(_sv_last_board(mover.env.observation_space), start_boards['ls20'])
                            and mover.__dict__.get('_sv_fresh'))
        t, r = _tick(KEEPALIVE_IDLE_S + 60)
        lad['tier1_after_restart'] = bool(t is mover and r)
        saved = list(_SV_GAMES)
        _SV_GAMES[:] = [ref for ref in saved if ref() is clicker]
        t, r = _tick(KEEPALIVE_LAST_RESORT_IDLE_S + 20)
        _SV_GAMES[:] = saved
        lad['tier4'] = bool(t is clicker and r and int(clicker.env.observation_space.levels_completed) == 1
                            and _sv_np.array_equal(_sv_last_board(clicker.env.observation_space), ft09_l2_board)
                            and int(server_env['ft09']._game._action_count) == 0)
        lad['tiers'] = dict(SURVIVAL_STATS['keepalive_tiers'])
        res['ladder'] = lad
        res['ladder_ok'] = all(v for k, v in lad.items() if k != 'tiers')
        res['ok'] = bool(res.get('slow_cmd_ok') and res['tracked_fresh_at_start'] == 3 and not res['mover_fresh_after_move']
                         and res['ft09_levels'] == 1 and res['ft09_fresh_after_clear'] and res['ft09_server_counter'] == 0
                         and res['keepalive_sent'] == 1 and res['keepalive_target'] == 'vc33'
                         and res['keepalive_board_changed'] == 0 and res['scorecard_timer_refreshed']
                         and res['vc33_server_counter'] == 0 and res['ls20_server_counter'] == 1
                         and res['no_keepalive_when_recent'] and res['ladder_ok'])
    except Exception as exc:
        res['error'] = repr(exc)[:300]
    finally:
        if srv is not None:
            try:
                srv.stop()
            except Exception:
                pass
    res['stats'] = dict(SURVIVAL_STATS)
    try:
        if out_path:
            Path(out_path).write_text(json.dumps(res, indent=1, default=str))
    except Exception:
        pass
    print('SURVIVAL_SMOKE', json.dumps(res, default=str), flush=True)
    return res


if ENABLE_GATEWAY_PATIENCE:
    _sv_install_patience()
if ENABLE_SCORECARD_KEEPALIVE:
    _sv_install_tracking()
    if TRUE_SUBMISSION:
        _sv_start_keepalive()
print('v32 survival:', json.dumps({k: SURVIVAL_STATS[k] for k in ('patched_modules', 'tracking', 'keepalive_thread')}),
      flush=True)


## 6f. Trim memory (v41)

When the context is full and the oldest turns are dropped, the model loses its own interpretation of them (the board and action record stays in `history`). This cell keeps a verbatim excerpt of the reasoning of each dropped turn of the current level - its first and last 400 characters, plus the actions that turn executed - in one message at the head of the retained history (oldest first, at most 10,000 characters, the newest win). No summaries, no extra requests, no generated tokens; the message changes only when a trim happens. A new level starts a new list. `ENABLE_TRIM_MEMORY = False` (config cell) disables it.

In [ ]:
# ===== v41: trim memory - the model's own reasoning from turns a history trim removes (installed on top of the harness) =====
# v40 (20 games, history trimmed at 68K down to ~35K) cleared about 20% fewer levels per generated token than every
# earlier run once its trims started, and right after each trim its clear rate halved. A trim does not lose the facts:
# every frame and action of the game stays in `history` in Python. It loses the model's interpretation - its findings,
# the ideas it ruled out, its plan.
# When a trim drops assistant turns of the CURRENT level, this keeps a short verbatim excerpt of each one's reasoning
# (its first and last few hundred characters, usually the newest finding and the next test) plus the actions that turn
# executed. The excerpts go, oldest first, into one user message at the head of the retained history. Nothing is
# summarised or rewritten, no extra request is made and no model tokens are spent. The message is rebuilt only when a
# trim happens - the one moment the cached prefix is lost anyway - so between trims the prefix cache works as before.
# A new level starts a new list (cleared levels are covered by the solved-level memory). Below the budget nothing
# changes. ENABLE_TRIM_MEMORY = False (config cell) disables it.
import re as _tm_re
import threading as _tm_threading

TM_HEAD_CHARS = 400          # start of each removed reasoning: usually what the last result showed
TM_TAIL_CHARS = 400          # end of it: usually the conclusion and the next test
TM_EXEC_CHARS = 200          # the actions that turn executed (from the next turn opener)
TM_MAX_CHARS = 10000         # the whole message (~2.5K tokens); the newest excerpts win
TM_MAX_ITEMS = 80            # excerpts remembered per level
TM_HEADER = ("EARLIER ON THIS LEVEL - excerpts of your own reasoning from turns that were removed from your context to "
             "save memory (oldest first, verbatim; some ideas here may have been ruled out later). The full board and "
             "action record of this level is still in `history` in Python.")
TM_STATS = {"trims": 0, "excerpts": 0, "digests": 0, "digest_chars_max": 0, "level_resets": 0, "no_level": 0,
            "errors": 0}
_TM_LOCK = _tm_threading.Lock()
_TM_STATE_RX = _tm_re.compile(r"Current state: step (\d+), level (\d+)\.")
_TM_EXEC_RX = _tm_re.compile(r"Executed actions: ([^\n]*)")


def _tm_inc(key, n=1):
    with _TM_LOCK:
        TM_STATS[key] += n


def _tm_note_max(key, value):
    with _TM_LOCK:
        TM_STATS[key] = max(TM_STATS[key], value)


def _tm_role(message):
    return str(message.get("role", "")).strip() if isinstance(message, dict) else ""


def _tm_text(message):
    content = message.get("content") if isinstance(message, dict) else None
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "\n".join(p["text"] for p in content
                         if isinstance(p, dict) and p.get("type") == "text" and isinstance(p.get("text"), str))
    return ""


def _tm_is_digest(message):
    return (_tm_role(message) == "user" and isinstance(message.get("content"), str)
            and message["content"].startswith(TM_HEADER))


def _tm_state(text):
    """(step, level) of the LAST state line in an opener, or None."""
    found = _TM_STATE_RX.findall(text)
    return (int(found[-1][0]), int(found[-1][1])) if found else None


def _tm_reasoning(message):
    for key in ("reasoning", "reasoning_content"):
        value = message.get(key)
        if isinstance(value, str) and value.strip():
            return value.strip()
    return ""


def _tm_excerpt(text):
    if len(text) <= TM_HEAD_CHARS + TM_TAIL_CHARS + 40:
        return text
    return text[:TM_HEAD_CHARS].rstrip() + " [...] " + text[-TM_TAIL_CHARS:].lstrip()


def _tm_digest(items):
    """The message text: header, then the newest excerpts that fit TM_MAX_CHARS, oldest first."""
    parts, total = [], len(TM_HEADER) + 2
    for step, excerpt, executed in reversed(items):
        label = f"[step {step}]" if step is not None else "[earlier]"
        entry = f"{label} {excerpt}" + (f"\n-> executed: {executed}" if executed else "")
        if parts and total + len(entry) + 2 > TM_MAX_CHARS:
            break
        parts.append(entry)
        total += len(entry) + 2
    return TM_HEADER + "\n\n" + "\n\n".join(reversed(parts))


def trim_memory_update(state, messages, out):
    """Pure core: given the agent's state, the trimmer's input `messages` and output `out` (both with the system
    prompt first), return the history to send. `out` is returned unchanged unless the trim dropped assistant turns or
    the previous excerpt message."""
    kept = {id(m) for m in out}
    dropped = {id(m) for m in messages[1:] if _tm_role(m) == "assistant" and id(m) not in kept}
    had_digest = any(_tm_is_digest(m) for m in messages[1:])
    keeps_digest = any(_tm_is_digest(m) for m in out[1:])
    if not dropped and (keeps_digest or not had_digest):
        return out, 0
    level_now = None
    for m in reversed(messages[1:]):
        if _tm_role(m) == "user" and not _tm_is_digest(m):
            st = _tm_state(_tm_text(m))
            if st:
                level_now = st[1]
                break
    if level_now is None:
        _tm_inc("no_level")
        return out, 0
    if state["level"] != level_now:
        if state["items"]:
            _tm_inc("level_resets")
        state["level"], state["items"] = level_now, []
    step = level = None
    last = None
    new = 0
    for m in messages[1:]:
        role = _tm_role(m)
        if role == "user":
            if _tm_is_digest(m):
                continue
            text = _tm_text(m)
            if last is not None:
                ex = _TM_EXEC_RX.search(text)
                if ex and not last[2]:
                    last[2] = ex.group(1).strip()[:TM_EXEC_CHARS]
                last = None
            st = _tm_state(text)
            if st:
                step, level = st
        elif role == "assistant" and id(m) in dropped and level == level_now:
            reasoning = _tm_reasoning(m)
            if reasoning:
                last = [step, _tm_excerpt(reasoning), ""]
                state["items"].append(last)
                new += 1
    del state["items"][:-TM_MAX_ITEMS]
    history = [m for m in out[1:] if not _tm_is_digest(m)]
    if state["items"]:
        digest = _tm_digest(state["items"])
        _tm_inc("digests")
        _tm_note_max("digest_chars_max", len(digest))
        return [out[0], {"role": "user", "content": digest}, *history], new
    return [out[0], *history], new


def install_trim_memory():
    import inference.agent.tool_agent as _ta
    agent_cls = _ta.ToolAgent
    inner = agent_cls._trim_messages_for_context
    if getattr(inner, "_tm_wrapped", False):
        return False

    def _trim_messages_for_context(self, messages, *, tools=None, preserve_recent=1, extra_safety_tokens=0,
                                   force_drain=False):
        out = inner(self, messages, tools=tools, preserve_recent=preserve_recent,
                    extra_safety_tokens=extra_safety_tokens, force_drain=force_drain)
        try:
            if messages and out and len(messages) > 1 and len(out) > 1:
                state = self.__dict__.get("_tm_state")
                if state is None:
                    state = self.__dict__.setdefault("_tm_state", {"level": None, "items": []})
                result, new = trim_memory_update(state, messages, out)
                if result is not out:
                    _tm_inc("trims")
                    _tm_inc("excerpts", new)
                    return result
        except Exception:
            _tm_inc("errors")
        return out

    _trim_messages_for_context._tm_wrapped = True
    agent_cls._trim_messages_for_context = _trim_messages_for_context
    return True


if ENABLE_TRIM_MEMORY:
    print(f"v41 trim memory installed: {install_trim_memory()}", flush=True)
else:
    print("v41 trim memory disabled", flush=True)


## 7. Customization hook

Optional: tweak `bm`, `bm.games`, or `bm.solver` here before the run starts — the safe place
for one-off experiments once the deployed bundle has loaded.

In [ ]:
# Make one-off changes to `bm`, `bm.games`, or `bm.solver` here before the run starts.
bm.solver.max_runtime_s_per_game = 48*60 #7920.0
bm.solver.analyzer_timeout = 900.0
bm.solver.concurrency = 10
bm.solver.max_actions_per_game = None
bm.solver.save_request_logs = True
bm.n_passes = 1
bm.game_weights = None

demo_excluded_games = [] if TRUE_SUBMISSION else [g for g in _ALL25_GAMES if g not in _LOCAL_SETS[LOCAL_GAME_SET]]

if TRUE_SUBMISSION:  # overrides for contest run
    bm.solver.save_request_logs = False
    bm.n_passes = 1
    if USE_PRIORITY_SCHEDULING:
        bm.solver.concurrency = 120  # game count should be 110
        bm.solver.max_runtime_s_per_game = 532*60
else:
    if USE_PRIORITY_SCHEDULING:
        bm.solver.concurrency = (25 - len(demo_excluded_games)) * bm.n_passes
        bm.solver.max_runtime_s_per_game = LOCAL_PLAY_MINUTES*60
    if float(getattr(target, 'max_runtime_s', 0.0) or 0.0) != 32400.0:
        raise RuntimeError(
            f'Expected the 32400-second notebook budget, got {target.max_runtime_s!r}.'
        )

# show config
print('benchmark.solver:', bm.solver)

## 8. Run the system monitor

Periodically show free main memory, VRAM and disk space.

In [ ]:
# --- system census, one line every 10 minutes -------------------------------
# Everything reads as used/total. RAM is total - MemAvailable, which runs higher
# than the "used" column in `free -h`: that one excludes reclaimable page cache,
# and MemAvailable is the figure that actually predicts an OOM. Disk likewise
# counts the root reserve as unavailable, matching df.
#
# A daemon thread rather than a hook in the run loop: the point of watching a
# long run is to see what happens when it stops behaving, and a print driven by
# the harness stops with it. This keeps reporting through a deadlock.
import os, subprocess, threading, sys

def _sysline(paths=("/", "/kaggle/working")):
    GB = 1024 ** 3
    out = []
    mem = {}
    try:
        for line in open("/proc/meminfo"):
            k, _, rest = line.partition(":")
            p = rest.split()
            if p:
                mem[k] = int(p[0]) * 1024
    except OSError:
        pass
    if mem.get("MemTotal") and mem.get("MemAvailable") is not None:
        out.append(f"RAM {(mem['MemTotal']-mem['MemAvailable'])/GB:.1f}/{mem['MemTotal']/GB:.1f}")
    if mem.get("SwapTotal"):
        out.append(f"swap {(mem['SwapTotal']-mem['SwapFree'])/GB:.1f}/{mem['SwapTotal']/GB:.1f}")
    else:
        out.append("swap none")
    st = {}
    try:
        for line in open("/proc/self/status"):
            k, _, rest = line.partition(":")
            p = rest.split()
            if k in ("VmRSS", "Threads") and p:
                st[k] = int(p[0])
    except OSError:
        pass
    proc = []
    if "VmRSS" in st:
        proc.append(f"{st['VmRSS']*1024/GB:.1f} GB")
    proc.append(f"{st.get('Threads', threading.active_count())} threads")
    try:
        proc.append(f"{len(os.listdir('/proc/self/fd'))} fds")
    except OSError:
        pass
    out.append("proc " + ", ".join(proc))
    try:
        r = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total",
                            "--format=csv,noheader,nounits"],
                           capture_output=True, text=True, timeout=5, check=False)
        gpus = []
        for i, line in enumerate(r.stdout.splitlines()):
            p = [x.strip() for x in line.split(",")]
            if len(p) == 2:
                gpus.append(f"GPU{i} {float(p[0])/1024:.1f}/{float(p[1])/1024:.1f}")
        out.append(" ".join(gpus) if gpus else "GPU unavailable")
    except Exception:
        out.append("GPU unavailable")          # nvidia-smi hangs on a wedged GPU
    disks = []
    for path in paths:
        try:
            s = os.statvfs(path)
        except OSError:
            continue
        total = s.f_blocks * s.f_frsize
        disks.append(f"{path} {(total - s.f_bavail*s.f_frsize)/GB:.1f}/{total/GB:.1f}")
    if disks:
        out.append("disk " + ", ".join(disks))
    return "[sys] " + " | ".join(out)


def start_monitor(interval_minutes=10.0, paths=("/", "/kaggle/working")):
    stop = threading.Event()
    def loop():
        while not stop.is_set():
            try:
                print(_sysline(paths), flush=True)
            except Exception as exc:
                print(f"[sys] census failed: {exc!r}", flush=True)
            stop.wait(interval_minutes * 60.0)
    threading.Thread(target=loop, name="system-stats", daemon=True).start()
    return stop

if not TRUE_SUBMISSION:
    _stop_monitor = start_monitor(2)
    print(_sysline())

## 8a. D′ slot priority (affectify, v5)

This is affectify's scheduler patch from their public notebook, embedded verbatim (its comments are in Japanese). It replaces the priority that decides which waiting game gets an LLM slot, and stops unstarted games from jumping the queue. It prints `#OURS_FORM ok version=d_prime ...` once installed, and raises an error if the scheduler it patches has changed. All credit for it goes to affectify. Set `ENABLE_DPRIME_SCHEDULER = False` to keep da-fr's priority.

v34 changed two of its numbers before it is installed (`DPRIME_OVERRIDES` in the config cell: `level_pow` 0.75, `token_scale` 400K) and v36 raises `token_scale` to 700K for 14 slots (see the intro). `DPRIME_OVERRIDES = {}` keeps affectify's values.

In [ ]:
# --- arc-agi-3-agent: replace the slot priority with the formula found in simulation ---
# Source: arceval/duck/form_priority.py, embedded verbatim. Active in the real submission.
# Keeps the scheduler's flow (unstarted games first, swaps only at context trims) and
# replaces only priority_value (plus a pace tracker: mean tokens per cleared level).
import sys as _fp_sys, types as _fp_types
_FP_SOURCE = '"""dfranzen の配分の優先度を、模擬で見つけた式に差し替える（本番でも効かせる）。\n\n彼の配分の流れ（まだ始めていないゲームが先、動いているゲームは履歴を削るときだけ\n入れ替える）はそのまま使い、**優先度の値を出す関数だけ**を替える。\n\n    優先度 = e^{lp·(ℓ-1)}·norm(N) × 2^{-a/ec} × ペース × 2^{-(t/ts)²} × (1 + 伸びしろ/8)\n\n    ℓ  挑戦中のレベル（1始まり）      N  総レベル数（分からなければ10）。norm は 6〜10 に丸めて\n                                         55/(N(N+1)/2)（彼と同じ）\n    a  このレベルで打った手数          t  このレベルで生成したトークン\n    ペース = clip((ref / これまでの1レベルあたりの平均トークン)^pp, 0.25, 4)。\n             1レベルも越えていなければ 1\n    伸びしろ = 残りのレベル数で 3以上→b3, 2→b2, 1→b1, 0→0\n\narceval/sim/simulate.py の Form（level_form=2, eff_form=1, shape=0, c_action=0,\ncombine=2, fade=0）と同じ値を出す（テストで見張る）。\n\n## 取り付け\n\n    install(tool_agent_module)\n\n  - `tool_agent.priority_value` を差し替える（彼の `_snapshot_priority` はこの名前で呼ぶ）\n  - `tool_agent.ProgressPace` を差し替える。各ゲームの分析役は作られるときにこの名前で\n    ペースの入れ物を作り、レベルを越えるたびに `record_completion(そのレベルのトークン, 基準)`\n    を呼ぶ。我々の入れ物は「これまでの1レベルあたりの平均トークン」を返す\n  - `ARC3_PRIORITY_PACE=1` にする。そうしないと彼のコードはペースを記録せず、優先度の\n    材料（PrioritySnapshot.cost_multiplier）にも載せない\n\n★ 本番の提出でも効かせる（そのための差し替え）。取り付けに失敗したら raise する ――\n  提出の前の Save & Run で必ず同じコードが走るので、そこで見つかる。\n"""\nfrom __future__ import annotations\n\nimport math\nimport os\nimport threading\nimport time\n\n# 模擬で選んだ式（data/datarun/forms-joint、2026-10-02）\nCHOSEN = {"level_pow": 0.2, "eff_const": 100.0, "pace_pow": 0.25, "pace_ref": 30_000.0,\n          "token_scale": 124_000.0, "b1": 2.5, "b2": 3.5, "b3": 4.0}\nSCALE = 1000.0   # 彼の優先度は整数。帯（100万）より十分小さく、順位が潰れない桁にする\n\n\nclass FormPace:\n    """これまでに越えたレベルの、1レベルあたりの平均生成トークン。\n\n    彼の ProgressPace と同じ呼ばれ方をする（`record_completion`, `cost_multiplier`）。\n    `cost_multiplier()` は倍率ではなく**平均トークンそのもの**を返す（0 なら、まだ無い）。\n    これを読むのは差し替えた `form_priority` だけ。\n    """\n\n    def __init__(self):\n        self.completed = 0\n        self.tokens = 0.0\n\n    def record_completion(self, tokens_used: float, reference_tokens: float = 0.0) -> None:\n        self.completed += 1\n        self.tokens += max(0.0, float(tokens_used))\n\n    def cost_multiplier(self) -> float:\n        return self.tokens / self.completed if self.completed else 0.0\n\n\ndef _levels(total) -> int | None:\n    try:\n        n = int(total)\n    except (TypeError, ValueError):\n        return None\n    return n if n > 0 else None\n\n\ndef form_value(level: int, actions: float, tokens: float, pace: float, total_levels,\n               params: dict = CHOSEN) -> float:\n    """優先度（小数）。pace はこれまでの1レベルあたりの平均トークン（0 ならまだ無い）。"""\n    level = max(1, int(level))\n    n = _levels(total_levels) or 10\n    n_clamped = min(10, max(6, n))\n    norm = 55 / (n_clamped * (n_clamped + 1) / 2)\n    a = max(0.0, float(actions))\n    t = max(0.0, float(tokens))\n    value = math.exp(params["level_pow"] * (level - 1)) * norm\n    value *= 2 ** (-(a / params["eff_const"]))\n    if pace > 0:\n        value *= min(4.0, max(0.25, (params["pace_ref"] / max(1.0, pace)) ** params["pace_pow"]))\n    value *= 2 ** (-((t / params["token_scale"]) ** 2))\n    remaining = max(0, n - level)\n    bonus = (0.0, params["b1"], params["b2"], params["b3"])[min(remaining, 3)]\n    return value * (1.0 + bonus / 8.0)\n\n\ndef form_priority(state, **_ignored) -> int:\n    """彼の `priority_value(state, ...)` と同じ呼ばれ方で、整数の優先度を返す。\n\n    state は PrioritySnapshot（level, actions, tokens, cost_multiplier, total_levels）。\n    彼の寄せ（tail_fraction）や係数の引数は使わない（この式は寄せを持たない）。\n    """\n    value = form_value(state.level, state.actions, state.tokens,\n                       float(getattr(state, "cost_multiplier", 0.0) or 0.0),\n                       getattr(state, "total_levels", None))\n    return max(1, int(value * SCALE))\n\n\n# ---- D′（2026-10-03、data/datarun/refine-fine。LB の v1 から模擬で +1.5 前後） ----\n#\n#     優先度 = A·M·C + B·φ\n#\n#     A = (1 + 0.5·(ℓ-1))·norm(N) × (300/(300+a))^2.5          深さ（線形）× 手数の効率\n#     M = clip((3万 / ペース)^0.4, 0.25, 4)（1レベルも越えていなければ 1）\n#     C = 0.1·max(0.1, 1 - a/115) + 0.9·max(0.1, 1 - t/T)        詰まりの減衰（線形、下限 0.1）\n#         T = 22.5万 × clip(ペース/3万, 0.5, 2)^0.5（遅いゲームほど我慢を長く。越えていなければ ×1）\n#     B = 残りのレベル数で 3以上→16, 2→14, 1→10, 0→0（伸びしろ。足し算）\n#     φ = 寄せ。走行の最後の4割で 1 → 0（彼の tail_fraction をそのまま使う）\n#\n#   加えて「全員まず1回」をやめる: まだ始めていないゲームも、この式の値（ℓ=1, a=t=0）で並べる。\n#\n# arceval/sim/simulate.py の Form（D_PRIME_FORM）と同じ値を出す（テストで見張る）。\n\nD_PRIME = {"level_pow": 0.5, "eff_const": 300.0, "eff_pow": 2.5, "pace_pow": 0.4,\n           "pace_ref": 30_000.0, "c_action": 0.1, "action_scale": 115.0, "c_token": 0.9,\n           "token_scale": 225_000.0, "lin_floor": 0.1, "rel_pow": 0.5,\n           "b1": 10.0, "b2": 14.0, "b3": 16.0, "fade": 0.4}\n# 模擬の Form の引数で書いた同じ式（テストで突き合わせる）\nD_PRIME_FORM = {"fade": 0.4, "level_form": 3, "level_pow": 0.5, "eff_form": 0, "eff_const": 300.0,\n                "eff_pow": 2.5, "shape": 3, "c_action": 0.1, "c_token": 0.9, "token_scale": 225000.0,\n                "combine": 1, "b1": 10.0, "b2": 14.0, "b3": 16.0, "pace_pow": 0.4, "form_fresh": 0,\n                "lin_floor": 0.1, "eff_floor": 0.2, "rel_pow": 0.5}\n\n\ndef d_parts(level: int, actions: float, tokens: float, pace: float, total_levels,\n            params: dict = D_PRIME) -> tuple:\n    """(A·M·C, B)。pace はこれまでの1レベルあたりの平均トークン（0 ならまだ無い）。"""\n    level = max(1, int(level))\n    n = _levels(total_levels) or 10\n    n_clamped = min(10, max(6, n))\n    norm = 55 / (n_clamped * (n_clamped + 1) / 2)\n    a = max(0.0, float(actions))\n    t = max(0.0, float(tokens))\n    A = (1.0 + params["level_pow"] * (level - 1)) * norm\n    A *= (params["eff_const"] / (params["eff_const"] + a)) ** params["eff_pow"]\n    if pace > 0:\n        A *= min(4.0, max(0.25, (params["pace_ref"] / max(1.0, pace)) ** params["pace_pow"]))\n    scale = params["token_scale"]\n    rel = pace if pace > 0 else params["pace_ref"]\n    scale *= min(2.0, max(0.5, rel / params["pace_ref"])) ** params["rel_pow"]\n    floor = params["lin_floor"]\n    C = (params["c_action"] * max(floor, 1.0 - a / params["action_scale"])\n         + params["c_token"] * max(floor, 1.0 - t / scale))\n    remaining = max(0, n - level)\n    B = (0.0, params["b1"], params["b2"], params["b3"])[min(remaining, 3)]\n    return A * C, B\n\n\ndef d_value(level, actions, tokens, pace, total_levels, phi: float = 1.0) -> float:\n    u, v = d_parts(level, actions, tokens, pace, total_levels)\n    return u + v * phi\n\n\ndef d_priority(state, *, tail_fraction=None, **_ignored) -> int:\n    """彼の `priority_value(state, tail_fraction=..., ...)` と同じ呼ばれ方。\n\n    寄せ φ は彼が渡す tail_fraction（残り時間 ÷ 寄せの窓。窓の外では 1 より大きい）を\n    0〜1 に切って使う。渡されなければ 1（寄せ無し）。\n    """\n    phi = 1.0 if tail_fraction is None else min(1.0, max(0.0, float(tail_fraction)))\n    value = d_value(state.level, state.actions, state.tokens,\n                    float(getattr(state, "cost_multiplier", 0.0) or 0.0),\n                    getattr(state, "total_levels", None), phi)\n    return max(1, int(value * SCALE))\n\n\nFRESH = {"replaced": 0}\n_TL = threading.local()\n\n\ndef install_d(tool_agent, solver) -> str:\n    """D′ を取り付ける。v1 の install と違い、「全員まず1回」も外す。\n\n    彼のコードで、まだ始めていないゲームは `gate.acquire(200万 − 待ち順)` で並ぶ（どの\n    優先度より上の帯）。これを、この式の値（ℓ=1, a=t=0、そのゲームの総レベル数）と、\n    寄せで値が変わるようにスナップショットつきで並べ直す。総レベル数は、その遊びを\n    走らせているスレッドで `_HarnessGameSession.play` が始まるときに覚えておく。\n    """\n    for name in ("priority_value", "ProgressPace", "_PriorityGate", "PrioritySnapshot",\n                 "_PRIORITY_UNTRIMMED_BASE", "_PRIORITY_BAND"):\n        if not hasattr(tool_agent, name):\n            raise RuntimeError(f"tool_agent に {name} が無い。彼の版が変わった。")\n    session_cls = getattr(solver, "_HarnessGameSession", None)\n    if session_cls is None or not callable(getattr(session_cls, "play", None)):\n        raise RuntimeError("solver に _HarnessGameSession.play が無い。彼の版が変わった。")\n    gate_cls = tool_agent._PriorityGate\n    for name in ("acquire", "_enqueue_and_wait", "_snapshot_priority"):\n        if not callable(getattr(gate_cls, name, None)):\n            raise RuntimeError(f"_PriorityGate に {name} が無い。彼の版が変わった。")\n\n    tool_agent.priority_value = d_priority\n    tool_agent.ProgressPace = FormPace\n    os.environ["ARC3_PRIORITY_PACE"] = "1"\n    os.environ["ARC3_PRIORITY_REFRESH_QUEUE"] = "1"\n    os.environ["ARC3_PRIORITY_TAIL_FADE"] = "1"\n    os.environ["ARC3_PRIORITY_TAIL_FADE_FRACTION"] = str(D_PRIME["fade"])\n\n    if not getattr(session_cls.play, "_ours_d", False):\n        orig_play = session_cls.play\n\n        def play(self, *args, **kwargs):\n            run = getattr(getattr(self, "game", None), "game_run", None)\n            _TL.n = getattr(run, "number_of_levels", None)\n            try:\n                return orig_play(self, *args, **kwargs)\n            finally:\n                _TL.n = None\n\n        play._ours_d = True\n        session_cls.play = play\n\n    if not getattr(gate_cls.acquire, "_ours_d", False):\n        orig_acquire = gate_cls.acquire\n        fresh_floor = tool_agent._PRIORITY_UNTRIMMED_BASE - tool_agent._PRIORITY_BAND\n        snapshot_cls = tool_agent.PrioritySnapshot\n\n        def acquire(self, priority):\n            if priority < fresh_floor:\n                return orig_acquire(self, priority)\n            # まだ始めていないゲーム: 帯で先頭に並べず、この式の値で並べる\n            snap = snapshot_cls(1, 0, 0.0, 0.0, getattr(_TL, "n", None))\n            value = self._snapshot_priority(snap, time.monotonic())\n            with self._cond:\n                self._enqueue_and_wait(value, snap)\n            FRESH["replaced"] += 1\n\n        acquire._ours_d = True\n        gate_cls.acquire = acquire\n\n    line = ("#OURS_FORM ok version=d_prime priority_value replaced; fresh-first removed; "\n            "tail fade " + str(D_PRIME["fade"]) + "; "\n            + " ".join(f"{k}={v:g}" for k, v in D_PRIME.items() if k != "fade"))\n    print(line, flush=True)\n    return line\n\n\ndef install(tool_agent) -> str:\n    for name in ("priority_value", "ProgressPace", "_PriorityGate"):\n        if not hasattr(tool_agent, name):\n            raise RuntimeError(f"tool_agent に {name} が無い。彼の版が変わった。")\n    tool_agent.priority_value = form_priority\n    tool_agent.ProgressPace = FormPace\n    os.environ["ARC3_PRIORITY_PACE"] = "1"\n    line = ("#OURS_FORM ok priority_value replaced; pace=mean tokens per cleared level; "\n            + " ".join(f"{k}={v:g}" for k, v in CHOSEN.items()))\n    print(line, flush=True)\n    return line\n'
_fp_mod = _fp_types.ModuleType('ours_form_priority')
_fp_sys.modules['ours_form_priority'] = _fp_mod
exec(compile(_FP_SOURCE, 'ours_form_priority.py', 'exec'), _fp_mod.__dict__)
import inference.agent.tool_agent as _fp_ta
import inference.framework.solver as _fp_solver
if ENABLE_DPRIME_SCHEDULER:
    # v34: two D' numbers retuned for hidden-set difficulty (contest simulator on our recorded runs); {} = v33
    _dp_unknown = sorted(set(DPRIME_OVERRIDES) - set(_fp_mod.D_PRIME))
    if _dp_unknown:
        raise KeyError(f'DPRIME_OVERRIDES: unknown D-prime keys {_dp_unknown}')
    _fp_mod.D_PRIME.update(DPRIME_OVERRIDES)
    _fp_mod.install_d(_fp_ta, _fp_solver)
else:
    print('D-prime scheduler disabled: da-fr priority in use', flush=True)


## 8c. First Light (v46)

A game stuck on level 1 scores 0, and level 1 carries the smallest weight, so search moves spent there cost almost nothing while clearing it opens the more valuable levels to the model. When such a game (still on level 1 after `FL_MIN_LEVEL1_TOKENS`) hands its GPU seat over at a history trim and has to wait, a background thread plays level 1 with a plain graph search (every key, a click on every object, boards remembered by hash, RESET + replay to reach unexplored ones). It stops the moment the seat comes back: if level 1 was cleared, the harness's state moves to level 2 and the model's next request says so; if not, the board is put back exactly where the model left it. Never on level 2 or later. Limits: `FL_MAX_MOVES_PER_GAME`, `FL_MAX_MOVES_TOTAL`, `FL_MAX_CONCURRENT`.


In [ ]:
# ===== v46: FIRST LIGHT - a CPU search for games stuck on level 1, run while they wait for the GPU =====
# A game stuck on level 1 scores 0 for the whole game, and level 1 has the smallest weight of all levels, so moves
# spent on it cost almost nothing, while clearing it opens levels 2, 3, ... to the model. In the contest about 110
# games share 16 GPU seats: a game whose context was just trimmed hands its seat over (_maybe_handover) and waits.
# First Light uses exactly that wait, on games that are still on level 1 after FL_MIN_LEVEL1_TOKENS generated tokens:
# a background thread plays level 1 with a plain graph search - every keyboard action and a click on each object of
# every board it has seen, boards remembered by a hash (edge rows/columns masked: move-budget bars), unexplored
# boards reached again by RESET + replaying their path, GAME OVER -> RESET. Measured on the 25 public games
# (2026-10-20): it cleared level 1 of 16 games within 60-90 s (5 to 11,700 moves), and 3 of the 8 level-1s our model
# found hardest (cd82, ls20, m0r0). It never runs on level 2 or later: there the model still clears 74% of its stalled
# levels at ~1.1x the human move count, and search moves would wipe those scores.
# When the game gets its seat back the search stops at once, and:
#  - level 1 NOT cleared: the board is put back exactly where the model left it (RESET + the model's own moves on
#    this level since its last RESET or level start, checked cell by cell); the model is told in one line;
#  - level 1 cleared: the harness's game state, history and runtime state are moved to the new board, the per-level
#    token/move counters restart (the scheduler prices the game on level 2), and the model's very next request carries
#    a note: level 1 was cleared by the search, everything seen earlier in the turn is out of date, and the moves that
#    cleared it.
# Limits: FL_MAX_MOVES_PER_GAME per game, FL_MAX_MOVES_TOTAL per run, FL_MAX_CONCURRENT games searching at once, a
# start delay so a wait that ends at once costs no move. Any failure leaves the game to the model and is counted.
import collections as _fl_collections
import hashlib as _fl_hashlib
import json as _fl_json
import threading as _fl_threading
import time as _fl_time

FL_EDGE = 2                 # rows/cols masked at each board edge for the state hash (HUD / move-budget bars)
FL_MAX_CLICKS = 80          # objects clicked per board at most (rare colours first, then small objects)
FL_START_DELAY_S = 4.0      # a wait shorter than this costs no move
FL_JOIN_TIMEOUT_S = 90.0    # longest wait for the search thread after the game got its seat back (one gateway call)
FL_NOTE_PATH_MOVES = 40     # winning moves quoted to the model
FL_STATS = {"jobs": 0, "jobs_with_moves": 0, "cleared": 0, "restored": 0, "restore_mismatch": 0, "moves": 0,
            "game_overs": 0, "resets": 0, "stopped_by_seat": 0, "budget_ends": 0, "exhausted": 0, "notes": 0,
            "skipped_busy": 0, "skipped_budget": 0, "skipped_untracked": 0, "sync_errors": 0, "errors": 0,
            "last_error": ""}
_FL_LOCK = _fl_threading.Lock()
_FL_TL = _fl_threading.local()          # the game session that runs in this thread
_FL_ACTIVE = set()                      # id(game) of games searching now
_FL_GRAPHS = {}                         # id(game) -> _FlGraph, kept across waits
_FL_MOVES_USED = {}                     # id(game) -> moves spent on it
_FL_STOP_ALL = _fl_threading.Event()    # set when the benchmark ends: every search stops
FL_TAIL_STOP_S = 900.0                  # no new search in the last 15 minutes of the run
FIRST_LIGHT_LOG = (None if globals().get("TRUE_SUBMISSION", True)
                   else str(globals().get("WORKING_DIR", "/kaggle/working")) + "/first_light.jsonl")


def _fl_err(exc):
    with _FL_LOCK:
        FL_STATS["errors"] += 1
        FL_STATS["last_error"] = f"{type(exc).__name__}: {exc}"[:300]


def _fl_log(record):
    if not FIRST_LIGHT_LOG:
        return
    try:
        with _FL_LOCK:
            with open(FIRST_LIGHT_LOG, "a", encoding="utf-8") as f:
                f.write(_fl_json.dumps(record, default=str) + "\n")
    except Exception:
        pass


def _fl_board(obs):
    """The last frame layer as a tuple of row tuples, or None."""
    try:
        frames = list(obs.frame or [])
        if not frames:
            return None
        return tuple(tuple(int(v) for v in row) for row in frames[-1])
    except Exception:
        return None


def _fl_key(board):
    h = len(board)
    w = len(board[0]) if h else 0
    rows = []
    for r in range(FL_EDGE, h - FL_EDGE):
        rows.append(bytes(board[r][c] & 0xFF for c in range(FL_EDGE, w - FL_EDGE)))
    return _fl_hashlib.blake2b(b"\n".join(rows), digest_size=12).hexdigest()


def _fl_components(board):
    """(size, colour, (row, col) of a central cell) per 4-connected same-colour object inside the masked edge."""
    h = len(board)
    w = len(board[0]) if h else 0
    seen = [[False] * w for _ in range(h)]
    out = []
    for r in range(FL_EDGE, h - FL_EDGE):
        for c in range(FL_EDGE, w - FL_EDGE):
            if seen[r][c]:
                continue
            col = board[r][c]
            stack, cells = [(r, c)], []
            seen[r][c] = True
            while stack:
                y, x = stack.pop()
                cells.append((y, x))
                for ny, nx in ((y + 1, x), (y - 1, x), (y, x + 1), (y, x - 1)):
                    if FL_EDGE <= ny < h - FL_EDGE and FL_EDGE <= nx < w - FL_EDGE and not seen[ny][nx] \
                            and board[ny][nx] == col:
                        seen[ny][nx] = True
                        stack.append((ny, nx))
            cy = sum(p[0] for p in cells) / len(cells)
            cx = sum(p[1] for p in cells) / len(cells)
            out.append((len(cells), col, min(cells, key=lambda p: (p[0] - cy) ** 2 + (p[1] - cx) ** 2)))
    return out


def _fl_candidates(available, board):
    """Keyboard actions first, then a click on each object: rare colours first, then small objects."""
    acts = [(k, None) for k in (1, 2, 3, 4, 5) if k in available]
    if 6 in available:
        comps = _fl_components(board)
        if comps:
            big = max(comps, key=lambda t: t[0])[0]
            comps = [t for t in comps if t[0] < big]
            count = _fl_collections.Counter(t[1] for t in comps)
            comps.sort(key=lambda t: (count[t[1]], t[0]))
            acts += [(6, rc) for _size, _col, rc in comps[:FL_MAX_CLICKS]]
    return acts


class _FlGraph:
    """Level-1 search state of one game: board key -> untested actions, shortest known path from the level start."""

    def __init__(self):
        self.untested = {}
        self.path = {}
        self.start = None
        self.cleared_path = None


class _FlEnv:
    """The game's own arc_agi environment, driven directly (the harness never sees these moves)."""

    def __init__(self, env):
        import arcengine as _fl_ae
        self.env = env
        self.ae = _fl_ae
        self.since_reset = None          # moves since the level start / last RESET (None = unknown)

    @property
    def obs(self):
        return self.env.observation_space

    def available(self):
        try:
            return {int(a) for a in (self.obs.available_actions or [])}
        except Exception:
            return set()

    def step(self, k, rc):
        ga = self.ae.GameAction
        action = {1: ga.ACTION1, 2: ga.ACTION2, 3: ga.ACTION3, 4: ga.ACTION4, 5: ga.ACTION5,
                  6: ga.ACTION6, 7: ga.ACTION7}[k]
        data = {"x": int(rc[1]), "y": int(rc[0])} if rc is not None else {}
        out = self.env.step(action, data=data)
        if out is None:
            raise RuntimeError(f"step {k} {rc} returned no frame")
        self.since_reset = (self.since_reset or 0) + 1
        return out

    def step_action(self, action_id, data):
        out = self.env.step(action_id, data=dict(data or {}))
        if out is None:
            raise RuntimeError(f"replay {action_id} returned no frame")
        self.since_reset = (self.since_reset or 0) + 1
        return out

    def reset_level(self):
        """RESET only after a move (a RESET at a level start would restart the whole game outside competition mode)."""
        if self.since_reset == 0:
            return self.obs
        out = self.env.reset()
        if out is None:
            raise RuntimeError("reset returned no frame")
        self.since_reset = 0
        with _FL_LOCK:
            FL_STATS["resets"] += 1
        return out

    def state_name(self):
        try:
            return self.obs.state.name
        except Exception:
            return str(getattr(self.obs, "state", ""))

    def levels(self):
        return int(self.obs.levels_completed)


class _FlJob:
    """One search session on one game, for the length of one wait."""

    def __init__(self, game, budget):
        self.game = game
        self.budget = int(budget)
        self.stop = _fl_threading.Event()
        self.moves = 0
        self.cleared = False
        self.error = None
        self.reason = ""
        self.fe = _FlEnv(game.env)
        moves = game.__dict__.get("_fl_moves")
        self.fe.since_reset = len(moves) if isinstance(moves, list) else None
        self.board_before = _fl_board(game.env.observation_space)
        self.graph = _FL_GRAPHS.setdefault(id(game), _FlGraph())
        self.thread = _fl_threading.Thread(target=self.run, name="first-light", daemon=True)

    def _count(self, n=1):
        self.moves += n
        with _FL_LOCK:
            FL_STATS["moves"] += n
            _FL_MOVES_USED[id(self.game)] = _FL_MOVES_USED.get(id(self.game), 0) + n

    def _register(self, board, via):
        g = self.graph
        key = _fl_key(board)
        if key not in g.untested:
            g.untested[key] = _fl_candidates(self.fe.available(), board)
            g.path[key] = list(via)
        elif len(via) < len(g.path[key]):
            g.path[key] = list(via)
        return key

    def _act(self, k, rc):
        obs = self.fe.step(k, rc)
        self._count()
        return obs

    def _reset(self):
        before = self.fe.since_reset
        obs = self.fe.reset_level()
        if before != 0:
            self._count()
        return obs

    def run(self):
        try:
            if self.stop.wait(FL_START_DELAY_S):
                return
            if self.fe.since_reset is None:
                self.reason = "untracked"
                return
            self._explore()
        except Exception as exc:
            self.error = exc
            _fl_err(exc)

    def _explore(self):
        fe, g = self.fe, self.graph
        if fe.levels() != 0 or fe.state_name() != "NOT_FINISHED":
            self.reason = "not on level 1"
            return
        self._reset()
        if g.start is None:
            g.start = self._register(_fl_board(fe.obs), [])
        cur, cur_path = g.start, []
        while not self.stop.is_set() and not _FL_STOP_ALL.is_set():
            if self.moves >= self.budget:
                self.reason = "budget"
                with _FL_LOCK:
                    FL_STATS["budget_ends"] += 1
                return
            if fe.state_name() == "GAME_OVER":
                with _FL_LOCK:
                    FL_STATS["game_overs"] += 1
                self._reset()
                cur, cur_path = g.start, []
                continue
            if g.untested.get(cur):
                k, rc = g.untested[cur].pop(0)
                self._act(k, rc)
                if fe.levels() > 0 or fe.state_name() == "WIN":
                    self.cleared = True
                    g.cleared_path = cur_path + [(k, rc)]
                    self.reason = "cleared"
                    return
                if fe.state_name() == "GAME_OVER":
                    continue
                nb = _fl_board(fe.obs)
                if nb is None:
                    raise RuntimeError("no board")
                nk = _fl_key(nb)
                if nk == cur:
                    continue
                nk = self._register(nb, cur_path + [(k, rc)])
                cur, cur_path = nk, g.path[nk]
                continue
            frontier = [key for key, acts in g.untested.items() if acts and key != cur]
            if not frontier:
                self.reason = "exhausted"
                with _FL_LOCK:
                    FL_STATS["exhausted"] += 1
                return
            target = min(frontier, key=lambda key: len(g.path[key]))
            self._reset()
            cur, cur_path = g.start, []
            for k, rc in g.path[target]:
                if self.stop.is_set() or _FL_STOP_ALL.is_set():
                    return
                self._act(k, rc)
                if fe.state_name() != "NOT_FINISHED" or fe.levels() > 0:
                    break
            if fe.levels() > 0 or fe.state_name() == "WIN":
                self.cleared = True
                g.cleared_path = list(g.path[target])
                self.reason = "cleared"
                return
            if fe.state_name() == "GAME_OVER":
                continue
            nb = _fl_board(fe.obs)
            cur = self._register(nb, g.path[target]) if nb is not None else g.start
            cur_path = g.path[cur]

    def restore(self):
        """Put the board back where the model left it: RESET + the model's moves on this level since its last RESET."""
        moves = self.game.__dict__.get("_fl_moves")
        if not isinstance(moves, list):
            return False
        self._reset()
        for action_id, data in moves:
            self.fe.step_action(action_id, data)
            self._count()
            if self.fe.state_name() != "NOT_FINISHED":
                break
        return _fl_board(self.fe.obs) == self.board_before


def _fl_move_text(k, rc):
    names = {1: "UP", 2: "DOWN", 3: "LEFT", 4: "RIGHT", 5: "SPACE"}
    return names.get(k) or f"MOUSE(row={rc[0]}, col={rc[1]})"


def _fl_sync(job, agent):
    """Move the harness's view of the game to the board the search left (after a clear or a failed restore)."""
    game = job.game
    obs = game.env.observation_space
    try:
        import taaf.game as _fl_tg
        game._current_state = _fl_tg.GameState(raw=obs)
    except Exception as exc:
        with _FL_LOCK:
            FL_STATS["sync_errors"] += 1
        _fl_err(exc)
    session = getattr(_FL_TL, "session", None)
    if session is not None:
        try:
            entries = getattr(session, "history_entries", None)
            if entries:
                entry_cls = type(entries[-1])
                result = {"automatic": True, "first_light": True, "level_completed": bool(job.cleared),
                          "level": int(obs.levels_completed) + 1, "state": getattr(obs.state, "name", str(obs.state))}
                entries.append(entry_cls(action="FIRST_LIGHT_SEARCH", frame=session.current_frame(), result=result))
            session.write_runtime_state()
        except Exception as exc:
            with _FL_LOCK:
                FL_STATS["sync_errors"] += 1
            _fl_err(exc)
    if job.cleared and agent is not None:
        try:
            import inference.agent.tool_agent as _fl_ta
            agent._tokens_at_level_start = agent._session_generated_tokens
            agent._actions_at_level_start = _fl_ta._priority_action_count(agent._last_step_summary or {})
        except Exception as exc:
            with _FL_LOCK:
                FL_STATS["sync_errors"] += 1
            _fl_err(exc)
    game.__dict__["_fl_moves"] = [] if job.cleared else None
    game.__dict__["_sv_fresh"] = bool(job.cleared)        # a cleared level starts with the server's counter at 0


def _fl_note(job, restored):
    if job.cleared:
        path = job.graph.cleared_path or []
        shown = path[-FL_NOTE_PATH_MOVES:]
        skipped = len(path) - len(shown)
        return ("FIRST LIGHT (automatic search, run by the harness while you waited for the GPU): LEVEL 1 WAS CLEARED "
                f"by a systematic search ({job.moves} moves in all). You are now on LEVEL 2 - every board you saw earlier "
                "in this turn is out of date: inspect current_frame before acting. The moves that cleared level 1 from "
                f"its start{f' (last {len(shown)} of {len(path)})' if skipped > 0 else ''}: "
                + ", ".join(_fl_move_text(k, rc) for k, rc in shown) + ".")
    if restored:
        return ("FIRST LIGHT (automatic search, run by the harness while you waited for the GPU): it tried "
                f"{job.moves} moves on level 1 and visited {len(job.graph.untested)} different boards without clearing "
                "it; your board was then put back exactly as you left it.")
    return ("FIRST LIGHT (automatic search, run by the harness while you waited for the GPU): it tried moves on "
            "level 1 and could not put your board back exactly - the board is not what you last saw: inspect "
            "current_frame before acting.")


def _fl_eligible(agent, game):
    if not ENABLE_FIRST_LIGHT or game is None or getattr(game, "env", None) is None:
        return False, ""
    try:
        state = game.current_state
        raw = getattr(state, "raw", state)
        if int(raw.levels_completed) != 0 or getattr(raw.state, "name", str(raw.state)) != "NOT_FINISHED":
            return False, ""
    except Exception:
        return False, ""
    tokens = int(getattr(agent, "_session_generated_tokens", 0) or 0) - int(getattr(agent, "_tokens_at_level_start", 0) or 0)
    if tokens < FL_MIN_LEVEL1_TOKENS or _FL_STOP_ALL.is_set():
        return False, ""
    try:
        import inference.agent.tool_agent as _fl_ta
        deadline = getattr(_fl_ta._priority_gate(), "_clock_deadline", None)
        if deadline is not None and deadline - _fl_time.monotonic() < FL_TAIL_STOP_S:
            return False, ""
    except Exception:
        pass
    if not isinstance(game.__dict__.get("_fl_moves"), list):
        return False, "untracked"
    with _FL_LOCK:
        if FL_STATS["moves"] >= FL_MAX_MOVES_TOTAL or _FL_MOVES_USED.get(id(game), 0) >= FL_MAX_MOVES_PER_GAME:
            return False, "budget"
        if len(_FL_ACTIVE) >= FL_MAX_CONCURRENT or id(game) in _FL_ACTIVE:
            return False, "busy"
        _FL_ACTIVE.add(id(game))
    return True, ""


def install_first_light():
    import arcengine as _fl_ae
    import inference.agent.tool_agent as _fl_ta
    import inference.framework.solver as _fl_solver
    import taaf.game_api as _fl_ga
    done = []
    # 1. the model's own moves on the current level since its last RESET or level start (to restore its board)
    G = _fl_ga.GameAPI
    if not getattr(G, "_fl_tracked", False):
        exec_inner = G._execute_action
        reset_id = _fl_ae.GameAction.RESET

        def _execute_action(self, action):
            try:
                before = int(self.env.observation_space.levels_completed)
            except Exception:
                before = None
            state = exec_inner(self, action)
            try:
                raw = state.raw
                moves = self.__dict__.get("_fl_moves")
                if action.id == reset_id or (before is not None and int(raw.levels_completed) != before):
                    self.__dict__["_fl_moves"] = []          # a level start
                elif before is None or raw.state != _fl_ae.GameState.NOT_FINISHED:
                    self.__dict__["_fl_moves"] = None        # unknown or ended: the board cannot be rebuilt by replay
                elif isinstance(moves, list):
                    moves.append((action.id, dict(getattr(action, "data", None) or {})))
            except Exception:
                self.__dict__["_fl_moves"] = None
            return state

        start_inner = G._start_game

        def _start_game(self, session):
            state = start_inner(self, session)
            self.__dict__["_fl_moves"] = []        # arcade.make ends with a RESET: a level start
            return state

        G._execute_action, G._start_game, G._fl_tracked = _execute_action, _start_game, True
        done.append("moves")
    # 2. which game session runs in this thread
    S = _fl_solver._HarnessGameSession
    if not getattr(S.play, "_fl_wrapped", False):
        play_inner = S.play

        def play(self, *args, **kwargs):
            _FL_TL.session = self
            try:
                return play_inner(self, *args, **kwargs)
            finally:
                _FL_TL.session = None

        play._fl_wrapped = True
        for name in dir(play_inner):
            if name.endswith("_d") or name.endswith("_wrapped"):
                try:
                    setattr(play, name, getattr(play_inner, name))
                except Exception:
                    pass
        S.play = play
        done.append("session")
    # 3. the outgoing request (a note can join it) and the seat handover (the wait the search uses)
    A = _fl_ta.ToolAgent
    if not getattr(A._trim_messages_for_context, "_fl_wrapped", False):
        trim_inner = A._trim_messages_for_context

        def _trim_messages_for_context(self, *args, **kwargs):
            out = trim_inner(self, *args, **kwargs)
            self._fl_outgoing = out
            return out

        for name in dir(trim_inner):
            if name.endswith("_wrapped"):
                setattr(_trim_messages_for_context, name, getattr(trim_inner, name))
        _trim_messages_for_context._fl_wrapped = True
        A._trim_messages_for_context = _trim_messages_for_context
        done.append("outgoing")
    if not getattr(A._maybe_handover, "_fl_wrapped", False):
        handover_inner = A._maybe_handover

        def _maybe_handover(self, *args, **kwargs):
            job = None
            try:
                if getattr(self, "_context_was_trimmed", False) and _fl_ta._priority_gate() is not None:
                    session = getattr(_FL_TL, "session", None)
                    game = getattr(session, "game", None)
                    ok, why = _fl_eligible(self, game)
                    if ok:
                        try:
                            job = _FlJob(game, FL_MAX_MOVES_PER_GAME - _FL_MOVES_USED.get(id(game), 0))
                            with _FL_LOCK:
                                FL_STATS["jobs"] += 1
                            job.thread.start()
                        except Exception:
                            job = None
                            with _FL_LOCK:
                                _FL_ACTIVE.discard(id(game))      # never leak a search seat
                            raise
                    elif why:
                        with _FL_LOCK:
                            FL_STATS["skipped_" + why] += 1
            except Exception as exc:
                _fl_err(exc)
                job = None
            try:
                return handover_inner(self, *args, **kwargs)
            finally:
                if job is not None:
                    _fl_finish(job, self)

        for name in dir(handover_inner):
            if name.endswith("_wrapped"):
                setattr(_maybe_handover, name, getattr(handover_inner, name))
        _maybe_handover._fl_wrapped = True
        A._maybe_handover = _maybe_handover
        done.append("handover")
    return done


def _fl_finish(job, agent):
    """In the game's own thread, right after it got its seat back: stop the search, restore or sync, note."""
    try:
        job.stop.set()
        job.thread.join(FL_JOIN_TIMEOUT_S)
        if job.thread.is_alive():
            raise RuntimeError("search thread did not stop")
        if job.moves == 0:
            return
        with _FL_LOCK:
            FL_STATS["jobs_with_moves"] += 1
            if job.reason not in ("cleared", "budget", "exhausted"):
                FL_STATS["stopped_by_seat"] += 1
        restored = False
        if job.cleared:
            with _FL_LOCK:
                FL_STATS["cleared"] += 1
            _fl_sync(job, agent)
        else:
            restored = job.restore()
            if restored:
                with _FL_LOCK:
                    FL_STATS["restored"] += 1
            else:
                with _FL_LOCK:
                    FL_STATS["restore_mismatch"] += 1
                _fl_sync(job, agent)
        note = _fl_note(job, restored)
        outgoing = getattr(agent, "_fl_outgoing", None)
        if isinstance(outgoing, list):
            outgoing.append({"role": "user", "content": note})
            with _FL_LOCK:
                FL_STATS["notes"] += 1
        _fl_log({"t": round(_fl_time.time(), 1), "moves": job.moves, "cleared": job.cleared, "restored": restored,
                 "reason": job.reason, "boards": len(job.graph.untested),
                 "path_len": len(job.graph.cleared_path or [])})
    except Exception as exc:
        _fl_err(exc)
    finally:
        with _FL_LOCK:
            _FL_ACTIVE.discard(id(job.game))




def run_first_light_smoke(env_dir, out_path=None):
    """Save & Run only (after the benchmark): First Light on the REAL taaf GameAPI with offline games - vc33 level 1
    cleared by the search and the harness game state moved to level 2; ls20: three moves through the real GameAPI
    (tracked), a short search, the board restored exactly. Measurement only; never raises."""
    res = {'ok': False}
    was_stopped = _FL_STOP_ALL.is_set()
    _FL_STOP_ALL.clear()
    try:
        import arc_agi as _fs_arc
        import arcengine as _fs_ae
        import taaf.game as _fs_tg
        import taaf.game_api as _fs_ga
        spec = _fs_ga.ArcadeSpec(operation_mode=_fs_arc.OperationMode.OFFLINE, environments_dir=str(env_dir))
        ids = {e.game_id.split('-')[0]: e.game_id
               for e in _fs_arc.Arcade(operation_mode=_fs_arc.OperationMode.OFFLINE,
                                      environments_dir=str(env_dir)).available_environments}
        session = _fs_tg.RunSession(record_intermediate_states=False)
        g = _fs_ga.GameAPI(env_name=ids['vc33'], arcade_spec=spec)
        g.start_game(session)
        res['vc33_tracked_at_start'] = g.__dict__.get('_fl_moves') == []
        job = _FlJob(g, 3000)
        job._explore()
        res['vc33_cleared'] = bool(job.cleared)
        res['vc33_moves'] = job.moves
        _fl_sync(job, None)
        res['vc33_harness_levels'] = int(g.current_state.raw.levels_completed)
        g2 = _fs_ga.GameAPI(env_name=ids['ls20'], arcade_spec=spec)
        g2.start_game(session)
        for a in (_fs_ae.GameAction.ACTION1, _fs_ae.GameAction.ACTION4, _fs_ae.GameAction.ACTION4):
            g2.execute_action(_fs_ae.ActionInput(id=a, data={}))
        res['ls20_tracked_moves'] = len(g2.__dict__.get('_fl_moves') or [])
        job2 = _FlJob(g2, 300)
        before = job2.board_before
        job2._explore()
        res['ls20_search_moves'] = job2.moves
        res['ls20_restored'] = bool(job2.restore()) and _fl_board(g2.env.observation_space) == before
        res['ls20_harness_board_unchanged'] = _fl_board(g2.current_state.raw) == before
        res['ok'] = bool(res['vc33_tracked_at_start'] and res['vc33_cleared'] and res['vc33_harness_levels'] == 1
                         and res['ls20_tracked_moves'] == 3 and res['ls20_restored']
                         and res['ls20_harness_board_unchanged'])
    except Exception as exc:
        res['error'] = repr(exc)[:300]
    finally:
        if was_stopped:
            _FL_STOP_ALL.set()
    res['stats'] = {k: v for k, v in FL_STATS.items() if k != 'last_error'}
    try:
        if out_path:
            with open(str(out_path), 'w') as f:
                f.write(_fl_json.dumps(res, indent=1, default=str))
    except Exception:
        pass
    print('FIRST_LIGHT_SMOKE', _fl_json.dumps(res, default=str), flush=True)
    return res


if ENABLE_FIRST_LIGHT:
    print("first light installed:", install_first_light(), flush=True)


## 8b. End-of-run report + experiment record (v3)

This cell only measures; it never changes play. After a Save & Run, it prints the prefix-cache reuse, decode throughput and speculative accept length from `serve.log` and the server's `/metrics`. It also prints tokens per action and per level for each game. Any part that cannot be read prints `n/a`.

In [ ]:
# ===== End-of-run speed / cache report + experiment record (measurement only; never changes play) =====
# Printed after the benchmark (Save & Run only): prefix-cache reuse and generation speed from the SGLang log and
# its /metrics endpoint, plus tokens per action and per level for each game. Every part is best-effort: a missing
# log line or metric prints "n/a" instead of failing the notebook.
import json as _json_rep
import re as _re_rep
import statistics as _st_rep
import urllib.request as _ur_rep

_REP_NEW = _re_rep.compile(r"#new-token:\s*(\d+)")
_REP_CACHED = _re_rep.compile(r"#cached-token:\s*(\d+)")
_REP_GEN = _re_rep.compile(r"gen throughput \(token/s\):\s*([\d.]+)")
_REP_ACCEPT = _re_rep.compile(r"accept len:\s*([\d.]+)")
# v12: per-request timing lines (enable_request_time_stats_logging): prompt length, cached part, prefill time
_REP_RT_IN = _re_rep.compile(r", input_len=(\d+)")
_REP_RT_CACHED = _re_rep.compile(r"cached_input_len=(\d+)")
_REP_RT_PRE = _re_rep.compile(r"initial_prefill_elapsed=([\d.]+)ms")
_REP_METRICS = ("cache_hit", "cached_token", "prompt_tokens_total", "generation_tokens_total", "spec_accept",
                "prefix_cache", "spec_decode")   # v35: vLLM metric names too
# v14: reply statistics from the request logs (keys are JSON keys, so quotes inside message text are escaped)
_REP_COMPL = _re_rep.compile(r'"completion_tokens": (\d+)')
_REP_REASON = _re_rep.compile(r'"reasoning_tokens": (\d+)')
_REP_FINISH = _re_rep.compile(r'"finish_reason": "(\w+)"')
_REP_EFFORT = _re_rep.compile(r'"reasoning_effort": "(\w+)"')
# v16: GPU batch occupancy, from the periodic decode-batch lines
_REP_RUNNING = _re_rep.compile(r"Decode batch, #running-req:\s*(\d+)")
_REP_KV = _re_rep.compile(r"full token usage:\s*([\d.]+)")
_REP_QUEUE = _re_rep.compile(r"#queue-req:\s*(\d+)")


def _report_serve_log(path):
    new = cached = 0
    gen, accept = [], []
    pre_s = cold_s = 0.0
    cold_n = 0
    cold_big_n, cold_big_s = 0, 0.0
    running, kv, queued, retract = [], [], 0, 0
    with open(path, "r", errors="replace") as f:
        for line in f:
            if "#new-token" in line:
                m1, m2 = _REP_NEW.search(line), _REP_CACHED.search(line)
                if m1 and m2:
                    new += int(m1.group(1))
                    cached += int(m2.group(1))
            elif "ReqTimeStats" in line:
                mi, mc, mp = _REP_RT_IN.search(line), _REP_RT_CACHED.search(line), _REP_RT_PRE.search(line)
                if mi and mc and mp and int(mi.group(1)) >= 500:   # skip warmup / health checks
                    pre = float(mp.group(1)) / 1000.0
                    pre_s += pre
                    if int(mc.group(1)) < 0.5 * int(mi.group(1)):   # history was trimmed: prompt re-prefilled
                        cold_n += 1
                        cold_s += pre
                        if int(mi.group(1)) >= 20000:   # v17: a long history re-read (drain or eviction)
                            cold_big_n += 1
                            cold_big_s += pre
            elif "gen throughput" in line:
                mr, mk, mq = _REP_RUNNING.search(line), _REP_KV.search(line), _REP_QUEUE.search(line)
                if mr:
                    running.append(int(mr.group(1)))
                    kv.append(float(mk.group(1)) if mk else 0.0)
                    queued += bool(mq) and int(mq.group(1)) > 0
                m = _REP_GEN.search(line)
                if m:
                    gen.append(float(m.group(1)))
                m = _REP_ACCEPT.search(line)
                if m:
                    accept.append(float(m.group(1)))
            elif "etract" in line and "server_args" not in line:   # KV pool full: running requests retracted
                retract += 1
    hit = f"{100.0 * cached / (cached + new):.1f}%" if cached + new else "n/a"
    print(f"[report] prefix cache (serve.log, approx): cached {cached:,} / prefilled {new:,} tokens -> reuse {hit}")
    print("[report] server decode throughput tok/s: "
          + (f"median {_st_rep.median(gen):.0f}, mean {_st_rep.mean(gen):.0f} over {len(gen)} samples" if gen else "n/a"))
    if accept:
        print(f"[report] speculative accept length: median {_st_rep.median(accept):.2f}")
    print(f"[report] prefill time (decode paused): {pre_s / 60:.1f} min, of which {cold_n} cold re-prefills "
          f"after history trims {cold_s / 60:.1f} min")
    occupancy = {}
    if running:
        try:
            seats = int(CFG["MAXREQ"])
        except Exception:
            seats = max(running)
        occupancy = {"mean_running": round(_st_rep.mean(running), 2),
                     "full_batch_pct": round(100.0 * sum(r >= seats for r in running) / len(running), 1),
                     "kv_mean": round(_st_rep.mean(kv), 2), "kv_max": round(max(kv), 2),
                     "queue_pct": round(100.0 * queued / len(running), 1), "retract_lines": retract}
        print(f"[report] GPU batch: mean {occupancy['mean_running']} of {seats} seats busy, full batch in "
              f"{occupancy['full_batch_pct']}% of decode steps, KV mean {occupancy['kv_mean']} max "
              f"{occupancy['kv_max']}, queue in {occupancy['queue_pct']}%, retract lines {retract}")
    return {**occupancy, "cache_reuse_pct": round(100.0 * cached / (cached + new), 1) if cached + new else None,
            "prefill_min": round(pre_s / 60, 1), "cold_prefills": cold_n, "cold_prefill_min": round(cold_s / 60, 1),
            "cold_big_prefills": cold_big_n, "cold_big_min": round(cold_big_s / 60, 1),
            "server_decode_tok_s_median": round(_st_rep.median(gen)) if gen else None,
            "spec_accept_median": round(_st_rep.median(accept), 2) if accept else None}


import re as _re_late_mod
_re_late = _re_late_mod.compile(r'"action": (\d+)')


def _report_requests(run_dir):
    paths = sorted(run_dir.glob("*_requests.jsonl")) or sorted(run_dir.glob("*/*_requests.jsonl"))
    compl, reason, trunc, effort = [], 0, 0, {}
    summ, summ_trunc = [], 0   # v17: compaction summaries (requests sent with thinking off)
    late_tok, late_act = 0, 0   # v18: moves per generated token after each game's first 24 replies
    for p in paths:
        n_resp, prev_act = 0, 0
        with open(p, "r", encoding="utf-8", errors="replace") as f:
            for line in f:
                if '"event": "response"' not in line:
                    continue
                mc = _REP_COMPL.findall(line)
                if not mc:
                    continue
                if '"enable_thinking": false' in line:
                    summ.append(int(mc[-1]))
                    mf = _REP_FINISH.findall(line)
                    summ_trunc += bool(mf) and mf[-1] == "length"
                    continue
                compl.append(int(mc[-1]))
                ma = _re_late.findall(line)
                act = int(ma[-1]) if ma else prev_act
                if n_resp >= 24:
                    late_tok += int(mc[-1])
                    late_act += max(0, act - prev_act)
                prev_act = max(prev_act, act)
                n_resp += 1
                mr = _REP_REASON.findall(line)
                reason += int(mr[-1]) if mr else 0
                mf = _REP_FINISH.findall(line)
                trunc += bool(mf) and mf[-1] == "length"
                me = _REP_EFFORT.findall(line)
                key = me[-1] if me else "default(xhigh)"
                effort[key] = effort.get(key, 0) + 1
    if not compl:
        print(f"[report] request logs: none found in {run_dir}")
        return {}
    print(f"[report] replies: {len(compl)} from {len(paths)} logs, tokens median {_st_rep.median(compl):.0f} "
          f"mean {_st_rep.mean(compl):.0f}, hidden reasoning {100.0 * reason / max(1, sum(compl)):.1f}%, "
          f"truncated {trunc}, effort {effort}")
    if summ:
        print(f"[report] compaction summaries: {len(summ)}, tokens total {sum(summ)} mean {_st_rep.mean(summ):.0f}, "
              f"truncated {summ_trunc}")
    late = round(1000.0 * late_act / late_tok, 2) if late_tok else None
    print(f"[report] moves per 1K generated tokens after the first 24 replies of each game: {late} "
          f"(v7..v16 1.07-1.63, v17 0.57)")
    return {"act_per_1k_late": late, "summaries": len(summ), "summary_tokens": sum(summ),
            "summary_tokens_mean": round(_st_rep.mean(summ)) if summ else None, "summaries_truncated": summ_trunc,
            "reply_tokens_median": round(_st_rep.median(compl)), "reply_tokens_mean": round(_st_rep.mean(compl)),
            "reasoning_pct": round(100.0 * reason / max(1, sum(compl)), 1), "truncated_replies": trunc,
            "reply_effort": effort}


def _report_metrics(url):
    with _ur_rep.urlopen(url, timeout=10) as r:
        text = r.read().decode(errors="replace")
    rows = [l for l in text.splitlines() if l and not l.startswith("#") and any(k in l for k in _REP_METRICS)]
    for l in rows[:25]:
        print("[report] metric", l)
    if not rows:
        print("[report] metrics: no cache/token metrics exposed")


def _report_games(bm):
    import taaf.diagnostics as _dg
    print("[report] per game: score | levels | actions | tokens | tokens/action | tokens/level")
    games = {}
    for g in _dg._game_ids(bm):
        for r in _dg._game_runs_by_id(bm, g):
            acts, toks, lv = _dg._count_actions(r), _dg._total_tokens(r), r.levels_completed
            sc = _dg._live_score(r)
            games[g.split("-")[0]] = [round(sc, 2), lv]
            print(f"[report]   {g}: {sc:6.2f} | {lv}/{r.number_of_levels} | {acts} | {toks} | "
                  f"{toks / acts if acts else 0:,.0f} | {toks / lv if lv else 0:,.0f}")
    scores = [v[0] for v in games.values()]
    toks_all = sum(_dg._total_tokens(r) for r in bm.game_runs)
    try:
        elapsed = _dg._benchmark_elapsed_seconds(bm)
        job_tok_s = round(toks_all / elapsed, 1) if elapsed else None
    except Exception:
        job_tok_s = None
    return {"job_tok_s": job_tok_s, "games": len(games), "mean_score": round(_st_rep.mean(scores), 2) if scores else None,
            "levels": sum(v[1] for v in games.values()),
            "actions": sum(_dg._count_actions(r) for r in bm.game_runs),
            "tokens": sum(_dg._total_tokens(r) for r in bm.game_runs), "per_game": games}


def _dump_sglang_src(out_zip):
    # The installed SGLang python sources (wheel + any install-time patches), so the reply cache-miss can be
    # diagnosed against the exact code that served this run. Python files only; nothing is imported.
    import subprocess as _sp_rep
    import zipfile as _zip_rep
    from pathlib import Path as _P_rep
    probe = 'import importlib.util as u; print(list(u.find_spec("sglang").submodule_search_locations)[0])'
    root = _P_rep(_sp_rep.run([PYTHON, '-c', probe], capture_output=True, text=True, timeout=60).stdout.strip())
    files = sorted(root.rglob('*.py'))
    with _zip_rep.ZipFile(out_zip, 'w', _zip_rep.ZIP_DEFLATED) as z:
        for p in files:
            z.write(p, str(p.relative_to(root.parent)))
    print(f'[report] SGLang sources: {len(files)} python files from {root} -> {out_zip}')
    return {}


def _report_cache_diag(path):
    # v19: classify every follow-up request (same game, next turn) by why its previous reply was or was not reused:
    # track_ok = the server kept a Mamba checkpoint past the previous prompt; same_text = the new prompt repeats the
    # previous prompt+reply token for token; reused = the cache hit reached the end of the previous reply.
    import json as _json_cd
    fin, new = {}, []
    with open(path) as f:
        for line in f:
            try:
                r = _json_cd.loads(line)
            except ValueError:
                continue
            if r.get("ev") == "finish":
                fin[r["rid"]] = r
            elif r.get("ev") == "new":
                new.append(r)
    follow = [r for r in new if r.get("prev_rid") and r.get("prev_P", 0) > 8000 and r.get("lcp", 0) >= r["prev_P"] - 64
              and r.get("matched") is not None]
    cells, div_tok, reread = {}, {}, 0
    for r in follow:
        end = (r["prev_kv"] // 64) * 64
        track_ok = r.get("prev_last_track") is not None and r["prev_last_track"] > (r["prev_P"] // 64) * 64
        same_text = r["lcp"] >= r["prev_kv"] - 1
        reused = r["matched"] >= end - 64
        key = f"track_{'ok' if track_ok else 'no'}/text_{'same' if same_text else 'diff'}/{'reused' if reused else 'reread'}"
        cells[key] = cells.get(key, 0) + 1
        if not reused:
            reread += max(0, r["prev_kv"] - r["matched"])
        if not same_text and r.get("prev_tok") is not None:   # v20: which generated token the next prompt changed
            i = r["lcp"] - r.get("tok_from", 0)
            if 0 <= i < len(r["prev_tok"]):
                t = str(r["prev_tok"][i])
                div_tok[t] = div_tok.get(t, 0) + 1
    out = {"finish": len(fin), "new": len(new), "follow": len(follow),
           "reused_pct": round(100.0 * sum(v for k, v in cells.items() if k.endswith("/reused")) / max(1, len(follow)), 1),
           "cells": dict(sorted(cells.items())), "reread_tokens": reread,
           "div_tok_top": dict(sorted(div_tok.items(), key=lambda kv: -kv[1])[:5])}
    print(f"[report] cache diagnostics: {out}")
    return {"cache_diag": out}


# v35: vLLM's periodic stats lines (Save & Run only; the contest run starts vLLM with --disable-log-stats)
_VL_REP_GEN = _re_rep.compile(r"Avg generation throughput:\s*([\d.]+)")
_VL_REP_PROMPT = _re_rep.compile(r"Avg prompt throughput:\s*([\d.]+)")
_VL_REP_RUN = _re_rep.compile(r"Running:\s*(\d+)\s*req")
_VL_REP_WAIT = _re_rep.compile(r"Waiting:\s*(\d+)\s*req")
_VL_REP_KV = _re_rep.compile(r"KV cache usage:\s*([\d.]+)\s*%")
_VL_REP_HIT = _re_rep.compile(r"Prefix cache hit rate:\s*([\d.]+)\s*%")
_VL_REP_ACC = _re_rep.compile(r"Mean acceptance length:\s*([\d.]+)")
_VL_REP_KVSIZE = _re_rep.compile(r"GPU KV cache size:\s*([\d,]+)\s*tokens")


def _report_vllm_log(path):
    gen, prompt, running, waiting, kv, hit, acc = [], [], [], [], [], [], []
    kv_size, preempt, restarts = None, 0, 0
    with open(path, "r", errors="replace") as f:
        for line in f:
            if "Avg generation throughput" in line:
                vals = {}
                for key, rx, cast in (("gen", _VL_REP_GEN, float), ("prompt", _VL_REP_PROMPT, float),
                                      ("run", _VL_REP_RUN, int), ("wait", _VL_REP_WAIT, int),
                                      ("kv", _VL_REP_KV, float), ("hit", _VL_REP_HIT, float)):
                    m = rx.search(line)
                    vals[key] = cast(m.group(1)) if m else None
                if vals["gen"] is not None:
                    gen.append(vals["gen"])
                    running.append(vals["run"] if vals["run"] is not None else 0)
                    waiting.append(vals["wait"] or 0)
                    prompt.append(vals["prompt"] or 0.0)
                    if vals["kv"] is not None:
                        kv.append(vals["kv"])
                    if vals["hit"] is not None:
                        hit.append(vals["hit"])
            elif "acceptance length" in line:
                m = _VL_REP_ACC.search(line)
                if m:
                    acc.append(float(m.group(1)))
            elif "GPU KV cache size" in line:
                m = _VL_REP_KVSIZE.search(line)
                if m:
                    kv_size = int(m.group(1).replace(",", ""))
            if "==== watchdog restart" in line:
                restarts += 1
            elif "preempt" in line.lower():
                preempt += 1
    busy = [g for g, r in zip(gen, running) if r > 0]
    try:
        seats = int(VLLM_MAX_NUM_SEQS)
    except Exception:
        seats = max(running) if running else 0
    out = {"server_engine": "vllm", "kv_cache_tokens": kv_size, "preempt_lines": preempt, "server_restarts": restarts,
           "server_decode_tok_s_median": round(_st_rep.median(busy)) if busy else None,
           "server_prompt_tok_s_mean": round(_st_rep.mean(prompt)) if prompt else None,
           "spec_accept_median": round(_st_rep.median(acc), 2) if acc else None,
           "prefix_hit_median": round(_st_rep.median(hit), 1) if hit else None}
    if running:
        out.update({"mean_running": round(_st_rep.mean(running), 2),
                    "full_batch_pct": round(100.0 * sum(r >= seats for r in running) / len(running), 1),
                    "kv_mean": round(_st_rep.mean(kv) / 100.0, 2) if kv else None,
                    "kv_max": round(max(kv) / 100.0, 2) if kv else None,
                    "queue_pct": round(100.0 * sum(w > 0 for w in waiting) / len(waiting), 1)})
    print(f"[report] vLLM: KV pool {kv_size} tokens | generation tok/s median {out['server_decode_tok_s_median']} over "
          f"{len(busy)} busy samples | prompt tok/s mean {out['server_prompt_tok_s_mean']} | draft acceptance length "
          f"median {out['spec_accept_median']} | prefix-cache hit median {out['prefix_hit_median']}% | running mean "
          f"{out.get('mean_running')} of {seats} | KV mean/max {out.get('kv_mean')}/{out.get('kv_max')} | waiting in "
          f"{out.get('queue_pct')}% of samples | preempt lines {preempt} | server restarts {restarts}")
    return out


def run_server_report(bm, experiment=None):
    record = dict(experiment or {})
    _serve_report = _report_vllm_log if globals().get("SERVER_ENGINE_USED") == "vllm" else _report_serve_log
    for name, fn, arg in (("serve.log", _serve_report, LOG),
                          ("metrics", _report_metrics, f"http://127.0.0.1:{SERVED_MODEL_PORT}/metrics"),
                          ("games", _report_games, bm),
                          ("request logs", _report_requests, WORKING_DIR),
                          ("cache diag", _report_cache_diag, str(WORKING_DIR / 'cache_diag.jsonl'))):
        try:
            out = fn(arg)
            if isinstance(out, dict):
                record.update(out)
        except Exception as exc:
            print(f"[report] {name}: n/a ({type(exc).__name__}: {exc})"[:300])
    try:   # v16: the 10 games every earlier local run played, for comparison when extra games were added
        _m10 = [record['per_game'][g][0] for g in _LOCAL_SETS['mixed10'] if g in record.get('per_game', {})]
        if len(_m10) == len(_LOCAL_SETS['mixed10']):
            record['mean_score_mixed10'] = round(_st_rep.mean(_m10), 2)
            record['levels_mixed10'] = sum(record['per_game'][g][1] for g in _LOCAL_SETS['mixed10'])
    except Exception:
        pass
    # one line to paste back for comparison across runs; also saved to /kaggle/working
    try:
        record["addons"] = {k: v for k, v in addon_summary().items() if not k.endswith("_last_error")}
    except Exception:
        pass
    try:   # v18
        record["dedup_stats"] = dict(DEDUP_STATS)
    except Exception:
        pass
    try:   # v20
        record["exact_replay_stats"] = dict(XR_STATS)
    except Exception:
        pass
    try:   # v41
        record["trim_memory_stats"] = dict(TM_STATS)
    except Exception:
        pass
    line = _json_rep.dumps(record, sort_keys=True, default=str)
    print("EXPERIMENT_RESULT " + line, flush=True)
    try:
        with open(str(WORKING_DIR / "experiment_result.json"), "w") as f:
            f.write(line + "\n")
    except Exception as exc:
        print(f"[report] could not save experiment_result.json: {exc}")
    return record


## 9. Run the benchmark

In a real competition rerun (`KAGGLE_IS_COMPETITION_RERUN`), wait for the Kaggle gateway and
play the **live competition Arcade**. Otherwise — an interactive "Save & Run" — play the
competition's **bundled environment files offline**, with no gateway required, so the notebook
runs end-to-end without a submission. Teardown commands run afterward even if the run raises.

In [ ]:
print('Starting benchmark...')

# Build the live competition game list from the gateway's available environments.
def _competition_games():
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ["ARC_BASE_URL"],
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed zero environments.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


# Build the offline game list from the competition's bundled environment files.
def _offline_games(env_dir: str):
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    arcade = arc_agi.Arcade(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError(f"No offline environments found under {env_dir}.")
    game_ids = [game_id for game_id in game_ids if not any(game_id.startswith(g) for g in demo_excluded_games)]
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


# The gateway can take a while to come up; poll until it answers.
def _wait_for_gateway(base_url: str, timeout_s: float = 600.0) -> None:
    deadline = time.monotonic() + timeout_s
    last_error = ""
    while time.monotonic() < deadline:
        try:
            with urlopen(f"{base_url}api/games", timeout=10) as response:
                if response.status < 500:
                    return
        except Exception as exc:
            last_error = repr(exc)
        time.sleep(5)
    raise RuntimeError(f"Kaggle gateway did not become ready: {last_error}")


# Print the run preamble and persist the launcher's git status for diagnostics.
#print((BUNDLE_DIR / "preamble.txt").read_text())
(WORKING_DIR / "git_status.txt").write_text((BUNDLE_DIR / "git_status.txt").read_text())

# arc_agi reads RECORDINGS_DIR and ARC_API_KEY from env (ArcadeSpec carries neither); operation
# mode, environments dir, and base url are all passed explicitly via the spec, so no env is needed.
os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))

if TRUE_SUBMISSION:
    # Real submission: play the live competition Arcade served by the Kaggle gateway.
    os.environ.setdefault("ARC_API_KEY", "test-key-123")
    os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
    # The gateway boots asynchronously; wait before swapping in its game list.
    _wait_for_gateway(os.environ["ARC_BASE_URL"])
    bm.games = _competition_games()
    # v33: every game must start at once - the 532-min per-game budget and the deadline guard assume it. A game keeps
    # its place until it wins or its budget ends, so games beyond the concurrency limit would wait until the run ends.
    if ENABLE_GAME_COUNT_GUARD and len(bm.games) > int(bm.solver.concurrency):
        print(f'game-count guard: {len(bm.games)} games > concurrency {bm.solver.concurrency} -> {len(bm.games)}', flush=True)
        bm.solver.concurrency = len(bm.games)
    print(f'competition games: {len(bm.games)}, concurrency {bm.solver.concurrency}', flush=True)
else:
    # Interactive run: play the bundled competition environments offline (no gateway).
    # The competition's environment files ship alongside the wheelhouse in the competition dataset.
    competition_env_files = str(Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels").parent / "environment_files")
    bm.games = _offline_games(competition_env_files)

# Outside a real submission, stop ~10 min before the wall-clock budget for a graceful exit.
soft_end = None
if TRUE_SUBMISSION and ENABLE_DEADLINE_GUARD:
    # Every game starts at once (concurrency >= game count), so the per-game budget IS the play wall clock.
    # Cap it so play + drain end inside the notebook limit, however long the server/gateway took to come up.
    _elapsed = time.time() - NOTEBOOK_START_TIME
    _cap = KAGGLE_RUNTIME_LIMIT_S - DEADLINE_SAFETY_MARGIN_S - _elapsed - float(bm.solver.cancel_drain_timeout_s or 0)
    if _cap < bm.solver.max_runtime_s_per_game:
        print(f'deadline guard: max_runtime_s_per_game {bm.solver.max_runtime_s_per_game:.0f}s -> {_cap:.0f}s '
              f'(startup took {_elapsed:.0f}s)', flush=True)
        bm.solver.max_runtime_s_per_game = max(600.0, _cap)
if not TRUE_SUBMISSION:
    budget = float(getattr(target, "max_runtime_s", 0.0) or 0.0)
    if budget > 0:
        soft_end = datetime.fromtimestamp(NOTEBOOK_START_EPOCH) + timedelta(seconds=budget - min(600.0, budget / 2))

# Play the benchmark; teardown commands run even if the run raises.
try:
    await bm.run(soft_end_time=soft_end, runtime_environment=target, minimal_diagnostics=TRUE_SUBMISSION)
finally:
    _FL_STOP_ALL.set()   # v46: every First Light search stops with the benchmark
    print('ADDON_SUMMARY', json.dumps(addon_summary()), flush=True)
    print('FIRST_LIGHT_SUMMARY', json.dumps({k: v for k, v in FL_STATS.items()}), flush=True)
    print('DPRIME_SUMMARY', json.dumps({'enabled': ENABLE_DPRIME_SCHEDULER, 'fresh_games_queued': _fp_mod.FRESH['replaced'],
                                        'params': dict(_fp_mod.D_PRIME)}), flush=True)
    print('SERVER_SUMMARY', json.dumps({'engine': SERVER_ENGINE_USED, 'vllm': globals().get('VLLM_STATE')}, default=str), flush=True)
    if ENABLE_SERVER_REPORT and not TRUE_SUBMISSION:
        run_server_report(bm, {'profile': PROFILE, 'extra_env': EXTRA_ENV, 'spec_steps': SERVER_SPEC_STEPS,
                               'memory': MEMORY_PLACEMENT, 'all25': LOCAL_EVAL_ALL_25_GAMES,
                               'play_minutes': LOCAL_PLAY_MINUTES, 'wm_tools': ENABLE_WORLD_MODEL_TOOLS,
                               'dprime': ENABLE_DPRIME_SCHEDULER, 'game_set': LOCAL_GAME_SET,
                               'dprime_params': dict(_fp_mod.D_PRIME) if ENABLE_DPRIME_SCHEDULER else None,
                               'server_engine': SERVER_ENGINE_USED, 'vllm_state': globals().get('VLLM_STATE'),
                               'hints': ENABLE_ACTION_HINTS, 'nudge': ENABLE_TOOL_NUDGE,
                               'budget_meter': ENABLE_BUDGET_METER, 'dedup': ENABLE_HISTORY_DEDUP,
                               'cache_diag_hooked': CACHE_DIAG_FILE is not None,
                               'exact_replay': ENABLE_EXACT_REPLAY, 'mamba_cache': SERVER_MAMBA_CACHE,
                               'trim_memory': ENABLE_TRIM_MEMORY,
                               'reap_keep': VLLM_REAP_KEEP, 'spec_relax_t': VLLM_SPEC_RELAX_T,
                               'stuck_facts': ENABLE_STUCK_FACTS, 'keep_functions': ENABLE_KEEP_FUNCTIONS_ON_TIMEOUT,
                               'second_opinion': ENABLE_SECOND_OPINION, 'so_shared_rules': ENABLE_SO_SHARED_RULES,
                               'first_impressions': ENABLE_FIRST_IMPRESSIONS,
                               'v60': {'effort_ladder': V60_EFFORT_LADDER, 'light_trims': V60_LIGHT_TRIMS,
                                       'trim_keep_k': V60_TRIM_KEEP_K, 'no_diff_image': V60_NO_DIFF_IMAGE},
                               'so_catalogue': ENABLE_SO_CATALOGUE,
                               'cross_game_rules': ENABLE_CROSS_GAME_RULES,
                               'xg_library': ENABLE_XG_LIBRARY,
                               'max_num_seqs': VLLM_MAX_NUM_SEQS, 'history_budget_k': VLLM_HISTORY_BUDGET_K,
                               'first_light': ENABLE_FIRST_LIGHT,
                               'first_light_stats': {k: v for k, v in FL_STATS.items() if k != 'last_error'},
                               'game_catalogue': ENABLE_GAME_CATALOGUE,
                               'gateway_patience': ENABLE_GATEWAY_PATIENCE,
                               'scorecard_keepalive': ENABLE_SCORECARD_KEEPALIVE})
    if ENABLE_SERVER_WATCHDOG:
        print('WATCHDOG_SUMMARY', json.dumps(SERVER_WATCHDOG), flush=True)
    _SV_STOP.set()
    print('SURVIVAL_SUMMARY', json.dumps(SURVIVAL_STATS), flush=True)
if not TRUE_SUBMISSION and ENABLE_SURVIVAL_SMOKE:
    run_survival_smoke(competition_env_files, WORKING_DIR / 'survival_smoke.json')
if not TRUE_SUBMISSION and ENABLE_FIRST_LIGHT:
    run_first_light_smoke(competition_env_files, WORKING_DIR / 'first_light_smoke.json')
if not TRUE_SUBMISSION:
    # An offline run isn't scored, but Kaggle still expects a submission.parquet output.
    import pandas as pd

    pd.DataFrame(
        [["1_0", "1", True, 1]],
        columns=["row_id", "game_id", "end_of_game", "score"],
    ).to_parquet(WORKING_DIR / "submission.parquet", index=False)

## 10. Show the diagnostics

A non-submission run writes `diagnostics.html` to `/kaggle/working`; it is rendered inline below
(and downloadable from the working directory). You should be able to click around through the links.

In [ ]:
from html import escape

from IPython.display import HTML, display

diagnostics_html = WORKING_DIR / "diagnostics.html"
if diagnostics_html.is_file():
    # Isolate the full document in an iframe so its styles don't leak into the notebook.
    display(
        HTML(
            f'<iframe srcdoc="{escape(diagnostics_html.read_text(), quote=True)}" '
            'width="100%" height="900" style="border:0"></iframe>'
        )
    )
else:
    print("No diagnostics.html — minimal diagnostics (real submission) suppresses it.")